### I found that bulk is not really necessary and also removing bulk, low demand and zero demand products might improve the model fot the final model.

## This might be the last modelling starting with creating a dataset for that.

In [2]:
from __future__ import annotations

import hashlib
import json
import os
import shutil
import stat
import uuid
from datetime import datetime, timezone
from pathlib import Path
from zoneinfo import ZoneInfo

import numpy as np
import pandas as pd

# =============================================================================
# EDEN NORMAL-DEMAND MODEL V2
# ND01 — CREATE ORGANISED WORKSPACE AND COPY CANONICAL SOURCE DATASET
# =============================================================================

PROJECT_ROOT = Path("/Users/ryansmac/Desktop/Meng Project")
EDEN_ROOT = PROJECT_ROOT / "eden_datasets"
SOURCE_FILENAME = "UL_EDEN_canonical_product_daily_demand_forecasting_final.csv"

MODEL_FOLDER_NAME = "eden_normal_demand_model_v2"
MODEL_ROOT = EDEN_ROOT / MODEL_FOLDER_NAME

MEMORY_ROOT = MODEL_ROOT / "00_project_memory"
DATASET_ROOT = MODEL_ROOT / "01_datasets"
SOURCE_COPY_ROOT = DATASET_ROOT / "00_source_copy"
INTERMEDIATE_DATA_ROOT = DATASET_ROOT / "01_intermediate"
FINAL_DATA_ROOT = DATASET_ROOT / "02_final"
FEATURE_ROOT = MODEL_ROOT / "02_feature_engineering"
MODEL_CANDIDATE_ROOT = MODEL_ROOT / "03_models" / "00_candidates"
MODEL_FINAL_ROOT = MODEL_ROOT / "03_models" / "01_final"
PREDICTION_ROOT = MODEL_ROOT / "04_predictions"
VALIDATION_ROOT = MODEL_ROOT / "05_validation"
REPORT_ROOT = MODEL_ROOT / "06_reports"
NOTEBOOK_ROOT = MODEL_ROOT / "07_notebooks"
CHECKPOINT_ROOT = MODEL_ROOT / "08_checkpoints"
LOG_ROOT = MODEL_ROOT / "09_logs"

SOURCE_COPY_PATH = SOURCE_COPY_ROOT / SOURCE_FILENAME
SOURCE_SNAPSHOT_PATH = SOURCE_COPY_ROOT / "ND01_source_snapshot.json"
SOURCE_README_PATH = SOURCE_COPY_ROOT / "README.md"

README_PATH = MODEL_ROOT / "README.md"
AGENTS_PATH = MODEL_ROOT / "AGENTS.md"
GITIGNORE_PATH = MODEL_ROOT / ".gitignore"

PROJECT_CONTEXT_PATH = MEMORY_ROOT / "PROJECT_CONTEXT.md"
WORKFLOW_PATH = MEMORY_ROOT / "WORKFLOW.md"
DECISIONS_PATH = MEMORY_ROOT / "DECISIONS.md"
FILES_AND_PATHS_PATH = MEMORY_ROOT / "FILES_AND_PATHS.md"
METRICS_AND_RESULTS_PATH = MEMORY_ROOT / "METRICS_AND_RESULTS.md"
CURRENT_HANDOFF_PATH = MEMORY_ROOT / "CURRENT_HANDOFF.md"
CHAT_INDEX_PATH = MEMORY_ROOT / "CHAT_INDEX.md"
DATA_DICTIONARY_PATH = MEMORY_ROOT / "SOURCE_DATA_DICTIONARY.md"

VALIDATION_PATH = VALIDATION_ROOT / "ND01_source_copy_validation.csv"
MANIFEST_PATH = VALIDATION_ROOT / "ND01_artifact_hash_manifest.csv"
CHECKPOINT_PATH = CHECKPOINT_ROOT / "ND01_checkpoint.json"
CHECKPOINT_SHA_PATH = CHECKPOINT_ROOT / "ND01_checkpoint.sha256"
LOCK_PATH = CHECKPOINT_ROOT / "ND01_workspace_and_source_copy_lock.json"
LOCK_SHA_PATH = CHECKPOINT_ROOT / "ND01_workspace_and_source_copy_lock.sha256"
LOG_PATH = LOG_ROOT / "ND01_workspace_setup_log.txt"

STEP_ID = "ND01"
STATUS = "ND01_SOURCE_WORKSPACE_CREATED_READY_FOR_ND02"
NOW_UTC = datetime.now(timezone.utc)
NOW_LOCAL = NOW_UTC.astimezone(ZoneInfo("Europe/Dublin"))


def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


def write_text(path: Path, text: str) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(text, encoding="utf-8")


def write_json(path: Path, payload: dict) -> None:
    write_text(path, json.dumps(payload, indent=2, ensure_ascii=False) + "\n")


def write_csv(path: Path, frame: pd.DataFrame) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    frame.to_csv(path, index=False)


def make_read_only(path: Path) -> None:
    if path.is_file():
        path.chmod(stat.S_IRUSR | stat.S_IRGRP | stat.S_IROTH)


def locate_source_dataset() -> Path:
    expected = EDEN_ROOT / SOURCE_FILENAME
    if expected.is_file():
        return expected

    matches = [
        path
        for path in EDEN_ROOT.rglob(SOURCE_FILENAME)
        if MODEL_FOLDER_NAME not in path.parts
    ]

    if len(matches) == 1:
        return matches[0]
    if not matches:
        raise FileNotFoundError(
            "The canonical source dataset was not found.\n"
            f"Expected location:\n{expected}"
        )
    raise RuntimeError(
        "More than one matching canonical source dataset was found.\n"
        + "\n".join(f"- {path}" for path in matches)
    )


# =============================================================================
# PRE-FLIGHT SAFETY
# =============================================================================

if not PROJECT_ROOT.is_dir():
    raise FileNotFoundError(f"Project root does not exist:\n{PROJECT_ROOT}")
if not EDEN_ROOT.is_dir():
    raise FileNotFoundError(f"Eden dataset root does not exist:\n{EDEN_ROOT}")

SOURCE_PATH = locate_source_dataset()

if MODEL_ROOT.exists():
    raise FileExistsError(
        "The modelling workspace already exists, so ND01 will not overwrite it.\n"
        f"Existing path:\n{MODEL_ROOT}"
    )

for old_stage in EDEN_ROOT.glob(f".{MODEL_FOLDER_NAME}_staging_*"):
    if old_stage.is_dir():
        shutil.rmtree(old_stage)

STAGING_ROOT = EDEN_ROOT / f".{MODEL_FOLDER_NAME}_staging_{uuid.uuid4().hex}"
STAGING_ROOT.mkdir(parents=True, exist_ok=False)

relative_directories = [
    Path("00_project_memory"),
    Path("01_datasets/00_source_copy"),
    Path("01_datasets/01_intermediate"),
    Path("01_datasets/02_final"),
    Path("02_feature_engineering"),
    Path("03_models/00_candidates"),
    Path("03_models/01_final"),
    Path("04_predictions"),
    Path("05_validation"),
    Path("06_reports"),
    Path("07_notebooks"),
    Path("08_checkpoints"),
    Path("09_logs"),
]

try:
    for relative_directory in relative_directories:
        (STAGING_ROOT / relative_directory).mkdir(parents=True, exist_ok=True)

    staged_source_copy = STAGING_ROOT / SOURCE_COPY_PATH.relative_to(MODEL_ROOT)
    shutil.copy2(SOURCE_PATH, staged_source_copy)

    source_sha256 = sha256_file(SOURCE_PATH)
    copied_sha256 = sha256_file(staged_source_copy)
    if source_sha256 != copied_sha256:
        raise AssertionError("The copied source does not match the original source.")

    # =========================================================================
    # SOURCE VALIDATION
    # =========================================================================

    source_df = pd.read_csv(staged_source_copy, low_memory=False)

    required_columns = {
        "Date",
        "CanonicalProductID",
        "NormalDemand",
        "BulkDemand",
        "TotalDemand",
    }
    missing_columns = sorted(required_columns - set(source_df.columns))
    if missing_columns:
        raise AssertionError(
            "The source dataset is missing required columns:\n"
            + "\n".join(f"- {column}" for column in missing_columns)
        )

    source_df["Date"] = pd.to_datetime(source_df["Date"], errors="raise")
    source_df["CanonicalProductID"] = source_df["CanonicalProductID"].astype(str)

    demand_columns = ["NormalDemand", "BulkDemand", "TotalDemand"]
    for column in demand_columns:
        source_df[column] = pd.to_numeric(source_df[column], errors="raise").astype(float)

    demand_array = source_df[demand_columns].to_numpy(dtype=float)
    non_finite_demand_values = int((~np.isfinite(demand_array)).sum())
    negative_demand_rows = int((source_df[demand_columns] < 0).any(axis=1).sum())
    duplicate_product_date_rows = int(
        source_df.duplicated(["Date", "CanonicalProductID"]).sum()
    )

    reconciliation_difference = (
        source_df["NormalDemand"]
        + source_df["BulkDemand"]
        - source_df["TotalDemand"]
    )
    maximum_reconciliation_difference = float(reconciliation_difference.abs().max())

    if non_finite_demand_values != 0:
        raise AssertionError("Non-finite demand values were found.")
    if negative_demand_rows != 0:
        raise AssertionError("Negative demand rows were found.")
    if duplicate_product_date_rows != 0:
        raise AssertionError("Duplicate product-date rows were found.")
    if maximum_reconciliation_difference > 1e-9:
        raise AssertionError(
            "NormalDemand + BulkDemand does not reconcile to TotalDemand."
        )

    row_count = int(len(source_df))
    column_count = int(len(source_df.columns))
    product_count = int(source_df["CanonicalProductID"].nunique())
    operating_date_count = int(source_df["Date"].nunique())
    minimum_date = source_df["Date"].min().date().isoformat()
    maximum_date = source_df["Date"].max().date().isoformat()

    normal_demand_total = float(source_df["NormalDemand"].sum())
    bulk_demand_total = float(source_df["BulkDemand"].sum())
    total_demand_total = float(source_df["TotalDemand"].sum())
    bulk_share_percentage = (
        100.0 * bulk_demand_total / total_demand_total
        if total_demand_total != 0
        else np.nan
    )

    source_snapshot = {
        "StepID": STEP_ID,
        "Status": STATUS,
        "CreatedUTC": NOW_UTC.isoformat(),
        "CreatedLocal": NOW_LOCAL.isoformat(),
        "OriginalSourcePath": str(SOURCE_PATH),
        "CopiedSourcePath": str(SOURCE_COPY_PATH),
        "OriginalSourceSHA256": source_sha256,
        "CopiedSourceSHA256": copied_sha256,
        "CopyIsByteIdentical": True,
        "SourceBytes": int(SOURCE_PATH.stat().st_size),
        "Rows": row_count,
        "Columns": column_count,
        "Products": product_count,
        "OperatingDates": operating_date_count,
        "MinimumDate": minimum_date,
        "MaximumDate": maximum_date,
        "NormalDemandTotal": normal_demand_total,
        "BulkDemandTotal": bulk_demand_total,
        "TotalDemandTotal": total_demand_total,
        "BulkSharePercentage": bulk_share_percentage,
        "MaximumDemandReconciliationDifference": maximum_reconciliation_difference,
        "DuplicateProductDateRows": duplicate_product_date_rows,
        "NegativeDemandRows": negative_demand_rows,
        "NonFiniteDemandValues": non_finite_demand_values,
        "SourceModifiedDuringStep": False,
        "CopiedDatasetModifiedDuringStep": False,
    }

    staged_source_snapshot = STAGING_ROOT / SOURCE_SNAPSHOT_PATH.relative_to(MODEL_ROOT)
    write_json(staged_source_snapshot, source_snapshot)

    validation = pd.DataFrame(
        [
            {
                "Check": "Original source exists",
                "Expected": True,
                "Actual": SOURCE_PATH.is_file(),
                "Passed": SOURCE_PATH.is_file(),
            },
            {
                "Check": "Copied source exists",
                "Expected": True,
                "Actual": staged_source_copy.is_file(),
                "Passed": staged_source_copy.is_file(),
            },
            {
                "Check": "Source and copied hashes match",
                "Expected": source_sha256,
                "Actual": copied_sha256,
                "Passed": source_sha256 == copied_sha256,
            },
            {
                "Check": "Required demand columns present",
                "Expected": True,
                "Actual": len(missing_columns) == 0,
                "Passed": len(missing_columns) == 0,
            },
            {
                "Check": "Duplicate product-date rows",
                "Expected": 0,
                "Actual": duplicate_product_date_rows,
                "Passed": duplicate_product_date_rows == 0,
            },
            {
                "Check": "Negative demand rows",
                "Expected": 0,
                "Actual": negative_demand_rows,
                "Passed": negative_demand_rows == 0,
            },
            {
                "Check": "Non-finite demand values",
                "Expected": 0,
                "Actual": non_finite_demand_values,
                "Passed": non_finite_demand_values == 0,
            },
            {
                "Check": "Maximum demand reconciliation difference",
                "Expected": "<= 1e-9",
                "Actual": maximum_reconciliation_difference,
                "Passed": maximum_reconciliation_difference <= 1e-9,
            },
            {
                "Check": "Original source modified",
                "Expected": False,
                "Actual": False,
                "Passed": True,
            },
            {
                "Check": "Dataset transformed during ND01",
                "Expected": False,
                "Actual": False,
                "Passed": True,
            },
        ]
    )

    if not validation["Passed"].all():
        raise AssertionError(
            "ND01 validation failed:\n"
            + validation.loc[~validation["Passed"]].to_string(index=False)
        )

    staged_validation = STAGING_ROOT / VALIDATION_PATH.relative_to(MODEL_ROOT)
    write_csv(staged_validation, validation)

    # =========================================================================
    # DOCUMENTATION
    # No Markdown triple-backtick fences are used inside these Python strings.
    # =========================================================================

    folder_tree = "\n".join(
        [
            f"{MODEL_FOLDER_NAME}/",
            "  AGENTS.md",
            "  README.md",
            "  00_project_memory/",
            "  01_datasets/",
            "    00_source_copy/",
            "    01_intermediate/",
            "    02_final/",
            "  02_feature_engineering/",
            "  03_models/",
            "    00_candidates/",
            "    01_final/",
            "  04_predictions/",
            "  05_validation/",
            "  06_reports/",
            "  07_notebooks/",
            "  08_checkpoints/",
            "  09_logs/",
        ]
    )

    readme_text = f"""# Eden Normal-Demand Model V2

## Purpose

This workspace contains the revised Eden Restaurant forecasting workflow.
The revised model will forecast ordinary normal product demand. Confirmed bulk
orders will be handled as known external operational inputs.

## Operational quantity rule

Required quantity = forecast normal demand + confirmed bulk-order quantity

## Current status

- Step completed: {STEP_ID}
- Status: {STATUS}
- Canonical source copied unchanged: {SOURCE_COPY_PATH}
- No feature engineering has been performed.
- No products have been filtered.
- No model has been fitted.
- No previous Eden artefact has been overwritten.

## Modelling principles

1. Use NormalDemand as the revised model target.
2. Retain BulkDemand for audit and reconciliation only.
3. Rebuild lag and rolling predictors using past NormalDemand only.
4. Determine product eligibility using information available before each forecast.
5. Focus the main model on established high- and moderate-demand products.
6. Retain low-demand and cold-start products for fallback and audit.
7. Report forecasting error together with demand coverage.
8. Use chronological, leakage-safe evaluation.
9. Preserve the original locked Eden models as benchmarks.
10. Do not describe 100 minus WAPE as conventional accuracy.

## Folder structure

{folder_tree}

## Next step

ND02 will create the leakage-safe normal-demand panel, prior-only product
eligibility fields, demand coverage summaries and fallback routes.
"""

    agents_text = f"""# AGENTS.md

## Project

Eden Restaurant normal-demand forecasting model, version 2.

## Authoritative status

- Completed step: {STEP_ID}
- Status: {STATUS}
- Next step: ND02
- Workspace root: {MODEL_ROOT}
- Immutable source copy: {SOURCE_COPY_PATH}

## Objective

Create a product-level daily forecasting system for ordinary restaurant demand.
Bulk orders are treated as confirmed external quantities rather than uncertain
routine demand that must be inferred from point-of-sale history.

## Non-negotiable rules

1. The revised target is NormalDemand.
2. Do not train the revised model on TotalDemand.
3. Do not create revised historical predictors from TotalDemand.
4. All lags, rolling statistics and expanding statistics must use prior NormalDemand.
5. Do not classify products using complete-period or future demand.
6. Product rank, segment and eligibility must be recalculated from prior data only.
7. Low-demand and cold-start products must remain in the data and use an explicit fallback.
8. Report WAPE together with the percentage of normal demand covered.
9. Bias is prediction minus actual.
10. Do not round forecasts before scoring.
11. Use chronological validation only.
12. March 2026 has already been examined and is not a new untouched final test.
13. Never overwrite an existing lock, checkpoint, source snapshot or final model.
14. Never modify the CSV in 01_datasets/00_source_copy.
15. Do not commit raw or derived Eden datasets to Git.

## Planned routing

Established high/moderate products -> main normal-demand model
Low-demand/zero-heavy/insufficient-history products -> explicit fallback
Confirmed bulk orders -> external quantity added after prediction

## Safe working procedure

1. Read 00_project_memory/CURRENT_HANDOFF.md.
2. Verify the previous lock and checkpoint.
3. Write new artefacts into staging.
4. Validate all outputs.
5. Calculate SHA-256 hashes.
6. Commit atomically.
7. Create a new immutable lock.
"""

    source_readme_text = f"""# Canonical Source Copy

This directory contains the immutable source snapshot for the revised
normal-demand modelling workflow.

Original file: {SOURCE_PATH}
Copied file: {SOURCE_COPY_PATH}
Original SHA-256: {source_sha256}
Copied SHA-256: {copied_sha256}
Byte-identical copy: True

Rules:
- Do not edit this CSV.
- Do not add engineered features to this CSV.
- Do not remove bulk rows from this CSV.
- Do not remove low-demand products from this CSV.
- Write transformations to 01_datasets/01_intermediate.
- Write final prepared datasets to 01_datasets/02_final.
"""

    project_context_text = f"""# Project Context

## Objective

Develop an operational forecasting prototype for Eden Restaurant that predicts
product demand and supports food-waste prevention.

## Revised modelling objective

Forecast NormalDemand rather than TotalDemand. Confirmed bulk orders will be
added externally to the predicted normal demand.

## Source profile

- Rows: {row_count:,}
- Columns: {column_count:,}
- Products: {product_count:,}
- Operating dates: {operating_date_count:,}
- Date range: {minimum_date} to {maximum_date}
- Normal demand: {normal_demand_total:,.6f}
- Bulk demand: {bulk_demand_total:,.6f}
- Total demand: {total_demand_total:,.6f}
- Bulk share: {bulk_share_percentage:.6f}%

## Current status

The workspace and immutable source snapshot are complete. No modelling
transformation has yet been applied.
"""

    workflow_text = """# Workflow

## Completed

### ND01 — Workspace and source snapshot

- Created the complete modelling workspace.
- Copied the canonical product-day dataset unchanged.
- Verified source integrity and demand reconciliation.
- Created agent documentation, project memory, validation, hashes and lock.

## Planned

### ND02 — Normal-demand panel and product eligibility
- Preserve all source rows.
- Set NormalDemand as the modelling target.
- Create prior-only product history and ranking fields.
- Define main-model and fallback routes.
- Quantify demand coverage.

### ND03 — Normal-demand feature engineering
- Rebuild lag and rolling features from NormalDemand.
- Produce chronological model-selection datasets.
- Complete leakage and missingness audits.

### ND04 — Baselines and fallback evaluation

### ND05 — Candidate daily models

### ND06 — Tuning, calibration and final selection

### ND07 — Daily-to-weekly aggregation and rolling updates

### ND08 — Arbitrary future-date inference pipeline

### ND09 — Untouched future evaluation
"""

    decisions_text = """# Decisions

## ND01 locked decisions

1. The canonical product-day dataset is the source of truth.
2. The revised target is NormalDemand.
3. BulkDemand remains in source and audit data but is excluded from the statistical target.
4. Confirmed bulk quantities will be added externally.
5. Low-demand and zero-heavy products will not be deleted from the source.
6. Main-model eligibility will be calculated dynamically from prior data.
7. Insufficient-history and outside-scope products will use an explicit fallback.
8. The original locked daily model remains the benchmark.
9. All model evaluation must be chronological and leakage-safe.
10. Accuracy results must be accompanied by demand coverage.
"""

    files_and_paths_text = f"""# Files and Paths

Workspace root: {MODEL_ROOT}
Original canonical source: {SOURCE_PATH}
Immutable copied source: {SOURCE_COPY_PATH}
Source snapshot: {SOURCE_SNAPSHOT_PATH}
Validation: {VALIDATION_PATH}
Hash manifest: {MANIFEST_PATH}
Checkpoint: {CHECKPOINT_PATH}
Lock: {LOCK_PATH}
Intermediate datasets: {INTERMEDIATE_DATA_ROOT}
Final datasets: {FINAL_DATA_ROOT}
Feature artefacts: {FEATURE_ROOT}
"""

    metrics_text = f"""# Metrics and Results

## ND01 source profile

- Rows: {row_count:,}
- Columns: {column_count:,}
- Products: {product_count:,}
- Operating dates: {operating_date_count:,}
- Minimum date: {minimum_date}
- Maximum date: {maximum_date}
- Normal demand total: {normal_demand_total:,.6f}
- Bulk demand total: {bulk_demand_total:,.6f}
- Total demand total: {total_demand_total:,.6f}
- Bulk share of total demand: {bulk_share_percentage:.6f}%
- Maximum reconciliation difference: {maximum_reconciliation_difference:.12g}
- Duplicate product-date rows: {duplicate_product_date_rows}
- Negative demand rows: {negative_demand_rows}
- Non-finite demand values: {non_finite_demand_values}

No forecast or fitted model was produced in ND01.
"""

    current_handoff_text = f"""# Current Handoff

- Current completed step: {STEP_ID}
- Status: {STATUS}
- Updated local time: {NOW_LOCAL.isoformat()}
- Workspace root: {MODEL_ROOT}
- Source copied unchanged: {SOURCE_COPY_PATH}
- Source SHA-256: {source_sha256}
- Planned target: NormalDemand
- Bulk included in statistical target: no
- Low-demand products deleted: no
- Models fitted: no
- Next step: ND02

## ND02 objective

Create a leakage-safe normal-demand panel that preserves all products,
calculates product history from prior observations only, identifies established
high/moderate-demand products dynamically, routes other products to fallbacks,
and reports normal-demand coverage before any model is fitted.
"""

    chat_index_text = f"""# Chat and Decision Index

| Date and time | Step | Topic | Status |
|---|---|---|---|
| {NOW_LOCAL.isoformat()} | ND01 | Create normal-demand workspace and copy canonical source | {STATUS} |
"""

    data_dictionary_lines = [
        "# Source Data Dictionary",
        "",
        f"Source file: {SOURCE_COPY_PATH}",
        "",
        "| Column | Data type | Non-null rows | Unique values |",
        "|---|---:|---:|---:|",
    ]
    for column in source_df.columns:
        data_dictionary_lines.append(
            f"| {column} | {source_df[column].dtype} | "
            f"{int(source_df[column].notna().sum()):,} | "
            f"{int(source_df[column].nunique(dropna=True)):,} |"
        )
    data_dictionary_lines.extend(
        [
            "",
            "## Demand-field interpretation",
            "",
            "- NormalDemand: ordinary demand to be modelled in version 2.",
            "- BulkDemand: audit-only demand from confirmed or exceptional bulk transactions.",
            "- TotalDemand: reconciliation field equal to NormalDemand plus BulkDemand.",
            "",
            "No feature engineering was applied in ND01.",
        ]
    )
    data_dictionary_text = "\n".join(data_dictionary_lines) + "\n"

    gitignore_text = """# Eden data and generated model artefacts
01_datasets/00_source_copy/*.csv
01_datasets/01_intermediate/**
01_datasets/02_final/**
03_models/**/*.joblib
03_models/**/*.pkl
03_models/**/*.pickle
04_predictions/**
*.parquet
*.feather

# Jupyter
.ipynb_checkpoints/

# Python
__pycache__/
*.py[cod]

# macOS
.DS_Store
"""

    documentation_files = {
        README_PATH: readme_text,
        AGENTS_PATH: agents_text,
        GITIGNORE_PATH: gitignore_text,
        SOURCE_README_PATH: source_readme_text,
        PROJECT_CONTEXT_PATH: project_context_text,
        WORKFLOW_PATH: workflow_text,
        DECISIONS_PATH: decisions_text,
        FILES_AND_PATHS_PATH: files_and_paths_text,
        METRICS_AND_RESULTS_PATH: metrics_text,
        CURRENT_HANDOFF_PATH: current_handoff_text,
        CHAT_INDEX_PATH: chat_index_text,
        DATA_DICTIONARY_PATH: data_dictionary_text,
    }

    for final_path, text in documentation_files.items():
        staged_path = STAGING_ROOT / final_path.relative_to(MODEL_ROOT)
        write_text(staged_path, text)

    log_text = "\n".join(
        [
            f"Step: {STEP_ID}",
            f"Status: {STATUS}",
            f"Created local: {NOW_LOCAL.isoformat()}",
            f"Created UTC: {NOW_UTC.isoformat()}",
            f"Original source: {SOURCE_PATH}",
            f"Copied source: {SOURCE_COPY_PATH}",
            f"Source SHA256: {source_sha256}",
            f"Rows: {row_count}",
            f"Columns: {column_count}",
            f"Products: {product_count}",
            f"Operating dates: {operating_date_count}",
            f"Normal demand: {normal_demand_total}",
            f"Bulk demand: {bulk_demand_total}",
            f"Total demand: {total_demand_total}",
            "Source modified: False",
            "Dataset transformed: False",
            "Models fitted: False",
            "",
        ]
    )
    staged_log = STAGING_ROOT / LOG_PATH.relative_to(MODEL_ROOT)
    write_text(staged_log, log_text)

    # =========================================================================
    # MANIFEST
    # =========================================================================

    manifest_exclusions = {
        MANIFEST_PATH.relative_to(MODEL_ROOT),
        CHECKPOINT_PATH.relative_to(MODEL_ROOT),
        CHECKPOINT_SHA_PATH.relative_to(MODEL_ROOT),
        LOCK_PATH.relative_to(MODEL_ROOT),
        LOCK_SHA_PATH.relative_to(MODEL_ROOT),
    }

    staged_artifact_files = sorted(
        path
        for path in STAGING_ROOT.rglob("*")
        if path.is_file() and path.relative_to(STAGING_ROOT) not in manifest_exclusions
    )

    manifest = pd.DataFrame(
        [
            {
                "RelativePath": str(path.relative_to(STAGING_ROOT)),
                "Bytes": int(path.stat().st_size),
                "SHA256": sha256_file(path),
            }
            for path in staged_artifact_files
        ]
    ).sort_values("RelativePath").reset_index(drop=True)

    staged_manifest = STAGING_ROOT / MANIFEST_PATH.relative_to(MODEL_ROOT)
    write_csv(staged_manifest, manifest)
    manifest_sha256 = sha256_file(staged_manifest)

    # =========================================================================
    # CHECKPOINT
    # =========================================================================

    checkpoint = {
        "StepID": STEP_ID,
        "Status": STATUS,
        "CreatedUTC": NOW_UTC.isoformat(),
        "CreatedLocal": NOW_LOCAL.isoformat(),
        "WorkspaceRoot": str(MODEL_ROOT),
        "Source": {
            "OriginalPath": str(SOURCE_PATH),
            "CopiedPath": str(SOURCE_COPY_PATH),
            "OriginalSHA256": source_sha256,
            "CopiedSHA256": copied_sha256,
            "ByteIdentical": True,
        },
        "SourceProfile": {
            "Rows": row_count,
            "Columns": column_count,
            "Products": product_count,
            "OperatingDates": operating_date_count,
            "MinimumDate": minimum_date,
            "MaximumDate": maximum_date,
            "NormalDemandTotal": normal_demand_total,
            "BulkDemandTotal": bulk_demand_total,
            "TotalDemandTotal": total_demand_total,
            "BulkSharePercentage": bulk_share_percentage,
        },
        "Manifest": {
            "Path": str(MANIFEST_PATH),
            "SHA256": manifest_sha256,
            "FilesListed": int(len(manifest)),
        },
        "Safety": {
            "OriginalSourceModified": False,
            "CopiedSourceModified": False,
            "RowsRemoved": False,
            "BulkRemoved": False,
            "LowDemandProductsRemoved": False,
            "FeaturesGenerated": False,
            "ModelsFitted": False,
            "ExistingModelLocksModified": False,
        },
        "ReadyForND02": True,
        "NextStep": "ND02",
    }

    staged_checkpoint = STAGING_ROOT / CHECKPOINT_PATH.relative_to(MODEL_ROOT)
    write_json(staged_checkpoint, checkpoint)
    checkpoint_sha256 = sha256_file(staged_checkpoint)

    staged_checkpoint_sha = STAGING_ROOT / CHECKPOINT_SHA_PATH.relative_to(MODEL_ROOT)
    write_text(
        staged_checkpoint_sha,
        f"{checkpoint_sha256}  {CHECKPOINT_PATH.name}\n",
    )

    # =========================================================================
    # LOCK
    # =========================================================================

    lock = {
        "StepID": STEP_ID,
        "Status": STATUS,
        "CreatedUTC": NOW_UTC.isoformat(),
        "WorkspaceRoot": str(MODEL_ROOT),
        "ImmutableSourceCopy": {
            "Path": str(SOURCE_COPY_PATH),
            "SHA256": copied_sha256,
        },
        "SourceSnapshot": {
            "Path": str(SOURCE_SNAPSHOT_PATH),
            "SHA256": sha256_file(staged_source_snapshot),
        },
        "Validation": {
            "Path": str(VALIDATION_PATH),
            "SHA256": sha256_file(staged_validation),
            "AllChecksPassed": bool(validation["Passed"].all()),
        },
        "Manifest": {
            "Path": str(MANIFEST_PATH),
            "SHA256": manifest_sha256,
        },
        "Checkpoint": {
            "Path": str(CHECKPOINT_PATH),
            "SHA256": checkpoint_sha256,
        },
        "SafetyAssertions": checkpoint["Safety"],
        "ReadyForND02": True,
        "NextStep": "ND02",
    }

    staged_lock = STAGING_ROOT / LOCK_PATH.relative_to(MODEL_ROOT)
    write_json(staged_lock, lock)
    lock_sha256 = sha256_file(staged_lock)

    staged_lock_sha = STAGING_ROOT / LOCK_SHA_PATH.relative_to(MODEL_ROOT)
    write_text(staged_lock_sha, f"{lock_sha256}  {LOCK_PATH.name}\n")

    required_staged_files = [
        staged_source_copy,
        staged_source_snapshot,
        staged_validation,
        staged_manifest,
        staged_checkpoint,
        staged_checkpoint_sha,
        staged_lock,
        staged_lock_sha,
        STAGING_ROOT / "README.md",
        STAGING_ROOT / "AGENTS.md",
        STAGING_ROOT / "00_project_memory/CURRENT_HANDOFF.md",
    ]
    missing_staged_files = [path for path in required_staged_files if not path.is_file()]
    if missing_staged_files:
        raise AssertionError(
            "Required staged files are missing:\n"
            + "\n".join(f"- {path}" for path in missing_staged_files)
        )

    if sha256_file(staged_source_copy) != source_sha256:
        raise AssertionError("The staged source-copy hash changed unexpectedly.")

    os.replace(STAGING_ROOT, MODEL_ROOT)

    for protected_path in [
        SOURCE_COPY_PATH,
        SOURCE_SNAPSHOT_PATH,
        VALIDATION_PATH,
        MANIFEST_PATH,
        CHECKPOINT_PATH,
        CHECKPOINT_SHA_PATH,
        LOCK_PATH,
        LOCK_SHA_PATH,
    ]:
        make_read_only(protected_path)

except Exception:
    if STAGING_ROOT.exists():
        shutil.rmtree(STAGING_ROOT)
    raise


# =============================================================================
# FINAL OUTPUT
# =============================================================================

print("=" * 100)
print("EDEN NORMAL-DEMAND MODEL V2 — ND01 COMPLETE")
print("=" * 100)
print(f"Status: {STATUS}")
print(f"Local time: {NOW_LOCAL.isoformat()}")
print(f"Workspace root: {MODEL_ROOT}")

print("\nSOURCE COPY")
print(f"Original source: {SOURCE_PATH}")
print(f"Copied source: {SOURCE_COPY_PATH}")
print(f"SHA-256: {source_sha256}")
print("Byte-identical copy: True")
print("Source modified: False")

print("\nSOURCE PROFILE")
print(f"Rows: {row_count:,}")
print(f"Columns: {column_count:,}")
print(f"Products: {product_count:,}")
print(f"Operating dates: {operating_date_count:,}")
print(f"Date range: {minimum_date} to {maximum_date}")
print(f"Normal demand: {normal_demand_total:,.6f}")
print(f"Bulk demand: {bulk_demand_total:,.6f}")
print(f"Total demand: {total_demand_total:,.6f}")
print(f"Bulk share: {bulk_share_percentage:.6f}%")
print(
    "Maximum reconciliation difference: "
    f"{maximum_reconciliation_difference:.12g}"
)

print("\nAGENT AND PROJECT DOCUMENTATION")
print(f"- AGENTS.md: {AGENTS_PATH}")
print(f"- README.md: {README_PATH}")
print(f"- Current handoff: {CURRENT_HANDOFF_PATH}")
print(f"- Workflow: {WORKFLOW_PATH}")
print(f"- Decisions: {DECISIONS_PATH}")
print(f"- Source dictionary: {DATA_DICTIONARY_PATH}")

print("\nCONTROL FILES")
print(f"- Validation: {VALIDATION_PATH}")
print(f"- Manifest: {MANIFEST_PATH}")
print(f"- Checkpoint: {CHECKPOINT_PATH}")
print(f"- Checkpoint SHA-256: {checkpoint_sha256}")
print(f"- Lock: {LOCK_PATH}")
print(f"- Lock SHA-256: {lock_sha256}")

print("\nSAFETY")
print("- Original dataset overwritten: False")
print("- Source-copy dataset transformed: False")
print("- Bulk demand removed: False")
print("- Low-demand products removed: False")
print("- Features generated: False")
print("- Models fitted: False")
print("- Existing Eden model locks modified: False")

print("\nNEXT STEP")
print(
    "ND02 — create the leakage-safe normal-demand panel, dynamic product "
    "eligibility register, fallback routes and demand-coverage audit."
)
print("=" * 100)

EDEN NORMAL-DEMAND MODEL V2 — ND01 COMPLETE
Status: ND01_SOURCE_WORKSPACE_CREATED_READY_FOR_ND02
Local time: 2026-08-07T21:44:23.660298+01:00
Workspace root: /Users/ryansmac/Desktop/Meng Project/eden_datasets/eden_normal_demand_model_v2

SOURCE COPY
Original source: /Users/ryansmac/Desktop/Meng Project/eden_datasets/UL_EDEN_canonical_product_daily_demand_forecasting_final.csv
Copied source: /Users/ryansmac/Desktop/Meng Project/eden_datasets/eden_normal_demand_model_v2/01_datasets/00_source_copy/UL_EDEN_canonical_product_daily_demand_forecasting_final.csv
SHA-256: f8538b31df4a2751a6b34cbc0a0f82c7d0e1d441480e1c6bdadd8899bca293e2
Byte-identical copy: True
Source modified: False

SOURCE PROFILE
Rows: 25,405
Columns: 38
Products: 227
Operating dates: 245
Date range: 2025-04-01 to 2026-03-30
Normal demand: 114,186.000000
Bulk demand: 1,972.000000
Total demand: 116,158.000000
Bulk share: 1.697688%
Maximum reconciliation difference: 0

AGENT AND PROJECT DOCUMENTATION
- AGENTS.md: /Users/ryansm

In [3]:
from __future__ import annotations

# =============================================================================
# EDEN NORMAL-DEMAND MODEL V2
# ND02 — BULK-FREE NORMAL-DEMAND PANEL AND PRIOR-ONLY PRODUCT ROUTING
#
# This step:
#   1. Verifies the immutable ND01 source copy and checkpoint.
#   2. Preserves every canonical product-date row.
#   3. Creates a modelling panel with NormalDemand as the only demand target.
#   4. Excludes BulkDemand, TotalDemand, and same-day demand-status fields.
#   5. Creates historical bulk audit outputs separately.
#   6. Calculates product history using dates before each row only.
#   7. Creates dynamic 80%, 90%, and 95% prior-demand scopes.
#   8. Routes established products to the main model and others to fallbacks.
#   9. Records daily, weekly, arbitrary-date, rolling-update, and bulk contracts.
#  10. Creates validation, leakage, coverage, hashes, checkpoint, and handoff.
#
# No model is fitted in ND02.
# No existing model, prediction, lock, checkpoint, or source file is modified.
# March 2026 is retained but marked ineligible for method selection.
#
# Run this as one complete Jupyter cell.
# =============================================================================

import hashlib
import json
import math
import os
import shutil
import stat
import uuid
from datetime import datetime, timezone
from pathlib import Path
from zoneinfo import ZoneInfo

import numpy as np
import pandas as pd


# =============================================================================
# CONFIGURATION
# =============================================================================

PROJECT_ROOT = Path("/Users/ryansmac/Desktop/Meng Project")
EDEN_ROOT = PROJECT_ROOT / "eden_datasets"
MODEL_ROOT = EDEN_ROOT / "eden_normal_demand_model_v2"

SOURCE_FILENAME = "UL_EDEN_canonical_product_daily_demand_forecasting_final.csv"

SOURCE_COPY_PATH = (
    MODEL_ROOT
    / "01_datasets"
    / "00_source_copy"
    / SOURCE_FILENAME
)

ND01_LOCK_PATH = (
    MODEL_ROOT
    / "08_checkpoints"
    / "ND01_workspace_and_source_copy_lock.json"
)

ND01_CHECKPOINT_PATH = (
    MODEL_ROOT
    / "08_checkpoints"
    / "ND01_checkpoint.json"
)

EXPECTED_SOURCE_SHA256 = (
    "f8538b31df4a2751a6b34cbc0a0f82c7"
    "d0e1d441480e1c6bdadd8899bca293e2"
)

EXPECTED_ND01_LOCK_SHA256 = (
    "07678543a1ea5246a3e4b0d2a10ede51"
    "6bcaf8d1696249c46e67109835d18ccf"
)

EXPECTED_ND01_CHECKPOINT_SHA256 = (
    "81ca2ff46f7a9fbc7f80d74207cd26fa"
    "ba3137f1cd784a2ca3ba5ffd35249157"
)

ND02_ROOT = (
    MODEL_ROOT
    / "01_datasets"
    / "01_intermediate"
    / "ND02_normal_demand_preparation"
)

DATA_DIR = ND02_ROOT / "01_data"
AUDIT_DIR = ND02_ROOT / "02_audits"
CONTRACT_DIR = ND02_ROOT / "03_contracts"
REPORT_DIR = ND02_ROOT / "04_reports"
CONTROL_DIR = ND02_ROOT / "05_control"

PANEL_PATH = DATA_DIR / "ND02_normal_demand_daily_panel.csv"
ELIGIBILITY_PATH = DATA_DIR / "ND02_product_date_eligibility_register.csv"
LATEST_STATUS_PATH = DATA_DIR / "ND02_latest_product_status.csv"
OPERATING_CALENDAR_PATH = DATA_DIR / "ND02_operating_calendar.csv"

BULK_AUDIT_PATH = DATA_DIR / "ND02_historical_bulk_order_audit.csv"
BULK_PRODUCT_SUMMARY_PATH = DATA_DIR / "ND02_historical_bulk_summary_by_product.csv"
BULK_DATE_SUMMARY_PATH = DATA_DIR / "ND02_historical_bulk_summary_by_date.csv"
BULK_INPUT_TEMPLATE_PATH = DATA_DIR / "ND02_confirmed_bulk_order_input_template.csv"

COVERAGE_BY_DATE_PATH = AUDIT_DIR / "ND02_scope_coverage_by_date.csv"
COVERAGE_SUMMARY_PATH = AUDIT_DIR / "ND02_scope_coverage_summary.csv"
ROUTE_SUMMARY_PATH = AUDIT_DIR / "ND02_forecast_route_summary.csv"
SPLIT_SUMMARY_PATH = AUDIT_DIR / "ND02_split_summary.csv"
LEAKAGE_AUDIT_PATH = AUDIT_DIR / "ND02_leakage_audit.csv"
VALIDATION_PATH = AUDIT_DIR / "ND02_validation_summary.csv"
SCHEMA_AUDIT_PATH = AUDIT_DIR / "ND02_output_schema_audit.csv"

DATASET_CONTRACT_JSON_PATH = (
    CONTRACT_DIR / "ND02_normal_demand_dataset_contract.json"
)
FORECAST_CONTRACT_JSON_PATH = (
    CONTRACT_DIR / "ND02_forecasting_architecture_contract.json"
)
DATASET_CONTRACT_MD_PATH = (
    CONTRACT_DIR / "ND02_normal_demand_dataset_contract.md"
)
FORECAST_CONTRACT_MD_PATH = (
    CONTRACT_DIR / "ND02_forecasting_architecture_contract.md"
)

REPORT_SUMMARY_PATH = REPORT_DIR / "ND02_preparation_summary.md"
README_PATH = ND02_ROOT / "README.md"

MANIFEST_PATH = CONTROL_DIR / "ND02_artifact_hash_manifest.csv"
CHECKPOINT_PATH = CONTROL_DIR / "ND02_checkpoint.json"
CHECKPOINT_SHA_PATH = CONTROL_DIR / "ND02_checkpoint.sha256"

TOP_LEVEL_CHECKPOINT_PATH = (
    MODEL_ROOT / "08_checkpoints" / "ND02_checkpoint.json"
)
TOP_LEVEL_CHECKPOINT_SHA_PATH = (
    MODEL_ROOT / "08_checkpoints" / "ND02_checkpoint.sha256"
)

AGENTS_PATH = MODEL_ROOT / "AGENTS.md"
MEMORY_ROOT = MODEL_ROOT / "00_project_memory"
CURRENT_HANDOFF_PATH = MEMORY_ROOT / "CURRENT_HANDOFF.md"
ND02_HANDOFF_PATH = MEMORY_ROOT / "ND02_HANDOFF.md"
WORKFLOW_PATH = MEMORY_ROOT / "WORKFLOW.md"
DECISIONS_PATH = MEMORY_ROOT / "DECISIONS.md"
METRICS_AND_RESULTS_PATH = MEMORY_ROOT / "METRICS_AND_RESULTS.md"

LOG_PATH = MODEL_ROOT / "09_logs" / "ND02_preparation_log.txt"

MINIMUM_HISTORY_OPERATING_DAYS = 20
PRIMARY_SCOPE_PERCENTAGE = 95
DEVELOPMENT_END_EXCLUSIVE = pd.Timestamp("2026-03-02")
OPENED_MARCH_START = pd.Timestamp("2026-03-02")
OPENED_MARCH_END = pd.Timestamp("2026-03-30")

EXPECTED_ROWS = 25_405
EXPECTED_PRODUCTS = 227
EXPECTED_OPERATING_DATES = 245
EXPECTED_NORMAL_DEMAND_TOTAL = 114_186.0
EXPECTED_BULK_DEMAND_TOTAL = 1_972.0
EXPECTED_TOTAL_DEMAND_TOTAL = 116_158.0

ALLOW_OVERWRITE = False
STEP_ID = "ND02"
STATUS = "ND02_NORMAL_DEMAND_PANEL_CREATED_READY_FOR_ND03"

NOW_UTC = datetime.now(timezone.utc)
NOW_LOCAL = NOW_UTC.astimezone(ZoneInfo("Europe/Dublin"))


# =============================================================================
# HELPERS
# =============================================================================

def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()

    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)

    return digest.hexdigest()


def write_csv(path: Path, frame: pd.DataFrame) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    frame.to_csv(path, index=False)


def write_json(path: Path, payload: dict) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(
        json.dumps(
            payload,
            indent=2,
            ensure_ascii=False,
            default=str,
        )
        + "\n",
        encoding="utf-8",
    )


def write_text(path: Path, text: str) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(text, encoding="utf-8")


def atomic_write_text(path: Path, text: str) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary_path = path.with_name(
        f".{path.name}.{uuid.uuid4().hex}.tmp"
    )
    temporary_path.write_text(text, encoding="utf-8")
    os.replace(temporary_path, path)


def make_read_only(path: Path) -> None:
    if path.is_file():
        path.chmod(
            stat.S_IRUSR
            | stat.S_IRGRP
            | stat.S_IROTH
        )


def percentage(
    numerator: float,
    denominator: float,
) -> float:
    if denominator == 0:
        return float("nan")

    return float(100.0 * numerator / denominator)


def validate_required_columns(
    frame: pd.DataFrame,
    required_columns: set[str],
    frame_name: str,
) -> None:
    missing = sorted(required_columns - set(frame.columns))

    if missing:
        raise AssertionError(
            f"{frame_name} is missing required columns:\n"
            + "\n".join(f"- {column}" for column in missing)
        )


def append_marked_section(
    path: Path,
    marker: str,
    section_text: str,
) -> None:
    existing = (
        path.read_text(encoding="utf-8")
        if path.is_file()
        else ""
    )

    if marker in existing:
        return

    separator = "\n" if existing.endswith("\n") else "\n\n"
    updated = (
        existing
        + separator
        + section_text.strip()
        + "\n"
    )

    atomic_write_text(path, updated)


def dataframe_schema(frame: pd.DataFrame) -> list[dict]:
    records = []

    for column in frame.columns:
        records.append(
            {
                "Column": str(column),
                "DataType": str(frame[column].dtype),
                "NonNullRows": int(frame[column].notna().sum()),
                "UniqueNonNullValues": int(
                    frame[column].nunique(dropna=True)
                ),
            }
        )

    return records


# =============================================================================
# PREFLIGHT
# =============================================================================

required_inputs = [
    SOURCE_COPY_PATH,
    ND01_LOCK_PATH,
    ND01_CHECKPOINT_PATH,
    AGENTS_PATH,
    CURRENT_HANDOFF_PATH,
    WORKFLOW_PATH,
    DECISIONS_PATH,
]

missing_inputs = [
    path for path in required_inputs if not path.is_file()
]

if missing_inputs:
    raise FileNotFoundError(
        "ND02 required input files are missing:\n"
        + "\n".join(f"- {path}" for path in missing_inputs)
    )

source_sha256_before = sha256_file(SOURCE_COPY_PATH)
nd01_lock_sha256_before = sha256_file(ND01_LOCK_PATH)
nd01_checkpoint_sha256_before = sha256_file(
    ND01_CHECKPOINT_PATH
)

if source_sha256_before != EXPECTED_SOURCE_SHA256:
    raise AssertionError(
        "The immutable ND01 source-copy hash does not match.\n"
        f"Expected: {EXPECTED_SOURCE_SHA256}\n"
        f"Actual:   {source_sha256_before}\n"
        f"Path:     {SOURCE_COPY_PATH}"
    )

if nd01_lock_sha256_before != EXPECTED_ND01_LOCK_SHA256:
    raise AssertionError(
        "The ND01 lock hash does not match.\n"
        f"Expected: {EXPECTED_ND01_LOCK_SHA256}\n"
        f"Actual:   {nd01_lock_sha256_before}\n"
        f"Path:     {ND01_LOCK_PATH}"
    )

if (
    nd01_checkpoint_sha256_before
    != EXPECTED_ND01_CHECKPOINT_SHA256
):
    raise AssertionError(
        "The ND01 checkpoint hash does not match.\n"
        f"Expected: {EXPECTED_ND01_CHECKPOINT_SHA256}\n"
        f"Actual:   {nd01_checkpoint_sha256_before}\n"
        f"Path:     {ND01_CHECKPOINT_PATH}"
    )

with ND01_LOCK_PATH.open("r", encoding="utf-8") as handle:
    nd01_lock_payload = json.load(handle)

with ND01_CHECKPOINT_PATH.open(
    "r",
    encoding="utf-8",
) as handle:
    nd01_checkpoint_payload = json.load(handle)

if not bool(nd01_lock_payload.get("ReadyForND02", False)):
    raise AssertionError(
        "The ND01 lock does not indicate readiness for ND02."
    )

if not bool(
    nd01_checkpoint_payload.get("ReadyForND02", False)
):
    raise AssertionError(
        "The ND01 checkpoint does not indicate readiness for ND02."
    )

if ND02_ROOT.exists():
    if not ALLOW_OVERWRITE:
        raise FileExistsError(
            "The ND02 output folder already exists. "
            "No files were changed:\n"
            f"{ND02_ROOT}\n\n"
            "Review the existing ND02 checkpoint instead of "
            "silently rebuilding it."
        )

    shutil.rmtree(ND02_ROOT)

for path in [
    TOP_LEVEL_CHECKPOINT_PATH,
    TOP_LEVEL_CHECKPOINT_SHA_PATH,
]:
    if path.exists():
        if not ALLOW_OVERWRITE:
            raise FileExistsError(
                "An ND02 top-level checkpoint already exists. "
                "No files were changed:\n"
                f"{path}"
            )
        path.unlink()

STAGING_ROOT = (
    ND02_ROOT.parent
    / f".ND02_staging_{uuid.uuid4().hex}"
)

STAGING_ROOT.mkdir(parents=True, exist_ok=False)


# =============================================================================
# LOAD AND VALIDATE SOURCE
# =============================================================================

try:
    source = pd.read_csv(
        SOURCE_COPY_PATH,
        low_memory=False,
    )

    required_source_columns = {
        "Date",
        "CanonicalProductID",
        "CanonicalProductName",
        "ProductFirstObservedDate",
        "ProductAgeOperatingDays",
        "OperatingDaySequence",
        "DayOfWeekNumber",
        "DayOfWeek",
        "NormalDemand",
        "BulkDemand",
        "TotalDemand",
        "IsObservedProductDate",
        "IsZeroDemandRow",
        "DemandRecordSource",
    }

    validate_required_columns(
        source,
        required_source_columns,
        "Canonical source dataset",
    )

    source["Date"] = pd.to_datetime(
        source["Date"],
        errors="raise",
    )

    source["ProductFirstObservedDate"] = pd.to_datetime(
        source["ProductFirstObservedDate"],
        errors="coerce",
    )

    for demand_column in [
        "NormalDemand",
        "BulkDemand",
        "TotalDemand",
    ]:
        source[demand_column] = pd.to_numeric(
            source[demand_column],
            errors="raise",
        ).astype(float)

    source_row_count = int(len(source))
    source_product_count = int(
        source["CanonicalProductID"].nunique()
    )
    source_operating_date_count = int(
        source["Date"].nunique()
    )

    source_normal_total = float(
        source["NormalDemand"].sum()
    )
    source_bulk_total = float(
        source["BulkDemand"].sum()
    )
    source_total_total = float(
        source["TotalDemand"].sum()
    )

    source_minimum_date = source["Date"].min()
    source_maximum_date = source["Date"].max()

    source_duplicate_keys = int(
        source.duplicated(
            ["Date", "CanonicalProductID"]
        ).sum()
    )

    source_non_finite_demand_values = int(
        (
            ~np.isfinite(
                source[
                    [
                        "NormalDemand",
                        "BulkDemand",
                        "TotalDemand",
                    ]
                ].to_numpy(dtype=float)
            )
        ).sum()
    )

    source_negative_demand_rows = int(
        (
            source[
                [
                    "NormalDemand",
                    "BulkDemand",
                    "TotalDemand",
                ]
            ]
            < 0
        )
        .any(axis=1)
        .sum()
    )

    reconciliation_difference = (
        source["NormalDemand"]
        + source["BulkDemand"]
        - source["TotalDemand"]
    )

    maximum_reconciliation_difference = float(
        reconciliation_difference.abs().max()
    )

    source_profile_assertions = {
        "Rows": (
            source_row_count,
            EXPECTED_ROWS,
        ),
        "Products": (
            source_product_count,
            EXPECTED_PRODUCTS,
        ),
        "OperatingDates": (
            source_operating_date_count,
            EXPECTED_OPERATING_DATES,
        ),
        "NormalDemandTotal": (
            source_normal_total,
            EXPECTED_NORMAL_DEMAND_TOTAL,
        ),
        "BulkDemandTotal": (
            source_bulk_total,
            EXPECTED_BULK_DEMAND_TOTAL,
        ),
        "TotalDemandTotal": (
            source_total_total,
            EXPECTED_TOTAL_DEMAND_TOTAL,
        ),
    }

    profile_failures = []

    for label, (
        actual_value,
        expected_value,
    ) in source_profile_assertions.items():
        if not math.isclose(
            float(actual_value),
            float(expected_value),
            rel_tol=0.0,
            abs_tol=1e-9,
        ):
            profile_failures.append(
                f"{label}: expected {expected_value}, "
                f"actual {actual_value}"
            )

    if profile_failures:
        raise AssertionError(
            "The canonical source profile differs from the "
            "accepted ND01 profile:\n"
            + "\n".join(
                f"- {failure}"
                for failure in profile_failures
            )
        )

    if source_duplicate_keys != 0:
        raise AssertionError(
            "Duplicate Date + CanonicalProductID rows were found."
        )

    if source_non_finite_demand_values != 0:
        raise AssertionError(
            "Non-finite demand values were found."
        )

    if source_negative_demand_rows != 0:
        raise AssertionError(
            "Negative demand values were found."
        )

    if maximum_reconciliation_difference > 1e-9:
        raise AssertionError(
            "NormalDemand + BulkDemand does not reconcile "
            "to TotalDemand."
        )

    # =========================================================================
    # SOURCE ORDER AND PRIOR-ONLY HISTORY
    # =========================================================================

    working = source.sort_values(
        [
            "CanonicalProductID",
            "Date",
            "OperatingDaySequence",
        ],
        kind="mergesort",
    ).reset_index(drop=True)

    product_group = working.groupby(
        "CanonicalProductID",
        sort=False,
    )

    working["PriorOperatingDayCount"] = (
        product_group.cumcount().astype(int)
    )

    cumulative_normal = product_group[
        "NormalDemand"
    ].cumsum()

    working["PriorCumulativeNormalDemand"] = (
        cumulative_normal - working["NormalDemand"]
    )

    current_positive = (
        working["NormalDemand"] > 0
    ).astype(int)

    cumulative_positive = current_positive.groupby(
        working["CanonicalProductID"]
    ).cumsum()

    working["PriorPositiveNormalDemandDays"] = (
        cumulative_positive - current_positive
    ).astype(int)

    working["PriorZeroNormalDemandDays"] = (
        working["PriorOperatingDayCount"]
        - working["PriorPositiveNormalDemandDays"]
    ).astype(int)

    working["PriorMeanNormalDemand"] = np.where(
        working["PriorOperatingDayCount"] > 0,
        (
            working["PriorCumulativeNormalDemand"]
            / working["PriorOperatingDayCount"]
        ),
        np.nan,
    )

    working["PriorPositiveDemandRate"] = np.where(
        working["PriorOperatingDayCount"] > 0,
        (
            working["PriorPositiveNormalDemandDays"]
            / working["PriorOperatingDayCount"]
        ),
        np.nan,
    )

    working["PriorZeroDemandRate"] = np.where(
        working["PriorOperatingDayCount"] > 0,
        (
            working["PriorZeroNormalDemandDays"]
            / working["PriorOperatingDayCount"]
        ),
        np.nan,
    )

    working[
        "PriorLastPositiveOperatingDaySequence"
    ] = np.nan

    for _, index_values in product_group.groups.items():
        index_array = np.asarray(
            list(index_values),
            dtype=int,
        )

        group_frame = working.loc[index_array]

        previous_positive_sequence = (
            group_frame["OperatingDaySequence"]
            .where(group_frame["NormalDemand"] > 0)
            .shift(1)
            .ffill()
        )

        working.loc[
            index_array,
            "PriorLastPositiveOperatingDaySequence",
        ] = previous_positive_sequence.to_numpy()

    working[
        "OperatingDaysSincePriorPositiveDemand"
    ] = (
        working["OperatingDaySequence"]
        - working[
            "PriorLastPositiveOperatingDaySequence"
        ]
    )

    working["HasSufficientHistory20"] = (
        working["PriorOperatingDayCount"]
        >= MINIMUM_HISTORY_OPERATING_DAYS
    )

    working["ColdStartFlag"] = (
        ~working["HasSufficientHistory20"]
    )

    working["ZeroPriorNormalDemandFlag"] = (
        working["PriorCumulativeNormalDemand"] <= 0
    )

    # =========================================================================
    # DYNAMIC PRIOR-DEMAND RANKING
    # =========================================================================

    ranking_columns = [
        "PriorDemandUniverseTotal",
        "PriorActiveProductCount",
        "PriorDemandRank",
        "PriorDemandSharePercentage",
        "PriorCumulativeDemandShareBeforePercentage",
        "PriorCumulativeDemandSharePercentage",
        "InPrior80DemandScope",
        "InPrior90DemandScope",
        "InPrior95DemandScope",
    ]

    for column in ranking_columns:
        if column.startswith("InPrior"):
            working[column] = False
        else:
            working[column] = np.nan

    for forecast_date, date_frame in working.groupby(
        "Date",
        sort=True,
    ):
        ranked = date_frame.sort_values(
            [
                "PriorCumulativeNormalDemand",
                "PriorPositiveNormalDemandDays",
                "CanonicalProductID",
            ],
            ascending=[False, False, True],
            kind="mergesort",
        )

        ranked_index = ranked.index
        prior_universe_total = float(
            ranked["PriorCumulativeNormalDemand"].sum()
        )
        active_product_count = int(len(ranked))

        working.loc[
            ranked_index,
            "PriorDemandUniverseTotal",
        ] = prior_universe_total

        working.loc[
            ranked_index,
            "PriorActiveProductCount",
        ] = active_product_count

        working.loc[
            ranked_index,
            "PriorDemandRank",
        ] = np.arange(
            1,
            active_product_count + 1,
            dtype=int,
        )

        if prior_universe_total > 0:
            demand_share_percentage = (
                100.0
                * ranked["PriorCumulativeNormalDemand"]
                / prior_universe_total
            )

            cumulative_share = (
                demand_share_percentage.cumsum()
            )

            cumulative_share_before = (
                cumulative_share - demand_share_percentage
            )

            working.loc[
                ranked_index,
                "PriorDemandSharePercentage",
            ] = demand_share_percentage.to_numpy()

            working.loc[
                ranked_index,
                "PriorCumulativeDemandShareBeforePercentage",
            ] = cumulative_share_before.to_numpy()

            working.loc[
                ranked_index,
                "PriorCumulativeDemandSharePercentage",
            ] = cumulative_share.to_numpy()

            positive_prior_demand = (
                ranked["PriorCumulativeNormalDemand"] > 0
            )

            for threshold in [80, 90, 95]:
                scope_flag = (
                    positive_prior_demand
                    & (
                        cumulative_share_before
                        < float(threshold)
                    )
                )

                working.loc[
                    ranked_index,
                    f"InPrior{threshold}DemandScope",
                ] = scope_flag.to_numpy()

    working["PriorDemandRank"] = (
        working["PriorDemandRank"]
        .astype("Int64")
    )

    working["PriorActiveProductCount"] = (
        working["PriorActiveProductCount"]
        .astype("Int64")
    )

    # =========================================================================
    # DYNAMIC SEGMENT AND ROUTING
    # =========================================================================

    no_prior_universe = (
        working["PriorDemandUniverseTotal"] <= 0
    )

    working["PriorDemandVolumeSegment"] = np.select(
        [
            no_prior_universe,
            working["ZeroPriorNormalDemandFlag"],
            working["InPrior80DemandScope"],
            (
                ~working["InPrior80DemandScope"]
                & working["InPrior95DemandScope"]
            ),
        ],
        [
            "NO_PRIOR_DEMAND_UNIVERSE",
            "ZERO_PRIOR_DEMAND",
            "HIGH_DEMAND",
            "MODERATE_DEMAND",
        ],
        default="LOW_DEMAND",
    )

    for threshold in [80, 90, 95]:
        working[
            f"EligibleForMainModel{threshold}"
        ] = (
            working["HasSufficientHistory20"]
            & working[
                f"InPrior{threshold}DemandScope"
            ]
            & ~working["ZeroPriorNormalDemandFlag"]
        )

    working["EligibleForMainModel"] = working[
        f"EligibleForMainModel{PRIMARY_SCOPE_PERCENTAGE}"
    ]

    working["ForecastRoute"] = np.select(
        [
            working["ZeroPriorNormalDemandFlag"],
            working["ColdStartFlag"],
            working["EligibleForMainModel"],
        ],
        [
            "ZERO_HISTORY_FALLBACK",
            "COLD_START_FALLBACK",
            "MAIN_MODEL",
        ],
        default="LOW_DEMAND_FALLBACK",
    )

    working["ForecastRouteReason"] = np.select(
        [
            working["ZeroPriorNormalDemandFlag"],
            working["ColdStartFlag"],
            working["EligibleForMainModel"],
        ],
        [
            "No positive normal-demand history before forecast date",
            (
                "Fewer than 20 prior operating-day observations "
                "before forecast date"
            ),
            (
                "At least 20 prior operating days and inside "
                "the dynamic prior-demand 95 percent scope"
            ),
        ],
        default=(
            "Sufficient history but outside the dynamic "
            "prior-demand 95 percent scope"
        ),
    )

    working["FallbackMethodStatus"] = np.where(
        working["ForecastRoute"] == "MAIN_MODEL",
        "NOT_APPLICABLE",
        "TO_BE_SELECTED_IN_ND04",
    )

    # =========================================================================
    # SPLIT AND FUTURE-USE FLAGS
    # =========================================================================

    working["IsStandardMondayToFriday"] = (
        working["DayOfWeekNumber"].between(0, 4)
    )

    working["IsExceptionalWeekendOperatingDate"] = (
        working["DayOfWeekNumber"] > 4
    )

    working["IsPreMarchDevelopmentPeriod"] = (
        working["Date"] < DEVELOPMENT_END_EXCLUSIVE
    )

    working["IsOpenedMarchDiagnosticPeriod"] = (
        working["Date"].between(
            OPENED_MARCH_START,
            OPENED_MARCH_END,
            inclusive="both",
        )
    )

    working["EligibleForMethodSelection"] = (
        working["IsPreMarchDevelopmentPeriod"]
    )

    working[
        "EligibleForNewUnbiasedFinalEvaluation"
    ] = False

    # =========================================================================
    # BULK-FREE MODELLING PANEL
    # =========================================================================

    forbidden_model_panel_columns = {
        "BulkDemand",
        "TotalDemand",
        "IsObservedProductDate",
        "IsZeroDemandRow",
        "DemandRecordSource",
    }

    panel_columns = [
        column
        for column in working.columns
        if column not in forbidden_model_panel_columns
    ]

    panel = working[panel_columns].copy()

    panel = panel.sort_values(
        ["Date", "CanonicalProductID"],
        kind="mergesort",
    ).reset_index(drop=True)

    panel_forbidden_columns_present = sorted(
        forbidden_model_panel_columns
        & set(panel.columns)
    )

    if panel_forbidden_columns_present:
        raise AssertionError(
            "Forbidden bulk, total-demand, or same-day status "
            "columns remain in the model panel:\n"
            + "\n".join(
                f"- {column}"
                for column in panel_forbidden_columns_present
            )
        )

    # =========================================================================
    # ELIGIBILITY REGISTER
    # =========================================================================

    eligibility_columns = [
        "Date",
        "CanonicalProductID",
        "CanonicalProductName",
        "OperatingDaySequence",
        "PriorOperatingDayCount",
        "PriorCumulativeNormalDemand",
        "PriorPositiveNormalDemandDays",
        "PriorZeroNormalDemandDays",
        "PriorMeanNormalDemand",
        "PriorPositiveDemandRate",
        "PriorZeroDemandRate",
        "PriorLastPositiveOperatingDaySequence",
        "OperatingDaysSincePriorPositiveDemand",
        "PriorDemandUniverseTotal",
        "PriorActiveProductCount",
        "PriorDemandRank",
        "PriorDemandSharePercentage",
        "PriorCumulativeDemandShareBeforePercentage",
        "PriorCumulativeDemandSharePercentage",
        "InPrior80DemandScope",
        "InPrior90DemandScope",
        "InPrior95DemandScope",
        "PriorDemandVolumeSegment",
        "HasSufficientHistory20",
        "ColdStartFlag",
        "ZeroPriorNormalDemandFlag",
        "EligibleForMainModel80",
        "EligibleForMainModel90",
        "EligibleForMainModel95",
        "EligibleForMainModel",
        "ForecastRoute",
        "ForecastRouteReason",
        "FallbackMethodStatus",
        "IsPreMarchDevelopmentPeriod",
        "IsOpenedMarchDiagnosticPeriod",
        "EligibleForMethodSelection",
    ]

    eligibility_register = panel[
        eligibility_columns
    ].copy()

    # =========================================================================
    # LATEST PRODUCT STATUS
    # =========================================================================

    product_last_date = panel.groupby(
        "CanonicalProductID"
    )["Date"].transform("max")

    latest_product_status = panel.loc[
        panel["Date"] == product_last_date,
        [
            "Date",
            "CanonicalProductID",
            "CanonicalProductName",
            "TierProductFamily",
            "BeverageSeries",
            "BeverageType",
            "NominalPriceTier",
            "PriorOperatingDayCount",
            "PriorCumulativeNormalDemand",
            "PriorPositiveNormalDemandDays",
            "PriorPositiveDemandRate",
            "PriorDemandRank",
            "PriorDemandVolumeSegment",
            "HasSufficientHistory20",
            "EligibleForMainModel",
            "ForecastRoute",
            "FallbackMethodStatus",
        ],
    ].copy()

    latest_product_status = (
        latest_product_status.sort_values(
            [
                "EligibleForMainModel",
                "PriorCumulativeNormalDemand",
                "CanonicalProductID",
            ],
            ascending=[False, False, True],
            kind="mergesort",
        )
        .reset_index(drop=True)
    )

    latest_product_status[
        "IsPresentOnFinalObservedDate"
    ] = (
        latest_product_status["Date"]
        == source_maximum_date
    )

    # =========================================================================
    # OPERATING CALENDAR
    # =========================================================================

    operating_calendar_columns = [
        "Date",
        "OperatingDaySequence",
        "Year",
        "Month",
        "MonthName",
        "Quarter",
        "DayOfWeekNumber",
        "DayOfWeek",
        "ISOYear",
        "ISOWeek",
        "DayOfYear",
        "IsWeekend",
        "DaysSincePreviousOperatingDate",
        "IsConsecutiveCalendarDay",
    ]

    operating_calendar = (
        source[operating_calendar_columns]
        .drop_duplicates()
        .sort_values("Date")
        .reset_index(drop=True)
    )

    operating_calendar[
        "IsStandardMondayToFriday"
    ] = operating_calendar[
        "DayOfWeekNumber"
    ].between(0, 4)

    operating_calendar[
        "IsExceptionalWeekendOperatingDate"
    ] = operating_calendar[
        "DayOfWeekNumber"
    ] > 4

    operating_calendar[
        "FutureWeeklyDefaultPolicy"
    ] = np.where(
        operating_calendar[
            "IsStandardMondayToFriday"
        ],
        "INCLUDED_IN_STANDARD_MONDAY_TO_FRIDAY_WEEK",
        "HISTORICAL_EXCEPTION_REQUIRES_EXPLICIT_OVERRIDE",
    )

    # =========================================================================
    # SEPARATE HISTORICAL BULK AUDIT
    # =========================================================================

    bulk_audit_columns = [
        "Date",
        "CanonicalProductID",
        "CanonicalProductName",
        "TierProductFamily",
        "BeverageSeries",
        "BeverageType",
        "NominalPriceTier",
        "NormalDemand",
        "BulkDemand",
        "TotalDemand",
        "OperatingDaySequence",
        "DayOfWeek",
        "ISOYear",
        "ISOWeek",
    ]

    bulk_audit = source.loc[
        source["BulkDemand"] > 0,
        bulk_audit_columns,
    ].copy()

    bulk_audit = bulk_audit.rename(
        columns={
            "NormalDemand":
                "NormalDemandOnBulkOrderDate",
            "BulkDemand":
                "HistoricalConfirmedBulkDemand",
            "TotalDemand":
                "HistoricalCombinedDemand",
        }
    )

    bulk_audit = bulk_audit.sort_values(
        ["Date", "CanonicalProductID"]
    ).reset_index(drop=True)

    bulk_product_summary = (
        bulk_audit.groupby(
            [
                "CanonicalProductID",
                "CanonicalProductName",
            ],
            as_index=False,
        )
        .agg(
            BulkOrderDates=(
                "Date",
                "nunique",
            ),
            HistoricalConfirmedBulkDemand=(
                "HistoricalConfirmedBulkDemand",
                "sum",
            ),
            NormalDemandOnBulkOrderDates=(
                "NormalDemandOnBulkOrderDate",
                "sum",
            ),
            FirstBulkOrderDate=(
                "Date",
                "min",
            ),
            LastBulkOrderDate=(
                "Date",
                "max",
            ),
        )
        .sort_values(
            "HistoricalConfirmedBulkDemand",
            ascending=False,
        )
        .reset_index(drop=True)
    )

    bulk_date_summary = (
        bulk_audit.groupby(
            "Date",
            as_index=False,
        )
        .agg(
            ProductsWithBulkOrders=(
                "CanonicalProductID",
                "nunique",
            ),
            HistoricalConfirmedBulkDemand=(
                "HistoricalConfirmedBulkDemand",
                "sum",
            ),
            NormalDemandOnBulkOrderDateRows=(
                "NormalDemandOnBulkOrderDate",
                "sum",
            ),
        )
        .sort_values("Date")
        .reset_index(drop=True)
    )

    bulk_input_template = pd.DataFrame(
        columns=[
            "Date",
            "CanonicalProductID",
            "CanonicalProductName",
            "ConfirmedBulkQuantity",
            "OrderReference",
            "Notes",
        ]
    )

    # =========================================================================
    # COVERAGE BY DATE
    # =========================================================================

    coverage_by_date_records = []

    for forecast_date, date_frame in panel.groupby(
        "Date",
        sort=True,
    ):
        actual_total = float(
            date_frame["NormalDemand"].sum()
        )

        established_actual_total = float(
            date_frame.loc[
                date_frame["HasSufficientHistory20"],
                "NormalDemand",
            ].sum()
        )

        record = {
            "Date": forecast_date,
            "ProductsInPanel": int(
                date_frame[
                    "CanonicalProductID"
                ].nunique()
            ),
            "ProductsWithSufficientHistory20": int(
                date_frame.loc[
                    date_frame[
                        "HasSufficientHistory20"
                    ],
                    "CanonicalProductID",
                ].nunique()
            ),
            "ActualNormalDemandAllProducts":
                actual_total,
            "ActualNormalDemandEstablishedProducts":
                established_actual_total,
            "EstablishedDemandSharePercentage":
                percentage(
                    established_actual_total,
                    actual_total,
                ),
            "IsPreMarchDevelopmentPeriod": bool(
                (
                    date_frame[
                        "IsPreMarchDevelopmentPeriod"
                    ]
                ).all()
            ),
            "IsOpenedMarchDiagnosticPeriod": bool(
                (
                    date_frame[
                        "IsOpenedMarchDiagnosticPeriod"
                    ]
                ).all()
            ),
        }

        for threshold in [80, 90, 95]:
            eligible_flag = (
                f"EligibleForMainModel{threshold}"
            )

            eligible_frame = date_frame.loc[
                date_frame[eligible_flag]
            ]

            eligible_actual = float(
                eligible_frame["NormalDemand"].sum()
            )

            record[
                f"EligibleProducts{threshold}"
            ] = int(
                eligible_frame[
                    "CanonicalProductID"
                ].nunique()
            )

            record[
                f"ActualNormalDemandEligible{threshold}"
            ] = eligible_actual

            record[
                f"RetrospectiveActualCoverage{threshold}Percentage"
            ] = percentage(
                eligible_actual,
                actual_total,
            )

            record[
                f"CoverageAmongEstablishedDemand{threshold}Percentage"
            ] = percentage(
                eligible_actual,
                established_actual_total,
            )

        coverage_by_date_records.append(record)

    coverage_by_date = pd.DataFrame(
        coverage_by_date_records
    )

    # =========================================================================
    # COVERAGE SUMMARY
    # =========================================================================

    period_definitions = {
        "ALL_SOURCE_DATES": (
            pd.Series(True, index=panel.index)
        ),
        "PRE_MARCH_DEVELOPMENT_PERIOD": (
            panel["IsPreMarchDevelopmentPeriod"]
        ),
        "OPENED_MARCH_DIAGNOSTIC_PERIOD": (
            panel["IsOpenedMarchDiagnosticPeriod"]
        ),
        "STANDARD_MONDAY_TO_FRIDAY_ROWS": (
            panel["IsStandardMondayToFriday"]
        ),
    }

    coverage_summary_records = []

    for period_name, period_mask in period_definitions.items():
        period_frame = panel.loc[period_mask].copy()

        period_actual_total = float(
            period_frame["NormalDemand"].sum()
        )

        period_established_total = float(
            period_frame.loc[
                period_frame[
                    "HasSufficientHistory20"
                ],
                "NormalDemand",
            ].sum()
        )

        for threshold in [80, 90, 95]:
            eligibility_column = (
                f"EligibleForMainModel{threshold}"
            )

            eligible_frame = period_frame.loc[
                period_frame[eligibility_column]
            ]

            eligible_actual_total = float(
                eligible_frame["NormalDemand"].sum()
            )

            coverage_summary_records.append(
                {
                    "Period": period_name,
                    "ScopePercentage": threshold,
                    "RowsInPeriod": int(
                        len(period_frame)
                    ),
                    "DatesInPeriod": int(
                        period_frame["Date"].nunique()
                    ),
                    "ProductsInPeriod": int(
                        period_frame[
                            "CanonicalProductID"
                        ].nunique()
                    ),
                    "EligibleRows": int(
                        len(eligible_frame)
                    ),
                    "EligibleProductsAppearing": int(
                        eligible_frame[
                            "CanonicalProductID"
                        ].nunique()
                    ),
                    "ActualNormalDemandAllProducts":
                        period_actual_total,
                    "ActualNormalDemandEstablishedProducts":
                        period_established_total,
                    "ActualNormalDemandEligible":
                        eligible_actual_total,
                    "RetrospectiveActualCoveragePercentage":
                        percentage(
                            eligible_actual_total,
                            period_actual_total,
                        ),
                    "CoverageAmongEstablishedDemandPercentage":
                        percentage(
                            eligible_actual_total,
                            period_established_total,
                        ),
                    "SelectionUse":
                        (
                            "METHOD_SELECTION_ALLOWED"
                            if period_name
                            == "PRE_MARCH_DEVELOPMENT_PERIOD"
                            else "DIAGNOSTIC_OR_DESCRIPTIVE_ONLY"
                        ),
                }
            )

    coverage_summary = pd.DataFrame(
        coverage_summary_records
    )

    # =========================================================================
    # ROUTE SUMMARY
    # =========================================================================

    route_summary = (
        panel.groupby(
            "ForecastRoute",
            as_index=False,
        )
        .agg(
            ProductDateRows=(
                "CanonicalProductID",
                "size",
            ),
            ProductsAppearing=(
                "CanonicalProductID",
                "nunique",
            ),
            DatesAppearing=(
                "Date",
                "nunique",
            ),
            ActualNormalDemand=(
                "NormalDemand",
                "sum",
            ),
            PositiveNormalDemandRows=(
                "NormalDemand",
                lambda values:
                    int((values > 0).sum()),
            ),
        )
    )

    route_summary[
        "ShareOfAllNormalDemandPercentage"
    ] = (
        100.0
        * route_summary["ActualNormalDemand"]
        / source_normal_total
    )

    route_order = {
        "MAIN_MODEL": 1,
        "LOW_DEMAND_FALLBACK": 2,
        "COLD_START_FALLBACK": 3,
        "ZERO_HISTORY_FALLBACK": 4,
    }

    route_summary["_SortOrder"] = (
        route_summary["ForecastRoute"].map(
            route_order
        )
    )

    route_summary = (
        route_summary.sort_values("_SortOrder")
        .drop(columns=["_SortOrder"])
        .reset_index(drop=True)
    )

    # =========================================================================
    # SPLIT SUMMARY
    # =========================================================================

    split_summary_records = []

    split_definitions = {
        "PRE_MARCH_DEVELOPMENT_PERIOD": (
            panel["IsPreMarchDevelopmentPeriod"]
        ),
        "OPENED_MARCH_DIAGNOSTIC_PERIOD": (
            panel["IsOpenedMarchDiagnosticPeriod"]
        ),
    }

    for split_name, split_mask in split_definitions.items():
        split_frame = panel.loc[split_mask]

        split_summary_records.append(
            {
                "Split": split_name,
                "MinimumDate":
                    split_frame["Date"].min(),
                "MaximumDate":
                    split_frame["Date"].max(),
                "Rows": int(len(split_frame)),
                "OperatingDates": int(
                    split_frame["Date"].nunique()
                ),
                "Products": int(
                    split_frame[
                        "CanonicalProductID"
                    ].nunique()
                ),
                "NormalDemand": float(
                    split_frame["NormalDemand"].sum()
                ),
                "EligibleForMethodSelection":
                    (
                        split_name
                        == "PRE_MARCH_DEVELOPMENT_PERIOD"
                    ),
                "EligibleForNewUnbiasedFinalEvaluation":
                    False,
                "Reason":
                    (
                        "Available for chronological model development"
                        if split_name
                        == "PRE_MARCH_DEVELOPMENT_PERIOD"
                        else (
                            "Targets have already been opened; "
                            "diagnostic use only"
                        )
                    ),
            }
        )

    split_summary = pd.DataFrame(
        split_summary_records
    )

    # =========================================================================
    # LEAKAGE AUDIT
    # =========================================================================

    first_rows = (
        working.groupby(
            "CanonicalProductID",
            sort=False,
        )
        .head(1)
    )

    expected_prior_cumulative = (
        working.groupby(
            "CanonicalProductID",
            sort=False,
        )["NormalDemand"].cumsum()
        - working["NormalDemand"]
    )

    expected_prior_positive = (
        (working["NormalDemand"] > 0)
        .astype(int)
        .groupby(
            working["CanonicalProductID"]
        )
        .cumsum()
        - (working["NormalDemand"] > 0).astype(int)
    )

    prior_cumulative_max_difference = float(
        (
            working["PriorCumulativeNormalDemand"]
            - expected_prior_cumulative
        )
        .abs()
        .max()
    )

    prior_positive_max_difference = float(
        (
            working["PriorPositiveNormalDemandDays"]
            - expected_prior_positive
        )
        .abs()
        .max()
    )

    first_row_prior_count_max = int(
        first_rows["PriorOperatingDayCount"].max()
    )

    first_row_prior_demand_max = float(
        first_rows[
            "PriorCumulativeNormalDemand"
        ].max()
    )

    first_row_prior_positive_max = int(
        first_rows[
            "PriorPositiveNormalDemandDays"
        ].max()
    )

    eligibility_invalid_history_rows = int(
        (
            panel["EligibleForMainModel"]
            & (
                panel["PriorOperatingDayCount"]
                < MINIMUM_HISTORY_OPERATING_DAYS
            )
        ).sum()
    )

    eligibility_outside_95_rows = int(
        (
            panel["EligibleForMainModel"]
            & ~panel["InPrior95DemandScope"]
        ).sum()
    )

    eligibility_zero_history_rows = int(
        (
            panel["EligibleForMainModel"]
            & panel["ZeroPriorNormalDemandFlag"]
        ).sum()
    )

    leakage_audit = pd.DataFrame(
        [
            {
                "Check":
                    "BulkDemand absent from model panel",
                "Expected": True,
                "Actual":
                    "BulkDemand" not in panel.columns,
                "Passed":
                    "BulkDemand" not in panel.columns,
                "Interpretation":
                    "Bulk is stored only in the separate audit route",
            },
            {
                "Check":
                    "TotalDemand absent from model panel",
                "Expected": True,
                "Actual":
                    "TotalDemand" not in panel.columns,
                "Passed":
                    "TotalDemand" not in panel.columns,
                "Interpretation":
                    "Historical model features will be rebuilt from NormalDemand",
            },
            {
                "Check":
                    "Same-day IsObservedProductDate absent",
                "Expected": True,
                "Actual":
                    "IsObservedProductDate"
                    not in panel.columns,
                "Passed":
                    "IsObservedProductDate"
                    not in panel.columns,
                "Interpretation":
                    "Same-day observed-status leakage removed",
            },
            {
                "Check":
                    "Same-day IsZeroDemandRow absent",
                "Expected": True,
                "Actual":
                    "IsZeroDemandRow"
                    not in panel.columns,
                "Passed":
                    "IsZeroDemandRow"
                    not in panel.columns,
                "Interpretation":
                    "Same-day target-derived zero flag removed",
            },
            {
                "Check":
                    "DemandRecordSource absent",
                "Expected": True,
                "Actual":
                    "DemandRecordSource"
                    not in panel.columns,
                "Passed":
                    "DemandRecordSource"
                    not in panel.columns,
                "Interpretation":
                    "Same-day observation-source indicator removed",
            },
            {
                "Check":
                    "Prior cumulative demand uses earlier rows only",
                "Expected": 0.0,
                "Actual":
                    prior_cumulative_max_difference,
                "Passed":
                    prior_cumulative_max_difference
                    <= 1e-9,
                "Interpretation":
                    "Cumulative demand is shifted by excluding the current target",
            },
            {
                "Check":
                    "Prior positive-day count uses earlier rows only",
                "Expected": 0.0,
                "Actual":
                    prior_positive_max_difference,
                "Passed":
                    prior_positive_max_difference
                    <= 1e-9,
                "Interpretation":
                    "Positive-day count excludes the current target",
            },
            {
                "Check":
                    "First row prior operating-day count",
                "Expected": 0,
                "Actual":
                    first_row_prior_count_max,
                "Passed":
                    first_row_prior_count_max == 0,
                "Interpretation":
                    "No product receives invented earlier history",
            },
            {
                "Check":
                    "First row prior cumulative demand",
                "Expected": 0.0,
                "Actual":
                    first_row_prior_demand_max,
                "Passed":
                    first_row_prior_demand_max
                    <= 1e-9,
                "Interpretation":
                    "Current-day demand is not included in first-row history",
            },
            {
                "Check":
                    "First row prior positive-day count",
                "Expected": 0,
                "Actual":
                    first_row_prior_positive_max,
                "Passed":
                    first_row_prior_positive_max
                    == 0,
                "Interpretation":
                    "Current-day positivity is not included",
            },
            {
                "Check":
                    "Main-model eligibility requires 20 prior days",
                "Expected": 0,
                "Actual":
                    eligibility_invalid_history_rows,
                "Passed":
                    eligibility_invalid_history_rows
                    == 0,
                "Interpretation":
                    "Cold-start products cannot enter the main model",
            },
            {
                "Check":
                    "Main-model eligibility remains inside prior 95 percent scope",
                "Expected": 0,
                "Actual":
                    eligibility_outside_95_rows,
                "Passed":
                    eligibility_outside_95_rows
                    == 0,
                "Interpretation":
                    "Primary scope is based on prior demand only",
            },
            {
                "Check":
                    "Zero-history products excluded from main model",
                "Expected": 0,
                "Actual":
                    eligibility_zero_history_rows,
                "Passed":
                    eligibility_zero_history_rows
                    == 0,
                "Interpretation":
                    "Products without prior normal demand use fallback routing",
            },
            {
                "Check":
                    "March 2026 rows excluded from method selection",
                "Expected": 0,
                "Actual": int(
                    (
                        panel[
                            "IsOpenedMarchDiagnosticPeriod"
                        ]
                        & panel[
                            "EligibleForMethodSelection"
                        ]
                    ).sum()
                ),
                "Passed": int(
                    (
                        panel[
                            "IsOpenedMarchDiagnosticPeriod"
                        ]
                        & panel[
                            "EligibleForMethodSelection"
                        ]
                    ).sum()
                )
                == 0,
                "Interpretation":
                    "Opened March targets are diagnostic only",
            },
        ]
    )

    if not leakage_audit["Passed"].all():
        raise AssertionError(
            "ND02 leakage audit failed:\n"
            + leakage_audit.loc[
                ~leakage_audit["Passed"]
            ].to_string(index=False)
        )

    # =========================================================================
    # VALIDATION
    # =========================================================================

    source_key_set = set(
        zip(
            source["Date"].astype("int64"),
            source["CanonicalProductID"].astype(str),
        )
    )

    panel_key_set = set(
        zip(
            panel["Date"].astype("int64"),
            panel["CanonicalProductID"].astype(str),
        )
    )

    route_missing_rows = int(
        panel["ForecastRoute"].isna().sum()
    )

    panel_duplicate_keys = int(
        panel.duplicated(
            ["Date", "CanonicalProductID"]
        ).sum()
    )

    panel_non_finite_target_values = int(
        (
            ~np.isfinite(
                panel["NormalDemand"].to_numpy(
                    dtype=float
                )
            )
        ).sum()
    )

    panel_negative_target_rows = int(
        (panel["NormalDemand"] < 0).sum()
    )

    panel_normal_total = float(
        panel["NormalDemand"].sum()
    )

    bulk_audit_total = float(
        bulk_audit[
            "HistoricalConfirmedBulkDemand"
        ].sum()
    )

    bulk_audit_rows = int(len(bulk_audit))
    bulk_audit_dates = int(
        bulk_audit["Date"].nunique()
    )
    bulk_audit_products = int(
        bulk_audit[
            "CanonicalProductID"
        ].nunique()
    )

    validation = pd.DataFrame(
        [
            {
                "Check": "ND01 source hash verified",
                "Expected": EXPECTED_SOURCE_SHA256,
                "Actual": source_sha256_before,
                "Passed":
                    source_sha256_before
                    == EXPECTED_SOURCE_SHA256,
            },
            {
                "Check": "ND01 lock hash verified",
                "Expected": EXPECTED_ND01_LOCK_SHA256,
                "Actual": nd01_lock_sha256_before,
                "Passed":
                    nd01_lock_sha256_before
                    == EXPECTED_ND01_LOCK_SHA256,
            },
            {
                "Check": "ND01 checkpoint hash verified",
                "Expected":
                    EXPECTED_ND01_CHECKPOINT_SHA256,
                "Actual":
                    nd01_checkpoint_sha256_before,
                "Passed":
                    nd01_checkpoint_sha256_before
                    == EXPECTED_ND01_CHECKPOINT_SHA256,
            },
            {
                "Check": "Panel row count preserved",
                "Expected": source_row_count,
                "Actual": int(len(panel)),
                "Passed":
                    int(len(panel))
                    == source_row_count,
            },
            {
                "Check": "Panel product count preserved",
                "Expected": source_product_count,
                "Actual": int(
                    panel[
                        "CanonicalProductID"
                    ].nunique()
                ),
                "Passed": int(
                    panel[
                        "CanonicalProductID"
                    ].nunique()
                )
                == source_product_count,
            },
            {
                "Check":
                    "Panel operating-date count preserved",
                "Expected":
                    source_operating_date_count,
                "Actual": int(
                    panel["Date"].nunique()
                ),
                "Passed": int(
                    panel["Date"].nunique()
                )
                == source_operating_date_count,
            },
            {
                "Check":
                    "Source and panel key sets match",
                "Expected": True,
                "Actual":
                    source_key_set
                    == panel_key_set,
                "Passed":
                    source_key_set
                    == panel_key_set,
            },
            {
                "Check":
                    "Panel duplicate product-date keys",
                "Expected": 0,
                "Actual": panel_duplicate_keys,
                "Passed":
                    panel_duplicate_keys == 0,
            },
            {
                "Check":
                    "Panel normal-demand total preserved",
                "Expected": source_normal_total,
                "Actual": panel_normal_total,
                "Passed": math.isclose(
                    panel_normal_total,
                    source_normal_total,
                    rel_tol=0.0,
                    abs_tol=1e-9,
                ),
            },
            {
                "Check":
                    "Historical bulk audit total preserved",
                "Expected": source_bulk_total,
                "Actual": bulk_audit_total,
                "Passed": math.isclose(
                    bulk_audit_total,
                    source_bulk_total,
                    rel_tol=0.0,
                    abs_tol=1e-9,
                ),
            },
            {
                "Check":
                    "Demand reconciliation maximum difference",
                "Expected": "<= 1e-9",
                "Actual":
                    maximum_reconciliation_difference,
                "Passed":
                    maximum_reconciliation_difference
                    <= 1e-9,
            },
            {
                "Check":
                    "Forbidden model-panel columns",
                "Expected": [],
                "Actual":
                    panel_forbidden_columns_present,
                "Passed":
                    len(
                        panel_forbidden_columns_present
                    )
                    == 0,
            },
            {
                "Check":
                    "Missing forecast routes",
                "Expected": 0,
                "Actual": route_missing_rows,
                "Passed": route_missing_rows == 0,
            },
            {
                "Check":
                    "Non-finite normal-demand target values",
                "Expected": 0,
                "Actual":
                    panel_non_finite_target_values,
                "Passed":
                    panel_non_finite_target_values
                    == 0,
            },
            {
                "Check":
                    "Negative normal-demand target rows",
                "Expected": 0,
                "Actual":
                    panel_negative_target_rows,
                "Passed":
                    panel_negative_target_rows
                    == 0,
            },
            {
                "Check": "Models fitted",
                "Expected": False,
                "Actual": False,
                "Passed": True,
            },
            {
                "Check": "Source copy modified",
                "Expected": False,
                "Actual": False,
                "Passed": True,
            },
            {
                "Check":
                    "Existing model locks modified",
                "Expected": False,
                "Actual": False,
                "Passed": True,
            },
            {
                "Check":
                    "ND02 step lock created",
                "Expected": False,
                "Actual": False,
                "Passed": True,
            },
        ]
    )

    if not validation["Passed"].all():
        raise AssertionError(
            "ND02 validation failed:\n"
            + validation.loc[
                ~validation["Passed"]
            ].to_string(index=False)
        )

    # =========================================================================
    # CONTRACTS
    # =========================================================================

    target_columns = ["NormalDemand"]

    identifier_columns = [
        "Date",
        "CanonicalProductID",
        "CanonicalProductName",
    ]

    administrative_non_predictor_columns = [
        "DailyPanelVersion",
        "ProductMetadataVersion",
        "IsPreMarchDevelopmentPeriod",
        "IsOpenedMarchDiagnosticPeriod",
        "EligibleForMethodSelection",
        "EligibleForNewUnbiasedFinalEvaluation",
    ]

    routing_columns = [
        "PriorDemandVolumeSegment",
        "HasSufficientHistory20",
        "ColdStartFlag",
        "ZeroPriorNormalDemandFlag",
        "EligibleForMainModel80",
        "EligibleForMainModel90",
        "EligibleForMainModel95",
        "EligibleForMainModel",
        "ForecastRoute",
        "ForecastRouteReason",
        "FallbackMethodStatus",
    ]

    prior_only_columns = [
        "PriorOperatingDayCount",
        "PriorCumulativeNormalDemand",
        "PriorPositiveNormalDemandDays",
        "PriorZeroNormalDemandDays",
        "PriorMeanNormalDemand",
        "PriorPositiveDemandRate",
        "PriorZeroDemandRate",
        "PriorLastPositiveOperatingDaySequence",
        "OperatingDaysSincePriorPositiveDemand",
        "PriorDemandUniverseTotal",
        "PriorActiveProductCount",
        "PriorDemandRank",
        "PriorDemandSharePercentage",
        "PriorCumulativeDemandShareBeforePercentage",
        "PriorCumulativeDemandSharePercentage",
        "InPrior80DemandScope",
        "InPrior90DemandScope",
        "InPrior95DemandScope",
    ]

    dataset_contract = {
        "StepID": STEP_ID,
        "Status": STATUS,
        "CreatedUTC": NOW_UTC.isoformat(),
        "Source": {
            "Path": str(SOURCE_COPY_PATH),
            "SHA256": source_sha256_before,
            "Rows": source_row_count,
            "Products": source_product_count,
            "OperatingDates":
                source_operating_date_count,
            "MinimumDate":
                source_minimum_date.date().isoformat(),
            "MaximumDate":
                source_maximum_date.date().isoformat(),
        },
        "OutputPanel": {
            "Path": str(PANEL_PATH),
            "Rows": int(len(panel)),
            "Columns": int(len(panel.columns)),
            "TargetColumns": target_columns,
            "IdentifierColumns": identifier_columns,
            "AdministrativeNonPredictorColumns":
                administrative_non_predictor_columns,
            "RoutingColumns": routing_columns,
            "PriorOnlyColumns": prior_only_columns,
            "ForbiddenColumns": sorted(
                forbidden_model_panel_columns
            ),
            "BulkFree": True,
            "TotalDemandFree": True,
            "AllSourceProductDateRowsPreserved": True,
        },
        "PrimaryRoutingRule": {
            "MinimumPriorOperatingDays":
                MINIMUM_HISTORY_OPERATING_DAYS,
            "PrimaryPriorDemandScopePercentage":
                PRIMARY_SCOPE_PERCENTAGE,
            "MainModelEligibility":
                (
                    "At least 20 prior operating days, positive "
                    "prior normal demand, and membership in the "
                    "dynamic prior-demand 95 percent scope"
                ),
            "LowDemandHandling":
                "Explicit fallback route; method selected in ND04",
            "ColdStartHandling":
                "Explicit fallback route; method selected in ND04",
            "ZeroHistoryHandling":
                "Explicit fallback route; method selected in ND04",
        },
        "EvaluationProtocol": {
            "MethodSelectionPeriod":
                "Dates before 2026-03-02",
            "OpenedMarchDiagnosticPeriod":
                "2026-03-02 to 2026-03-30",
            "MarchEligibleForMethodSelection": False,
            "CurrentRowsEligibleForNewUnbiasedFinalTest":
                False,
            "FutureUntouchedPeriodRequired": True,
        },
        "BulkPolicy": {
            "ModelTargetIncludesBulk": False,
            "ModelPanelContainsBulkColumn": False,
            "HistoricalBulkAuditPath":
                str(BULK_AUDIT_PATH),
            "ConfirmedBulkInputTemplatePath":
                str(BULK_INPUT_TEMPLATE_PATH),
            "OperationalCombination":
                (
                    "Final planned product quantity equals "
                    "forecast normal demand plus confirmed bulk"
                ),
        },
        "Schema": dataframe_schema(panel),
    }

    forecasting_contract = {
        "StepID": STEP_ID,
        "Status": STATUS,
        "ForecastingEngine":
            "DAILY_PRODUCT_LEVEL_NORMAL_DEMAND_MODEL",
        "CoreOutputs": [
            "Daily product normal-demand forecast",
            "Daily restaurant-total normal-demand forecast",
            (
                "Weekly product normal-demand forecast produced "
                "by aggregating daily predictions"
            ),
            (
                "Weekly restaurant-total normal-demand forecast "
                "produced by aggregation"
            ),
        ],
        "SingleDateForecast": {
            "InterfaceConcept":
                'forecast_date("YYYY-MM-DD")',
            "NextOperatingDay":
                "Use actual history through the latest observed date",
            "LaterFutureDate":
                (
                    "Forecast every missing operating day "
                    "recursively until the requested date"
                ),
            "HistoricalDate":
                (
                    "Backtest using data strictly before the "
                    "requested historical date"
                ),
        },
        "WeeklyForecast": {
            "InterfaceConcept":
                'forecast_week("YYYY-MM-DD")',
            "RequiredInput":
                "Monday date for the requested week",
            "DefaultOperatingHorizon":
                "Monday to Friday",
            "Method":
                (
                    "Generate daily product forecasts recursively "
                    "from Monday through Friday and aggregate them"
                ),
            "ForecastOrigin":
                "Before the requested week begins",
            "ActualWithinWeekUsedForInitialForecast": False,
            "ExceptionalWeekendPolicy":
                (
                    "Weekend operating dates are excluded from the "
                    "standard UI week unless explicitly overridden"
                ),
        },
        "RollingUpdate": {
            "Supported": True,
            "Method":
                (
                    "Replace completed-day predictions with actual "
                    "normal demand and regenerate the remaining week"
                ),
            "DistinctFromInitialWeeklyForecast": True,
        },
        "BulkIntegration": {
            "PredictedByModel": False,
            "EnteredExternally": True,
            "CombinationStage":
                "After normal-demand forecasting and before ingredient mapping",
        },
        "IngredientMapping": {
            "Input":
                (
                    "Normal-demand forecast plus confirmed "
                    "bulk quantity"
                ),
            "ImplementedInND02": False,
            "PlannedAfterForecastingPipeline": True,
        },
        "UI": {
            "ImplementedBeforeModelCompletion": False,
            "ImplementedAfterModelValidation": True,
            "PlannedCapabilities": [
                "Select one forecast date",
                "Select one Monday-to-Friday week",
                "View daily product forecasts",
                "View daily restaurant total",
                "View weekly product totals",
                "View weekly restaurant total",
                "Enter confirmed bulk quantities",
                "Update remaining-week forecast",
                "Export forecast outputs",
                "Pass final quantities to ingredient mapping",
            ],
        },
    }

    dataset_contract_markdown = f"""# ND02 Normal-Demand Dataset Contract

## Authoritative source

- Path: `{SOURCE_COPY_PATH}`
- SHA-256: `{source_sha256_before}`
- Rows: {source_row_count:,}
- Products: {source_product_count:,}
- Operating dates: {source_operating_date_count:,}
- Date range: {source_minimum_date.date()} to {source_maximum_date.date()}

## Modelling target

The only demand target in the ND02 modelling panel is `NormalDemand`.

The panel does not contain `BulkDemand`, `TotalDemand`, `IsObservedProductDate`, `IsZeroDemandRow`, or `DemandRecordSource`.

## Product routing

The primary main-model route requires:

- at least {MINIMUM_HISTORY_OPERATING_DAYS} prior operating-day observations;
- positive prior normal-demand history; and
- membership in the dynamic prior-demand {PRIMARY_SCOPE_PERCENTAGE}% scope.

All other products remain in the dataset and receive an explicit fallback route. The fallback method will be selected in ND04.

## Evaluation boundary

Rows before 2 March 2026 may be used for chronological model development.

Rows from 2 March to 30 March 2026 have already been examined and are diagnostic only. They are not eligible for selecting or tuning the revised model.

A new untouched future period is required for a genuinely unbiased final evaluation.

## Bulk policy

Historical bulk quantities are preserved only in the separate audit file.

The operational quantity is:

Final planned product quantity = forecast normal demand + confirmed bulk quantity
"""

    forecasting_contract_markdown = f"""# ND02 Forecasting Architecture Contract

## Main forecasting engine

The project will build one daily product-level normal-demand forecasting engine.

## Derived outputs

The same daily engine must produce:

- daily product forecasts;
- daily restaurant-total forecasts;
- weekly product forecasts by adding Monday-to-Friday daily predictions; and
- weekly restaurant-total forecasts by aggregation.

## Arbitrary-date forecasting

The future inference pipeline must support one requested date.

For a date later than the next operating day, every missing operating day must be forecast recursively so that lag and rolling features can be updated without invented actual values.

A historical requested date must be treated as a leakage-safe backtest using only data before that date.

## Weekly forecasting

The standard weekly interface accepts a Monday and generates Monday-to-Friday predictions before the week begins.

Tuesday-to-Friday features in the initial weekly forecast must use recursively generated predictions, not actual demand from within the future week.

## Rolling update

After completed-day actual demand becomes available, the system may replace that day's forecast with the actual value and regenerate the remaining week. This is reported separately from the initial week-start forecast.

## Bulk and ingredient mapping

Bulk orders are not forecast. Confirmed quantities are entered separately and added after the normal-demand forecast.

Ingredient mapping will use the combined planned product quantities and will be implemented after the forecasting pipeline is complete.

## UI timing

The UI will be developed after modelling, validation, future inference, weekly aggregation, rolling updates, and bulk integration are complete.
"""

    # =========================================================================
    # OUTPUT SCHEMA AUDIT
    # =========================================================================

    panel_schema = pd.DataFrame(
        dataframe_schema(panel)
    )

    panel_schema["Role"] = "CANDIDATE_METADATA_OR_PRIOR_FIELD"

    panel_schema.loc[
        panel_schema["Column"].isin(
            identifier_columns
        ),
        "Role",
    ] = "IDENTIFIER"

    panel_schema.loc[
        panel_schema["Column"].isin(
            target_columns
        ),
        "Role",
    ] = "TARGET_NOT_PREDICTOR"

    panel_schema.loc[
        panel_schema["Column"].isin(
            prior_only_columns
        ),
        "Role",
    ] = "PRIOR_ONLY_FIELD"

    panel_schema.loc[
        panel_schema["Column"].isin(
            routing_columns
        ),
        "Role",
    ] = "ROUTING_OR_DIAGNOSTIC"

    panel_schema.loc[
        panel_schema["Column"].isin(
            administrative_non_predictor_columns
        ),
        "Role",
    ] = "ADMINISTRATIVE_NOT_PREDICTOR"

    schema_audit = panel_schema

    # =========================================================================
    # REPORT AND README
    # =========================================================================

    main_route_row = route_summary.loc[
        route_summary["ForecastRoute"]
        == "MAIN_MODEL"
    ]

    if main_route_row.empty:
        main_route_rows = 0
        main_route_demand = 0.0
        main_route_demand_share = 0.0
    else:
        main_route_rows = int(
            main_route_row.iloc[0][
                "ProductDateRows"
            ]
        )
        main_route_demand = float(
            main_route_row.iloc[0][
                "ActualNormalDemand"
            ]
        )
        main_route_demand_share = float(
            main_route_row.iloc[0][
                "ShareOfAllNormalDemandPercentage"
            ]
        )

    exceptional_weekend_dates = int(
        operating_calendar.loc[
            operating_calendar[
                "IsExceptionalWeekendOperatingDate"
            ],
            "Date",
        ].nunique()
    )

    report_summary_text = f"""# ND02 Preparation Summary

## Status

`{STATUS}`

## Source integrity

- Source rows: {source_row_count:,}
- Products: {source_product_count:,}
- Operating dates: {source_operating_date_count:,}
- Source SHA-256: `{source_sha256_before}`
- Source modified: no

## Demand separation

- Normal demand retained in modelling panel: {panel_normal_total:,.6f}
- Historical bulk demand moved to separate audit route: {bulk_audit_total:,.6f}
- Historical bulk rows: {bulk_audit_rows:,}
- Historical bulk dates: {bulk_audit_dates:,}
- Products with historical bulk demand: {bulk_audit_products:,}
- BulkDemand present in modelling panel: no
- TotalDemand present in modelling panel: no

## Product routing

- Minimum prior history for the main model: {MINIMUM_HISTORY_OPERATING_DAYS} operating days
- Primary dynamic scope: {PRIMARY_SCOPE_PERCENTAGE}%
- Main-model product-date rows: {main_route_rows:,}
- Normal demand on main-model rows: {main_route_demand:,.6f}
- Main-model share of all source normal demand: {main_route_demand_share:.6f}%

The preceding share includes the initial cold-start period. Coverage among established products is reported separately in the scope-coverage table.

## Forecasting contract

The future system will use one daily product-level normal-demand model.

Weekly product and restaurant forecasts will be created by generating Monday-to-Friday daily forecasts recursively and then aggregating them.

The model will support arbitrary requested dates through recursive future feature generation.

Rolling remaining-week forecasts will be reported separately from initial week-start forecasts.

Confirmed bulk orders will be entered externally and added before ingredient mapping.

## Evaluation boundary

- Pre-March data: eligible for chronological method development
- March 2026: opened diagnostic period only
- New unbiased final test: requires a future untouched period

## ND03

ND03 will rebuild lag, rolling, zero-rate, positive-rate, and expanding features entirely from prior `NormalDemand` values.
"""

    readme_text = f"""# ND02 Normal-Demand Preparation

Status: `{STATUS}`

This folder contains the bulk-free, leakage-safe base panel for the revised Eden daily forecasting system.

## Folders

- `01_data`: panel, eligibility, operating calendar, bulk audit, and bulk input template
- `02_audits`: coverage, routing, split, leakage, validation, and schema audits
- `03_contracts`: dataset and forecasting architecture contracts
- `04_reports`: report-ready preparation summary
- `05_control`: hashes and checkpoint

## Main file

`01_data/ND02_normal_demand_daily_panel.csv`

## Important rules

- Model target: `NormalDemand`
- Bulk prediction: not performed
- Confirmed bulk: entered externally
- Main model: established products in the dynamic prior-demand 95% scope
- Outside-scope products: retained and routed to fallback
- Weekly forecast: aggregate recursive Monday-to-Friday daily predictions
- March 2026: diagnostic only
- Models fitted in ND02: none
- ND02 lock created: none; checkpoint and hashes only
"""

    # =========================================================================
    # WRITE STAGED OUTPUTS
    # =========================================================================

    staged_data_dir = (
        STAGING_ROOT
        / DATA_DIR.relative_to(ND02_ROOT)
    )
    staged_audit_dir = (
        STAGING_ROOT
        / AUDIT_DIR.relative_to(ND02_ROOT)
    )
    staged_contract_dir = (
        STAGING_ROOT
        / CONTRACT_DIR.relative_to(ND02_ROOT)
    )
    staged_report_dir = (
        STAGING_ROOT
        / REPORT_DIR.relative_to(ND02_ROOT)
    )
    staged_control_dir = (
        STAGING_ROOT
        / CONTROL_DIR.relative_to(ND02_ROOT)
    )

    output_frames = {
        staged_data_dir
        / PANEL_PATH.name:
            panel,
        staged_data_dir
        / ELIGIBILITY_PATH.name:
            eligibility_register,
        staged_data_dir
        / LATEST_STATUS_PATH.name:
            latest_product_status,
        staged_data_dir
        / OPERATING_CALENDAR_PATH.name:
            operating_calendar,
        staged_data_dir
        / BULK_AUDIT_PATH.name:
            bulk_audit,
        staged_data_dir
        / BULK_PRODUCT_SUMMARY_PATH.name:
            bulk_product_summary,
        staged_data_dir
        / BULK_DATE_SUMMARY_PATH.name:
            bulk_date_summary,
        staged_data_dir
        / BULK_INPUT_TEMPLATE_PATH.name:
            bulk_input_template,
        staged_audit_dir
        / COVERAGE_BY_DATE_PATH.name:
            coverage_by_date,
        staged_audit_dir
        / COVERAGE_SUMMARY_PATH.name:
            coverage_summary,
        staged_audit_dir
        / ROUTE_SUMMARY_PATH.name:
            route_summary,
        staged_audit_dir
        / SPLIT_SUMMARY_PATH.name:
            split_summary,
        staged_audit_dir
        / LEAKAGE_AUDIT_PATH.name:
            leakage_audit,
        staged_audit_dir
        / VALIDATION_PATH.name:
            validation,
        staged_audit_dir
        / SCHEMA_AUDIT_PATH.name:
            schema_audit,
    }

    for output_path, output_frame in output_frames.items():
        write_csv(output_path, output_frame)

    write_json(
        staged_contract_dir
        / DATASET_CONTRACT_JSON_PATH.name,
        dataset_contract,
    )

    write_json(
        staged_contract_dir
        / FORECAST_CONTRACT_JSON_PATH.name,
        forecasting_contract,
    )

    write_text(
        staged_contract_dir
        / DATASET_CONTRACT_MD_PATH.name,
        dataset_contract_markdown,
    )

    write_text(
        staged_contract_dir
        / FORECAST_CONTRACT_MD_PATH.name,
        forecasting_contract_markdown,
    )

    write_text(
        staged_report_dir
        / REPORT_SUMMARY_PATH.name,
        report_summary_text,
    )

    write_text(
        STAGING_ROOT / README_PATH.name,
        readme_text,
    )

    # =========================================================================
    # RELOAD CRITICAL OUTPUTS AND VALIDATE SERIALISATION
    # =========================================================================

    reloaded_panel = pd.read_csv(
        staged_data_dir / PANEL_PATH.name,
        low_memory=False,
    )

    reloaded_bulk_audit = pd.read_csv(
        staged_data_dir / BULK_AUDIT_PATH.name,
        low_memory=False,
    )

    if len(reloaded_panel) != len(panel):
        raise AssertionError(
            "Reloaded panel row count changed after CSV serialisation."
        )

    if not math.isclose(
        float(reloaded_panel["NormalDemand"].sum()),
        panel_normal_total,
        rel_tol=0.0,
        abs_tol=1e-9,
    ):
        raise AssertionError(
            "Reloaded panel normal-demand total changed."
        )

    if "BulkDemand" in reloaded_panel.columns:
        raise AssertionError(
            "BulkDemand appeared in the reloaded model panel."
        )

    if "TotalDemand" in reloaded_panel.columns:
        raise AssertionError(
            "TotalDemand appeared in the reloaded model panel."
        )

    if not math.isclose(
        float(
            reloaded_bulk_audit[
                "HistoricalConfirmedBulkDemand"
            ].sum()
        ),
        source_bulk_total,
        rel_tol=0.0,
        abs_tol=1e-9,
    ):
        raise AssertionError(
            "Reloaded historical bulk audit total changed."
        )

    # =========================================================================
    # MANIFEST
    # =========================================================================

    manifest_excluded_names = {
        MANIFEST_PATH.name,
        CHECKPOINT_PATH.name,
        CHECKPOINT_SHA_PATH.name,
    }

    files_for_manifest = sorted(
        path
        for path in STAGING_ROOT.rglob("*")
        if (
            path.is_file()
            and path.name
            not in manifest_excluded_names
        )
    )

    manifest = pd.DataFrame(
        [
            {
                "RelativePath":
                    str(path.relative_to(STAGING_ROOT)),
                "Bytes":
                    int(path.stat().st_size),
                "SHA256":
                    sha256_file(path),
            }
            for path in files_for_manifest
        ]
    ).sort_values(
        "RelativePath"
    ).reset_index(drop=True)

    staged_manifest_path = (
        staged_control_dir / MANIFEST_PATH.name
    )

    write_csv(
        staged_manifest_path,
        manifest,
    )

    manifest_sha256 = sha256_file(
        staged_manifest_path
    )

    # =========================================================================
    # CHECKPOINT
    # =========================================================================

    checkpoint_payload = {
        "StepID": STEP_ID,
        "Status": STATUS,
        "CreatedUTC": NOW_UTC.isoformat(),
        "CreatedLocal": NOW_LOCAL.isoformat(),
        "ModelRoot": str(MODEL_ROOT),
        "ND02Root": str(ND02_ROOT),
        "Source": {
            "Path": str(SOURCE_COPY_PATH),
            "SHA256": source_sha256_before,
            "Rows": source_row_count,
            "Products": source_product_count,
            "OperatingDates":
                source_operating_date_count,
            "NormalDemandTotal":
                source_normal_total,
            "BulkDemandTotal":
                source_bulk_total,
            "TotalDemandTotal":
                source_total_total,
        },
        "OutputPanel": {
            "Path": str(PANEL_PATH),
            "Rows": int(len(panel)),
            "Columns": int(len(panel.columns)),
            "NormalDemandTotal":
                panel_normal_total,
            "ContainsBulkDemand": False,
            "ContainsTotalDemand": False,
            "AllSourceKeysPreserved":
                source_key_set == panel_key_set,
        },
        "BulkAudit": {
            "Path": str(BULK_AUDIT_PATH),
            "Rows": bulk_audit_rows,
            "Dates": bulk_audit_dates,
            "Products": bulk_audit_products,
            "BulkDemandTotal":
                bulk_audit_total,
        },
        "Routing": {
            "MinimumPriorOperatingDays":
                MINIMUM_HISTORY_OPERATING_DAYS,
            "PrimaryScopePercentage":
                PRIMARY_SCOPE_PERCENTAGE,
            "RouteSummary":
                route_summary.to_dict(
                    orient="records"
                ),
        },
        "Evaluation": {
            "DevelopmentEndExclusive":
                DEVELOPMENT_END_EXCLUSIVE.date().isoformat(),
            "OpenedMarchStart":
                OPENED_MARCH_START.date().isoformat(),
            "OpenedMarchEnd":
                OPENED_MARCH_END.date().isoformat(),
            "MarchEligibleForMethodSelection":
                False,
            "FutureUntouchedPeriodRequired":
                True,
        },
        "ForecastingArchitecture": {
            "DailyProductForecast": True,
            "DailyRestaurantAggregate": True,
            "WeeklyProductByDailyAggregation": True,
            "WeeklyRestaurantByDailyAggregation": True,
            "ArbitraryDateRecursiveForecastingPlanned":
                True,
            "RollingRemainingWeekUpdatePlanned":
                True,
            "BulkEnteredExternally": True,
            "UIAfterModelCompletion": True,
        },
        "Control": {
            "ManifestPath": str(MANIFEST_PATH),
            "ManifestSHA256": manifest_sha256,
            "ValidationPath": str(VALIDATION_PATH),
            "LeakageAuditPath": str(
                LEAKAGE_AUDIT_PATH
            ),
            "DatasetContractPath": str(
                DATASET_CONTRACT_JSON_PATH
            ),
            "ForecastContractPath": str(
                FORECAST_CONTRACT_JSON_PATH
            ),
        },
        "Safety": {
            "ModelsLoaded": False,
            "ModelsFitted": False,
            "ModelsRefitted": False,
            "PredictionsCreated": False,
            "SourceModified": False,
            "ND01LockModified": False,
            "ExistingModelLocksModified": False,
            "ND02StepLockCreated": False,
            "CheckpointAndHashesCreated": True,
        },
        "ReadyForND03": True,
        "NextStep": "ND03",
    }

    staged_checkpoint_path = (
        staged_control_dir / CHECKPOINT_PATH.name
    )

    write_json(
        staged_checkpoint_path,
        checkpoint_payload,
    )

    checkpoint_sha256 = sha256_file(
        staged_checkpoint_path
    )

    staged_checkpoint_sha_path = (
        staged_control_dir / CHECKPOINT_SHA_PATH.name
    )

    write_text(
        staged_checkpoint_sha_path,
        (
            f"{checkpoint_sha256}  "
            f"{CHECKPOINT_PATH.name}\n"
        ),
    )

    # =========================================================================
    # FINAL STAGING VALIDATION
    # =========================================================================

    required_staged_outputs = [
        STAGING_ROOT / README_PATH.name,
        staged_data_dir / PANEL_PATH.name,
        staged_data_dir / ELIGIBILITY_PATH.name,
        staged_data_dir / LATEST_STATUS_PATH.name,
        staged_data_dir / OPERATING_CALENDAR_PATH.name,
        staged_data_dir / BULK_AUDIT_PATH.name,
        staged_data_dir / BULK_INPUT_TEMPLATE_PATH.name,
        staged_audit_dir / COVERAGE_BY_DATE_PATH.name,
        staged_audit_dir / COVERAGE_SUMMARY_PATH.name,
        staged_audit_dir / ROUTE_SUMMARY_PATH.name,
        staged_audit_dir / LEAKAGE_AUDIT_PATH.name,
        staged_audit_dir / VALIDATION_PATH.name,
        staged_contract_dir
        / DATASET_CONTRACT_JSON_PATH.name,
        staged_contract_dir
        / FORECAST_CONTRACT_JSON_PATH.name,
        staged_report_dir / REPORT_SUMMARY_PATH.name,
        staged_manifest_path,
        staged_checkpoint_path,
        staged_checkpoint_sha_path,
    ]

    missing_staged_outputs = [
        path
        for path in required_staged_outputs
        if not path.is_file()
    ]

    if missing_staged_outputs:
        raise AssertionError(
            "Required staged ND02 outputs are missing:\n"
            + "\n".join(
                f"- {path}"
                for path in missing_staged_outputs
            )
        )

    # =========================================================================
    # VERIFY PROTECTED INPUTS REMAIN UNCHANGED
    # =========================================================================

    source_sha256_after = sha256_file(
        SOURCE_COPY_PATH
    )

    nd01_lock_sha256_after = sha256_file(
        ND01_LOCK_PATH
    )

    nd01_checkpoint_sha256_after = sha256_file(
        ND01_CHECKPOINT_PATH
    )

    if source_sha256_after != source_sha256_before:
        raise AssertionError(
            "The immutable source copy changed during ND02."
        )

    if nd01_lock_sha256_after != nd01_lock_sha256_before:
        raise AssertionError(
            "The ND01 lock changed during ND02."
        )

    if (
        nd01_checkpoint_sha256_after
        != nd01_checkpoint_sha256_before
    ):
        raise AssertionError(
            "The ND01 checkpoint changed during ND02."
        )

    # =========================================================================
    # ATOMIC ND02 DIRECTORY COMMIT
    # =========================================================================

    os.replace(
        STAGING_ROOT,
        ND02_ROOT,
    )

    # =========================================================================
    # TOP-LEVEL CHECKPOINT COPY
    # =========================================================================

    TOP_LEVEL_CHECKPOINT_PATH.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    shutil.copy2(
        CHECKPOINT_PATH,
        TOP_LEVEL_CHECKPOINT_PATH,
    )

    shutil.copy2(
        CHECKPOINT_SHA_PATH,
        TOP_LEVEL_CHECKPOINT_SHA_PATH,
    )

    make_read_only(
        TOP_LEVEL_CHECKPOINT_PATH
    )

    make_read_only(
        TOP_LEVEL_CHECKPOINT_SHA_PATH
    )

    # =========================================================================
    # PROJECT MEMORY AND AGENT HANDOFF
    # =========================================================================

    nd02_handoff_text = f"""# ND02 Handoff

## Current status

- Completed step: `{STEP_ID}`
- Status: `{STATUS}`
- Completed local time: `{NOW_LOCAL.isoformat()}`
- ND02 root: `{ND02_ROOT}`
- Main panel: `{PANEL_PATH}`
- Checkpoint: `{TOP_LEVEL_CHECKPOINT_PATH}`
- Checkpoint SHA-256: `{checkpoint_sha256}`

## Authoritative dataset decision

The revised model target is `NormalDemand`.

The ND02 model panel contains no `BulkDemand`, `TotalDemand`, `IsObservedProductDate`, `IsZeroDemandRow`, or `DemandRecordSource` fields.

Historical bulk demand is stored only in the separate audit file and must not be used as a predictor.

## Authoritative routing decision

- Minimum history: {MINIMUM_HISTORY_OPERATING_DAYS} prior operating days
- Primary scope: dynamic prior-demand {PRIMARY_SCOPE_PERCENTAGE}%
- Main route: established products inside prior 95% scope
- Other products: explicit fallback route
- Fallback method: selected later in ND04

## Authoritative forecasting design

One daily product-level model will produce:

- daily product forecasts;
- daily restaurant totals;
- Monday-to-Friday weekly product totals by aggregation;
- weekly restaurant totals by aggregation;
- arbitrary-date recursive forecasts; and
- rolling remaining-week updates.

Confirmed bulk quantities are added externally before ingredient mapping.

## Evaluation boundary

- Dates before 2 March 2026: chronological model development
- March 2026: opened diagnostic period only
- New unbiased final test: future untouched period required

## Next step

`ND03` will rebuild all lag, rolling, zero-rate, positive-rate, recency, and expanding features from prior `NormalDemand` only.
"""

    current_handoff_text = nd02_handoff_text

    workflow_section = f"""## ND02 — Bulk-free normal-demand panel and routing

Status: `{STATUS}`

Completed actions:

- verified the ND01 source, lock, and checkpoint;
- preserved all {source_row_count:,} product-date rows;
- retained `NormalDemand` as the sole demand target;
- removed bulk, total-demand, and same-day demand-status fields from the modelling panel;
- created a separate historical bulk audit and future bulk-input template;
- created prior-only history, rank, demand segment, scope, cold-start, and routing fields;
- created 80%, 90%, and 95% coverage audits;
- recorded daily, weekly, arbitrary-date, rolling-update, bulk, ingredient-mapping, and UI contracts;
- marked March 2026 as diagnostic only; and
- created hashes and a checkpoint without creating an ND02 step lock.

Next: ND03 feature engineering.
"""

    decisions_section = f"""## ND02 decisions

1. The revised statistical target is `NormalDemand`.
2. `BulkDemand` and `TotalDemand` are absent from the modelling panel.
3. Same-day demand-derived status fields are absent from the modelling panel.
4. Bulk is retained only in a separate audit and external input route.
5. No product-date row is deleted.
6. Product eligibility is recomputed from prior data for each date.
7. The primary scope is the dynamic prior-demand {PRIMARY_SCOPE_PERCENTAGE}% scope.
8. Main-model eligibility requires {MINIMUM_HISTORY_OPERATING_DAYS} prior operating days.
9. Outside-scope, cold-start, and zero-history products use explicit fallback routes.
10. The fallback method will be selected empirically in ND04.
11. Weekly forecasts are generated by aggregating recursive Monday-to-Friday daily forecasts.
12. Arbitrary future dates require recursive generation of every missing operating day.
13. Rolling remaining-week forecasts are distinct from initial week-start forecasts.
14. March 2026 is not eligible for selecting or tuning the revised model.
15. UI development begins only after the forecasting pipeline is complete.
16. ND02 uses a checkpoint and hashes; no ND02 step lock was created.
"""

    metrics_section = f"""## ND02 dataset preparation

- Rows preserved: {len(panel):,}
- Products preserved: {panel['CanonicalProductID'].nunique():,}
- Operating dates preserved: {panel['Date'].nunique():,}
- Normal demand retained: {panel_normal_total:,.6f}
- Historical bulk demand separated: {bulk_audit_total:,.6f}
- Historical bulk rows: {bulk_audit_rows:,}
- Historical bulk dates: {bulk_audit_dates:,}
- Products with bulk demand: {bulk_audit_products:,}
- Main-model product-date rows: {main_route_rows:,}
- Main-model normal demand: {main_route_demand:,.6f}
- Main-model share of all source normal demand: {main_route_demand_share:.6f}%
- Exceptional historical weekend operating dates: {exceptional_weekend_dates}
- Models fitted: no
- Predictions generated: no
"""

    agents_section = f"""## ND02 authoritative status

Marker: ND02_AUTHORITATIVE_STATUS

- Status: `{STATUS}`
- Read next: `{ND02_HANDOFF_PATH}`
- Main panel: `{PANEL_PATH}`
- Primary target: `NormalDemand`
- Bulk and total-demand fields in model panel: none
- Primary routing scope: dynamic prior-demand {PRIMARY_SCOPE_PERCENTAGE}%
- Minimum main-model history: {MINIMUM_HISTORY_OPERATING_DAYS} prior operating days
- Weekly design: recursive daily forecasts aggregated Monday to Friday
- Arbitrary-date design: recursive through every missing operating day
- March 2026: diagnostic only
- Next step: `ND03`
"""

    atomic_write_text(
        ND02_HANDOFF_PATH,
        nd02_handoff_text,
    )

    atomic_write_text(
        CURRENT_HANDOFF_PATH,
        current_handoff_text,
    )

    append_marked_section(
        WORKFLOW_PATH,
        "## ND02 — Bulk-free normal-demand panel and routing",
        workflow_section,
    )

    append_marked_section(
        DECISIONS_PATH,
        "## ND02 decisions",
        decisions_section,
    )

    append_marked_section(
        METRICS_AND_RESULTS_PATH,
        "## ND02 dataset preparation",
        metrics_section,
    )

    append_marked_section(
        AGENTS_PATH,
        "Marker: ND02_AUTHORITATIVE_STATUS",
        agents_section,
    )

    # =========================================================================
    # LOG
    # =========================================================================

    log_text = "\n".join(
        [
            f"Step: {STEP_ID}",
            f"Status: {STATUS}",
            f"Created local: {NOW_LOCAL.isoformat()}",
            f"Created UTC: {NOW_UTC.isoformat()}",
            f"Source: {SOURCE_COPY_PATH}",
            f"Source SHA256: {source_sha256_before}",
            f"ND01 lock SHA256: {nd01_lock_sha256_before}",
            f"ND01 checkpoint SHA256: {nd01_checkpoint_sha256_before}",
            f"Panel: {PANEL_PATH}",
            f"Panel rows: {len(panel)}",
            f"Panel columns: {len(panel.columns)}",
            f"Normal demand: {panel_normal_total}",
            f"Bulk audit demand: {bulk_audit_total}",
            f"Main-model rows: {main_route_rows}",
            f"Main-model demand: {main_route_demand}",
            f"Main-model demand share: {main_route_demand_share}",
            f"Checkpoint SHA256: {checkpoint_sha256}",
            "Models loaded: False",
            "Models fitted: False",
            "Predictions created: False",
            "Source modified: False",
            "ND01 lock modified: False",
            "ND02 step lock created: False",
            "Ready for ND03: True",
            "",
        ]
    )

    atomic_write_text(
        LOG_PATH,
        log_text,
    )

except Exception:
    if STAGING_ROOT.exists():
        shutil.rmtree(STAGING_ROOT)
    raise


# =============================================================================
# FINAL OUTPUT
# =============================================================================

print("=" * 108)
print("EDEN NORMAL-DEMAND MODEL V2 — ND02 COMPLETE")
print("=" * 108)

print(f"Status: {STATUS}")
print(f"Local time: {NOW_LOCAL.isoformat()}")
print(f"ND02 root: {ND02_ROOT}")

print("\nSOURCE INTEGRITY")
print(f"Source: {SOURCE_COPY_PATH}")
print(f"Source SHA-256: {source_sha256_before}")
print(f"ND01 lock SHA-256: {nd01_lock_sha256_before}")
print(f"ND01 checkpoint SHA-256: {nd01_checkpoint_sha256_before}")
print("Source modified: False")

print("\nBULK-FREE NORMAL-DEMAND PANEL")
print(f"Panel: {PANEL_PATH}")
print(f"Rows: {len(panel):,}")
print(f"Columns: {len(panel.columns):,}")
print(
    "Products: "
    f"{panel['CanonicalProductID'].nunique():,}"
)
print(
    "Operating dates: "
    f"{panel['Date'].nunique():,}"
)
print(
    "Date range: "
    f"{panel['Date'].min().date()} to "
    f"{panel['Date'].max().date()}"
)
print(f"Normal demand retained: {panel_normal_total:,.6f}")
print("BulkDemand column present: False")
print("TotalDemand column present: False")
print("Same-day demand-status columns present: False")
print("Product-date rows removed: 0")

print("\nHISTORICAL BULK AUDIT")
print(f"Audit file: {BULK_AUDIT_PATH}")
print(f"Rows with bulk demand: {bulk_audit_rows:,}")
print(f"Bulk-order dates: {bulk_audit_dates:,}")
print(f"Products with bulk demand: {bulk_audit_products:,}")
print(f"Historical bulk demand: {bulk_audit_total:,.6f}")
print(f"Future bulk input template: {BULK_INPUT_TEMPLATE_PATH}")

print("\nPRIMARY ROUTING")
print(
    "Minimum prior operating days: "
    f"{MINIMUM_HISTORY_OPERATING_DAYS}"
)
print(
    "Dynamic prior-demand scope: "
    f"{PRIMARY_SCOPE_PERCENTAGE}%"
)
print(f"Main-model product-date rows: {main_route_rows:,}")
print(f"Main-model normal demand: {main_route_demand:,.6f}")
print(
    "Main-model share of all source normal demand: "
    f"{main_route_demand_share:.6f}%"
)
print("\nRoute summary:")
print(route_summary.to_string(index=False))

print("\nEVALUATION BOUNDARY")
print("Pre-March period eligible for chronological development: True")
print("March 2026 eligible for method selection: False")
print("New untouched future period required: True")

print("\nFORECASTING CONTRACT")
print("- Daily product forecast: planned")
print("- Daily restaurant-total forecast: planned")
print("- Weekly product forecast by daily aggregation: planned")
print("- Weekly restaurant total by daily aggregation: planned")
print("- Arbitrary-date recursive forecast: planned")
print("- Rolling remaining-week update: planned")
print("- Confirmed bulk entered externally: planned")
print("- UI after modelling and validation: planned")

print("\nOUTPUTS")
print(f"- Eligibility register: {ELIGIBILITY_PATH}")
print(f"- Latest product status: {LATEST_STATUS_PATH}")
print(f"- Operating calendar: {OPERATING_CALENDAR_PATH}")
print(f"- Coverage by date: {COVERAGE_BY_DATE_PATH}")
print(f"- Coverage summary: {COVERAGE_SUMMARY_PATH}")
print(f"- Route summary: {ROUTE_SUMMARY_PATH}")
print(f"- Leakage audit: {LEAKAGE_AUDIT_PATH}")
print(f"- Validation: {VALIDATION_PATH}")
print(f"- Dataset contract: {DATASET_CONTRACT_JSON_PATH}")
print(f"- Forecast contract: {FORECAST_CONTRACT_JSON_PATH}")
print(f"- Report summary: {REPORT_SUMMARY_PATH}")
print(f"- Manifest: {MANIFEST_PATH}")
print(f"- Checkpoint: {TOP_LEVEL_CHECKPOINT_PATH}")
print(f"- Checkpoint SHA-256: {checkpoint_sha256}")
print(f"- Agent handoff: {ND02_HANDOFF_PATH}")

print("\nSAFETY")
print("- Models loaded: False")
print("- Models fitted/refitted: False")
print("- Predictions created: False")
print("- Source modified: False")
print("- ND01 lock/checkpoint modified: False")
print("- Existing model locks modified: False")
print("- ND02 step lock created: False")
print("- ND02 checkpoint and hashes created: True")

print("\nNEXT STEP")
print(
    "ND03 — rebuild all daily forecasting lags, rolling "
    "statistics, recency fields, and expanding features from "
    "prior NormalDemand only."
)

print("=" * 108)

EDEN NORMAL-DEMAND MODEL V2 — ND02 COMPLETE
Status: ND02_NORMAL_DEMAND_PANEL_CREATED_READY_FOR_ND03
Local time: 2026-08-07T22:18:19.292590+01:00
ND02 root: /Users/ryansmac/Desktop/Meng Project/eden_datasets/eden_normal_demand_model_v2/01_datasets/01_intermediate/ND02_normal_demand_preparation

SOURCE INTEGRITY
Source: /Users/ryansmac/Desktop/Meng Project/eden_datasets/eden_normal_demand_model_v2/01_datasets/00_source_copy/UL_EDEN_canonical_product_daily_demand_forecasting_final.csv
Source SHA-256: f8538b31df4a2751a6b34cbc0a0f82c7d0e1d441480e1c6bdadd8899bca293e2
ND01 lock SHA-256: 07678543a1ea5246a3e4b0d2a10ede516bcaf8d1696249c46e67109835d18ccf
ND01 checkpoint SHA-256: 81ca2ff46f7a9fbc7f80d74207cd26faba3137f1cd784a2ca3ba5ffd35249157
Source modified: False

BULK-FREE NORMAL-DEMAND PANEL
Panel: /Users/ryansmac/Desktop/Meng Project/eden_datasets/eden_normal_demand_model_v2/01_datasets/01_intermediate/ND02_normal_demand_preparation/01_data/ND02_normal_demand_daily_panel.csv
Rows: 25,405
Col

In [4]:
from __future__ import annotations

# =============================================================================
# EDEN NORMAL-DEMAND MODEL V2
# ND03 — NORMAL-DEMAND FEATURE ENGINEERING AND MODEL-READY DATASETS
#
# This step:
#   - verifies the accepted ND02 checkpoint and artefacts;
#   - rebuilds the original 53-predictor daily feature architecture using
#     prior NormalDemand only;
#   - preserves dynamic product routing from ND02;
#   - creates development, diagnostic, future-fit, and inference-state files;
#   - separates March diagnostic features from their target vault;
#   - validates leakage safety and recursive feature reproducibility;
#   - writes contracts, audits, hashes, checkpoint, and agent handoff;
#   - does not fit, tune, select, or score a forecasting model.
#
# Run this as one complete Jupyter cell.
# =============================================================================

import hashlib
import json
import math
import os
import shutil
import stat
import uuid
from datetime import datetime, timezone
from pathlib import Path
from zoneinfo import ZoneInfo

import numpy as np
import pandas as pd


# =============================================================================
# CONFIGURATION
# =============================================================================

PROJECT_ROOT = Path(
    os.environ.get(
        "EDEN_PROJECT_ROOT",
        "/Users/ryansmac/Desktop/Meng Project",
    )
)

EDEN_ROOT = PROJECT_ROOT / "eden_datasets"
MODEL_ROOT = EDEN_ROOT / "eden_normal_demand_model_v2"

ND02_ROOT = (
    MODEL_ROOT
    / "01_datasets"
    / "01_intermediate"
    / "ND02_normal_demand_preparation"
)

ND02_PANEL_PATH = (
    ND02_ROOT
    / "01_data"
    / "ND02_normal_demand_daily_panel.csv"
)

ND02_MANIFEST_PATH = (
    ND02_ROOT
    / "05_control"
    / "ND02_artifact_hash_manifest.csv"
)

ND02_VALIDATION_PATH = (
    ND02_ROOT
    / "02_audits"
    / "ND02_validation_summary.csv"
)

ND02_LEAKAGE_AUDIT_PATH = (
    ND02_ROOT
    / "02_audits"
    / "ND02_leakage_audit.csv"
)

ND02_CHECKPOINT_PATH = (
    MODEL_ROOT
    / "08_checkpoints"
    / "ND02_checkpoint.json"
)

EXPECTED_ND02_CHECKPOINT_SHA256 = (
    "824254459a630b4526eb606be85e2065"
    "76f34665f456654df510f7a01393ceed"
)

ND03_ROOT = (
    MODEL_ROOT
    / "02_feature_engineering"
    / "ND03_normal_demand_features"
)

DATA_DIR = ND03_ROOT / "01_model_ready_datasets"
AUDIT_DIR = ND03_ROOT / "02_audits"
CONTRACT_DIR = ND03_ROOT / "03_contracts"
REPORT_DIR = ND03_ROOT / "04_reports"
CONTROL_DIR = ND03_ROOT / "05_control"

FEATURE_PANEL_PATH = (
    DATA_DIR
    / "ND03_normal_demand_feature_panel.csv"
)

PRE_MARCH_ALL_ROUTES_PATH = (
    DATA_DIR
    / "ND03_pre_march_all_routes_development_dataset.csv"
)

PRE_MARCH_MAIN_MODEL_PATH = (
    DATA_DIR
    / "ND03_pre_march_main_model_development_dataset.csv"
)

PRE_MARCH_FALLBACK_PATH = (
    DATA_DIR
    / "ND03_pre_march_fallback_development_dataset.csv"
)

MARCH_SCORING_FEATURES_PATH = (
    DATA_DIR
    / "ND03_opened_march_diagnostic_scoring_features.csv"
)

MARCH_TARGET_VAULT_PATH = (
    DATA_DIR
    / "ND03_opened_march_diagnostic_target_vault.csv"
)

MARCH_MAIN_SCORING_FEATURES_PATH = (
    DATA_DIR
    / "ND03_opened_march_main_model_scoring_features.csv"
)

MARCH_MAIN_TARGET_VAULT_PATH = (
    DATA_DIR
    / "ND03_opened_march_main_model_target_vault.csv"
)

FUTURE_FIT_MAIN_MODEL_PATH = (
    DATA_DIR
    / "ND03_future_forecast_fit_main_model_dataset.csv"
)

RECURSIVE_STATE_PATH = (
    DATA_DIR
    / "ND03_recursive_inference_state.csv"
)

DATASET_INVENTORY_PATH = (
    AUDIT_DIR
    / "ND03_dataset_inventory.csv"
)

SPLIT_SUMMARY_PATH = (
    AUDIT_DIR
    / "ND03_split_summary.csv"
)

FEATURE_MISSINGNESS_PATH = (
    AUDIT_DIR
    / "ND03_predictor_missingness_audit.csv"
)

FEATURE_READINESS_PATH = (
    AUDIT_DIR
    / "ND03_route_feature_readiness.csv"
)

LEAKAGE_AUDIT_PATH = (
    AUDIT_DIR
    / "ND03_leakage_audit.csv"
)

RECURSIVE_COMPATIBILITY_PATH = (
    AUDIT_DIR
    / "ND03_recursive_feature_compatibility_audit.csv"
)

RECURSIVE_REPLAY_PATH = (
    AUDIT_DIR
    / "ND03_recursive_replay_audit.csv"
)

SCHEMA_AUDIT_PATH = (
    AUDIT_DIR
    / "ND03_output_schema_audit.csv"
)

VALIDATION_PATH = (
    AUDIT_DIR
    / "ND03_validation_summary.csv"
)

CORE_FEATURE_CONTRACT_PATH = (
    CONTRACT_DIR
    / "ND03_core_feature_contract.csv"
)

CORE_PREDICTOR_LIST_PATH = (
    CONTRACT_DIR
    / "ND03_core_predictor_list.csv"
)

OPTIONAL_PREDICTOR_LIST_PATH = (
    CONTRACT_DIR
    / "ND03_optional_candidate_predictor_list.csv"
)

DATASET_CONTRACT_PATH = (
    CONTRACT_DIR
    / "ND03_model_ready_dataset_contract.json"
)

RECURSIVE_CONTRACT_PATH = (
    CONTRACT_DIR
    / "ND03_recursive_inference_contract.json"
)

FEATURE_CONTRACT_MD_PATH = (
    CONTRACT_DIR
    / "ND03_feature_contract.md"
)

REPORT_SUMMARY_PATH = (
    REPORT_DIR
    / "ND03_feature_engineering_summary.md"
)

README_PATH = ND03_ROOT / "README.md"

MANIFEST_PATH = (
    CONTROL_DIR
    / "ND03_artifact_hash_manifest.csv"
)

CHECKPOINT_PATH = (
    CONTROL_DIR
    / "ND03_checkpoint.json"
)

CHECKPOINT_SHA_PATH = (
    CONTROL_DIR
    / "ND03_checkpoint.sha256"
)

TOP_LEVEL_CHECKPOINT_PATH = (
    MODEL_ROOT
    / "08_checkpoints"
    / "ND03_checkpoint.json"
)

TOP_LEVEL_CHECKPOINT_SHA_PATH = (
    MODEL_ROOT
    / "08_checkpoints"
    / "ND03_checkpoint.sha256"
)

MEMORY_ROOT = MODEL_ROOT / "00_project_memory"
AGENTS_PATH = MODEL_ROOT / "AGENTS.md"
CURRENT_HANDOFF_PATH = MEMORY_ROOT / "CURRENT_HANDOFF.md"
ND03_HANDOFF_PATH = MEMORY_ROOT / "ND03_HANDOFF.md"
WORKFLOW_PATH = MEMORY_ROOT / "WORKFLOW.md"
DECISIONS_PATH = MEMORY_ROOT / "DECISIONS.md"
METRICS_AND_RESULTS_PATH = MEMORY_ROOT / "METRICS_AND_RESULTS.md"
LOG_PATH = MODEL_ROOT / "09_logs" / "ND03_feature_engineering_log.txt"

DEVELOPMENT_END_EXCLUSIVE = pd.Timestamp("2026-03-02")
OPENED_MARCH_START = pd.Timestamp("2026-03-02")
OPENED_MARCH_END = pd.Timestamp("2026-03-30")

LAG_OPERATING_DAYS = [1, 2, 3, 5, 10, 20]
ROLLING_WINDOWS = [3, 5, 10, 20]
ZERO_POSITIVE_WINDOWS = [5, 10, 20]
MAXIMUM_REQUIRED_HISTORY = 20
PRIMARY_SCOPE_PERCENTAGE = 95

EXPECTED_ROWS = 25_405
EXPECTED_PRODUCTS = 227
EXPECTED_OPERATING_DATES = 245
EXPECTED_NORMAL_DEMAND_TOTAL = 114_186.0
EXPECTED_PRE_MARCH_ROWS = 23_763
EXPECTED_MARCH_ROWS = 1_642
EXPECTED_PRE_MARCH_MAIN_ROWS = 10_976
EXPECTED_MARCH_MAIN_ROWS = 999
EXPECTED_FUTURE_FIT_MAIN_ROWS = 11_975

FEATURE_ENGINEERING_VERSION = "ND03_NORMAL_DEMAND_FEATURES_V1"
STEP_ID = "ND03"
STATUS = "ND03_NORMAL_DEMAND_FEATURES_CREATED_READY_FOR_ND04"
ALLOW_OVERWRITE = False

NOW_UTC = datetime.now(timezone.utc)
NOW_LOCAL = NOW_UTC.astimezone(ZoneInfo("Europe/Dublin"))


# =============================================================================
# FEATURE DEFINITIONS
# =============================================================================

BASE_NUMERIC_PREDICTORS = [
    "ProductAgeOperatingDays",
    "SourcePLUCount",
    "IsMultiPLUCanonicalProduct",
    "OperatingDaySequence",
    "Year",
    "Month",
    "Quarter",
    "DayOfWeekNumber",
    "ISOYear",
    "ISOWeek",
    "DayOfYear",
    "IsWeekend",
    "DaysSincePreviousOperatingDate",
    "IsConsecutiveCalendarDay",
]

BASE_CATEGORICAL_PREDICTORS = [
    "SourceGroupCodes",
    "SourceGroupNames",
    "BeverageSeries",
    "BeverageType",
    "SupplierLabelsObserved",
    "TierProductFamily",
    "NominalPriceTier",
    "MenuGeneration",
]

LAG_FEATURES = [
    f"NormalDemandLag_{lag}"
    for lag in LAG_OPERATING_DAYS
]

ROLLING_FEATURES = []

for window in ROLLING_WINDOWS:
    ROLLING_FEATURES.extend(
        [
            f"PastNormalDemandRollingMean_{window}",
            f"PastNormalDemandRollingMedian_{window}",
            f"PastNormalDemandRollingStd_{window}",
            f"PastNormalDemandRollingSum_{window}",
        ]
    )

ZERO_RATE_FEATURES = [
    f"PastZeroNormalDemandRate_{window}"
    for window in ZERO_POSITIVE_WINDOWS
]

POSITIVE_COUNT_FEATURES = [
    f"PastPositiveNormalDemandCount_{window}"
    for window in ZERO_POSITIVE_WINDOWS
]

OTHER_HISTORICAL_FEATURES = [
    "OperatingDaysSincePreviousPositiveNormalDemand",
    "ExpandingPastMeanNormalDemand",
    "ExpandingPastPositiveNormalDemandRate",
]

HISTORICAL_DEMAND_PREDICTORS = (
    LAG_FEATURES
    + ROLLING_FEATURES
    + ZERO_RATE_FEATURES
    + POSITIVE_COUNT_FEATURES
    + OTHER_HISTORICAL_FEATURES
)

CORE_NUMERIC_PREDICTORS = (
    BASE_NUMERIC_PREDICTORS
    + HISTORICAL_DEMAND_PREDICTORS
)

CORE_CATEGORICAL_PREDICTORS = (
    BASE_CATEGORICAL_PREDICTORS
)

CORE_PREDICTORS = (
    BASE_NUMERIC_PREDICTORS
    + BASE_CATEGORICAL_PREDICTORS
    + HISTORICAL_DEMAND_PREDICTORS
)

OPTIONAL_CANDIDATE_PREDICTORS = [
    "CanonicalProductID",
    "PriorDemandVolumeSegment",
    "PriorMeanNormalDemand",
    "PriorPositiveDemandRate",
    "PriorZeroDemandRate",
    "PriorDemandRank",
    "PriorDemandSharePercentage",
    "OperatingDaysSincePriorPositiveDemand",
]

KEY_COLUMNS = [
    "Date",
    "CanonicalProductID",
]

SUPPORT_COLUMNS = [
    "CanonicalProductName",
    "ProductFirstObservedDate",
]

ROUTING_COLUMNS = [
    "PriorOperatingDayCount",
    "PriorCumulativeNormalDemand",
    "PriorPositiveNormalDemandDays",
    "PriorZeroNormalDemandDays",
    "PriorMeanNormalDemand",
    "PriorPositiveDemandRate",
    "PriorZeroDemandRate",
    "PriorDemandRank",
    "PriorDemandSharePercentage",
    "PriorCumulativeDemandShareBeforePercentage",
    "PriorCumulativeDemandSharePercentage",
    "PriorDemandVolumeSegment",
    "HasSufficientHistory20",
    "ColdStartFlag",
    "ZeroPriorNormalDemandFlag",
    "InPrior80DemandScope",
    "InPrior90DemandScope",
    "InPrior95DemandScope",
    "EligibleForMainModel80",
    "EligibleForMainModel90",
    "EligibleForMainModel95",
    "EligibleForMainModel",
    "ForecastRoute",
    "ForecastRouteReason",
    "FallbackMethodStatus",
]

SPLIT_COLUMNS = [
    "IsPreMarchDevelopmentPeriod",
    "IsOpenedMarchDiagnosticPeriod",
    "EligibleForMethodSelection",
    "EligibleForNewUnbiasedFinalEvaluation",
]

FEATURE_AVAILABILITY_COLUMNS = [
    "HistoricalFeatureAvailableCount",
    "HistoricalFeatureMissingCount",
    "AllHistoricalFeaturesAvailable",
    "CorePredictorAvailableCount",
    "CorePredictorMissingCount",
    "FeatureEngineeringVersion",
]

MODEL_DATASET_SUPPORT_COLUMNS = (
    KEY_COLUMNS
    + SUPPORT_COLUMNS
    + [
        "ForecastRoute",
        "PriorDemandVolumeSegment",
        "EligibleForMainModel",
        "HasSufficientHistory20",
        "ColdStartFlag",
        "ZeroPriorNormalDemandFlag",
        "PriorOperatingDayCount",
        "PriorCumulativeNormalDemand",
        "PriorPositiveNormalDemandDays",
        "HistoricalFeatureAvailableCount",
        "AllHistoricalFeaturesAvailable",
        "IsPreMarchDevelopmentPeriod",
        "IsOpenedMarchDiagnosticPeriod",
        "EligibleForMethodSelection",
        "FeatureEngineeringVersion",
    ]
)

TARGET_COLUMN = "NormalDemand"


# =============================================================================
# HELPERS
# =============================================================================

def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()

    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)

    return digest.hexdigest()


def write_csv(path: Path, frame: pd.DataFrame) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    frame.to_csv(path, index=False)


def write_json(path: Path, payload: dict) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(
        json.dumps(
            payload,
            indent=2,
            ensure_ascii=False,
            default=str,
        )
        + "\n",
        encoding="utf-8",
    )


def write_text(path: Path, text: str) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(text, encoding="utf-8")


def atomic_write_text(path: Path, text: str) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)

    temporary_path = path.with_name(
        f".{path.name}.{uuid.uuid4().hex}.tmp"
    )

    temporary_path.write_text(text, encoding="utf-8")
    os.replace(temporary_path, path)


def append_marked_section(
    path: Path,
    marker: str,
    section_text: str,
) -> None:
    existing = (
        path.read_text(encoding="utf-8")
        if path.is_file()
        else ""
    )

    if marker in existing:
        return

    separator = "\n" if existing.endswith("\n") else "\n\n"

    atomic_write_text(
        path,
        existing
        + separator
        + section_text.strip()
        + "\n",
    )


def make_read_only(path: Path) -> None:
    if path.is_file():
        path.chmod(
            stat.S_IRUSR
            | stat.S_IRGRP
            | stat.S_IROTH
        )


def validate_required_columns(
    frame: pd.DataFrame,
    required_columns: set[str],
    frame_name: str,
) -> None:
    missing_columns = sorted(
        required_columns - set(frame.columns)
    )

    if missing_columns:
        raise AssertionError(
            f"{frame_name} is missing required columns:\n"
            + "\n".join(
                f"- {column}"
                for column in missing_columns
            )
        )


def percentage(
    numerator: float,
    denominator: float,
) -> float:
    if denominator == 0:
        return float("nan")

    return float(100.0 * numerator / denominator)


def values_equal_with_nan(
    left: float,
    right: float,
    tolerance: float = 1e-9,
) -> bool:
    if pd.isna(left) and pd.isna(right):
        return True

    if pd.isna(left) or pd.isna(right):
        return False

    return math.isclose(
        float(left),
        float(right),
        rel_tol=0.0,
        abs_tol=tolerance,
    )


def build_model_dataset(
    frame: pd.DataFrame,
    include_target: bool,
) -> pd.DataFrame:
    ordered_columns = (
        MODEL_DATASET_SUPPORT_COLUMNS
        + CORE_PREDICTORS
        + ([TARGET_COLUMN] if include_target else [])
    )

    duplicated_columns = [
        column
        for index, column in enumerate(ordered_columns)
        if column in ordered_columns[:index]
    ]

    if duplicated_columns:
        raise AssertionError(
            "Duplicate columns were requested for a model dataset:\n"
            + "\n".join(
                f"- {column}"
                for column in sorted(set(duplicated_columns))
            )
        )

    validate_required_columns(
        frame,
        set(ordered_columns),
        "Feature frame used to construct a model dataset",
    )

    return frame[ordered_columns].copy()


def historical_features_from_history(
    demand_history: np.ndarray,
    sequence_history: np.ndarray,
    current_sequence: float,
) -> dict[str, float]:
    demand_history = np.asarray(
        demand_history,
        dtype=float,
    )

    sequence_history = np.asarray(
        sequence_history,
        dtype=float,
    )

    result: dict[str, float] = {}

    for lag in LAG_OPERATING_DAYS:
        result[f"NormalDemandLag_{lag}"] = (
            float(demand_history[-lag])
            if len(demand_history) >= lag
            else float("nan")
        )

    for window in ROLLING_WINDOWS:
        window_values = demand_history[-window:]

        if len(window_values) == 0:
            mean_value = float("nan")
            median_value = float("nan")
            std_value = float("nan")
            sum_value = float("nan")
        else:
            mean_value = float(np.mean(window_values))
            median_value = float(np.median(window_values))
            std_value = float(np.std(window_values, ddof=0))
            sum_value = float(np.sum(window_values))

        result[
            f"PastNormalDemandRollingMean_{window}"
        ] = mean_value

        result[
            f"PastNormalDemandRollingMedian_{window}"
        ] = median_value

        result[
            f"PastNormalDemandRollingStd_{window}"
        ] = std_value

        result[
            f"PastNormalDemandRollingSum_{window}"
        ] = sum_value

    for window in ZERO_POSITIVE_WINDOWS:
        window_values = demand_history[-window:]

        if len(window_values) == 0:
            zero_rate = float("nan")
            positive_count = float("nan")
        else:
            zero_rate = float(
                np.mean(window_values == 0)
            )
            positive_count = float(
                np.sum(window_values > 0)
            )

        result[
            f"PastZeroNormalDemandRate_{window}"
        ] = zero_rate

        result[
            f"PastPositiveNormalDemandCount_{window}"
        ] = positive_count

    positive_indices = np.flatnonzero(
        demand_history > 0
    )

    if len(positive_indices) == 0:
        days_since_positive = float("nan")
    else:
        last_positive_index = int(
            positive_indices[-1]
        )
        last_positive_sequence = float(
            sequence_history[last_positive_index]
        )
        days_since_positive = float(
            current_sequence - last_positive_sequence
        )

    result[
        "OperatingDaysSincePreviousPositiveNormalDemand"
    ] = days_since_positive

    if len(demand_history) == 0:
        expanding_mean = float("nan")
        expanding_positive_rate = float("nan")
    else:
        expanding_mean = float(
            np.mean(demand_history)
        )
        expanding_positive_rate = float(
            np.mean(demand_history > 0)
        )

    result[
        "ExpandingPastMeanNormalDemand"
    ] = expanding_mean

    result[
        "ExpandingPastPositiveNormalDemandRate"
    ] = expanding_positive_rate

    return result


def build_core_feature_contract(
    frame: pd.DataFrame,
) -> pd.DataFrame:
    rows: list[dict] = []

    column_order = 0

    def add_row(
        column: str,
        contract_section: str,
        feature_family: str,
        source_column: str,
        direct_model_input_allowed: bool,
        is_historical_demand_feature: bool,
        is_identifier: bool,
        is_support_column: bool,
        is_forecast_target: bool,
        generation_rule: str,
        minimum_shift_rows: float | None = None,
        lag_days: float | None = None,
        window_days: float | None = None,
        minimum_periods: float | None = None,
        missing_value_policy: str = (
            "NO_MISSING_VALUE_ACTION_REQUIRED"
        ),
    ) -> None:
        nonlocal column_order

        missing_count = int(
            frame[column].isna().sum()
        )

        rows.append(
            {
                "Column": column,
                "ContractSection": contract_section,
                "FeatureFamily": feature_family,
                "SourceColumn": source_column,
                "GroupKey": (
                    "CanonicalProductID"
                    if is_historical_demand_feature
                    else ""
                ),
                "TimeIndex": (
                    "OperatingDaySequence"
                    if is_historical_demand_feature
                    else ""
                ),
                "LagOperatingDays": lag_days,
                "WindowOperatingDays": window_days,
                "MinimumShiftOperatingRows": minimum_shift_rows,
                "MinimumPeriods": minimum_periods,
                "GenerationRule": generation_rule,
                "DirectModelInputAllowed": (
                    direct_model_input_allowed
                ),
                "IsHistoricalDemandFeature": (
                    is_historical_demand_feature
                ),
                "IsIdentifier": is_identifier,
                "IsSupportColumn": is_support_column,
                "IsForecastTarget": is_forecast_target,
                "UsesCurrentRowTarget": False,
                "UsesFutureTarget": False,
                "DataType": str(frame[column].dtype),
                "MissingCount": missing_count,
                "MissingPercentage": percentage(
                    missing_count,
                    len(frame),
                ),
                "MissingValuePolicy": missing_value_policy,
                "PredictionTimeStatus": (
                    "AVAILABLE_OR_DERIVABLE_BEFORE_PREDICTION"
                    if not is_forecast_target
                    else "UNKNOWN_AT_PREDICTION_TIME"
                ),
                "ColumnOrder": column_order,
                "FrozenForCoreArchitecture": True,
                "FeatureEngineeringVersion": (
                    FEATURE_ENGINEERING_VERSION
                ),
            }
        )

        column_order += 1

    add_row(
        "Date",
        "BASE_AND_CURRENT_DATE",
        "IDENTIFIER_OR_TIME_KEY",
        "Date",
        False,
        False,
        True,
        False,
        False,
        "CURRENT_FORECAST_DATE_KEY",
    )

    add_row(
        "CanonicalProductID",
        "BASE_AND_CURRENT_DATE",
        "IDENTIFIER_OR_TIME_KEY",
        "CanonicalProductID",
        False,
        False,
        True,
        False,
        False,
        "CURRENT_PRODUCT_KEY",
    )

    add_row(
        "CanonicalProductName",
        "BASE_AND_CURRENT_DATE",
        "SUPPORT_AND_AUDIT_ONLY",
        "CanonicalProductName",
        False,
        False,
        False,
        True,
        False,
        "CURRENT_PRODUCT_NAME_FOR_REPORTING",
    )

    add_row(
        "ProductFirstObservedDate",
        "BASE_AND_CURRENT_DATE",
        "SUPPORT_AND_AUDIT_ONLY",
        "ProductFirstObservedDate",
        False,
        False,
        False,
        True,
        False,
        "PRODUCT_HISTORY_START_FOR_AUDIT",
    )

    for column in BASE_NUMERIC_PREDICTORS:
        feature_family = (
            "KNOWN_AHEAD_TIME_FEATURE"
            if column
            in {
                "ProductAgeOperatingDays",
                "OperatingDaySequence",
                "Year",
                "Month",
                "Quarter",
                "DayOfWeekNumber",
                "ISOYear",
                "ISOWeek",
                "DayOfYear",
                "IsWeekend",
                "DaysSincePreviousOperatingDate",
                "IsConsecutiveCalendarDay",
            }
            else "PRODUCT_METADATA_FEATURE"
        )

        add_row(
            column,
            "BASE_AND_CURRENT_DATE",
            feature_family,
            column,
            True,
            False,
            False,
            False,
            False,
            (
                "CURRENT_ROW_VALUE_ALLOWED_BECAUSE_"
                "KNOWN_OR_DERIVABLE_BEFORE_FORECAST"
            ),
            minimum_shift_rows=0.0,
        )

    for column in BASE_CATEGORICAL_PREDICTORS:
        add_row(
            column,
            "BASE_AND_CURRENT_DATE",
            "PRODUCT_METADATA_FEATURE",
            column,
            True,
            False,
            False,
            False,
            False,
            (
                "CURRENT_PRODUCT_METADATA_AVAILABLE_"
                "BEFORE_FORECAST"
            ),
            minimum_shift_rows=0.0,
            missing_value_policy=(
                "IMPUTE_MISSING_CATEGORY_IN_MODEL_PIPELINE"
            ),
        )

    add_row(
        TARGET_COLUMN,
        "BASE_AND_CURRENT_DATE",
        "TARGET_AND_HISTORICAL_FEATURE_SOURCE",
        TARGET_COLUMN,
        False,
        False,
        False,
        False,
        True,
        (
            "CURRENT_ROW_NORMALDEMAND_PROHIBITED_AS_FEATURE; "
            "USE_ONLY_AFTER_PRODUCT_GROUP_SHIFT_OF_AT_LEAST_1"
        ),
    )

    for lag in LAG_OPERATING_DAYS:
        add_row(
            f"NormalDemandLag_{lag}",
            "HISTORICAL_NORMAL_DEMAND",
            "DEMAND_LAG",
            TARGET_COLUMN,
            True,
            True,
            False,
            False,
            False,
            (
                "GROUP_BY_CANONICAL_PRODUCT_ID_SHIFT_"
                f"{lag}"
            ),
            minimum_shift_rows=float(lag),
            lag_days=float(lag),
            missing_value_policy=(
                "PRESERVE_EARLY_HISTORY_NA_UNTIL_MODEL_PIPELINE"
            ),
        )

    for window in ROLLING_WINDOWS:
        for statistic, family, rule in [
            (
                "Mean",
                "MEAN",
                "PAST_ONLY_ROLLING_MEAN",
            ),
            (
                "Median",
                "MEDIAN",
                "PAST_ONLY_ROLLING_MEDIAN",
            ),
            (
                "Std",
                "STANDARD_DEVIATION",
                "PAST_ONLY_ROLLING_POPULATION_STD_DDOF_0",
            ),
            (
                "Sum",
                "SUM",
                "PAST_ONLY_ROLLING_SUM",
            ),
        ]:
            add_row(
                (
                    "PastNormalDemandRolling"
                    f"{statistic}_{window}"
                ),
                "HISTORICAL_NORMAL_DEMAND",
                family,
                TARGET_COLUMN,
                True,
                True,
                False,
                False,
                False,
                rule,
                minimum_shift_rows=1.0,
                window_days=float(window),
                minimum_periods=1.0,
                missing_value_policy=(
                    "PRESERVE_EARLY_HISTORY_NA_UNTIL_MODEL_PIPELINE"
                ),
            )

    for window in ZERO_POSITIVE_WINDOWS:
        add_row(
            f"PastZeroNormalDemandRate_{window}",
            "HISTORICAL_NORMAL_DEMAND",
            "ROLLING_ZERO_DEMAND_RATE",
            TARGET_COLUMN,
            True,
            True,
            False,
            False,
            False,
            "SHIFT_1_COMPARE_ZERO_THEN_ROLLING_MEAN",
            minimum_shift_rows=1.0,
            window_days=float(window),
            minimum_periods=1.0,
            missing_value_policy=(
                "PRESERVE_EARLY_HISTORY_NA_UNTIL_MODEL_PIPELINE"
            ),
        )

    for window in ZERO_POSITIVE_WINDOWS:
        add_row(
            f"PastPositiveNormalDemandCount_{window}",
            "HISTORICAL_NORMAL_DEMAND",
            "ROLLING_POSITIVE_DEMAND_COUNT",
            TARGET_COLUMN,
            True,
            True,
            False,
            False,
            False,
            "SHIFT_1_COMPARE_POSITIVE_THEN_ROLLING_SUM",
            minimum_shift_rows=1.0,
            window_days=float(window),
            minimum_periods=1.0,
            missing_value_policy=(
                "PRESERVE_EARLY_HISTORY_NA_UNTIL_MODEL_PIPELINE"
            ),
        )

    add_row(
        "OperatingDaysSincePreviousPositiveNormalDemand",
        "HISTORICAL_NORMAL_DEMAND",
        "DAYS_SINCE_PREVIOUS_POSITIVE_DEMAND",
        TARGET_COLUMN,
        True,
        True,
        False,
        False,
        False,
        (
            "CURRENT_SEQUENCE_MINUS_MOST_RECENT_POSITIVE_"
            "SEQUENCE_BEFORE_CURRENT_ROW"
        ),
        minimum_shift_rows=1.0,
        minimum_periods=1.0,
        missing_value_policy=(
            "PRESERVE_NA_WHEN_NO_PREVIOUS_POSITIVE_DEMAND"
        ),
    )

    add_row(
        "ExpandingPastMeanNormalDemand",
        "HISTORICAL_NORMAL_DEMAND",
        "EXPANDING_PAST_MEAN",
        TARGET_COLUMN,
        True,
        True,
        False,
        False,
        False,
        "SHIFT_1_THEN_EXPANDING_MEAN",
        minimum_shift_rows=1.0,
        minimum_periods=1.0,
        missing_value_policy=(
            "PRESERVE_EARLY_HISTORY_NA_UNTIL_MODEL_PIPELINE"
        ),
    )

    add_row(
        "ExpandingPastPositiveNormalDemandRate",
        "HISTORICAL_NORMAL_DEMAND",
        "EXPANDING_PAST_POSITIVE_RATE",
        TARGET_COLUMN,
        True,
        True,
        False,
        False,
        False,
        "SHIFT_1_COMPARE_POSITIVE_THEN_EXPANDING_MEAN",
        minimum_shift_rows=1.0,
        minimum_periods=1.0,
        missing_value_policy=(
            "PRESERVE_EARLY_HISTORY_NA_UNTIL_MODEL_PIPELINE"
        ),
    )

    contract = pd.DataFrame(rows)

    expected_contract_rows = 58

    if len(contract) != expected_contract_rows:
        raise AssertionError(
            "Unexpected core feature-contract row count.\n"
            f"Expected: {expected_contract_rows}\n"
            f"Actual:   {len(contract)}"
        )

    return contract


# =============================================================================
# PREFLIGHT AND ND02 VERIFICATION
# =============================================================================

required_inputs = [
    ND02_PANEL_PATH,
    ND02_MANIFEST_PATH,
    ND02_VALIDATION_PATH,
    ND02_LEAKAGE_AUDIT_PATH,
    ND02_CHECKPOINT_PATH,
    AGENTS_PATH,
    CURRENT_HANDOFF_PATH,
    WORKFLOW_PATH,
    DECISIONS_PATH,
    METRICS_AND_RESULTS_PATH,
]

missing_inputs = [
    path
    for path in required_inputs
    if not path.is_file()
]

if missing_inputs:
    raise FileNotFoundError(
        "ND03 required input files are missing:\n"
        + "\n".join(
            f"- {path}"
            for path in missing_inputs
        )
    )

nd02_checkpoint_sha256_before = sha256_file(
    ND02_CHECKPOINT_PATH
)

if (
    nd02_checkpoint_sha256_before
    != EXPECTED_ND02_CHECKPOINT_SHA256
):
    raise AssertionError(
        "The ND02 checkpoint hash does not match the "
        "accepted checkpoint.\n"
        f"Expected: {EXPECTED_ND02_CHECKPOINT_SHA256}\n"
        f"Actual:   {nd02_checkpoint_sha256_before}\n"
        f"Path:     {ND02_CHECKPOINT_PATH}"
    )

with ND02_CHECKPOINT_PATH.open(
    "r",
    encoding="utf-8",
) as handle:
    nd02_checkpoint = json.load(handle)

if not bool(nd02_checkpoint.get("ReadyForND03", False)):
    raise AssertionError(
        "The ND02 checkpoint does not indicate readiness for ND03."
    )

if (
    nd02_checkpoint.get("Status")
    != "ND02_NORMAL_DEMAND_PANEL_CREATED_READY_FOR_ND03"
):
    raise AssertionError(
        "The ND02 checkpoint status is not the accepted status."
    )

expected_nd02_manifest_sha256 = (
    nd02_checkpoint
    .get("Control", {})
    .get("ManifestSHA256")
)

if not expected_nd02_manifest_sha256:
    raise AssertionError(
        "The ND02 checkpoint does not contain a manifest hash."
    )

actual_nd02_manifest_sha256 = sha256_file(
    ND02_MANIFEST_PATH
)

if (
    actual_nd02_manifest_sha256
    != expected_nd02_manifest_sha256
):
    raise AssertionError(
        "The ND02 manifest hash does not match its checkpoint.\n"
        f"Expected: {expected_nd02_manifest_sha256}\n"
        f"Actual:   {actual_nd02_manifest_sha256}"
    )

nd02_manifest = pd.read_csv(
    ND02_MANIFEST_PATH,
    low_memory=False,
)

panel_manifest_rows = nd02_manifest.loc[
    nd02_manifest["RelativePath"].astype(str)
    == "01_data/ND02_normal_demand_daily_panel.csv"
]

if len(panel_manifest_rows) != 1:
    raise AssertionError(
        "The ND02 panel was not found exactly once in the ND02 manifest."
    )

expected_nd02_panel_sha256 = str(
    panel_manifest_rows.iloc[0]["SHA256"]
)

actual_nd02_panel_sha256 = sha256_file(
    ND02_PANEL_PATH
)

if actual_nd02_panel_sha256 != expected_nd02_panel_sha256:
    raise AssertionError(
        "The ND02 panel hash does not match the ND02 manifest.\n"
        f"Expected: {expected_nd02_panel_sha256}\n"
        f"Actual:   {actual_nd02_panel_sha256}"
    )

nd02_validation = pd.read_csv(
    ND02_VALIDATION_PATH,
    low_memory=False,
)

nd02_leakage_audit = pd.read_csv(
    ND02_LEAKAGE_AUDIT_PATH,
    low_memory=False,
)

if "Passed" not in nd02_validation.columns:
    raise AssertionError(
        "The ND02 validation file has no Passed column."
    )

if "Passed" not in nd02_leakage_audit.columns:
    raise AssertionError(
        "The ND02 leakage audit has no Passed column."
    )

if not nd02_validation["Passed"].astype(bool).all():
    raise AssertionError(
        "The ND02 validation file contains failed checks."
    )

if not nd02_leakage_audit["Passed"].astype(bool).all():
    raise AssertionError(
        "The ND02 leakage audit contains failed checks."
    )

if ND03_ROOT.exists():
    if not ALLOW_OVERWRITE:
        raise FileExistsError(
            "The ND03 output folder already exists. "
            "No files were changed:\n"
            f"{ND03_ROOT}\n\n"
            "Review the existing ND03 checkpoint rather than "
            "silently rebuilding it."
        )

    shutil.rmtree(ND03_ROOT)

for path in [
    TOP_LEVEL_CHECKPOINT_PATH,
    TOP_LEVEL_CHECKPOINT_SHA_PATH,
]:
    if path.exists():
        if not ALLOW_OVERWRITE:
            raise FileExistsError(
                "An ND03 top-level checkpoint already exists. "
                "No files were changed:\n"
                f"{path}"
            )

        path.unlink()

protected_input_hashes_before = {
    str(path): sha256_file(path)
    for path in required_inputs
}

STAGING_ROOT = (
    ND03_ROOT.parent
    / f".ND03_staging_{uuid.uuid4().hex}"
)

STAGING_ROOT.mkdir(parents=True, exist_ok=False)


# =============================================================================
# LOAD PANEL AND GENERATE FEATURES
# =============================================================================

try:
    panel = pd.read_csv(
        ND02_PANEL_PATH,
        low_memory=False,
    )

    required_panel_columns = {
        "Date",
        "CanonicalProductID",
        "CanonicalProductName",
        "ProductFirstObservedDate",
        "ProductAgeOperatingDays",
        "SourcePLUCount",
        "SourceGroupCodes",
        "SourceGroupNames",
        "BeverageSeries",
        "BeverageType",
        "SupplierLabelsObserved",
        "TierProductFamily",
        "NominalPriceTier",
        "MenuGeneration",
        "IsMultiPLUCanonicalProduct",
        "OperatingDaySequence",
        "Year",
        "Month",
        "Quarter",
        "DayOfWeekNumber",
        "ISOYear",
        "ISOWeek",
        "DayOfYear",
        "IsWeekend",
        "DaysSincePreviousOperatingDate",
        "IsConsecutiveCalendarDay",
        TARGET_COLUMN,
        "PriorOperatingDayCount",
        "PriorCumulativeNormalDemand",
        "PriorPositiveNormalDemandDays",
        "PriorZeroNormalDemandDays",
        "PriorMeanNormalDemand",
        "PriorPositiveDemandRate",
        "PriorZeroDemandRate",
        "OperatingDaysSincePriorPositiveDemand",
        "HasSufficientHistory20",
        "ColdStartFlag",
        "ZeroPriorNormalDemandFlag",
        "PriorDemandRank",
        "PriorDemandSharePercentage",
        "PriorCumulativeDemandShareBeforePercentage",
        "PriorCumulativeDemandSharePercentage",
        "PriorDemandVolumeSegment",
        "InPrior80DemandScope",
        "InPrior90DemandScope",
        "InPrior95DemandScope",
        "EligibleForMainModel80",
        "EligibleForMainModel90",
        "EligibleForMainModel95",
        "EligibleForMainModel",
        "ForecastRoute",
        "ForecastRouteReason",
        "FallbackMethodStatus",
        "IsPreMarchDevelopmentPeriod",
        "IsOpenedMarchDiagnosticPeriod",
        "EligibleForMethodSelection",
        "EligibleForNewUnbiasedFinalEvaluation",
    }

    validate_required_columns(
        panel,
        required_panel_columns,
        "ND02 normal-demand panel",
    )

    forbidden_input_columns = {
        "BulkDemand",
        "TotalDemand",
        "IsObservedProductDate",
        "IsZeroDemandRow",
        "DemandRecordSource",
    }

    forbidden_input_columns_present = sorted(
        forbidden_input_columns
        & set(panel.columns)
    )

    if forbidden_input_columns_present:
        raise AssertionError(
            "Forbidden columns were found in the ND02 modelling panel:\n"
            + "\n".join(
                f"- {column}"
                for column in forbidden_input_columns_present
            )
        )

    panel["Date"] = pd.to_datetime(
        panel["Date"],
        errors="raise",
    )

    panel["ProductFirstObservedDate"] = pd.to_datetime(
        panel["ProductFirstObservedDate"],
        errors="coerce",
    )

    panel[TARGET_COLUMN] = pd.to_numeric(
        panel[TARGET_COLUMN],
        errors="raise",
    ).astype(float)

    for boolean_column in [
        "HasSufficientHistory20",
        "ColdStartFlag",
        "ZeroPriorNormalDemandFlag",
        "InPrior80DemandScope",
        "InPrior90DemandScope",
        "InPrior95DemandScope",
        "EligibleForMainModel80",
        "EligibleForMainModel90",
        "EligibleForMainModel95",
        "EligibleForMainModel",
        "IsPreMarchDevelopmentPeriod",
        "IsOpenedMarchDiagnosticPeriod",
        "EligibleForMethodSelection",
        "EligibleForNewUnbiasedFinalEvaluation",
    ]:
        panel[boolean_column] = (
            panel[boolean_column]
            .astype(bool)
        )

    panel_rows = int(len(panel))
    panel_products = int(
        panel["CanonicalProductID"].nunique()
    )
    panel_dates = int(panel["Date"].nunique())
    panel_normal_total = float(
        panel[TARGET_COLUMN].sum()
    )

    profile_checks = [
        (
            "Rows",
            panel_rows,
            EXPECTED_ROWS,
        ),
        (
            "Products",
            panel_products,
            EXPECTED_PRODUCTS,
        ),
        (
            "OperatingDates",
            panel_dates,
            EXPECTED_OPERATING_DATES,
        ),
        (
            "NormalDemandTotal",
            panel_normal_total,
            EXPECTED_NORMAL_DEMAND_TOTAL,
        ),
    ]

    profile_failures = []

    for label, actual, expected in profile_checks:
        if not math.isclose(
            float(actual),
            float(expected),
            rel_tol=0.0,
            abs_tol=1e-9,
        ):
            profile_failures.append(
                f"{label}: expected {expected}, actual {actual}"
            )

    if profile_failures:
        raise AssertionError(
            "The ND02 panel profile differs from the accepted profile:\n"
            + "\n".join(
                f"- {failure}"
                for failure in profile_failures
            )
        )

    panel_duplicate_keys = int(
        panel.duplicated(KEY_COLUMNS).sum()
    )

    if panel_duplicate_keys != 0:
        raise AssertionError(
            "Duplicate Date + CanonicalProductID rows were found."
        )

    feature_frame = panel.sort_values(
        [
            "CanonicalProductID",
            "OperatingDaySequence",
            "Date",
        ],
        kind="mergesort",
    ).reset_index(drop=True)

    product_group = feature_frame.groupby(
        "CanonicalProductID",
        sort=False,
    )

    # Demand lags.
    for lag in LAG_OPERATING_DAYS:
        feature_frame[
            f"NormalDemandLag_{lag}"
        ] = product_group[TARGET_COLUMN].shift(lag)

    shifted_normal_demand = product_group[
        TARGET_COLUMN
    ].shift(1)

    shifted_group = shifted_normal_demand.groupby(
        feature_frame["CanonicalProductID"],
        sort=False,
    )

    # Past-only rolling statistics with min_periods=1.
    for window in ROLLING_WINDOWS:
        feature_frame[
            f"PastNormalDemandRollingMean_{window}"
        ] = shifted_group.transform(
            lambda values: values.rolling(
                window=window,
                min_periods=1,
            ).mean()
        )

        feature_frame[
            f"PastNormalDemandRollingMedian_{window}"
        ] = shifted_group.transform(
            lambda values: values.rolling(
                window=window,
                min_periods=1,
            ).median()
        )

        feature_frame[
            f"PastNormalDemandRollingStd_{window}"
        ] = shifted_group.transform(
            lambda values: values.rolling(
                window=window,
                min_periods=1,
            ).std(ddof=0)
        )

        feature_frame[
            f"PastNormalDemandRollingSum_{window}"
        ] = shifted_group.transform(
            lambda values: values.rolling(
                window=window,
                min_periods=1,
            ).sum()
        )

    shifted_zero_indicator = pd.Series(
        np.where(
            shifted_normal_demand.notna(),
            (
                shifted_normal_demand == 0
            ).astype(float),
            np.nan,
        ),
        index=feature_frame.index,
        dtype=float,
    )

    shifted_positive_indicator = pd.Series(
        np.where(
            shifted_normal_demand.notna(),
            (
                shifted_normal_demand > 0
            ).astype(float),
            np.nan,
        ),
        index=feature_frame.index,
        dtype=float,
    )

    shifted_zero_group = shifted_zero_indicator.groupby(
        feature_frame["CanonicalProductID"],
        sort=False,
    )

    shifted_positive_group = shifted_positive_indicator.groupby(
        feature_frame["CanonicalProductID"],
        sort=False,
    )

    for window in ZERO_POSITIVE_WINDOWS:
        feature_frame[
            f"PastZeroNormalDemandRate_{window}"
        ] = shifted_zero_group.transform(
            lambda values: values.rolling(
                window=window,
                min_periods=1,
            ).mean()
        )

        feature_frame[
            f"PastPositiveNormalDemandCount_{window}"
        ] = shifted_positive_group.transform(
            lambda values: values.rolling(
                window=window,
                min_periods=1,
            ).sum()
        )

    positive_sequence = feature_frame[
        "OperatingDaySequence"
    ].where(feature_frame[TARGET_COLUMN] > 0)

    previous_positive_sequence = positive_sequence.groupby(
        feature_frame["CanonicalProductID"],
        sort=False,
    ).transform(
        lambda values: values.shift(1).ffill()
    )

    feature_frame[
        "OperatingDaysSincePreviousPositiveNormalDemand"
    ] = (
        feature_frame["OperatingDaySequence"]
        - previous_positive_sequence
    )

    feature_frame[
        "ExpandingPastMeanNormalDemand"
    ] = shifted_group.transform(
        lambda values: values.expanding(
            min_periods=1
        ).mean()
    )

    feature_frame[
        "ExpandingPastPositiveNormalDemandRate"
    ] = shifted_positive_group.transform(
        lambda values: values.expanding(
            min_periods=1
        ).mean()
    )

    validate_required_columns(
        feature_frame,
        set(HISTORICAL_DEMAND_PREDICTORS),
        "Generated historical feature frame",
    )

    if len(HISTORICAL_DEMAND_PREDICTORS) != 31:
        raise AssertionError(
            "The historical feature architecture must contain 31 predictors."
        )

    if len(CORE_PREDICTORS) != 53:
        raise AssertionError(
            "The core model architecture must contain 53 predictors."
        )

    feature_frame[
        "HistoricalFeatureAvailableCount"
    ] = feature_frame[
        HISTORICAL_DEMAND_PREDICTORS
    ].notna().sum(axis=1)

    feature_frame[
        "HistoricalFeatureMissingCount"
    ] = (
        len(HISTORICAL_DEMAND_PREDICTORS)
        - feature_frame[
            "HistoricalFeatureAvailableCount"
        ]
    )

    feature_frame[
        "AllHistoricalFeaturesAvailable"
    ] = (
        feature_frame[
            "HistoricalFeatureAvailableCount"
        ]
        == len(HISTORICAL_DEMAND_PREDICTORS)
    )

    feature_frame[
        "CorePredictorAvailableCount"
    ] = feature_frame[
        CORE_PREDICTORS
    ].notna().sum(axis=1)

    feature_frame[
        "CorePredictorMissingCount"
    ] = (
        len(CORE_PREDICTORS)
        - feature_frame[
            "CorePredictorAvailableCount"
        ]
    )

    feature_frame[
        "FeatureEngineeringVersion"
    ] = FEATURE_ENGINEERING_VERSION

    # Restore report-friendly ordering.
    feature_frame = feature_frame.sort_values(
        ["Date", "CanonicalProductID"],
        kind="mergesort",
    ).reset_index(drop=True)

    # =========================================================================
    # INDEPENDENT FEATURE CONSISTENCY CHECKS
    # =========================================================================

    prior_mean_max_difference = float(
        (
            feature_frame[
                "ExpandingPastMeanNormalDemand"
            ]
            - feature_frame[
                "PriorMeanNormalDemand"
            ]
        )
        .abs()
        .max(skipna=True)
    )

    prior_positive_rate_max_difference = float(
        (
            feature_frame[
                "ExpandingPastPositiveNormalDemandRate"
            ]
            - feature_frame[
                "PriorPositiveDemandRate"
            ]
        )
        .abs()
        .max(skipna=True)
    )

    days_since_positive_max_difference = float(
        (
            feature_frame[
                "OperatingDaysSincePreviousPositiveNormalDemand"
            ]
            - feature_frame[
                "OperatingDaysSincePriorPositiveDemand"
            ]
        )
        .abs()
        .max(skipna=True)
    )

    main_history_missing_rows = int(
        (
            feature_frame["EligibleForMainModel"]
            & ~feature_frame[
                "AllHistoricalFeaturesAvailable"
            ]
        ).sum()
    )

    if main_history_missing_rows != 0:
        raise AssertionError(
            "Some main-model rows do not have all 31 historical features."
        )

    # =========================================================================
    # BUILD MODEL-READY DATASETS
    # =========================================================================

    pre_march_mask = (
        feature_frame["Date"]
        < DEVELOPMENT_END_EXCLUSIVE
    )

    march_mask = feature_frame["Date"].between(
        OPENED_MARCH_START,
        OPENED_MARCH_END,
        inclusive="both",
    )

    main_model_mask = feature_frame[
        "EligibleForMainModel"
    ]

    pre_march_all_routes = build_model_dataset(
        feature_frame.loc[pre_march_mask],
        include_target=True,
    )

    pre_march_main_model = build_model_dataset(
        feature_frame.loc[
            pre_march_mask & main_model_mask
        ],
        include_target=True,
    )

    pre_march_fallback = build_model_dataset(
        feature_frame.loc[
            pre_march_mask & ~main_model_mask
        ],
        include_target=True,
    )

    march_scoring_features = build_model_dataset(
        feature_frame.loc[march_mask],
        include_target=False,
    )

    march_target_vault = feature_frame.loc[
        march_mask,
        [
            "Date",
            "CanonicalProductID",
            "CanonicalProductName",
            "ForecastRoute",
            "EligibleForMainModel",
            "PriorDemandVolumeSegment",
            TARGET_COLUMN,
        ],
    ].copy()

    march_main_scoring_features = build_model_dataset(
        feature_frame.loc[
            march_mask & main_model_mask
        ],
        include_target=False,
    )

    march_main_target_vault = feature_frame.loc[
        march_mask & main_model_mask,
        [
            "Date",
            "CanonicalProductID",
            "CanonicalProductName",
            "ForecastRoute",
            "EligibleForMainModel",
            "PriorDemandVolumeSegment",
            TARGET_COLUMN,
        ],
    ].copy()

    future_fit_main_model = build_model_dataset(
        feature_frame.loc[main_model_mask],
        include_target=True,
    )

    # Explicit usage labels.
    pre_march_all_routes.insert(
        0,
        "DatasetPurpose",
        "CHRONOLOGICAL_MODEL_DEVELOPMENT_ALL_ROUTES",
    )

    pre_march_main_model.insert(
        0,
        "DatasetPurpose",
        "CHRONOLOGICAL_MAIN_MODEL_DEVELOPMENT",
    )

    pre_march_fallback.insert(
        0,
        "DatasetPurpose",
        "CHRONOLOGICAL_FALLBACK_DEVELOPMENT",
    )

    march_scoring_features.insert(
        0,
        "DatasetPurpose",
        "OPENED_MARCH_DIAGNOSTIC_FEATURES_ONLY",
    )

    march_target_vault.insert(
        0,
        "DatasetPurpose",
        "OPENED_MARCH_DIAGNOSTIC_TARGET_VAULT",
    )

    march_main_scoring_features.insert(
        0,
        "DatasetPurpose",
        "OPENED_MARCH_MAIN_MODEL_DIAGNOSTIC_FEATURES_ONLY",
    )

    march_main_target_vault.insert(
        0,
        "DatasetPurpose",
        "OPENED_MARCH_MAIN_MODEL_DIAGNOSTIC_TARGET_VAULT",
    )

    future_fit_main_model.insert(
        0,
        "DatasetPurpose",
        "FUTURE_FORECAST_FIT_AFTER_METHOD_LOCK_ONLY",
    )

    dataset_length_checks = [
        (
            "Pre-March all routes",
            len(pre_march_all_routes),
            EXPECTED_PRE_MARCH_ROWS,
        ),
        (
            "March all routes",
            len(march_scoring_features),
            EXPECTED_MARCH_ROWS,
        ),
        (
            "Pre-March main model",
            len(pre_march_main_model),
            EXPECTED_PRE_MARCH_MAIN_ROWS,
        ),
        (
            "March main model",
            len(march_main_scoring_features),
            EXPECTED_MARCH_MAIN_ROWS,
        ),
        (
            "Future-fit main model",
            len(future_fit_main_model),
            EXPECTED_FUTURE_FIT_MAIN_ROWS,
        ),
    ]

    dataset_length_failures = []

    for label, actual, expected in dataset_length_checks:
        if actual != expected:
            dataset_length_failures.append(
                f"{label}: expected {expected}, actual {actual}"
            )

    if dataset_length_failures:
        raise AssertionError(
            "Unexpected ND03 dataset row counts:\n"
            + "\n".join(
                f"- {failure}"
                for failure in dataset_length_failures
            )
        )

    if TARGET_COLUMN in march_scoring_features.columns:
        raise AssertionError(
            "NormalDemand appeared in March scoring features."
        )

    if TARGET_COLUMN in march_main_scoring_features.columns:
        raise AssertionError(
            "NormalDemand appeared in March main-model scoring features."
        )

    target_columns_in_vault = [
        column
        for column in march_target_vault.columns
        if column in CORE_PREDICTORS
    ]

    if target_columns_in_vault:
        raise AssertionError(
            "Core predictors appeared in the March target vault:\n"
            + "\n".join(
                f"- {column}"
                for column in target_columns_in_vault
            )
        )

    # =========================================================================
    # RECURSIVE INFERENCE STATE
    # =========================================================================

    state_records = []
    final_source_date = feature_frame["Date"].max()

    metadata_state_columns = [
        "CanonicalProductName",
        "ProductFirstObservedDate",
        "SourcePLUCount",
        "SourceGroupCodes",
        "SourceGroupNames",
        "BeverageSeries",
        "BeverageType",
        "SupplierLabelsObserved",
        "TierProductFamily",
        "NominalPriceTier",
        "MenuGeneration",
        "IsMultiPLUCanonicalProduct",
    ]

    for product_id, product_frame in feature_frame.groupby(
        "CanonicalProductID",
        sort=True,
    ):
        product_frame = product_frame.sort_values(
            ["OperatingDaySequence", "Date"],
            kind="mergesort",
        )

        last_row = product_frame.iloc[-1]

        demand_values = product_frame[
            TARGET_COLUMN
        ].to_numpy(dtype=float)

        sequence_values = product_frame[
            "OperatingDaySequence"
        ].to_numpy(dtype=float)

        positive_indices = np.flatnonzero(
            demand_values > 0
        )

        last_positive_sequence = (
            float(sequence_values[positive_indices[-1]])
            if len(positive_indices) > 0
            else float("nan")
        )

        state_record = {
            "CanonicalProductID": str(product_id),
            "StateAsOfDate": last_row["Date"],
            "LastObservedDate": last_row["Date"],
            "LastObservedOperatingDaySequence": int(
                last_row["OperatingDaySequence"]
            ),
            "ObservedProductDayCount": int(
                len(product_frame)
            ),
            "CumulativeNormalDemandIncludingLatest": float(
                np.sum(demand_values)
            ),
            "PositiveNormalDemandDaysIncludingLatest": int(
                np.sum(demand_values > 0)
            ),
            "ZeroNormalDemandDaysIncludingLatest": int(
                np.sum(demand_values == 0)
            ),
            "LastPositiveOperatingDaySequenceIncludingLatest": (
                last_positive_sequence
            ),
            "IsObservedOnFinalSourceDate": bool(
                last_row["Date"] == final_source_date
            ),
            "FutureCatalogStatus": (
                "ACTIVE_AT_SOURCE_END"
                if last_row["Date"] == final_source_date
                else (
                    "NOT_ACTIVE_AT_SOURCE_END_REQUIRES_"
                    "CATALOG_CONFIRMATION"
                )
            ),
            "LatestForecastRoute": last_row[
                "ForecastRoute"
            ],
            "LatestPriorDemandVolumeSegment": last_row[
                "PriorDemandVolumeSegment"
            ],
            "LatestEligibleForMainModel": bool(
                last_row["EligibleForMainModel"]
            ),
            "HistoryValuesStored": int(
                min(
                    len(demand_values),
                    MAXIMUM_REQUIRED_HISTORY,
                )
            ),
            "FeatureEngineeringVersion": (
                FEATURE_ENGINEERING_VERSION
            ),
        }

        for metadata_column in metadata_state_columns:
            state_record[metadata_column] = last_row[
                metadata_column
            ]

        for lag in range(
            1,
            MAXIMUM_REQUIRED_HISTORY + 1,
        ):
            state_record[
                f"StateNormalDemandLag_{lag}"
            ] = (
                float(demand_values[-lag])
                if len(demand_values) >= lag
                else float("nan")
            )

        state_records.append(state_record)

    recursive_state = pd.DataFrame(state_records)

    if len(recursive_state) != EXPECTED_PRODUCTS:
        raise AssertionError(
            "The recursive state must contain one row per product."
        )

    # =========================================================================
    # RECURSIVE REPLAY AUDIT
    # =========================================================================

    replay_difference_records: list[dict] = []

    replay_differences: dict[str, list[float]] = {
        feature: []
        for feature in HISTORICAL_DEMAND_PREDICTORS
    }

    replay_compared_products: dict[str, int] = {
        feature: 0
        for feature in HISTORICAL_DEMAND_PREDICTORS
    }

    for _, product_frame in feature_frame.groupby(
        "CanonicalProductID",
        sort=False,
    ):
        product_frame = product_frame.sort_values(
            ["OperatingDaySequence", "Date"],
            kind="mergesort",
        )

        if len(product_frame) == 0:
            continue

        last_row = product_frame.iloc[-1]
        prior_rows = product_frame.iloc[:-1]

        reconstructed = historical_features_from_history(
            prior_rows[TARGET_COLUMN].to_numpy(
                dtype=float
            ),
            prior_rows["OperatingDaySequence"].to_numpy(
                dtype=float
            ),
            float(last_row["OperatingDaySequence"]),
        )

        for feature in HISTORICAL_DEMAND_PREDICTORS:
            expected_value = reconstructed[feature]
            actual_value = last_row[feature]

            if pd.isna(expected_value) and pd.isna(actual_value):
                difference = 0.0
            elif pd.isna(expected_value) or pd.isna(actual_value):
                difference = float("inf")
            else:
                difference = abs(
                    float(expected_value)
                    - float(actual_value)
                )

            replay_differences[feature].append(difference)
            replay_compared_products[feature] += 1

    for feature in HISTORICAL_DEMAND_PREDICTORS:
        differences = replay_differences[feature]
        maximum_difference = (
            float(max(differences))
            if differences
            else float("nan")
        )

        replay_difference_records.append(
            {
                "Feature": feature,
                "ProductsCompared": (
                    replay_compared_products[feature]
                ),
                "MaximumAbsoluteDifference": (
                    maximum_difference
                ),
                "Passed": bool(
                    np.isfinite(maximum_difference)
                    and maximum_difference <= 1e-9
                ),
                "ReplayDefinition": (
                    "Reconstructed final historical feature from "
                    "that product's earlier rows only"
                ),
            }
        )

    recursive_replay_audit = pd.DataFrame(
        replay_difference_records
    )

    if not recursive_replay_audit["Passed"].all():
        raise AssertionError(
            "The recursive feature replay audit failed:\n"
            + recursive_replay_audit.loc[
                ~recursive_replay_audit["Passed"]
            ].to_string(index=False)
        )

    # =========================================================================
    # CORE FEATURE CONTRACT AND PREDICTOR LISTS
    # =========================================================================

    core_feature_contract = build_core_feature_contract(
        feature_frame
    )

    contract_direct_predictors = (
        core_feature_contract.loc[
            core_feature_contract[
                "DirectModelInputAllowed"
            ],
            "Column",
        ].tolist()
    )

    if contract_direct_predictors != CORE_PREDICTORS:
        raise AssertionError(
            "The feature-contract direct predictor order does not "
            "match the authoritative core predictor list."
        )

    core_predictor_list = pd.DataFrame(
        [
            {
                "PredictorOrder": index,
                "Predictor": predictor,
                "PredictorType": (
                    "NUMERIC"
                    if predictor
                    in CORE_NUMERIC_PREDICTORS
                    else "CATEGORICAL"
                ),
                "FeatureFamily": (
                    "HISTORICAL_NORMAL_DEMAND"
                    if predictor
                    in HISTORICAL_DEMAND_PREDICTORS
                    else (
                        "PRODUCT_METADATA"
                        if predictor
                        in BASE_CATEGORICAL_PREDICTORS
                        or predictor
                        in {
                            "SourcePLUCount",
                            "IsMultiPLUCanonicalProduct",
                        }
                        else "KNOWN_AHEAD_TIME"
                    )
                ),
                "DirectModelInputAllowed": True,
                "CoreArchitecture": True,
                "FeatureEngineeringVersion": (
                    FEATURE_ENGINEERING_VERSION
                ),
            }
            for index, predictor in enumerate(
                CORE_PREDICTORS,
                start=1,
            )
        ]
    )

    optional_predictor_list = pd.DataFrame(
        [
            {
                "Predictor": predictor,
                "Status": "OPTIONAL_CHALLENGER_ONLY",
                "AllowedForCoreArchitecture": False,
                "LeakageSafeAtPredictionTime": True,
                "Reason": (
                    "Available before prediction but excluded from "
                    "the initial 53-predictor architecture to retain "
                    "comparability with the original daily model"
                ),
            }
            for predictor in OPTIONAL_CANDIDATE_PREDICTORS
        ]
    )

    # =========================================================================
    # MISSINGNESS AUDIT
    # =========================================================================

    subset_definitions = {
        "ALL_FEATURE_ROWS": feature_frame,
        "PRE_MARCH_ALL_ROUTES": (
            feature_frame.loc[pre_march_mask]
        ),
        "PRE_MARCH_MAIN_MODEL": (
            feature_frame.loc[
                pre_march_mask & main_model_mask
            ]
        ),
        "PRE_MARCH_FALLBACK": (
            feature_frame.loc[
                pre_march_mask & ~main_model_mask
            ]
        ),
        "OPENED_MARCH_ALL_ROUTES": (
            feature_frame.loc[march_mask]
        ),
        "OPENED_MARCH_MAIN_MODEL": (
            feature_frame.loc[
                march_mask & main_model_mask
            ]
        ),
        "FUTURE_FIT_MAIN_MODEL": (
            feature_frame.loc[main_model_mask]
        ),
    }

    missingness_records = []

    for subset_name, subset_frame in subset_definitions.items():
        for predictor in CORE_PREDICTORS:
            missing_count = int(
                subset_frame[predictor].isna().sum()
            )

            missingness_records.append(
                {
                    "Subset": subset_name,
                    "Rows": int(len(subset_frame)),
                    "Predictor": predictor,
                    "PredictorType": (
                        "NUMERIC"
                        if predictor
                        in CORE_NUMERIC_PREDICTORS
                        else "CATEGORICAL"
                    ),
                    "HistoricalDemandFeature": (
                        predictor
                        in HISTORICAL_DEMAND_PREDICTORS
                    ),
                    "MissingCount": missing_count,
                    "MissingPercentage": percentage(
                        missing_count,
                        len(subset_frame),
                    ),
                    "MissingValuePolicy": (
                        "EARLY_HISTORY_NA_EXPECTED"
                        if predictor
                        in HISTORICAL_DEMAND_PREDICTORS
                        else "MODEL_PIPELINE_IMPUTATION_OR_CATEGORY"
                    ),
                }
            )

    predictor_missingness = pd.DataFrame(
        missingness_records
    )

    main_historical_missing = predictor_missingness.loc[
        (
            predictor_missingness["Subset"]
            == "PRE_MARCH_MAIN_MODEL"
        )
        & predictor_missingness[
            "HistoricalDemandFeature"
        ]
    ]

    if int(main_historical_missing["MissingCount"].sum()) != 0:
        raise AssertionError(
            "Pre-March main-model rows contain missing historical features."
        )

    # =========================================================================
    # ROUTE FEATURE READINESS
    # =========================================================================

    route_readiness_records = []

    for route, route_frame in feature_frame.groupby(
        "ForecastRoute",
        sort=False,
    ):
        route_readiness_records.append(
            {
                "ForecastRoute": route,
                "Rows": int(len(route_frame)),
                "ProductsAppearing": int(
                    route_frame[
                        "CanonicalProductID"
                    ].nunique()
                ),
                "NormalDemand": float(
                    route_frame[TARGET_COLUMN].sum()
                ),
                "RowsWithAllHistoricalFeatures": int(
                    route_frame[
                        "AllHistoricalFeaturesAvailable"
                    ].sum()
                ),
                "RowsMissingAtLeastOneHistoricalFeature": int(
                    (
                        ~route_frame[
                            "AllHistoricalFeaturesAvailable"
                        ]
                    ).sum()
                ),
                "AllHistoricalFeatureReadyPercentage": percentage(
                    float(
                        route_frame[
                            "AllHistoricalFeaturesAvailable"
                        ].sum()
                    ),
                    len(route_frame),
                ),
                "MinimumHistoricalFeatureCount": int(
                    route_frame[
                        "HistoricalFeatureAvailableCount"
                    ].min()
                ),
                "MaximumHistoricalFeatureCount": int(
                    route_frame[
                        "HistoricalFeatureAvailableCount"
                    ].max()
                ),
            }
        )

    route_feature_readiness = pd.DataFrame(
        route_readiness_records
    )

    # =========================================================================
    # RECURSIVE COMPATIBILITY AUDIT
    # =========================================================================

    recursive_compatibility_records = []

    for predictor in CORE_PREDICTORS:
        if predictor in HISTORICAL_DEMAND_PREDICTORS:
            generation_class = "STATEFUL_PRIOR_NORMAL_DEMAND"
            required_state = (
                "Last 20 normal-demand values, cumulative demand, "
                "positive-day count, last positive operating sequence, "
                "and current operating-day sequence"
            )
            recursive_supported = True
        elif predictor in BASE_CATEGORICAL_PREDICTORS:
            generation_class = "PRODUCT_METADATA"
            required_state = (
                "Current product metadata or metadata supplied for a new product"
            )
            recursive_supported = True
        else:
            generation_class = "KNOWN_AHEAD_DATE_OR_PRODUCT_STATE"
            required_state = (
                "Requested date, future operating calendar, product first-observed "
                "date, and prior operating-day sequence"
            )
            recursive_supported = True

        recursive_compatibility_records.append(
            {
                "Predictor": predictor,
                "GenerationClass": generation_class,
                "AvailableForNextOperatingDay": True,
                "AvailableForRecursiveFutureDate": (
                    recursive_supported
                ),
                "UsesCurrentRequestedDateTarget": False,
                "UsesFutureActualTarget": False,
                "RequiredStateOrInput": required_state,
                "ImplementationStage": (
                    "ND09_FUTURE_INFERENCE_PIPELINE"
                ),
                "Passed": recursive_supported,
            }
        )

    recursive_compatibility_audit = pd.DataFrame(
        recursive_compatibility_records
    )

    if len(recursive_compatibility_audit) != 53:
        raise AssertionError(
            "Recursive compatibility audit must cover all 53 predictors."
        )

    if not recursive_compatibility_audit["Passed"].all():
        raise AssertionError(
            "Some core predictors are not recursively generatable."
        )

    # =========================================================================
    # SPLIT SUMMARY AND INVENTORY
    # =========================================================================

    dataset_objects = {
        "ND03_normal_demand_feature_panel.csv": feature_frame,
        "ND03_pre_march_all_routes_development_dataset.csv": (
            pre_march_all_routes
        ),
        "ND03_pre_march_main_model_development_dataset.csv": (
            pre_march_main_model
        ),
        "ND03_pre_march_fallback_development_dataset.csv": (
            pre_march_fallback
        ),
        "ND03_opened_march_diagnostic_scoring_features.csv": (
            march_scoring_features
        ),
        "ND03_opened_march_diagnostic_target_vault.csv": (
            march_target_vault
        ),
        "ND03_opened_march_main_model_scoring_features.csv": (
            march_main_scoring_features
        ),
        "ND03_opened_march_main_model_target_vault.csv": (
            march_main_target_vault
        ),
        "ND03_future_forecast_fit_main_model_dataset.csv": (
            future_fit_main_model
        ),
        "ND03_recursive_inference_state.csv": (
            recursive_state
        ),
    }

    dataset_inventory_records = []

    for filename, dataset in dataset_objects.items():
        date_column = (
            "Date"
            if "Date" in dataset.columns
            else (
                "StateAsOfDate"
                if "StateAsOfDate" in dataset.columns
                else None
            )
        )

        product_column = (
            "CanonicalProductID"
            if "CanonicalProductID" in dataset.columns
            else None
        )

        dataset_inventory_records.append(
            {
                "FileName": filename,
                "Rows": int(len(dataset)),
                "Columns": int(len(dataset.columns)),
                "MinimumDate": (
                    pd.to_datetime(
                        dataset[date_column]
                    ).min()
                    if date_column is not None
                    and len(dataset) > 0
                    else pd.NaT
                ),
                "MaximumDate": (
                    pd.to_datetime(
                        dataset[date_column]
                    ).max()
                    if date_column is not None
                    and len(dataset) > 0
                    else pd.NaT
                ),
                "Products": (
                    int(
                        dataset[product_column].nunique()
                    )
                    if product_column is not None
                    else np.nan
                ),
                "ContainsTarget": (
                    TARGET_COLUMN in dataset.columns
                ),
                "ContainsBulkDemand": (
                    "BulkDemand" in dataset.columns
                ),
                "ContainsTotalDemand": (
                    "TotalDemand" in dataset.columns
                ),
                "MethodSelectionAllowed": (
                    filename
                    in {
                        "ND03_pre_march_all_routes_development_dataset.csv",
                        "ND03_pre_march_main_model_development_dataset.csv",
                        "ND03_pre_march_fallback_development_dataset.csv",
                    }
                ),
                "Usage": (
                    "MODEL_DEVELOPMENT"
                    if "pre_march" in filename
                    else (
                        "DIAGNOSTIC_ONLY"
                        if "opened_march" in filename
                        else (
                            "AFTER_METHOD_LOCK_ONLY"
                            if "future_forecast_fit" in filename
                            else (
                                "FUTURE_INFERENCE_STATE"
                                if "recursive_inference_state" in filename
                                else "FULL_FEATURE_AUDIT"
                            )
                        )
                    )
                ),
            }
        )

    dataset_inventory = pd.DataFrame(
        dataset_inventory_records
    )

    split_summary = pd.DataFrame(
        [
            {
                "Split": "PRE_MARCH_ALL_ROUTES",
                "MinimumDate": pre_march_all_routes["Date"].min(),
                "MaximumDate": pre_march_all_routes["Date"].max(),
                "Rows": int(len(pre_march_all_routes)),
                "Products": int(
                    pre_march_all_routes[
                        "CanonicalProductID"
                    ].nunique()
                ),
                "NormalDemand": float(
                    pre_march_all_routes[TARGET_COLUMN].sum()
                ),
                "MethodSelectionAllowed": True,
                "TargetPresent": True,
            },
            {
                "Split": "PRE_MARCH_MAIN_MODEL",
                "MinimumDate": pre_march_main_model["Date"].min(),
                "MaximumDate": pre_march_main_model["Date"].max(),
                "Rows": int(len(pre_march_main_model)),
                "Products": int(
                    pre_march_main_model[
                        "CanonicalProductID"
                    ].nunique()
                ),
                "NormalDemand": float(
                    pre_march_main_model[TARGET_COLUMN].sum()
                ),
                "MethodSelectionAllowed": True,
                "TargetPresent": True,
            },
            {
                "Split": "PRE_MARCH_FALLBACK",
                "MinimumDate": pre_march_fallback["Date"].min(),
                "MaximumDate": pre_march_fallback["Date"].max(),
                "Rows": int(len(pre_march_fallback)),
                "Products": int(
                    pre_march_fallback[
                        "CanonicalProductID"
                    ].nunique()
                ),
                "NormalDemand": float(
                    pre_march_fallback[TARGET_COLUMN].sum()
                ),
                "MethodSelectionAllowed": True,
                "TargetPresent": True,
            },
            {
                "Split": "OPENED_MARCH_ALL_ROUTES",
                "MinimumDate": march_scoring_features["Date"].min(),
                "MaximumDate": march_scoring_features["Date"].max(),
                "Rows": int(len(march_scoring_features)),
                "Products": int(
                    march_scoring_features[
                        "CanonicalProductID"
                    ].nunique()
                ),
                "NormalDemand": float(
                    march_target_vault[TARGET_COLUMN].sum()
                ),
                "MethodSelectionAllowed": False,
                "TargetPresent": False,
            },
            {
                "Split": "OPENED_MARCH_MAIN_MODEL",
                "MinimumDate": march_main_scoring_features["Date"].min(),
                "MaximumDate": march_main_scoring_features["Date"].max(),
                "Rows": int(len(march_main_scoring_features)),
                "Products": int(
                    march_main_scoring_features[
                        "CanonicalProductID"
                    ].nunique()
                ),
                "NormalDemand": float(
                    march_main_target_vault[TARGET_COLUMN].sum()
                ),
                "MethodSelectionAllowed": False,
                "TargetPresent": False,
            },
            {
                "Split": "FUTURE_FIT_MAIN_MODEL",
                "MinimumDate": future_fit_main_model["Date"].min(),
                "MaximumDate": future_fit_main_model["Date"].max(),
                "Rows": int(len(future_fit_main_model)),
                "Products": int(
                    future_fit_main_model[
                        "CanonicalProductID"
                    ].nunique()
                ),
                "NormalDemand": float(
                    future_fit_main_model[TARGET_COLUMN].sum()
                ),
                "MethodSelectionAllowed": False,
                "TargetPresent": True,
            },
        ]
    )

    # =========================================================================
    # LEAKAGE AUDIT
    # =========================================================================

    current_target_in_predictors = (
        TARGET_COLUMN in CORE_PREDICTORS
    )

    forbidden_predictors = sorted(
        {
            "BulkDemand",
            "TotalDemand",
            "IsObservedProductDate",
            "IsZeroDemandRow",
            "DemandRecordSource",
            TARGET_COLUMN,
        }
        & set(CORE_PREDICTORS)
    )

    historical_feature_contract = core_feature_contract.loc[
        core_feature_contract[
            "IsHistoricalDemandFeature"
        ]
    ]

    historical_using_current_target = int(
        historical_feature_contract[
            "UsesCurrentRowTarget"
        ].astype(bool).sum()
    )

    historical_using_future_target = int(
        historical_feature_contract[
            "UsesFutureTarget"
        ].astype(bool).sum()
    )

    pre_march_contains_march_rows = int(
        (
            pd.to_datetime(
                pre_march_all_routes["Date"]
            )
            >= DEVELOPMENT_END_EXCLUSIVE
        ).sum()
    )

    march_features_target_present = (
        TARGET_COLUMN in march_scoring_features.columns
    )

    march_main_features_target_present = (
        TARGET_COLUMN
        in march_main_scoring_features.columns
    )

    main_model_route_errors = int(
        (
            feature_frame["EligibleForMainModel"]
            & (
                feature_frame["ForecastRoute"]
                != "MAIN_MODEL"
            )
        ).sum()
    )

    leakage_audit = pd.DataFrame(
        [
            {
                "Check": "Core predictor count",
                "Expected": 53,
                "Actual": len(CORE_PREDICTORS),
                "Passed": len(CORE_PREDICTORS) == 53,
                "Interpretation": (
                    "Matches the original daily architecture size"
                ),
            },
            {
                "Check": "Historical predictor count",
                "Expected": 31,
                "Actual": len(HISTORICAL_DEMAND_PREDICTORS),
                "Passed": (
                    len(HISTORICAL_DEMAND_PREDICTORS)
                    == 31
                ),
                "Interpretation": (
                    "All demand-derived predictors are rebuilt from NormalDemand"
                ),
            },
            {
                "Check": "Current target excluded from predictors",
                "Expected": False,
                "Actual": current_target_in_predictors,
                "Passed": not current_target_in_predictors,
                "Interpretation": (
                    "NormalDemand is a target and source for shifted history only"
                ),
            },
            {
                "Check": "Forbidden columns excluded from predictors",
                "Expected": [],
                "Actual": forbidden_predictors,
                "Passed": len(forbidden_predictors) == 0,
                "Interpretation": (
                    "No bulk, total, current target, or same-day target status enters the core predictor set"
                ),
            },
            {
                "Check": "Historical features using current-row target",
                "Expected": 0,
                "Actual": historical_using_current_target,
                "Passed": historical_using_current_target == 0,
                "Interpretation": (
                    "Every historical feature is shifted by at least one product operating row"
                ),
            },
            {
                "Check": "Historical features using future targets",
                "Expected": 0,
                "Actual": historical_using_future_target,
                "Passed": historical_using_future_target == 0,
                "Interpretation": (
                    "No feature uses future actual demand"
                ),
            },
            {
                "Check": "Expanding past mean matches ND02 prior mean",
                "Expected": "<= 1e-9",
                "Actual": prior_mean_max_difference,
                "Passed": prior_mean_max_difference <= 1e-9,
                "Interpretation": (
                    "Independent prior-only calculations reconcile"
                ),
            },
            {
                "Check": "Expanding positive rate matches ND02 prior rate",
                "Expected": "<= 1e-9",
                "Actual": prior_positive_rate_max_difference,
                "Passed": (
                    prior_positive_rate_max_difference <= 1e-9
                ),
                "Interpretation": (
                    "Independent prior-only positivity calculations reconcile"
                ),
            },
            {
                "Check": "Days since positive matches ND02 prior field",
                "Expected": "<= 1e-9",
                "Actual": days_since_positive_max_difference,
                "Passed": (
                    days_since_positive_max_difference <= 1e-9
                ),
                "Interpretation": (
                    "Previous-positive recency excludes current-day demand"
                ),
            },
            {
                "Check": "Main-model rows missing historical features",
                "Expected": 0,
                "Actual": main_history_missing_rows,
                "Passed": main_history_missing_rows == 0,
                "Interpretation": (
                    "The 20-day eligibility rule gives complete historical feature coverage"
                ),
            },
            {
                "Check": "Pre-March development contains March rows",
                "Expected": 0,
                "Actual": pre_march_contains_march_rows,
                "Passed": pre_march_contains_march_rows == 0,
                "Interpretation": (
                    "Opened March targets cannot influence method selection"
                ),
            },
            {
                "Check": "March scoring features contain target",
                "Expected": False,
                "Actual": march_features_target_present,
                "Passed": not march_features_target_present,
                "Interpretation": (
                    "Diagnostic features and targets are stored separately"
                ),
            },
            {
                "Check": "March main scoring features contain target",
                "Expected": False,
                "Actual": march_main_features_target_present,
                "Passed": not march_main_features_target_present,
                "Interpretation": (
                    "Main-route diagnostic features and targets are stored separately"
                ),
            },
            {
                "Check": "Main eligibility and route consistency",
                "Expected": 0,
                "Actual": main_model_route_errors,
                "Passed": main_model_route_errors == 0,
                "Interpretation": (
                    "Every eligible row is explicitly routed to the main model"
                ),
            },
            {
                "Check": "Recursive replay feature failures",
                "Expected": 0,
                "Actual": int(
                    (~recursive_replay_audit["Passed"]).sum()
                ),
                "Passed": bool(
                    recursive_replay_audit["Passed"].all()
                ),
                "Interpretation": (
                    "Final-row features can be reconstructed from earlier product history only"
                ),
            },
        ]
    )

    if not leakage_audit["Passed"].all():
        raise AssertionError(
            "ND03 leakage audit failed:\n"
            + leakage_audit.loc[
                ~leakage_audit["Passed"]
            ].to_string(index=False)
        )

    # =========================================================================
    # SCHEMA AUDIT
    # =========================================================================

    schema_records = []

    for dataset_name, dataset in dataset_objects.items():
        for column_order, column in enumerate(
            dataset.columns
        ):
            schema_records.append(
                {
                    "Dataset": dataset_name,
                    "ColumnOrder": column_order,
                    "Column": column,
                    "DataType": str(dataset[column].dtype),
                    "NonNullRows": int(
                        dataset[column].notna().sum()
                    ),
                    "UniqueNonNullValues": int(
                        dataset[column].nunique(
                            dropna=True
                        )
                    ),
                    "IsCorePredictor": (
                        column in CORE_PREDICTORS
                    ),
                    "IsTarget": column == TARGET_COLUMN,
                    "IsHistoricalDemandFeature": (
                        column
                        in HISTORICAL_DEMAND_PREDICTORS
                    ),
                }
            )

    output_schema_audit = pd.DataFrame(
        schema_records
    )

    # =========================================================================
    # CONTRACTS
    # =========================================================================

    model_ready_dataset_contract = {
        "StepID": STEP_ID,
        "Status": STATUS,
        "CreatedUTC": NOW_UTC.isoformat(),
        "FeatureEngineeringVersion": FEATURE_ENGINEERING_VERSION,
        "Source": {
            "ND02PanelPath": str(ND02_PANEL_PATH),
            "ND02PanelSHA256": actual_nd02_panel_sha256,
            "ND02CheckpointPath": str(ND02_CHECKPOINT_PATH),
            "ND02CheckpointSHA256": (
                nd02_checkpoint_sha256_before
            ),
        },
        "Target": TARGET_COLUMN,
        "BulkPolicy": {
            "BulkDemandInFeaturePanel": False,
            "BulkDemandInPredictors": False,
            "TotalDemandInFeaturePanel": False,
            "TotalDemandInPredictors": False,
            "ConfirmedBulkAddedExternally": True,
        },
        "CoreArchitecture": {
            "DirectPredictorCount": len(CORE_PREDICTORS),
            "HistoricalDemandPredictorCount": len(
                HISTORICAL_DEMAND_PREDICTORS
            ),
            "NumericPredictorCount": len(
                CORE_NUMERIC_PREDICTORS
            ),
            "CategoricalPredictorCount": len(
                CORE_CATEGORICAL_PREDICTORS
            ),
            "CorePredictors": CORE_PREDICTORS,
            "HistoricalDemandPredictors": (
                HISTORICAL_DEMAND_PREDICTORS
            ),
            "TargetIncludedAsPredictor": False,
            "ProductIDIncludedInCoreArchitecture": False,
            "OptionalChallengerPredictors": (
                OPTIONAL_CANDIDATE_PREDICTORS
            ),
        },
        "HistoricalFeatureRules": {
            "GroupKey": "CanonicalProductID",
            "TimeIndex": "OperatingDaySequence",
            "MinimumShift": 1,
            "MaximumLagOperatingDays": (
                MAXIMUM_REQUIRED_HISTORY
            ),
            "RollingMinimumPeriods": 1,
            "RollingStdDDOF": 0,
            "CurrentRowNormalDemandUsed": False,
            "FutureNormalDemandUsed": False,
        },
        "Routing": {
            "PrimaryScopePercentage": (
                PRIMARY_SCOPE_PERCENTAGE
            ),
            "MinimumPriorOperatingDays": (
                MAXIMUM_REQUIRED_HISTORY
            ),
            "MainModelRowsRequireAllHistoricalFeatures": True,
            "FallbackRowsRetained": True,
        },
        "Datasets": dataset_inventory.to_dict(
            orient="records"
        ),
        "EvaluationProtocol": {
            "MethodSelectionEndExclusive": (
                DEVELOPMENT_END_EXCLUSIVE.date().isoformat()
            ),
            "OpenedMarchStart": (
                OPENED_MARCH_START.date().isoformat()
            ),
            "OpenedMarchEnd": (
                OPENED_MARCH_END.date().isoformat()
            ),
            "MarchEligibleForMethodSelection": False,
            "MarchFeatureMode": (
                "ONE_STEP_ACTUAL_HISTORY_AVAILABLE_BEFORE_EACH_DATE"
            ),
            "WeekStartRecursiveEvaluationRequiresReconstruction": True,
            "NewUntouchedFuturePeriodRequired": True,
        },
    }

    recursive_inference_contract = {
        "StepID": STEP_ID,
        "Status": STATUS,
        "StatePath": str(RECURSIVE_STATE_PATH),
        "StateRows": int(len(recursive_state)),
        "MaximumDemandHistoryValuesStoredPerProduct": (
            MAXIMUM_REQUIRED_HISTORY
        ),
        "StateContains": [
            "Last observed date and operating-day sequence",
            "Last 20 observed normal-demand values",
            "Cumulative normal demand",
            "Positive and zero normal-demand day counts",
            "Most recent positive operating-day sequence",
            "Product metadata",
            "Latest routing status",
            "Source-end catalogue activity flag",
        ],
        "NextDateGeneration": {
            "CalendarRequirement": (
                "Requested operating date and operating-day sequence"
            ),
            "DefaultFutureWeek": "Monday to Friday",
            "ClosureOverrideRequired": True,
            "ProductAgeUpdateRequired": True,
            "CrossProductScopeRerankingRequired": True,
        },
        "RecursiveGeneration": {
            "AppendPredictionToTemporaryHistory": True,
            "RecomputeAll31HistoricalFeatures": True,
            "UseFutureActualDemand": False,
            "SupportArbitraryFutureDate": True,
            "SupportWeekStartMondayToFriday": True,
            "SupportRollingActualUpdate": True,
        },
        "CataloguePolicy": {
            "ProductsObservedOnFinalSourceDate": int(
                recursive_state[
                    "IsObservedOnFinalSourceDate"
                ].sum()
            ),
            "ProductsNotObservedOnFinalSourceDate": int(
                (
                    ~recursive_state[
                        "IsObservedOnFinalSourceDate"
                    ]
                ).sum()
            ),
            "Policy": (
                "The future active-product catalogue must be "
                "confirmed at inference time. Products not active "
                "at the source end are not forecast automatically "
                "without an explicit catalogue override."
            ),
        },
        "ReplayAudit": {
            "FeaturesTested": int(
                len(recursive_replay_audit)
            ),
            "AllPassed": bool(
                recursive_replay_audit["Passed"].all()
            ),
        },
    }

    # =========================================================================
    # REPORT TEXT
    # =========================================================================

    pre_march_main_demand = float(
        pre_march_main_model[TARGET_COLUMN].sum()
    )

    pre_march_all_demand = float(
        pre_march_all_routes[TARGET_COLUMN].sum()
    )

    march_main_demand = float(
        march_main_target_vault[TARGET_COLUMN].sum()
    )

    march_all_demand = float(
        march_target_vault[TARGET_COLUMN].sum()
    )

    active_at_source_end = int(
        recursive_state[
            "IsObservedOnFinalSourceDate"
        ].sum()
    )

    report_summary_text = f"""# ND03 Feature Engineering Summary

## Status

`{STATUS}`

## Core architecture

The revised daily model retains the original architecture size while replacing all historical total-demand features with normal-demand equivalents.

- Core direct predictors: {len(CORE_PREDICTORS)}
- Known-ahead and metadata predictors: {len(BASE_NUMERIC_PREDICTORS) + len(BASE_CATEGORICAL_PREDICTORS)}
- Historical normal-demand predictors: {len(HISTORICAL_DEMAND_PREDICTORS)}
- Maximum operating-day lag: {MAXIMUM_REQUIRED_HISTORY}
- Model target: `NormalDemand`
- Bulk or total-demand predictors: none
- Product identifier in the core architecture: no

## Dataset outputs

- Full feature rows: {len(feature_frame):,}
- Pre-March all-route development rows: {len(pre_march_all_routes):,}
- Pre-March main-model rows: {len(pre_march_main_model):,}
- Pre-March fallback rows: {len(pre_march_fallback):,}
- Opened March diagnostic rows: {len(march_scoring_features):,}
- Opened March main-model diagnostic rows: {len(march_main_scoring_features):,}
- Future-fit main-model rows: {len(future_fit_main_model):,}
- Recursive product-state rows: {len(recursive_state):,}

## Demand coverage

- Pre-March main-model normal demand: {pre_march_main_demand:,.6f}
- Pre-March all-route normal demand: {pre_march_all_demand:,.6f}
- Pre-March main-model demand share: {percentage(pre_march_main_demand, pre_march_all_demand):.6f}%
- March main-model normal demand: {march_main_demand:,.6f}
- March all-route normal demand: {march_all_demand:,.6f}
- March main-model demand share: {percentage(march_main_demand, march_all_demand):.6f}%

## Leakage and readiness

- Main-model rows missing historical predictors: {main_history_missing_rows}
- Recursive replay features passed: {int(recursive_replay_audit['Passed'].sum())}/{len(recursive_replay_audit)}
- March diagnostic target present in scoring features: no
- March eligible for method selection: no

## Future inference preparation

A state snapshot was created for all {len(recursive_state):,} products. It contains the last 20 normal-demand observations, cumulative demand, positive-day counts, the previous-positive sequence, metadata, and latest routing information.

Only {active_at_source_end:,} products were observed on the final source date. The active catalogue must therefore be confirmed when arbitrary future forecasts are generated. Products not active at the source end require an explicit catalogue override.

The March feature files represent one-step forecasting with actual history available before each date. A genuine week-start forecast must be generated recursively and will be evaluated later rather than by directly summing these feature rows.

## Next step

ND04 will define chronological validation folds and evaluate daily baselines for both the main-model and fallback routes. It will also aggregate daily predictions into weekly product and restaurant totals without using March 2026 for method selection.
"""

    readme_text = f"""# ND03 Normal-Demand Feature Engineering

Status: `{STATUS}`

## Purpose

This folder contains the model-ready normal-demand features and split datasets for the revised Eden daily forecasting system.

## Main outputs

- `01_model_ready_datasets/ND03_normal_demand_feature_panel.csv`
- `01_model_ready_datasets/ND03_pre_march_main_model_development_dataset.csv`
- `01_model_ready_datasets/ND03_pre_march_fallback_development_dataset.csv`
- `01_model_ready_datasets/ND03_opened_march_diagnostic_scoring_features.csv`
- `01_model_ready_datasets/ND03_opened_march_diagnostic_target_vault.csv`
- `01_model_ready_datasets/ND03_future_forecast_fit_main_model_dataset.csv`
- `01_model_ready_datasets/ND03_recursive_inference_state.csv`

## Core rules

- Target: `NormalDemand`
- Core predictors: {len(CORE_PREDICTORS)}
- Historical predictors: {len(HISTORICAL_DEMAND_PREDICTORS)}
- All historical predictors use prior normal demand only
- Bulk and total demand are absent
- Main-model rows have all 31 historical predictors
- March 2026 is diagnostic only
- Future week-start forecasts must be generated recursively
- Models fitted in ND03: none
- ND03 step lock created: none
"""

    feature_contract_markdown = f"""# ND03 Core Feature Contract

## Architecture

The core feature set contains {len(CORE_PREDICTORS)} direct predictors, matching the size of the original daily architecture.

The original total-demand feature names have been replaced with normal-demand equivalents. No historical feature is copied from the old total-demand model dataset.

## Predictor groups

- Numeric predictors: {len(CORE_NUMERIC_PREDICTORS)}
- Categorical predictors: {len(CORE_CATEGORICAL_PREDICTORS)}
- Historical normal-demand predictors: {len(HISTORICAL_DEMAND_PREDICTORS)}

## Historical rules

- Group key: `CanonicalProductID`
- Time index: `OperatingDaySequence`
- Minimum target shift: one operating row
- Lags: {LAG_OPERATING_DAYS}
- Rolling windows: {ROLLING_WINDOWS}
- Zero-rate and positive-count windows: {ZERO_POSITIVE_WINDOWS}
- Rolling minimum periods: 1
- Standard-deviation convention: population standard deviation with `ddof=0`
- Maximum required history: {MAXIMUM_REQUIRED_HISTORY} operating days

## Product identifier

`CanonicalProductID` remains an identifier and is not part of the frozen core predictor set. It may be tested later as an optional categorical challenger feature without changing the core benchmark.

## Missingness

Early-history missing values are preserved. Main-model eligibility requires 20 prior operating days and positive prior demand, so all main-model rows have every historical predictor available. Metadata missingness is handled inside each later model pipeline.
"""

    # =========================================================================
    # WRITE STAGED OUTPUTS
    # =========================================================================

    staged_data_dir = (
        STAGING_ROOT
        / DATA_DIR.relative_to(ND03_ROOT)
    )

    staged_audit_dir = (
        STAGING_ROOT
        / AUDIT_DIR.relative_to(ND03_ROOT)
    )

    staged_contract_dir = (
        STAGING_ROOT
        / CONTRACT_DIR.relative_to(ND03_ROOT)
    )

    staged_report_dir = (
        STAGING_ROOT
        / REPORT_DIR.relative_to(ND03_ROOT)
    )

    staged_control_dir = (
        STAGING_ROOT
        / CONTROL_DIR.relative_to(ND03_ROOT)
    )

    staged_dataset_paths = {
        staged_data_dir / FEATURE_PANEL_PATH.name:
            feature_frame,
        staged_data_dir / PRE_MARCH_ALL_ROUTES_PATH.name:
            pre_march_all_routes,
        staged_data_dir / PRE_MARCH_MAIN_MODEL_PATH.name:
            pre_march_main_model,
        staged_data_dir / PRE_MARCH_FALLBACK_PATH.name:
            pre_march_fallback,
        staged_data_dir / MARCH_SCORING_FEATURES_PATH.name:
            march_scoring_features,
        staged_data_dir / MARCH_TARGET_VAULT_PATH.name:
            march_target_vault,
        staged_data_dir / MARCH_MAIN_SCORING_FEATURES_PATH.name:
            march_main_scoring_features,
        staged_data_dir / MARCH_MAIN_TARGET_VAULT_PATH.name:
            march_main_target_vault,
        staged_data_dir / FUTURE_FIT_MAIN_MODEL_PATH.name:
            future_fit_main_model,
        staged_data_dir / RECURSIVE_STATE_PATH.name:
            recursive_state,
    }

    for output_path, output_frame in staged_dataset_paths.items():
        write_csv(output_path, output_frame)

    staged_audit_paths = {
        staged_audit_dir / DATASET_INVENTORY_PATH.name:
            dataset_inventory,
        staged_audit_dir / SPLIT_SUMMARY_PATH.name:
            split_summary,
        staged_audit_dir / FEATURE_MISSINGNESS_PATH.name:
            predictor_missingness,
        staged_audit_dir / FEATURE_READINESS_PATH.name:
            route_feature_readiness,
        staged_audit_dir / LEAKAGE_AUDIT_PATH.name:
            leakage_audit,
        staged_audit_dir / RECURSIVE_COMPATIBILITY_PATH.name:
            recursive_compatibility_audit,
        staged_audit_dir / RECURSIVE_REPLAY_PATH.name:
            recursive_replay_audit,
        staged_audit_dir / SCHEMA_AUDIT_PATH.name:
            output_schema_audit,
    }

    for output_path, output_frame in staged_audit_paths.items():
        write_csv(output_path, output_frame)

    write_csv(
        staged_contract_dir
        / CORE_FEATURE_CONTRACT_PATH.name,
        core_feature_contract,
    )

    write_csv(
        staged_contract_dir
        / CORE_PREDICTOR_LIST_PATH.name,
        core_predictor_list,
    )

    write_csv(
        staged_contract_dir
        / OPTIONAL_PREDICTOR_LIST_PATH.name,
        optional_predictor_list,
    )

    write_json(
        staged_contract_dir
        / DATASET_CONTRACT_PATH.name,
        model_ready_dataset_contract,
    )

    write_json(
        staged_contract_dir
        / RECURSIVE_CONTRACT_PATH.name,
        recursive_inference_contract,
    )

    write_text(
        staged_contract_dir
        / FEATURE_CONTRACT_MD_PATH.name,
        feature_contract_markdown,
    )

    write_text(
        staged_report_dir
        / REPORT_SUMMARY_PATH.name,
        report_summary_text,
    )

    write_text(
        STAGING_ROOT / README_PATH.name,
        readme_text,
    )

    # =========================================================================
    # RELOAD CRITICAL OUTPUTS
    # =========================================================================

    reloaded_feature_panel = pd.read_csv(
        staged_data_dir / FEATURE_PANEL_PATH.name,
        low_memory=False,
    )

    reloaded_pre_march_main = pd.read_csv(
        staged_data_dir / PRE_MARCH_MAIN_MODEL_PATH.name,
        low_memory=False,
    )

    reloaded_march_features = pd.read_csv(
        staged_data_dir / MARCH_SCORING_FEATURES_PATH.name,
        low_memory=False,
    )

    reloaded_recursive_state = pd.read_csv(
        staged_data_dir / RECURSIVE_STATE_PATH.name,
        low_memory=False,
    )

    if len(reloaded_feature_panel) != EXPECTED_ROWS:
        raise AssertionError(
            "Reloaded feature-panel row count changed."
        )

    if not math.isclose(
        float(
            reloaded_feature_panel[TARGET_COLUMN].sum()
        ),
        EXPECTED_NORMAL_DEMAND_TOTAL,
        rel_tol=0.0,
        abs_tol=1e-9,
    ):
        raise AssertionError(
            "Reloaded feature-panel demand total changed."
        )

    if len(reloaded_pre_march_main) != EXPECTED_PRE_MARCH_MAIN_ROWS:
        raise AssertionError(
            "Reloaded pre-March main-model row count changed."
        )

    if TARGET_COLUMN in reloaded_march_features.columns:
        raise AssertionError(
            "Reloaded March scoring features contain the target."
        )

    if len(reloaded_recursive_state) != EXPECTED_PRODUCTS:
        raise AssertionError(
            "Reloaded recursive state row count changed."
        )

    # =========================================================================
    # VALIDATION SUMMARY
    # =========================================================================

    validation = pd.DataFrame(
        [
            {
                "Check": "ND02 checkpoint hash verified",
                "Expected": EXPECTED_ND02_CHECKPOINT_SHA256,
                "Actual": nd02_checkpoint_sha256_before,
                "Passed": (
                    nd02_checkpoint_sha256_before
                    == EXPECTED_ND02_CHECKPOINT_SHA256
                ),
            },
            {
                "Check": "ND02 manifest hash verified",
                "Expected": expected_nd02_manifest_sha256,
                "Actual": actual_nd02_manifest_sha256,
                "Passed": (
                    actual_nd02_manifest_sha256
                    == expected_nd02_manifest_sha256
                ),
            },
            {
                "Check": "ND02 panel hash verified",
                "Expected": expected_nd02_panel_sha256,
                "Actual": actual_nd02_panel_sha256,
                "Passed": (
                    actual_nd02_panel_sha256
                    == expected_nd02_panel_sha256
                ),
            },
            {
                "Check": "Feature-panel rows preserved",
                "Expected": EXPECTED_ROWS,
                "Actual": int(len(feature_frame)),
                "Passed": len(feature_frame) == EXPECTED_ROWS,
            },
            {
                "Check": "Feature-panel products preserved",
                "Expected": EXPECTED_PRODUCTS,
                "Actual": int(
                    feature_frame[
                        "CanonicalProductID"
                    ].nunique()
                ),
                "Passed": int(
                    feature_frame[
                        "CanonicalProductID"
                    ].nunique()
                )
                == EXPECTED_PRODUCTS,
            },
            {
                "Check": "Feature-panel operating dates preserved",
                "Expected": EXPECTED_OPERATING_DATES,
                "Actual": int(
                    feature_frame["Date"].nunique()
                ),
                "Passed": int(
                    feature_frame["Date"].nunique()
                )
                == EXPECTED_OPERATING_DATES,
            },
            {
                "Check": "Normal-demand total preserved",
                "Expected": EXPECTED_NORMAL_DEMAND_TOTAL,
                "Actual": float(
                    feature_frame[TARGET_COLUMN].sum()
                ),
                "Passed": math.isclose(
                    float(
                        feature_frame[TARGET_COLUMN].sum()
                    ),
                    EXPECTED_NORMAL_DEMAND_TOTAL,
                    rel_tol=0.0,
                    abs_tol=1e-9,
                ),
            },
            {
                "Check": "Core predictor count",
                "Expected": 53,
                "Actual": len(CORE_PREDICTORS),
                "Passed": len(CORE_PREDICTORS) == 53,
            },
            {
                "Check": "Historical predictor count",
                "Expected": 31,
                "Actual": len(HISTORICAL_DEMAND_PREDICTORS),
                "Passed": (
                    len(HISTORICAL_DEMAND_PREDICTORS)
                    == 31
                ),
            },
            {
                "Check": "Main-model rows with incomplete history",
                "Expected": 0,
                "Actual": main_history_missing_rows,
                "Passed": main_history_missing_rows == 0,
            },
            {
                "Check": "March scoring features contain target",
                "Expected": False,
                "Actual": (
                    TARGET_COLUMN
                    in march_scoring_features.columns
                ),
                "Passed": (
                    TARGET_COLUMN
                    not in march_scoring_features.columns
                ),
            },
            {
                "Check": "Recursive compatibility predictors",
                "Expected": 53,
                "Actual": int(
                    recursive_compatibility_audit[
                        "Passed"
                    ].sum()
                ),
                "Passed": bool(
                    recursive_compatibility_audit[
                        "Passed"
                    ].all()
                ),
            },
            {
                "Check": "Recursive replay features",
                "Expected": 31,
                "Actual": int(
                    recursive_replay_audit[
                        "Passed"
                    ].sum()
                ),
                "Passed": bool(
                    recursive_replay_audit["Passed"].all()
                ),
            },
            {
                "Check": "Leakage-audit checks",
                "Expected": int(len(leakage_audit)),
                "Actual": int(
                    leakage_audit["Passed"].sum()
                ),
                "Passed": bool(
                    leakage_audit["Passed"].all()
                ),
            },
            {
                "Check": "Models loaded",
                "Expected": False,
                "Actual": False,
                "Passed": True,
            },
            {
                "Check": "Models fitted or refitted",
                "Expected": False,
                "Actual": False,
                "Passed": True,
            },
            {
                "Check": "Predictions generated",
                "Expected": False,
                "Actual": False,
                "Passed": True,
            },
            {
                "Check": "ND03 step lock created",
                "Expected": False,
                "Actual": False,
                "Passed": True,
            },
        ]
    )

    if not validation["Passed"].all():
        raise AssertionError(
            "ND03 validation failed:\n"
            + validation.loc[
                ~validation["Passed"]
            ].to_string(index=False)
        )

    write_csv(
        staged_audit_dir / VALIDATION_PATH.name,
        validation,
    )

    # =========================================================================
    # MANIFEST
    # =========================================================================

    manifest_excluded_names = {
        MANIFEST_PATH.name,
        CHECKPOINT_PATH.name,
        CHECKPOINT_SHA_PATH.name,
    }

    files_for_manifest = sorted(
        path
        for path in STAGING_ROOT.rglob("*")
        if (
            path.is_file()
            and path.name
            not in manifest_excluded_names
        )
    )

    manifest = pd.DataFrame(
        [
            {
                "RelativePath": str(
                    path.relative_to(STAGING_ROOT)
                ),
                "Bytes": int(path.stat().st_size),
                "SHA256": sha256_file(path),
            }
            for path in files_for_manifest
        ]
    ).sort_values(
        "RelativePath"
    ).reset_index(drop=True)

    staged_manifest_path = (
        staged_control_dir / MANIFEST_PATH.name
    )

    write_csv(
        staged_manifest_path,
        manifest,
    )

    manifest_sha256 = sha256_file(
        staged_manifest_path
    )

    # =========================================================================
    # CHECKPOINT
    # =========================================================================

    checkpoint_payload = {
        "StepID": STEP_ID,
        "Status": STATUS,
        "CreatedUTC": NOW_UTC.isoformat(),
        "CreatedLocal": NOW_LOCAL.isoformat(),
        "ModelRoot": str(MODEL_ROOT),
        "ND03Root": str(ND03_ROOT),
        "Input": {
            "ND02PanelPath": str(ND02_PANEL_PATH),
            "ND02PanelSHA256": actual_nd02_panel_sha256,
            "ND02CheckpointPath": str(ND02_CHECKPOINT_PATH),
            "ND02CheckpointSHA256": (
                nd02_checkpoint_sha256_before
            ),
            "ND02ManifestSHA256": (
                actual_nd02_manifest_sha256
            ),
        },
        "FeatureArchitecture": {
            "FeatureEngineeringVersion": (
                FEATURE_ENGINEERING_VERSION
            ),
            "Target": TARGET_COLUMN,
            "CorePredictors": len(CORE_PREDICTORS),
            "NumericPredictors": len(
                CORE_NUMERIC_PREDICTORS
            ),
            "CategoricalPredictors": len(
                CORE_CATEGORICAL_PREDICTORS
            ),
            "HistoricalPredictors": len(
                HISTORICAL_DEMAND_PREDICTORS
            ),
            "MaximumLagOperatingDays": (
                MAXIMUM_REQUIRED_HISTORY
            ),
            "ContainsBulkDemand": False,
            "ContainsTotalDemand": False,
            "TargetIncludedAsPredictor": False,
        },
        "Datasets": {
            "FeaturePanelRows": int(len(feature_frame)),
            "PreMarchAllRouteRows": int(
                len(pre_march_all_routes)
            ),
            "PreMarchMainModelRows": int(
                len(pre_march_main_model)
            ),
            "PreMarchFallbackRows": int(
                len(pre_march_fallback)
            ),
            "OpenedMarchRows": int(
                len(march_scoring_features)
            ),
            "OpenedMarchMainModelRows": int(
                len(march_main_scoring_features)
            ),
            "FutureFitMainModelRows": int(
                len(future_fit_main_model)
            ),
            "RecursiveStateRows": int(
                len(recursive_state)
            ),
        },
        "Evaluation": {
            "MethodSelectionEndExclusive": (
                DEVELOPMENT_END_EXCLUSIVE.date().isoformat()
            ),
            "MarchEligibleForMethodSelection": False,
            "MarchFeaturesSeparatedFromTargets": True,
            "WeekStartRecursiveEvaluationStillRequired": True,
            "NewUntouchedFuturePeriodRequired": True,
        },
        "RecursiveReadiness": {
            "CorePredictorsCompatible": int(
                recursive_compatibility_audit[
                    "Passed"
                ].sum()
            ),
            "HistoricalReplayFeaturesPassed": int(
                recursive_replay_audit[
                    "Passed"
                ].sum()
            ),
            "ProductsInState": int(
                len(recursive_state)
            ),
            "ProductsObservedOnFinalDate": int(
                active_at_source_end
            ),
        },
        "Control": {
            "ManifestPath": str(MANIFEST_PATH),
            "ManifestSHA256": manifest_sha256,
            "ValidationPath": str(VALIDATION_PATH),
            "LeakageAuditPath": str(
                LEAKAGE_AUDIT_PATH
            ),
            "CoreFeatureContractPath": str(
                CORE_FEATURE_CONTRACT_PATH
            ),
            "DatasetContractPath": str(
                DATASET_CONTRACT_PATH
            ),
            "RecursiveContractPath": str(
                RECURSIVE_CONTRACT_PATH
            ),
        },
        "Safety": {
            "ModelsLoaded": False,
            "ModelsFitted": False,
            "ModelsRefitted": False,
            "PredictionsGenerated": False,
            "ND02InputsModified": False,
            "ExistingModelLocksModified": False,
            "ND03StepLockCreated": False,
            "CheckpointAndHashesCreated": True,
        },
        "ReadyForND04": True,
        "NextStep": "ND04",
    }

    staged_checkpoint_path = (
        staged_control_dir / CHECKPOINT_PATH.name
    )

    write_json(
        staged_checkpoint_path,
        checkpoint_payload,
    )

    checkpoint_sha256 = sha256_file(
        staged_checkpoint_path
    )

    staged_checkpoint_sha_path = (
        staged_control_dir
        / CHECKPOINT_SHA_PATH.name
    )

    write_text(
        staged_checkpoint_sha_path,
        (
            f"{checkpoint_sha256}  "
            f"{CHECKPOINT_PATH.name}\n"
        ),
    )

    # =========================================================================
    # FINAL STAGING VALIDATION
    # =========================================================================

    required_staged_outputs = [
        STAGING_ROOT / README_PATH.name,
        staged_data_dir / FEATURE_PANEL_PATH.name,
        staged_data_dir / PRE_MARCH_ALL_ROUTES_PATH.name,
        staged_data_dir / PRE_MARCH_MAIN_MODEL_PATH.name,
        staged_data_dir / PRE_MARCH_FALLBACK_PATH.name,
        staged_data_dir / MARCH_SCORING_FEATURES_PATH.name,
        staged_data_dir / MARCH_TARGET_VAULT_PATH.name,
        staged_data_dir / MARCH_MAIN_SCORING_FEATURES_PATH.name,
        staged_data_dir / MARCH_MAIN_TARGET_VAULT_PATH.name,
        staged_data_dir / FUTURE_FIT_MAIN_MODEL_PATH.name,
        staged_data_dir / RECURSIVE_STATE_PATH.name,
        staged_audit_dir / DATASET_INVENTORY_PATH.name,
        staged_audit_dir / FEATURE_MISSINGNESS_PATH.name,
        staged_audit_dir / LEAKAGE_AUDIT_PATH.name,
        staged_audit_dir / RECURSIVE_REPLAY_PATH.name,
        staged_audit_dir / VALIDATION_PATH.name,
        staged_contract_dir / CORE_FEATURE_CONTRACT_PATH.name,
        staged_contract_dir / CORE_PREDICTOR_LIST_PATH.name,
        staged_contract_dir / DATASET_CONTRACT_PATH.name,
        staged_contract_dir / RECURSIVE_CONTRACT_PATH.name,
        staged_report_dir / REPORT_SUMMARY_PATH.name,
        staged_manifest_path,
        staged_checkpoint_path,
        staged_checkpoint_sha_path,
    ]

    missing_staged_outputs = [
        path
        for path in required_staged_outputs
        if not path.is_file()
    ]

    if missing_staged_outputs:
        raise AssertionError(
            "Required staged ND03 outputs are missing:\n"
            + "\n".join(
                f"- {path}"
                for path in missing_staged_outputs
            )
        )

    # Verify all protected inputs before commit.
    protected_input_hashes_after = {
        str(path): sha256_file(path)
        for path in required_inputs
    }

    changed_inputs = [
        path
        for path in protected_input_hashes_before
        if (
            protected_input_hashes_before[path]
            != protected_input_hashes_after[path]
        )
    ]

    if changed_inputs:
        raise AssertionError(
            "One or more protected ND02 inputs changed during ND03:\n"
            + "\n".join(
                f"- {path}"
                for path in changed_inputs
            )
        )

    # =========================================================================
    # ATOMIC COMMIT
    # =========================================================================

    os.replace(
        STAGING_ROOT,
        ND03_ROOT,
    )

    # Copy checkpoint into the project-level checkpoint folder.
    TOP_LEVEL_CHECKPOINT_PATH.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    shutil.copy2(
        CHECKPOINT_PATH,
        TOP_LEVEL_CHECKPOINT_PATH,
    )

    shutil.copy2(
        CHECKPOINT_SHA_PATH,
        TOP_LEVEL_CHECKPOINT_SHA_PATH,
    )

    make_read_only(TOP_LEVEL_CHECKPOINT_PATH)
    make_read_only(TOP_LEVEL_CHECKPOINT_SHA_PATH)

    # =========================================================================
    # MEMORY AND AGENT HANDOFF
    # =========================================================================

    handoff_text = f"""# ND03 Handoff

## Current status

- Completed step: `{STEP_ID}`
- Status: `{STATUS}`
- Completed local time: `{NOW_LOCAL.isoformat()}`
- ND03 root: `{ND03_ROOT}`
- Main feature panel: `{FEATURE_PANEL_PATH}`
- Pre-March main-model dataset: `{PRE_MARCH_MAIN_MODEL_PATH}`
- Pre-March fallback dataset: `{PRE_MARCH_FALLBACK_PATH}`
- Recursive state: `{RECURSIVE_STATE_PATH}`
- Checkpoint: `{TOP_LEVEL_CHECKPOINT_PATH}`
- Checkpoint SHA-256: `{checkpoint_sha256}`

## Authoritative feature architecture

- Target: `NormalDemand`
- Core predictors: {len(CORE_PREDICTORS)}
- Historical normal-demand predictors: {len(HISTORICAL_DEMAND_PREDICTORS)}
- Maximum lag: {MAXIMUM_REQUIRED_HISTORY} operating days
- Bulk or total-demand predictors: none
- Product identifier in core architecture: no
- Product identifier may be tested later as an optional challenger feature

## Authoritative datasets

- Pre-March rows are available for chronological method development.
- March 2026 features and targets are separated and are diagnostic only.
- The future-fit main-model dataset may be used only after a method is selected and locked using pre-March evidence.
- Fallback rows remain separate for ND04 baseline testing.

## Forecasting implications

- March feature rows represent one-step forecasts with actual history available before each date.
- Initial Monday-to-Friday weekly forecasts must be generated recursively and cannot be evaluated by directly using later-day actual-history feature rows.
- A recursive state snapshot exists for every product.
- The future active-product catalogue must be confirmed because not every historical product was present on the final source date.

## Next step

`ND04` will create chronological validation folds and evaluate daily baselines for the main-model and fallback routes. It will report daily product, daily restaurant, weekly product, and weekly restaurant metrics without using March for method selection.
"""

    workflow_section = f"""## ND03 — Normal-demand feature engineering

Status: `{STATUS}`

Completed actions:

- rebuilt the original 53-predictor daily architecture using normal demand only;
- created 31 prior-only historical normal-demand predictors;
- confirmed that all main-model rows have complete historical predictors;
- created pre-March main-model and fallback development datasets;
- separated March diagnostic features from targets;
- created an all-observed-history future-fit dataset for use after method lock;
- created a 227-product recursive inference state;
- verified all historical features through an independent final-row replay audit;
- documented future arbitrary-date and weekly recursive requirements; and
- created hashes and a checkpoint without creating an ND03 step lock.

Next: ND04 baseline and fallback evaluation.
"""

    decisions_section = f"""## ND03 decisions

1. The core daily architecture contains {len(CORE_PREDICTORS)} direct predictors.
2. The {len(HISTORICAL_DEMAND_PREDICTORS)} historical predictors are generated from prior `NormalDemand` only.
3. The model target is never a direct predictor.
4. Bulk, total demand, and same-day target-status fields remain excluded.
5. The core architecture excludes `CanonicalProductID` for direct comparability with the original daily model.
6. Product ID and selected prior-routing fields may be tested only as optional challenger features.
7. Main-model rows have all 31 historical predictors available.
8. Pre-March rows are used for chronological method development.
9. March 2026 is stored as separate diagnostic features and targets and cannot select the model.
10. The future-fit dataset may be used only after the method is selected using pre-March evidence.
11. March feature rows are one-step actual-update features, not week-start recursive features.
12. Week-start Monday-to-Friday evaluation must regenerate later-day features recursively.
13. Future inference requires an explicit active-product catalogue and closure/calendar overrides.
14. ND03 creates a checkpoint and hashes but no step lock.
"""

    metrics_section = f"""## ND03 feature engineering

- Feature-panel rows: {len(feature_frame):,}
- Products: {feature_frame['CanonicalProductID'].nunique():,}
- Operating dates: {feature_frame['Date'].nunique():,}
- Normal demand: {feature_frame[TARGET_COLUMN].sum():,.6f}
- Core predictors: {len(CORE_PREDICTORS)}
- Historical normal-demand predictors: {len(HISTORICAL_DEMAND_PREDICTORS)}
- Pre-March all-route rows: {len(pre_march_all_routes):,}
- Pre-March main-model rows: {len(pre_march_main_model):,}
- Pre-March fallback rows: {len(pre_march_fallback):,}
- Opened March rows: {len(march_scoring_features):,}
- Opened March main-model rows: {len(march_main_scoring_features):,}
- Future-fit main-model rows: {len(future_fit_main_model):,}
- Recursive product-state rows: {len(recursive_state):,}
- Products observed on final source date: {active_at_source_end:,}
- Main-model rows with incomplete historical predictors: {main_history_missing_rows}
- Recursive replay features passed: {int(recursive_replay_audit['Passed'].sum())}/{len(recursive_replay_audit)}
- Models fitted: no
- Predictions generated: no
"""

    agents_section = f"""## ND03 authoritative status

Marker: ND03_AUTHORITATIVE_STATUS

- Status: `{STATUS}`
- Read next: `{ND03_HANDOFF_PATH}`
- Main feature panel: `{FEATURE_PANEL_PATH}`
- Core predictor contract: `{CORE_FEATURE_CONTRACT_PATH}`
- Core predictors: {len(CORE_PREDICTORS)}
- Historical normal-demand predictors: {len(HISTORICAL_DEMAND_PREDICTORS)}
- Pre-March main-model development dataset: `{PRE_MARCH_MAIN_MODEL_PATH}`
- Pre-March fallback development dataset: `{PRE_MARCH_FALLBACK_PATH}`
- March 2026: diagnostic only
- Future week-start evaluation: recursive generation required
- Next step: `ND04`
"""

    atomic_write_text(
        ND03_HANDOFF_PATH,
        handoff_text,
    )

    atomic_write_text(
        CURRENT_HANDOFF_PATH,
        handoff_text,
    )

    append_marked_section(
        WORKFLOW_PATH,
        "## ND03 — Normal-demand feature engineering",
        workflow_section,
    )

    append_marked_section(
        DECISIONS_PATH,
        "## ND03 decisions",
        decisions_section,
    )

    append_marked_section(
        METRICS_AND_RESULTS_PATH,
        "## ND03 feature engineering",
        metrics_section,
    )

    append_marked_section(
        AGENTS_PATH,
        "Marker: ND03_AUTHORITATIVE_STATUS",
        agents_section,
    )

    log_text = "\n".join(
        [
            f"Step: {STEP_ID}",
            f"Status: {STATUS}",
            f"Created local: {NOW_LOCAL.isoformat()}",
            f"Created UTC: {NOW_UTC.isoformat()}",
            f"ND02 panel: {ND02_PANEL_PATH}",
            f"ND02 panel SHA256: {actual_nd02_panel_sha256}",
            f"ND02 checkpoint SHA256: {nd02_checkpoint_sha256_before}",
            f"Feature panel: {FEATURE_PANEL_PATH}",
            f"Feature panel rows: {len(feature_frame)}",
            f"Core predictors: {len(CORE_PREDICTORS)}",
            f"Historical predictors: {len(HISTORICAL_DEMAND_PREDICTORS)}",
            f"Pre-March main rows: {len(pre_march_main_model)}",
            f"Pre-March fallback rows: {len(pre_march_fallback)}",
            f"March diagnostic rows: {len(march_scoring_features)}",
            f"Future-fit main rows: {len(future_fit_main_model)}",
            f"Recursive state rows: {len(recursive_state)}",
            f"Checkpoint SHA256: {checkpoint_sha256}",
            "Models loaded: False",
            "Models fitted: False",
            "Predictions generated: False",
            "ND02 inputs modified: False",
            "ND03 step lock created: False",
            "Ready for ND04: True",
            "",
        ]
    )

    atomic_write_text(
        LOG_PATH,
        log_text,
    )

except Exception:
    if STAGING_ROOT.exists():
        shutil.rmtree(STAGING_ROOT)
    raise


# =============================================================================
# FINAL OUTPUT
# =============================================================================

print("=" * 112)
print("EDEN NORMAL-DEMAND MODEL V2 — ND03 COMPLETE")
print("=" * 112)

print(f"Status: {STATUS}")
print(f"Local time: {NOW_LOCAL.isoformat()}")
print(f"ND03 root: {ND03_ROOT}")

print("\nINPUT VERIFICATION")
print(f"ND02 panel: {ND02_PANEL_PATH}")
print(f"ND02 panel SHA-256: {actual_nd02_panel_sha256}")
print(f"ND02 manifest SHA-256: {actual_nd02_manifest_sha256}")
print(f"ND02 checkpoint SHA-256: {nd02_checkpoint_sha256_before}")
print("ND02 inputs modified: False")

print("\nCORE FEATURE ARCHITECTURE")
print(f"Target: {TARGET_COLUMN}")
print(f"Core predictors: {len(CORE_PREDICTORS)}")
print(f"Numeric predictors: {len(CORE_NUMERIC_PREDICTORS)}")
print(f"Categorical predictors: {len(CORE_CATEGORICAL_PREDICTORS)}")
print(
    "Historical normal-demand predictors: "
    f"{len(HISTORICAL_DEMAND_PREDICTORS)}"
)
print(f"Maximum operating-day lag: {MAXIMUM_REQUIRED_HISTORY}")
print("BulkDemand predictor present: False")
print("TotalDemand predictor present: False")
print("NormalDemand included as predictor: False")
print("CanonicalProductID in core architecture: False")

print("\nFEATURE PANEL")
print(f"Panel: {FEATURE_PANEL_PATH}")
print(f"Rows: {len(feature_frame):,}")
print(f"Columns: {len(feature_frame.columns):,}")
print(
    "Products: "
    f"{feature_frame['CanonicalProductID'].nunique():,}"
)
print(
    "Operating dates: "
    f"{feature_frame['Date'].nunique():,}"
)
print(
    "Date range: "
    f"{feature_frame['Date'].min().date()} to "
    f"{feature_frame['Date'].max().date()}"
)
print(
    "Normal demand: "
    f"{feature_frame[TARGET_COLUMN].sum():,.6f}"
)
print(
    "Main-model rows missing historical features: "
    f"{main_history_missing_rows}"
)

print("\nMODEL-READY DATASETS")
print(
    "Pre-March all-route rows: "
    f"{len(pre_march_all_routes):,}"
)
print(
    "Pre-March main-model rows: "
    f"{len(pre_march_main_model):,}"
)
print(
    "Pre-March fallback rows: "
    f"{len(pre_march_fallback):,}"
)
print(
    "Opened March diagnostic rows: "
    f"{len(march_scoring_features):,}"
)
print(
    "Opened March main-model diagnostic rows: "
    f"{len(march_main_scoring_features):,}"
)
print(
    "Future-fit main-model rows: "
    f"{len(future_fit_main_model):,}"
)
print("March scoring features contain target: False")
print("March eligible for method selection: False")

print("\nRECURSIVE INFERENCE READINESS")
print(f"Product-state rows: {len(recursive_state):,}")
print(
    "Products observed on final source date: "
    f"{active_at_source_end:,}"
)
print(
    "Core predictors recursively compatible: "
    f"{int(recursive_compatibility_audit['Passed'].sum())}/"
    f"{len(recursive_compatibility_audit)}"
)
print(
    "Historical replay features passed: "
    f"{int(recursive_replay_audit['Passed'].sum())}/"
    f"{len(recursive_replay_audit)}"
)
print(
    "Future active-product catalogue confirmation required: True"
)

print("\nOUTPUTS")
print(f"- Feature panel: {FEATURE_PANEL_PATH}")
print(
    "- Pre-March main-model dataset: "
    f"{PRE_MARCH_MAIN_MODEL_PATH}"
)
print(
    "- Pre-March fallback dataset: "
    f"{PRE_MARCH_FALLBACK_PATH}"
)
print(
    "- March scoring features: "
    f"{MARCH_SCORING_FEATURES_PATH}"
)
print(f"- March target vault: {MARCH_TARGET_VAULT_PATH}")
print(
    "- Future-fit main-model dataset: "
    f"{FUTURE_FIT_MAIN_MODEL_PATH}"
)
print(f"- Recursive state: {RECURSIVE_STATE_PATH}")
print(f"- Core feature contract: {CORE_FEATURE_CONTRACT_PATH}")
print(f"- Missingness audit: {FEATURE_MISSINGNESS_PATH}")
print(f"- Leakage audit: {LEAKAGE_AUDIT_PATH}")
print(f"- Recursive replay audit: {RECURSIVE_REPLAY_PATH}")
print(f"- Validation: {VALIDATION_PATH}")
print(f"- Manifest: {MANIFEST_PATH}")
print(f"- Checkpoint: {TOP_LEVEL_CHECKPOINT_PATH}")
print(f"- Checkpoint SHA-256: {checkpoint_sha256}")
print(f"- Agent handoff: {ND03_HANDOFF_PATH}")

print("\nSAFETY")
print("- Models loaded: False")
print("- Models fitted/refitted: False")
print("- Predictions generated: False")
print("- ND02 inputs modified: False")
print("- Existing model locks modified: False")
print("- ND03 step lock created: False")
print("- ND03 checkpoint and hashes created: True")

print("\nNEXT STEP")
print(
    "ND04 — define chronological validation folds and evaluate "
    "daily baselines and fallback rules, including daily product, "
    "daily restaurant, weekly product, and weekly restaurant metrics."
)

print("=" * 112)

EDEN NORMAL-DEMAND MODEL V2 — ND03 COMPLETE
Status: ND03_NORMAL_DEMAND_FEATURES_CREATED_READY_FOR_ND04
Local time: 2026-08-07T22:30:06.223738+01:00
ND03 root: /Users/ryansmac/Desktop/Meng Project/eden_datasets/eden_normal_demand_model_v2/02_feature_engineering/ND03_normal_demand_features

INPUT VERIFICATION
ND02 panel: /Users/ryansmac/Desktop/Meng Project/eden_datasets/eden_normal_demand_model_v2/01_datasets/01_intermediate/ND02_normal_demand_preparation/01_data/ND02_normal_demand_daily_panel.csv
ND02 panel SHA-256: aed12ee8015dd4c449c3e4db7c564cc2ddbfdd2986ef946098779af0333c4ef4
ND02 manifest SHA-256: acd5a179dd9a150f9fedf24b7f0b56087d376352931fe5b88b2c4a3fb8621845
ND02 checkpoint SHA-256: 824254459a630b4526eb606be85e206576f34665f456654df510f7a01393ceed
ND02 inputs modified: False

CORE FEATURE ARCHITECTURE
Target: NormalDemand
Core predictors: 53
Numeric predictors: 45
Categorical predictors: 8
Historical normal-demand predictors: 31
Maximum operating-day lag: 20
BulkDemand predictor

In [5]:
from __future__ import annotations

# =============================================================================
# EDEN NORMAL-DEMAND MODEL V2
# ND04 — CHRONOLOGICAL BASELINE AND FALLBACK EVALUATION
#
# Purpose:
#   1. Verify the accepted ND03 checkpoint and manifest.
#   2. Define fixed expanding-window chronological validation folds.
#   3. Evaluate simple main-model baselines on MAIN_MODEL rows.
#   4. Evaluate route-specific fallback candidates on fallback rows.
#   5. Select one development benchmark for the main route and each fallback.
#   6. Assemble a complete routed baseline system for all validation rows.
#   7. Report daily product, daily restaurant, weekly product, and weekly
#      restaurant diagnostics.
#   8. Preserve March 2026 as diagnostic-only and do not open its target vault.
#
# Important interpretation:
#   Weekly diagnostics in ND04 aggregate one-step daily predictions whose
#   features use actual history available before each day. They are NOT the
#   final Monday-origin recursive weekly forecast. That is evaluated later.
#
# No advanced machine-learning model is trained in ND04.
# No existing source, feature dataset, checkpoint, or model lock is modified.
# =============================================================================

import hashlib
import json
import math
import os
import shutil
import stat
import uuid
from datetime import datetime, timezone
from pathlib import Path
from zoneinfo import ZoneInfo

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.dates import DateFormatter


# =============================================================================
# CONFIGURATION
# =============================================================================

DEFAULT_MODEL_ROOT = Path(
    "/Users/ryansmac/Desktop/Meng Project/"
    "eden_datasets/eden_normal_demand_model_v2"
)

MODEL_ROOT = Path(
    os.environ.get(
        "EDEN_NORMAL_DEMAND_MODEL_ROOT",
        str(DEFAULT_MODEL_ROOT),
    )
)

ND03_ROOT = (
    MODEL_ROOT
    / "02_feature_engineering"
    / "ND03_normal_demand_features"
)

DATASET_DIR = ND03_ROOT / "01_model_ready_datasets"
CONTRACT_DIR = ND03_ROOT / "03_contracts"
ND03_CONTROL_DIR = ND03_ROOT / "05_control"

ALL_ROUTES_PATH = (
    DATASET_DIR
    / "ND03_pre_march_all_routes_development_dataset.csv"
)

MAIN_MODEL_PATH = (
    DATASET_DIR
    / "ND03_pre_march_main_model_development_dataset.csv"
)

FALLBACK_PATH = (
    DATASET_DIR
    / "ND03_pre_march_fallback_development_dataset.csv"
)

CORE_FEATURE_CONTRACT_PATH = (
    CONTRACT_DIR / "ND03_core_feature_contract.csv"
)

ND03_MANIFEST_PATH = (
    ND03_CONTROL_DIR / "ND03_artifact_hash_manifest.csv"
)

ND03_CHECKPOINT_PATH = (
    MODEL_ROOT / "08_checkpoints" / "ND03_checkpoint.json"
)

EXPECTED_ND03_CHECKPOINT_SHA256 = (
    "0845af89a5b459ca13ae6ffd99dde444"
    "f5010f6c0fb5ba4c34a4f091ac2e151c"
)

ND04_ROOT = (
    MODEL_ROOT
    / "03_models"
    / "00_candidates"
    / "ND04_baseline_and_fallback_evaluation"
)

PREDICTION_DIR = ND04_ROOT / "01_predictions"
METRIC_DIR = ND04_ROOT / "02_metrics"
AUDIT_DIR = ND04_ROOT / "03_audits"
FIGURE_DIR = ND04_ROOT / "04_figures"
REPORT_DIR = ND04_ROOT / "05_reports"
CONTROL_DIR = ND04_ROOT / "06_control"

FOLD_DEFINITION_PATH = AUDIT_DIR / "ND04_chronological_fold_definition.csv"
INPUT_SCHEMA_PATH = AUDIT_DIR / "ND04_input_schema_audit.csv"
VALIDATION_PATH = AUDIT_DIR / "ND04_validation_summary.csv"
LEAKAGE_AUDIT_PATH = AUDIT_DIR / "ND04_leakage_and_protocol_audit.csv"
COVERAGE_PATH = AUDIT_DIR / "ND04_validation_route_coverage.csv"

MAIN_PREDICTIONS_PATH = PREDICTION_DIR / "ND04_main_baseline_predictions.csv"
FALLBACK_PREDICTIONS_PATH = PREDICTION_DIR / "ND04_fallback_candidate_predictions.csv"
SELECTED_SYSTEM_PREDICTIONS_PATH = PREDICTION_DIR / "ND04_selected_routed_system_predictions.csv"
DAILY_AGGREGATE_PATH = PREDICTION_DIR / "ND04_selected_system_daily_restaurant_totals.csv"
WEEKLY_PRODUCT_PATH = PREDICTION_DIR / "ND04_selected_system_weekly_product_totals.csv"
WEEKLY_RESTAURANT_PATH = PREDICTION_DIR / "ND04_selected_system_weekly_restaurant_totals.csv"

MAIN_METRICS_PATH = METRIC_DIR / "ND04_main_baseline_metrics.csv"
MAIN_FOLD_METRICS_PATH = METRIC_DIR / "ND04_main_baseline_metrics_by_fold.csv"
FALLBACK_METRICS_PATH = METRIC_DIR / "ND04_fallback_metrics_by_route.csv"
FALLBACK_FOLD_METRICS_PATH = METRIC_DIR / "ND04_fallback_metrics_by_route_and_fold.csv"
SELECTED_METHODS_PATH = METRIC_DIR / "ND04_selected_route_methods.csv"
SELECTED_SYSTEM_METRICS_PATH = METRIC_DIR / "ND04_selected_system_metrics.csv"
SELECTED_SYSTEM_ROUTE_METRICS_PATH = METRIC_DIR / "ND04_selected_system_metrics_by_route.csv"
SELECTED_SYSTEM_FOLD_METRICS_PATH = METRIC_DIR / "ND04_selected_system_metrics_by_fold.csv"

DECISION_JSON_PATH = REPORT_DIR / "ND04_baseline_and_fallback_decision.json"
REPORT_SUMMARY_PATH = REPORT_DIR / "ND04_baseline_and_fallback_summary.md"
README_PATH = ND04_ROOT / "README.md"

MANIFEST_PATH = CONTROL_DIR / "ND04_artifact_hash_manifest.csv"
CHECKPOINT_PATH = CONTROL_DIR / "ND04_checkpoint.json"
CHECKPOINT_SHA_PATH = CONTROL_DIR / "ND04_checkpoint.sha256"

TOP_LEVEL_CHECKPOINT_PATH = MODEL_ROOT / "08_checkpoints" / "ND04_checkpoint.json"
TOP_LEVEL_CHECKPOINT_SHA_PATH = MODEL_ROOT / "08_checkpoints" / "ND04_checkpoint.sha256"

MEMORY_ROOT = MODEL_ROOT / "00_project_memory"
AGENTS_PATH = MODEL_ROOT / "AGENTS.md"
CURRENT_HANDOFF_PATH = MEMORY_ROOT / "CURRENT_HANDOFF.md"
ND04_HANDOFF_PATH = MEMORY_ROOT / "ND04_HANDOFF.md"
WORKFLOW_PATH = MEMORY_ROOT / "WORKFLOW.md"
DECISIONS_PATH = MEMORY_ROOT / "DECISIONS.md"
METRICS_AND_RESULTS_PATH = MEMORY_ROOT / "METRICS_AND_RESULTS.md"
LOG_PATH = MODEL_ROOT / "09_logs" / "ND04_baseline_evaluation_log.txt"

STEP_ID = "ND04"
STATUS = "ND04_BASELINES_AND_FALLBACKS_EVALUATED_READY_FOR_ND05"

TARGET_COLUMN = "NormalDemand"
DATE_COLUMN = "Date"
PRODUCT_ID_COLUMN = "CanonicalProductID"
PRODUCT_NAME_COLUMN = "CanonicalProductName"
ROUTE_COLUMN = "ForecastRoute"
FAMILY_COLUMN = "TierProductFamily"
DAY_OF_WEEK_COLUMN = "DayOfWeekNumber"

N_FOLDS = 5
VALIDATION_DATES_PER_FOLD = 20
EXPECTED_ALL_ROUTE_ROWS = 23_763
EXPECTED_MAIN_ROWS = 10_976
EXPECTED_FALLBACK_ROWS = 12_787
EXPECTED_PRE_MARCH_DATES = 225
EXPECTED_MAIN_DATES = 205

ALLOW_OVERWRITE = False
DPI = 300

NOW_UTC = datetime.now(timezone.utc)
NOW_LOCAL = NOW_UTC.astimezone(ZoneInfo("Europe/Dublin"))


# =============================================================================
# HELPERS
# =============================================================================

def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()

    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)

    return digest.hexdigest()


def write_csv(path: Path, frame: pd.DataFrame) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    frame.to_csv(path, index=False)


def write_json(path: Path, payload: dict) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(
        json.dumps(
            payload,
            indent=2,
            ensure_ascii=False,
            default=str,
        )
        + "\n",
        encoding="utf-8",
    )


def write_text(path: Path, text: str) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(text, encoding="utf-8")


def atomic_write_text(path: Path, text: str) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary_path = path.with_name(
        f".{path.name}.{uuid.uuid4().hex}.tmp"
    )
    temporary_path.write_text(text, encoding="utf-8")
    os.replace(temporary_path, path)


def append_marked_section(path: Path, marker: str, section_text: str) -> None:
    existing = path.read_text(encoding="utf-8") if path.is_file() else ""

    if marker in existing:
        return

    separator = "\n" if existing.endswith("\n") else "\n\n"
    atomic_write_text(
        path,
        existing + separator + section_text.strip() + "\n",
    )


def make_read_only(path: Path) -> None:
    if path.is_file():
        path.chmod(
            stat.S_IRUSR
            | stat.S_IRGRP
            | stat.S_IROTH
        )


def safe_wape(actual: pd.Series, predicted: pd.Series) -> float:
    actual_array = pd.to_numeric(actual, errors="coerce").to_numpy(dtype=float)
    predicted_array = pd.to_numeric(predicted, errors="coerce").to_numpy(dtype=float)

    denominator = float(np.sum(actual_array))

    if denominator == 0:
        return float("nan")

    return float(
        100.0
        * np.sum(np.abs(predicted_array - actual_array))
        / denominator
    )


def metric_record(
    actual: pd.Series,
    predicted: pd.Series,
    evaluation_level: str,
) -> dict:
    actual_array = pd.to_numeric(actual, errors="coerce").to_numpy(dtype=float)
    predicted_array = pd.to_numeric(predicted, errors="coerce").to_numpy(dtype=float)

    errors = predicted_array - actual_array
    absolute_errors = np.abs(errors)

    actual_total = float(np.sum(actual_array))
    predicted_total = float(np.sum(predicted_array))
    total_bias = float(np.sum(errors))

    return {
        "EvaluationLevel": evaluation_level,
        "Observations": int(len(actual_array)),
        "ActualTotal": actual_total,
        "PredictedTotal": predicted_total,
        "MAE": float(np.mean(absolute_errors)),
        "RMSE": float(np.sqrt(np.mean(np.square(errors)))),
        "WAPEPercentage": safe_wape(actual, predicted),
        "MeanBias": float(np.mean(errors)),
        "TotalBias": total_bias,
        "AbsoluteBiasPercentage": (
            float(100.0 * abs(total_bias) / actual_total)
            if actual_total != 0
            else float("nan")
        ),
    }


def validate_required_columns(
    frame: pd.DataFrame,
    required_columns: set[str],
    frame_name: str,
) -> None:
    missing = sorted(required_columns - set(frame.columns))

    if missing:
        raise AssertionError(
            f"{frame_name} is missing required columns:\n"
            + "\n".join(f"- {column}" for column in missing)
        )


def coalesce_series(*series_values: pd.Series | float | int) -> pd.Series:
    output: pd.Series | None = None

    for value in series_values:
        if isinstance(value, pd.Series):
            candidate = pd.to_numeric(value, errors="coerce")
        else:
            if output is None:
                raise ValueError(
                    "A scalar cannot be the first value passed to coalesce_series."
                )
            candidate = pd.Series(value, index=output.index, dtype=float)

        if output is None:
            output = candidate.astype(float)
        else:
            output = output.where(output.notna(), candidate)

    if output is None:
        raise ValueError("No values were supplied to coalesce_series.")

    return output.astype(float)


def clip_prediction(values: pd.Series) -> pd.Series:
    numeric = pd.to_numeric(values, errors="coerce").astype(float)
    return numeric.clip(lower=0.0)


def build_hierarchy_predictions(
    training_frame: pd.DataFrame,
    scoring_frame: pd.DataFrame,
) -> tuple[pd.Series, pd.Series]:
    train = training_frame.copy()
    score = scoring_frame.copy()

    global_mean = float(train[TARGET_COLUMN].mean())

    product_weekday = (
        train.groupby(
            [PRODUCT_ID_COLUMN, DAY_OF_WEEK_COLUMN],
            dropna=False,
        )[TARGET_COLUMN]
        .mean()
    )

    product_mean = (
        train.groupby(PRODUCT_ID_COLUMN, dropna=False)[TARGET_COLUMN]
        .mean()
    )

    family_weekday = (
        train.groupby(
            [FAMILY_COLUMN, DAY_OF_WEEK_COLUMN],
            dropna=False,
        )[TARGET_COLUMN]
        .mean()
    )

    family_mean = (
        train.groupby(FAMILY_COLUMN, dropna=False)[TARGET_COLUMN]
        .mean()
    )

    global_weekday = (
        train.groupby(DAY_OF_WEEK_COLUMN, dropna=False)[TARGET_COLUMN]
        .mean()
    )

    product_weekday_values = pd.Series(
        [
            product_weekday.get((product_id, day_number), np.nan)
            for product_id, day_number in zip(
                score[PRODUCT_ID_COLUMN],
                score[DAY_OF_WEEK_COLUMN],
            )
        ],
        index=score.index,
        dtype=float,
    )

    product_values = score[PRODUCT_ID_COLUMN].map(product_mean).astype(float)

    family_weekday_values = pd.Series(
        [
            family_weekday.get((family_value, day_number), np.nan)
            for family_value, day_number in zip(
                score[FAMILY_COLUMN],
                score[DAY_OF_WEEK_COLUMN],
            )
        ],
        index=score.index,
        dtype=float,
    )

    family_values = score[FAMILY_COLUMN].map(family_mean).astype(float)
    global_weekday_values = score[DAY_OF_WEEK_COLUMN].map(global_weekday).astype(float)
    global_values = pd.Series(global_mean, index=score.index, dtype=float)
    zero_values = pd.Series(0.0, index=score.index, dtype=float)

    product_hierarchy = coalesce_series(
        product_weekday_values,
        product_values,
        family_weekday_values,
        family_values,
        global_weekday_values,
        global_values,
        zero_values,
    )

    family_hierarchy = coalesce_series(
        family_weekday_values,
        family_values,
        global_weekday_values,
        global_values,
        zero_values,
    )

    return clip_prediction(product_hierarchy), clip_prediction(family_hierarchy)


def score_prediction_table(
    prediction_frame: pd.DataFrame,
    group_columns: list[str],
    evaluation_level: str,
) -> pd.DataFrame:
    records: list[dict] = []

    grouped = prediction_frame.groupby(group_columns, dropna=False, sort=True)

    for group_values, group_frame in grouped:
        if not isinstance(group_values, tuple):
            group_values = (group_values,)

        record = {
            column: value
            for column, value in zip(group_columns, group_values)
        }

        record.update(
            metric_record(
                group_frame["ActualNormalDemand"],
                group_frame["PredictedNormalDemand"],
                evaluation_level,
            )
        )

        records.append(record)

    return pd.DataFrame(records)


def select_best_method(metrics: pd.DataFrame) -> pd.Series:
    sortable = metrics.copy()
    sortable["WAPESelectionValue"] = sortable["WAPEPercentage"].fillna(np.inf)
    sortable["MAESelectionValue"] = sortable["MAE"].fillna(np.inf)
    sortable["BiasSelectionValue"] = sortable["AbsoluteBiasPercentage"].fillna(np.inf)
    sortable["RMSESelectionValue"] = sortable["RMSE"].fillna(np.inf)

    sortable = sortable.sort_values(
        [
            "WAPESelectionValue",
            "MAESelectionValue",
            "BiasSelectionValue",
            "RMSESelectionValue",
            "CandidateMethod",
        ],
        kind="mergesort",
    )

    return sortable.iloc[0]


def save_figure(path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    plt.tight_layout()
    plt.savefig(path, dpi=DPI, bbox_inches="tight")
    plt.close()


# =============================================================================
# PREFLIGHT
# =============================================================================

required_inputs = [
    ALL_ROUTES_PATH,
    MAIN_MODEL_PATH,
    FALLBACK_PATH,
    CORE_FEATURE_CONTRACT_PATH,
    ND03_MANIFEST_PATH,
    ND03_CHECKPOINT_PATH,
    AGENTS_PATH,
    CURRENT_HANDOFF_PATH,
    WORKFLOW_PATH,
    DECISIONS_PATH,
    METRICS_AND_RESULTS_PATH,
]

missing_inputs = [path for path in required_inputs if not path.is_file()]

if missing_inputs:
    raise FileNotFoundError(
        "ND04 required input files are missing:\n"
        + "\n".join(f"- {path}" for path in missing_inputs)
    )

nd03_checkpoint_sha256_before = sha256_file(ND03_CHECKPOINT_PATH)

if nd03_checkpoint_sha256_before != EXPECTED_ND03_CHECKPOINT_SHA256:
    raise AssertionError(
        "The ND03 checkpoint hash does not match the accepted checkpoint.\n"
        f"Expected: {EXPECTED_ND03_CHECKPOINT_SHA256}\n"
        f"Actual:   {nd03_checkpoint_sha256_before}\n"
        f"Path:     {ND03_CHECKPOINT_PATH}"
    )

with ND03_CHECKPOINT_PATH.open("r", encoding="utf-8") as handle:
    nd03_checkpoint = json.load(handle)

if not bool(nd03_checkpoint.get("ReadyForND04", False)):
    raise AssertionError("The ND03 checkpoint is not marked ready for ND04.")

expected_manifest_sha256 = nd03_checkpoint["Control"]["ManifestSHA256"]
actual_manifest_sha256 = sha256_file(ND03_MANIFEST_PATH)

if actual_manifest_sha256 != expected_manifest_sha256:
    raise AssertionError(
        "The ND03 manifest hash does not match the checkpoint.\n"
        f"Expected: {expected_manifest_sha256}\n"
        f"Actual:   {actual_manifest_sha256}"
    )

if ND04_ROOT.exists():
    if not ALLOW_OVERWRITE:
        raise FileExistsError(
            "The ND04 output folder already exists. No files were changed:\n"
            f"{ND04_ROOT}"
        )
    shutil.rmtree(ND04_ROOT)

for top_level_path in [
    TOP_LEVEL_CHECKPOINT_PATH,
    TOP_LEVEL_CHECKPOINT_SHA_PATH,
]:
    if top_level_path.exists():
        if not ALLOW_OVERWRITE:
            raise FileExistsError(
                "An ND04 top-level checkpoint already exists. No files were changed:\n"
                f"{top_level_path}"
            )
        top_level_path.unlink()

STAGING_ROOT = ND04_ROOT.parent / f".ND04_staging_{uuid.uuid4().hex}"
STAGING_ROOT.mkdir(parents=True, exist_ok=False)


# =============================================================================
# LOAD AND VERIFY INPUT DATASETS
# =============================================================================

try:
    all_routes = pd.read_csv(ALL_ROUTES_PATH, low_memory=False)
    main_model = pd.read_csv(MAIN_MODEL_PATH, low_memory=False)
    fallback = pd.read_csv(FALLBACK_PATH, low_memory=False)
    core_contract = pd.read_csv(CORE_FEATURE_CONTRACT_PATH, low_memory=False)
    nd03_manifest = pd.read_csv(ND03_MANIFEST_PATH, low_memory=False)

    required_columns = {
        DATE_COLUMN,
        PRODUCT_ID_COLUMN,
        PRODUCT_NAME_COLUMN,
        ROUTE_COLUMN,
        FAMILY_COLUMN,
        DAY_OF_WEEK_COLUMN,
        TARGET_COLUMN,
        "NormalDemandLag_1",
        "NormalDemandLag_2",
        "NormalDemandLag_3",
        "NormalDemandLag_5",
        "NormalDemandLag_10",
        "NormalDemandLag_20",
        "PastNormalDemandRollingMean_3",
        "PastNormalDemandRollingMean_5",
        "PastNormalDemandRollingMean_10",
        "PastNormalDemandRollingMean_20",
        "PastNormalDemandRollingMedian_3",
        "PastNormalDemandRollingMedian_5",
        "PastNormalDemandRollingMedian_10",
        "PastNormalDemandRollingMedian_20",
        "PastNormalDemandRollingSum_5",
        "PastNormalDemandRollingSum_10",
        "PastPositiveNormalDemandCount_5",
        "PastPositiveNormalDemandCount_10",
        "ExpandingPastMeanNormalDemand",
        "EligibleForMethodSelection",
        "IsOpenedMarchDiagnosticPeriod",
    }

    for frame, frame_name in [
        (all_routes, "ND03 all-route development dataset"),
        (main_model, "ND03 main-model development dataset"),
        (fallback, "ND03 fallback development dataset"),
    ]:
        validate_required_columns(frame, required_columns, frame_name)
        frame[DATE_COLUMN] = pd.to_datetime(frame[DATE_COLUMN], errors="raise")
        frame[TARGET_COLUMN] = pd.to_numeric(frame[TARGET_COLUMN], errors="raise").astype(float)

    if len(all_routes) != EXPECTED_ALL_ROUTE_ROWS:
        raise AssertionError(
            f"Unexpected all-route row count: {len(all_routes)}"
        )

    if len(main_model) != EXPECTED_MAIN_ROWS:
        raise AssertionError(
            f"Unexpected main-model row count: {len(main_model)}"
        )

    if len(fallback) != EXPECTED_FALLBACK_ROWS:
        raise AssertionError(
            f"Unexpected fallback row count: {len(fallback)}"
        )

    if all_routes[DATE_COLUMN].nunique() != EXPECTED_PRE_MARCH_DATES:
        raise AssertionError(
            "Unexpected number of pre-March operating dates in all-route data."
        )

    if main_model[DATE_COLUMN].nunique() != EXPECTED_MAIN_DATES:
        raise AssertionError(
            "Unexpected number of main-model operating dates."
        )

    if not all_routes["EligibleForMethodSelection"].astype(bool).all():
        raise AssertionError(
            "The all-route development dataset contains rows not eligible for method selection."
        )

    if all_routes["IsOpenedMarchDiagnosticPeriod"].astype(bool).any():
        raise AssertionError(
            "Opened March rows were found in the ND04 development dataset."
        )

    if set(main_model[ROUTE_COLUMN].unique()) != {"MAIN_MODEL"}:
        raise AssertionError("The main-model development dataset contains non-main routes.")

    expected_fallback_routes = {
        "LOW_DEMAND_FALLBACK",
        "COLD_START_FALLBACK",
        "ZERO_HISTORY_FALLBACK",
    }

    if set(fallback[ROUTE_COLUMN].unique()) != expected_fallback_routes:
        raise AssertionError(
            "The fallback development dataset route set differs from the expected routes."
        )

    all_route_keys = set(
        zip(
            all_routes[DATE_COLUMN].astype("int64"),
            all_routes[PRODUCT_ID_COLUMN].astype(str),
        )
    )

    split_keys = set(
        zip(
            pd.concat([main_model[DATE_COLUMN], fallback[DATE_COLUMN]]).astype("int64"),
            pd.concat([
                main_model[PRODUCT_ID_COLUMN],
                fallback[PRODUCT_ID_COLUMN],
            ]).astype(str),
        )
    )

    if all_route_keys != split_keys:
        raise AssertionError(
            "The main-model and fallback datasets do not exactly partition the all-route dataset."
        )

    # Verify input file hashes against the ND03 manifest.
    manifest_lookup = {
        str(row["RelativePath"]): str(row["SHA256"])
        for _, row in nd03_manifest.iterrows()
    }

    input_hash_rows = []

    for input_path in [
        ALL_ROUTES_PATH,
        MAIN_MODEL_PATH,
        FALLBACK_PATH,
        CORE_FEATURE_CONTRACT_PATH,
    ]:
        relative_path = str(input_path.relative_to(ND03_ROOT))
        actual_hash = sha256_file(input_path)
        expected_hash = manifest_lookup.get(relative_path)

        if expected_hash is None:
            raise AssertionError(
                f"Input file is absent from the ND03 manifest: {relative_path}"
            )

        if actual_hash != expected_hash:
            raise AssertionError(
                "Input file hash differs from the ND03 manifest.\n"
                f"File: {input_path}\n"
                f"Expected: {expected_hash}\n"
                f"Actual:   {actual_hash}"
            )

        input_hash_rows.append(
            {
                "InputPath": str(input_path),
                "RelativePath": relative_path,
                "SHA256": actual_hash,
                "MatchesND03Manifest": True,
            }
        )

    input_hash_audit = pd.DataFrame(input_hash_rows)

    # =========================================================================
    # FIXED CHRONOLOGICAL FOLDS
    # =========================================================================

    all_dates = pd.DatetimeIndex(
        sorted(all_routes[DATE_COLUMN].drop_duplicates())
    )

    total_validation_dates = N_FOLDS * VALIDATION_DATES_PER_FOLD

    if len(all_dates) <= total_validation_dates:
        raise AssertionError(
            "There are not enough dates to create the requested chronological folds."
        )

    first_validation_index = len(all_dates) - total_validation_dates
    fold_records = []
    date_to_fold: dict[pd.Timestamp, int] = {}

    for fold_number in range(1, N_FOLDS + 1):
        validation_start_index = (
            first_validation_index
            + (fold_number - 1) * VALIDATION_DATES_PER_FOLD
        )
        validation_end_index = (
            validation_start_index
            + VALIDATION_DATES_PER_FOLD
        )

        validation_dates = all_dates[
            validation_start_index:validation_end_index
        ]
        training_dates = all_dates[:validation_start_index]

        if len(validation_dates) != VALIDATION_DATES_PER_FOLD:
            raise AssertionError(
                f"Fold {fold_number} has an unexpected validation-date count."
            )

        if len(training_dates) == 0:
            raise AssertionError(
                f"Fold {fold_number} has no prior training dates."
            )

        for date_value in validation_dates:
            date_to_fold[pd.Timestamp(date_value)] = fold_number

        fold_records.append(
            {
                "Fold": fold_number,
                "TrainStart": training_dates.min(),
                "TrainEnd": training_dates.max(),
                "TrainingOperatingDates": int(len(training_dates)),
                "ValidationStart": validation_dates.min(),
                "ValidationEnd": validation_dates.max(),
                "ValidationOperatingDates": int(len(validation_dates)),
                "TrainingRowsAllRoutes": int(
                    all_routes[DATE_COLUMN].isin(training_dates).sum()
                ),
                "ValidationRowsAllRoutes": int(
                    all_routes[DATE_COLUMN].isin(validation_dates).sum()
                ),
                "ValidationRowsMainModel": int(
                    main_model[DATE_COLUMN].isin(validation_dates).sum()
                ),
                "ValidationRowsFallback": int(
                    fallback[DATE_COLUMN].isin(validation_dates).sum()
                ),
                "Protocol": "EXPANDING_WINDOW_FIXED_20_OPERATING_DATE_VALIDATION",
            }
        )

    fold_definition = pd.DataFrame(fold_records)
    validation_date_set = set(date_to_fold)

    # =========================================================================
    # CANDIDATE DEFINITIONS
    # =========================================================================

    main_candidate_names = [
        "ZERO",
        "NAIVE_LAG_1",
        "NAIVE_LAG_2",
        "NAIVE_LAG_3",
        "NAIVE_LAG_5",
        "NAIVE_LAG_10",
        "NAIVE_LAG_20",
        "ROLLING_MEAN_3",
        "ROLLING_MEAN_5",
        "ROLLING_MEAN_10",
        "ROLLING_MEAN_20",
        "ROLLING_MEDIAN_3",
        "ROLLING_MEDIAN_5",
        "ROLLING_MEDIAN_10",
        "ROLLING_MEDIAN_20",
        "EXPANDING_MEAN",
        "PRODUCT_WEEKDAY_HIERARCHY",
        "FAMILY_WEEKDAY_HIERARCHY",
        "BLEND_LAG5_MEAN5_50",
        "BLEND_LAG5_MEAN10_50",
    ]

    fallback_candidate_names = [
        "ZERO",
        "LAST_AVAILABLE",
        "NAIVE5_WITH_BACKOFF",
        "RECENT_MEAN3_WITH_BACKOFF",
        "RECENT_MEAN5_WITH_BACKOFF",
        "RECENT_MEAN10_WITH_BACKOFF",
        "RECENT_MEDIAN5_WITH_BACKOFF",
        "POSITIVE_MEAN5_WITH_BACKOFF",
        "POSITIVE_MEAN10_WITH_BACKOFF",
        "EXPANDING_MEAN_WITH_BACKOFF",
        "PRODUCT_WEEKDAY_HIERARCHY",
        "FAMILY_WEEKDAY_HIERARCHY",
    ]

    main_prediction_parts: list[pd.DataFrame] = []
    fallback_prediction_parts: list[pd.DataFrame] = []

    # =========================================================================
    # WALK-FORWARD CANDIDATE PREDICTIONS
    # =========================================================================

    for fold_row in fold_definition.itertuples(index=False):
        fold_number = int(fold_row.Fold)
        validation_start = pd.Timestamp(fold_row.ValidationStart)
        validation_end = pd.Timestamp(fold_row.ValidationEnd)

        training_frame = all_routes.loc[
            all_routes[DATE_COLUMN] < validation_start
        ].copy()

        main_validation = main_model.loc[
            main_model[DATE_COLUMN].between(
                validation_start,
                validation_end,
                inclusive="both",
            )
        ].copy()

        fallback_validation = fallback.loc[
            fallback[DATE_COLUMN].between(
                validation_start,
                validation_end,
                inclusive="both",
            )
        ].copy()

        if main_validation.empty:
            raise AssertionError(f"Fold {fold_number} has no main-model rows.")

        if fallback_validation.empty:
            raise AssertionError(f"Fold {fold_number} has no fallback rows.")

        main_product_hierarchy, main_family_hierarchy = build_hierarchy_predictions(
            training_frame,
            main_validation,
        )

        fallback_product_hierarchy, fallback_family_hierarchy = build_hierarchy_predictions(
            training_frame,
            fallback_validation,
        )

        main_candidates: dict[str, pd.Series] = {
            "ZERO": pd.Series(0.0, index=main_validation.index),
            "NAIVE_LAG_1": main_validation["NormalDemandLag_1"],
            "NAIVE_LAG_2": main_validation["NormalDemandLag_2"],
            "NAIVE_LAG_3": main_validation["NormalDemandLag_3"],
            "NAIVE_LAG_5": main_validation["NormalDemandLag_5"],
            "NAIVE_LAG_10": main_validation["NormalDemandLag_10"],
            "NAIVE_LAG_20": main_validation["NormalDemandLag_20"],
            "ROLLING_MEAN_3": main_validation["PastNormalDemandRollingMean_3"],
            "ROLLING_MEAN_5": main_validation["PastNormalDemandRollingMean_5"],
            "ROLLING_MEAN_10": main_validation["PastNormalDemandRollingMean_10"],
            "ROLLING_MEAN_20": main_validation["PastNormalDemandRollingMean_20"],
            "ROLLING_MEDIAN_3": main_validation["PastNormalDemandRollingMedian_3"],
            "ROLLING_MEDIAN_5": main_validation["PastNormalDemandRollingMedian_5"],
            "ROLLING_MEDIAN_10": main_validation["PastNormalDemandRollingMedian_10"],
            "ROLLING_MEDIAN_20": main_validation["PastNormalDemandRollingMedian_20"],
            "EXPANDING_MEAN": main_validation["ExpandingPastMeanNormalDemand"],
            "PRODUCT_WEEKDAY_HIERARCHY": main_product_hierarchy,
            "FAMILY_WEEKDAY_HIERARCHY": main_family_hierarchy,
            "BLEND_LAG5_MEAN5_50": (
                0.5 * main_validation["NormalDemandLag_5"]
                + 0.5 * main_validation["PastNormalDemandRollingMean_5"]
            ),
            "BLEND_LAG5_MEAN10_50": (
                0.5 * main_validation["NormalDemandLag_5"]
                + 0.5 * main_validation["PastNormalDemandRollingMean_10"]
            ),
        }

        if set(main_candidates) != set(main_candidate_names):
            raise AssertionError("Main candidate definition mismatch.")

        for candidate_method, prediction_values in main_candidates.items():
            output = main_validation[
                [
                    DATE_COLUMN,
                    PRODUCT_ID_COLUMN,
                    PRODUCT_NAME_COLUMN,
                    ROUTE_COLUMN,
                    FAMILY_COLUMN,
                    DAY_OF_WEEK_COLUMN,
                    TARGET_COLUMN,
                ]
            ].copy()

            output["Fold"] = fold_number
            output["CandidateMethod"] = candidate_method
            output["ActualNormalDemand"] = output.pop(TARGET_COLUMN).astype(float)
            output["PredictedNormalDemand"] = clip_prediction(prediction_values).to_numpy()
            output["ForecastError"] = (
                output["PredictedNormalDemand"]
                - output["ActualNormalDemand"]
            )
            output["AbsoluteError"] = output["ForecastError"].abs()
            main_prediction_parts.append(output)

        zero_fallback = pd.Series(0.0, index=fallback_validation.index, dtype=float)

        last_available = coalesce_series(
            fallback_validation["NormalDemandLag_1"],
            fallback_validation["NormalDemandLag_2"],
            fallback_validation["NormalDemandLag_3"],
            fallback_validation["NormalDemandLag_5"],
            fallback_validation["NormalDemandLag_10"],
            fallback_validation["NormalDemandLag_20"],
            fallback_validation["ExpandingPastMeanNormalDemand"],
            fallback_product_hierarchy,
            zero_fallback,
        )

        naive5_backoff = coalesce_series(
            fallback_validation["NormalDemandLag_5"],
            fallback_validation["NormalDemandLag_1"],
            fallback_validation["PastNormalDemandRollingMean_3"],
            fallback_validation["ExpandingPastMeanNormalDemand"],
            fallback_product_hierarchy,
            zero_fallback,
        )

        recent_mean3_backoff = coalesce_series(
            fallback_validation["PastNormalDemandRollingMean_3"],
            fallback_validation["NormalDemandLag_1"],
            fallback_validation["ExpandingPastMeanNormalDemand"],
            fallback_family_hierarchy,
            zero_fallback,
        )

        recent_mean5_backoff = coalesce_series(
            fallback_validation["PastNormalDemandRollingMean_5"],
            fallback_validation["PastNormalDemandRollingMean_3"],
            fallback_validation["NormalDemandLag_1"],
            fallback_validation["ExpandingPastMeanNormalDemand"],
            fallback_family_hierarchy,
            zero_fallback,
        )

        recent_mean10_backoff = coalesce_series(
            fallback_validation["PastNormalDemandRollingMean_10"],
            fallback_validation["PastNormalDemandRollingMean_5"],
            fallback_validation["PastNormalDemandRollingMean_3"],
            fallback_validation["ExpandingPastMeanNormalDemand"],
            fallback_family_hierarchy,
            zero_fallback,
        )

        recent_median5_backoff = coalesce_series(
            fallback_validation["PastNormalDemandRollingMedian_5"],
            fallback_validation["PastNormalDemandRollingMedian_3"],
            fallback_validation["NormalDemandLag_1"],
            fallback_validation["ExpandingPastMeanNormalDemand"],
            fallback_family_hierarchy,
            zero_fallback,
        )

        positive_mean5 = (
            fallback_validation["PastNormalDemandRollingSum_5"]
            / fallback_validation["PastPositiveNormalDemandCount_5"].replace(0, np.nan)
        )

        positive_mean10 = (
            fallback_validation["PastNormalDemandRollingSum_10"]
            / fallback_validation["PastPositiveNormalDemandCount_10"].replace(0, np.nan)
        )

        positive_mean5_backoff = coalesce_series(
            positive_mean5,
            fallback_validation["PastNormalDemandRollingMean_5"],
            fallback_validation["ExpandingPastMeanNormalDemand"],
            fallback_family_hierarchy,
            zero_fallback,
        )

        positive_mean10_backoff = coalesce_series(
            positive_mean10,
            fallback_validation["PastNormalDemandRollingMean_10"],
            fallback_validation["ExpandingPastMeanNormalDemand"],
            fallback_family_hierarchy,
            zero_fallback,
        )

        expanding_mean_backoff = coalesce_series(
            fallback_validation["ExpandingPastMeanNormalDemand"],
            fallback_product_hierarchy,
            fallback_family_hierarchy,
            zero_fallback,
        )

        fallback_candidates: dict[str, pd.Series] = {
            "ZERO": zero_fallback,
            "LAST_AVAILABLE": last_available,
            "NAIVE5_WITH_BACKOFF": naive5_backoff,
            "RECENT_MEAN3_WITH_BACKOFF": recent_mean3_backoff,
            "RECENT_MEAN5_WITH_BACKOFF": recent_mean5_backoff,
            "RECENT_MEAN10_WITH_BACKOFF": recent_mean10_backoff,
            "RECENT_MEDIAN5_WITH_BACKOFF": recent_median5_backoff,
            "POSITIVE_MEAN5_WITH_BACKOFF": positive_mean5_backoff,
            "POSITIVE_MEAN10_WITH_BACKOFF": positive_mean10_backoff,
            "EXPANDING_MEAN_WITH_BACKOFF": expanding_mean_backoff,
            "PRODUCT_WEEKDAY_HIERARCHY": fallback_product_hierarchy,
            "FAMILY_WEEKDAY_HIERARCHY": fallback_family_hierarchy,
        }

        if set(fallback_candidates) != set(fallback_candidate_names):
            raise AssertionError("Fallback candidate definition mismatch.")

        for candidate_method, prediction_values in fallback_candidates.items():
            output = fallback_validation[
                [
                    DATE_COLUMN,
                    PRODUCT_ID_COLUMN,
                    PRODUCT_NAME_COLUMN,
                    ROUTE_COLUMN,
                    FAMILY_COLUMN,
                    DAY_OF_WEEK_COLUMN,
                    TARGET_COLUMN,
                ]
            ].copy()

            output["Fold"] = fold_number
            output["CandidateMethod"] = candidate_method
            output["ActualNormalDemand"] = output.pop(TARGET_COLUMN).astype(float)
            output["PredictedNormalDemand"] = clip_prediction(prediction_values).to_numpy()
            output["ForecastError"] = (
                output["PredictedNormalDemand"]
                - output["ActualNormalDemand"]
            )
            output["AbsoluteError"] = output["ForecastError"].abs()
            fallback_prediction_parts.append(output)

    main_predictions = pd.concat(main_prediction_parts, ignore_index=True)
    fallback_predictions = pd.concat(fallback_prediction_parts, ignore_index=True)

    # =========================================================================
    # SCORE AND SELECT METHODS
    # =========================================================================

    main_metrics = score_prediction_table(
        main_predictions,
        ["CandidateMethod"],
        "DAILY_PRODUCT_MAIN_ROUTE",
    )

    main_metrics = main_metrics.sort_values(
        ["WAPEPercentage", "MAE", "AbsoluteBiasPercentage", "CandidateMethod"],
        kind="mergesort",
    ).reset_index(drop=True)
    main_metrics.insert(0, "Rank", np.arange(1, len(main_metrics) + 1))

    main_fold_metrics = score_prediction_table(
        main_predictions,
        ["CandidateMethod", "Fold"],
        "DAILY_PRODUCT_MAIN_ROUTE_FOLD",
    )

    selected_main_row = select_best_method(main_metrics)
    selected_main_method = str(selected_main_row["CandidateMethod"])

    fallback_metrics = score_prediction_table(
        fallback_predictions,
        [ROUTE_COLUMN, "CandidateMethod"],
        "DAILY_PRODUCT_FALLBACK_ROUTE",
    )

    fallback_fold_metrics = score_prediction_table(
        fallback_predictions,
        [ROUTE_COLUMN, "CandidateMethod", "Fold"],
        "DAILY_PRODUCT_FALLBACK_ROUTE_FOLD",
    )

    selected_fallback_rows = []
    selected_fallback_methods: dict[str, str] = {}

    for route_name, route_metrics in fallback_metrics.groupby(ROUTE_COLUMN, sort=True):
        selected_route_row = select_best_method(route_metrics)
        selected_method = str(selected_route_row["CandidateMethod"])
        selected_fallback_methods[str(route_name)] = selected_method

        selection_record = selected_route_row.to_dict()
        selection_record["SelectionRole"] = "FALLBACK_ROUTE_BENCHMARK"
        selected_fallback_rows.append(selection_record)

    fallback_metrics = fallback_metrics.copy()
    fallback_metrics["SelectedForRoute"] = fallback_metrics.apply(
        lambda row: (
            selected_fallback_methods.get(str(row[ROUTE_COLUMN]))
            == str(row["CandidateMethod"])
        ),
        axis=1,
    )

    fallback_metrics = fallback_metrics.sort_values(
        [ROUTE_COLUMN, "WAPEPercentage", "MAE", "CandidateMethod"],
        kind="mergesort",
    ).reset_index(drop=True)

    selected_methods_records = [
        {
            "ForecastRoute": "MAIN_MODEL",
            "SelectedMethod": selected_main_method,
            "SelectionRole": "MAIN_ROUTE_DEVELOPMENT_BENCHMARK",
            "WAPEPercentage": float(selected_main_row["WAPEPercentage"]),
            "MAE": float(selected_main_row["MAE"]),
            "RMSE": float(selected_main_row["RMSE"]),
            "TotalBias": float(selected_main_row["TotalBias"]),
            "AbsoluteBiasPercentage": float(selected_main_row["AbsoluteBiasPercentage"]),
            "SelectionPeriod": "PRE_MARCH_CHRONOLOGICAL_VALIDATION",
        }
    ]

    for selected_route_row in selected_fallback_rows:
        selected_methods_records.append(
            {
                "ForecastRoute": str(selected_route_row[ROUTE_COLUMN]),
                "SelectedMethod": str(selected_route_row["CandidateMethod"]),
                "SelectionRole": "FALLBACK_ROUTE_DEVELOPMENT_BENCHMARK",
                "WAPEPercentage": float(selected_route_row["WAPEPercentage"]),
                "MAE": float(selected_route_row["MAE"]),
                "RMSE": float(selected_route_row["RMSE"]),
                "TotalBias": float(selected_route_row["TotalBias"]),
                "AbsoluteBiasPercentage": float(selected_route_row["AbsoluteBiasPercentage"]),
                "SelectionPeriod": "PRE_MARCH_CHRONOLOGICAL_VALIDATION",
            }
        )

    selected_methods = pd.DataFrame(selected_methods_records)

    # =========================================================================
    # ASSEMBLE COMPLETE ROUTED BASELINE SYSTEM
    # =========================================================================

    selected_main_predictions = main_predictions.loc[
        main_predictions["CandidateMethod"] == selected_main_method
    ].copy()

    selected_fallback_prediction_parts = []

    for route_name, selected_method in selected_fallback_methods.items():
        selected_part = fallback_predictions.loc[
            (fallback_predictions[ROUTE_COLUMN] == route_name)
            & (fallback_predictions["CandidateMethod"] == selected_method)
        ].copy()
        selected_fallback_prediction_parts.append(selected_part)

    selected_fallback_predictions = pd.concat(
        selected_fallback_prediction_parts,
        ignore_index=True,
    )

    selected_system = pd.concat(
        [selected_main_predictions, selected_fallback_predictions],
        ignore_index=True,
    )

    selected_system = selected_system.rename(
        columns={"CandidateMethod": "SelectedMethod"}
    )

    selected_system["ForecastMode"] = "ONE_STEP_DAILY_ACTUAL_HISTORY"
    selected_system["WeeklyInterpretation"] = (
        "DAILY_UPDATED_AGGREGATION_DIAGNOSTIC_NOT_WEEK_START_RECURSIVE"
    )

    selected_system = selected_system.sort_values(
        [DATE_COLUMN, PRODUCT_ID_COLUMN],
        kind="mergesort",
    ).reset_index(drop=True)

    selected_duplicate_keys = int(
        selected_system.duplicated([DATE_COLUMN, PRODUCT_ID_COLUMN]).sum()
    )

    validation_universe = all_routes.loc[
        all_routes[DATE_COLUMN].isin(validation_date_set)
    ].copy()

    expected_validation_keys = set(
        zip(
            validation_universe[DATE_COLUMN].astype("int64"),
            validation_universe[PRODUCT_ID_COLUMN].astype(str),
        )
    )

    selected_validation_keys = set(
        zip(
            selected_system[DATE_COLUMN].astype("int64"),
            selected_system[PRODUCT_ID_COLUMN].astype(str),
        )
    )

    if selected_duplicate_keys != 0:
        raise AssertionError("Duplicate keys were found in the selected routed system.")

    if expected_validation_keys != selected_validation_keys:
        raise AssertionError(
            "The selected routed system does not cover the complete validation universe."
        )

    if selected_system["PredictedNormalDemand"].isna().any():
        raise AssertionError("Missing predictions were found in the selected routed system.")

    if (selected_system["PredictedNormalDemand"] < 0).any():
        raise AssertionError("Negative predictions were found in the selected routed system.")

    # =========================================================================
    # SELECTED SYSTEM METRICS
    # =========================================================================

    selected_product_metric = metric_record(
        selected_system["ActualNormalDemand"],
        selected_system["PredictedNormalDemand"],
        "DAILY_PRODUCT_ALL_ROUTES",
    )
    selected_product_metric["Scope"] = "ALL_ROUTES"
    selected_product_metric["ForecastMode"] = "ONE_STEP_DAILY_ACTUAL_HISTORY"

    selected_route_metrics = score_prediction_table(
        selected_system,
        [ROUTE_COLUMN],
        "DAILY_PRODUCT_ROUTE",
    )

    selected_fold_metrics = score_prediction_table(
        selected_system,
        ["Fold"],
        "DAILY_PRODUCT_ALL_ROUTES_FOLD",
    )

    daily_aggregate = (
        selected_system.groupby(DATE_COLUMN, as_index=False)
        .agg(
            ActualRestaurantNormalDemand=("ActualNormalDemand", "sum"),
            PredictedRestaurantNormalDemand=("PredictedNormalDemand", "sum"),
            ProductsScored=(PRODUCT_ID_COLUMN, "nunique"),
            ProductRows=(PRODUCT_ID_COLUMN, "size"),
            Fold=("Fold", "first"),
        )
        .sort_values(DATE_COLUMN)
        .reset_index(drop=True)
    )

    daily_aggregate["ForecastError"] = (
        daily_aggregate["PredictedRestaurantNormalDemand"]
        - daily_aggregate["ActualRestaurantNormalDemand"]
    )
    daily_aggregate["AbsoluteError"] = daily_aggregate["ForecastError"].abs()

    daily_restaurant_metric = metric_record(
        daily_aggregate["ActualRestaurantNormalDemand"],
        daily_aggregate["PredictedRestaurantNormalDemand"],
        "DAILY_RESTAURANT_TOTAL",
    )
    daily_restaurant_metric["Scope"] = "ALL_ROUTES"
    daily_restaurant_metric["ForecastMode"] = "ONE_STEP_DAILY_ACTUAL_HISTORY"

    standard_week_rows = selected_system.loc[
        selected_system[DAY_OF_WEEK_COLUMN].between(0, 4)
    ].copy()

    standard_week_rows["WeekStart"] = (
        standard_week_rows[DATE_COLUMN]
        - pd.to_timedelta(
            standard_week_rows[DAY_OF_WEEK_COLUMN],
            unit="D",
        )
    )

    weekly_calendar = (
        standard_week_rows.groupby("WeekStart", as_index=False)
        .agg(
            OperatingDatesInWeek=(DATE_COLUMN, "nunique"),
            MinimumDayOfWeek=(DAY_OF_WEEK_COLUMN, "min"),
            MaximumDayOfWeek=(DAY_OF_WEEK_COLUMN, "max"),
        )
    )

    weekly_calendar["CompleteMondayToFridayWeek"] = (
        (weekly_calendar["OperatingDatesInWeek"] == 5)
        & (weekly_calendar["MinimumDayOfWeek"] == 0)
        & (weekly_calendar["MaximumDayOfWeek"] == 4)
    )

    weekly_product = (
        standard_week_rows.groupby(
            [
                "WeekStart",
                PRODUCT_ID_COLUMN,
                PRODUCT_NAME_COLUMN,
                ROUTE_COLUMN,
            ],
            as_index=False,
        )
        .agg(
            ActualWeeklyNormalDemand=("ActualNormalDemand", "sum"),
            PredictedWeeklyNormalDemand=("PredictedNormalDemand", "sum"),
            OperatingDatesContributed=(DATE_COLUMN, "nunique"),
        )
    )

    weekly_product = weekly_product.merge(
        weekly_calendar,
        on="WeekStart",
        how="left",
        validate="many_to_one",
    )

    weekly_product["ForecastError"] = (
        weekly_product["PredictedWeeklyNormalDemand"]
        - weekly_product["ActualWeeklyNormalDemand"]
    )
    weekly_product["AbsoluteError"] = weekly_product["ForecastError"].abs()
    weekly_product["ForecastMode"] = "DAILY_UPDATED_WEEKLY_AGGREGATION_DIAGNOSTIC"

    weekly_restaurant = (
        weekly_product.groupby(
            [
                "WeekStart",
                "OperatingDatesInWeek",
                "CompleteMondayToFridayWeek",
            ],
            as_index=False,
        )
        .agg(
            ActualWeeklyRestaurantNormalDemand=("ActualWeeklyNormalDemand", "sum"),
            PredictedWeeklyRestaurantNormalDemand=("PredictedWeeklyNormalDemand", "sum"),
            ProductRows=(PRODUCT_ID_COLUMN, "size"),
            ProductsScored=(PRODUCT_ID_COLUMN, "nunique"),
        )
        .sort_values("WeekStart")
        .reset_index(drop=True)
    )

    weekly_restaurant["ForecastError"] = (
        weekly_restaurant["PredictedWeeklyRestaurantNormalDemand"]
        - weekly_restaurant["ActualWeeklyRestaurantNormalDemand"]
    )
    weekly_restaurant["AbsoluteError"] = weekly_restaurant["ForecastError"].abs()
    weekly_restaurant["ForecastMode"] = "DAILY_UPDATED_WEEKLY_AGGREGATION_DIAGNOSTIC"

    weekly_product_all_metric = metric_record(
        weekly_product["ActualWeeklyNormalDemand"],
        weekly_product["PredictedWeeklyNormalDemand"],
        "WEEKLY_PRODUCT_ALL_EVALUATION_WEEKS",
    )
    weekly_product_all_metric["Scope"] = "ALL_ROUTES_STANDARD_WEEKDAYS"
    weekly_product_all_metric["ForecastMode"] = "DAILY_UPDATED_WEEKLY_AGGREGATION_DIAGNOSTIC"

    complete_weekly_product = weekly_product.loc[
        weekly_product["CompleteMondayToFridayWeek"]
    ]

    weekly_product_complete_metric = metric_record(
        complete_weekly_product["ActualWeeklyNormalDemand"],
        complete_weekly_product["PredictedWeeklyNormalDemand"],
        "WEEKLY_PRODUCT_COMPLETE_MONDAY_TO_FRIDAY_WEEKS",
    )
    weekly_product_complete_metric["Scope"] = "ALL_ROUTES_STANDARD_WEEKDAYS"
    weekly_product_complete_metric["ForecastMode"] = "DAILY_UPDATED_WEEKLY_AGGREGATION_DIAGNOSTIC"

    weekly_restaurant_all_metric = metric_record(
        weekly_restaurant["ActualWeeklyRestaurantNormalDemand"],
        weekly_restaurant["PredictedWeeklyRestaurantNormalDemand"],
        "WEEKLY_RESTAURANT_ALL_EVALUATION_WEEKS",
    )
    weekly_restaurant_all_metric["Scope"] = "ALL_ROUTES_STANDARD_WEEKDAYS"
    weekly_restaurant_all_metric["ForecastMode"] = "DAILY_UPDATED_WEEKLY_AGGREGATION_DIAGNOSTIC"

    complete_weekly_restaurant = weekly_restaurant.loc[
        weekly_restaurant["CompleteMondayToFridayWeek"]
    ]

    weekly_restaurant_complete_metric = metric_record(
        complete_weekly_restaurant["ActualWeeklyRestaurantNormalDemand"],
        complete_weekly_restaurant["PredictedWeeklyRestaurantNormalDemand"],
        "WEEKLY_RESTAURANT_COMPLETE_MONDAY_TO_FRIDAY_WEEKS",
    )
    weekly_restaurant_complete_metric["Scope"] = "ALL_ROUTES_STANDARD_WEEKDAYS"
    weekly_restaurant_complete_metric["ForecastMode"] = "DAILY_UPDATED_WEEKLY_AGGREGATION_DIAGNOSTIC"

    selected_system_metrics = pd.DataFrame(
        [
            selected_product_metric,
            daily_restaurant_metric,
            weekly_product_all_metric,
            weekly_product_complete_metric,
            weekly_restaurant_all_metric,
            weekly_restaurant_complete_metric,
        ]
    )

    # =========================================================================
    # COVERAGE AND AUDITS
    # =========================================================================

    validation_actual_total = float(selected_system["ActualNormalDemand"].sum())

    coverage = (
        selected_system.groupby(ROUTE_COLUMN, as_index=False)
        .agg(
            Rows=(PRODUCT_ID_COLUMN, "size"),
            Products=(PRODUCT_ID_COLUMN, "nunique"),
            Dates=(DATE_COLUMN, "nunique"),
            ActualNormalDemand=("ActualNormalDemand", "sum"),
        )
    )

    coverage["ActualDemandSharePercentage"] = np.where(
        validation_actual_total > 0,
        100.0 * coverage["ActualNormalDemand"] / validation_actual_total,
        np.nan,
    )

    coverage = coverage.merge(
        selected_methods[
            [ROUTE_COLUMN, "SelectedMethod"]
        ],
        on=ROUTE_COLUMN,
        how="left",
        validate="one_to_one",
    )

    input_schema_audit = pd.DataFrame(
        [
            {
                "Dataset": "ALL_ROUTES",
                "Path": str(ALL_ROUTES_PATH),
                "Rows": len(all_routes),
                "Columns": len(all_routes.columns),
                "Dates": all_routes[DATE_COLUMN].nunique(),
                "Products": all_routes[PRODUCT_ID_COLUMN].nunique(),
                "Routes": ", ".join(sorted(all_routes[ROUTE_COLUMN].unique())),
            },
            {
                "Dataset": "MAIN_MODEL",
                "Path": str(MAIN_MODEL_PATH),
                "Rows": len(main_model),
                "Columns": len(main_model.columns),
                "Dates": main_model[DATE_COLUMN].nunique(),
                "Products": main_model[PRODUCT_ID_COLUMN].nunique(),
                "Routes": ", ".join(sorted(main_model[ROUTE_COLUMN].unique())),
            },
            {
                "Dataset": "FALLBACK",
                "Path": str(FALLBACK_PATH),
                "Rows": len(fallback),
                "Columns": len(fallback.columns),
                "Dates": fallback[DATE_COLUMN].nunique(),
                "Products": fallback[PRODUCT_ID_COLUMN].nunique(),
                "Routes": ", ".join(sorted(fallback[ROUTE_COLUMN].unique())),
            },
        ]
    )

    leakage_audit = pd.DataFrame(
        [
            {
                "Check": "ND03 checkpoint hash verified",
                "Expected": EXPECTED_ND03_CHECKPOINT_SHA256,
                "Actual": nd03_checkpoint_sha256_before,
                "Passed": nd03_checkpoint_sha256_before == EXPECTED_ND03_CHECKPOINT_SHA256,
            },
            {
                "Check": "ND03 manifest hash verified",
                "Expected": expected_manifest_sha256,
                "Actual": actual_manifest_sha256,
                "Passed": actual_manifest_sha256 == expected_manifest_sha256,
            },
            {
                "Check": "All input files match ND03 manifest",
                "Expected": True,
                "Actual": bool(input_hash_audit["MatchesND03Manifest"].all()),
                "Passed": bool(input_hash_audit["MatchesND03Manifest"].all()),
            },
            {
                "Check": "Opened March rows used for selection",
                "Expected": 0,
                "Actual": int(all_routes["IsOpenedMarchDiagnosticPeriod"].astype(bool).sum()),
                "Passed": int(all_routes["IsOpenedMarchDiagnosticPeriod"].astype(bool).sum()) == 0,
            },
            {
                "Check": "Validation folds have training dates strictly before validation dates",
                "Expected": True,
                "Actual": bool((fold_definition["TrainEnd"] < fold_definition["ValidationStart"]).all()),
                "Passed": bool((fold_definition["TrainEnd"] < fold_definition["ValidationStart"]).all()),
            },
            {
                "Check": "Hierarchy statistics fitted using dates before each fold",
                "Expected": True,
                "Actual": True,
                "Passed": True,
            },
            {
                "Check": "Selected system covers every validation product-date key",
                "Expected": True,
                "Actual": expected_validation_keys == selected_validation_keys,
                "Passed": expected_validation_keys == selected_validation_keys,
            },
            {
                "Check": "Selected system duplicate keys",
                "Expected": 0,
                "Actual": selected_duplicate_keys,
                "Passed": selected_duplicate_keys == 0,
            },
            {
                "Check": "Weekly result described as week-start recursive",
                "Expected": False,
                "Actual": False,
                "Passed": True,
            },
            {
                "Check": "Advanced machine-learning models fitted",
                "Expected": False,
                "Actual": False,
                "Passed": True,
            },
        ]
    )

    validation = pd.DataFrame(
        [
            {
                "Check": "All-route row count",
                "Expected": EXPECTED_ALL_ROUTE_ROWS,
                "Actual": len(all_routes),
                "Passed": len(all_routes) == EXPECTED_ALL_ROUTE_ROWS,
            },
            {
                "Check": "Main-model row count",
                "Expected": EXPECTED_MAIN_ROWS,
                "Actual": len(main_model),
                "Passed": len(main_model) == EXPECTED_MAIN_ROWS,
            },
            {
                "Check": "Fallback row count",
                "Expected": EXPECTED_FALLBACK_ROWS,
                "Actual": len(fallback),
                "Passed": len(fallback) == EXPECTED_FALLBACK_ROWS,
            },
            {
                "Check": "Chronological folds",
                "Expected": N_FOLDS,
                "Actual": len(fold_definition),
                "Passed": len(fold_definition) == N_FOLDS,
            },
            {
                "Check": "Validation dates per fold",
                "Expected": VALIDATION_DATES_PER_FOLD,
                "Actual": int(fold_definition["ValidationOperatingDates"].min()),
                "Passed": bool((fold_definition["ValidationOperatingDates"] == VALIDATION_DATES_PER_FOLD).all()),
            },
            {
                "Check": "Main baseline candidates",
                "Expected": len(main_candidate_names),
                "Actual": main_predictions["CandidateMethod"].nunique(),
                "Passed": main_predictions["CandidateMethod"].nunique() == len(main_candidate_names),
            },
            {
                "Check": "Fallback candidates",
                "Expected": len(fallback_candidate_names),
                "Actual": fallback_predictions["CandidateMethod"].nunique(),
                "Passed": fallback_predictions["CandidateMethod"].nunique() == len(fallback_candidate_names),
            },
            {
                "Check": "Fallback routes selected",
                "Expected": len(expected_fallback_routes),
                "Actual": len(selected_fallback_methods),
                "Passed": len(selected_fallback_methods) == len(expected_fallback_routes),
            },
            {
                "Check": "Selected routed rows",
                "Expected": len(validation_universe),
                "Actual": len(selected_system),
                "Passed": len(selected_system) == len(validation_universe),
            },
            {
                "Check": "Missing selected predictions",
                "Expected": 0,
                "Actual": int(selected_system["PredictedNormalDemand"].isna().sum()),
                "Passed": int(selected_system["PredictedNormalDemand"].isna().sum()) == 0,
            },
            {
                "Check": "Negative selected predictions",
                "Expected": 0,
                "Actual": int((selected_system["PredictedNormalDemand"] < 0).sum()),
                "Passed": int((selected_system["PredictedNormalDemand"] < 0).sum()) == 0,
            },
            {
                "Check": "March target vault opened",
                "Expected": False,
                "Actual": False,
                "Passed": True,
            },
            {
                "Check": "Models fitted",
                "Expected": False,
                "Actual": False,
                "Passed": True,
            },
            {
                "Check": "ND04 step lock created",
                "Expected": False,
                "Actual": False,
                "Passed": True,
            },
        ]
    )

    if not leakage_audit["Passed"].all():
        raise AssertionError(
            "ND04 leakage/protocol audit failed:\n"
            + leakage_audit.loc[~leakage_audit["Passed"]].to_string(index=False)
        )

    if not validation["Passed"].all():
        raise AssertionError(
            "ND04 validation failed:\n"
            + validation.loc[~validation["Passed"]].to_string(index=False)
        )

    # =========================================================================
    # FIGURES
    # =========================================================================

    staged_prediction_dir = STAGING_ROOT / PREDICTION_DIR.relative_to(ND04_ROOT)
    staged_metric_dir = STAGING_ROOT / METRIC_DIR.relative_to(ND04_ROOT)
    staged_audit_dir = STAGING_ROOT / AUDIT_DIR.relative_to(ND04_ROOT)
    staged_figure_dir = STAGING_ROOT / FIGURE_DIR.relative_to(ND04_ROOT)
    staged_report_dir = STAGING_ROOT / REPORT_DIR.relative_to(ND04_ROOT)
    staged_control_dir = STAGING_ROOT / CONTROL_DIR.relative_to(ND04_ROOT)

    main_plot = main_metrics.sort_values("WAPEPercentage", ascending=True)
    plt.figure(figsize=(12, 9))
    plt.barh(main_plot["CandidateMethod"], main_plot["WAPEPercentage"])
    plt.axvline(
        float(selected_main_row["WAPEPercentage"]),
        linestyle="--",
        label=f"Selected: {selected_main_method}",
    )
    plt.title("Main-route daily baseline comparison")
    plt.xlabel("Product-day WAPE (%)")
    plt.ylabel("Baseline method")
    plt.grid(axis="x", alpha=0.3)
    plt.legend()
    save_figure(staged_figure_dir / "ND04_figure_01_main_baseline_wape.png")

    selected_fallback_plot = selected_methods.loc[
        selected_methods["ForecastRoute"] != "MAIN_MODEL"
    ].sort_values("WAPEPercentage")
    plt.figure(figsize=(10, 6))
    plt.barh(
        selected_fallback_plot["ForecastRoute"],
        selected_fallback_plot["WAPEPercentage"],
    )
    plt.title("Selected fallback benchmark WAPE by route")
    plt.xlabel("Product-day WAPE (%)")
    plt.ylabel("Fallback route")
    plt.grid(axis="x", alpha=0.3)
    save_figure(staged_figure_dir / "ND04_figure_02_selected_fallback_wape.png")

    plt.figure(figsize=(14, 6))
    plt.plot(
        daily_aggregate[DATE_COLUMN],
        daily_aggregate["ActualRestaurantNormalDemand"],
        marker="o",
        label="Actual normal demand",
    )
    plt.plot(
        daily_aggregate[DATE_COLUMN],
        daily_aggregate["PredictedRestaurantNormalDemand"],
        marker="o",
        label="Selected routed baseline",
    )
    plt.title("Daily restaurant normal demand: actual versus routed baseline")
    plt.xlabel("Operating date")
    plt.ylabel("Normal-demand units")
    plt.grid(alpha=0.3)
    plt.legend()
    plt.gca().xaxis.set_major_formatter(DateFormatter("%Y-%m-%d"))
    plt.xticks(rotation=45, ha="right")
    save_figure(staged_figure_dir / "ND04_figure_03_daily_restaurant_actual_vs_forecast.png")

    plt.figure(figsize=(14, 6))
    plt.bar(daily_aggregate[DATE_COLUMN], daily_aggregate["ForecastError"])
    plt.axhline(0, linewidth=1)
    plt.title("Daily restaurant forecast error")
    plt.xlabel("Operating date")
    plt.ylabel("Prediction minus actual")
    plt.grid(axis="y", alpha=0.3)
    plt.gca().xaxis.set_major_formatter(DateFormatter("%Y-%m-%d"))
    plt.xticks(rotation=45, ha="right")
    save_figure(staged_figure_dir / "ND04_figure_04_daily_restaurant_error.png")

    plt.figure(figsize=(14, 6))
    plt.plot(
        weekly_restaurant["WeekStart"],
        weekly_restaurant["ActualWeeklyRestaurantNormalDemand"],
        marker="o",
        label="Actual weekly normal demand",
    )
    plt.plot(
        weekly_restaurant["WeekStart"],
        weekly_restaurant["PredictedWeeklyRestaurantNormalDemand"],
        marker="o",
        label="Aggregated one-step daily baseline",
    )
    plt.title("Weekly restaurant totals from daily-updated baseline predictions")
    plt.xlabel("Week starting")
    plt.ylabel("Normal-demand units")
    plt.grid(alpha=0.3)
    plt.legend()
    plt.gca().xaxis.set_major_formatter(DateFormatter("%Y-%m-%d"))
    plt.xticks(rotation=45, ha="right")
    save_figure(staged_figure_dir / "ND04_figure_05_weekly_restaurant_actual_vs_forecast.png")

    coverage_plot = coverage.sort_values("ActualDemandSharePercentage", ascending=True)
    plt.figure(figsize=(10, 6))
    plt.barh(
        coverage_plot[ROUTE_COLUMN],
        coverage_plot["ActualDemandSharePercentage"],
    )
    plt.title("Validation normal-demand share by forecast route")
    plt.xlabel("Share of actual validation normal demand (%)")
    plt.ylabel("Forecast route")
    plt.grid(axis="x", alpha=0.3)
    save_figure(staged_figure_dir / "ND04_figure_06_route_demand_coverage.png")

    selected_main_fold_plot = main_fold_metrics.loc[
        main_fold_metrics["CandidateMethod"] == selected_main_method
    ].sort_values("Fold")
    plt.figure(figsize=(10, 6))
    plt.plot(
        selected_main_fold_plot["Fold"],
        selected_main_fold_plot["WAPEPercentage"],
        marker="o",
    )
    plt.axhline(
        float(selected_main_row["WAPEPercentage"]),
        linestyle="--",
        label="Pooled selected-method WAPE",
    )
    plt.title("Selected main baseline WAPE by chronological fold")
    plt.xlabel("Fold")
    plt.ylabel("Product-day WAPE (%)")
    plt.xticks(range(1, N_FOLDS + 1))
    plt.grid(alpha=0.3)
    plt.legend()
    save_figure(staged_figure_dir / "ND04_figure_07_selected_main_wape_by_fold.png")

    scatter_max = float(
        max(
            selected_system["ActualNormalDemand"].max(),
            selected_system["PredictedNormalDemand"].max(),
        )
    )
    scatter_limit = math.ceil(scatter_max / 10.0) * 10.0 if scatter_max > 0 else 1.0
    plt.figure(figsize=(8, 8))
    plt.scatter(
        selected_system["ActualNormalDemand"],
        selected_system["PredictedNormalDemand"],
        alpha=0.25,
    )
    plt.plot([0, scatter_limit], [0, scatter_limit], linestyle="--", label="Perfect forecast")
    plt.title("Selected routed baseline: actual versus predicted product-day demand")
    plt.xlabel("Actual normal demand")
    plt.ylabel("Predicted normal demand")
    plt.xlim(left=0)
    plt.ylim(bottom=0)
    plt.grid(alpha=0.3)
    plt.legend()
    save_figure(staged_figure_dir / "ND04_figure_08_product_day_actual_vs_predicted.png")

    level_plot = selected_system_metrics[
        ["EvaluationLevel", "WAPEPercentage"]
    ].copy()
    plt.figure(figsize=(12, 7))
    plt.barh(level_plot["EvaluationLevel"], level_plot["WAPEPercentage"])
    plt.title("Selected routed baseline WAPE by evaluation level")
    plt.xlabel("WAPE (%)")
    plt.ylabel("Evaluation level")
    plt.grid(axis="x", alpha=0.3)
    save_figure(staged_figure_dir / "ND04_figure_09_selected_system_wape_by_level.png")

    # =========================================================================
    # WRITE TABLES
    # =========================================================================

    output_frames = {
        staged_prediction_dir / MAIN_PREDICTIONS_PATH.name: main_predictions,
        staged_prediction_dir / FALLBACK_PREDICTIONS_PATH.name: fallback_predictions,
        staged_prediction_dir / SELECTED_SYSTEM_PREDICTIONS_PATH.name: selected_system,
        staged_prediction_dir / DAILY_AGGREGATE_PATH.name: daily_aggregate,
        staged_prediction_dir / WEEKLY_PRODUCT_PATH.name: weekly_product,
        staged_prediction_dir / WEEKLY_RESTAURANT_PATH.name: weekly_restaurant,
        staged_metric_dir / MAIN_METRICS_PATH.name: main_metrics,
        staged_metric_dir / MAIN_FOLD_METRICS_PATH.name: main_fold_metrics,
        staged_metric_dir / FALLBACK_METRICS_PATH.name: fallback_metrics,
        staged_metric_dir / FALLBACK_FOLD_METRICS_PATH.name: fallback_fold_metrics,
        staged_metric_dir / SELECTED_METHODS_PATH.name: selected_methods,
        staged_metric_dir / SELECTED_SYSTEM_METRICS_PATH.name: selected_system_metrics,
        staged_metric_dir / SELECTED_SYSTEM_ROUTE_METRICS_PATH.name: selected_route_metrics,
        staged_metric_dir / SELECTED_SYSTEM_FOLD_METRICS_PATH.name: selected_fold_metrics,
        staged_audit_dir / FOLD_DEFINITION_PATH.name: fold_definition,
        staged_audit_dir / INPUT_SCHEMA_PATH.name: input_schema_audit,
        staged_audit_dir / VALIDATION_PATH.name: validation,
        staged_audit_dir / LEAKAGE_AUDIT_PATH.name: leakage_audit,
        staged_audit_dir / COVERAGE_PATH.name: coverage,
        staged_audit_dir / "ND04_input_hash_audit.csv": input_hash_audit,
    }

    for output_path, output_frame in output_frames.items():
        write_csv(output_path, output_frame)

    # =========================================================================
    # DECISION AND REPORT
    # =========================================================================

    decision_payload = {
        "StepID": STEP_ID,
        "Status": STATUS,
        "CreatedUTC": NOW_UTC.isoformat(),
        "SelectionPeriod": "PRE_MARCH_CHRONOLOGICAL_VALIDATION",
        "FoldProtocol": {
            "Folds": N_FOLDS,
            "ValidationOperatingDatesPerFold": VALIDATION_DATES_PER_FOLD,
            "TrainingWindow": "EXPANDING",
            "FirstValidationDate": fold_definition["ValidationStart"].min().date().isoformat(),
            "LastValidationDate": fold_definition["ValidationEnd"].max().date().isoformat(),
        },
        "SelectedMainMethod": selected_main_method,
        "SelectedFallbackMethods": selected_fallback_methods,
        "SelectedMethods": selected_methods.to_dict(orient="records"),
        "SelectedSystemMetrics": selected_system_metrics.to_dict(orient="records"),
        "Interpretation": {
            "DailyPredictions": "ONE_STEP_USING_ACTUAL_PRIOR_HISTORY",
            "WeeklyAggregation": "DAILY_UPDATED_DIAGNOSTIC_ONLY",
            "WeekStartRecursive": False,
            "MarchUsedForSelection": False,
            "AdvancedMLModelFitted": False,
        },
        "NextStep": "ND05",
    }

    write_json(staged_report_dir / DECISION_JSON_PATH.name, decision_payload)

    selected_route_lines = []
    for row in selected_methods.itertuples(index=False):
        selected_route_lines.append(
            f"- {row.ForecastRoute}: `{row.SelectedMethod}` "
            f"(WAPE {row.WAPEPercentage:.6f}%)"
        )

    report_summary_text = f"""# ND04 Baseline and Fallback Evaluation Summary

## Status

`{STATUS}`

## Evaluation protocol

- Development data only: dates before March 2026
- Chronological folds: {N_FOLDS}
- Validation operating dates per fold: {VALIDATION_DATES_PER_FOLD}
- Training window: expanding
- March 2026 target vault opened: no
- Advanced machine-learning model fitted: no

## Selected development benchmarks

{chr(10).join(selected_route_lines)}

## Selected routed system

- Daily product WAPE: {selected_product_metric['WAPEPercentage']:.6f}%
- Daily product MAE: {selected_product_metric['MAE']:.6f}
- Daily product RMSE: {selected_product_metric['RMSE']:.6f}
- Daily product total bias: {selected_product_metric['TotalBias']:.6f}
- Daily restaurant-total WAPE: {daily_restaurant_metric['WAPEPercentage']:.6f}%
- Weekly product WAPE, all evaluation weeks: {weekly_product_all_metric['WAPEPercentage']:.6f}%
- Weekly product WAPE, complete Monday-to-Friday weeks: {weekly_product_complete_metric['WAPEPercentage']:.6f}%
- Weekly restaurant WAPE, all evaluation weeks: {weekly_restaurant_all_metric['WAPEPercentage']:.6f}%
- Weekly restaurant WAPE, complete Monday-to-Friday weeks: {weekly_restaurant_complete_metric['WAPEPercentage']:.6f}%

## Important interpretation

The weekly values in ND04 are produced by aggregating one-step daily predictions. Each daily row uses actual history available before that day. Therefore, these weekly values represent a daily-updated aggregation diagnostic.

They are not the final Monday-origin recursive weekly forecast. The week-start forecast will later generate Monday through Friday recursively without using actual demand from inside the future week.

## Purpose of ND04

The selected methods are development benchmarks. ND05 will rebuild the original hurdle-plus-naive daily architecture using the same chronological folds. Later candidate models must be compared against these benchmarks on the same dates and routes.
"""

    readme_text = f"""# ND04 Baseline and Fallback Evaluation

Status: `{STATUS}`

This folder contains chronological development benchmarks for the bulk-free normal-demand forecasting system.

## Selected main benchmark

`{selected_main_method}`

## Forecast routes

- MAIN_MODEL: selected simple benchmark
- LOW_DEMAND_FALLBACK: selected route-specific benchmark
- COLD_START_FALLBACK: selected route-specific benchmark
- ZERO_HISTORY_FALLBACK: selected route-specific benchmark

## Weekly interpretation

Weekly outputs are aggregated one-step daily diagnostics. They are not week-start recursive forecasts.

## Next step

ND05 rebuilds the original daily hurdle-plus-naive architecture and compares it with the ND04 benchmark.
"""

    write_text(staged_report_dir / REPORT_SUMMARY_PATH.name, report_summary_text)
    write_text(STAGING_ROOT / README_PATH.name, readme_text)

    # =========================================================================
    # MANIFEST AND CHECKPOINT
    # =========================================================================

    manifest_excluded_names = {
        MANIFEST_PATH.name,
        CHECKPOINT_PATH.name,
        CHECKPOINT_SHA_PATH.name,
    }

    files_for_manifest = sorted(
        path
        for path in STAGING_ROOT.rglob("*")
        if path.is_file() and path.name not in manifest_excluded_names
    )

    manifest = pd.DataFrame(
        [
            {
                "RelativePath": str(path.relative_to(STAGING_ROOT)),
                "Bytes": int(path.stat().st_size),
                "SHA256": sha256_file(path),
            }
            for path in files_for_manifest
        ]
    ).sort_values("RelativePath").reset_index(drop=True)

    staged_manifest_path = staged_control_dir / MANIFEST_PATH.name
    write_csv(staged_manifest_path, manifest)
    manifest_sha256 = sha256_file(staged_manifest_path)

    checkpoint_payload = {
        "StepID": STEP_ID,
        "Status": STATUS,
        "CreatedUTC": NOW_UTC.isoformat(),
        "CreatedLocal": NOW_LOCAL.isoformat(),
        "ModelRoot": str(MODEL_ROOT),
        "ND04Root": str(ND04_ROOT),
        "Input": {
            "ND03CheckpointPath": str(ND03_CHECKPOINT_PATH),
            "ND03CheckpointSHA256": nd03_checkpoint_sha256_before,
            "ND03ManifestPath": str(ND03_MANIFEST_PATH),
            "ND03ManifestSHA256": actual_manifest_sha256,
            "AllRouteRows": len(all_routes),
            "MainRows": len(main_model),
            "FallbackRows": len(fallback),
        },
        "FoldProtocol": {
            "Folds": N_FOLDS,
            "ValidationOperatingDatesPerFold": VALIDATION_DATES_PER_FOLD,
            "TrainingWindow": "EXPANDING",
            "ValidationRows": len(selected_system),
        },
        "Selection": {
            "MainMethod": selected_main_method,
            "FallbackMethods": selected_fallback_methods,
        },
        "SelectedSystemMetrics": selected_system_metrics.to_dict(orient="records"),
        "Coverage": coverage.to_dict(orient="records"),
        "Interpretation": {
            "DailyForecastMode": "ONE_STEP_DAILY_ACTUAL_HISTORY",
            "WeeklyForecastMode": "DAILY_UPDATED_AGGREGATION_DIAGNOSTIC",
            "WeekStartRecursiveEvaluationCompleted": False,
            "MarchUsedForSelection": False,
        },
        "Control": {
            "ManifestPath": str(MANIFEST_PATH),
            "ManifestSHA256": manifest_sha256,
            "ValidationPath": str(VALIDATION_PATH),
            "LeakageAuditPath": str(LEAKAGE_AUDIT_PATH),
            "DecisionPath": str(DECISION_JSON_PATH),
        },
        "Safety": {
            "AdvancedModelsLoaded": False,
            "AdvancedModelsFitted": False,
            "MarchTargetVaultOpened": False,
            "ND03InputsModified": False,
            "ExistingModelLocksModified": False,
            "ND04StepLockCreated": False,
            "CheckpointAndHashesCreated": True,
        },
        "ReadyForND05": True,
        "NextStep": "ND05",
    }

    staged_checkpoint_path = staged_control_dir / CHECKPOINT_PATH.name
    write_json(staged_checkpoint_path, checkpoint_payload)
    checkpoint_sha256 = sha256_file(staged_checkpoint_path)

    staged_checkpoint_sha_path = staged_control_dir / CHECKPOINT_SHA_PATH.name
    write_text(
        staged_checkpoint_sha_path,
        f"{checkpoint_sha256}  {CHECKPOINT_PATH.name}\n",
    )

    # =========================================================================
    # VERIFY PROTECTED INPUTS REMAIN UNCHANGED
    # =========================================================================

    if sha256_file(ND03_CHECKPOINT_PATH) != nd03_checkpoint_sha256_before:
        raise AssertionError("The ND03 checkpoint changed during ND04.")

    if sha256_file(ND03_MANIFEST_PATH) != actual_manifest_sha256:
        raise AssertionError("The ND03 manifest changed during ND04.")

    for row in input_hash_rows:
        if sha256_file(Path(row["InputPath"])) != row["SHA256"]:
            raise AssertionError(
                f"Protected ND03 input changed during ND04: {row['InputPath']}"
            )

    # =========================================================================
    # ATOMIC COMMIT
    # =========================================================================

    os.replace(STAGING_ROOT, ND04_ROOT)

    TOP_LEVEL_CHECKPOINT_PATH.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(CHECKPOINT_PATH, TOP_LEVEL_CHECKPOINT_PATH)
    shutil.copy2(CHECKPOINT_SHA_PATH, TOP_LEVEL_CHECKPOINT_SHA_PATH)
    make_read_only(TOP_LEVEL_CHECKPOINT_PATH)
    make_read_only(TOP_LEVEL_CHECKPOINT_SHA_PATH)

    # =========================================================================
    # MEMORY AND HANDOFF
    # =========================================================================

    selected_method_text = "\n".join(selected_route_lines)

    handoff_text = f"""# ND04 Handoff

## Current status

- Completed step: `{STEP_ID}`
- Status: `{STATUS}`
- Completed local time: `{NOW_LOCAL.isoformat()}`
- ND04 root: `{ND04_ROOT}`
- Checkpoint: `{TOP_LEVEL_CHECKPOINT_PATH}`
- Checkpoint SHA-256: `{checkpoint_sha256}`

## Chronological validation protocol

- Folds: {N_FOLDS}
- Validation operating dates per fold: {VALIDATION_DATES_PER_FOLD}
- Training window: expanding
- March 2026 used for selection: no

## Selected development benchmarks

{selected_method_text}

## Selected routed-system development result

- Daily product WAPE: {selected_product_metric['WAPEPercentage']:.6f}%
- Daily restaurant-total WAPE: {daily_restaurant_metric['WAPEPercentage']:.6f}%
- Weekly product WAPE, complete Monday-to-Friday weeks: {weekly_product_complete_metric['WAPEPercentage']:.6f}%
- Weekly restaurant WAPE, complete Monday-to-Friday weeks: {weekly_restaurant_complete_metric['WAPEPercentage']:.6f}%

## Critical interpretation

The ND04 weekly metrics aggregate one-step daily predictions that use actual prior-day history. They are daily-updated diagnostics, not Monday-origin recursive forecasts.

## Next step

`ND05` rebuilds the original occurrence-plus-positive-quantity hurdle architecture and its naive blend using the same ND04 fold dates.
"""

    workflow_section = f"""## ND04 — Chronological baselines and fallback evaluation

Status: `{STATUS}`

- Created {N_FOLDS} fixed expanding-window folds with {VALIDATION_DATES_PER_FOLD} validation operating dates each.
- Evaluated {len(main_candidate_names)} main-route baselines.
- Evaluated {len(fallback_candidate_names)} fallback candidates for each fallback route.
- Selected one development benchmark for MAIN_MODEL and each fallback route.
- Assembled a full routed baseline system covering every validation product-date row.
- Produced daily product, daily restaurant, weekly product, and weekly restaurant diagnostics.
- Kept March 2026 closed for method selection.
- Did not fit advanced machine-learning models.

Next: ND05.
"""

    decisions_section = f"""## ND04 decisions

1. Chronological model comparison will use the fixed ND04 fold definition.
2. The selected MAIN_MODEL benchmark is `{selected_main_method}`.
3. Fallback benchmarks are selected separately by route.
4. Baseline selection uses product-day WAPE first, then MAE, absolute bias percentage, RMSE, and method name for deterministic tie-breaking.
5. No forecast is rounded before scoring.
6. Bias is prediction minus actual.
7. March 2026 is not used for selection.
8. Weekly ND04 results are daily-updated aggregation diagnostics, not week-start recursive forecasts.
9. ND04 creates a checkpoint and hashes but no step lock.
"""

    metrics_section = f"""## ND04 baseline and fallback evaluation

- Selected main baseline: `{selected_main_method}`
- Daily product WAPE, all routes: {selected_product_metric['WAPEPercentage']:.6f}%
- Daily product MAE, all routes: {selected_product_metric['MAE']:.6f}
- Daily product RMSE, all routes: {selected_product_metric['RMSE']:.6f}
- Daily product total bias: {selected_product_metric['TotalBias']:.6f}
- Daily restaurant-total WAPE: {daily_restaurant_metric['WAPEPercentage']:.6f}%
- Weekly product WAPE, complete Monday-to-Friday weeks: {weekly_product_complete_metric['WAPEPercentage']:.6f}%
- Weekly restaurant WAPE, complete Monday-to-Friday weeks: {weekly_restaurant_complete_metric['WAPEPercentage']:.6f}%
- Forecast mode for weekly diagnostics: daily-updated aggregation
"""

    agents_section = f"""## ND04 authoritative status

Marker: ND04_AUTHORITATIVE_STATUS

- Status: `{STATUS}`
- Read next: `{ND04_HANDOFF_PATH}`
- Fixed fold definition: `{FOLD_DEFINITION_PATH}`
- Selected methods: `{SELECTED_METHODS_PATH}`
- Selected system predictions: `{SELECTED_SYSTEM_PREDICTIONS_PATH}`
- Main benchmark: `{selected_main_method}`
- March 2026 used for selection: no
- Weekly metrics are daily-updated aggregation diagnostics
- Next step: `ND05`
"""

    atomic_write_text(ND04_HANDOFF_PATH, handoff_text)
    atomic_write_text(CURRENT_HANDOFF_PATH, handoff_text)
    append_marked_section(WORKFLOW_PATH, "## ND04 — Chronological baselines and fallback evaluation", workflow_section)
    append_marked_section(DECISIONS_PATH, "## ND04 decisions", decisions_section)
    append_marked_section(METRICS_AND_RESULTS_PATH, "## ND04 baseline and fallback evaluation", metrics_section)
    append_marked_section(AGENTS_PATH, "Marker: ND04_AUTHORITATIVE_STATUS", agents_section)

    log_text = "\n".join(
        [
            f"Step: {STEP_ID}",
            f"Status: {STATUS}",
            f"Created local: {NOW_LOCAL.isoformat()}",
            f"ND03 checkpoint SHA256: {nd03_checkpoint_sha256_before}",
            f"ND03 manifest SHA256: {actual_manifest_sha256}",
            f"Folds: {N_FOLDS}",
            f"Validation dates per fold: {VALIDATION_DATES_PER_FOLD}",
            f"Selected main method: {selected_main_method}",
            f"Selected fallback methods: {json.dumps(selected_fallback_methods, sort_keys=True)}",
            f"Daily product WAPE: {selected_product_metric['WAPEPercentage']}",
            f"Daily restaurant WAPE: {daily_restaurant_metric['WAPEPercentage']}",
            f"Weekly product complete-week WAPE: {weekly_product_complete_metric['WAPEPercentage']}",
            f"Weekly restaurant complete-week WAPE: {weekly_restaurant_complete_metric['WAPEPercentage']}",
            f"Checkpoint SHA256: {checkpoint_sha256}",
            "Advanced models loaded: False",
            "Advanced models fitted: False",
            "March target vault opened: False",
            "ND03 inputs modified: False",
            "ND04 step lock created: False",
            "Ready for ND05: True",
            "",
        ]
    )

    atomic_write_text(LOG_PATH, log_text)

except Exception:
    if STAGING_ROOT.exists():
        shutil.rmtree(STAGING_ROOT)
    raise


# =============================================================================
# FINAL OUTPUT
# =============================================================================

print("=" * 110)
print("EDEN NORMAL-DEMAND MODEL V2 — ND04 COMPLETE")
print("=" * 110)

print(f"Status: {STATUS}")
print(f"Local time: {NOW_LOCAL.isoformat()}")
print(f"ND04 root: {ND04_ROOT}")

print("\nINPUT VERIFICATION")
print(f"ND03 checkpoint SHA-256: {nd03_checkpoint_sha256_before}")
print(f"ND03 manifest SHA-256: {actual_manifest_sha256}")
print(f"All-route rows: {len(all_routes):,}")
print(f"Main-model rows: {len(main_model):,}")
print(f"Fallback rows: {len(fallback):,}")
print("March target vault opened: False")
print("ND03 inputs modified: False")

print("\nCHRONOLOGICAL VALIDATION")
print(f"Folds: {N_FOLDS}")
print(f"Validation operating dates per fold: {VALIDATION_DATES_PER_FOLD}")
print(f"First validation date: {fold_definition['ValidationStart'].min().date()}")
print(f"Last validation date: {fold_definition['ValidationEnd'].max().date()}")
print(f"Selected-system validation rows: {len(selected_system):,}")

print("\nSELECTED DEVELOPMENT BENCHMARKS")
print(selected_methods.to_string(index=False))

print("\nSELECTED ROUTED SYSTEM — DAILY PRODUCT")
print(f"Rows: {selected_product_metric['Observations']:,}")
print(f"Actual total: {selected_product_metric['ActualTotal']:,.6f}")
print(f"Predicted total: {selected_product_metric['PredictedTotal']:,.6f}")
print(f"MAE: {selected_product_metric['MAE']:.6f}")
print(f"RMSE: {selected_product_metric['RMSE']:.6f}")
print(f"WAPE: {selected_product_metric['WAPEPercentage']:.6f}%")
print(f"Mean bias: {selected_product_metric['MeanBias']:.6f}")
print(f"Total bias: {selected_product_metric['TotalBias']:.6f}")

print("\nSELECTED ROUTED SYSTEM — DAILY RESTAURANT TOTAL")
print(f"Operating dates: {daily_restaurant_metric['Observations']}")
print(f"MAE: {daily_restaurant_metric['MAE']:.6f}")
print(f"RMSE: {daily_restaurant_metric['RMSE']:.6f}")
print(f"WAPE: {daily_restaurant_metric['WAPEPercentage']:.6f}%")
print(f"Total bias: {daily_restaurant_metric['TotalBias']:.6f}")

print("\nWEEKLY AGGREGATION DIAGNOSTIC")
print("Forecast mode: DAILY_UPDATED_AGGREGATION_DIAGNOSTIC")
print("Week-start recursive forecast: False")
print(f"All evaluation weeks: {weekly_restaurant_all_metric['Observations']}")
print(f"Complete Monday-to-Friday weeks: {weekly_restaurant_complete_metric['Observations']}")
print(f"Weekly product WAPE, all weeks: {weekly_product_all_metric['WAPEPercentage']:.6f}%")
print(f"Weekly product WAPE, complete weeks: {weekly_product_complete_metric['WAPEPercentage']:.6f}%")
print(f"Weekly restaurant WAPE, all weeks: {weekly_restaurant_all_metric['WAPEPercentage']:.6f}%")
print(f"Weekly restaurant WAPE, complete weeks: {weekly_restaurant_complete_metric['WAPEPercentage']:.6f}%")

print("\nVALIDATION DEMAND COVERAGE BY ROUTE")
print(coverage.to_string(index=False))

print("\nOUTPUTS")
print(f"- Fold definition: {FOLD_DEFINITION_PATH}")
print(f"- Main predictions: {MAIN_PREDICTIONS_PATH}")
print(f"- Fallback predictions: {FALLBACK_PREDICTIONS_PATH}")
print(f"- Selected system predictions: {SELECTED_SYSTEM_PREDICTIONS_PATH}")
print(f"- Main metrics: {MAIN_METRICS_PATH}")
print(f"- Fallback metrics: {FALLBACK_METRICS_PATH}")
print(f"- Selected methods: {SELECTED_METHODS_PATH}")
print(f"- Selected system metrics: {SELECTED_SYSTEM_METRICS_PATH}")
print(f"- Figures: {FIGURE_DIR}")
print(f"- Report summary: {REPORT_SUMMARY_PATH}")
print(f"- Validation: {VALIDATION_PATH}")
print(f"- Manifest: {MANIFEST_PATH}")
print(f"- Checkpoint: {TOP_LEVEL_CHECKPOINT_PATH}")
print(f"- Checkpoint SHA-256: {checkpoint_sha256}")
print(f"- Agent handoff: {ND04_HANDOFF_PATH}")

print("\nSAFETY")
print("- Advanced models loaded: False")
print("- Advanced models fitted/refitted: False")
print("- March target vault opened: False")
print("- ND03 inputs modified: False")
print("- Existing model locks modified: False")
print("- ND04 step lock created: False")
print("- ND04 checkpoint and hashes created: True")

print("\nNEXT STEP")
print(
    "ND05 — rebuild the original daily occurrence/positive-quantity hurdle model "
    "and naive blend using the same fixed chronological folds."
)

print("=" * 110)

EDEN NORMAL-DEMAND MODEL V2 — ND04 COMPLETE
Status: ND04_BASELINES_AND_FALLBACKS_EVALUATED_READY_FOR_ND05
Local time: 2026-08-07T22:57:59.861183+01:00
ND04 root: /Users/ryansmac/Desktop/Meng Project/eden_datasets/eden_normal_demand_model_v2/03_models/00_candidates/ND04_baseline_and_fallback_evaluation

INPUT VERIFICATION
ND03 checkpoint SHA-256: 0845af89a5b459ca13ae6ffd99dde444f5010f6c0fb5ba4c34a4f091ac2e151c
ND03 manifest SHA-256: faa7f60b99f2c48cd110786a560969ddccac5c9e6da871f4dd41f46a553ce654
All-route rows: 23,763
Main-model rows: 10,976
Fallback rows: 12,787
March target vault opened: False
ND03 inputs modified: False

CHRONOLOGICAL VALIDATION
Folds: 5
Validation operating dates per fold: 20
First validation date: 2025-09-30
Last validation date: 2026-02-27
Selected-system validation rows: 8,277

SELECTED DEVELOPMENT BENCHMARKS
        ForecastRoute              SelectedMethod                        SelectionRole  WAPEPercentage      MAE      RMSE  TotalBias  AbsoluteBiasPercentag

In [6]:
from __future__ import annotations

# =============================================================================
# EDEN NORMAL-DEMAND MODEL V2
# ND05 — REBUILD AND EVALUATE THE ORIGINAL DAILY HURDLE + NAIVE5 ARCHITECTURE
#
# Purpose:
#   - Reuse the accepted ND03 53-predictor normal-demand architecture.
#   - Reuse the fixed ND04 expanding-window chronological folds.
#   - Fit, within every fold:
#       1. occurrence LogisticRegression on all main-route rows;
#       2. positive-quantity MLPRegressor on positive-demand rows only.
#   - Form the soft-hurdle expected value:
#       P(NormalDemand > 0) * predicted positive quantity.
#   - Compare pure soft hurdle, lag-5, the original 75/25 blend, nearby fixed
#     blends, and the accepted ROLLING_MEAN_5 benchmark.
#   - Combine the best ND05 main-route challenger with the already-selected
#     ND04 fallback predictions for full routed-system diagnostics.
#   - Produce daily and daily-updated weekly aggregation diagnostics.
#
# Important:
#   - This step fits candidate models inside chronological folds.
#   - It does NOT open March 2026 targets.
#   - It does NOT fit or save the final production model.
#   - It does NOT perform a Monday-origin recursive weekly forecast.
#   - It does NOT modify ND03, ND04, existing locks, or previous checkpoints.
#   - No ND05 lock is created; only a checkpoint and hashes are created.
#
# Run this as one complete Jupyter cell.
# =============================================================================

import hashlib
import json
import math
import os
import shutil
import stat
import time
import uuid
import warnings
from datetime import datetime, timezone
from pathlib import Path
from zoneinfo import ZoneInfo

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.dates import DateFormatter
from sklearn.compose import ColumnTransformer
from sklearn.exceptions import ConvergenceWarning
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    average_precision_score,
    balanced_accuracy_score,
    brier_score_loss,
    f1_score,
    log_loss,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.neural_network import MLPRegressor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler


# =============================================================================
# CONFIGURATION
# =============================================================================

PROJECT_ROOT = Path("/Users/ryansmac/Desktop/Meng Project")
EDEN_ROOT = PROJECT_ROOT / "eden_datasets"
MODEL_ROOT = EDEN_ROOT / "eden_normal_demand_model_v2"

ND03_ROOT = (
    MODEL_ROOT
    / "02_feature_engineering"
    / "ND03_normal_demand_features"
)

ND04_ROOT = (
    MODEL_ROOT
    / "03_models"
    / "00_candidates"
    / "ND04_baseline_and_fallback_evaluation"
)

MAIN_MODEL_PATH = (
    ND03_ROOT
    / "01_model_ready_datasets"
    / "ND03_pre_march_main_model_development_dataset.csv"
)

CORE_PREDICTOR_LIST_PATH = (
    ND03_ROOT
    / "03_contracts"
    / "ND03_core_predictor_list.csv"
)

CORE_FEATURE_CONTRACT_PATH = (
    ND03_ROOT
    / "03_contracts"
    / "ND03_core_feature_contract.csv"
)

ND03_MANIFEST_PATH = (
    ND03_ROOT
    / "05_control"
    / "ND03_artifact_hash_manifest.csv"
)

ND03_CHECKPOINT_PATH = (
    MODEL_ROOT
    / "08_checkpoints"
    / "ND03_checkpoint.json"
)

FOLD_DEFINITION_PATH = (
    ND04_ROOT
    / "03_audits"
    / "ND04_chronological_fold_definition.csv"
)

ND04_SELECTED_METHODS_PATH = (
    ND04_ROOT
    / "02_metrics"
    / "ND04_selected_route_methods.csv"
)

ND04_MAIN_METRICS_PATH = (
    ND04_ROOT
    / "02_metrics"
    / "ND04_main_baseline_metrics.csv"
)

ND04_SELECTED_ROUTED_PREDICTIONS_PATH = (
    ND04_ROOT
    / "01_predictions"
    / "ND04_selected_routed_system_predictions.csv"
)

ND04_MANIFEST_PATH = (
    ND04_ROOT
    / "06_control"
    / "ND04_artifact_hash_manifest.csv"
)

ND04_CHECKPOINT_PATH = (
    MODEL_ROOT
    / "08_checkpoints"
    / "ND04_checkpoint.json"
)

EXPECTED_ND03_CHECKPOINT_SHA256 = (
    "0845af89a5b459ca13ae6ffd99dde444"
    "f5010f6c0fb5ba4c34a4f091ac2e151c"
)

EXPECTED_ND04_CHECKPOINT_SHA256 = (
    "2fdc5d2c64c38f85b2669ca942042884"
    "209d80111cc840261307da98b1e9cf54"
)

EXPECTED_MAIN_DEVELOPMENT_ROWS = 10_976
EXPECTED_MAIN_VALIDATION_ROWS = 4_124
EXPECTED_VALIDATION_DATES = 100
EXPECTED_FOLDS = 5
EXPECTED_PREDICTORS = 53
EXPECTED_NUMERIC_PREDICTORS = 45
EXPECTED_CATEGORICAL_PREDICTORS = 8
EXPECTED_ND04_MAIN_BENCHMARK = "ROLLING_MEAN_5"
EXPECTED_ND04_MAIN_WAPE = 41.331256

DATE_COLUMN = "Date"
PRODUCT_ID_COLUMN = "CanonicalProductID"
PRODUCT_NAME_COLUMN = "CanonicalProductName"
TARGET_COLUMN = "NormalDemand"
ROUTE_COLUMN = "ForecastRoute"
FAMILY_COLUMN = "TierProductFamily"
DAY_OF_WEEK_COLUMN = "DayOfWeekNumber"

BINARY_PREDICTORS = [
    "IsMultiPLUCanonicalProduct",
    "IsWeekend",
    "IsConsecutiveCalendarDay",
]

HURDLE_BLEND_WEIGHTS = [0.25, 0.50, 0.75]
ORIGINAL_BLEND_WEIGHT = 0.75

OCCURRENCE_PARAMETERS = {
    "C": 1.0,
    "class_weight": "balanced",
    "max_iter": 3000,
    "solver": "lbfgs",
    "tol": 1e-5,
}

QUANTITY_PARAMETERS = {
    "activation": "relu",
    "alpha": 0.0001,
    "batch_size": 256,
    "early_stopping": True,
    "hidden_layer_sizes": (64, 32),
    "learning_rate": "adaptive",
    "learning_rate_init": 0.001,
    "max_iter": 300,
    "n_iter_no_change": 20,
    "random_state": 42,
    "solver": "adam",
    "validation_fraction": 0.1,
}

MINIMUM_WAPE_IMPROVEMENT_PP = 0.50
MAXIMUM_ACCEPTABLE_ABSOLUTE_BIAS_PERCENTAGE = 3.0
MAXIMUM_MAE_RATIO_TO_BENCHMARK = 1.02

DPI = 300
ALLOW_OVERWRITE = False
STEP_ID = "ND05"
STATUS = "ND05_ORIGINAL_HURDLE_ARCHITECTURE_EVALUATED_READY_FOR_ND06"

ND05_ROOT = (
    MODEL_ROOT
    / "03_models"
    / "00_candidates"
    / "ND05_original_hurdle_model_rebuild"
)

PREDICTION_DIR = ND05_ROOT / "01_predictions"
METRIC_DIR = ND05_ROOT / "02_metrics"
AUDIT_DIR = ND05_ROOT / "03_audits"
FIGURE_DIR = ND05_ROOT / "04_figures"
REPORT_DIR = ND05_ROOT / "05_reports"
CONTROL_DIR = ND05_ROOT / "06_control"

COMPONENT_PREDICTIONS_PATH = (
    PREDICTION_DIR / "ND05_hurdle_component_predictions.csv"
)
CANDIDATE_PREDICTIONS_PATH = (
    PREDICTION_DIR / "ND05_main_candidate_predictions.csv"
)
SELECTED_MAIN_PREDICTIONS_PATH = (
    PREDICTION_DIR / "ND05_selected_main_challenger_predictions.csv"
)
ROUTED_SYSTEM_PREDICTIONS_PATH = (
    PREDICTION_DIR / "ND05_selected_routed_system_predictions.csv"
)
DAILY_RESTAURANT_PATH = (
    PREDICTION_DIR / "ND05_selected_routed_daily_restaurant_totals.csv"
)
WEEKLY_PRODUCT_PATH = (
    PREDICTION_DIR / "ND05_selected_routed_weekly_product_totals.csv"
)
WEEKLY_RESTAURANT_PATH = (
    PREDICTION_DIR / "ND05_selected_routed_weekly_restaurant_totals.csv"
)

MAIN_METRICS_PATH = (
    METRIC_DIR / "ND05_main_candidate_metrics.csv"
)
MAIN_FOLD_METRICS_PATH = (
    METRIC_DIR / "ND05_main_candidate_metrics_by_fold.csv"
)
OCCURRENCE_METRICS_PATH = (
    METRIC_DIR / "ND05_occurrence_component_metrics_by_fold.csv"
)
QUANTITY_METRICS_PATH = (
    METRIC_DIR / "ND05_positive_quantity_metrics_by_fold.csv"
)
COMPARISON_PATH = (
    METRIC_DIR / "ND05_hurdle_vs_baseline_comparison.csv"
)
ROUTED_METRICS_PATH = (
    METRIC_DIR / "ND05_selected_routed_system_metrics.csv"
)
ROUTED_ROUTE_METRICS_PATH = (
    METRIC_DIR / "ND05_selected_routed_metrics_by_route.csv"
)

MODEL_CONFIG_PATH = (
    AUDIT_DIR / "ND05_original_architecture_configuration.json"
)
INPUT_HASH_AUDIT_PATH = (
    AUDIT_DIR / "ND05_input_hash_audit.csv"
)
FOLD_TRAINING_AUDIT_PATH = (
    AUDIT_DIR / "ND05_fold_training_audit.csv"
)
PROTOCOL_AUDIT_PATH = (
    AUDIT_DIR / "ND05_leakage_and_protocol_audit.csv"
)
VALIDATION_PATH = (
    AUDIT_DIR / "ND05_validation_summary.csv"
)

DECISION_PATH = (
    REPORT_DIR / "ND05_original_hurdle_architecture_decision.json"
)
REPORT_SUMMARY_PATH = (
    REPORT_DIR / "ND05_original_hurdle_architecture_summary.md"
)
README_PATH = ND05_ROOT / "README.md"

MANIFEST_PATH = (
    CONTROL_DIR / "ND05_artifact_hash_manifest.csv"
)
CHECKPOINT_PATH = CONTROL_DIR / "ND05_checkpoint.json"
CHECKPOINT_SHA_PATH = CONTROL_DIR / "ND05_checkpoint.sha256"

TOP_LEVEL_CHECKPOINT_PATH = (
    MODEL_ROOT / "08_checkpoints" / "ND05_checkpoint.json"
)
TOP_LEVEL_CHECKPOINT_SHA_PATH = (
    MODEL_ROOT / "08_checkpoints" / "ND05_checkpoint.sha256"
)

MEMORY_ROOT = MODEL_ROOT / "00_project_memory"
AGENTS_PATH = MODEL_ROOT / "AGENTS.md"
CURRENT_HANDOFF_PATH = MEMORY_ROOT / "CURRENT_HANDOFF.md"
ND05_HANDOFF_PATH = MEMORY_ROOT / "ND05_HANDOFF.md"
WORKFLOW_PATH = MEMORY_ROOT / "WORKFLOW.md"
DECISIONS_PATH = MEMORY_ROOT / "DECISIONS.md"
METRICS_AND_RESULTS_PATH = MEMORY_ROOT / "METRICS_AND_RESULTS.md"
LOG_PATH = MODEL_ROOT / "09_logs" / "ND05_hurdle_rebuild_log.txt"

NOW_UTC = datetime.now(timezone.utc)
NOW_LOCAL = NOW_UTC.astimezone(ZoneInfo("Europe/Dublin"))


# =============================================================================
# HELPERS
# =============================================================================

def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


def write_csv(path: Path, frame: pd.DataFrame) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    frame.to_csv(path, index=False)


def write_json(path: Path, payload: dict) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(
        json.dumps(payload, indent=2, ensure_ascii=False, default=str) + "\n",
        encoding="utf-8",
    )


def write_text(path: Path, text: str) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(text, encoding="utf-8")


def atomic_write_text(path: Path, text: str) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary_path = path.with_name(f".{path.name}.{uuid.uuid4().hex}.tmp")
    temporary_path.write_text(text, encoding="utf-8")
    os.replace(temporary_path, path)


def append_marked_section(path: Path, marker: str, section_text: str) -> None:
    existing = path.read_text(encoding="utf-8") if path.is_file() else ""
    if marker in existing:
        return
    separator = "\n" if existing.endswith("\n") else "\n\n"
    atomic_write_text(path, existing + separator + section_text.strip() + "\n")


def make_read_only(path: Path) -> None:
    if path.is_file():
        path.chmod(stat.S_IRUSR | stat.S_IRGRP | stat.S_IROTH)


def validate_required_columns(
    frame: pd.DataFrame,
    required_columns: set[str],
    frame_name: str,
) -> None:
    missing = sorted(required_columns - set(frame.columns))
    if missing:
        raise AssertionError(
            f"{frame_name} is missing required columns:\n"
            + "\n".join(f"- {column}" for column in missing)
        )


def safe_wape(actual: pd.Series, predicted: pd.Series) -> float:
    actual_array = pd.to_numeric(actual, errors="coerce").to_numpy(dtype=float)
    predicted_array = pd.to_numeric(predicted, errors="coerce").to_numpy(dtype=float)
    denominator = float(np.sum(actual_array))
    if denominator == 0:
        return float("nan")
    return float(
        100.0
        * np.sum(np.abs(predicted_array - actual_array))
        / denominator
    )


def metric_record(
    actual: pd.Series,
    predicted: pd.Series,
    evaluation_level: str,
) -> dict:
    actual_array = pd.to_numeric(actual, errors="coerce").to_numpy(dtype=float)
    predicted_array = pd.to_numeric(predicted, errors="coerce").to_numpy(dtype=float)
    errors = predicted_array - actual_array
    absolute_errors = np.abs(errors)
    actual_total = float(np.sum(actual_array))
    predicted_total = float(np.sum(predicted_array))
    total_bias = float(np.sum(errors))
    absolute_bias_percentage = (
        100.0 * abs(total_bias) / actual_total
        if actual_total != 0
        else float("nan")
    )
    return {
        "EvaluationLevel": evaluation_level,
        "Observations": int(len(actual_array)),
        "ActualTotal": actual_total,
        "PredictedTotal": predicted_total,
        "MAE": float(np.mean(absolute_errors)),
        "RMSE": float(np.sqrt(np.mean(np.square(errors)))),
        "WAPEPercentage": safe_wape(actual, predicted),
        "MeanBias": float(np.mean(errors)),
        "TotalBias": total_bias,
        "AbsoluteBiasPercentage": float(absolute_bias_percentage),
    }


def score_prediction_table(
    prediction_frame: pd.DataFrame,
    group_columns: list[str],
    evaluation_level: str,
) -> pd.DataFrame:
    records: list[dict] = []
    grouped = prediction_frame.groupby(group_columns, dropna=False, sort=True)
    for group_values, group_frame in grouped:
        if not isinstance(group_values, tuple):
            group_values = (group_values,)
        record = {
            column: value
            for column, value in zip(group_columns, group_values)
        }
        record.update(
            metric_record(
                group_frame["ActualNormalDemand"],
                group_frame["PredictedNormalDemand"],
                evaluation_level,
            )
        )
        records.append(record)
    return pd.DataFrame(records)


def select_best_method(metrics: pd.DataFrame) -> pd.Series:
    sortable = metrics.copy()
    sortable["WAPESelectionValue"] = sortable["WAPEPercentage"].fillna(np.inf)
    sortable["MAESelectionValue"] = sortable["MAE"].fillna(np.inf)
    sortable["BiasSelectionValue"] = sortable[
        "AbsoluteBiasPercentage"
    ].fillna(np.inf)
    sortable["RMSESelectionValue"] = sortable["RMSE"].fillna(np.inf)
    sortable = sortable.sort_values(
        [
            "WAPESelectionValue",
            "MAESelectionValue",
            "BiasSelectionValue",
            "RMSESelectionValue",
            "CandidateMethod",
        ],
        kind="mergesort",
    )
    return sortable.iloc[0]


def clip_prediction(values: pd.Series | np.ndarray) -> np.ndarray:
    output = np.asarray(values, dtype=float)
    return np.clip(output, 0.0, None)


def prepare_model_frame(
    frame: pd.DataFrame,
    predictors: list[str],
    numeric_predictors: list[str],
    categorical_predictors: list[str],
    binary_predictors: list[str],
) -> pd.DataFrame:
    output = frame[predictors].copy()

    continuous_numeric = [
        column
        for column in numeric_predictors
        if column not in binary_predictors
    ]

    for column in continuous_numeric:
        output[column] = pd.to_numeric(output[column], errors="coerce")

    for column in binary_predictors:
        mapped = output[column].map(
            {True: 1.0, False: 0.0, "True": 1.0, "False": 0.0}
        )
        output[column] = pd.to_numeric(mapped, errors="coerce")

    for column in categorical_predictors:
        output[column] = (
            output[column]
            .astype("string")
            .fillna("__NOT_APPLICABLE_OR_MISSING__")
            .astype(str)
        )

    return output


def build_preprocessor(
    numeric_predictors: list[str],
    categorical_predictors: list[str],
    binary_predictors: list[str],
) -> ColumnTransformer:
    continuous_numeric = [
        column
        for column in numeric_predictors
        if column not in binary_predictors
    ]

    numeric_pipeline = Pipeline(
        steps=[
            (
                "median_imputer",
                SimpleImputer(strategy="median", add_indicator=True),
            ),
            ("standard_scaler", StandardScaler()),
        ]
    )

    categorical_pipeline = Pipeline(
        steps=[
            (
                "missing_category_imputer",
                SimpleImputer(
                    strategy="constant",
                    fill_value="__NOT_APPLICABLE_OR_MISSING__",
                ),
            ),
            (
                "one_hot_encoder",
                OneHotEncoder(
                    handle_unknown="ignore",
                    sparse_output=False,
                ),
            ),
        ]
    )

    return ColumnTransformer(
        transformers=[
            ("numeric", numeric_pipeline, continuous_numeric),
            ("categorical", categorical_pipeline, categorical_predictors),
            ("binary", "passthrough", binary_predictors),
        ],
        remainder="drop",
        sparse_threshold=0.0,
    )


def build_occurrence_pipeline(
    numeric_predictors: list[str],
    categorical_predictors: list[str],
    binary_predictors: list[str],
) -> Pipeline:
    return Pipeline(
        steps=[
            (
                "preprocessor",
                build_preprocessor(
                    numeric_predictors,
                    categorical_predictors,
                    binary_predictors,
                ),
            ),
            (
                "model",
                LogisticRegression(**OCCURRENCE_PARAMETERS),
            ),
        ]
    )


def build_quantity_pipeline(
    numeric_predictors: list[str],
    categorical_predictors: list[str],
    binary_predictors: list[str],
) -> Pipeline:
    return Pipeline(
        steps=[
            (
                "preprocessor",
                build_preprocessor(
                    numeric_predictors,
                    categorical_predictors,
                    binary_predictors,
                ),
            ),
            (
                "model",
                MLPRegressor(**QUANTITY_PARAMETERS),
            ),
        ]
    )


def save_figure(path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    plt.tight_layout()
    plt.savefig(path, dpi=DPI, bbox_inches="tight")
    plt.close()


def week_start_from_date(date_series: pd.Series) -> pd.Series:
    parsed = pd.to_datetime(date_series, errors="raise")
    return parsed - pd.to_timedelta(parsed.dt.dayofweek, unit="D")


# =============================================================================
# PREFLIGHT
# =============================================================================

required_inputs = [
    MAIN_MODEL_PATH,
    CORE_PREDICTOR_LIST_PATH,
    CORE_FEATURE_CONTRACT_PATH,
    ND03_MANIFEST_PATH,
    ND03_CHECKPOINT_PATH,
    FOLD_DEFINITION_PATH,
    ND04_SELECTED_METHODS_PATH,
    ND04_MAIN_METRICS_PATH,
    ND04_SELECTED_ROUTED_PREDICTIONS_PATH,
    ND04_MANIFEST_PATH,
    ND04_CHECKPOINT_PATH,
    AGENTS_PATH,
    CURRENT_HANDOFF_PATH,
    WORKFLOW_PATH,
    DECISIONS_PATH,
    METRICS_AND_RESULTS_PATH,
]

missing_inputs = [path for path in required_inputs if not path.is_file()]
if missing_inputs:
    raise FileNotFoundError(
        "ND05 required input files are missing:\n"
        + "\n".join(f"- {path}" for path in missing_inputs)
    )

nd03_checkpoint_sha256_before = sha256_file(ND03_CHECKPOINT_PATH)
nd04_checkpoint_sha256_before = sha256_file(ND04_CHECKPOINT_PATH)

if nd03_checkpoint_sha256_before != EXPECTED_ND03_CHECKPOINT_SHA256:
    raise AssertionError(
        "The ND03 checkpoint hash does not match the accepted checkpoint.\n"
        f"Expected: {EXPECTED_ND03_CHECKPOINT_SHA256}\n"
        f"Actual:   {nd03_checkpoint_sha256_before}"
    )

if nd04_checkpoint_sha256_before != EXPECTED_ND04_CHECKPOINT_SHA256:
    raise AssertionError(
        "The ND04 checkpoint hash does not match the accepted checkpoint.\n"
        f"Expected: {EXPECTED_ND04_CHECKPOINT_SHA256}\n"
        f"Actual:   {nd04_checkpoint_sha256_before}"
    )

with ND03_CHECKPOINT_PATH.open("r", encoding="utf-8") as handle:
    nd03_checkpoint = json.load(handle)

with ND04_CHECKPOINT_PATH.open("r", encoding="utf-8") as handle:
    nd04_checkpoint = json.load(handle)

if not bool(nd03_checkpoint.get("ReadyForND04", False)):
    raise AssertionError("The ND03 checkpoint is not marked ready for ND04.")

if not bool(nd04_checkpoint.get("ReadyForND05", False)):
    raise AssertionError("The ND04 checkpoint is not marked ready for ND05.")

expected_nd03_manifest_sha256 = nd03_checkpoint["Control"]["ManifestSHA256"]
actual_nd03_manifest_sha256 = sha256_file(ND03_MANIFEST_PATH)

if actual_nd03_manifest_sha256 != expected_nd03_manifest_sha256:
    raise AssertionError(
        "The ND03 manifest hash does not match the ND03 checkpoint.\n"
        f"Expected: {expected_nd03_manifest_sha256}\n"
        f"Actual:   {actual_nd03_manifest_sha256}"
    )

expected_nd04_manifest_sha256 = nd04_checkpoint["Control"]["ManifestSHA256"]
actual_nd04_manifest_sha256 = sha256_file(ND04_MANIFEST_PATH)

if actual_nd04_manifest_sha256 != expected_nd04_manifest_sha256:
    raise AssertionError(
        "The ND04 manifest hash does not match the ND04 checkpoint.\n"
        f"Expected: {expected_nd04_manifest_sha256}\n"
        f"Actual:   {actual_nd04_manifest_sha256}"
    )

if ND05_ROOT.exists():
    if not ALLOW_OVERWRITE:
        raise FileExistsError(
            "The ND05 output folder already exists. No files were changed:\n"
            f"{ND05_ROOT}"
        )
    shutil.rmtree(ND05_ROOT)

for top_level_path in [
    TOP_LEVEL_CHECKPOINT_PATH,
    TOP_LEVEL_CHECKPOINT_SHA_PATH,
]:
    if top_level_path.exists():
        if not ALLOW_OVERWRITE:
            raise FileExistsError(
                "An ND05 top-level checkpoint already exists. No files were changed:\n"
                f"{top_level_path}"
            )
        top_level_path.unlink()

STAGING_ROOT = ND05_ROOT.parent / f".ND05_staging_{uuid.uuid4().hex}"
STAGING_ROOT.mkdir(parents=True, exist_ok=False)


# =============================================================================
# LOAD AND VERIFY INPUTS
# =============================================================================

try:
    main_model = pd.read_csv(MAIN_MODEL_PATH, low_memory=False)
    predictor_register = pd.read_csv(CORE_PREDICTOR_LIST_PATH, low_memory=False)
    feature_contract = pd.read_csv(CORE_FEATURE_CONTRACT_PATH, low_memory=False)
    nd03_manifest = pd.read_csv(ND03_MANIFEST_PATH, low_memory=False)
    fold_definition = pd.read_csv(FOLD_DEFINITION_PATH, low_memory=False)
    nd04_selected_methods = pd.read_csv(
        ND04_SELECTED_METHODS_PATH,
        low_memory=False,
    )
    nd04_main_metrics = pd.read_csv(ND04_MAIN_METRICS_PATH, low_memory=False)
    nd04_selected_routed = pd.read_csv(
        ND04_SELECTED_ROUTED_PREDICTIONS_PATH,
        low_memory=False,
    )
    nd04_manifest = pd.read_csv(ND04_MANIFEST_PATH, low_memory=False)

    required_main_columns = {
        DATE_COLUMN,
        PRODUCT_ID_COLUMN,
        PRODUCT_NAME_COLUMN,
        ROUTE_COLUMN,
        FAMILY_COLUMN,
        DAY_OF_WEEK_COLUMN,
        TARGET_COLUMN,
        "NormalDemandLag_5",
        "PastNormalDemandRollingMean_5",
        "EligibleForMethodSelection",
        "IsOpenedMarchDiagnosticPeriod",
    }

    validate_required_columns(
        main_model,
        required_main_columns,
        "ND03 pre-March main-model dataset",
    )

    main_model[DATE_COLUMN] = pd.to_datetime(
        main_model[DATE_COLUMN],
        errors="raise",
    )
    main_model[TARGET_COLUMN] = pd.to_numeric(
        main_model[TARGET_COLUMN],
        errors="raise",
    ).astype(float)

    if len(main_model) != EXPECTED_MAIN_DEVELOPMENT_ROWS:
        raise AssertionError(
            "Unexpected ND03 main-model development row count.\n"
            f"Expected: {EXPECTED_MAIN_DEVELOPMENT_ROWS}\n"
            f"Actual:   {len(main_model)}"
        )

    if set(main_model[ROUTE_COLUMN].astype(str).unique()) != {"MAIN_MODEL"}:
        raise AssertionError("The ND03 main-model dataset contains non-main routes.")

    if not main_model["EligibleForMethodSelection"].astype(bool).all():
        raise AssertionError(
            "The main-model development dataset contains rows not eligible for method selection."
        )

    if main_model["IsOpenedMarchDiagnosticPeriod"].astype(bool).any():
        raise AssertionError("Opened March rows were found in ND05 development data.")

    predictor_register = predictor_register.sort_values("PredictorOrder").reset_index(drop=True)
    core_predictors = predictor_register["Predictor"].astype(str).tolist()
    numeric_predictors = predictor_register.loc[
        predictor_register["PredictorType"].astype(str) == "NUMERIC",
        "Predictor",
    ].astype(str).tolist()
    categorical_predictors = predictor_register.loc[
        predictor_register["PredictorType"].astype(str) == "CATEGORICAL",
        "Predictor",
    ].astype(str).tolist()

    if len(core_predictors) != EXPECTED_PREDICTORS:
        raise AssertionError(
            f"Expected {EXPECTED_PREDICTORS} core predictors; found {len(core_predictors)}."
        )

    if len(numeric_predictors) != EXPECTED_NUMERIC_PREDICTORS:
        raise AssertionError(
            f"Expected {EXPECTED_NUMERIC_PREDICTORS} numeric predictors; "
            f"found {len(numeric_predictors)}."
        )

    if len(categorical_predictors) != EXPECTED_CATEGORICAL_PREDICTORS:
        raise AssertionError(
            f"Expected {EXPECTED_CATEGORICAL_PREDICTORS} categorical predictors; "
            f"found {len(categorical_predictors)}."
        )

    missing_core_predictors = sorted(set(core_predictors) - set(main_model.columns))
    if missing_core_predictors:
        raise AssertionError(
            "The main-model dataset is missing core predictors:\n"
            + "\n".join(f"- {column}" for column in missing_core_predictors)
        )

    if TARGET_COLUMN in core_predictors:
        raise AssertionError("NormalDemand is incorrectly included as a predictor.")

    forbidden_predictors = {
        "BulkDemand",
        "TotalDemand",
        "IsObservedProductDate",
        "IsZeroDemandRow",
        "DemandRecordSource",
    }
    forbidden_present = sorted(forbidden_predictors & set(core_predictors))
    if forbidden_present:
        raise AssertionError(
            "Forbidden leakage or bulk fields appear in the predictor set:\n"
            + "\n".join(f"- {column}" for column in forbidden_present)
        )

    missing_binary_predictors = sorted(set(BINARY_PREDICTORS) - set(numeric_predictors))
    if missing_binary_predictors:
        raise AssertionError(
            "The expected original-architecture binary predictors are absent:\n"
            + "\n".join(f"- {column}" for column in missing_binary_predictors)
        )

    fold_date_columns = [
        "TrainStart",
        "TrainEnd",
        "ValidationStart",
        "ValidationEnd",
    ]
    for column in fold_date_columns:
        fold_definition[column] = pd.to_datetime(
            fold_definition[column],
            errors="raise",
        )

    if len(fold_definition) != EXPECTED_FOLDS:
        raise AssertionError(
            f"Expected {EXPECTED_FOLDS} ND04 folds; found {len(fold_definition)}."
        )

    if not (
        fold_definition["TrainEnd"]
        < fold_definition["ValidationStart"]
    ).all():
        raise AssertionError(
            "At least one ND04 fold has training dates that are not strictly before validation dates."
        )

    selected_main_rows = nd04_selected_methods.loc[
        nd04_selected_methods["ForecastRoute"].astype(str) == "MAIN_MODEL"
    ]
    if len(selected_main_rows) != 1:
        raise AssertionError("Expected exactly one selected ND04 main-route method.")

    nd04_selected_main_method = str(
        selected_main_rows.iloc[0]["SelectedMethod"]
    )
    nd04_selected_main_wape = float(
        selected_main_rows.iloc[0]["WAPEPercentage"]
    )

    if nd04_selected_main_method != EXPECTED_ND04_MAIN_BENCHMARK:
        raise AssertionError(
            "The accepted ND04 main benchmark differs from the expected benchmark.\n"
            f"Expected: {EXPECTED_ND04_MAIN_BENCHMARK}\n"
            f"Actual:   {nd04_selected_main_method}"
        )

    if not math.isclose(
        nd04_selected_main_wape,
        EXPECTED_ND04_MAIN_WAPE,
        rel_tol=0.0,
        abs_tol=1e-6,
    ):
        raise AssertionError(
            "The accepted ND04 main benchmark WAPE differs from the expected value.\n"
            f"Expected: {EXPECTED_ND04_MAIN_WAPE:.12f}\n"
            f"Actual:   {nd04_selected_main_wape:.12f}"
        )

    # Verify critical inputs against the ND03 and ND04 manifests.
    nd03_manifest_lookup = {
        str(row["RelativePath"]): str(row["SHA256"])
        for _, row in nd03_manifest.iterrows()
    }
    nd04_manifest_lookup = {
        str(row["RelativePath"]): str(row["SHA256"])
        for _, row in nd04_manifest.iterrows()
    }

    input_hash_records: list[dict] = []

    for path, root, lookup, source_step in [
        (MAIN_MODEL_PATH, ND03_ROOT, nd03_manifest_lookup, "ND03"),
        (CORE_PREDICTOR_LIST_PATH, ND03_ROOT, nd03_manifest_lookup, "ND03"),
        (CORE_FEATURE_CONTRACT_PATH, ND03_ROOT, nd03_manifest_lookup, "ND03"),
        (FOLD_DEFINITION_PATH, ND04_ROOT, nd04_manifest_lookup, "ND04"),
        (ND04_SELECTED_METHODS_PATH, ND04_ROOT, nd04_manifest_lookup, "ND04"),
        (ND04_MAIN_METRICS_PATH, ND04_ROOT, nd04_manifest_lookup, "ND04"),
        (
            ND04_SELECTED_ROUTED_PREDICTIONS_PATH,
            ND04_ROOT,
            nd04_manifest_lookup,
            "ND04",
        ),
    ]:
        relative_path = str(path.relative_to(root))
        actual_hash = sha256_file(path)
        expected_hash = lookup.get(relative_path)
        if expected_hash is None:
            raise AssertionError(
                f"{source_step} manifest does not list required input: {relative_path}"
            )
        if actual_hash != expected_hash:
            raise AssertionError(
                f"{source_step} input hash differs from its manifest.\n"
                f"File: {path}\n"
                f"Expected: {expected_hash}\n"
                f"Actual:   {actual_hash}"
            )
        input_hash_records.append(
            {
                "SourceStep": source_step,
                "InputPath": str(path),
                "RelativePath": relative_path,
                "SHA256": actual_hash,
                "MatchesManifest": True,
            }
        )

    input_hash_audit = pd.DataFrame(input_hash_records)

    protected_input_hashes_before = {
        str(path): sha256_file(path)
        for path in required_inputs
        if path.is_file()
    }

    # =========================================================================
    # FIT THE ORIGINAL ARCHITECTURE INSIDE FIXED CHRONOLOGICAL FOLDS
    # =========================================================================

    candidate_prediction_parts: list[pd.DataFrame] = []
    component_prediction_parts: list[pd.DataFrame] = []
    occurrence_metric_records: list[dict] = []
    quantity_metric_records: list[dict] = []
    fold_training_records: list[dict] = []

    total_fit_start = time.perf_counter()

    for fold_row in fold_definition.itertuples(index=False):
        fold_number = int(fold_row.Fold)
        validation_start = pd.Timestamp(fold_row.ValidationStart)
        validation_end = pd.Timestamp(fold_row.ValidationEnd)

        train_mask = main_model[DATE_COLUMN] < validation_start
        validation_mask = main_model[DATE_COLUMN].between(
            validation_start,
            validation_end,
            inclusive="both",
        )

        train_frame = main_model.loc[train_mask].copy()
        validation_frame = main_model.loc[validation_mask].copy()

        if train_frame.empty:
            raise AssertionError(f"Fold {fold_number} has no training rows.")
        if validation_frame.empty:
            raise AssertionError(f"Fold {fold_number} has no validation rows.")
        if train_frame[DATE_COLUMN].max() >= validation_frame[DATE_COLUMN].min():
            raise AssertionError(
                f"Fold {fold_number} violates the chronological boundary."
            )

        positive_train_frame = train_frame.loc[
            train_frame[TARGET_COLUMN] > 0
        ].copy()
        positive_validation_frame = validation_frame.loc[
            validation_frame[TARGET_COLUMN] > 0
        ].copy()

        occurrence_target_train = (
            train_frame[TARGET_COLUMN] > 0
        ).astype(int)
        occurrence_target_validation = (
            validation_frame[TARGET_COLUMN] > 0
        ).astype(int)

        if occurrence_target_train.nunique() != 2:
            raise AssertionError(
                f"Fold {fold_number} occurrence training target does not contain both classes."
            )
        if len(positive_train_frame) < 2:
            raise AssertionError(
                f"Fold {fold_number} has insufficient positive rows for the quantity model."
            )

        X_train = prepare_model_frame(
            train_frame,
            core_predictors,
            numeric_predictors,
            categorical_predictors,
            BINARY_PREDICTORS,
        )
        X_validation = prepare_model_frame(
            validation_frame,
            core_predictors,
            numeric_predictors,
            categorical_predictors,
            BINARY_PREDICTORS,
        )
        X_positive_train = prepare_model_frame(
            positive_train_frame,
            core_predictors,
            numeric_predictors,
            categorical_predictors,
            BINARY_PREDICTORS,
        )

        occurrence_pipeline = build_occurrence_pipeline(
            numeric_predictors,
            categorical_predictors,
            BINARY_PREDICTORS,
        )
        quantity_pipeline = build_quantity_pipeline(
            numeric_predictors,
            categorical_predictors,
            BINARY_PREDICTORS,
        )

        fold_fit_start = time.perf_counter()
        convergence_messages: list[str] = []

        with warnings.catch_warnings(record=True) as caught_warnings:
            warnings.simplefilter("always", ConvergenceWarning)
            occurrence_pipeline.fit(X_train, occurrence_target_train)
            quantity_pipeline.fit(
                X_positive_train,
                positive_train_frame[TARGET_COLUMN].to_numpy(dtype=float),
            )

            for warning_record in caught_warnings:
                if issubclass(warning_record.category, ConvergenceWarning):
                    convergence_messages.append(str(warning_record.message))

        fold_fit_seconds = time.perf_counter() - fold_fit_start

        occurrence_probability = occurrence_pipeline.predict_proba(
            X_validation
        )[:, 1]
        occurrence_probability = np.clip(occurrence_probability, 0.0, 1.0)

        positive_quantity_prediction = clip_prediction(
            quantity_pipeline.predict(X_validation)
        )
        soft_hurdle_prediction = clip_prediction(
            occurrence_probability * positive_quantity_prediction
        )
        lag5_prediction = clip_prediction(
            validation_frame["NormalDemandLag_5"]
        )
        rolling_mean5_prediction = clip_prediction(
            validation_frame["PastNormalDemandRollingMean_5"]
        )

        component_frame = pd.DataFrame(
            {
                "Date": validation_frame[DATE_COLUMN].to_numpy(),
                "CanonicalProductID": validation_frame[
                    PRODUCT_ID_COLUMN
                ].astype(str).to_numpy(),
                "CanonicalProductName": validation_frame[
                    PRODUCT_NAME_COLUMN
                ].astype(str).to_numpy(),
                "ForecastRoute": "MAIN_MODEL",
                "TierProductFamily": validation_frame[FAMILY_COLUMN].to_numpy(),
                "DayOfWeekNumber": validation_frame[
                    DAY_OF_WEEK_COLUMN
                ].to_numpy(),
                "Fold": fold_number,
                "ActualNormalDemand": validation_frame[
                    TARGET_COLUMN
                ].to_numpy(dtype=float),
                "OccurrenceActual": occurrence_target_validation.to_numpy(dtype=int),
                "OccurrenceProbability": occurrence_probability,
                "OccurrenceClassAt050": (
                    occurrence_probability >= 0.5
                ).astype(int),
                "PositiveQuantityPrediction": positive_quantity_prediction,
                "SoftHurdlePrediction": soft_hurdle_prediction,
                "NaiveLag5Prediction": lag5_prediction,
                "RollingMean5BenchmarkPrediction": rolling_mean5_prediction,
                "ForecastMode": "ONE_STEP_DAILY_ACTUAL_HISTORY",
                "WeeklyInterpretation": (
                    "DAILY_UPDATED_AGGREGATION_DIAGNOSTIC_NOT_WEEK_START_RECURSIVE"
                ),
            }
        )

        component_prediction_parts.append(component_frame)

        candidate_arrays: dict[str, np.ndarray] = {
            "ROLLING_MEAN_5_BENCHMARK": rolling_mean5_prediction,
            "NAIVE_LAG_5": lag5_prediction,
            "SOFT_HURDLE_LOGISTIC_MLP": soft_hurdle_prediction,
        }

        for hurdle_weight in HURDLE_BLEND_WEIGHTS:
            method_name = (
                "BLEND_HURDLE_NAIVE5__H"
                f"{int(round(hurdle_weight * 100)):02d}"
            )
            candidate_arrays[method_name] = clip_prediction(
                hurdle_weight * soft_hurdle_prediction
                + (1.0 - hurdle_weight) * lag5_prediction
            )

        for candidate_method, predicted_values in candidate_arrays.items():
            output = component_frame[
                [
                    "Date",
                    "CanonicalProductID",
                    "CanonicalProductName",
                    "ForecastRoute",
                    "TierProductFamily",
                    "DayOfWeekNumber",
                    "Fold",
                    "ActualNormalDemand",
                    "ForecastMode",
                    "WeeklyInterpretation",
                ]
            ].copy()
            output["CandidateMethod"] = candidate_method
            output["PredictedNormalDemand"] = predicted_values
            output["ForecastError"] = (
                output["PredictedNormalDemand"]
                - output["ActualNormalDemand"]
            )
            output["AbsoluteError"] = output["ForecastError"].abs()
            candidate_prediction_parts.append(output)

        # Occurrence diagnostics.
        occurrence_actual_array = occurrence_target_validation.to_numpy(dtype=int)
        occurrence_class_array = (
            occurrence_probability >= 0.5
        ).astype(int)
        occurrence_record = {
            "Fold": fold_number,
            "ValidationRows": int(len(validation_frame)),
            "PositiveValidationRows": int(occurrence_actual_array.sum()),
            "PositiveRate": float(occurrence_actual_array.mean()),
            "BrierScore": float(
                brier_score_loss(
                    occurrence_actual_array,
                    occurrence_probability,
                )
            ),
            "LogLoss": float(
                log_loss(
                    occurrence_actual_array,
                    occurrence_probability,
                    labels=[0, 1],
                )
            ),
            "BalancedAccuracyAt050": float(
                balanced_accuracy_score(
                    occurrence_actual_array,
                    occurrence_class_array,
                )
            ),
            "PrecisionAt050": float(
                precision_score(
                    occurrence_actual_array,
                    occurrence_class_array,
                    zero_division=0,
                )
            ),
            "RecallAt050": float(
                recall_score(
                    occurrence_actual_array,
                    occurrence_class_array,
                    zero_division=0,
                )
            ),
            "F1At050": float(
                f1_score(
                    occurrence_actual_array,
                    occurrence_class_array,
                    zero_division=0,
                )
            ),
            "ROCAUC": (
                float(
                    roc_auc_score(
                        occurrence_actual_array,
                        occurrence_probability,
                    )
                )
                if np.unique(occurrence_actual_array).size == 2
                else float("nan")
            ),
            "AveragePrecision": float(
                average_precision_score(
                    occurrence_actual_array,
                    occurrence_probability,
                )
            ),
        }
        occurrence_metric_records.append(occurrence_record)

        # Quantity-model diagnostics on positive validation rows only.
        positive_mask_validation = occurrence_actual_array == 1
        quantity_actual_positive = validation_frame.loc[
            positive_mask_validation,
            TARGET_COLUMN,
        ].reset_index(drop=True)
        quantity_predicted_positive = pd.Series(
            positive_quantity_prediction[positive_mask_validation]
        )
        quantity_record = {
            "Fold": fold_number,
            **metric_record(
                quantity_actual_positive,
                quantity_predicted_positive,
                "POSITIVE_QUANTITY_VALIDATION_ROWS",
            ),
        }
        quantity_metric_records.append(quantity_record)

        occurrence_model = occurrence_pipeline.named_steps["model"]
        quantity_model = quantity_pipeline.named_steps["model"]
        transformed_feature_count_occurrence = int(
            occurrence_pipeline.named_steps["preprocessor"]
            .transform(X_validation.iloc[:1])
            .shape[1]
        )
        transformed_feature_count_quantity = int(
            quantity_pipeline.named_steps["preprocessor"]
            .transform(X_validation.iloc[:1])
            .shape[1]
        )

        fold_training_records.append(
            {
                "Fold": fold_number,
                "TrainStart": train_frame[DATE_COLUMN].min(),
                "TrainEnd": train_frame[DATE_COLUMN].max(),
                "ValidationStart": validation_frame[DATE_COLUMN].min(),
                "ValidationEnd": validation_frame[DATE_COLUMN].max(),
                "TrainingRows": int(len(train_frame)),
                "PositiveTrainingRows": int(len(positive_train_frame)),
                "OccurrenceTrainingPositiveRate": float(
                    occurrence_target_train.mean()
                ),
                "ValidationRows": int(len(validation_frame)),
                "PositiveValidationRows": int(len(positive_validation_frame)),
                "OccurrenceTransformedFeatureCount": (
                    transformed_feature_count_occurrence
                ),
                "QuantityTransformedFeatureCount": (
                    transformed_feature_count_quantity
                ),
                "OccurrenceIterations": int(occurrence_model.n_iter_[0]),
                "OccurrenceConverged": bool(
                    occurrence_model.n_iter_[0]
                    < OCCURRENCE_PARAMETERS["max_iter"]
                ),
                "QuantityIterations": int(quantity_model.n_iter_),
                "QuantityConvergedOrEarlyStopped": bool(
                    quantity_model.n_iter_
                    < QUANTITY_PARAMETERS["max_iter"]
                ),
                "ConvergenceWarningCount": int(len(convergence_messages)),
                "ConvergenceMessages": " | ".join(convergence_messages),
                "FitSeconds": float(fold_fit_seconds),
                "TrainingStrictlyBeforeValidation": bool(
                    train_frame[DATE_COLUMN].max()
                    < validation_frame[DATE_COLUMN].min()
                ),
            }
        )

    total_fit_seconds = time.perf_counter() - total_fit_start

    component_predictions = pd.concat(
        component_prediction_parts,
        ignore_index=True,
    ).sort_values(
        ["Date", "CanonicalProductID"]
    ).reset_index(drop=True)

    candidate_predictions = pd.concat(
        candidate_prediction_parts,
        ignore_index=True,
    ).sort_values(
        ["CandidateMethod", "Date", "CanonicalProductID"]
    ).reset_index(drop=True)

    occurrence_metrics = pd.DataFrame(occurrence_metric_records)
    quantity_metrics = pd.DataFrame(quantity_metric_records)
    fold_training_audit = pd.DataFrame(fold_training_records)

    if component_predictions.duplicated(
        ["Date", "CanonicalProductID"]
    ).any():
        raise AssertionError(
            "Duplicate main validation product-date rows were produced."
        )

    main_validation_rows = int(len(component_predictions))
    validation_dates = int(component_predictions["Date"].nunique())

    if main_validation_rows != EXPECTED_MAIN_VALIDATION_ROWS:
        raise AssertionError(
            "Unexpected main validation prediction row count.\n"
            f"Expected: {EXPECTED_MAIN_VALIDATION_ROWS}\n"
            f"Actual:   {main_validation_rows}"
        )

    if validation_dates != EXPECTED_VALIDATION_DATES:
        raise AssertionError(
            "Unexpected validation operating-date count.\n"
            f"Expected: {EXPECTED_VALIDATION_DATES}\n"
            f"Actual:   {validation_dates}"
        )

    # =========================================================================
    # SCORE AND SELECT THE BEST ORIGINAL-ARCHITECTURE CHALLENGER
    # =========================================================================

    main_metrics = score_prediction_table(
        candidate_predictions,
        ["CandidateMethod"],
        "DAILY_PRODUCT_MAIN_ROUTE",
    )

    main_metrics = main_metrics.sort_values(
        [
            "WAPEPercentage",
            "MAE",
            "AbsoluteBiasPercentage",
            "RMSE",
            "CandidateMethod",
        ],
        kind="mergesort",
    ).reset_index(drop=True)
    main_metrics.insert(0, "Rank", np.arange(1, len(main_metrics) + 1))

    main_fold_metrics = score_prediction_table(
        candidate_predictions,
        ["CandidateMethod", "Fold"],
        "DAILY_PRODUCT_MAIN_ROUTE_BY_FOLD",
    )

    architecture_methods = [
        "SOFT_HURDLE_LOGISTIC_MLP",
        *[
            "BLEND_HURDLE_NAIVE5__H"
            f"{int(round(weight * 100)):02d}"
            for weight in HURDLE_BLEND_WEIGHTS
        ],
    ]

    architecture_metrics = main_metrics.loc[
        main_metrics["CandidateMethod"].isin(architecture_methods)
    ].copy()

    selected_architecture_row = select_best_method(architecture_metrics)
    selected_architecture_method = str(
        selected_architecture_row["CandidateMethod"]
    )

    benchmark_row = main_metrics.loc[
        main_metrics["CandidateMethod"] == "ROLLING_MEAN_5_BENCHMARK"
    ]
    if len(benchmark_row) != 1:
        raise AssertionError("Expected exactly one ND05 benchmark row.")
    benchmark_row = benchmark_row.iloc[0]

    original_h75_row = main_metrics.loc[
        main_metrics["CandidateMethod"] == "BLEND_HURDLE_NAIVE5__H75"
    ]
    if len(original_h75_row) != 1:
        raise AssertionError("The exact original H75 blend result is missing.")
    original_h75_row = original_h75_row.iloc[0]

    selected_wape_improvement_pp = float(
        benchmark_row["WAPEPercentage"]
        - selected_architecture_row["WAPEPercentage"]
    )
    selected_relative_wape_improvement_percentage = float(
        100.0
        * selected_wape_improvement_pp
        / benchmark_row["WAPEPercentage"]
    )
    selected_mae_ratio = float(
        selected_architecture_row["MAE"]
        / benchmark_row["MAE"]
    )

    selected_architecture_qualifies = bool(
        selected_wape_improvement_pp >= MINIMUM_WAPE_IMPROVEMENT_PP
        and selected_mae_ratio <= MAXIMUM_MAE_RATIO_TO_BENCHMARK
        and float(selected_architecture_row["AbsoluteBiasPercentage"])
        <= MAXIMUM_ACCEPTABLE_ABSOLUTE_BIAS_PERCENTAGE
    )

    development_recommendation = (
        "ADVANCE_SELECTED_HURDLE_ARCHITECTURE_AS_ND06_CHALLENGER"
        if selected_architecture_qualifies
        else "ROLLING_MEAN_5_REMAINS_PRIMARY_BENCHMARK_FOR_ND06"
    )

    comparison = pd.DataFrame(
        [
            {
                "ComparisonRole": "ACCEPTED_ND04_MAIN_BENCHMARK",
                "CandidateMethod": "ROLLING_MEAN_5_BENCHMARK",
                "WAPEPercentage": float(benchmark_row["WAPEPercentage"]),
                "MAE": float(benchmark_row["MAE"]),
                "RMSE": float(benchmark_row["RMSE"]),
                "TotalBias": float(benchmark_row["TotalBias"]),
                "AbsoluteBiasPercentage": float(
                    benchmark_row["AbsoluteBiasPercentage"]
                ),
                "WAPEImprovementVersusBenchmarkPP": 0.0,
                "RelativeWAPEImprovementPercentage": 0.0,
                "MAERatioToBenchmark": 1.0,
                "QualifiesAgainstAcceptanceRule": True,
            },
            {
                "ComparisonRole": "EXACT_ORIGINAL_H75_REBUILD",
                "CandidateMethod": "BLEND_HURDLE_NAIVE5__H75",
                "WAPEPercentage": float(original_h75_row["WAPEPercentage"]),
                "MAE": float(original_h75_row["MAE"]),
                "RMSE": float(original_h75_row["RMSE"]),
                "TotalBias": float(original_h75_row["TotalBias"]),
                "AbsoluteBiasPercentage": float(
                    original_h75_row["AbsoluteBiasPercentage"]
                ),
                "WAPEImprovementVersusBenchmarkPP": float(
                    benchmark_row["WAPEPercentage"]
                    - original_h75_row["WAPEPercentage"]
                ),
                "RelativeWAPEImprovementPercentage": float(
                    100.0
                    * (
                        benchmark_row["WAPEPercentage"]
                        - original_h75_row["WAPEPercentage"]
                    )
                    / benchmark_row["WAPEPercentage"]
                ),
                "MAERatioToBenchmark": float(
                    original_h75_row["MAE"] / benchmark_row["MAE"]
                ),
                "QualifiesAgainstAcceptanceRule": bool(
                    (
                        benchmark_row["WAPEPercentage"]
                        - original_h75_row["WAPEPercentage"]
                    )
                    >= MINIMUM_WAPE_IMPROVEMENT_PP
                    and float(original_h75_row["MAE"] / benchmark_row["MAE"])
                    <= MAXIMUM_MAE_RATIO_TO_BENCHMARK
                    and float(original_h75_row["AbsoluteBiasPercentage"])
                    <= MAXIMUM_ACCEPTABLE_ABSOLUTE_BIAS_PERCENTAGE
                ),
            },
            {
                "ComparisonRole": "BEST_ND05_HURDLE_ARCHITECTURE",
                "CandidateMethod": selected_architecture_method,
                "WAPEPercentage": float(
                    selected_architecture_row["WAPEPercentage"]
                ),
                "MAE": float(selected_architecture_row["MAE"]),
                "RMSE": float(selected_architecture_row["RMSE"]),
                "TotalBias": float(selected_architecture_row["TotalBias"]),
                "AbsoluteBiasPercentage": float(
                    selected_architecture_row["AbsoluteBiasPercentage"]
                ),
                "WAPEImprovementVersusBenchmarkPP": (
                    selected_wape_improvement_pp
                ),
                "RelativeWAPEImprovementPercentage": (
                    selected_relative_wape_improvement_percentage
                ),
                "MAERatioToBenchmark": selected_mae_ratio,
                "QualifiesAgainstAcceptanceRule": (
                    selected_architecture_qualifies
                ),
            },
        ]
    )

    selected_main_predictions = candidate_predictions.loc[
        candidate_predictions["CandidateMethod"]
        == selected_architecture_method
    ].copy()
    selected_main_predictions = selected_main_predictions.rename(
        columns={"CandidateMethod": "SelectedMethod"}
    )

    # =========================================================================
    # COMBINE ND05 MAIN CHALLENGER WITH ND04 FALLBACK PREDICTIONS
    # =========================================================================

    nd04_selected_routed["Date"] = pd.to_datetime(
        nd04_selected_routed["Date"],
        errors="raise",
    )

    fallback_selected_predictions = nd04_selected_routed.loc[
        nd04_selected_routed["ForecastRoute"].astype(str) != "MAIN_MODEL"
    ].copy()

    required_selected_columns = {
        "Date",
        "CanonicalProductID",
        "CanonicalProductName",
        "ForecastRoute",
        "TierProductFamily",
        "DayOfWeekNumber",
        "Fold",
        "SelectedMethod",
        "ActualNormalDemand",
        "PredictedNormalDemand",
        "ForecastMode",
        "WeeklyInterpretation",
    }
    validate_required_columns(
        fallback_selected_predictions,
        required_selected_columns,
        "ND04 selected fallback prediction rows",
    )

    main_routed = selected_main_predictions[
        [
            "Date",
            "CanonicalProductID",
            "CanonicalProductName",
            "ForecastRoute",
            "TierProductFamily",
            "DayOfWeekNumber",
            "Fold",
            "SelectedMethod",
            "ActualNormalDemand",
            "PredictedNormalDemand",
            "ForecastMode",
            "WeeklyInterpretation",
        ]
    ].copy()

    fallback_routed = fallback_selected_predictions[
        [
            "Date",
            "CanonicalProductID",
            "CanonicalProductName",
            "ForecastRoute",
            "TierProductFamily",
            "DayOfWeekNumber",
            "Fold",
            "SelectedMethod",
            "ActualNormalDemand",
            "PredictedNormalDemand",
            "ForecastMode",
            "WeeklyInterpretation",
        ]
    ].copy()

    routed_predictions = pd.concat(
        [main_routed, fallback_routed],
        ignore_index=True,
    ).sort_values(
        ["Date", "CanonicalProductID"]
    ).reset_index(drop=True)

    routed_predictions["ForecastError"] = (
        routed_predictions["PredictedNormalDemand"]
        - routed_predictions["ActualNormalDemand"]
    )
    routed_predictions["AbsoluteError"] = (
        routed_predictions["ForecastError"].abs()
    )

    if routed_predictions.duplicated(
        ["Date", "CanonicalProductID"]
    ).any():
        raise AssertionError(
            "The combined ND05 routed system contains duplicate product-date rows."
        )

    expected_routed_rows = int(len(nd04_selected_routed))
    if len(routed_predictions) != expected_routed_rows:
        raise AssertionError(
            "The ND05 routed system does not preserve the ND04 validation row count.\n"
            f"Expected: {expected_routed_rows}\n"
            f"Actual:   {len(routed_predictions)}"
        )

    routed_metrics_records = []
    routed_metrics_records.append(
        metric_record(
            routed_predictions["ActualNormalDemand"],
            routed_predictions["PredictedNormalDemand"],
            "DAILY_PRODUCT_ALL_ROUTES",
        )
    )

    daily_restaurant = (
        routed_predictions.groupby("Date", as_index=False)
        .agg(
            ActualNormalDemand=("ActualNormalDemand", "sum"),
            PredictedNormalDemand=("PredictedNormalDemand", "sum"),
            ProductRows=("CanonicalProductID", "size"),
            ProductsScored=("CanonicalProductID", "nunique"),
        )
        .sort_values("Date")
        .reset_index(drop=True)
    )
    daily_restaurant["ForecastError"] = (
        daily_restaurant["PredictedNormalDemand"]
        - daily_restaurant["ActualNormalDemand"]
    )
    daily_restaurant["AbsoluteError"] = daily_restaurant["ForecastError"].abs()

    routed_metrics_records.append(
        metric_record(
            daily_restaurant["ActualNormalDemand"],
            daily_restaurant["PredictedNormalDemand"],
            "DAILY_RESTAURANT_TOTAL_ALL_ROUTES",
        )
    )

    routed_predictions["WeekStart"] = week_start_from_date(
        routed_predictions["Date"]
    )

    weekly_product = (
        routed_predictions.groupby(
            ["WeekStart", "CanonicalProductID", "CanonicalProductName"],
            as_index=False,
        )
        .agg(
            ActualNormalDemand=("ActualNormalDemand", "sum"),
            PredictedNormalDemand=("PredictedNormalDemand", "sum"),
            ProductDateRows=("Date", "size"),
            DistinctOperatingDates=("Date", "nunique"),
        )
        .sort_values(["WeekStart", "CanonicalProductID"])
        .reset_index(drop=True)
    )
    weekly_product["ForecastError"] = (
        weekly_product["PredictedNormalDemand"]
        - weekly_product["ActualNormalDemand"]
    )
    weekly_product["AbsoluteError"] = weekly_product["ForecastError"].abs()

    week_completeness = (
        routed_predictions.groupby("WeekStart", as_index=False)
        .agg(
            DistinctOperatingDates=("Date", "nunique"),
            MinimumDayOfWeek=("DayOfWeekNumber", "min"),
            MaximumDayOfWeek=("DayOfWeekNumber", "max"),
        )
    )
    week_completeness["IsCompleteMondayToFridayWeek"] = (
        (week_completeness["DistinctOperatingDates"] == 5)
        & (week_completeness["MinimumDayOfWeek"] == 0)
        & (week_completeness["MaximumDayOfWeek"] == 4)
    )

    weekly_product = weekly_product.merge(
        week_completeness[["WeekStart", "IsCompleteMondayToFridayWeek"]],
        on="WeekStart",
        how="left",
        validate="many_to_one",
    )

    weekly_restaurant = (
        weekly_product.groupby("WeekStart", as_index=False)
        .agg(
            ActualNormalDemand=("ActualNormalDemand", "sum"),
            PredictedNormalDemand=("PredictedNormalDemand", "sum"),
            ProductWeekRows=("CanonicalProductID", "size"),
            ProductsScored=("CanonicalProductID", "nunique"),
            IsCompleteMondayToFridayWeek=(
                "IsCompleteMondayToFridayWeek",
                "first",
            ),
        )
        .sort_values("WeekStart")
        .reset_index(drop=True)
    )
    weekly_restaurant["ForecastError"] = (
        weekly_restaurant["PredictedNormalDemand"]
        - weekly_restaurant["ActualNormalDemand"]
    )
    weekly_restaurant["AbsoluteError"] = (
        weekly_restaurant["ForecastError"].abs()
    )

    routed_metrics_records.append(
        metric_record(
            weekly_product["ActualNormalDemand"],
            weekly_product["PredictedNormalDemand"],
            "WEEKLY_PRODUCT_DAILY_UPDATED_AGGREGATION_ALL_WEEKS",
        )
    )
    routed_metrics_records.append(
        metric_record(
            weekly_product.loc[
                weekly_product["IsCompleteMondayToFridayWeek"],
                "ActualNormalDemand",
            ],
            weekly_product.loc[
                weekly_product["IsCompleteMondayToFridayWeek"],
                "PredictedNormalDemand",
            ],
            "WEEKLY_PRODUCT_DAILY_UPDATED_AGGREGATION_COMPLETE_WEEKS",
        )
    )
    routed_metrics_records.append(
        metric_record(
            weekly_restaurant["ActualNormalDemand"],
            weekly_restaurant["PredictedNormalDemand"],
            "WEEKLY_RESTAURANT_DAILY_UPDATED_AGGREGATION_ALL_WEEKS",
        )
    )
    routed_metrics_records.append(
        metric_record(
            weekly_restaurant.loc[
                weekly_restaurant["IsCompleteMondayToFridayWeek"],
                "ActualNormalDemand",
            ],
            weekly_restaurant.loc[
                weekly_restaurant["IsCompleteMondayToFridayWeek"],
                "PredictedNormalDemand",
            ],
            "WEEKLY_RESTAURANT_DAILY_UPDATED_AGGREGATION_COMPLETE_WEEKS",
        )
    )

    routed_metrics = pd.DataFrame(routed_metrics_records)
    routed_metrics["SelectedMainMethod"] = selected_architecture_method
    routed_metrics["ForecastMode"] = "DAILY_UPDATED_AGGREGATION_DIAGNOSTIC"
    routed_metrics["WeekStartRecursiveForecast"] = False

    routed_route_metrics = score_prediction_table(
        routed_predictions,
        ["ForecastRoute", "SelectedMethod"],
        "DAILY_PRODUCT_BY_ROUTE",
    )

    # =========================================================================
    # AUDITS AND VALIDATION
    # =========================================================================

    transformed_feature_counts_equal = bool(
        (
            fold_training_audit["OccurrenceTransformedFeatureCount"]
            == fold_training_audit["QuantityTransformedFeatureCount"]
        ).all()
    )

    non_finite_component_values = int(
        (
            ~np.isfinite(
                component_predictions[
                    [
                        "OccurrenceProbability",
                        "PositiveQuantityPrediction",
                        "SoftHurdlePrediction",
                        "NaiveLag5Prediction",
                        "RollingMean5BenchmarkPrediction",
                    ]
                ].to_numpy(dtype=float)
            )
        ).sum()
    )

    non_finite_candidate_values = int(
        (
            ~np.isfinite(
                candidate_predictions[
                    ["ActualNormalDemand", "PredictedNormalDemand"]
                ].to_numpy(dtype=float)
            )
        ).sum()
    )

    negative_candidate_predictions = int(
        (candidate_predictions["PredictedNormalDemand"] < 0).sum()
    )

    maximum_benchmark_prediction_difference = float(
        np.max(
            np.abs(
                component_predictions["RollingMean5BenchmarkPrediction"].to_numpy(dtype=float)
                - component_predictions[
                    "RollingMean5BenchmarkPrediction"
                ].to_numpy(dtype=float)
            )
        )
    )

    nd05_benchmark_wape_difference_from_nd04 = abs(
        float(benchmark_row["WAPEPercentage"])
        - nd04_selected_main_wape
    )

    protocol_audit = pd.DataFrame(
        [
            {
                "Check": "Training dates strictly precede validation dates",
                "Expected": True,
                "Actual": bool(
                    fold_training_audit[
                        "TrainingStrictlyBeforeValidation"
                    ].all()
                ),
                "Passed": bool(
                    fold_training_audit[
                        "TrainingStrictlyBeforeValidation"
                    ].all()
                ),
                "Interpretation": "No fold trains on its validation dates",
            },
            {
                "Check": "Opened March target vault accessed",
                "Expected": False,
                "Actual": False,
                "Passed": True,
                "Interpretation": "ND05 uses pre-March development data only",
            },
            {
                "Check": "NormalDemand included as predictor",
                "Expected": False,
                "Actual": TARGET_COLUMN in core_predictors,
                "Passed": TARGET_COLUMN not in core_predictors,
                "Interpretation": "Current-row target is not a model input",
            },
            {
                "Check": "Bulk or total demand included as predictor",
                "Expected": False,
                "Actual": bool(forbidden_present),
                "Passed": not bool(forbidden_present),
                "Interpretation": "The rebuilt architecture remains bulk-free",
            },
            {
                "Check": "Quantity model trained only on positive-demand rows",
                "Expected": True,
                "Actual": bool(
                    (
                        fold_training_audit["PositiveTrainingRows"]
                        < fold_training_audit["TrainingRows"]
                    ).all()
                ),
                "Passed": bool(
                    (
                        fold_training_audit["PositiveTrainingRows"]
                        < fold_training_audit["TrainingRows"]
                    ).all()
                ),
                "Interpretation": "The positive-quantity component follows the hurdle design",
            },
            {
                "Check": "Predictions clipped at zero",
                "Expected": 0,
                "Actual": negative_candidate_predictions,
                "Passed": negative_candidate_predictions == 0,
                "Interpretation": "No negative demand forecast is retained",
            },
            {
                "Check": "Weekly metrics are marked daily-updated diagnostics",
                "Expected": False,
                "Actual": False,
                "Passed": True,
                "Interpretation": "ND05 does not claim a Monday-origin recursive result",
            },
        ]
    )

    validation = pd.DataFrame(
        [
            {
                "Check": "ND03 checkpoint hash verified",
                "Expected": EXPECTED_ND03_CHECKPOINT_SHA256,
                "Actual": nd03_checkpoint_sha256_before,
                "Passed": nd03_checkpoint_sha256_before
                == EXPECTED_ND03_CHECKPOINT_SHA256,
            },
            {
                "Check": "ND04 checkpoint hash verified",
                "Expected": EXPECTED_ND04_CHECKPOINT_SHA256,
                "Actual": nd04_checkpoint_sha256_before,
                "Passed": nd04_checkpoint_sha256_before
                == EXPECTED_ND04_CHECKPOINT_SHA256,
            },
            {
                "Check": "Core predictor count",
                "Expected": EXPECTED_PREDICTORS,
                "Actual": len(core_predictors),
                "Passed": len(core_predictors) == EXPECTED_PREDICTORS,
            },
            {
                "Check": "Numeric predictor count",
                "Expected": EXPECTED_NUMERIC_PREDICTORS,
                "Actual": len(numeric_predictors),
                "Passed": len(numeric_predictors)
                == EXPECTED_NUMERIC_PREDICTORS,
            },
            {
                "Check": "Categorical predictor count",
                "Expected": EXPECTED_CATEGORICAL_PREDICTORS,
                "Actual": len(categorical_predictors),
                "Passed": len(categorical_predictors)
                == EXPECTED_CATEGORICAL_PREDICTORS,
            },
            {
                "Check": "Chronological fold count",
                "Expected": EXPECTED_FOLDS,
                "Actual": len(fold_training_audit),
                "Passed": len(fold_training_audit) == EXPECTED_FOLDS,
            },
            {
                "Check": "Main validation rows",
                "Expected": EXPECTED_MAIN_VALIDATION_ROWS,
                "Actual": main_validation_rows,
                "Passed": main_validation_rows
                == EXPECTED_MAIN_VALIDATION_ROWS,
            },
            {
                "Check": "Validation operating dates",
                "Expected": EXPECTED_VALIDATION_DATES,
                "Actual": validation_dates,
                "Passed": validation_dates == EXPECTED_VALIDATION_DATES,
            },
            {
                "Check": "Occurrence and quantity transformed feature counts match",
                "Expected": True,
                "Actual": transformed_feature_counts_equal,
                "Passed": transformed_feature_counts_equal,
            },
            {
                "Check": "Non-finite component predictions",
                "Expected": 0,
                "Actual": non_finite_component_values,
                "Passed": non_finite_component_values == 0,
            },
            {
                "Check": "Non-finite candidate predictions",
                "Expected": 0,
                "Actual": non_finite_candidate_values,
                "Passed": non_finite_candidate_values == 0,
            },
            {
                "Check": "Negative candidate predictions",
                "Expected": 0,
                "Actual": negative_candidate_predictions,
                "Passed": negative_candidate_predictions == 0,
            },
            {
                "Check": "ND05 rolling-mean benchmark matches ND04 WAPE",
                "Expected": "<= 1e-6 difference",
                "Actual": nd05_benchmark_wape_difference_from_nd04,
                "Passed": nd05_benchmark_wape_difference_from_nd04 <= 1e-6,
            },
            {
                "Check": "Exact original H75 blend scored",
                "Expected": True,
                "Actual": "BLEND_HURDLE_NAIVE5__H75"
                in set(main_metrics["CandidateMethod"]),
                "Passed": "BLEND_HURDLE_NAIVE5__H75"
                in set(main_metrics["CandidateMethod"]),
            },
            {
                "Check": "Routed validation row count preserved",
                "Expected": expected_routed_rows,
                "Actual": len(routed_predictions),
                "Passed": len(routed_predictions) == expected_routed_rows,
            },
            {
                "Check": "Final production model fitted",
                "Expected": False,
                "Actual": False,
                "Passed": True,
            },
            {
                "Check": "ND05 step lock created",
                "Expected": False,
                "Actual": False,
                "Passed": True,
            },
        ]
    )

    if not protocol_audit["Passed"].all():
        raise AssertionError(
            "ND05 protocol audit failed:\n"
            + protocol_audit.loc[
                ~protocol_audit["Passed"]
            ].to_string(index=False)
        )

    if not validation["Passed"].all():
        raise AssertionError(
            "ND05 validation failed:\n"
            + validation.loc[~validation["Passed"]].to_string(index=False)
        )

    # =========================================================================
    # FIGURES
    # =========================================================================

    staged_prediction_dir = (
        STAGING_ROOT / PREDICTION_DIR.relative_to(ND05_ROOT)
    )
    staged_metric_dir = STAGING_ROOT / METRIC_DIR.relative_to(ND05_ROOT)
    staged_audit_dir = STAGING_ROOT / AUDIT_DIR.relative_to(ND05_ROOT)
    staged_figure_dir = STAGING_ROOT / FIGURE_DIR.relative_to(ND05_ROOT)
    staged_report_dir = STAGING_ROOT / REPORT_DIR.relative_to(ND05_ROOT)
    staged_control_dir = STAGING_ROOT / CONTROL_DIR.relative_to(ND05_ROOT)

    # Figure 1 — main candidate WAPE.
    main_plot = main_metrics.sort_values(
        "WAPEPercentage",
        ascending=True,
    )
    plt.figure(figsize=(11, 7))
    plt.barh(main_plot["CandidateMethod"], main_plot["WAPEPercentage"])
    plt.axvline(
        float(benchmark_row["WAPEPercentage"]),
        linestyle="--",
        label=f"ND04 benchmark: {float(benchmark_row['WAPEPercentage']):.2f}%",
    )
    plt.title("ND05 original hurdle architecture: daily product WAPE")
    plt.xlabel("WAPE (%)")
    plt.ylabel("Candidate method")
    plt.grid(axis="x", alpha=0.3)
    plt.legend()
    save_figure(staged_figure_dir / "ND05_figure_01_main_candidate_wape.png")

    # Figure 2 — fold stability.
    selected_fold_plot = main_fold_metrics.loc[
        main_fold_metrics["CandidateMethod"] == selected_architecture_method
    ].sort_values("Fold")
    benchmark_fold_plot = main_fold_metrics.loc[
        main_fold_metrics["CandidateMethod"] == "ROLLING_MEAN_5_BENCHMARK"
    ].sort_values("Fold")
    plt.figure(figsize=(10, 6))
    plt.plot(
        selected_fold_plot["Fold"],
        selected_fold_plot["WAPEPercentage"],
        marker="o",
        label=selected_architecture_method,
    )
    plt.plot(
        benchmark_fold_plot["Fold"],
        benchmark_fold_plot["WAPEPercentage"],
        marker="o",
        label="ROLLING_MEAN_5_BENCHMARK",
    )
    plt.title("WAPE by chronological fold")
    plt.xlabel("Fold")
    plt.ylabel("WAPE (%)")
    plt.xticks(sorted(fold_training_audit["Fold"].unique()))
    plt.grid(alpha=0.3)
    plt.legend()
    save_figure(staged_figure_dir / "ND05_figure_02_wape_by_fold.png")

    # Figure 3 — main product actual vs predicted.
    maximum_scatter = float(
        max(
            selected_main_predictions["ActualNormalDemand"].max(),
            selected_main_predictions["PredictedNormalDemand"].max(),
        )
    )
    scatter_limit = (
        math.ceil(maximum_scatter / 10.0) * 10.0
        if maximum_scatter > 0
        else 1.0
    )
    plt.figure(figsize=(8, 8))
    plt.scatter(
        selected_main_predictions["ActualNormalDemand"],
        selected_main_predictions["PredictedNormalDemand"],
        alpha=0.35,
    )
    plt.plot(
        [0, scatter_limit],
        [0, scatter_limit],
        linestyle="--",
        label="Perfect forecast",
    )
    plt.title("Selected ND05 main challenger: actual versus predicted")
    plt.xlabel("Actual daily normal demand")
    plt.ylabel("Predicted daily normal demand")
    plt.xlim(left=0)
    plt.ylim(bottom=0)
    plt.grid(alpha=0.3)
    plt.legend()
    save_figure(staged_figure_dir / "ND05_figure_03_main_actual_vs_predicted.png")

    # Figure 4 — occurrence probability distribution.
    plt.figure(figsize=(10, 6))
    plt.hist(
        component_predictions.loc[
            component_predictions["OccurrenceActual"] == 0,
            "OccurrenceProbability",
        ],
        bins=30,
        alpha=0.6,
        label="Actual zero demand",
    )
    plt.hist(
        component_predictions.loc[
            component_predictions["OccurrenceActual"] == 1,
            "OccurrenceProbability",
        ],
        bins=30,
        alpha=0.6,
        label="Actual positive demand",
    )
    plt.title("Occurrence-model probability distribution")
    plt.xlabel("Predicted probability of positive demand")
    plt.ylabel("Product-day rows")
    plt.grid(axis="y", alpha=0.3)
    plt.legend()
    save_figure(staged_figure_dir / "ND05_figure_04_occurrence_probabilities.png")

    # Figure 5 — routed daily restaurant totals.
    plt.figure(figsize=(14, 6))
    plt.plot(
        daily_restaurant["Date"],
        daily_restaurant["ActualNormalDemand"],
        label="Actual",
    )
    plt.plot(
        daily_restaurant["Date"],
        daily_restaurant["PredictedNormalDemand"],
        label="Predicted",
    )
    plt.title("Selected ND05 routed system: daily restaurant normal demand")
    plt.xlabel("Date")
    plt.ylabel("Normal-demand units")
    plt.grid(alpha=0.3)
    plt.legend()
    plt.gca().xaxis.set_major_formatter(DateFormatter("%Y-%m-%d"))
    plt.xticks(rotation=45, ha="right")
    save_figure(staged_figure_dir / "ND05_figure_05_daily_restaurant_actual_vs_forecast.png")

    # Figure 6 — routed weekly restaurant totals.
    plt.figure(figsize=(13, 6))
    plt.plot(
        weekly_restaurant["WeekStart"],
        weekly_restaurant["ActualNormalDemand"],
        marker="o",
        label="Actual",
    )
    plt.plot(
        weekly_restaurant["WeekStart"],
        weekly_restaurant["PredictedNormalDemand"],
        marker="o",
        label="Predicted",
    )
    plt.title(
        "Selected ND05 routed system: weekly restaurant totals "
        "(daily-updated diagnostic)"
    )
    plt.xlabel("Week starting")
    plt.ylabel("Normal-demand units")
    plt.grid(alpha=0.3)
    plt.legend()
    plt.gca().xaxis.set_major_formatter(DateFormatter("%Y-%m-%d"))
    plt.xticks(rotation=45, ha="right")
    save_figure(staged_figure_dir / "ND05_figure_06_weekly_restaurant_actual_vs_forecast.png")

    # Figure 7 — WAPE by evaluation level.
    plt.figure(figsize=(11, 6))
    plt.bar(
        routed_metrics["EvaluationLevel"],
        routed_metrics["WAPEPercentage"],
    )
    plt.title("Selected ND05 routed-system WAPE by evaluation level")
    plt.xlabel("Evaluation level")
    plt.ylabel("WAPE (%)")
    plt.xticks(rotation=30, ha="right")
    plt.grid(axis="y", alpha=0.3)
    save_figure(staged_figure_dir / "ND05_figure_07_routed_wape_by_level.png")

    # Figure 8 — exact original, best hurdle, and benchmark.
    plt.figure(figsize=(10, 6))
    plt.bar(
        comparison["ComparisonRole"],
        comparison["WAPEPercentage"],
    )
    plt.title("ND05 benchmark versus original and best hurdle rebuild")
    plt.xlabel("Comparison role")
    plt.ylabel("Daily product WAPE (%)")
    plt.xticks(rotation=25, ha="right")
    plt.grid(axis="y", alpha=0.3)
    save_figure(staged_figure_dir / "ND05_figure_08_key_method_comparison.png")

    # =========================================================================
    # REPORTS AND CONFIGURATION
    # =========================================================================

    model_configuration = {
        "StepID": STEP_ID,
        "Status": STATUS,
        "CreatedUTC": NOW_UTC.isoformat(),
        "Architecture": {
            "Name": "ORIGINAL_DAILY_SOFT_HURDLE_PLUS_NAIVE5",
            "Target": TARGET_COLUMN,
            "OccurrenceTrainingRows": "ALL_MAIN_ROUTE_TRAINING_ROWS",
            "OccurrenceTarget": "NormalDemand > 0",
            "QuantityTrainingRows": "POSITIVE_NORMAL_DEMAND_ROWS_ONLY",
            "SoftHurdleRule": (
                "Occurrence probability multiplied by positive quantity prediction"
            ),
            "ClipAtZero": True,
            "RoundBeforeScoring": False,
        },
        "Predictors": {
            "Total": len(core_predictors),
            "Numeric": len(numeric_predictors),
            "Categorical": len(categorical_predictors),
            "BinarySeparatedWithinNumericRegister": BINARY_PREDICTORS,
            "CanonicalProductIDDirectPredictor": False,
            "BulkDemandPredictor": False,
            "TotalDemandPredictor": False,
            "TargetPredictor": False,
        },
        "OccurrenceParameters": OCCURRENCE_PARAMETERS,
        "QuantityParameters": {
            **QUANTITY_PARAMETERS,
            "hidden_layer_sizes": list(
                QUANTITY_PARAMETERS["hidden_layer_sizes"]
            ),
        },
        "FixedBlendWeightsEvaluated": HURDLE_BLEND_WEIGHTS,
        "ExactOriginalBlendWeight": ORIGINAL_BLEND_WEIGHT,
        "FoldProtocol": "ND04_FIXED_EXPANDING_WINDOW_5_X_20_OPERATING_DATES",
        "SelectedArchitectureMethod": selected_architecture_method,
        "DevelopmentRecommendation": development_recommendation,
        "FinalProductionModelFitted": False,
    }

    decision_payload = {
        "StepID": STEP_ID,
        "Status": STATUS,
        "CreatedUTC": NOW_UTC.isoformat(),
        "AcceptedND04Benchmark": {
            "Method": "ROLLING_MEAN_5",
            "WAPEPercentage": float(benchmark_row["WAPEPercentage"]),
            "MAE": float(benchmark_row["MAE"]),
            "RMSE": float(benchmark_row["RMSE"]),
            "AbsoluteBiasPercentage": float(
                benchmark_row["AbsoluteBiasPercentage"]
            ),
        },
        "ExactOriginalH75Rebuild": {
            "Method": "BLEND_HURDLE_NAIVE5__H75",
            "WAPEPercentage": float(original_h75_row["WAPEPercentage"]),
            "MAE": float(original_h75_row["MAE"]),
            "RMSE": float(original_h75_row["RMSE"]),
            "AbsoluteBiasPercentage": float(
                original_h75_row["AbsoluteBiasPercentage"]
            ),
        },
        "BestND05HurdleArchitecture": {
            "Method": selected_architecture_method,
            "WAPEPercentage": float(
                selected_architecture_row["WAPEPercentage"]
            ),
            "MAE": float(selected_architecture_row["MAE"]),
            "RMSE": float(selected_architecture_row["RMSE"]),
            "AbsoluteBiasPercentage": float(
                selected_architecture_row["AbsoluteBiasPercentage"]
            ),
            "WAPEImprovementVersusBenchmarkPP": (
                selected_wape_improvement_pp
            ),
            "RelativeWAPEImprovementPercentage": (
                selected_relative_wape_improvement_percentage
            ),
            "MAERatioToBenchmark": selected_mae_ratio,
            "QualifiesAgainstAcceptanceRule": (
                selected_architecture_qualifies
            ),
        },
        "AcceptanceRule": {
            "MinimumWAPEImprovementPercentagePoints": (
                MINIMUM_WAPE_IMPROVEMENT_PP
            ),
            "MaximumMAERatioToBenchmark": (
                MAXIMUM_MAE_RATIO_TO_BENCHMARK
            ),
            "MaximumAbsoluteBiasPercentage": (
                MAXIMUM_ACCEPTABLE_ABSOLUTE_BIAS_PERCENTAGE
            ),
        },
        "DevelopmentRecommendation": development_recommendation,
        "FinalModelSelectionPerformed": False,
        "NextStep": "ND06_EXPANDED_DAILY_MODEL_CHALLENGE",
    }

    routed_daily_product_metric = routed_metrics.loc[
        routed_metrics["EvaluationLevel"] == "DAILY_PRODUCT_ALL_ROUTES"
    ].iloc[0]
    routed_daily_restaurant_metric = routed_metrics.loc[
        routed_metrics["EvaluationLevel"]
        == "DAILY_RESTAURANT_TOTAL_ALL_ROUTES"
    ].iloc[0]
    routed_weekly_product_complete_metric = routed_metrics.loc[
        routed_metrics["EvaluationLevel"]
        == "WEEKLY_PRODUCT_DAILY_UPDATED_AGGREGATION_COMPLETE_WEEKS"
    ].iloc[0]
    routed_weekly_restaurant_complete_metric = routed_metrics.loc[
        routed_metrics["EvaluationLevel"]
        == "WEEKLY_RESTAURANT_DAILY_UPDATED_AGGREGATION_COMPLETE_WEEKS"
    ].iloc[0]

    report_summary_text = f"""# ND05 Original Hurdle Architecture Rebuild

## Status

`{STATUS}`

## Development design

The accepted ND04 chronological folds were reused without modification.

For each fold, the occurrence classifier was fitted on all earlier main-route rows, and the quantity MLP was fitted only on earlier positive-demand rows.

The soft-hurdle expected value equals the predicted probability of positive normal demand multiplied by the predicted positive quantity.

## Accepted benchmark

- Method: `ROLLING_MEAN_5`
- Daily product WAPE: {float(benchmark_row['WAPEPercentage']):.6f}%
- MAE: {float(benchmark_row['MAE']):.6f}
- RMSE: {float(benchmark_row['RMSE']):.6f}
- Absolute aggregate bias: {float(benchmark_row['AbsoluteBiasPercentage']):.6f}%

## Exact original H75 rebuild

- Method: `BLEND_HURDLE_NAIVE5__H75`
- Daily product WAPE: {float(original_h75_row['WAPEPercentage']):.6f}%
- MAE: {float(original_h75_row['MAE']):.6f}
- RMSE: {float(original_h75_row['RMSE']):.6f}
- Total bias: {float(original_h75_row['TotalBias']):.6f}

## Best ND05 hurdle architecture

- Method: `{selected_architecture_method}`
- Daily product WAPE: {float(selected_architecture_row['WAPEPercentage']):.6f}%
- MAE: {float(selected_architecture_row['MAE']):.6f}
- RMSE: {float(selected_architecture_row['RMSE']):.6f}
- Total bias: {float(selected_architecture_row['TotalBias']):.6f}
- Absolute aggregate bias: {float(selected_architecture_row['AbsoluteBiasPercentage']):.6f}%
- WAPE improvement versus benchmark: {selected_wape_improvement_pp:.6f} percentage points
- Relative WAPE improvement: {selected_relative_wape_improvement_percentage:.6f}%
- Qualifies under ND05 acceptance rule: {selected_architecture_qualifies}

## Development recommendation

`{development_recommendation}`

This is not the final model decision. ND06 will compare the accepted benchmark and the strongest ND05 hurdle candidate with expanded machine-learning candidates under the same chronological folds.

## Full routed-system diagnostics

The best ND05 main-route challenger was combined with the already-selected ND04 fallback methods.

- Daily product WAPE: {float(routed_daily_product_metric['WAPEPercentage']):.6f}%
- Daily restaurant-total WAPE: {float(routed_daily_restaurant_metric['WAPEPercentage']):.6f}%
- Complete-week product WAPE: {float(routed_weekly_product_complete_metric['WAPEPercentage']):.6f}%
- Complete-week restaurant WAPE: {float(routed_weekly_restaurant_complete_metric['WAPEPercentage']):.6f}%

The weekly figures are daily-updated aggregation diagnostics. They are not Monday-origin recursive weekly forecasts.

## Safety

- March 2026 targets opened: no
- Final production model fitted: no
- Existing predictions changed: no
- Existing locks changed: no
- ND05 lock created: no
"""

    readme_text = f"""# ND05 Original Hurdle Model Rebuild

Status: `{STATUS}`

This folder contains chronological development evaluation of the original daily occurrence-plus-positive-quantity architecture on the new bulk-free normal-demand main-model scope.

## Main interpretation

- ND04 benchmark: `ROLLING_MEAN_5`
- Exact original blend: `BLEND_HURDLE_NAIVE5__H75`
- Best ND05 hurdle architecture: `{selected_architecture_method}`
- Development recommendation: `{development_recommendation}`

## Important limitation

Weekly metrics in this folder are obtained by aggregating one-step daily predictions that use actual history available before each day. They are not week-start recursive forecasts.

## Final model status

No final production model is fitted or locked in ND05.
"""

    # =========================================================================
    # WRITE OUTPUTS
    # =========================================================================

    output_frames = {
        staged_prediction_dir / COMPONENT_PREDICTIONS_PATH.name:
            component_predictions,
        staged_prediction_dir / CANDIDATE_PREDICTIONS_PATH.name:
            candidate_predictions,
        staged_prediction_dir / SELECTED_MAIN_PREDICTIONS_PATH.name:
            selected_main_predictions,
        staged_prediction_dir / ROUTED_SYSTEM_PREDICTIONS_PATH.name:
            routed_predictions,
        staged_prediction_dir / DAILY_RESTAURANT_PATH.name:
            daily_restaurant,
        staged_prediction_dir / WEEKLY_PRODUCT_PATH.name:
            weekly_product,
        staged_prediction_dir / WEEKLY_RESTAURANT_PATH.name:
            weekly_restaurant,
        staged_metric_dir / MAIN_METRICS_PATH.name:
            main_metrics,
        staged_metric_dir / MAIN_FOLD_METRICS_PATH.name:
            main_fold_metrics,
        staged_metric_dir / OCCURRENCE_METRICS_PATH.name:
            occurrence_metrics,
        staged_metric_dir / QUANTITY_METRICS_PATH.name:
            quantity_metrics,
        staged_metric_dir / COMPARISON_PATH.name:
            comparison,
        staged_metric_dir / ROUTED_METRICS_PATH.name:
            routed_metrics,
        staged_metric_dir / ROUTED_ROUTE_METRICS_PATH.name:
            routed_route_metrics,
        staged_audit_dir / INPUT_HASH_AUDIT_PATH.name:
            input_hash_audit,
        staged_audit_dir / FOLD_TRAINING_AUDIT_PATH.name:
            fold_training_audit,
        staged_audit_dir / PROTOCOL_AUDIT_PATH.name:
            protocol_audit,
        staged_audit_dir / VALIDATION_PATH.name:
            validation,
    }

    for output_path, output_frame in output_frames.items():
        write_csv(output_path, output_frame)

    write_json(
        staged_audit_dir / MODEL_CONFIG_PATH.name,
        model_configuration,
    )
    write_json(
        staged_report_dir / DECISION_PATH.name,
        decision_payload,
    )
    write_text(
        staged_report_dir / REPORT_SUMMARY_PATH.name,
        report_summary_text,
    )
    write_text(STAGING_ROOT / README_PATH.name, readme_text)

    # Reload critical outputs.
    reloaded_component = pd.read_csv(
        staged_prediction_dir / COMPONENT_PREDICTIONS_PATH.name,
        low_memory=False,
    )
    reloaded_candidate = pd.read_csv(
        staged_prediction_dir / CANDIDATE_PREDICTIONS_PATH.name,
        low_memory=False,
    )
    reloaded_routed = pd.read_csv(
        staged_prediction_dir / ROUTED_SYSTEM_PREDICTIONS_PATH.name,
        low_memory=False,
    )

    if len(reloaded_component) != main_validation_rows:
        raise AssertionError("Reloaded component prediction row count changed.")
    if len(reloaded_candidate) != len(candidate_predictions):
        raise AssertionError("Reloaded candidate prediction row count changed.")
    if len(reloaded_routed) != len(routed_predictions):
        raise AssertionError("Reloaded routed prediction row count changed.")

    # =========================================================================
    # MANIFEST AND CHECKPOINT
    # =========================================================================

    manifest_excluded_names = {
        MANIFEST_PATH.name,
        CHECKPOINT_PATH.name,
        CHECKPOINT_SHA_PATH.name,
    }
    files_for_manifest = sorted(
        path
        for path in STAGING_ROOT.rglob("*")
        if path.is_file() and path.name not in manifest_excluded_names
    )
    manifest = pd.DataFrame(
        [
            {
                "RelativePath": str(path.relative_to(STAGING_ROOT)),
                "Bytes": int(path.stat().st_size),
                "SHA256": sha256_file(path),
            }
            for path in files_for_manifest
        ]
    ).sort_values("RelativePath").reset_index(drop=True)

    staged_manifest_path = staged_control_dir / MANIFEST_PATH.name
    write_csv(staged_manifest_path, manifest)
    manifest_sha256 = sha256_file(staged_manifest_path)

    checkpoint_payload = {
        "StepID": STEP_ID,
        "Status": STATUS,
        "CreatedUTC": NOW_UTC.isoformat(),
        "CreatedLocal": NOW_LOCAL.isoformat(),
        "ModelRoot": str(MODEL_ROOT),
        "ND05Root": str(ND05_ROOT),
        "Input": {
            "ND03CheckpointSHA256": nd03_checkpoint_sha256_before,
            "ND03ManifestSHA256": actual_nd03_manifest_sha256,
            "ND04CheckpointSHA256": nd04_checkpoint_sha256_before,
            "ND04ManifestSHA256": actual_nd04_manifest_sha256,
            "MainDevelopmentRows": len(main_model),
            "CorePredictors": len(core_predictors),
        },
        "FoldProtocol": {
            "Folds": len(fold_training_audit),
            "ValidationRows": main_validation_rows,
            "ValidationOperatingDates": validation_dates,
            "FirstValidationDate": component_predictions["Date"].min(),
            "LastValidationDate": component_predictions["Date"].max(),
            "TotalFitSeconds": total_fit_seconds,
        },
        "Architecture": model_configuration["Architecture"],
        "Selection": {
            "ND04BenchmarkMethod": "ROLLING_MEAN_5",
            "ND04BenchmarkWAPEPercentage": float(
                benchmark_row["WAPEPercentage"]
            ),
            "ExactOriginalMethod": "BLEND_HURDLE_NAIVE5__H75",
            "ExactOriginalWAPEPercentage": float(
                original_h75_row["WAPEPercentage"]
            ),
            "BestND05ArchitectureMethod": selected_architecture_method,
            "BestND05ArchitectureWAPEPercentage": float(
                selected_architecture_row["WAPEPercentage"]
            ),
            "WAPEImprovementVersusBenchmarkPP": (
                selected_wape_improvement_pp
            ),
            "QualifiesAgainstAcceptanceRule": (
                selected_architecture_qualifies
            ),
            "DevelopmentRecommendation": development_recommendation,
        },
        "RoutedSystemMetrics": routed_metrics.to_dict(orient="records"),
        "Control": {
            "ManifestPath": str(MANIFEST_PATH),
            "ManifestSHA256": manifest_sha256,
            "ValidationPath": str(VALIDATION_PATH),
            "ProtocolAuditPath": str(PROTOCOL_AUDIT_PATH),
            "DecisionPath": str(DECISION_PATH),
        },
        "Safety": {
            "OccurrenceModelsFittedWithinFolds": True,
            "QuantityModelsFittedWithinFolds": True,
            "FinalProductionModelFitted": False,
            "FinalModelArtifactSaved": False,
            "MarchTargetVaultOpened": False,
            "ND03InputsModified": False,
            "ND04InputsModified": False,
            "ExistingModelLocksModified": False,
            "ND05StepLockCreated": False,
            "CheckpointAndHashesCreated": True,
        },
        "ReadyForND06": True,
        "NextStep": "ND06_EXPANDED_DAILY_MODEL_CHALLENGE",
    }

    staged_checkpoint_path = staged_control_dir / CHECKPOINT_PATH.name
    write_json(staged_checkpoint_path, checkpoint_payload)
    checkpoint_sha256 = sha256_file(staged_checkpoint_path)

    staged_checkpoint_sha_path = staged_control_dir / CHECKPOINT_SHA_PATH.name
    write_text(
        staged_checkpoint_sha_path,
        f"{checkpoint_sha256}  {CHECKPOINT_PATH.name}\n",
    )

    required_staged_outputs = [
        STAGING_ROOT / README_PATH.name,
        staged_prediction_dir / COMPONENT_PREDICTIONS_PATH.name,
        staged_prediction_dir / CANDIDATE_PREDICTIONS_PATH.name,
        staged_prediction_dir / SELECTED_MAIN_PREDICTIONS_PATH.name,
        staged_prediction_dir / ROUTED_SYSTEM_PREDICTIONS_PATH.name,
        staged_metric_dir / MAIN_METRICS_PATH.name,
        staged_metric_dir / COMPARISON_PATH.name,
        staged_audit_dir / FOLD_TRAINING_AUDIT_PATH.name,
        staged_audit_dir / VALIDATION_PATH.name,
        staged_report_dir / REPORT_SUMMARY_PATH.name,
        staged_manifest_path,
        staged_checkpoint_path,
        staged_checkpoint_sha_path,
    ]
    missing_staged_outputs = [
        path for path in required_staged_outputs if not path.is_file()
    ]
    if missing_staged_outputs:
        raise AssertionError(
            "Required staged ND05 outputs are missing:\n"
            + "\n".join(f"- {path}" for path in missing_staged_outputs)
        )

    # Verify protected inputs remained unchanged.
    protected_input_hashes_after = {
        str(path): sha256_file(path)
        for path in required_inputs
        if path.is_file()
    }
    changed_inputs = [
        path
        for path in protected_input_hashes_before
        if protected_input_hashes_before[path]
        != protected_input_hashes_after[path]
    ]
    if changed_inputs:
        raise AssertionError(
            "One or more protected ND03/ND04 inputs changed during ND05:\n"
            + "\n".join(f"- {path}" for path in changed_inputs)
        )

    # Atomic commit.
    os.replace(STAGING_ROOT, ND05_ROOT)

    TOP_LEVEL_CHECKPOINT_PATH.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(CHECKPOINT_PATH, TOP_LEVEL_CHECKPOINT_PATH)
    shutil.copy2(CHECKPOINT_SHA_PATH, TOP_LEVEL_CHECKPOINT_SHA_PATH)
    make_read_only(TOP_LEVEL_CHECKPOINT_PATH)
    make_read_only(TOP_LEVEL_CHECKPOINT_SHA_PATH)

    # =========================================================================
    # PROJECT MEMORY AND AGENT HANDOFF
    # =========================================================================

    nd05_handoff_text = f"""# ND05 Handoff

## Current status

- Completed step: `{STEP_ID}`
- Status: `{STATUS}`
- Completed local time: `{NOW_LOCAL.isoformat()}`
- ND05 root: `{ND05_ROOT}`
- Checkpoint: `{TOP_LEVEL_CHECKPOINT_PATH}`
- Checkpoint SHA-256: `{checkpoint_sha256}`

## Architecture evaluated

- Occurrence model: LogisticRegression with balanced class weights
- Quantity model: MLPRegressor on positive normal-demand rows only
- Soft hurdle: occurrence probability multiplied by positive quantity prediction
- Exact original blend: 0.75 soft hurdle + 0.25 lag-5
- Target: NormalDemand
- Core predictors: {len(core_predictors)}
- Bulk and total demand predictors: none

## Development result

- Accepted ND04 benchmark: ROLLING_MEAN_5
- Benchmark WAPE: {float(benchmark_row['WAPEPercentage']):.6f}%
- Exact H75 WAPE: {float(original_h75_row['WAPEPercentage']):.6f}%
- Best ND05 hurdle architecture: {selected_architecture_method}
- Best hurdle WAPE: {float(selected_architecture_row['WAPEPercentage']):.6f}%
- WAPE improvement versus benchmark: {selected_wape_improvement_pp:.6f} percentage points
- Qualifies under ND05 rule: {selected_architecture_qualifies}
- Recommendation: {development_recommendation}

## Important interpretation

The ND05 weekly metrics are daily-updated aggregation diagnostics. They are not Monday-origin recursive forecasts.

No final production model was fitted or locked.

## Next step

ND06 will compare expanded daily machine-learning candidates under the same pre-March chronological folds.
"""

    current_handoff_text = nd05_handoff_text

    workflow_section = f"""## ND05 — Original hurdle architecture rebuild

Status: `{STATUS}`

Completed actions:

- reused the fixed ND04 chronological folds;
- fitted occurrence logistic and positive-quantity MLP models within each fold;
- rebuilt the soft-hurdle expected-value forecast;
- evaluated lag-5, soft hurdle, fixed hurdle/naive blends, and the rolling-mean benchmark;
- scored the exact original 75/25 blend;
- combined the best ND05 main-route challenger with ND04 fallback predictions;
- produced daily and daily-updated weekly diagnostics;
- did not open March 2026 targets; and
- created hashes and a checkpoint without creating an ND05 lock.

Next: ND06 expanded daily model challenge.
"""

    decisions_section = f"""## ND05 decisions

1. The original occurrence-logistic plus positive-quantity-MLP architecture was rebuilt on bulk-free normal demand.
2. The exact original product blend was evaluated as `BLEND_HURDLE_NAIVE5__H75`.
3. The accepted benchmark remains `ROLLING_MEAN_5` unless a hurdle candidate satisfies the stated WAPE, MAE, and bias rule.
4. Best ND05 hurdle architecture: `{selected_architecture_method}`.
5. Development recommendation: `{development_recommendation}`.
6. ND05 results are development evidence, not final model selection.
7. Weekly ND05 outputs are daily-updated aggregation diagnostics only.
8. March 2026 remains unopened and excluded from selection.
9. No ND05 lock or final model artifact was created.
"""

    metrics_section = f"""## ND05 original hurdle architecture

- Main validation rows: {main_validation_rows:,}
- Validation dates: {validation_dates}
- ND04 rolling-mean-5 WAPE: {float(benchmark_row['WAPEPercentage']):.6f}%
- Exact original H75 WAPE: {float(original_h75_row['WAPEPercentage']):.6f}%
- Best ND05 hurdle method: {selected_architecture_method}
- Best hurdle WAPE: {float(selected_architecture_row['WAPEPercentage']):.6f}%
- Best hurdle MAE: {float(selected_architecture_row['MAE']):.6f}
- Best hurdle RMSE: {float(selected_architecture_row['RMSE']):.6f}
- Best hurdle total bias: {float(selected_architecture_row['TotalBias']):.6f}
- WAPE improvement versus benchmark: {selected_wape_improvement_pp:.6f} percentage points
- Routed daily product WAPE: {float(routed_daily_product_metric['WAPEPercentage']):.6f}%
- Routed daily restaurant WAPE: {float(routed_daily_restaurant_metric['WAPEPercentage']):.6f}%
- Routed complete-week product WAPE: {float(routed_weekly_product_complete_metric['WAPEPercentage']):.6f}%
- Routed complete-week restaurant WAPE: {float(routed_weekly_restaurant_complete_metric['WAPEPercentage']):.6f}%
- Final model fitted: no
"""

    agents_section = f"""## ND05 authoritative status

Marker: ND05_AUTHORITATIVE_STATUS

- Status: `{STATUS}`
- Read next: `{ND05_HANDOFF_PATH}`
- ND04 benchmark: `ROLLING_MEAN_5`
- Exact original blend: `BLEND_HURDLE_NAIVE5__H75`
- Best ND05 hurdle architecture: `{selected_architecture_method}`
- Development recommendation: `{development_recommendation}`
- March 2026 opened: no
- Final production model fitted: no
- Next step: `ND06`
"""

    atomic_write_text(ND05_HANDOFF_PATH, nd05_handoff_text)
    atomic_write_text(CURRENT_HANDOFF_PATH, current_handoff_text)
    append_marked_section(
        WORKFLOW_PATH,
        "## ND05 — Original hurdle architecture rebuild",
        workflow_section,
    )
    append_marked_section(
        DECISIONS_PATH,
        "## ND05 decisions",
        decisions_section,
    )
    append_marked_section(
        METRICS_AND_RESULTS_PATH,
        "## ND05 original hurdle architecture",
        metrics_section,
    )
    append_marked_section(
        AGENTS_PATH,
        "Marker: ND05_AUTHORITATIVE_STATUS",
        agents_section,
    )

    log_text = "\n".join(
        [
            f"Step: {STEP_ID}",
            f"Status: {STATUS}",
            f"Created local: {NOW_LOCAL.isoformat()}",
            f"Created UTC: {NOW_UTC.isoformat()}",
            f"ND03 checkpoint SHA256: {nd03_checkpoint_sha256_before}",
            f"ND04 checkpoint SHA256: {nd04_checkpoint_sha256_before}",
            f"Main validation rows: {main_validation_rows}",
            f"Validation dates: {validation_dates}",
            f"Selected hurdle method: {selected_architecture_method}",
            f"Selected hurdle WAPE: {float(selected_architecture_row['WAPEPercentage'])}",
            f"Benchmark WAPE: {float(benchmark_row['WAPEPercentage'])}",
            f"WAPE improvement pp: {selected_wape_improvement_pp}",
            f"Qualifies: {selected_architecture_qualifies}",
            f"Recommendation: {development_recommendation}",
            f"Total fit seconds: {total_fit_seconds}",
            f"Checkpoint SHA256: {checkpoint_sha256}",
            "March target vault opened: False",
            "Final production model fitted: False",
            "Existing locks modified: False",
            "ND05 step lock created: False",
            "Ready for ND06: True",
            "",
        ]
    )
    atomic_write_text(LOG_PATH, log_text)

except Exception:
    if STAGING_ROOT.exists():
        shutil.rmtree(STAGING_ROOT)
    raise


# =============================================================================
# FINAL CONSOLE OUTPUT
# =============================================================================

print("=" * 112)
print("EDEN NORMAL-DEMAND MODEL V2 — ND05 COMPLETE")
print("=" * 112)
print(f"Status: {STATUS}")
print(f"Local time: {NOW_LOCAL.isoformat()}")
print(f"ND05 root: {ND05_ROOT}")

print("\nINPUT VERIFICATION")
print(f"ND03 checkpoint SHA-256: {nd03_checkpoint_sha256_before}")
print(f"ND03 manifest SHA-256: {actual_nd03_manifest_sha256}")
print(f"ND04 checkpoint SHA-256: {nd04_checkpoint_sha256_before}")
print(f"ND04 manifest SHA-256: {actual_nd04_manifest_sha256}")
print(f"Main development rows: {len(main_model):,}")
print(f"Core predictors: {len(core_predictors)}")
print(f"Numeric predictors: {len(numeric_predictors)}")
print(f"Categorical predictors: {len(categorical_predictors)}")
print("March target vault opened: False")
print("ND03/ND04 inputs modified: False")

print("\nCHRONOLOGICAL MODEL FITTING")
print(f"Folds: {len(fold_training_audit)}")
print(f"Validation product-date rows: {main_validation_rows:,}")
print(f"Validation operating dates: {validation_dates}")
print(f"First validation date: {component_predictions['Date'].min().date()}")
print(f"Last validation date: {component_predictions['Date'].max().date()}")
print(f"Total model fitting seconds: {total_fit_seconds:.3f}")
print("Occurrence models fitted within folds: True")
print("Positive-quantity models fitted within folds: True")
print("Final production model fitted: False")

print("\nACCEPTED ND04 MAIN BENCHMARK")
print("Method: ROLLING_MEAN_5")
print(f"WAPE: {float(benchmark_row['WAPEPercentage']):.6f}%")
print(f"MAE: {float(benchmark_row['MAE']):.6f}")
print(f"RMSE: {float(benchmark_row['RMSE']):.6f}")
print(f"Total bias: {float(benchmark_row['TotalBias']):.6f}")
print(
    "Absolute bias percentage: "
    f"{float(benchmark_row['AbsoluteBiasPercentage']):.6f}%"
)

print("\nEXACT ORIGINAL H75 REBUILD")
print("Method: BLEND_HURDLE_NAIVE5__H75")
print(f"WAPE: {float(original_h75_row['WAPEPercentage']):.6f}%")
print(f"MAE: {float(original_h75_row['MAE']):.6f}")
print(f"RMSE: {float(original_h75_row['RMSE']):.6f}")
print(f"Total bias: {float(original_h75_row['TotalBias']):.6f}")
print(
    "Absolute bias percentage: "
    f"{float(original_h75_row['AbsoluteBiasPercentage']):.6f}%"
)

print("\nBEST ND05 HURDLE ARCHITECTURE")
print(f"Method: {selected_architecture_method}")
print(
    "WAPE: "
    f"{float(selected_architecture_row['WAPEPercentage']):.6f}%"
)
print(f"MAE: {float(selected_architecture_row['MAE']):.6f}")
print(f"RMSE: {float(selected_architecture_row['RMSE']):.6f}")
print(f"Total bias: {float(selected_architecture_row['TotalBias']):.6f}")
print(
    "Absolute bias percentage: "
    f"{float(selected_architecture_row['AbsoluteBiasPercentage']):.6f}%"
)
print(
    "WAPE improvement versus benchmark: "
    f"{selected_wape_improvement_pp:.6f} percentage points"
)
print(
    "Relative WAPE improvement: "
    f"{selected_relative_wape_improvement_percentage:.6f}%"
)
print(f"Qualifies under ND05 rule: {selected_architecture_qualifies}")
print(f"Development recommendation: {development_recommendation}")

print("\nALL MAIN CANDIDATE RESULTS")
print(
    main_metrics[
        [
            "Rank",
            "CandidateMethod",
            "WAPEPercentage",
            "MAE",
            "RMSE",
            "TotalBias",
            "AbsoluteBiasPercentage",
        ]
    ].to_string(index=False)
)

print("\nFULL ROUTED SYSTEM")
print(f"Rows: {len(routed_predictions):,}")
print(
    routed_metrics[
        [
            "EvaluationLevel",
            "Observations",
            "WAPEPercentage",
            "MAE",
            "RMSE",
            "TotalBias",
        ]
    ].to_string(index=False)
)
print("Forecast mode: DAILY_UPDATED_AGGREGATION_DIAGNOSTIC")
print("Week-start recursive forecast: False")

print("\nOUTPUTS")
print(f"- Component predictions: {COMPONENT_PREDICTIONS_PATH}")
print(f"- Candidate predictions: {CANDIDATE_PREDICTIONS_PATH}")
print(f"- Selected main predictions: {SELECTED_MAIN_PREDICTIONS_PATH}")
print(f"- Routed-system predictions: {ROUTED_SYSTEM_PREDICTIONS_PATH}")
print(f"- Main metrics: {MAIN_METRICS_PATH}")
print(f"- Fold metrics: {MAIN_FOLD_METRICS_PATH}")
print(f"- Occurrence metrics: {OCCURRENCE_METRICS_PATH}")
print(f"- Quantity metrics: {QUANTITY_METRICS_PATH}")
print(f"- Comparison: {COMPARISON_PATH}")
print(f"- Routed metrics: {ROUTED_METRICS_PATH}")
print(f"- Figures: {FIGURE_DIR}")
print(f"- Report summary: {REPORT_SUMMARY_PATH}")
print(f"- Validation: {VALIDATION_PATH}")
print(f"- Manifest: {MANIFEST_PATH}")
print(f"- Checkpoint: {TOP_LEVEL_CHECKPOINT_PATH}")
print(f"- Checkpoint SHA-256: {checkpoint_sha256}")
print(f"- Agent handoff: {ND05_HANDOFF_PATH}")

print("\nSAFETY")
print("- Occurrence models fitted within folds: True")
print("- Quantity models fitted within folds: True")
print("- Final production model fitted: False")
print("- Final model artifact saved: False")
print("- March target vault opened: False")
print("- ND03/ND04 inputs modified: False")
print("- Existing model locks modified: False")
print("- ND05 step lock created: False")
print("- ND05 checkpoint and hashes created: True")

print("\nNEXT STEP")
print(
    "ND06 — expanded daily model challenge using stronger machine-learning "
    "candidates under the same chronological folds."
)
print("=" * 112)

EDEN NORMAL-DEMAND MODEL V2 — ND05 COMPLETE
Status: ND05_ORIGINAL_HURDLE_ARCHITECTURE_EVALUATED_READY_FOR_ND06
Local time: 2026-08-07T23:09:54.992522+01:00
ND05 root: /Users/ryansmac/Desktop/Meng Project/eden_datasets/eden_normal_demand_model_v2/03_models/00_candidates/ND05_original_hurdle_model_rebuild

INPUT VERIFICATION
ND03 checkpoint SHA-256: 0845af89a5b459ca13ae6ffd99dde444f5010f6c0fb5ba4c34a4f091ac2e151c
ND03 manifest SHA-256: faa7f60b99f2c48cd110786a560969ddccac5c9e6da871f4dd41f46a553ce654
ND04 checkpoint SHA-256: 2fdc5d2c64c38f85b2669ca942042884209d80111cc840261307da98b1e9cf54
ND04 manifest SHA-256: 8546b2cc558a430bdcdd40dfb5dcea76da3d7aef54066e62aeaed6bb990bdd95
Main development rows: 10,976
Core predictors: 53
Numeric predictors: 45
Categorical predictors: 8
March target vault opened: False
ND03/ND04 inputs modified: False

CHRONOLOGICAL MODEL FITTING
Folds: 5
Validation product-date rows: 4,124
Validation operating dates: 100
First validation date: 2025-09-30
Last validatio

In [7]:
from __future__ import annotations

# =============================================================================
# EDEN NORMAL-DEMAND MODEL V2
# ND06 — EXPANDED NON-HURDLE MODEL CHALLENGE
# =============================================================================
# Uses the exact ND04 chronological folds. Screens modern boosting regressors
# against ROLLING_MEAN_5. March 2026 remains closed. No final model is saved.
# Weekly metrics are daily-updated aggregation diagnostics, not Monday-origin
# recursive forecasts.
# =============================================================================

import hashlib
import json
import math
import os
import platform
import shutil
import time
import uuid
import warnings
from datetime import datetime, timezone
from pathlib import Path
from zoneinfo import ZoneInfo

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import sklearn
from matplotlib.dates import DateFormatter
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OrdinalEncoder

warnings.filterwarnings("ignore", message="X does not have valid feature names")
os.environ.setdefault("OMP_NUM_THREADS", "1")
os.environ.setdefault("OPENBLAS_NUM_THREADS", "1")
os.environ.setdefault("MKL_NUM_THREADS", "1")
os.environ.setdefault("NUMEXPR_NUM_THREADS", "1")

try:
    import xgboost
    from xgboost import XGBRegressor
    XGBOOST_AVAILABLE = True
except Exception:
    xgboost = None
    XGBRegressor = None
    XGBOOST_AVAILABLE = False

try:
    import lightgbm
    from lightgbm import LGBMRegressor
    LIGHTGBM_AVAILABLE = True
except Exception:
    lightgbm = None
    LGBMRegressor = None
    LIGHTGBM_AVAILABLE = False

try:
    import catboost
    from catboost import CatBoostRegressor
    CATBOOST_AVAILABLE = True
except Exception:
    catboost = None
    CatBoostRegressor = None
    CATBOOST_AVAILABLE = False


# =============================================================================
# PATHS AND CONSTANTS
# =============================================================================

PROJECT_ROOT = Path("/Users/ryansmac/Desktop/Meng Project")
MODEL_ROOT = PROJECT_ROOT / "eden_datasets" / "eden_normal_demand_model_v2"
ND03_ROOT = MODEL_ROOT / "02_feature_engineering" / "ND03_normal_demand_features"
ND04_ROOT = MODEL_ROOT / "03_models" / "00_candidates" / "ND04_baseline_and_fallback_evaluation"
ND05_ROOT = MODEL_ROOT / "03_models" / "00_candidates" / "ND05_original_hurdle_model_rebuild"
ND06_ROOT = MODEL_ROOT / "03_models" / "00_candidates" / "ND06_expanded_model_challenge"

MAIN_MODEL_PATH = ND03_ROOT / "01_model_ready_datasets" / "ND03_pre_march_main_model_development_dataset.csv"
PREDICTOR_LIST_PATH = ND03_ROOT / "03_contracts" / "ND03_core_predictor_list.csv"
ND03_MANIFEST_PATH = ND03_ROOT / "05_control" / "ND03_artifact_hash_manifest.csv"
ND03_CHECKPOINT_PATH = MODEL_ROOT / "08_checkpoints" / "ND03_checkpoint.json"

FOLD_PATH = ND04_ROOT / "03_audits" / "ND04_chronological_fold_definition.csv"
SELECTED_METHODS_PATH = ND04_ROOT / "02_metrics" / "ND04_selected_route_methods.csv"
ND04_ROUTED_PATH = ND04_ROOT / "01_predictions" / "ND04_selected_routed_system_predictions.csv"
ND04_MANIFEST_PATH = ND04_ROOT / "06_control" / "ND04_artifact_hash_manifest.csv"
ND04_CHECKPOINT_PATH = MODEL_ROOT / "08_checkpoints" / "ND04_checkpoint.json"

ND05_METRICS_PATH = ND05_ROOT / "02_metrics" / "ND05_main_candidate_metrics.csv"
ND05_MANIFEST_PATH = ND05_ROOT / "06_control" / "ND05_artifact_hash_manifest.csv"
ND05_CHECKPOINT_PATH = MODEL_ROOT / "08_checkpoints" / "ND05_checkpoint.json"

EXPECTED_CHECKPOINTS = {
    "ND03": "0845af89a5b459ca13ae6ffd99dde444f5010f6c0fb5ba4c34a4f091ac2e151c",
    "ND04": "2fdc5d2c64c38f85b2669ca942042884209d80111cc840261307da98b1e9cf54",
    "ND05": "ce3342c8b960aa5c4791a114ab09ae1a86d1dab2a3eebb648aa060579a1378ff",
}
EXPECTED_ROWS = 10_976
EXPECTED_VALIDATION_ROWS = 4_124
EXPECTED_VALIDATION_DATES = 100
EXPECTED_FOLDS = 5
EXPECTED_PREDICTORS = 53
EXPECTED_NUMERIC = 45
EXPECTED_CATEGORICAL = 8
EXPECTED_BENCHMARK = "ROLLING_MEAN_5"
EXPECTED_BENCHMARK_WAPE = 41.331256

DATE = "Date"
PID = "CanonicalProductID"
PNAME = "CanonicalProductName"
TARGET = "NormalDemand"
ROUTE = "ForecastRoute"
FAMILY = "TierProductFamily"
DOW = "DayOfWeekNumber"
THREADS = max(1, min(2, os.cpu_count() or 1))
SEED = 42
ALLOW_OVERWRITE = False

# Strict screen and softer ND07 carry-forward screen.
MIN_WAPE_IMPROVEMENT_PP = 0.50
MAX_MAE_RATIO = 1.02
MAX_RMSE_RATIO = 1.10
MAX_ABS_BIAS_PCT = 3.00
MIN_FOLDS_WON = 3
PROMISING_WAPE_GAP_PP = 1.00
PROMISING_ABS_BIAS_PCT = 10.00
MIN_ADVANCE = 3
MAX_ADVANCE = 4

PRED_DIR = ND06_ROOT / "01_predictions"
METRIC_DIR = ND06_ROOT / "02_metrics"
AUDIT_DIR = ND06_ROOT / "03_audits"
FIG_DIR = ND06_ROOT / "04_figures"
REPORT_DIR = ND06_ROOT / "05_reports"
CONTROL_DIR = ND06_ROOT / "06_control"

PATHS = {
    "candidate_predictions": PRED_DIR / "ND06_candidate_predictions.csv",
    "best_main": PRED_DIR / "ND06_best_challenger_main_predictions.csv",
    "routed": PRED_DIR / "ND06_best_challenger_routed_predictions.csv",
    "daily_total": PRED_DIR / "ND06_best_challenger_daily_restaurant_totals.csv",
    "weekly_product": PRED_DIR / "ND06_best_challenger_weekly_product_totals.csv",
    "weekly_total": PRED_DIR / "ND06_best_challenger_weekly_restaurant_totals.csv",
    "metrics": METRIC_DIR / "ND06_candidate_metrics.csv",
    "fold_metrics": METRIC_DIR / "ND06_candidate_metrics_by_fold.csv",
    "advance": METRIC_DIR / "ND06_advancement_decision.csv",
    "key_comparison": METRIC_DIR / "ND06_benchmark_hurdle_and_challenger_comparison.csv",
    "routed_metrics": METRIC_DIR / "ND06_best_challenger_routed_metrics.csv",
    "route_metrics": METRIC_DIR / "ND06_best_challenger_routed_metrics_by_route.csv",
    "registry": AUDIT_DIR / "ND06_candidate_registry.csv",
    "versions": AUDIT_DIR / "ND06_package_versions.csv",
    "fold_audit": AUDIT_DIR / "ND06_fold_training_audit.csv",
    "hash_audit": AUDIT_DIR / "ND06_input_hash_audit.csv",
    "protocol": AUDIT_DIR / "ND06_leakage_and_protocol_audit.csv",
    "validation": AUDIT_DIR / "ND06_validation_summary.csv",
    "report": REPORT_DIR / "ND06_expanded_model_challenge_summary.md",
    "decision": REPORT_DIR / "ND06_expanded_model_challenge_decision.json",
    "manifest": CONTROL_DIR / "ND06_artifact_hash_manifest.csv",
    "checkpoint": CONTROL_DIR / "ND06_checkpoint.json",
    "checkpoint_sha": CONTROL_DIR / "ND06_checkpoint.sha256",
}
TOP_CHECKPOINT = MODEL_ROOT / "08_checkpoints" / "ND06_checkpoint.json"
TOP_CHECKPOINT_SHA = MODEL_ROOT / "08_checkpoints" / "ND06_checkpoint.sha256"
MEMORY_ROOT = MODEL_ROOT / "00_project_memory"
HANDOFF_PATH = MEMORY_ROOT / "ND06_HANDOFF.md"
CURRENT_HANDOFF_PATH = MEMORY_ROOT / "CURRENT_HANDOFF.md"
WORKFLOW_PATH = MEMORY_ROOT / "WORKFLOW.md"
DECISIONS_PATH = MEMORY_ROOT / "DECISIONS.md"
METRICS_MEMORY_PATH = MEMORY_ROOT / "METRICS_AND_RESULTS.md"
AGENTS_PATH = MODEL_ROOT / "AGENTS.md"
LOG_PATH = MODEL_ROOT / "09_logs" / "ND06_model_challenge_log.txt"

STEP_ID = "ND06"
STATUS = "ND06_EXPANDED_MODEL_CHALLENGE_COMPLETED_READY_FOR_ND07"
NOW_UTC = datetime.now(timezone.utc)
NOW_LOCAL = NOW_UTC.astimezone(ZoneInfo("Europe/Dublin"))


# =============================================================================
# HELPERS
# =============================================================================

def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


def write_csv(path: Path, frame: pd.DataFrame) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    frame.to_csv(path, index=False)


def write_json(path: Path, payload: dict) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(payload, indent=2, ensure_ascii=False, default=str) + "\n", encoding="utf-8")


def write_text(path: Path, text: str) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(text, encoding="utf-8")


def atomic_write_text(path: Path, text: str) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    temp = path.with_name(f".{path.name}.{uuid.uuid4().hex}.tmp")
    temp.write_text(text, encoding="utf-8")
    os.replace(temp, path)


def append_once(path: Path, marker: str, text: str) -> None:
    existing = path.read_text(encoding="utf-8") if path.is_file() else ""
    if marker in existing:
        return
    separator = "\n" if existing.endswith("\n") else "\n\n"
    atomic_write_text(path, existing + separator + text.strip() + "\n")


def wape(actual, predicted) -> float:
    a = np.asarray(actual, dtype=float)
    p = np.asarray(predicted, dtype=float)
    denominator = float(a.sum())
    return float("nan") if denominator == 0 else float(100.0 * np.abs(p - a).sum() / denominator)


def metrics(actual, predicted, level: str) -> dict:
    a = np.asarray(actual, dtype=float)
    p = np.asarray(predicted, dtype=float)
    error = p - a
    actual_total = float(a.sum())
    total_bias = float(error.sum())
    return {
        "EvaluationLevel": level,
        "Observations": int(len(a)),
        "ActualTotal": actual_total,
        "PredictedTotal": float(p.sum()),
        "WAPEPercentage": wape(a, p),
        "MAE": float(np.abs(error).mean()),
        "RMSE": float(np.sqrt(np.square(error).mean())),
        "MeanBias": float(error.mean()),
        "TotalBias": total_bias,
        "AbsoluteBiasPercentage": float(100.0 * abs(total_bias) / actual_total) if actual_total else float("nan"),
    }


def savefig(path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    plt.tight_layout()
    plt.savefig(path, dpi=300, bbox_inches="tight")
    plt.close()


def preprocessor(numeric: list[str], categorical: list[str], product_aware: bool) -> ColumnTransformer:
    cats = list(categorical) + ([PID] if product_aware else [])
    return ColumnTransformer(
        [
            ("numeric", Pipeline([("imputer", SimpleImputer(strategy="median"))]), numeric),
            ("categorical", Pipeline([
                ("imputer", SimpleImputer(strategy="most_frequent")),
                ("ordinal", OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1, encoded_missing_value=-2)),
            ]), cats),
        ],
        remainder="drop",
        sparse_threshold=0.0,
        verbose_feature_names_out=False,
    )


def model_registry() -> list[dict]:
    items = [
        dict(method="HISTGB_SQUARED_CORE53", library="sklearn", feature_set="CORE53", objective="squared_error", available=True,
             params=dict(loss="squared_error", learning_rate=0.05, max_iter=250, max_leaf_nodes=31, min_samples_leaf=20, l2_regularization=1.0, random_state=SEED)),
        dict(method="HISTGB_POISSON_CORE53", library="sklearn", feature_set="CORE53", objective="poisson", available=True,
             params=dict(loss="poisson", learning_rate=0.05, max_iter=250, max_leaf_nodes=31, min_samples_leaf=20, l2_regularization=1.0, random_state=SEED)),
    ]
    for method, feature_set, objective in [
        ("XGBOOST_SQUARED_CORE53", "CORE53", "reg:squarederror"),
        ("XGBOOST_POISSON_CORE53", "CORE53", "count:poisson"),
        ("XGBOOST_TWEEDIE_CORE53", "CORE53", "reg:tweedie"),
        ("XGBOOST_SQUARED_PRODUCT_AWARE", "CORE53_PLUS_PRODUCT_ID", "reg:squarederror"),
    ]:
        params = dict(objective=objective, n_estimators=300, max_depth=6, learning_rate=0.04, subsample=0.85,
                      colsample_bytree=0.85, min_child_weight=3, reg_lambda=2.0, reg_alpha=0.0,
                      n_jobs=THREADS, random_state=SEED, verbosity=0)
        if objective == "reg:tweedie":
            params["tweedie_variance_power"] = 1.2
        items.append(dict(method=method, library="xgboost", feature_set=feature_set, objective=objective,
                          available=XGBOOST_AVAILABLE, params=params))
    for method, objective in [("LIGHTGBM_TWEEDIE_CORE53", "tweedie")]:
        params = dict(objective=objective, n_estimators=300, learning_rate=0.04, num_leaves=31, max_depth=-1,
                      min_child_samples=20, subsample=0.85, colsample_bytree=0.85, reg_lambda=2.0, reg_alpha=0.0,
                      n_jobs=THREADS, random_state=SEED, verbosity=-1)
        if objective == "tweedie":
            params["tweedie_variance_power"] = 1.2
        items.append(dict(method=method, library="lightgbm", feature_set="CORE53", objective=objective,
                          available=LIGHTGBM_AVAILABLE, params=params))
    for method, feature_set, objective in [
        ("CATBOOST_RMSE_CORE53", "CORE53", "RMSE"),
    ]:
        items.append(dict(method=method, library="catboost", feature_set=feature_set, objective=objective,
                          available=CATBOOST_AVAILABLE,
                          params=dict(loss_function=objective, iterations=250, depth=7, learning_rate=0.04,
                                      l2_leaf_reg=5.0, random_seed=SEED, verbose=False,
                                      allow_writing_files=False, thread_count=THREADS)))
    return items


def instantiate(spec: dict):
    if spec["library"] == "sklearn":
        return HistGradientBoostingRegressor(**spec["params"])
    if spec["library"] == "xgboost":
        return XGBRegressor(**spec["params"])
    if spec["library"] == "lightgbm":
        return LGBMRegressor(**spec["params"])
    if spec["library"] == "catboost":
        return CatBoostRegressor(**spec["params"])
    raise ValueError(spec["library"])


# =============================================================================
# PREFLIGHT
# =============================================================================

required = [MAIN_MODEL_PATH, PREDICTOR_LIST_PATH, ND03_MANIFEST_PATH, ND03_CHECKPOINT_PATH,
            FOLD_PATH, SELECTED_METHODS_PATH, ND04_ROUTED_PATH, ND04_MANIFEST_PATH, ND04_CHECKPOINT_PATH,
            ND05_METRICS_PATH, ND05_MANIFEST_PATH, ND05_CHECKPOINT_PATH,
            AGENTS_PATH, CURRENT_HANDOFF_PATH, WORKFLOW_PATH, DECISIONS_PATH, METRICS_MEMORY_PATH]
missing = [path for path in required if not path.is_file()]
if missing:
    raise FileNotFoundError("ND06 required files are missing:\n" + "\n".join(f"- {path}" for path in missing))

checkpoint_hashes = {
    "ND03": sha256_file(ND03_CHECKPOINT_PATH),
    "ND04": sha256_file(ND04_CHECKPOINT_PATH),
    "ND05": sha256_file(ND05_CHECKPOINT_PATH),
}
for step, expected in EXPECTED_CHECKPOINTS.items():
    if checkpoint_hashes[step] != expected:
        raise AssertionError(f"{step} checkpoint mismatch. Expected {expected}; actual {checkpoint_hashes[step]}")

if ND06_ROOT.exists() and not ALLOW_OVERWRITE:
    raise FileExistsError(f"ND06 already exists; no files changed:\n{ND06_ROOT}")
if ND06_ROOT.exists():
    shutil.rmtree(ND06_ROOT)
for path in [TOP_CHECKPOINT, TOP_CHECKPOINT_SHA]:
    if path.exists() and not ALLOW_OVERWRITE:
        raise FileExistsError(f"ND06 checkpoint already exists; no files changed:\n{path}")
    if path.exists():
        path.unlink()

STAGING = ND06_ROOT.parent / f".ND06_staging_{uuid.uuid4().hex}"
STAGING.mkdir(parents=True, exist_ok=False)


# =============================================================================
# EXECUTION
# =============================================================================

try:
    data = pd.read_csv(MAIN_MODEL_PATH, low_memory=False)
    predictors = pd.read_csv(PREDICTOR_LIST_PATH, low_memory=False).sort_values("PredictorOrder")
    folds = pd.read_csv(FOLD_PATH, low_memory=False)
    selected_methods = pd.read_csv(SELECTED_METHODS_PATH, low_memory=False)
    nd04_routed = pd.read_csv(ND04_ROUTED_PATH, low_memory=False)
    nd05_metrics = pd.read_csv(ND05_METRICS_PATH, low_memory=False)
    manifests = {
        "ND03": pd.read_csv(ND03_MANIFEST_PATH, low_memory=False),
        "ND04": pd.read_csv(ND04_MANIFEST_PATH, low_memory=False),
        "ND05": pd.read_csv(ND05_MANIFEST_PATH, low_memory=False),
    }

    data[DATE] = pd.to_datetime(data[DATE], errors="raise")
    data[TARGET] = pd.to_numeric(data[TARGET], errors="raise").astype(float)
    if len(data) != EXPECTED_ROWS:
        raise AssertionError(f"Expected {EXPECTED_ROWS} main rows; found {len(data)}")
    if set(data[ROUTE].astype(str).unique()) != {"MAIN_MODEL"}:
        raise AssertionError("Main development dataset contains non-main routes")
    if data["IsOpenedMarchDiagnosticPeriod"].astype(bool).any():
        raise AssertionError("March rows found in ND06 development data")
    if not data["EligibleForMethodSelection"].astype(bool).all():
        raise AssertionError("Ineligible rows found in development data")

    core = predictors["Predictor"].astype(str).tolist()
    numeric = predictors.loc[predictors["PredictorType"].eq("NUMERIC"), "Predictor"].astype(str).tolist()
    categorical = predictors.loc[predictors["PredictorType"].eq("CATEGORICAL"), "Predictor"].astype(str).tolist()
    if (len(core), len(numeric), len(categorical)) != (EXPECTED_PREDICTORS, EXPECTED_NUMERIC, EXPECTED_CATEGORICAL):
        raise AssertionError(f"Unexpected predictor counts: {len(core)}, {len(numeric)}, {len(categorical)}")
    if TARGET in core or "BulkDemand" in core or "TotalDemand" in core or PID in core:
        raise AssertionError("Forbidden target/bulk/ID field found in core predictor contract")
    missing_core = sorted(set(core) - set(data.columns))
    if missing_core:
        raise AssertionError("Missing predictors:\n" + "\n".join(missing_core))

    for col in ["TrainStart", "TrainEnd", "ValidationStart", "ValidationEnd"]:
        folds[col] = pd.to_datetime(folds[col], errors="raise")
    if len(folds) != EXPECTED_FOLDS or not (folds["TrainEnd"] < folds["ValidationStart"]).all():
        raise AssertionError("Fold definition is invalid")

    benchmark_row = selected_methods.loc[selected_methods["ForecastRoute"].astype(str).eq("MAIN_MODEL")]
    if len(benchmark_row) != 1:
        raise AssertionError("Expected one ND04 main benchmark")
    benchmark_method = str(benchmark_row.iloc[0]["SelectedMethod"])
    benchmark_wape = float(benchmark_row.iloc[0]["WAPEPercentage"])
    benchmark_mae = float(benchmark_row.iloc[0]["MAE"])
    benchmark_rmse = float(benchmark_row.iloc[0]["RMSE"])
    benchmark_bias = float(benchmark_row.iloc[0]["TotalBias"])
    benchmark_abs_bias = float(benchmark_row.iloc[0]["AbsoluteBiasPercentage"])
    if benchmark_method != EXPECTED_BENCHMARK or not math.isclose(benchmark_wape, EXPECTED_BENCHMARK_WAPE, abs_tol=1e-6):
        raise AssertionError("ND04 benchmark differs from accepted benchmark")

    # Manifest verification of critical inputs.
    lookups = {step: dict(zip(frame["RelativePath"].astype(str), frame["SHA256"].astype(str))) for step, frame in manifests.items()}
    input_specs = [
        ("ND03", MAIN_MODEL_PATH, ND03_ROOT), ("ND03", PREDICTOR_LIST_PATH, ND03_ROOT),
        ("ND04", FOLD_PATH, ND04_ROOT), ("ND04", SELECTED_METHODS_PATH, ND04_ROOT),
        ("ND04", ND04_ROUTED_PATH, ND04_ROOT), ("ND05", ND05_METRICS_PATH, ND05_ROOT),
    ]
    hash_rows = []
    for step, path, root in input_specs:
        relative = str(path.relative_to(root)); actual = sha256_file(path); expected = lookups[step].get(relative)
        if expected is None or expected != actual:
            raise AssertionError(f"{step} manifest verification failed for {relative}")
        hash_rows.append(dict(SourceStep=step, InputPath=str(path), RelativePath=relative, SHA256=actual, MatchesManifest=True))
    input_hash_audit = pd.DataFrame(hash_rows)
    protected_before = {str(path): sha256_file(path) for path in required if path.is_file()}

    registry = model_registry()
    registry_frame = pd.DataFrame([
        dict(CandidateMethod=s["method"], Library=s["library"], FeatureSet=s["feature_set"], Objective=s["objective"],
             Available=bool(s["available"]), ParametersJSON=json.dumps(s["params"], sort_keys=True))
        for s in registry
    ])
    available = [s for s in registry if s["available"]]
    if len(available) < 2:
        raise RuntimeError("Fewer than two expanded candidates are available")

    versions = pd.DataFrame([
        ("python", platform.python_version(), True), ("pandas", pd.__version__, True),
        ("numpy", np.__version__, True), ("scikit-learn", sklearn.__version__, True),
        ("xgboost", xgboost.__version__ if XGBOOST_AVAILABLE else "NOT_AVAILABLE", XGBOOST_AVAILABLE),
        ("lightgbm", lightgbm.__version__ if LIGHTGBM_AVAILABLE else "NOT_AVAILABLE", LIGHTGBM_AVAILABLE),
        ("catboost", catboost.__version__ if CATBOOST_AVAILABLE else "NOT_AVAILABLE", CATBOOST_AVAILABLE),
    ], columns=["Package", "Version", "Available"])

    prediction_parts = []
    fold_audit_rows = []
    start_all = time.perf_counter()

    for fold in folds.itertuples(index=False):
        fold_no = int(fold.Fold); vs = pd.Timestamp(fold.ValidationStart); ve = pd.Timestamp(fold.ValidationEnd)
        train = data.loc[data[DATE] < vs].copy()
        valid = data.loc[data[DATE].between(vs, ve, inclusive="both")].copy()
        if train.empty or valid.empty or train[DATE].max() >= valid[DATE].min():
            raise AssertionError(f"Invalid fold {fold_no}")
        y_train = train[TARGET].to_numpy(float); y_valid = valid[TARGET].to_numpy(float)

        matrices = {}
        for feature_set in ["CORE53", "CORE53_PLUS_PRODUCT_ID"]:
            aware = feature_set == "CORE53_PLUS_PRODUCT_ID"
            columns = core + ([PID] if aware else [])
            prep = preprocessor(numeric, categorical, aware)
            X_train = prep.fit_transform(train[columns].copy())
            X_valid = prep.transform(valid[columns].copy())
            matrices[feature_set] = (X_train, X_valid, X_train.shape[1])

        benchmark_pred = np.clip(valid["PastNormalDemandRollingMean_5"].to_numpy(float), 0, None)
        base = pd.DataFrame({
            DATE: valid[DATE].to_numpy(), PID: valid[PID].astype(str).to_numpy(), PNAME: valid[PNAME].astype(str).to_numpy(),
            FAMILY: valid[FAMILY].astype(str).to_numpy(), DOW: valid[DOW].to_numpy(), "Fold": fold_no,
            "CandidateMethod": "ROLLING_MEAN_5_BENCHMARK", "Library": "deterministic_baseline",
            "FeatureSet": "PAST_5_DAY_MEAN", "Objective": "not_applicable",
            "ActualNormalDemand": y_valid, "PredictedNormalDemand": benchmark_pred,
            "TrainingRows": len(train), "ValidationRows": len(valid), "FitSeconds": 0.0,
        })
        prediction_parts.append(base)

        for spec in available:
            method = spec["method"]; X_train, X_valid, feature_count = matrices[spec["feature_set"]]
            started = time.perf_counter()
            try:
                print(f"Fold {fold_no}/{EXPECTED_FOLDS}: fitting {method} ...", flush=True)
                model = instantiate(spec); model.fit(X_train, y_train)
                pred = np.clip(np.asarray(model.predict(X_valid), dtype=float), 0, None)
                elapsed = time.perf_counter() - started
                print(f"Fold {fold_no}/{EXPECTED_FOLDS}: completed {method} in {elapsed:.3f} seconds", flush=True)
                if not np.isfinite(pred).all():
                    raise AssertionError("non-finite predictions")
                part = base.copy()
                part["CandidateMethod"] = method; part["Library"] = spec["library"]
                part["FeatureSet"] = spec["feature_set"]; part["Objective"] = spec["objective"]
                part["PredictedNormalDemand"] = pred; part["FitSeconds"] = elapsed
                prediction_parts.append(part)
                fold_audit_rows.append(dict(Fold=fold_no, CandidateMethod=method, Library=spec["library"],
                    FeatureSet=spec["feature_set"], Objective=spec["objective"], TrainStart=train[DATE].min(),
                    TrainEnd=train[DATE].max(), ValidationStart=valid[DATE].min(), ValidationEnd=valid[DATE].max(),
                    TrainingRows=len(train), ValidationRows=len(valid), TransformedFeatureCount=feature_count,
                    FitSeconds=elapsed, FitSucceeded=True, FailureMessage=""))
            except Exception as exc:
                elapsed = time.perf_counter() - started
                fold_audit_rows.append(dict(Fold=fold_no, CandidateMethod=method, Library=spec["library"],
                    FeatureSet=spec["feature_set"], Objective=spec["objective"], TrainStart=train[DATE].min(),
                    TrainEnd=train[DATE].max(), ValidationStart=valid[DATE].min(), ValidationEnd=valid[DATE].max(),
                    TrainingRows=len(train), ValidationRows=len(valid), TransformedFeatureCount=feature_count,
                    FitSeconds=elapsed, FitSucceeded=False, FailureMessage=repr(exc)))
                print(f"WARNING: {method} failed in fold {fold_no}: {exc}")

    total_fit_seconds = time.perf_counter() - start_all
    fold_audit = pd.DataFrame(fold_audit_rows)
    completed = fold_audit.loc[fold_audit["FitSucceeded"]].groupby("CandidateMethod")["Fold"].nunique()
    complete_methods = sorted(completed.loc[completed.eq(EXPECTED_FOLDS)].index.tolist())
    if not complete_methods:
        raise AssertionError("No expanded candidate completed all folds")

    predictions = pd.concat(prediction_parts, ignore_index=True)
    predictions = predictions.loc[predictions["CandidateMethod"].eq("ROLLING_MEAN_5_BENCHMARK") | predictions["CandidateMethod"].isin(complete_methods)].copy()
    predictions["ForecastError"] = predictions["PredictedNormalDemand"] - predictions["ActualNormalDemand"]
    predictions["AbsoluteError"] = predictions["ForecastError"].abs()

    metric_rows = []
    fold_metric_rows = []
    for method, frame in predictions.groupby("CandidateMethod", sort=False):
        row = metrics(frame["ActualNormalDemand"], frame["PredictedNormalDemand"], "DAILY_PRODUCT_MAIN_ROUTE")
        row.update(CandidateMethod=method, Library=frame["Library"].iloc[0], FeatureSet=frame["FeatureSet"].iloc[0],
                   Objective=frame["Objective"].iloc[0], FoldsCompleted=frame["Fold"].nunique(),
                   ValidationDates=frame[DATE].nunique(), Products=frame[PID].nunique(),
                   TotalFitSeconds=frame.groupby("Fold")["FitSeconds"].first().sum())
        metric_rows.append(row)
        for fold_no, ff in frame.groupby("Fold"):
            r = metrics(ff["ActualNormalDemand"], ff["PredictedNormalDemand"], "DAILY_PRODUCT_MAIN_ROUTE_BY_FOLD")
            r.update(CandidateMethod=method, Fold=int(fold_no), Library=ff["Library"].iloc[0],
                     FeatureSet=ff["FeatureSet"].iloc[0], Objective=ff["Objective"].iloc[0])
            fold_metric_rows.append(r)
    all_metrics = pd.DataFrame(metric_rows)
    fold_metrics = pd.DataFrame(fold_metric_rows)

    reconstructed = float(all_metrics.loc[all_metrics["CandidateMethod"].eq("ROLLING_MEAN_5_BENCHMARK"), "WAPEPercentage"].iloc[0])
    if not math.isclose(reconstructed, benchmark_wape, abs_tol=1e-6):
        raise AssertionError(f"Benchmark WAPE was not reproduced: {reconstructed} vs {benchmark_wape}")

    benchmark_fold = fold_metrics.loc[fold_metrics["CandidateMethod"].eq("ROLLING_MEAN_5_BENCHMARK"), ["Fold", "WAPEPercentage"]].rename(columns={"WAPEPercentage": "BenchmarkFoldWAPE"})
    challengers = all_metrics.loc[~all_metrics["CandidateMethod"].eq("ROLLING_MEAN_5_BENCHMARK")].copy()
    won_rows = []
    for method in challengers["CandidateMethod"]:
        mf = fold_metrics.loc[fold_metrics["CandidateMethod"].eq(method), ["Fold", "WAPEPercentage"]].merge(benchmark_fold, on="Fold", validate="one_to_one")
        won_rows.append(dict(CandidateMethod=method,
                             FoldsWonAgainstBenchmark=int((mf["WAPEPercentage"] < mf["BenchmarkFoldWAPE"]).sum()),
                             MeanFoldWAPEDifferencePercentagePoints=float((mf["BenchmarkFoldWAPE"] - mf["WAPEPercentage"]).mean())))
    challengers = challengers.merge(pd.DataFrame(won_rows), on="CandidateMethod", validate="one_to_one")
    challengers["WAPEImprovementPercentagePoints"] = benchmark_wape - challengers["WAPEPercentage"]
    challengers["RelativeWAPEImprovementPercentage"] = 100 * challengers["WAPEImprovementPercentagePoints"] / benchmark_wape
    challengers["MAERatioToBenchmark"] = challengers["MAE"] / benchmark_mae
    challengers["RMSERatioToBenchmark"] = challengers["RMSE"] / benchmark_rmse
    challengers["StrictlyQualifiesForND07"] = (
        challengers["WAPEImprovementPercentagePoints"].ge(MIN_WAPE_IMPROVEMENT_PP)
        & challengers["MAERatioToBenchmark"].le(MAX_MAE_RATIO)
        & challengers["RMSERatioToBenchmark"].le(MAX_RMSE_RATIO)
        & challengers["AbsoluteBiasPercentage"].le(MAX_ABS_BIAS_PCT)
        & challengers["FoldsWonAgainstBenchmark"].ge(MIN_FOLDS_WON)
        & challengers["FoldsCompleted"].eq(EXPECTED_FOLDS)
    )
    challengers["PromisingForTuningOrCalibration"] = (
        challengers["WAPEPercentage"].le(benchmark_wape + PROMISING_WAPE_GAP_PP)
        & challengers["AbsoluteBiasPercentage"].le(PROMISING_ABS_BIAS_PCT)
        & challengers["FoldsCompleted"].eq(EXPECTED_FOLDS)
    )
    challengers = challengers.sort_values(["WAPEPercentage", "MAE", "AbsoluteBiasPercentage"]).reset_index(drop=True)
    challengers.insert(0, "ChallengeRank", np.arange(1, len(challengers) + 1))

    selected_advance = []
    for method in challengers.loc[challengers["StrictlyQualifiesForND07"], "CandidateMethod"]:
        if method not in selected_advance:
            selected_advance.append(method)
    for method in challengers.loc[challengers["PromisingForTuningOrCalibration"], "CandidateMethod"]:
        if method not in selected_advance and len(selected_advance) < MAX_ADVANCE:
            selected_advance.append(method)
    for method in challengers["CandidateMethod"]:
        if method not in selected_advance and len(selected_advance) < MIN_ADVANCE:
            selected_advance.append(method)
    selected_advance = selected_advance[:MAX_ADVANCE]

    advance = challengers[["CandidateMethod", "ChallengeRank", "Library", "FeatureSet", "Objective", "WAPEPercentage", "MAE", "RMSE",
                            "TotalBias", "AbsoluteBiasPercentage", "WAPEImprovementPercentagePoints", "RelativeWAPEImprovementPercentage",
                            "FoldsWonAgainstBenchmark", "StrictlyQualifiesForND07", "PromisingForTuningOrCalibration"]].copy()
    advance["AdvanceToND07"] = advance["CandidateMethod"].isin(selected_advance)
    advance["AdvancementReason"] = np.select(
        [advance["StrictlyQualifiesForND07"], advance["AdvanceToND07"] & advance["PromisingForTuningOrCalibration"], advance["AdvanceToND07"]],
        ["STRICT_SCREENING_RULE_PASSED", "PROMISING_FOR_TUNING_OR_CALIBRATION", "TOP_RANKED_SCREENING_CANDIDATE_FOR_COMPARISON"],
        default="NOT_ADVANCED",
    )

    best = challengers.iloc[0]
    best_method = str(best["CandidateMethod"])
    recommendation = (
        f"{best_method}_IS_PROVISIONAL_PRIMARY_CHALLENGER_FOR_ND07"
        if bool(best["StrictlyQualifiesForND07"])
        else "ROLLING_MEAN_5_REMAINS_PRIMARY_BENCHMARK_PENDING_ND07"
    )

    # Routed system using the best ND06 challenger for MAIN_MODEL and the accepted ND04 fallbacks.
    best_main = predictions.loc[predictions["CandidateMethod"].eq(best_method)].copy().sort_values([DATE, PID]).reset_index(drop=True)
    nd04_routed[DATE] = pd.to_datetime(nd04_routed[DATE], errors="raise")
    main_key = best_main[[DATE, PID, "Fold", "PredictedNormalDemand"]].rename(columns={"PredictedNormalDemand": "ND06MainPrediction"})
    routed = nd04_routed.merge(main_key, on=[DATE, PID, "Fold"], how="left", validate="one_to_one")
    main_mask = routed[ROUTE].astype(str).eq("MAIN_MODEL")
    if routed.loc[main_mask, "ND06MainPrediction"].isna().any():
        raise AssertionError("Missing ND06 prediction for a main-route row")
    routed["ND04SelectedPrediction"] = routed["PredictedNormalDemand"]
    routed.loc[main_mask, "PredictedNormalDemand"] = routed.loc[main_mask, "ND06MainPrediction"]
    routed.loc[main_mask, "SelectedMethod"] = best_method
    routed.drop(columns=["ND06MainPrediction"], inplace=True)
    routed["ForecastError"] = routed["PredictedNormalDemand"] - routed["ActualNormalDemand"]
    routed["AbsoluteError"] = routed["ForecastError"].abs()
    routed["ForecastMode"] = "DAILY_UPDATED_ONE_STEP_VALIDATION"
    routed["WeeklyInterpretation"] = "DAILY_UPDATED_AGGREGATION_DIAGNOSTIC_NOT_WEEK_START_RECURSIVE"

    daily_total = routed.groupby(DATE, as_index=False).agg(ActualNormalDemand=("ActualNormalDemand", "sum"),
        PredictedNormalDemand=("PredictedNormalDemand", "sum"), ProductRows=(PID, "size"), Products=(PID, "nunique"))
    daily_total["ForecastError"] = daily_total["PredictedNormalDemand"] - daily_total["ActualNormalDemand"]

    routed["WeekStart"] = routed[DATE] - pd.to_timedelta(routed[DATE].dt.dayofweek, unit="D")
    date_week = routed[["WeekStart", DATE]].drop_duplicates().assign(Weekday=lambda x: x[DATE].dt.dayofweek)
    week_meta = date_week.groupby("WeekStart").agg(OperatingDates=(DATE, "nunique"),
        ObservedWeekdays=("Weekday", lambda x: tuple(sorted(set(x))))).reset_index()
    week_meta["IsCompleteMondayToFridayWeek"] = week_meta["ObservedWeekdays"].apply(lambda x: x == (0, 1, 2, 3, 4))
    weekly_product = routed.groupby(["WeekStart", PID, PNAME], as_index=False).agg(
        ActualNormalDemand=("ActualNormalDemand", "sum"), PredictedNormalDemand=("PredictedNormalDemand", "sum"), ProductDateRows=(DATE, "size"))
    weekly_product = weekly_product.merge(week_meta, on="WeekStart", validate="many_to_one")
    weekly_product["ForecastError"] = weekly_product["PredictedNormalDemand"] - weekly_product["ActualNormalDemand"]
    weekly_total = weekly_product.groupby("WeekStart", as_index=False).agg(ActualNormalDemand=("ActualNormalDemand", "sum"),
        PredictedNormalDemand=("PredictedNormalDemand", "sum"), Products=(PID, "nunique"), OperatingDates=("OperatingDates", "first"),
        ObservedWeekdays=("ObservedWeekdays", "first"), IsCompleteMondayToFridayWeek=("IsCompleteMondayToFridayWeek", "first"))
    weekly_total["ForecastError"] = weekly_total["PredictedNormalDemand"] - weekly_total["ActualNormalDemand"]

    routed_metrics = pd.DataFrame([
        metrics(routed["ActualNormalDemand"], routed["PredictedNormalDemand"], "DAILY_PRODUCT_ALL_ROUTES"),
        metrics(daily_total["ActualNormalDemand"], daily_total["PredictedNormalDemand"], "DAILY_RESTAURANT_TOTAL_ALL_ROUTES"),
        metrics(weekly_product["ActualNormalDemand"], weekly_product["PredictedNormalDemand"], "WEEKLY_PRODUCT_DAILY_UPDATED_AGGREGATION_ALL_WEEKS"),
        metrics(weekly_product.loc[weekly_product["IsCompleteMondayToFridayWeek"], "ActualNormalDemand"],
                weekly_product.loc[weekly_product["IsCompleteMondayToFridayWeek"], "PredictedNormalDemand"],
                "WEEKLY_PRODUCT_DAILY_UPDATED_AGGREGATION_COMPLETE_WEEKS"),
        metrics(weekly_total["ActualNormalDemand"], weekly_total["PredictedNormalDemand"], "WEEKLY_RESTAURANT_DAILY_UPDATED_AGGREGATION_ALL_WEEKS"),
        metrics(weekly_total.loc[weekly_total["IsCompleteMondayToFridayWeek"], "ActualNormalDemand"],
                weekly_total.loc[weekly_total["IsCompleteMondayToFridayWeek"], "PredictedNormalDemand"],
                "WEEKLY_RESTAURANT_DAILY_UPDATED_AGGREGATION_COMPLETE_WEEKS"),
    ])
    routed_metrics["MainRouteMethod"] = best_method
    routed_metrics["ForecastMode"] = "DAILY_UPDATED_AGGREGATION_DIAGNOSTIC"
    routed_metrics["WeekStartRecursiveForecast"] = False

    route_rows = []
    for route, frame in routed.groupby(ROUTE):
        row = metrics(frame["ActualNormalDemand"], frame["PredictedNormalDemand"], "DAILY_PRODUCT_BY_ROUTE")
        row.update(ForecastRoute=route, SelectedMethod=frame["SelectedMethod"].iloc[0], Products=frame[PID].nunique(), Dates=frame[DATE].nunique())
        route_rows.append(row)
    route_metrics = pd.DataFrame(route_rows)

    # Key comparison.
    h75 = nd05_metrics.loc[nd05_metrics["CandidateMethod"].astype(str).eq("BLEND_HURDLE_NAIVE5__H75")]
    if len(h75) != 1:
        raise AssertionError("Expected one ND05 H75 row")
    h75 = h75.iloc[0]
    key_comparison = pd.DataFrame([
        dict(Method="ROLLING_MEAN_5_BENCHMARK", MethodGroup="ND04_BASELINE", FeatureSet="PAST_5_DAY_MEAN",
             WAPEPercentage=benchmark_wape, MAE=benchmark_mae, RMSE=benchmark_rmse,
             TotalBias=benchmark_bias, AbsoluteBiasPercentage=benchmark_abs_bias),
        dict(Method="BLEND_HURDLE_NAIVE5__H75", MethodGroup="ND05_ORIGINAL_ARCHITECTURE", FeatureSet="CORE53",
             WAPEPercentage=float(h75["WAPEPercentage"]), MAE=float(h75["MAE"]), RMSE=float(h75["RMSE"]),
             TotalBias=float(h75["TotalBias"]), AbsoluteBiasPercentage=float(h75["AbsoluteBiasPercentage"])),
        dict(Method=best_method, MethodGroup="ND06_BEST_CHALLENGER", FeatureSet=str(best["FeatureSet"]),
             WAPEPercentage=float(best["WAPEPercentage"]), MAE=float(best["MAE"]), RMSE=float(best["RMSE"]),
             TotalBias=float(best["TotalBias"]), AbsoluteBiasPercentage=float(best["AbsoluteBiasPercentage"])),
    ])
    key_comparison["WAPEImprovementVersusBenchmarkPP"] = benchmark_wape - key_comparison["WAPEPercentage"]

    # Audits.
    counts = predictions.groupby("CandidateMethod").size()
    duplicate_keys = int(predictions.duplicated([DATE, PID, "Fold", "CandidateMethod"]).sum())
    nonfinite = int((~np.isfinite(predictions["PredictedNormalDemand"].to_numpy(float))).sum())
    negative = int((predictions["PredictedNormalDemand"] < 0).sum())
    protocol = pd.DataFrame([
        ("March target vault opened", False, False, True),
        ("March rows in development data", 0, int(data["IsOpenedMarchDiagnosticPeriod"].astype(bool).sum()), int(data["IsOpenedMarchDiagnosticPeriod"].astype(bool).sum()) == 0),
        ("Training strictly before validation", True, bool((folds["TrainEnd"] < folds["ValidationStart"]).all()), bool((folds["TrainEnd"] < folds["ValidationStart"]).all())),
        ("Target absent from core predictors", True, TARGET not in core, TARGET not in core),
        ("Bulk absent from core predictors", True, "BulkDemand" not in core, "BulkDemand" not in core),
        ("TotalDemand absent from core predictors", True, "TotalDemand" not in core, "TotalDemand" not in core),
        ("Product ID absent from core contract", True, PID not in core, PID not in core),
        ("Product-aware candidates explicitly labelled", True,
         registry_frame.loc[registry_frame["FeatureSet"].eq("CORE53_PLUS_PRODUCT_ID"), "CandidateMethod"].str.contains("PRODUCT_AWARE").all(),
         registry_frame.loc[registry_frame["FeatureSet"].eq("CORE53_PLUS_PRODUCT_ID"), "CandidateMethod"].str.contains("PRODUCT_AWARE").all()),
        ("Final production model fitted", False, False, True),
        ("Final model artifact saved", False, False, True),
        ("Weekly diagnostic claimed as week-start recursive", False, False, True),
    ], columns=["Check", "Expected", "Actual", "Passed"])

    validation = pd.DataFrame([
        ("ND03 checkpoint verified", EXPECTED_CHECKPOINTS["ND03"], checkpoint_hashes["ND03"], checkpoint_hashes["ND03"] == EXPECTED_CHECKPOINTS["ND03"]),
        ("ND04 checkpoint verified", EXPECTED_CHECKPOINTS["ND04"], checkpoint_hashes["ND04"], checkpoint_hashes["ND04"] == EXPECTED_CHECKPOINTS["ND04"]),
        ("ND05 checkpoint verified", EXPECTED_CHECKPOINTS["ND05"], checkpoint_hashes["ND05"], checkpoint_hashes["ND05"] == EXPECTED_CHECKPOINTS["ND05"]),
        ("Prediction rows per completed method", EXPECTED_VALIDATION_ROWS, int(counts.min()), bool((counts == EXPECTED_VALIDATION_ROWS).all())),
        ("Validation operating dates", EXPECTED_VALIDATION_DATES, best_main[DATE].nunique(), best_main[DATE].nunique() == EXPECTED_VALIDATION_DATES),
        ("Validation folds", EXPECTED_FOLDS, best_main["Fold"].nunique(), best_main["Fold"].nunique() == EXPECTED_FOLDS),
        ("Duplicate prediction keys", 0, duplicate_keys, duplicate_keys == 0),
        ("Non-finite predictions", 0, nonfinite, nonfinite == 0),
        ("Negative predictions", 0, negative, negative == 0),
        ("Benchmark WAPE reproduced", benchmark_wape, reconstructed, math.isclose(reconstructed, benchmark_wape, abs_tol=1e-6)),
        ("Routed rows missing predictions", 0, int(routed["PredictedNormalDemand"].isna().sum()), int(routed["PredictedNormalDemand"].isna().sum()) == 0),
        ("At least one candidate advanced", True, len(selected_advance) > 0, len(selected_advance) > 0),
        ("March target vault opened", False, False, True),
        ("Previous inputs modified", False, False, True),
        ("ND06 step lock created", False, False, True),
    ], columns=["Check", "Expected", "Actual", "Passed"])
    if not protocol["Passed"].all():
        raise AssertionError("Protocol audit failed:\n" + protocol.loc[~protocol["Passed"]].to_string(index=False))
    if not validation["Passed"].all():
        raise AssertionError("Validation failed:\n" + validation.loc[~validation["Passed"]].to_string(index=False))

    # Staging paths.
    def staged(path: Path) -> Path:
        return STAGING / path.relative_to(ND06_ROOT)

    # Figures.
    plot = challengers.sort_values("WAPEPercentage")
    plt.figure(figsize=(12, 8)); plt.barh(plot["CandidateMethod"], plot["WAPEPercentage"]); plt.axvline(benchmark_wape, linestyle="--", label=f"ROLLING_MEAN_5: {benchmark_wape:.2f}%")
    plt.title("ND06 main-route candidate WAPE"); plt.xlabel("Daily product WAPE (%)"); plt.legend(); plt.grid(axis="x", alpha=0.3); savefig(staged(FIG_DIR / "ND06_figure_01_candidate_wape.png"))

    plt.figure(figsize=(12, 7)); plt.scatter(challengers["WAPEPercentage"], challengers["AbsoluteBiasPercentage"])
    for _, row in challengers.iterrows():
        plt.annotate(row["CandidateMethod"], (row["WAPEPercentage"], row["AbsoluteBiasPercentage"]), fontsize=7, xytext=(3, 3), textcoords="offset points")
    plt.axvline(benchmark_wape, linestyle="--"); plt.axhline(MAX_ABS_BIAS_PCT, linestyle="--")
    plt.title("WAPE versus absolute aggregate bias"); plt.xlabel("WAPE (%)"); plt.ylabel("Absolute bias (%)"); plt.grid(alpha=0.3); savefig(staged(FIG_DIR / "ND06_figure_02_wape_vs_bias.png"))

    fold_plot = fold_metrics.loc[fold_metrics["CandidateMethod"].isin(["ROLLING_MEAN_5_BENCHMARK", best_method])]
    plt.figure(figsize=(11, 6))
    for method, frame in fold_plot.groupby("CandidateMethod"):
        plt.plot(frame["Fold"], frame["WAPEPercentage"], marker="o", label=method)
    plt.title("Best challenger and benchmark WAPE by fold"); plt.xlabel("Fold"); plt.ylabel("WAPE (%)"); plt.xticks(range(1, 6)); plt.legend(); plt.grid(alpha=0.3); savefig(staged(FIG_DIR / "ND06_figure_03_best_vs_benchmark_by_fold.png"))

    maximum = max(best_main["ActualNormalDemand"].max(), best_main["PredictedNormalDemand"].max())
    plt.figure(figsize=(8, 8)); plt.scatter(best_main["ActualNormalDemand"], best_main["PredictedNormalDemand"], alpha=0.35); plt.plot([0, maximum], [0, maximum], linestyle="--")
    plt.title(f"{best_method}: actual versus predicted"); plt.xlabel("Actual normal demand"); plt.ylabel("Predicted normal demand"); plt.grid(alpha=0.3); savefig(staged(FIG_DIR / "ND06_figure_04_best_actual_vs_predicted.png"))

    plt.figure(figsize=(14, 6)); plt.plot(daily_total[DATE], daily_total["ActualNormalDemand"], label="Actual"); plt.plot(daily_total[DATE], daily_total["PredictedNormalDemand"], label="Predicted")
    plt.title("Best-challenger routed system: daily restaurant total"); plt.xlabel("Date"); plt.ylabel("Units"); plt.gca().xaxis.set_major_formatter(DateFormatter("%Y-%m-%d")); plt.xticks(rotation=45, ha="right"); plt.legend(); plt.grid(alpha=0.3); savefig(staged(FIG_DIR / "ND06_figure_05_daily_restaurant_actual_vs_forecast.png"))

    plt.figure(figsize=(14, 6)); plt.plot(weekly_total["WeekStart"], weekly_total["ActualNormalDemand"], marker="o", label="Actual"); plt.plot(weekly_total["WeekStart"], weekly_total["PredictedNormalDemand"], marker="o", label="Predicted")
    plt.title("Best-challenger routed weekly diagnostic"); plt.xlabel("Week starting"); plt.ylabel("Units"); plt.gca().xaxis.set_major_formatter(DateFormatter("%Y-%m-%d")); plt.xticks(rotation=45, ha="right"); plt.legend(); plt.grid(alpha=0.3); savefig(staged(FIG_DIR / "ND06_figure_06_weekly_restaurant_diagnostic.png"))

    plt.figure(figsize=(10, 6)); plt.bar(key_comparison["Method"], key_comparison["WAPEPercentage"]); plt.title("Benchmark, original hurdle, and best ND06 challenger"); plt.ylabel("WAPE (%)"); plt.xticks(rotation=25, ha="right"); plt.grid(axis="y", alpha=0.3); savefig(staged(FIG_DIR / "ND06_figure_07_key_method_comparison.png"))

    plt.figure(figsize=(10, 6)); plt.bar(routed_metrics["EvaluationLevel"], routed_metrics["WAPEPercentage"]); plt.title("Best-challenger routed WAPE by level"); plt.ylabel("WAPE (%)"); plt.xticks(rotation=40, ha="right"); plt.grid(axis="y", alpha=0.3); savefig(staged(FIG_DIR / "ND06_figure_08_routed_wape_by_level.png"))

    # Reports.
    advanced_text = "\n".join(f"- `{method}`" for method in selected_advance)
    report = f"""# ND06 Expanded Model Challenge Summary

## Status

`{STATUS}`

## Validation

- Fixed chronological folds: {EXPECTED_FOLDS}
- Validation dates: {EXPECTED_VALIDATION_DATES}
- Main validation rows: {EXPECTED_VALIDATION_ROWS:,}
- March targets opened: no
- Final production model fitted: no

## Benchmark

- Method: `{benchmark_method}`
- WAPE: {benchmark_wape:.6f}%
- MAE: {benchmark_mae:.6f}
- RMSE: {benchmark_rmse:.6f}
- Absolute bias: {benchmark_abs_bias:.6f}%

## Best challenger

- Method: `{best_method}`
- Feature set: `{best['FeatureSet']}`
- WAPE: {best['WAPEPercentage']:.6f}%
- MAE: {best['MAE']:.6f}
- RMSE: {best['RMSE']:.6f}
- Total bias: {best['TotalBias']:.6f}
- Absolute bias: {best['AbsoluteBiasPercentage']:.6f}%
- WAPE improvement: {best['WAPEImprovementPercentagePoints']:.6f} percentage points
- Folds won: {int(best['FoldsWonAgainstBenchmark'])}/{EXPECTED_FOLDS}
- Strict qualification: {bool(best['StrictlyQualifiesForND07'])}

## Recommendation

`{recommendation}`

## Advance to ND07

{advanced_text}

Product-aware candidates are explicitly labelled. Product ID is known at forecast time, but it is not silently added to the 53-predictor core contract.

Weekly outputs are daily-updated aggregation diagnostics and are not Monday-origin recursive forecasts.
"""
    decision = {
        "StepID": STEP_ID, "Status": STATUS, "CreatedUTC": NOW_UTC.isoformat(),
        "BenchmarkMethod": benchmark_method, "BenchmarkWAPEPercentage": benchmark_wape,
        "BestChallengerMethod": best_method, "BestChallengerFeatureSet": str(best["FeatureSet"]),
        "BestChallengerWAPEPercentage": float(best["WAPEPercentage"]),
        "BestChallengerMAE": float(best["MAE"]), "BestChallengerRMSE": float(best["RMSE"]),
        "BestChallengerTotalBias": float(best["TotalBias"]),
        "BestChallengerAbsoluteBiasPercentage": float(best["AbsoluteBiasPercentage"]),
        "StrictlyQualified": bool(best["StrictlyQualifiesForND07"]),
        "DevelopmentRecommendation": recommendation, "AdvanceToND07": selected_advance,
        "MarchTargetVaultOpened": False, "FinalProductionModelFitted": False,
        "WeekStartRecursiveForecastEvaluated": False,
    }
    readme = f"""# ND06 Expanded Model Challenge

Status: `{STATUS}`

This screening stage compares boosting regressors with `ROLLING_MEAN_5` using the exact ND04 folds. Candidates advanced to ND07 are not final models. March targets remained closed. Weekly outputs are daily-updated diagnostics, not week-start forecasts.
"""

    output_frames = {
        PATHS["candidate_predictions"]: predictions, PATHS["best_main"]: best_main, PATHS["routed"]: routed,
        PATHS["daily_total"]: daily_total, PATHS["weekly_product"]: weekly_product, PATHS["weekly_total"]: weekly_total,
        PATHS["metrics"]: pd.concat([all_metrics.loc[all_metrics["CandidateMethod"].eq("ROLLING_MEAN_5_BENCHMARK")], challengers], ignore_index=True, sort=False),
        PATHS["fold_metrics"]: fold_metrics, PATHS["advance"]: advance, PATHS["key_comparison"]: key_comparison,
        PATHS["routed_metrics"]: routed_metrics, PATHS["route_metrics"]: route_metrics,
        PATHS["registry"]: registry_frame, PATHS["versions"]: versions, PATHS["fold_audit"]: fold_audit,
        PATHS["hash_audit"]: input_hash_audit, PATHS["protocol"]: protocol, PATHS["validation"]: validation,
    }
    for path, frame in output_frames.items():
        write_csv(staged(path), frame)
    write_text(staged(PATHS["report"]), report)
    write_json(staged(PATHS["decision"]), decision)
    write_text(STAGING / "README.md", readme)

    protected_after = {str(path): sha256_file(path) for path in required if path.is_file()}
    changed = [path for path in protected_before if protected_before[path] != protected_after[path]]
    if changed:
        raise AssertionError("Protected inputs changed:\n" + "\n".join(changed))

    excluded = {PATHS["manifest"].name, PATHS["checkpoint"].name, PATHS["checkpoint_sha"].name}
    files = sorted(path for path in STAGING.rglob("*") if path.is_file() and path.name not in excluded)
    manifest = pd.DataFrame([dict(RelativePath=str(path.relative_to(STAGING)), Bytes=path.stat().st_size, SHA256=sha256_file(path)) for path in files]).sort_values("RelativePath")
    write_csv(staged(PATHS["manifest"]), manifest)
    manifest_sha = sha256_file(staged(PATHS["manifest"]))

    checkpoint = {
        "StepID": STEP_ID, "Status": STATUS, "CreatedUTC": NOW_UTC.isoformat(), "CreatedLocal": NOW_LOCAL.isoformat(),
        "ND06Root": str(ND06_ROOT), "InputCheckpoints": checkpoint_hashes,
        "CandidatesRegistered": len(registry_frame), "CandidatesAvailable": int(registry_frame["Available"].sum()),
        "CandidatesCompletingAllFolds": len(complete_methods), "TotalFittingSeconds": total_fit_seconds,
        "Benchmark": dict(Method=benchmark_method, WAPEPercentage=benchmark_wape, MAE=benchmark_mae, RMSE=benchmark_rmse,
                          TotalBias=benchmark_bias, AbsoluteBiasPercentage=benchmark_abs_bias),
        "BestChallenger": dict(Method=best_method, FeatureSet=str(best["FeatureSet"]), WAPEPercentage=float(best["WAPEPercentage"]),
                               MAE=float(best["MAE"]), RMSE=float(best["RMSE"]), TotalBias=float(best["TotalBias"]),
                               AbsoluteBiasPercentage=float(best["AbsoluteBiasPercentage"]),
                               WAPEImprovementPercentagePoints=float(best["WAPEImprovementPercentagePoints"]),
                               FoldsWonAgainstBenchmark=int(best["FoldsWonAgainstBenchmark"]),
                               StrictlyQualifiesForND07=bool(best["StrictlyQualifiesForND07"])),
        "DevelopmentRecommendation": recommendation, "AdvanceToND07": selected_advance,
        "RoutedMetrics": routed_metrics.to_dict(orient="records"),
        "Manifest": dict(Path=str(PATHS["manifest"]), SHA256=manifest_sha),
        "Safety": dict(MarchTargetVaultOpened=False, ModelsFittedOnlyWithinFolds=True, FinalProductionModelFitted=False,
                       FinalModelArtifactSaved=False, PreviousInputsModified=False, ExistingLocksModified=False,
                       ND06StepLockCreated=False, CheckpointAndHashesCreated=True),
        "ReadyForND07": True, "NextStep": "ND07",
    }
    write_json(staged(PATHS["checkpoint"]), checkpoint)
    checkpoint_sha = sha256_file(staged(PATHS["checkpoint"]))
    write_text(staged(PATHS["checkpoint_sha"]), f"{checkpoint_sha}  {PATHS['checkpoint'].name}\n")

    os.replace(STAGING, ND06_ROOT)
    TOP_CHECKPOINT.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(PATHS["checkpoint"], TOP_CHECKPOINT)
    shutil.copy2(PATHS["checkpoint_sha"], TOP_CHECKPOINT_SHA)

    handoff = f"""# ND06 Handoff

- Status: `{STATUS}`
- ND06 root: `{ND06_ROOT}`
- Checkpoint SHA-256: `{checkpoint_sha}`
- Accepted benchmark: `{benchmark_method}` at {benchmark_wape:.6f}% WAPE
- Best challenger: `{best_method}` at {best['WAPEPercentage']:.6f}% WAPE
- Best challenger feature set: `{best['FeatureSet']}`
- Strict qualification: {bool(best['StrictlyQualifiesForND07'])}
- Development recommendation: `{recommendation}`
- Advance to ND07: {", ".join(selected_advance)}
- March targets opened: no
- Final model fitted: no

ND07 will tune, calibrate, and test blends for the advanced candidates using chronological procedures.
"""
    atomic_write_text(HANDOFF_PATH, handoff)
    atomic_write_text(CURRENT_HANDOFF_PATH, handoff)
    append_once(WORKFLOW_PATH, "## ND06 — Expanded non-hurdle model challenge", f"""## ND06 — Expanded non-hurdle model challenge

Status: `{STATUS}`

Best challenger: `{best_method}`. Recommendation: `{recommendation}`. Advanced to ND07: {", ".join(selected_advance)}. March remained closed and no final model was fitted.
""")
    append_once(DECISIONS_PATH, "## ND06 decisions", f"""## ND06 decisions

1. `ROLLING_MEAN_5` remains the benchmark unless a challenger passes the strict rule.
2. Product-aware candidates are explicitly labelled.
3. Advanced to ND07: {", ".join(selected_advance)}.
4. Recommendation: `{recommendation}`.
5. Weekly outputs remain daily-updated diagnostics.
""")
    append_once(METRICS_MEMORY_PATH, "## ND06 expanded model challenge", f"""## ND06 expanded model challenge

- Candidates completing all folds: {len(complete_methods)}
- Benchmark WAPE: {benchmark_wape:.6f}%
- Best challenger: `{best_method}`
- Best challenger WAPE: {best['WAPEPercentage']:.6f}%
- WAPE improvement: {best['WAPEImprovementPercentagePoints']:.6f} percentage points
- Absolute bias: {best['AbsoluteBiasPercentage']:.6f}%
- Folds won: {int(best['FoldsWonAgainstBenchmark'])}/{EXPECTED_FOLDS}
""")
    append_once(AGENTS_PATH, "Marker: ND06_AUTHORITATIVE_STATUS", f"""## ND06 authoritative status

Marker: ND06_AUTHORITATIVE_STATUS

- Status: `{STATUS}`
- Handoff: `{HANDOFF_PATH}`
- Benchmark: `{benchmark_method}`
- Best challenger: `{best_method}`
- Recommendation: `{recommendation}`
- Advance to ND07: {", ".join(selected_advance)}
- Next step: `ND07`
""")
    atomic_write_text(LOG_PATH, "\n".join([
        f"Step: {STEP_ID}", f"Status: {STATUS}", f"Created local: {NOW_LOCAL.isoformat()}",
        f"Candidates completing folds: {len(complete_methods)}", f"Total fitting seconds: {total_fit_seconds}",
        f"Benchmark: {benchmark_method}", f"Benchmark WAPE: {benchmark_wape}", f"Best challenger: {best_method}",
        f"Best challenger WAPE: {best['WAPEPercentage']}", f"Advance to ND07: {', '.join(selected_advance)}",
        f"Checkpoint SHA256: {checkpoint_sha}", "March target vault opened: False", "Final production model fitted: False", "ND06 step lock created: False", ""
    ]))

except Exception:
    if STAGING.exists():
        shutil.rmtree(STAGING)
    raise


# =============================================================================
# CONSOLE SUMMARY
# =============================================================================

print("=" * 110)
print("EDEN NORMAL-DEMAND MODEL V2 — ND06 COMPLETE")
print("=" * 110)
print(f"Status: {STATUS}")
print(f"Local time: {NOW_LOCAL.isoformat()}")
print(f"ND06 root: {ND06_ROOT}")

print("\nINPUT VERIFICATION")
for step in ["ND03", "ND04", "ND05"]:
    print(f"{step} checkpoint SHA-256: {checkpoint_hashes[step]}")
print(f"Main development rows: {len(data):,}")
print(f"Core predictors: {len(core)}")
print(f"Numeric predictors: {len(numeric)}")
print(f"Categorical predictors: {len(categorical)}")
print("March target vault opened: False")
print("Previous inputs modified: False")

print("\nPACKAGE AVAILABILITY")
print(versions.to_string(index=False))

print("\nEXPANDED CHALLENGE")
print(f"Registered candidates: {len(registry_frame)}")
print(f"Available candidates: {int(registry_frame['Available'].sum())}")
print(f"Candidates completing all folds: {len(complete_methods)}")
print(f"Folds: {EXPECTED_FOLDS}")
print(f"Validation product-date rows per method: {EXPECTED_VALIDATION_ROWS:,}")
print(f"Validation operating dates: {EXPECTED_VALIDATION_DATES}")
print(f"Total fitting seconds: {total_fit_seconds:.3f}")

print("\nACCEPTED BENCHMARK")
print(f"Method: {benchmark_method}")
print(f"WAPE: {benchmark_wape:.6f}%")
print(f"MAE: {benchmark_mae:.6f}")
print(f"RMSE: {benchmark_rmse:.6f}")
print(f"Total bias: {benchmark_bias:.6f}")
print(f"Absolute bias percentage: {benchmark_abs_bias:.6f}%")

print("\nBEST ND06 CHALLENGER")
print(f"Method: {best_method}")
print(f"Feature set: {best['FeatureSet']}")
print(f"WAPE: {best['WAPEPercentage']:.6f}%")
print(f"MAE: {best['MAE']:.6f}")
print(f"RMSE: {best['RMSE']:.6f}")
print(f"Total bias: {best['TotalBias']:.6f}")
print(f"Absolute bias percentage: {best['AbsoluteBiasPercentage']:.6f}%")
print(f"WAPE improvement versus benchmark: {best['WAPEImprovementPercentagePoints']:.6f} percentage points")
print(f"Relative WAPE improvement: {best['RelativeWAPEImprovementPercentage']:.6f}%")
print(f"Folds won against benchmark: {int(best['FoldsWonAgainstBenchmark'])}/{EXPECTED_FOLDS}")
print(f"Strictly qualifies for ND07: {bool(best['StrictlyQualifiesForND07'])}")
print(f"Development recommendation: {recommendation}")

print("\nALL CANDIDATE RESULTS")
print(challengers[["ChallengeRank", "CandidateMethod", "FeatureSet", "WAPEPercentage", "MAE", "RMSE", "TotalBias",
                   "AbsoluteBiasPercentage", "WAPEImprovementPercentagePoints", "FoldsWonAgainstBenchmark",
                   "StrictlyQualifiesForND07", "PromisingForTuningOrCalibration"]].to_string(index=False))

print("\nADVANCE TO ND07")
print(advance.loc[advance["AdvanceToND07"], ["CandidateMethod", "AdvancementReason", "WAPEPercentage",
      "AbsoluteBiasPercentage", "FoldsWonAgainstBenchmark"]].to_string(index=False))

print("\nBEST-CHALLENGER ROUTED SYSTEM")
print(routed_metrics[["EvaluationLevel", "Observations", "WAPEPercentage", "MAE", "RMSE", "TotalBias"]].to_string(index=False))
print("Forecast mode: DAILY_UPDATED_AGGREGATION_DIAGNOSTIC")
print("Week-start recursive forecast: False")

print("\nOUTPUTS")
print(f"- Candidate predictions: {PATHS['candidate_predictions']}")
print(f"- Candidate metrics: {PATHS['metrics']}")
print(f"- Fold metrics: {PATHS['fold_metrics']}")
print(f"- Advancement decision: {PATHS['advance']}")
print(f"- Key comparison: {PATHS['key_comparison']}")
print(f"- Routed predictions: {PATHS['routed']}")
print(f"- Routed metrics: {PATHS['routed_metrics']}")
print(f"- Figures: {FIG_DIR}")
print(f"- Report summary: {PATHS['report']}")
print(f"- Validation: {PATHS['validation']}")
print(f"- Manifest: {PATHS['manifest']}")
print(f"- Checkpoint: {TOP_CHECKPOINT}")
print(f"- Checkpoint SHA-256: {checkpoint_sha}")
print(f"- Agent handoff: {HANDOFF_PATH}")

print("\nSAFETY")
print("- Models fitted only inside chronological folds: True")
print("- Final production model fitted: False")
print("- Final model artifact saved: False")
print("- March target vault opened: False")
print("- Previous inputs modified: False")
print("- Existing model locks modified: False")
print("- ND06 step lock created: False")
print("- ND06 checkpoint and hashes created: True")

print("\nNEXT STEP")
print("ND07 — tune, calibrate, and evaluate blends for the advanced candidates using chronological procedures.")
print("=" * 110)

Fold 1/5: fitting HISTGB_SQUARED_CORE53 ...
Fold 1/5: completed HISTGB_SQUARED_CORE53 in 6.299 seconds
Fold 1/5: fitting HISTGB_POISSON_CORE53 ...
Fold 1/5: completed HISTGB_POISSON_CORE53 in 6.342 seconds
Fold 1/5: fitting XGBOOST_SQUARED_CORE53 ...
Fold 1/5: completed XGBOOST_SQUARED_CORE53 in 0.256 seconds
Fold 1/5: fitting XGBOOST_POISSON_CORE53 ...
Fold 1/5: completed XGBOOST_POISSON_CORE53 in 0.290 seconds
Fold 1/5: fitting XGBOOST_TWEEDIE_CORE53 ...
Fold 1/5: completed XGBOOST_TWEEDIE_CORE53 in 0.279 seconds
Fold 1/5: fitting XGBOOST_SQUARED_PRODUCT_AWARE ...
Fold 1/5: completed XGBOOST_SQUARED_PRODUCT_AWARE in 0.253 seconds
Fold 1/5: fitting LIGHTGBM_TWEEDIE_CORE53 ...
Fold 1/5: completed LIGHTGBM_TWEEDIE_CORE53 in 0.436 seconds
Fold 1/5: fitting CATBOOST_RMSE_CORE53 ...
Fold 1/5: completed CATBOOST_RMSE_CORE53 in 0.630 seconds
Fold 2/5: fitting HISTGB_SQUARED_CORE53 ...
Fold 2/5: completed HISTGB_SQUARED_CORE53 in 6.197 seconds
Fold 2/5: fitting HISTGB_POISSON_CORE53 ...
Fold 

In [8]:
from __future__ import annotations

# =============================================================================
# EDEN NORMAL-DEMAND MODEL V2
# ND07 — NESTED CHRONOLOGICAL TUNING, CALIBRATION, AND BLEND SELECTION
#
# Run as one complete Jupyter cell.
#
# This step:
#   1. Reuses the exact five ND04 outer chronological folds.
#   2. Creates a trailing inner validation window inside every outer train set.
#   3. Tunes the three ND06-advanced model families using inner data only.
#   4. Selects calibration and ROLLING_MEAN_5 blend settings using inner data.
#   5. Refits the selected family/configuration on the complete outer train set.
#   6. Evaluates once on the untouched outer validation dates.
#   7. Selects the development method for ND08.
#
# It does not:
#   - open March 2026 targets;
#   - fit or save a final production model;
#   - perform a Monday-origin recursive weekly forecast;
#   - alter any previous checkpoint, model lock, dataset, or prediction.
# =============================================================================

import hashlib
import json
import math
import os
import platform
import shutil
import sys
import time
import uuid
import warnings
from collections import Counter
from datetime import datetime, timezone
from pathlib import Path
from zoneinfo import ZoneInfo

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import sklearn
from matplotlib.dates import DateFormatter
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OrdinalEncoder

try:
    import xgboost
    from xgboost import XGBRegressor
except Exception as error:
    raise RuntimeError(
        "ND07 requires xgboost. Install or restore the environment used in ND06."
    ) from error

try:
    import catboost
    from catboost import CatBoostRegressor
except Exception as error:
    raise RuntimeError(
        "ND07 requires catboost. Install or restore the environment used in ND06."
    ) from error

warnings.filterwarnings(
    "ignore",
    message="X does not have valid feature names",
)

os.environ.setdefault("OMP_NUM_THREADS", "1")
os.environ.setdefault("OPENBLAS_NUM_THREADS", "1")
os.environ.setdefault("MKL_NUM_THREADS", "1")
os.environ.setdefault("NUMEXPR_NUM_THREADS", "1")


# =============================================================================
# CONFIGURATION
# =============================================================================

PROJECT_ROOT = Path("/Users/ryansmac/Desktop/Meng Project")
EDEN_ROOT = PROJECT_ROOT / "eden_datasets"
MODEL_ROOT = EDEN_ROOT / "eden_normal_demand_model_v2"

ND03_ROOT = (
    MODEL_ROOT
    / "02_feature_engineering"
    / "ND03_normal_demand_features"
)

ND04_ROOT = (
    MODEL_ROOT
    / "03_models"
    / "00_candidates"
    / "ND04_baseline_and_fallback_evaluation"
)

ND05_ROOT = (
    MODEL_ROOT
    / "03_models"
    / "00_candidates"
    / "ND05_original_hurdle_model_rebuild"
)

ND06_ROOT = (
    MODEL_ROOT
    / "03_models"
    / "00_candidates"
    / "ND06_expanded_model_challenge"
)

MAIN_MODEL_PATH = (
    ND03_ROOT
    / "01_model_ready_datasets"
    / "ND03_pre_march_main_model_development_dataset.csv"
)

CORE_PREDICTOR_LIST_PATH = (
    ND03_ROOT
    / "03_contracts"
    / "ND03_core_predictor_list.csv"
)

FOLD_DEFINITION_PATH = (
    ND04_ROOT
    / "03_audits"
    / "ND04_chronological_fold_definition.csv"
)

ND04_SELECTED_METHODS_PATH = (
    ND04_ROOT
    / "02_metrics"
    / "ND04_selected_route_methods.csv"
)

ND04_ROUTED_PREDICTIONS_PATH = (
    ND04_ROOT
    / "01_predictions"
    / "ND04_selected_routed_system_predictions.csv"
)

ND05_MAIN_METRICS_PATH = (
    ND05_ROOT
    / "02_metrics"
    / "ND05_main_candidate_metrics.csv"
)

ND06_CANDIDATE_METRICS_PATH = (
    ND06_ROOT
    / "02_metrics"
    / "ND06_candidate_metrics.csv"
)

ND06_ADVANCEMENT_PATH = (
    ND06_ROOT
    / "02_metrics"
    / "ND06_advancement_decision.csv"
)

ND06_MANIFEST_PATH = (
    ND06_ROOT
    / "06_control"
    / "ND06_artifact_hash_manifest.csv"
)

ND03_CHECKPOINT_PATH = (
    MODEL_ROOT
    / "08_checkpoints"
    / "ND03_checkpoint.json"
)

ND04_CHECKPOINT_PATH = (
    MODEL_ROOT
    / "08_checkpoints"
    / "ND04_checkpoint.json"
)

ND05_CHECKPOINT_PATH = (
    MODEL_ROOT
    / "08_checkpoints"
    / "ND05_checkpoint.json"
)

ND06_CHECKPOINT_PATH = (
    MODEL_ROOT
    / "08_checkpoints"
    / "ND06_checkpoint.json"
)

EXPECTED_ND03_CHECKPOINT_SHA256 = (
    "0845af89a5b459ca13ae6ffd99dde444"
    "f5010f6c0fb5ba4c34a4f091ac2e151c"
)

EXPECTED_ND04_CHECKPOINT_SHA256 = (
    "2fdc5d2c64c38f85b2669ca942042884"
    "209d80111cc840261307da98b1e9cf54"
)

EXPECTED_ND05_CHECKPOINT_SHA256 = (
    "ce3342c8b960aa5c4791a114ab09ae1"
    "a86d1dab2a3eebb648aa060579a1378ff"
)

EXPECTED_ND06_CHECKPOINT_SHA256 = (
    "e3b7bb75a1b2968e426c9a4c1654e104"
    "20d683af4bd5cb2b75fa4b5f0f18f357"
)

DATE_COLUMN = "Date"
PRODUCT_ID_COLUMN = "CanonicalProductID"
PRODUCT_NAME_COLUMN = "CanonicalProductName"
TARGET_COLUMN = "NormalDemand"
ROUTE_COLUMN = "ForecastRoute"
FAMILY_COLUMN = "TierProductFamily"
DAY_OF_WEEK_COLUMN = "DayOfWeekNumber"
BENCHMARK_FEATURE = "PastNormalDemandRollingMean_5"

EXPECTED_MAIN_ROWS = 10_976
EXPECTED_OUTER_FOLDS = 5
EXPECTED_OUTER_VALIDATION_DATES = 100
EXPECTED_OUTER_VALIDATION_ROWS = 4_124
EXPECTED_CORE_PREDICTORS = 53
EXPECTED_NUMERIC_PREDICTORS = 45
EXPECTED_CATEGORICAL_PREDICTORS = 8

EXPECTED_BENCHMARK_METHOD = "ROLLING_MEAN_5"
EXPECTED_BENCHMARK_WAPE = 41.331256
EXPECTED_BENCHMARK_MAE = 4.760912
EXPECTED_BENCHMARK_RMSE = 8.663054
EXPECTED_BENCHMARK_TOTAL_BIAS = -154.400000
EXPECTED_BENCHMARK_ABSOLUTE_BIAS_PERCENTAGE = 0.325025

EXPECTED_ND06_BEST_METHOD = "CATBOOST_RMSE_CORE53"
EXPECTED_ND06_BEST_WAPE = 40.386564

INNER_VALIDATION_OPERATING_DATES = 20
MINIMUM_INNER_TRAIN_OPERATING_DATES = 60

BLEND_WEIGHTS = [0.25, 0.50, 0.75, 1.00]
CALIBRATION_TYPES = [
    "NONE",
    "MULTIPLICATIVE",
    "ADDITIVE",
]

MULTIPLICATIVE_FACTOR_MIN = 0.80
MULTIPLICATIVE_FACTOR_MAX = 1.20
ADDITIVE_OFFSET_MIN = -5.00
ADDITIVE_OFFSET_MAX = 5.00
INNER_MAXIMUM_ABSOLUTE_BIAS_PERCENTAGE = 5.00
INNER_MAXIMUM_MAE_RATIO_TO_BENCHMARK = 1.05

MINIMUM_WAPE_IMPROVEMENT_PP = 0.50
MAXIMUM_MAE_RATIO = 1.02
MAXIMUM_RMSE_RATIO = 1.10
MAXIMUM_ABSOLUTE_BIAS_PERCENTAGE = 3.00
MINIMUM_FOLDS_WON = 3

RANDOM_SEED = 42
MAX_THREADS = max(1, min(4, os.cpu_count() or 1))

ND07_ROOT = (
    MODEL_ROOT
    / "03_models"
    / "01_tuning"
    / "ND07_tuning_calibration_blending"
)

PREDICTION_DIR = ND07_ROOT / "01_predictions"
METRIC_DIR = ND07_ROOT / "02_metrics"
AUDIT_DIR = ND07_ROOT / "03_audits"
CONTRACT_DIR = ND07_ROOT / "04_contracts"
FIGURE_DIR = ND07_ROOT / "05_figures"
REPORT_DIR = ND07_ROOT / "06_reports"
CONTROL_DIR = ND07_ROOT / "07_control"

INNER_TRIAL_METRICS_PATH = (
    METRIC_DIR
    / "ND07_inner_tuning_trial_metrics.csv"
)

OUTER_SELECTIONS_PATH = (
    METRIC_DIR
    / "ND07_outer_fold_selected_configurations.csv"
)

OUTER_CANDIDATE_METRICS_PATH = (
    METRIC_DIR
    / "ND07_outer_candidate_metrics.csv"
)

OUTER_FOLD_METRICS_PATH = (
    METRIC_DIR
    / "ND07_outer_candidate_metrics_by_fold.csv"
)

FINAL_SELECTION_PATH = (
    METRIC_DIR
    / "ND07_final_development_selection.csv"
)

KEY_COMPARISON_PATH = (
    METRIC_DIR
    / "ND07_key_method_comparison.csv"
)

ROUTED_METRICS_PATH = (
    METRIC_DIR
    / "ND07_selected_routed_system_metrics.csv"
)

ROUTED_ROUTE_METRICS_PATH = (
    METRIC_DIR
    / "ND07_selected_routed_metrics_by_route.csv"
)

OUTER_PREDICTIONS_PATH = (
    PREDICTION_DIR
    / "ND07_outer_candidate_predictions.csv"
)

SELECTED_MAIN_PREDICTIONS_PATH = (
    PREDICTION_DIR
    / "ND07_selected_main_predictions.csv"
)

SELECTED_ROUTED_PREDICTIONS_PATH = (
    PREDICTION_DIR
    / "ND07_selected_routed_predictions.csv"
)

DAILY_RESTAURANT_TOTALS_PATH = (
    PREDICTION_DIR
    / "ND07_selected_daily_restaurant_totals.csv"
)

WEEKLY_PRODUCT_TOTALS_PATH = (
    PREDICTION_DIR
    / "ND07_selected_weekly_product_totals.csv"
)

WEEKLY_RESTAURANT_TOTALS_PATH = (
    PREDICTION_DIR
    / "ND07_selected_weekly_restaurant_totals.csv"
)

FINAL_METHOD_CONTRACT_PATH = (
    CONTRACT_DIR
    / "ND07_selected_development_method_contract.json"
)

FINAL_METHOD_CONTRACT_MD_PATH = (
    CONTRACT_DIR
    / "ND07_selected_development_method_contract.md"
)

INPUT_HASH_AUDIT_PATH = (
    AUDIT_DIR
    / "ND07_input_hash_audit.csv"
)

NESTED_SPLIT_AUDIT_PATH = (
    AUDIT_DIR
    / "ND07_nested_split_audit.csv"
)

PACKAGE_VERSIONS_PATH = (
    AUDIT_DIR
    / "ND07_package_versions.csv"
)

PROTOCOL_AUDIT_PATH = (
    AUDIT_DIR
    / "ND07_leakage_and_protocol_audit.csv"
)

VALIDATION_PATH = (
    AUDIT_DIR
    / "ND07_validation_summary.csv"
)

REPORT_SUMMARY_PATH = (
    REPORT_DIR
    / "ND07_tuning_calibration_blending_summary.md"
)

DECISION_JSON_PATH = (
    REPORT_DIR
    / "ND07_decision.json"
)

README_PATH = ND07_ROOT / "README.md"

MANIFEST_PATH = (
    CONTROL_DIR
    / "ND07_artifact_hash_manifest.csv"
)

CHECKPOINT_PATH = (
    CONTROL_DIR
    / "ND07_checkpoint.json"
)

CHECKPOINT_SHA_PATH = (
    CONTROL_DIR
    / "ND07_checkpoint.sha256"
)

TOP_LEVEL_CHECKPOINT_PATH = (
    MODEL_ROOT
    / "08_checkpoints"
    / "ND07_checkpoint.json"
)

TOP_LEVEL_CHECKPOINT_SHA_PATH = (
    MODEL_ROOT
    / "08_checkpoints"
    / "ND07_checkpoint.sha256"
)

MEMORY_ROOT = MODEL_ROOT / "00_project_memory"
ND07_HANDOFF_PATH = MEMORY_ROOT / "ND07_HANDOFF.md"
CURRENT_HANDOFF_PATH = MEMORY_ROOT / "CURRENT_HANDOFF.md"
WORKFLOW_PATH = MEMORY_ROOT / "WORKFLOW.md"
DECISIONS_PATH = MEMORY_ROOT / "DECISIONS.md"
METRICS_AND_RESULTS_PATH = MEMORY_ROOT / "METRICS_AND_RESULTS.md"
AGENTS_PATH = MODEL_ROOT / "AGENTS.md"
LOG_PATH = MODEL_ROOT / "09_logs" / "ND07_tuning_log.txt"

ALLOW_OVERWRITE = False
STEP_ID = "ND07"
STATUS = (
    "ND07_TUNING_CALIBRATION_AND_BLEND_SELECTION_"
    "COMPLETED_READY_FOR_ND08"
)

NOW_UTC = datetime.now(timezone.utc)
NOW_LOCAL = NOW_UTC.astimezone(ZoneInfo("Europe/Dublin"))


# =============================================================================
# MODEL GRIDS
# =============================================================================

FAMILY_DEFINITIONS = {
    "NESTED_TUNED_CATBOOST_RMSE_CORE53": {
        "Library": "catboost",
        "FeatureSet": "CORE53",
        "OriginalND06Method": "CATBOOST_RMSE_CORE53",
        "Grid": [
            {
                "ConfigID": "CB_D5_LR003_I500_L25",
                "depth": 5,
                "learning_rate": 0.03,
                "iterations": 500,
                "l2_leaf_reg": 5.0,
            },
            {
                "ConfigID": "CB_D6_LR003_I500_L25",
                "depth": 6,
                "learning_rate": 0.03,
                "iterations": 500,
                "l2_leaf_reg": 5.0,
            },
            {
                "ConfigID": "CB_D7_LR003_I500_L25",
                "depth": 7,
                "learning_rate": 0.03,
                "iterations": 500,
                "l2_leaf_reg": 5.0,
            },
            {
                "ConfigID": "CB_D7_LR004_I400_L25",
                "depth": 7,
                "learning_rate": 0.04,
                "iterations": 400,
                "l2_leaf_reg": 5.0,
            },
            {
                "ConfigID": "CB_D8_LR003_I500_L28",
                "depth": 8,
                "learning_rate": 0.03,
                "iterations": 500,
                "l2_leaf_reg": 8.0,
            },
        ],
    },
    "NESTED_TUNED_XGBOOST_SQUARED_CORE53": {
        "Library": "xgboost",
        "FeatureSet": "CORE53",
        "OriginalND06Method": "XGBOOST_SQUARED_CORE53",
        "Grid": [
            {
                "ConfigID": "XGB_D4_LR003_I450_MC3",
                "max_depth": 4,
                "learning_rate": 0.03,
                "n_estimators": 450,
                "min_child_weight": 3,
            },
            {
                "ConfigID": "XGB_D5_LR003_I450_MC3",
                "max_depth": 5,
                "learning_rate": 0.03,
                "n_estimators": 450,
                "min_child_weight": 3,
            },
            {
                "ConfigID": "XGB_D6_LR003_I450_MC3",
                "max_depth": 6,
                "learning_rate": 0.03,
                "n_estimators": 450,
                "min_child_weight": 3,
            },
            {
                "ConfigID": "XGB_D6_LR004_I350_MC5",
                "max_depth": 6,
                "learning_rate": 0.04,
                "n_estimators": 350,
                "min_child_weight": 5,
            },
        ],
    },
    "NESTED_TUNED_XGBOOST_SQUARED_PRODUCT_AWARE": {
        "Library": "xgboost",
        "FeatureSet": "CORE53_PLUS_PRODUCT_ID",
        "OriginalND06Method": "XGBOOST_SQUARED_PRODUCT_AWARE",
        "Grid": [
            {
                "ConfigID": "XGBP_D4_LR003_I450_MC3",
                "max_depth": 4,
                "learning_rate": 0.03,
                "n_estimators": 450,
                "min_child_weight": 3,
            },
            {
                "ConfigID": "XGBP_D5_LR003_I450_MC3",
                "max_depth": 5,
                "learning_rate": 0.03,
                "n_estimators": 450,
                "min_child_weight": 3,
            },
            {
                "ConfigID": "XGBP_D6_LR003_I450_MC3",
                "max_depth": 6,
                "learning_rate": 0.03,
                "n_estimators": 450,
                "min_child_weight": 3,
            },
            {
                "ConfigID": "XGBP_D6_LR004_I350_MC5",
                "max_depth": 6,
                "learning_rate": 0.04,
                "n_estimators": 350,
                "min_child_weight": 5,
            },
        ],
    },
}


# =============================================================================
# HELPERS
# =============================================================================

def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


def write_csv(path: Path, frame: pd.DataFrame) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    frame.to_csv(path, index=False)


def write_json(path: Path, payload: dict) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(
        json.dumps(payload, indent=2, ensure_ascii=False, default=str)
        + "\n",
        encoding="utf-8",
    )


def write_text(path: Path, text: str) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(text, encoding="utf-8")


def atomic_write_text(path: Path, text: str) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_name(f".{path.name}.{uuid.uuid4().hex}.tmp")
    temporary.write_text(text, encoding="utf-8")
    os.replace(temporary, path)


def append_marked_section(
    path: Path,
    marker: str,
    section_text: str,
) -> None:
    existing = path.read_text(encoding="utf-8") if path.is_file() else ""
    if marker in existing:
        return
    separator = "\n" if existing.endswith("\n") else "\n\n"
    atomic_write_text(path, existing + separator + section_text.strip() + "\n")


def safe_wape(actual, predicted) -> float:
    actual_array = np.asarray(actual, dtype=float)
    predicted_array = np.asarray(predicted, dtype=float)
    denominator = float(actual_array.sum())
    if denominator == 0:
        return float("nan")
    return float(
        100.0
        * np.abs(predicted_array - actual_array).sum()
        / denominator
    )


def metric_record(actual, predicted, evaluation_level: str) -> dict:
    actual_array = np.asarray(actual, dtype=float)
    predicted_array = np.asarray(predicted, dtype=float)
    errors = predicted_array - actual_array
    actual_total = float(actual_array.sum())
    total_bias = float(errors.sum())
    return {
        "EvaluationLevel": evaluation_level,
        "Observations": int(len(actual_array)),
        "ActualTotal": actual_total,
        "PredictedTotal": float(predicted_array.sum()),
        "WAPEPercentage": safe_wape(actual_array, predicted_array),
        "MAE": float(np.abs(errors).mean()),
        "RMSE": float(np.sqrt(np.square(errors).mean())),
        "MeanBias": float(errors.mean()),
        "TotalBias": total_bias,
        "AbsoluteBiasPercentage": (
            float(100.0 * abs(total_bias) / actual_total)
            if actual_total != 0
            else float("nan")
        ),
    }


def validate_required_columns(
    frame: pd.DataFrame,
    columns: set[str],
    name: str,
) -> None:
    missing = sorted(columns - set(frame.columns))
    if missing:
        raise AssertionError(
            f"{name} is missing required columns:\n"
            + "\n".join(f"- {column}" for column in missing)
        )


def make_preprocessor(
    numeric_predictors: list[str],
    categorical_predictors: list[str],
    include_product_id: bool,
) -> ColumnTransformer:
    categorical_columns = list(categorical_predictors)
    if include_product_id:
        categorical_columns.append(PRODUCT_ID_COLUMN)

    numeric_pipeline = Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="median")),
        ]
    )

    categorical_pipeline = Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="most_frequent")),
            (
                "ordinal",
                OrdinalEncoder(
                    handle_unknown="use_encoded_value",
                    unknown_value=-1,
                    encoded_missing_value=-2,
                ),
            ),
        ]
    )

    return ColumnTransformer(
        transformers=[
            ("numeric", numeric_pipeline, numeric_predictors),
            ("categorical", categorical_pipeline, categorical_columns),
        ],
        remainder="drop",
        sparse_threshold=0.0,
        verbose_feature_names_out=False,
    )


def prepare_source_frame(
    frame: pd.DataFrame,
    numeric_predictors: list[str],
    categorical_predictors: list[str],
    include_product_id: bool,
) -> pd.DataFrame:
    columns = (
        numeric_predictors
        + categorical_predictors
        + ([PRODUCT_ID_COLUMN] if include_product_id else [])
    )
    prepared = frame[columns].copy()

    for column in numeric_predictors:
        prepared[column] = pd.to_numeric(
            prepared[column],
            errors="coerce",
        )

    for column in categorical_predictors:
        prepared[column] = (
            prepared[column]
            .astype("string")
            .fillna("__MISSING__")
            .astype(str)
        )

    if include_product_id:
        prepared[PRODUCT_ID_COLUMN] = (
            prepared[PRODUCT_ID_COLUMN]
            .astype("string")
            .fillna("__MISSING_PRODUCT__")
            .astype(str)
        )

    return prepared


def instantiate_model(
    library: str,
    config: dict,
):
    parameters = {
        key: value
        for key, value in config.items()
        if key != "ConfigID"
    }

    if library == "catboost":
        return CatBoostRegressor(
            loss_function="RMSE",
            random_seed=RANDOM_SEED,
            verbose=False,
            allow_writing_files=False,
            thread_count=MAX_THREADS,
            **parameters,
        )

    if library == "xgboost":
        return XGBRegressor(
            objective="reg:squarederror",
            subsample=0.85,
            colsample_bytree=0.85,
            reg_lambda=2.0,
            reg_alpha=0.0,
            n_jobs=MAX_THREADS,
            random_state=RANDOM_SEED,
            verbosity=0,
            **parameters,
        )

    raise ValueError(f"Unsupported library: {library}")


def fit_predict(
    train_frame: pd.DataFrame,
    validation_frame: pd.DataFrame,
    family_definition: dict,
    config: dict,
    numeric_predictors: list[str],
    categorical_predictors: list[str],
) -> tuple[np.ndarray, float, int]:
    include_product_id = (
        family_definition["FeatureSet"]
        == "CORE53_PLUS_PRODUCT_ID"
    )

    preprocessor = make_preprocessor(
        numeric_predictors,
        categorical_predictors,
        include_product_id,
    )

    X_train_source = prepare_source_frame(
        train_frame,
        numeric_predictors,
        categorical_predictors,
        include_product_id,
    )

    X_validation_source = prepare_source_frame(
        validation_frame,
        numeric_predictors,
        categorical_predictors,
        include_product_id,
    )

    X_train = preprocessor.fit_transform(X_train_source)
    X_validation = preprocessor.transform(X_validation_source)

    model = instantiate_model(
        family_definition["Library"],
        config,
    )

    start = time.perf_counter()
    model.fit(
        X_train,
        train_frame[TARGET_COLUMN].to_numpy(dtype=float),
    )
    prediction = np.asarray(
        model.predict(X_validation),
        dtype=float,
    )
    seconds = float(time.perf_counter() - start)

    prediction = np.clip(prediction, 0.0, None)

    if not np.isfinite(prediction).all():
        raise AssertionError("A model produced non-finite predictions.")

    return prediction, seconds, int(X_train.shape[1])


def derive_calibration(
    actual: np.ndarray,
    base_prediction: np.ndarray,
    calibration_type: str,
) -> float:
    if calibration_type == "NONE":
        return 1.0

    if calibration_type == "MULTIPLICATIVE":
        denominator = float(base_prediction.sum())
        if denominator <= 0:
            factor = 1.0
        else:
            factor = float(actual.sum() / denominator)
        return float(
            np.clip(
                factor,
                MULTIPLICATIVE_FACTOR_MIN,
                MULTIPLICATIVE_FACTOR_MAX,
            )
        )

    if calibration_type == "ADDITIVE":
        offset = float(np.mean(actual - base_prediction))
        return float(
            np.clip(
                offset,
                ADDITIVE_OFFSET_MIN,
                ADDITIVE_OFFSET_MAX,
            )
        )

    raise ValueError(f"Unsupported calibration type: {calibration_type}")


def apply_calibration(
    base_prediction: np.ndarray,
    calibration_type: str,
    parameter: float,
) -> np.ndarray:
    prediction = np.asarray(base_prediction, dtype=float).copy()

    if calibration_type == "NONE":
        pass
    elif calibration_type == "MULTIPLICATIVE":
        prediction = prediction * float(parameter)
    elif calibration_type == "ADDITIVE":
        prediction = prediction + float(parameter)
    else:
        raise ValueError(f"Unsupported calibration type: {calibration_type}")

    return np.clip(prediction, 0.0, None)


def save_figure(path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    plt.tight_layout()
    plt.savefig(path, dpi=300, bbox_inches="tight")
    plt.close()


def mode_with_tie_break(
    values: list,
    tie_scores: dict,
):
    counts = Counter(values)
    maximum_count = max(counts.values())
    candidates = [
        value
        for value, count in counts.items()
        if count == maximum_count
    ]
    return min(
        candidates,
        key=lambda value: (
            tie_scores.get(value, float("inf")),
            str(value),
        ),
    )


# =============================================================================
# PREFLIGHT
# =============================================================================

required_inputs = [
    MAIN_MODEL_PATH,
    CORE_PREDICTOR_LIST_PATH,
    FOLD_DEFINITION_PATH,
    ND04_SELECTED_METHODS_PATH,
    ND04_ROUTED_PREDICTIONS_PATH,
    ND05_MAIN_METRICS_PATH,
    ND06_CANDIDATE_METRICS_PATH,
    ND06_ADVANCEMENT_PATH,
    ND06_MANIFEST_PATH,
    ND03_CHECKPOINT_PATH,
    ND04_CHECKPOINT_PATH,
    ND05_CHECKPOINT_PATH,
    ND06_CHECKPOINT_PATH,
    AGENTS_PATH,
    CURRENT_HANDOFF_PATH,
    WORKFLOW_PATH,
    DECISIONS_PATH,
    METRICS_AND_RESULTS_PATH,
]

missing_inputs = [path for path in required_inputs if not path.is_file()]
if missing_inputs:
    raise FileNotFoundError(
        "ND07 required inputs are missing:\n"
        + "\n".join(f"- {path}" for path in missing_inputs)
    )

checkpoint_hashes = {
    "ND03": sha256_file(ND03_CHECKPOINT_PATH),
    "ND04": sha256_file(ND04_CHECKPOINT_PATH),
    "ND05": sha256_file(ND05_CHECKPOINT_PATH),
    "ND06": sha256_file(ND06_CHECKPOINT_PATH),
}

expected_checkpoint_hashes = {
    "ND03": EXPECTED_ND03_CHECKPOINT_SHA256,
    "ND04": EXPECTED_ND04_CHECKPOINT_SHA256,
    "ND05": EXPECTED_ND05_CHECKPOINT_SHA256,
    "ND06": EXPECTED_ND06_CHECKPOINT_SHA256,
}

for step, expected_hash in expected_checkpoint_hashes.items():
    actual_hash = checkpoint_hashes[step]
    if actual_hash != expected_hash:
        raise AssertionError(
            f"{step} checkpoint hash mismatch.\n"
            f"Expected: {expected_hash}\n"
            f"Actual:   {actual_hash}"
        )

if ND07_ROOT.exists():
    if not ALLOW_OVERWRITE:
        raise FileExistsError(
            "ND07 output already exists. No files were changed:\n"
            f"{ND07_ROOT}"
        )
    shutil.rmtree(ND07_ROOT)

for path in [
    TOP_LEVEL_CHECKPOINT_PATH,
    TOP_LEVEL_CHECKPOINT_SHA_PATH,
]:
    if path.exists():
        if not ALLOW_OVERWRITE:
            raise FileExistsError(
                "An ND07 top-level checkpoint already exists. "
                f"No files were changed:\n{path}"
            )
        path.unlink()

STAGING_ROOT = (
    ND07_ROOT.parent
    / f".ND07_staging_{uuid.uuid4().hex}"
)
STAGING_ROOT.mkdir(parents=True, exist_ok=False)


# =============================================================================
# LOAD AND VERIFY
# =============================================================================

try:
    main_model = pd.read_csv(MAIN_MODEL_PATH, low_memory=False)
    predictor_register = pd.read_csv(
        CORE_PREDICTOR_LIST_PATH,
        low_memory=False,
    )
    fold_definition = pd.read_csv(
        FOLD_DEFINITION_PATH,
        low_memory=False,
    )
    selected_methods = pd.read_csv(
        ND04_SELECTED_METHODS_PATH,
        low_memory=False,
    )
    nd04_routed_predictions = pd.read_csv(
        ND04_ROUTED_PREDICTIONS_PATH,
        low_memory=False,
    )
    nd05_main_metrics = pd.read_csv(
        ND05_MAIN_METRICS_PATH,
        low_memory=False,
    )
    nd06_candidate_metrics = pd.read_csv(
        ND06_CANDIDATE_METRICS_PATH,
        low_memory=False,
    )
    nd06_advancement = pd.read_csv(
        ND06_ADVANCEMENT_PATH,
        low_memory=False,
    )
    nd06_manifest = pd.read_csv(
        ND06_MANIFEST_PATH,
        low_memory=False,
    )

    required_main_columns = {
        DATE_COLUMN,
        PRODUCT_ID_COLUMN,
        PRODUCT_NAME_COLUMN,
        TARGET_COLUMN,
        ROUTE_COLUMN,
        FAMILY_COLUMN,
        DAY_OF_WEEK_COLUMN,
        BENCHMARK_FEATURE,
        "EligibleForMethodSelection",
        "IsOpenedMarchDiagnosticPeriod",
    }

    validate_required_columns(
        main_model,
        required_main_columns,
        "ND03 main-model development dataset",
    )

    main_model[DATE_COLUMN] = pd.to_datetime(
        main_model[DATE_COLUMN],
        errors="raise",
    )

    main_model[TARGET_COLUMN] = pd.to_numeric(
        main_model[TARGET_COLUMN],
        errors="raise",
    ).astype(float)

    if len(main_model) != EXPECTED_MAIN_ROWS:
        raise AssertionError(
            f"Expected {EXPECTED_MAIN_ROWS:,} development rows; "
            f"found {len(main_model):,}."
        )

    if (
        set(main_model[ROUTE_COLUMN].astype(str).unique())
        != {"MAIN_MODEL"}
    ):
        raise AssertionError("Non-main routes found in the main-model dataset.")

    if main_model["IsOpenedMarchDiagnosticPeriod"].astype(bool).any():
        raise AssertionError("Opened March rows found in ND07 development data.")

    predictor_register = predictor_register.sort_values(
        "PredictorOrder"
    ).reset_index(drop=True)

    core_predictors = (
        predictor_register["Predictor"].astype(str).tolist()
    )
    numeric_predictors = (
        predictor_register.loc[
            predictor_register["PredictorType"].astype(str)
            == "NUMERIC",
            "Predictor",
        ]
        .astype(str)
        .tolist()
    )
    categorical_predictors = (
        predictor_register.loc[
            predictor_register["PredictorType"].astype(str)
            == "CATEGORICAL",
            "Predictor",
        ]
        .astype(str)
        .tolist()
    )

    if len(core_predictors) != EXPECTED_CORE_PREDICTORS:
        raise AssertionError("Unexpected core predictor count.")
    if len(numeric_predictors) != EXPECTED_NUMERIC_PREDICTORS:
        raise AssertionError("Unexpected numeric predictor count.")
    if len(categorical_predictors) != EXPECTED_CATEGORICAL_PREDICTORS:
        raise AssertionError("Unexpected categorical predictor count.")

    missing_predictors = sorted(
        set(core_predictors) - set(main_model.columns)
    )
    if missing_predictors:
        raise AssertionError(
            "Predictors missing from the development dataset:\n"
            + "\n".join(f"- {column}" for column in missing_predictors)
        )

    forbidden_predictors = {
        TARGET_COLUMN,
        "BulkDemand",
        "TotalDemand",
        "IsObservedProductDate",
        "IsZeroDemandRow",
        "DemandRecordSource",
    }
    forbidden_present = sorted(
        forbidden_predictors & set(core_predictors)
    )
    if forbidden_present:
        raise AssertionError(
            "Forbidden predictors found:\n"
            + "\n".join(f"- {column}" for column in forbidden_present)
        )

    for column in [
        "TrainStart",
        "TrainEnd",
        "ValidationStart",
        "ValidationEnd",
    ]:
        fold_definition[column] = pd.to_datetime(
            fold_definition[column],
            errors="raise",
        )

    if len(fold_definition) != EXPECTED_OUTER_FOLDS:
        raise AssertionError("Unexpected number of outer folds.")

    if not (
        fold_definition["TrainEnd"]
        < fold_definition["ValidationStart"]
    ).all():
        raise AssertionError("Outer folds violate chronology.")

    selected_main = selected_methods.loc[
        selected_methods["ForecastRoute"].astype(str)
        == "MAIN_MODEL"
    ]

    if len(selected_main) != 1:
        raise AssertionError("Expected one ND04 main benchmark row.")

    benchmark_method = str(
        selected_main.iloc[0]["SelectedMethod"]
    )
    benchmark_wape = float(
        selected_main.iloc[0]["WAPEPercentage"]
    )
    benchmark_mae = float(selected_main.iloc[0]["MAE"])
    benchmark_rmse = float(selected_main.iloc[0]["RMSE"])
    benchmark_total_bias = float(
        selected_main.iloc[0]["TotalBias"]
    )
    benchmark_absolute_bias_percentage = float(
        selected_main.iloc[0]["AbsoluteBiasPercentage"]
    )

    expected_values = [
        ("method", benchmark_method, EXPECTED_BENCHMARK_METHOD),
        ("WAPE", benchmark_wape, EXPECTED_BENCHMARK_WAPE),
        ("MAE", benchmark_mae, EXPECTED_BENCHMARK_MAE),
        ("RMSE", benchmark_rmse, EXPECTED_BENCHMARK_RMSE),
        (
            "total bias",
            benchmark_total_bias,
            EXPECTED_BENCHMARK_TOTAL_BIAS,
        ),
        (
            "absolute bias",
            benchmark_absolute_bias_percentage,
            EXPECTED_BENCHMARK_ABSOLUTE_BIAS_PERCENTAGE,
        ),
    ]

    for label, actual, expected in expected_values:
        if isinstance(expected, str):
            passed = actual == expected
        else:
            passed = math.isclose(
                float(actual),
                float(expected),
                rel_tol=0.0,
                abs_tol=1e-6,
            )
        if not passed:
            raise AssertionError(
                f"Benchmark {label} mismatch. "
                f"Expected {expected}; found {actual}."
            )

    nd06_best = nd06_candidate_metrics.loc[
        nd06_candidate_metrics["CandidateMethod"].astype(str)
        == EXPECTED_ND06_BEST_METHOD
    ]

    if len(nd06_best) != 1:
        raise AssertionError("Expected one ND06 CatBoost metric row.")

    nd06_best_wape = float(nd06_best.iloc[0]["WAPEPercentage"])
    if not math.isclose(
        nd06_best_wape,
        EXPECTED_ND06_BEST_WAPE,
        rel_tol=0.0,
        abs_tol=1e-6,
    ):
        raise AssertionError(
            "ND06 CatBoost WAPE differs from the accepted result."
        )

    expected_advanced = {
        "CATBOOST_RMSE_CORE53",
        "XGBOOST_SQUARED_CORE53",
        "XGBOOST_SQUARED_PRODUCT_AWARE",
    }

    actual_advanced = set(
        nd06_advancement.loc[
            nd06_advancement["AdvanceToND07"].astype(bool),
            "CandidateMethod",
        ].astype(str)
    )

    if actual_advanced != expected_advanced:
        raise AssertionError(
            "ND06 advancement set differs from the expected three candidates.\n"
            f"Expected: {sorted(expected_advanced)}\n"
            f"Actual:   {sorted(actual_advanced)}"
        )

    # Verify ND06 files against the ND06 manifest.
    nd06_manifest_lookup = {
        str(row["RelativePath"]): str(row["SHA256"])
        for _, row in nd06_manifest.iterrows()
    }

    input_hash_records = []
    for path in [
        ND06_CANDIDATE_METRICS_PATH,
        ND06_ADVANCEMENT_PATH,
    ]:
        relative_path = str(path.relative_to(ND06_ROOT))
        expected_hash = nd06_manifest_lookup.get(relative_path)
        actual_hash = sha256_file(path)
        if expected_hash is None:
            raise AssertionError(
                f"ND06 manifest does not contain {relative_path}."
            )
        if actual_hash != expected_hash:
            raise AssertionError(
                f"ND06 input hash mismatch for {relative_path}."
            )
        input_hash_records.append(
            {
                "SourceStep": "ND06",
                "InputPath": str(path),
                "RelativePath": relative_path,
                "SHA256": actual_hash,
                "MatchesManifest": True,
            }
        )

    input_hash_audit = pd.DataFrame(input_hash_records)

    protected_hashes_before = {
        str(path): sha256_file(path)
        for path in required_inputs
        if path.is_file()
    }

    # =========================================================================
    # NESTED CHRONOLOGICAL TUNING
    # =========================================================================

    outer_prediction_parts = []
    inner_trial_records = []
    outer_selection_records = []
    nested_split_records = []

    total_start = time.perf_counter()

    for outer_row in fold_definition.itertuples(index=False):
        outer_fold = int(outer_row.Fold)
        outer_validation_start = pd.Timestamp(
            outer_row.ValidationStart
        )
        outer_validation_end = pd.Timestamp(
            outer_row.ValidationEnd
        )

        outer_train = main_model.loc[
            main_model[DATE_COLUMN] < outer_validation_start
        ].copy()

        outer_validation = main_model.loc[
            main_model[DATE_COLUMN].between(
                outer_validation_start,
                outer_validation_end,
                inclusive="both",
            )
        ].copy()

        outer_train_dates = sorted(
            outer_train[DATE_COLUMN].drop_duplicates()
        )

        if (
            len(outer_train_dates)
            < (
                MINIMUM_INNER_TRAIN_OPERATING_DATES
                + INNER_VALIDATION_OPERATING_DATES
            )
        ):
            raise AssertionError(
                f"Outer fold {outer_fold} lacks enough dates "
                "for the nested inner split."
            )

        inner_validation_dates = outer_train_dates[
            -INNER_VALIDATION_OPERATING_DATES:
        ]

        inner_validation_start = pd.Timestamp(
            inner_validation_dates[0]
        )
        inner_validation_end = pd.Timestamp(
            inner_validation_dates[-1]
        )

        inner_train = outer_train.loc[
            outer_train[DATE_COLUMN] < inner_validation_start
        ].copy()

        inner_validation = outer_train.loc[
            outer_train[DATE_COLUMN].between(
                inner_validation_start,
                inner_validation_end,
                inclusive="both",
            )
        ].copy()

        if (
            inner_train[DATE_COLUMN].max()
            >= inner_validation[DATE_COLUMN].min()
        ):
            raise AssertionError(
                f"Outer fold {outer_fold}: inner split violates chronology."
            )

        if (
            inner_validation[DATE_COLUMN].max()
            >= outer_validation[DATE_COLUMN].min()
        ):
            raise AssertionError(
                f"Outer fold {outer_fold}: inner validation overlaps outer validation."
            )

        nested_split_records.append(
            {
                "OuterFold": outer_fold,
                "InnerTrainStart": inner_train[DATE_COLUMN].min(),
                "InnerTrainEnd": inner_train[DATE_COLUMN].max(),
                "InnerValidationStart": inner_validation[DATE_COLUMN].min(),
                "InnerValidationEnd": inner_validation[DATE_COLUMN].max(),
                "OuterTrainStart": outer_train[DATE_COLUMN].min(),
                "OuterTrainEnd": outer_train[DATE_COLUMN].max(),
                "OuterValidationStart": outer_validation[DATE_COLUMN].min(),
                "OuterValidationEnd": outer_validation[DATE_COLUMN].max(),
                "InnerTrainRows": int(len(inner_train)),
                "InnerValidationRows": int(len(inner_validation)),
                "OuterTrainRows": int(len(outer_train)),
                "OuterValidationRows": int(len(outer_validation)),
                "InnerTrainDates": int(
                    inner_train[DATE_COLUMN].nunique()
                ),
                "InnerValidationDates": int(
                    inner_validation[DATE_COLUMN].nunique()
                ),
                "OuterValidationDates": int(
                    outer_validation[DATE_COLUMN].nunique()
                ),
                "ChronologyPassed": True,
            }
        )

        inner_actual = inner_validation[
            TARGET_COLUMN
        ].to_numpy(dtype=float)

        inner_benchmark = np.clip(
            inner_validation[BENCHMARK_FEATURE].to_numpy(dtype=float),
            0.0,
            None,
        )

        inner_benchmark_metrics = metric_record(
            inner_actual,
            inner_benchmark,
            "INNER_VALIDATION_BENCHMARK",
        )

        outer_actual = outer_validation[
            TARGET_COLUMN
        ].to_numpy(dtype=float)

        outer_benchmark = np.clip(
            outer_validation[BENCHMARK_FEATURE].to_numpy(dtype=float),
            0.0,
            None,
        )

        benchmark_part = pd.DataFrame(
            {
                DATE_COLUMN: outer_validation[DATE_COLUMN].to_numpy(),
                PRODUCT_ID_COLUMN: (
                    outer_validation[PRODUCT_ID_COLUMN]
                    .astype(str)
                    .to_numpy()
                ),
                PRODUCT_NAME_COLUMN: (
                    outer_validation[PRODUCT_NAME_COLUMN]
                    .astype(str)
                    .to_numpy()
                ),
                FAMILY_COLUMN: (
                    outer_validation[FAMILY_COLUMN]
                    .astype(str)
                    .to_numpy()
                ),
                DAY_OF_WEEK_COLUMN: outer_validation[
                    DAY_OF_WEEK_COLUMN
                ].to_numpy(),
                "OuterFold": outer_fold,
                "CandidateMethod": "ROLLING_MEAN_5_BENCHMARK",
                "Library": "deterministic_baseline",
                "FeatureSet": "PAST_5_DAY_MEAN",
                "ConfigID": "NOT_APPLICABLE",
                "BlendWeightAdvancedModel": 0.0,
                "CalibrationType": "NONE",
                "CalibrationParameter": 1.0,
                "ActualNormalDemand": outer_actual,
                "RawAdvancedPrediction": np.nan,
                "BenchmarkPrediction": outer_benchmark,
                "PredictedNormalDemand": outer_benchmark,
                "FitSeconds": 0.0,
            }
        )

        outer_prediction_parts.append(benchmark_part)

        for family_method, family_definition in FAMILY_DEFINITIONS.items():
            family_trial_records = []

            for config in family_definition["Grid"]:
                inner_prediction, inner_fit_seconds, transformed_count = (
                    fit_predict(
                        inner_train,
                        inner_validation,
                        family_definition,
                        config,
                        numeric_predictors,
                        categorical_predictors,
                    )
                )

                for blend_weight in BLEND_WEIGHTS:
                    base_prediction = (
                        float(blend_weight) * inner_prediction
                        + (1.0 - float(blend_weight)) * inner_benchmark
                    )

                    for calibration_type in CALIBRATION_TYPES:
                        calibration_parameter = derive_calibration(
                            inner_actual,
                            base_prediction,
                            calibration_type,
                        )

                        final_inner_prediction = apply_calibration(
                            base_prediction,
                            calibration_type,
                            calibration_parameter,
                        )

                        trial_metrics = metric_record(
                            inner_actual,
                            final_inner_prediction,
                            "INNER_VALIDATION_TRIAL",
                        )

                        inner_qualified = bool(
                            (
                                trial_metrics["AbsoluteBiasPercentage"]
                                <= INNER_MAXIMUM_ABSOLUTE_BIAS_PERCENTAGE
                            )
                            and (
                                trial_metrics["MAE"]
                                <= (
                                    INNER_MAXIMUM_MAE_RATIO_TO_BENCHMARK
                                    * inner_benchmark_metrics["MAE"]
                                )
                            )
                        )

                        record = {
                            "OuterFold": outer_fold,
                            "CandidateMethod": family_method,
                            "OriginalND06Method": (
                                family_definition["OriginalND06Method"]
                            ),
                            "Library": family_definition["Library"],
                            "FeatureSet": family_definition["FeatureSet"],
                            "ConfigID": config["ConfigID"],
                            "ConfigJSON": json.dumps(
                                config,
                                sort_keys=True,
                            ),
                            "BlendWeightAdvancedModel": float(blend_weight),
                            "BlendWeightRollingMean5": float(
                                1.0 - blend_weight
                            ),
                            "CalibrationType": calibration_type,
                            "CalibrationParameter": float(
                                calibration_parameter
                            ),
                            "InnerTrainRows": int(len(inner_train)),
                            "InnerValidationRows": int(
                                len(inner_validation)
                            ),
                            "InnerValidationDates": int(
                                inner_validation[DATE_COLUMN].nunique()
                            ),
                            "TransformedFeatureCount": transformed_count,
                            "InnerFitSeconds": inner_fit_seconds,
                            "InnerBenchmarkWAPEPercentage": (
                                inner_benchmark_metrics["WAPEPercentage"]
                            ),
                            "InnerBenchmarkMAE": (
                                inner_benchmark_metrics["MAE"]
                            ),
                            "InnerQualified": inner_qualified,
                            **trial_metrics,
                        }

                        family_trial_records.append(record)
                        inner_trial_records.append(record)

            family_trials = pd.DataFrame(family_trial_records)

            qualified_trials = family_trials.loc[
                family_trials["InnerQualified"]
            ].copy()

            selection_pool = (
                qualified_trials
                if not qualified_trials.empty
                else family_trials
            )

            selected_inner = (
                selection_pool.sort_values(
                    [
                        "WAPEPercentage",
                        "MAE",
                        "RMSE",
                        "AbsoluteBiasPercentage",
                        "BlendWeightAdvancedModel",
                    ],
                    ascending=[True, True, True, True, False],
                )
                .iloc[0]
            )

            selected_config_id = str(selected_inner["ConfigID"])

            selected_config = next(
                config
                for config in family_definition["Grid"]
                if config["ConfigID"] == selected_config_id
            )

            outer_raw_prediction, outer_fit_seconds, transformed_count = (
                fit_predict(
                    outer_train,
                    outer_validation,
                    family_definition,
                    selected_config,
                    numeric_predictors,
                    categorical_predictors,
                )
            )

            selected_blend_weight = float(
                selected_inner["BlendWeightAdvancedModel"]
            )
            selected_calibration_type = str(
                selected_inner["CalibrationType"]
            )
            selected_calibration_parameter = float(
                selected_inner["CalibrationParameter"]
            )

            outer_base_prediction = (
                selected_blend_weight * outer_raw_prediction
                + (1.0 - selected_blend_weight) * outer_benchmark
            )

            outer_final_prediction = apply_calibration(
                outer_base_prediction,
                selected_calibration_type,
                selected_calibration_parameter,
            )

            outer_part = pd.DataFrame(
                {
                    DATE_COLUMN: outer_validation[DATE_COLUMN].to_numpy(),
                    PRODUCT_ID_COLUMN: (
                        outer_validation[PRODUCT_ID_COLUMN]
                        .astype(str)
                        .to_numpy()
                    ),
                    PRODUCT_NAME_COLUMN: (
                        outer_validation[PRODUCT_NAME_COLUMN]
                        .astype(str)
                        .to_numpy()
                    ),
                    FAMILY_COLUMN: (
                        outer_validation[FAMILY_COLUMN]
                        .astype(str)
                        .to_numpy()
                    ),
                    DAY_OF_WEEK_COLUMN: outer_validation[
                        DAY_OF_WEEK_COLUMN
                    ].to_numpy(),
                    "OuterFold": outer_fold,
                    "CandidateMethod": family_method,
                    "Library": family_definition["Library"],
                    "FeatureSet": family_definition["FeatureSet"],
                    "ConfigID": selected_config_id,
                    "BlendWeightAdvancedModel": (
                        selected_blend_weight
                    ),
                    "CalibrationType": selected_calibration_type,
                    "CalibrationParameter": (
                        selected_calibration_parameter
                    ),
                    "ActualNormalDemand": outer_actual,
                    "RawAdvancedPrediction": outer_raw_prediction,
                    "BenchmarkPrediction": outer_benchmark,
                    "PredictedNormalDemand": outer_final_prediction,
                    "FitSeconds": outer_fit_seconds,
                }
            )

            outer_prediction_parts.append(outer_part)

            outer_selection_records.append(
                {
                    "OuterFold": outer_fold,
                    "CandidateMethod": family_method,
                    "OriginalND06Method": (
                        family_definition["OriginalND06Method"]
                    ),
                    "Library": family_definition["Library"],
                    "FeatureSet": family_definition["FeatureSet"],
                    "SelectedConfigID": selected_config_id,
                    "SelectedConfigJSON": json.dumps(
                        selected_config,
                        sort_keys=True,
                    ),
                    "SelectedBlendWeightAdvancedModel": (
                        selected_blend_weight
                    ),
                    "SelectedBlendWeightRollingMean5": (
                        1.0 - selected_blend_weight
                    ),
                    "SelectedCalibrationType": (
                        selected_calibration_type
                    ),
                    "SelectedCalibrationParameter": (
                        selected_calibration_parameter
                    ),
                    "SelectedInnerWAPEPercentage": float(
                        selected_inner["WAPEPercentage"]
                    ),
                    "SelectedInnerMAE": float(
                        selected_inner["MAE"]
                    ),
                    "SelectedInnerRMSE": float(
                        selected_inner["RMSE"]
                    ),
                    "SelectedInnerAbsoluteBiasPercentage": float(
                        selected_inner["AbsoluteBiasPercentage"]
                    ),
                    "SelectedInnerQualified": bool(
                        selected_inner["InnerQualified"]
                    ),
                    "InnerBenchmarkWAPEPercentage": float(
                        inner_benchmark_metrics["WAPEPercentage"]
                    ),
                    "InnerBenchmarkMAE": float(
                        inner_benchmark_metrics["MAE"]
                    ),
                    "OuterTrainRows": int(len(outer_train)),
                    "OuterValidationRows": int(
                        len(outer_validation)
                    ),
                    "OuterFitSeconds": outer_fit_seconds,
                    "TransformedFeatureCount": transformed_count,
                }
            )

            print(
                f"Outer fold {outer_fold}/{EXPECTED_OUTER_FOLDS}: "
                f"{family_method} selected {selected_config_id}, "
                f"blend={selected_blend_weight:.2f}, "
                f"calibration={selected_calibration_type}, "
                f"inner WAPE={selected_inner['WAPEPercentage']:.3f}%"
            )

    total_fitting_seconds = float(time.perf_counter() - total_start)

    outer_predictions = pd.concat(
        outer_prediction_parts,
        ignore_index=True,
    )

    outer_predictions["ForecastError"] = (
        outer_predictions["PredictedNormalDemand"]
        - outer_predictions["ActualNormalDemand"]
    )
    outer_predictions["AbsoluteError"] = (
        outer_predictions["ForecastError"].abs()
    )

    inner_trial_metrics = pd.DataFrame(inner_trial_records)
    outer_selections = pd.DataFrame(outer_selection_records)
    nested_split_audit = pd.DataFrame(nested_split_records)

    # =========================================================================
    # OUTER METRICS AND DEVELOPMENT SELECTION
    # =========================================================================

    candidate_metric_records = []
    fold_metric_records = []

    for method, method_frame in outer_predictions.groupby(
        "CandidateMethod",
        sort=False,
    ):
        overall = metric_record(
            method_frame["ActualNormalDemand"],
            method_frame["PredictedNormalDemand"],
            "OUTER_VALIDATION_DAILY_PRODUCT_MAIN_ROUTE",
        )
        overall.update(
            {
                "CandidateMethod": method,
                "Library": method_frame["Library"].iloc[0],
                "FeatureSet": method_frame["FeatureSet"].iloc[0],
                "OuterFoldsCompleted": int(
                    method_frame["OuterFold"].nunique()
                ),
                "ValidationDates": int(
                    method_frame[DATE_COLUMN].nunique()
                ),
                "Products": int(
                    method_frame[PRODUCT_ID_COLUMN].nunique()
                ),
                "TotalOuterFitSeconds": float(
                    method_frame["FitSeconds"]
                    .groupby(method_frame["OuterFold"])
                    .first()
                    .sum()
                ),
            }
        )
        candidate_metric_records.append(overall)

        for fold, fold_frame in method_frame.groupby(
            "OuterFold",
            sort=True,
        ):
            fold_record = metric_record(
                fold_frame["ActualNormalDemand"],
                fold_frame["PredictedNormalDemand"],
                "OUTER_VALIDATION_DAILY_PRODUCT_MAIN_ROUTE_BY_FOLD",
            )
            fold_record.update(
                {
                    "CandidateMethod": method,
                    "OuterFold": int(fold),
                    "Library": fold_frame["Library"].iloc[0],
                    "FeatureSet": fold_frame["FeatureSet"].iloc[0],
                }
            )
            fold_metric_records.append(fold_record)

    outer_candidate_metrics = pd.DataFrame(candidate_metric_records)
    outer_fold_metrics = pd.DataFrame(fold_metric_records)

    benchmark_row = outer_candidate_metrics.loc[
        outer_candidate_metrics["CandidateMethod"]
        == "ROLLING_MEAN_5_BENCHMARK"
    ]

    if len(benchmark_row) != 1:
        raise AssertionError("Expected one reconstructed benchmark row.")

    reconstructed_benchmark_wape = float(
        benchmark_row.iloc[0]["WAPEPercentage"]
    )

    if not math.isclose(
        reconstructed_benchmark_wape,
        benchmark_wape,
        rel_tol=0.0,
        abs_tol=1e-6,
    ):
        raise AssertionError(
            "Nested outer benchmark does not reproduce ND04 WAPE."
        )

    benchmark_fold_wape = (
        outer_fold_metrics.loc[
            outer_fold_metrics["CandidateMethod"]
            == "ROLLING_MEAN_5_BENCHMARK",
            ["OuterFold", "WAPEPercentage"],
        ]
        .rename(
            columns={
                "WAPEPercentage": "BenchmarkFoldWAPEPercentage"
            }
        )
    )

    challenger_metrics = outer_candidate_metrics.loc[
        outer_candidate_metrics["CandidateMethod"]
        != "ROLLING_MEAN_5_BENCHMARK"
    ].copy()

    fold_win_records = []
    for method in challenger_metrics["CandidateMethod"]:
        comparison = (
            outer_fold_metrics.loc[
                outer_fold_metrics["CandidateMethod"] == method,
                ["OuterFold", "WAPEPercentage"],
            ]
            .merge(
                benchmark_fold_wape,
                on="OuterFold",
                how="left",
                validate="one_to_one",
            )
        )

        fold_win_records.append(
            {
                "CandidateMethod": method,
                "FoldsWonAgainstBenchmark": int(
                    (
                        comparison["WAPEPercentage"]
                        < comparison["BenchmarkFoldWAPEPercentage"]
                    ).sum()
                ),
                "MeanFoldWAPEImprovementPercentagePoints": float(
                    (
                        comparison["BenchmarkFoldWAPEPercentage"]
                        - comparison["WAPEPercentage"]
                    ).mean()
                ),
            }
        )

    challenger_metrics = challenger_metrics.merge(
        pd.DataFrame(fold_win_records),
        on="CandidateMethod",
        how="left",
        validate="one_to_one",
    )

    challenger_metrics["WAPEImprovementPercentagePoints"] = (
        benchmark_wape - challenger_metrics["WAPEPercentage"]
    )
    challenger_metrics["RelativeWAPEImprovementPercentage"] = (
        100.0
        * challenger_metrics["WAPEImprovementPercentagePoints"]
        / benchmark_wape
    )
    challenger_metrics["MAERatioToBenchmark"] = (
        challenger_metrics["MAE"] / benchmark_mae
    )
    challenger_metrics["RMSERatioToBenchmark"] = (
        challenger_metrics["RMSE"] / benchmark_rmse
    )
    challenger_metrics["StrictlyQualifiesAsDevelopmentMethod"] = (
        (
            challenger_metrics["WAPEImprovementPercentagePoints"]
            >= MINIMUM_WAPE_IMPROVEMENT_PP
        )
        & (
            challenger_metrics["MAERatioToBenchmark"]
            <= MAXIMUM_MAE_RATIO
        )
        & (
            challenger_metrics["RMSERatioToBenchmark"]
            <= MAXIMUM_RMSE_RATIO
        )
        & (
            challenger_metrics["AbsoluteBiasPercentage"]
            <= MAXIMUM_ABSOLUTE_BIAS_PERCENTAGE
        )
        & (
            challenger_metrics["FoldsWonAgainstBenchmark"]
            >= MINIMUM_FOLDS_WON
        )
        & (
            challenger_metrics["OuterFoldsCompleted"]
            == EXPECTED_OUTER_FOLDS
        )
    )

    challenger_metrics = (
        challenger_metrics.sort_values(
            [
                "WAPEPercentage",
                "MAE",
                "AbsoluteBiasPercentage",
            ]
        )
        .reset_index(drop=True)
    )

    challenger_metrics.insert(
        0,
        "DevelopmentRank",
        np.arange(1, len(challenger_metrics) + 1),
    )

    qualified = challenger_metrics.loc[
        challenger_metrics[
            "StrictlyQualifiesAsDevelopmentMethod"
        ]
    ].copy()

    if qualified.empty:
        selected_method = "ROLLING_MEAN_5_BENCHMARK"
        selected_method_group = "BASELINE"
        selected_method_row = benchmark_row.iloc[0]
        development_recommendation = (
            "ROLLING_MEAN_5_SELECTED_FOR_ND08"
        )
    else:
        selected_method_row = (
            qualified.sort_values(
                [
                    "WAPEPercentage",
                    "MAE",
                    "AbsoluteBiasPercentage",
                ]
            )
            .iloc[0]
        )
        selected_method = str(
            selected_method_row["CandidateMethod"]
        )
        selected_method_group = "NESTED_TUNED_ADVANCED_MODEL"
        development_recommendation = (
            f"{selected_method}_SELECTED_FOR_ND08"
        )

    selected_main_predictions = (
        outer_predictions.loc[
            outer_predictions["CandidateMethod"]
            == selected_method
        ]
        .copy()
        .sort_values(
            [DATE_COLUMN, PRODUCT_ID_COLUMN]
        )
        .reset_index(drop=True)
    )

    if len(selected_main_predictions) != EXPECTED_OUTER_VALIDATION_ROWS:
        raise AssertionError(
            "Selected method does not contain the expected outer rows."
        )

    # Add comparison and selection flags.
    outer_candidate_metrics = outer_candidate_metrics.merge(
        challenger_metrics[
            [
                "CandidateMethod",
                "DevelopmentRank",
                "FoldsWonAgainstBenchmark",
                "MeanFoldWAPEImprovementPercentagePoints",
                "WAPEImprovementPercentagePoints",
                "RelativeWAPEImprovementPercentage",
                "MAERatioToBenchmark",
                "RMSERatioToBenchmark",
                "StrictlyQualifiesAsDevelopmentMethod",
            ]
        ],
        on="CandidateMethod",
        how="left",
    )

    outer_candidate_metrics["IsBenchmark"] = (
        outer_candidate_metrics["CandidateMethod"]
        == "ROLLING_MEAN_5_BENCHMARK"
    )
    outer_candidate_metrics["SelectedForND08"] = (
        outer_candidate_metrics["CandidateMethod"]
        == selected_method
    )

    final_selection = outer_candidate_metrics.loc[
        outer_candidate_metrics["SelectedForND08"]
    ].copy()

    final_selection["DevelopmentRecommendation"] = (
        development_recommendation
    )
    final_selection["SelectionUsesMarch2026"] = False
    final_selection["FinalUnbiasedEvaluationStillRequired"] = True
    final_selection["FinalProductionModelFitted"] = False
    final_selection["WeekStartRecursiveEvaluationCompleted"] = False

    # =========================================================================
    # ROBUST DEPLOYMENT SPECIFICATION
    # =========================================================================

    if selected_method == "ROLLING_MEAN_5_BENCHMARK":
        method_contract = {
            "StepID": STEP_ID,
            "SelectedMethod": selected_method,
            "MethodGroup": selected_method_group,
            "ForecastFormula": (
                "Mean of the previous five available operating-day "
                "NormalDemand observations for the product."
            ),
            "CorePredictorsUsed": [],
            "ProductIDAware": False,
            "BlendWeightAdvancedModel": 0.0,
            "BlendWeightRollingMean5": 1.0,
            "CalibrationType": "NONE",
            "CalibrationParameterEstimationForFinalFit": "NOT_APPLICABLE",
            "March2026UsedForSelection": False,
            "FinalUnbiasedEvaluationStillRequired": True,
        }
        representative_config_id = "NOT_APPLICABLE"
        representative_blend_weight = 0.0
        representative_calibration_type = "NONE"
    else:
        selected_outer_settings = outer_selections.loc[
            outer_selections["CandidateMethod"] == selected_method
        ].copy()

        config_mean_scores = (
            selected_outer_settings.groupby(
                "SelectedConfigID"
            )["SelectedInnerWAPEPercentage"]
            .mean()
            .to_dict()
        )
        representative_config_id = mode_with_tie_break(
            selected_outer_settings["SelectedConfigID"]
            .astype(str)
            .tolist(),
            config_mean_scores,
        )

        blend_mean_scores = (
            selected_outer_settings.groupby(
                "SelectedBlendWeightAdvancedModel"
            )["SelectedInnerWAPEPercentage"]
            .mean()
            .to_dict()
        )
        representative_blend_weight = float(
            mode_with_tie_break(
                selected_outer_settings[
                    "SelectedBlendWeightAdvancedModel"
                ].astype(float).tolist(),
                blend_mean_scores,
            )
        )

        calibration_mean_scores = (
            selected_outer_settings.groupby(
                "SelectedCalibrationType"
            )["SelectedInnerWAPEPercentage"]
            .mean()
            .to_dict()
        )
        representative_calibration_type = str(
            mode_with_tie_break(
                selected_outer_settings[
                    "SelectedCalibrationType"
                ].astype(str).tolist(),
                calibration_mean_scores,
            )
        )

        family_definition = FAMILY_DEFINITIONS[selected_method]
        representative_config = next(
            config
            for config in family_definition["Grid"]
            if config["ConfigID"] == representative_config_id
        )

        method_contract = {
            "StepID": STEP_ID,
            "SelectedMethod": selected_method,
            "MethodGroup": selected_method_group,
            "OriginalND06Method": (
                family_definition["OriginalND06Method"]
            ),
            "Library": family_definition["Library"],
            "FeatureSet": family_definition["FeatureSet"],
            "ProductIDAware": (
                family_definition["FeatureSet"]
                == "CORE53_PLUS_PRODUCT_ID"
            ),
            "RepresentativeConfigID": representative_config_id,
            "RepresentativeConfig": representative_config,
            "RepresentativeBlendWeightAdvancedModel": (
                representative_blend_weight
            ),
            "RepresentativeBlendWeightRollingMean5": (
                1.0 - representative_blend_weight
            ),
            "RepresentativeCalibrationType": (
                representative_calibration_type
            ),
            "CalibrationParameterEstimationForFinalFit": (
                "Estimate from prequential out-of-fold predictions "
                "inside the complete pre-March development period; "
                "do not copy the mean outer-fold parameter."
            ),
            "CorePredictorCount": len(core_predictors),
            "CorePredictors": core_predictors,
            "March2026UsedForSelection": False,
            "FinalUnbiasedEvaluationStillRequired": True,
            "FinalProductionModelFitted": False,
            "ND08Requirement": (
                "Evaluate genuine Monday-origin recursive daily-to-weekly "
                "forecasting using this development method."
            ),
        }

    contract_markdown = f"""# ND07 Selected Development Method Contract

## Selected method

`{selected_method}`

## Development recommendation

`{development_recommendation}`

## Representative settings

- Representative configuration: `{representative_config_id}`
- Advanced-model blend weight: {representative_blend_weight:.2f}
- Rolling-mean blend weight: {1.0 - representative_blend_weight:.2f}
- Calibration type: `{representative_calibration_type}`

The representative settings are based on the most frequently selected nested-fold settings, with mean inner-validation WAPE used to break ties.

A final calibration parameter has not been copied from the outer folds. It must be estimated from prequential out-of-fold predictions during final fitting.

## Restrictions

- March 2026 was not used for selection.
- No final production model was fitted.
- No final model artifact was saved.
- A new untouched future period remains necessary for unbiased final evaluation.
- ND08 must test genuine Monday-origin recursive weekly forecasting.
"""

    # =========================================================================
    # ROUTED SYSTEM
    # =========================================================================

    nd04_routed_predictions[DATE_COLUMN] = pd.to_datetime(
        nd04_routed_predictions[DATE_COLUMN],
        errors="raise",
    )

    routed_key = selected_main_predictions[
        [
            DATE_COLUMN,
            PRODUCT_ID_COLUMN,
            "OuterFold",
            "PredictedNormalDemand",
        ]
    ].rename(
        columns={
            "OuterFold": "Fold",
            "PredictedNormalDemand": "ND07MainPrediction",
        }
    )

    routed = nd04_routed_predictions.merge(
        routed_key,
        on=[DATE_COLUMN, PRODUCT_ID_COLUMN, "Fold"],
        how="left",
        validate="one_to_one",
    )

    main_mask = routed[ROUTE_COLUMN].astype(str) == "MAIN_MODEL"

    if routed.loc[
        main_mask,
        "ND07MainPrediction",
    ].isna().any():
        raise AssertionError(
            "Selected ND07 main predictions are missing for routed rows."
        )

    routed["ND04SelectedPrediction"] = routed[
        "PredictedNormalDemand"
    ]

    routed.loc[
        main_mask,
        "PredictedNormalDemand",
    ] = routed.loc[
        main_mask,
        "ND07MainPrediction",
    ]

    routed.loc[
        main_mask,
        "SelectedMethod",
    ] = selected_method

    routed = routed.drop(columns=["ND07MainPrediction"])

    routed["ActualNormalDemand"] = pd.to_numeric(
        routed["ActualNormalDemand"],
        errors="raise",
    )
    routed["PredictedNormalDemand"] = pd.to_numeric(
        routed["PredictedNormalDemand"],
        errors="raise",
    )
    routed["ForecastError"] = (
        routed["PredictedNormalDemand"]
        - routed["ActualNormalDemand"]
    )
    routed["AbsoluteError"] = routed["ForecastError"].abs()
    routed["ForecastMode"] = "DAILY_UPDATED_ONE_STEP_VALIDATION"
    routed["WeeklyInterpretation"] = (
        "DAILY_UPDATED_AGGREGATION_DIAGNOSTIC_"
        "NOT_WEEK_START_RECURSIVE"
    )

    daily_restaurant = (
        routed.groupby(DATE_COLUMN, as_index=False)
        .agg(
            ActualNormalDemand=("ActualNormalDemand", "sum"),
            PredictedNormalDemand=("PredictedNormalDemand", "sum"),
            ProductRows=(PRODUCT_ID_COLUMN, "size"),
            Products=(PRODUCT_ID_COLUMN, "nunique"),
        )
        .sort_values(DATE_COLUMN)
        .reset_index(drop=True)
    )
    daily_restaurant["ForecastError"] = (
        daily_restaurant["PredictedNormalDemand"]
        - daily_restaurant["ActualNormalDemand"]
    )

    routed["WeekStart"] = (
        routed[DATE_COLUMN]
        - pd.to_timedelta(
            routed[DATE_COLUMN].dt.dayofweek,
            unit="D",
        )
    )

    week_dates = (
        routed[[DATE_COLUMN, "WeekStart"]]
        .drop_duplicates()
        .assign(
            Weekday=lambda frame: frame[DATE_COLUMN].dt.dayofweek
        )
        .groupby("WeekStart", as_index=False)
        .agg(
            OperatingDates=(DATE_COLUMN, "nunique"),
            ObservedWeekdays=(
                "Weekday",
                lambda values: tuple(sorted(set(values))),
            ),
        )
    )
    week_dates["IsCompleteMondayToFridayWeek"] = (
        week_dates["ObservedWeekdays"].apply(
            lambda value: value == (0, 1, 2, 3, 4)
        )
    )

    weekly_product = (
        routed.groupby(
            [
                "WeekStart",
                PRODUCT_ID_COLUMN,
                PRODUCT_NAME_COLUMN,
            ],
            as_index=False,
        )
        .agg(
            ActualNormalDemand=("ActualNormalDemand", "sum"),
            PredictedNormalDemand=("PredictedNormalDemand", "sum"),
            ProductDateRows=(DATE_COLUMN, "size"),
        )
        .merge(
            week_dates,
            on="WeekStart",
            how="left",
            validate="many_to_one",
        )
    )
    weekly_product["ForecastError"] = (
        weekly_product["PredictedNormalDemand"]
        - weekly_product["ActualNormalDemand"]
    )

    weekly_restaurant = (
        weekly_product.groupby("WeekStart", as_index=False)
        .agg(
            ActualNormalDemand=("ActualNormalDemand", "sum"),
            PredictedNormalDemand=("PredictedNormalDemand", "sum"),
            Products=(PRODUCT_ID_COLUMN, "nunique"),
            OperatingDates=("OperatingDates", "first"),
            ObservedWeekdays=("ObservedWeekdays", "first"),
            IsCompleteMondayToFridayWeek=(
                "IsCompleteMondayToFridayWeek",
                "first",
            ),
        )
    )
    weekly_restaurant["ForecastError"] = (
        weekly_restaurant["PredictedNormalDemand"]
        - weekly_restaurant["ActualNormalDemand"]
    )

    routed_metric_records = [
        metric_record(
            routed["ActualNormalDemand"],
            routed["PredictedNormalDemand"],
            "DAILY_PRODUCT_ALL_ROUTES",
        ),
        metric_record(
            daily_restaurant["ActualNormalDemand"],
            daily_restaurant["PredictedNormalDemand"],
            "DAILY_RESTAURANT_TOTAL_ALL_ROUTES",
        ),
        metric_record(
            weekly_product["ActualNormalDemand"],
            weekly_product["PredictedNormalDemand"],
            "WEEKLY_PRODUCT_DAILY_UPDATED_AGGREGATION_ALL_WEEKS",
        ),
        metric_record(
            weekly_product.loc[
                weekly_product["IsCompleteMondayToFridayWeek"],
                "ActualNormalDemand",
            ],
            weekly_product.loc[
                weekly_product["IsCompleteMondayToFridayWeek"],
                "PredictedNormalDemand",
            ],
            (
                "WEEKLY_PRODUCT_DAILY_UPDATED_"
                "AGGREGATION_COMPLETE_WEEKS"
            ),
        ),
        metric_record(
            weekly_restaurant["ActualNormalDemand"],
            weekly_restaurant["PredictedNormalDemand"],
            "WEEKLY_RESTAURANT_DAILY_UPDATED_AGGREGATION_ALL_WEEKS",
        ),
        metric_record(
            weekly_restaurant.loc[
                weekly_restaurant["IsCompleteMondayToFridayWeek"],
                "ActualNormalDemand",
            ],
            weekly_restaurant.loc[
                weekly_restaurant["IsCompleteMondayToFridayWeek"],
                "PredictedNormalDemand",
            ],
            (
                "WEEKLY_RESTAURANT_DAILY_UPDATED_"
                "AGGREGATION_COMPLETE_WEEKS"
            ),
        ),
    ]

    routed_metrics = pd.DataFrame(routed_metric_records)
    routed_metrics["MainRouteMethod"] = selected_method
    routed_metrics["ForecastMode"] = (
        "DAILY_UPDATED_AGGREGATION_DIAGNOSTIC"
    )
    routed_metrics["WeekStartRecursiveForecast"] = False

    route_metric_records = []
    for route, route_frame in routed.groupby(ROUTE_COLUMN, sort=True):
        record = metric_record(
            route_frame["ActualNormalDemand"],
            route_frame["PredictedNormalDemand"],
            "DAILY_PRODUCT_BY_ROUTE",
        )
        record.update(
            {
                "ForecastRoute": route,
                "SelectedMethod": (
                    route_frame["SelectedMethod"].iloc[0]
                ),
                "Products": int(
                    route_frame[PRODUCT_ID_COLUMN].nunique()
                ),
                "Dates": int(route_frame[DATE_COLUMN].nunique()),
            }
        )
        route_metric_records.append(record)

    routed_route_metrics = pd.DataFrame(route_metric_records)

    # =========================================================================
    # KEY COMPARISON
    # =========================================================================

    h75_row = nd05_main_metrics.loc[
        nd05_main_metrics["CandidateMethod"].astype(str)
        == "BLEND_HURDLE_NAIVE5__H75"
    ]
    if len(h75_row) != 1:
        raise AssertionError("Expected one ND05 H75 row.")

    nd07_selected_metric = outer_candidate_metrics.loc[
        outer_candidate_metrics["CandidateMethod"] == selected_method
    ].iloc[0]

    key_comparison = pd.DataFrame(
        [
            {
                "Method": "ROLLING_MEAN_5_BENCHMARK",
                "Stage": "ND04",
                "WAPEPercentage": benchmark_wape,
                "MAE": benchmark_mae,
                "RMSE": benchmark_rmse,
                "TotalBias": benchmark_total_bias,
                "AbsoluteBiasPercentage": (
                    benchmark_absolute_bias_percentage
                ),
            },
            {
                "Method": "BLEND_HURDLE_NAIVE5__H75",
                "Stage": "ND05",
                "WAPEPercentage": float(
                    h75_row.iloc[0]["WAPEPercentage"]
                ),
                "MAE": float(h75_row.iloc[0]["MAE"]),
                "RMSE": float(h75_row.iloc[0]["RMSE"]),
                "TotalBias": float(h75_row.iloc[0]["TotalBias"]),
                "AbsoluteBiasPercentage": float(
                    h75_row.iloc[0]["AbsoluteBiasPercentage"]
                ),
            },
            {
                "Method": EXPECTED_ND06_BEST_METHOD,
                "Stage": "ND06",
                "WAPEPercentage": nd06_best_wape,
                "MAE": float(nd06_best.iloc[0]["MAE"]),
                "RMSE": float(nd06_best.iloc[0]["RMSE"]),
                "TotalBias": float(nd06_best.iloc[0]["TotalBias"]),
                "AbsoluteBiasPercentage": float(
                    nd06_best.iloc[0]["AbsoluteBiasPercentage"]
                ),
            },
            {
                "Method": selected_method,
                "Stage": "ND07",
                "WAPEPercentage": float(
                    nd07_selected_metric["WAPEPercentage"]
                ),
                "MAE": float(nd07_selected_metric["MAE"]),
                "RMSE": float(nd07_selected_metric["RMSE"]),
                "TotalBias": float(
                    nd07_selected_metric["TotalBias"]
                ),
                "AbsoluteBiasPercentage": float(
                    nd07_selected_metric[
                        "AbsoluteBiasPercentage"
                    ]
                ),
            },
        ]
    )
    key_comparison["WAPEImprovementVersusBenchmarkPP"] = (
        benchmark_wape - key_comparison["WAPEPercentage"]
    )

    # =========================================================================
    # AUDITS
    # =========================================================================

    package_versions = pd.DataFrame(
        [
            {
                "Package": "python",
                "Version": platform.python_version(),
            },
            {
                "Package": "pandas",
                "Version": pd.__version__,
            },
            {
                "Package": "numpy",
                "Version": np.__version__,
            },
            {
                "Package": "scikit-learn",
                "Version": sklearn.__version__,
            },
            {
                "Package": "xgboost",
                "Version": xgboost.__version__,
            },
            {
                "Package": "catboost",
                "Version": catboost.__version__,
            },
        ]
    )

    prediction_counts = (
        outer_predictions.groupby("CandidateMethod").size()
    )
    duplicate_prediction_keys = int(
        outer_predictions.duplicated(
            [
                DATE_COLUMN,
                PRODUCT_ID_COLUMN,
                "OuterFold",
                "CandidateMethod",
            ]
        ).sum()
    )
    nonfinite_predictions = int(
        (
            ~np.isfinite(
                outer_predictions["PredictedNormalDemand"]
                .to_numpy(dtype=float)
            )
        ).sum()
    )
    negative_predictions = int(
        (
            outer_predictions["PredictedNormalDemand"] < 0
        ).sum()
    )

    protocol_audit = pd.DataFrame(
        [
            {
                "Check": "Outer validation used for tuning",
                "Expected": False,
                "Actual": False,
                "Passed": True,
            },
            {
                "Check": "Inner validation precedes outer validation",
                "Expected": True,
                "Actual": bool(
                    (
                        nested_split_audit["InnerValidationEnd"]
                        < nested_split_audit["OuterValidationStart"]
                    ).all()
                ),
                "Passed": bool(
                    (
                        nested_split_audit["InnerValidationEnd"]
                        < nested_split_audit["OuterValidationStart"]
                    ).all()
                ),
            },
            {
                "Check": "Inner training precedes inner validation",
                "Expected": True,
                "Actual": bool(
                    (
                        nested_split_audit["InnerTrainEnd"]
                        < nested_split_audit["InnerValidationStart"]
                    ).all()
                ),
                "Passed": bool(
                    (
                        nested_split_audit["InnerTrainEnd"]
                        < nested_split_audit["InnerValidationStart"]
                    ).all()
                ),
            },
            {
                "Check": "March target vault opened",
                "Expected": False,
                "Actual": False,
                "Passed": True,
            },
            {
                "Check": "Final production model fitted",
                "Expected": False,
                "Actual": False,
                "Passed": True,
            },
            {
                "Check": "Final model artifact saved",
                "Expected": False,
                "Actual": False,
                "Passed": True,
            },
            {
                "Check": "Weekly results described as recursive",
                "Expected": False,
                "Actual": False,
                "Passed": True,
            },
            {
                "Check": "Target in predictor list",
                "Expected": False,
                "Actual": TARGET_COLUMN in core_predictors,
                "Passed": TARGET_COLUMN not in core_predictors,
            },
            {
                "Check": "BulkDemand in predictor list",
                "Expected": False,
                "Actual": "BulkDemand" in core_predictors,
                "Passed": "BulkDemand" not in core_predictors,
            },
            {
                "Check": "TotalDemand in predictor list",
                "Expected": False,
                "Actual": "TotalDemand" in core_predictors,
                "Passed": "TotalDemand" not in core_predictors,
            },
        ]
    )

    validation = pd.DataFrame(
        [
            {
                "Check": "ND06 checkpoint verified",
                "Expected": EXPECTED_ND06_CHECKPOINT_SHA256,
                "Actual": checkpoint_hashes["ND06"],
                "Passed": (
                    checkpoint_hashes["ND06"]
                    == EXPECTED_ND06_CHECKPOINT_SHA256
                ),
            },
            {
                "Check": "Outer folds",
                "Expected": EXPECTED_OUTER_FOLDS,
                "Actual": int(
                    selected_main_predictions[
                        "OuterFold"
                    ].nunique()
                ),
                "Passed": int(
                    selected_main_predictions[
                        "OuterFold"
                    ].nunique()
                )
                == EXPECTED_OUTER_FOLDS,
            },
            {
                "Check": "Outer validation rows per method",
                "Expected": EXPECTED_OUTER_VALIDATION_ROWS,
                "Actual": int(prediction_counts.min()),
                "Passed": bool(
                    (
                        prediction_counts
                        == EXPECTED_OUTER_VALIDATION_ROWS
                    ).all()
                ),
            },
            {
                "Check": "Outer validation dates",
                "Expected": EXPECTED_OUTER_VALIDATION_DATES,
                "Actual": int(
                    selected_main_predictions[
                        DATE_COLUMN
                    ].nunique()
                ),
                "Passed": int(
                    selected_main_predictions[
                        DATE_COLUMN
                    ].nunique()
                )
                == EXPECTED_OUTER_VALIDATION_DATES,
            },
            {
                "Check": "Duplicate outer prediction keys",
                "Expected": 0,
                "Actual": duplicate_prediction_keys,
                "Passed": duplicate_prediction_keys == 0,
            },
            {
                "Check": "Non-finite predictions",
                "Expected": 0,
                "Actual": nonfinite_predictions,
                "Passed": nonfinite_predictions == 0,
            },
            {
                "Check": "Negative predictions after clipping",
                "Expected": 0,
                "Actual": negative_predictions,
                "Passed": negative_predictions == 0,
            },
            {
                "Check": "Benchmark WAPE reproduced",
                "Expected": benchmark_wape,
                "Actual": reconstructed_benchmark_wape,
                "Passed": math.isclose(
                    reconstructed_benchmark_wape,
                    benchmark_wape,
                    rel_tol=0.0,
                    abs_tol=1e-6,
                ),
            },
            {
                "Check": "One selected development method",
                "Expected": 1,
                "Actual": int(
                    outer_candidate_metrics[
                        "SelectedForND08"
                    ].sum()
                ),
                "Passed": int(
                    outer_candidate_metrics[
                        "SelectedForND08"
                    ].sum()
                )
                == 1,
            },
            {
                "Check": "Routed rows missing prediction",
                "Expected": 0,
                "Actual": int(
                    routed["PredictedNormalDemand"].isna().sum()
                ),
                "Passed": int(
                    routed["PredictedNormalDemand"].isna().sum()
                )
                == 0,
            },
            {
                "Check": "Previous inputs modified",
                "Expected": False,
                "Actual": False,
                "Passed": True,
            },
            {
                "Check": "ND07 step lock created",
                "Expected": False,
                "Actual": False,
                "Passed": True,
            },
        ]
    )

    if not protocol_audit["Passed"].all():
        raise AssertionError(
            "ND07 protocol audit failed:\n"
            + protocol_audit.loc[
                ~protocol_audit["Passed"]
            ].to_string(index=False)
        )

    if not validation["Passed"].all():
        raise AssertionError(
            "ND07 validation failed:\n"
            + validation.loc[
                ~validation["Passed"]
            ].to_string(index=False)
        )

    # =========================================================================
    # STAGED OUTPUT DIRECTORIES
    # =========================================================================

    staged_prediction_dir = (
        STAGING_ROOT / PREDICTION_DIR.relative_to(ND07_ROOT)
    )
    staged_metric_dir = (
        STAGING_ROOT / METRIC_DIR.relative_to(ND07_ROOT)
    )
    staged_audit_dir = (
        STAGING_ROOT / AUDIT_DIR.relative_to(ND07_ROOT)
    )
    staged_contract_dir = (
        STAGING_ROOT / CONTRACT_DIR.relative_to(ND07_ROOT)
    )
    staged_figure_dir = (
        STAGING_ROOT / FIGURE_DIR.relative_to(ND07_ROOT)
    )
    staged_report_dir = (
        STAGING_ROOT / REPORT_DIR.relative_to(ND07_ROOT)
    )
    staged_control_dir = (
        STAGING_ROOT / CONTROL_DIR.relative_to(ND07_ROOT)
    )

    # =========================================================================
    # FIGURES
    # =========================================================================

    plot_metrics = outer_candidate_metrics.sort_values(
        "WAPEPercentage",
        ascending=True,
    )

    plt.figure(figsize=(11, 7))
    plt.barh(
        plot_metrics["CandidateMethod"],
        plot_metrics["WAPEPercentage"],
    )
    plt.axvline(
        benchmark_wape,
        linestyle="--",
        label=f"ROLLING_MEAN_5: {benchmark_wape:.2f}%",
    )
    plt.title("ND07 nested outer-validation WAPE")
    plt.xlabel("Daily product WAPE (%)")
    plt.ylabel("Method")
    plt.grid(axis="x", alpha=0.3)
    plt.legend()
    save_figure(
        staged_figure_dir
        / "ND07_figure_01_outer_candidate_wape.png"
    )

    plt.figure(figsize=(11, 6))
    for method, frame in outer_fold_metrics.groupby(
        "CandidateMethod"
    ):
        plt.plot(
            frame["OuterFold"],
            frame["WAPEPercentage"],
            marker="o",
            label=method,
        )
    plt.title("ND07 WAPE by outer fold")
    plt.xlabel("Outer fold")
    plt.ylabel("WAPE (%)")
    plt.xticks(range(1, EXPECTED_OUTER_FOLDS + 1))
    plt.grid(alpha=0.3)
    plt.legend(fontsize=8)
    save_figure(
        staged_figure_dir
        / "ND07_figure_02_outer_fold_wape.png"
    )

    plt.figure(figsize=(10, 6))
    plt.scatter(
        outer_candidate_metrics["WAPEPercentage"],
        outer_candidate_metrics["AbsoluteBiasPercentage"],
    )
    for _, row in outer_candidate_metrics.iterrows():
        plt.annotate(
            str(row["CandidateMethod"]),
            (
                row["WAPEPercentage"],
                row["AbsoluteBiasPercentage"],
            ),
            fontsize=7,
            xytext=(3, 3),
            textcoords="offset points",
        )
    plt.axvline(benchmark_wape, linestyle="--")
    plt.axhline(
        MAXIMUM_ABSOLUTE_BIAS_PERCENTAGE,
        linestyle="--",
    )
    plt.title("Outer WAPE versus absolute aggregate bias")
    plt.xlabel("WAPE (%)")
    plt.ylabel("Absolute bias (%)")
    plt.grid(alpha=0.3)
    save_figure(
        staged_figure_dir
        / "ND07_figure_03_wape_vs_bias.png"
    )

    maximum_scatter = float(
        max(
            selected_main_predictions[
                "ActualNormalDemand"
            ].max(),
            selected_main_predictions[
                "PredictedNormalDemand"
            ].max(),
        )
    )

    plt.figure(figsize=(8, 8))
    plt.scatter(
        selected_main_predictions["ActualNormalDemand"],
        selected_main_predictions["PredictedNormalDemand"],
        alpha=0.35,
    )
    plt.plot(
        [0, maximum_scatter],
        [0, maximum_scatter],
        linestyle="--",
    )
    plt.title(f"{selected_method}: actual versus predicted")
    plt.xlabel("Actual normal demand")
    plt.ylabel("Predicted normal demand")
    plt.grid(alpha=0.3)
    save_figure(
        staged_figure_dir
        / "ND07_figure_04_selected_actual_vs_predicted.png"
    )

    plt.figure(figsize=(14, 6))
    plt.plot(
        daily_restaurant[DATE_COLUMN],
        daily_restaurant["ActualNormalDemand"],
        label="Actual",
    )
    plt.plot(
        daily_restaurant[DATE_COLUMN],
        daily_restaurant["PredictedNormalDemand"],
        label="Predicted",
    )
    plt.title("ND07 selected routed system: daily restaurant total")
    plt.xlabel("Date")
    plt.ylabel("Normal-demand units")
    plt.gca().xaxis.set_major_formatter(
        DateFormatter("%Y-%m-%d")
    )
    plt.xticks(rotation=45, ha="right")
    plt.grid(alpha=0.3)
    plt.legend()
    save_figure(
        staged_figure_dir
        / "ND07_figure_05_daily_restaurant.png"
    )

    plt.figure(figsize=(14, 6))
    plt.plot(
        weekly_restaurant["WeekStart"],
        weekly_restaurant["ActualNormalDemand"],
        marker="o",
        label="Actual",
    )
    plt.plot(
        weekly_restaurant["WeekStart"],
        weekly_restaurant["PredictedNormalDemand"],
        marker="o",
        label="Predicted",
    )
    plt.title(
        "ND07 selected routed system: weekly aggregation diagnostic"
    )
    plt.xlabel("Week starting")
    plt.ylabel("Normal-demand units")
    plt.gca().xaxis.set_major_formatter(
        DateFormatter("%Y-%m-%d")
    )
    plt.xticks(rotation=45, ha="right")
    plt.grid(alpha=0.3)
    plt.legend()
    save_figure(
        staged_figure_dir
        / "ND07_figure_06_weekly_restaurant.png"
    )

    plt.figure(figsize=(10, 6))
    plt.bar(
        key_comparison["Method"],
        key_comparison["WAPEPercentage"],
    )
    plt.title("Benchmark, hurdle, ND06 CatBoost, and ND07 selection")
    plt.xlabel("Method")
    plt.ylabel("WAPE (%)")
    plt.xticks(rotation=30, ha="right")
    plt.grid(axis="y", alpha=0.3)
    save_figure(
        staged_figure_dir
        / "ND07_figure_07_key_method_comparison.png"
    )

    plt.figure(figsize=(10, 6))
    plt.bar(
        routed_metrics["EvaluationLevel"],
        routed_metrics["WAPEPercentage"],
    )
    plt.title("ND07 selected routed WAPE by evaluation level")
    plt.xlabel("Evaluation level")
    plt.ylabel("WAPE (%)")
    plt.xticks(rotation=40, ha="right")
    plt.grid(axis="y", alpha=0.3)
    save_figure(
        staged_figure_dir
        / "ND07_figure_08_routed_wape.png"
    )

    # =========================================================================
    # REPORTS
    # =========================================================================

    selected_wape = float(nd07_selected_metric["WAPEPercentage"])
    selected_mae = float(nd07_selected_metric["MAE"])
    selected_rmse = float(nd07_selected_metric["RMSE"])
    selected_bias = float(nd07_selected_metric["TotalBias"])
    selected_abs_bias = float(
        nd07_selected_metric["AbsoluteBiasPercentage"]
    )

    if selected_method == "ROLLING_MEAN_5_BENCHMARK":
        selected_folds_won = 0
        selected_improvement = 0.0
        selected_qualified = False
    else:
        selected_challenger_row = challenger_metrics.loc[
            challenger_metrics["CandidateMethod"]
            == selected_method
        ].iloc[0]
        selected_folds_won = int(
            selected_challenger_row[
                "FoldsWonAgainstBenchmark"
            ]
        )
        selected_improvement = float(
            selected_challenger_row[
                "WAPEImprovementPercentagePoints"
            ]
        )
        selected_qualified = bool(
            selected_challenger_row[
                "StrictlyQualifiesAsDevelopmentMethod"
            ]
        )

    report_text = f"""# ND07 Tuning, Calibration, and Blend Selection Summary

## Status

`{STATUS}`

## Nested chronological design

- Outer folds: {EXPECTED_OUTER_FOLDS}
- Outer validation dates: {EXPECTED_OUTER_VALIDATION_DATES}
- Inner validation dates within each outer train set: {INNER_VALIDATION_OPERATING_DATES}
- Outer validation used for tuning: no
- March 2026 target vault opened: no

For each outer fold, hyperparameters, blend weight, and calibration method were selected using only an earlier inner validation window. The selected setting was then refitted on the complete outer training period and evaluated once on the untouched outer validation dates.

## Accepted benchmark

- Method: `{benchmark_method}`
- WAPE: {benchmark_wape:.6f}%
- MAE: {benchmark_mae:.6f}
- RMSE: {benchmark_rmse:.6f}
- Absolute bias: {benchmark_absolute_bias_percentage:.6f}%

## Selected development method

- Method: `{selected_method}`
- WAPE: {selected_wape:.6f}%
- MAE: {selected_mae:.6f}
- RMSE: {selected_rmse:.6f}
- Total bias: {selected_bias:.6f}
- Absolute bias: {selected_abs_bias:.6f}%
- WAPE improvement versus benchmark: {selected_improvement:.6f} percentage points
- Folds won against benchmark: {selected_folds_won}/{EXPECTED_OUTER_FOLDS}
- Strict qualification: {selected_qualified}

## Representative deployment settings

- Configuration: `{representative_config_id}`
- Advanced-model blend weight: {representative_blend_weight:.2f}
- Rolling-mean blend weight: {1.0 - representative_blend_weight:.2f}
- Calibration type: `{representative_calibration_type}`

The final calibration parameter must be estimated from prequential out-of-fold predictions when the final fit is created. No outer-fold calibration parameter is copied directly.

## Development recommendation

`{development_recommendation}`

## Weekly interpretation

The weekly outputs created here aggregate daily-updated one-step predictions. They are not genuine Monday-origin recursive forecasts. ND08 must perform that evaluation.

## Safety

- March targets opened: no
- Final production model fitted: no
- Final model saved: no
- Previous inputs modified: no
- ND07 lock created: no
"""

    decision_payload = {
        "StepID": STEP_ID,
        "Status": STATUS,
        "CreatedUTC": NOW_UTC.isoformat(),
        "SelectedMethod": selected_method,
        "DevelopmentRecommendation": development_recommendation,
        "SelectedMetrics": {
            "WAPEPercentage": selected_wape,
            "MAE": selected_mae,
            "RMSE": selected_rmse,
            "TotalBias": selected_bias,
            "AbsoluteBiasPercentage": selected_abs_bias,
            "WAPEImprovementVersusBenchmarkPP": selected_improvement,
            "FoldsWonAgainstBenchmark": selected_folds_won,
            "StrictQualification": selected_qualified,
        },
        "RepresentativeSettings": {
            "ConfigID": representative_config_id,
            "BlendWeightAdvancedModel": representative_blend_weight,
            "BlendWeightRollingMean5": (
                1.0 - representative_blend_weight
            ),
            "CalibrationType": representative_calibration_type,
        },
        "March2026UsedForSelection": False,
        "FinalProductionModelFitted": False,
        "WeekStartRecursiveEvaluationCompleted": False,
        "FinalUnbiasedFutureEvaluationStillRequired": True,
        "NextStep": "ND08",
    }

    readme_text = f"""# ND07 Nested Tuning, Calibration, and Blend Selection

Status: `{STATUS}`

Selected development method: `{selected_method}`

ND07 used nested chronological validation. Each outer fold had an earlier inner validation window for hyperparameter, blend, and calibration selection.

March 2026 remained closed. No final production model was fitted or saved.

Weekly outputs are daily-updated aggregation diagnostics. ND08 must perform genuine Monday-origin recursive evaluation.
"""

    # =========================================================================
    # WRITE STAGED OUTPUTS
    # =========================================================================

    output_frames = {
        staged_prediction_dir
        / OUTER_PREDICTIONS_PATH.name:
            outer_predictions,
        staged_prediction_dir
        / SELECTED_MAIN_PREDICTIONS_PATH.name:
            selected_main_predictions,
        staged_prediction_dir
        / SELECTED_ROUTED_PREDICTIONS_PATH.name:
            routed,
        staged_prediction_dir
        / DAILY_RESTAURANT_TOTALS_PATH.name:
            daily_restaurant,
        staged_prediction_dir
        / WEEKLY_PRODUCT_TOTALS_PATH.name:
            weekly_product,
        staged_prediction_dir
        / WEEKLY_RESTAURANT_TOTALS_PATH.name:
            weekly_restaurant,
        staged_metric_dir
        / INNER_TRIAL_METRICS_PATH.name:
            inner_trial_metrics,
        staged_metric_dir
        / OUTER_SELECTIONS_PATH.name:
            outer_selections,
        staged_metric_dir
        / OUTER_CANDIDATE_METRICS_PATH.name:
            outer_candidate_metrics,
        staged_metric_dir
        / OUTER_FOLD_METRICS_PATH.name:
            outer_fold_metrics,
        staged_metric_dir
        / FINAL_SELECTION_PATH.name:
            final_selection,
        staged_metric_dir
        / KEY_COMPARISON_PATH.name:
            key_comparison,
        staged_metric_dir
        / ROUTED_METRICS_PATH.name:
            routed_metrics,
        staged_metric_dir
        / ROUTED_ROUTE_METRICS_PATH.name:
            routed_route_metrics,
        staged_audit_dir
        / INPUT_HASH_AUDIT_PATH.name:
            input_hash_audit,
        staged_audit_dir
        / NESTED_SPLIT_AUDIT_PATH.name:
            nested_split_audit,
        staged_audit_dir
        / PACKAGE_VERSIONS_PATH.name:
            package_versions,
        staged_audit_dir
        / PROTOCOL_AUDIT_PATH.name:
            protocol_audit,
        staged_audit_dir
        / VALIDATION_PATH.name:
            validation,
    }

    for path, frame in output_frames.items():
        write_csv(path, frame)

    write_json(
        staged_contract_dir
        / FINAL_METHOD_CONTRACT_PATH.name,
        method_contract,
    )
    write_text(
        staged_contract_dir
        / FINAL_METHOD_CONTRACT_MD_PATH.name,
        contract_markdown,
    )
    write_text(
        staged_report_dir
        / REPORT_SUMMARY_PATH.name,
        report_text,
    )
    write_json(
        staged_report_dir
        / DECISION_JSON_PATH.name,
        decision_payload,
    )
    write_text(
        STAGING_ROOT / README_PATH.name,
        readme_text,
    )

    # =========================================================================
    # INPUT IMMUTABILITY
    # =========================================================================

    protected_hashes_after = {
        str(path): sha256_file(path)
        for path in required_inputs
        if path.is_file()
    }

    changed_inputs = [
        path
        for path in protected_hashes_before
        if protected_hashes_before[path]
        != protected_hashes_after[path]
    ]

    if changed_inputs:
        raise AssertionError(
            "Protected inputs changed during ND07:\n"
            + "\n".join(f"- {path}" for path in changed_inputs)
        )

    # =========================================================================
    # MANIFEST AND CHECKPOINT
    # =========================================================================

    excluded_names = {
        MANIFEST_PATH.name,
        CHECKPOINT_PATH.name,
        CHECKPOINT_SHA_PATH.name,
    }

    files_for_manifest = sorted(
        path
        for path in STAGING_ROOT.rglob("*")
        if path.is_file() and path.name not in excluded_names
    )

    manifest = pd.DataFrame(
        [
            {
                "RelativePath": str(
                    path.relative_to(STAGING_ROOT)
                ),
                "Bytes": int(path.stat().st_size),
                "SHA256": sha256_file(path),
            }
            for path in files_for_manifest
        ]
    ).sort_values("RelativePath").reset_index(drop=True)

    staged_manifest_path = (
        staged_control_dir / MANIFEST_PATH.name
    )
    write_csv(staged_manifest_path, manifest)
    manifest_sha256 = sha256_file(staged_manifest_path)

    checkpoint_payload = {
        "StepID": STEP_ID,
        "Status": STATUS,
        "CreatedUTC": NOW_UTC.isoformat(),
        "CreatedLocal": NOW_LOCAL.isoformat(),
        "ND07Root": str(ND07_ROOT),
        "InputCheckpoints": checkpoint_hashes,
        "NestedValidation": {
            "OuterFolds": EXPECTED_OUTER_FOLDS,
            "OuterValidationDates": EXPECTED_OUTER_VALIDATION_DATES,
            "InnerValidationOperatingDates": (
                INNER_VALIDATION_OPERATING_DATES
            ),
            "OuterValidationUsedForTuning": False,
        },
        "SelectedMethod": selected_method,
        "DevelopmentRecommendation": development_recommendation,
        "SelectedMetrics": decision_payload["SelectedMetrics"],
        "RepresentativeSettings": (
            decision_payload["RepresentativeSettings"]
        ),
        "RoutedMetrics": routed_metrics.to_dict(orient="records"),
        "TotalFittingSeconds": total_fitting_seconds,
        "Manifest": {
            "Path": str(MANIFEST_PATH),
            "SHA256": manifest_sha256,
        },
        "Safety": {
            "MarchTargetVaultOpened": False,
            "FinalProductionModelFitted": False,
            "FinalModelArtifactSaved": False,
            "PreviousInputsModified": False,
            "ExistingLocksModified": False,
            "ND07StepLockCreated": False,
            "CheckpointAndHashesCreated": True,
        },
        "ReadyForND08": True,
        "NextStep": "ND08",
    }

    staged_checkpoint_path = (
        staged_control_dir / CHECKPOINT_PATH.name
    )
    write_json(staged_checkpoint_path, checkpoint_payload)
    checkpoint_sha256 = sha256_file(staged_checkpoint_path)

    staged_checkpoint_sha_path = (
        staged_control_dir / CHECKPOINT_SHA_PATH.name
    )
    write_text(
        staged_checkpoint_sha_path,
        f"{checkpoint_sha256}  {CHECKPOINT_PATH.name}\n",
    )

    required_outputs = [
        STAGING_ROOT / README_PATH.name,
        staged_prediction_dir
        / OUTER_PREDICTIONS_PATH.name,
        staged_prediction_dir
        / SELECTED_MAIN_PREDICTIONS_PATH.name,
        staged_prediction_dir
        / SELECTED_ROUTED_PREDICTIONS_PATH.name,
        staged_metric_dir
        / INNER_TRIAL_METRICS_PATH.name,
        staged_metric_dir
        / FINAL_SELECTION_PATH.name,
        staged_contract_dir
        / FINAL_METHOD_CONTRACT_PATH.name,
        staged_report_dir
        / REPORT_SUMMARY_PATH.name,
        staged_audit_dir
        / VALIDATION_PATH.name,
        staged_manifest_path,
        staged_checkpoint_path,
        staged_checkpoint_sha_path,
    ]

    missing_outputs = [
        path for path in required_outputs if not path.is_file()
    ]
    if missing_outputs:
        raise AssertionError(
            "Required ND07 outputs are missing:\n"
            + "\n".join(f"- {path}" for path in missing_outputs)
        )

    # =========================================================================
    # ATOMIC COMMIT
    # =========================================================================

    os.replace(STAGING_ROOT, ND07_ROOT)

    TOP_LEVEL_CHECKPOINT_PATH.parent.mkdir(
        parents=True,
        exist_ok=True,
    )
    shutil.copy2(
        CHECKPOINT_PATH,
        TOP_LEVEL_CHECKPOINT_PATH,
    )
    shutil.copy2(
        CHECKPOINT_SHA_PATH,
        TOP_LEVEL_CHECKPOINT_SHA_PATH,
    )

    # =========================================================================
    # PROJECT MEMORY
    # =========================================================================

    handoff_text = f"""# ND07 Handoff

## Status

- Completed step: `{STEP_ID}`
- Status: `{STATUS}`
- Completed local time: `{NOW_LOCAL.isoformat()}`
- Root: `{ND07_ROOT}`
- Checkpoint: `{TOP_LEVEL_CHECKPOINT_PATH}`
- Checkpoint SHA-256: `{checkpoint_sha256}`

## Selected development method

- Method: `{selected_method}`
- WAPE: {selected_wape:.6f}%
- MAE: {selected_mae:.6f}
- RMSE: {selected_rmse:.6f}
- Total bias: {selected_bias:.6f}
- Absolute bias: {selected_abs_bias:.6f}%
- WAPE improvement versus benchmark: {selected_improvement:.6f} percentage points
- Folds won: {selected_folds_won}/{EXPECTED_OUTER_FOLDS}
- Strict qualification: {selected_qualified}

## Representative settings

- Configuration: `{representative_config_id}`
- Advanced-model blend weight: {representative_blend_weight:.2f}
- Rolling-mean blend weight: {1.0 - representative_blend_weight:.2f}
- Calibration type: `{representative_calibration_type}`

## Methodology

- Five fixed outer chronological folds.
- Trailing 20-operating-date inner validation window inside every outer train set.
- Hyperparameters, calibration, and blend selected from inner data only.
- Outer validation untouched until the final fold prediction.
- March 2026 remained closed.

## Safety

- Final production model fitted: no
- Final model saved: no
- Existing inputs modified: no
- ND07 lock created: no
- New untouched future evaluation still required: yes

## Next step

ND08 will evaluate the selected method as a genuine Monday-origin recursive daily-to-weekly forecasting system.
"""

    workflow_section = f"""## ND07 — Nested tuning, calibration, and blend selection

Status: `{STATUS}`

- Used five outer chronological folds.
- Used a 20-operating-date inner validation window inside each outer training period.
- Selected method: `{selected_method}`.
- Development recommendation: `{development_recommendation}`.
- Representative configuration: `{representative_config_id}`.
- Representative advanced-model blend weight: {representative_blend_weight:.2f}.
- Representative calibration: `{representative_calibration_type}`.
- March 2026 remained closed.
- No final model was fitted or saved.
"""

    decisions_section = f"""## ND07 decisions

1. Hyperparameter, calibration, and blend choices must be made inside nested chronological validation.
2. Outer validation dates cannot influence tuning.
3. Selected development method: `{selected_method}`.
4. Representative configuration: `{representative_config_id}`.
5. Representative advanced-model blend weight: {representative_blend_weight:.2f}.
6. Representative calibration type: `{representative_calibration_type}`.
7. Final calibration parameters must be estimated from prequential out-of-fold predictions, not copied from outer folds.
8. March 2026 remains excluded from method selection.
9. ND08 must perform genuine Monday-origin recursive weekly evaluation.
10. A new untouched future period remains required for final unbiased evaluation.
"""

    metrics_section = f"""## ND07 nested tuning results

- Selected method: `{selected_method}`
- WAPE: {selected_wape:.6f}%
- MAE: {selected_mae:.6f}
- RMSE: {selected_rmse:.6f}
- Total bias: {selected_bias:.6f}
- Absolute bias: {selected_abs_bias:.6f}%
- WAPE improvement versus ROLLING_MEAN_5: {selected_improvement:.6f} percentage points
- Folds won: {selected_folds_won}/{EXPECTED_OUTER_FOLDS}
- Strict qualification: {selected_qualified}
- Total fitting time: {total_fitting_seconds:.3f} seconds
"""

    agents_section = f"""## ND07 authoritative status

Marker: ND07_AUTHORITATIVE_STATUS

- Status: `{STATUS}`
- Handoff: `{ND07_HANDOFF_PATH}`
- Selected development method: `{selected_method}`
- Recommendation: `{development_recommendation}`
- Representative configuration: `{representative_config_id}`
- Representative advanced blend weight: {representative_blend_weight:.2f}
- Representative calibration: `{representative_calibration_type}`
- March target vault opened: no
- Final production model fitted: no
- Next step: `ND08`
"""

    atomic_write_text(ND07_HANDOFF_PATH, handoff_text)
    atomic_write_text(CURRENT_HANDOFF_PATH, handoff_text)
    append_marked_section(
        WORKFLOW_PATH,
        "## ND07 — Nested tuning, calibration, and blend selection",
        workflow_section,
    )
    append_marked_section(
        DECISIONS_PATH,
        "## ND07 decisions",
        decisions_section,
    )
    append_marked_section(
        METRICS_AND_RESULTS_PATH,
        "## ND07 nested tuning results",
        metrics_section,
    )
    append_marked_section(
        AGENTS_PATH,
        "Marker: ND07_AUTHORITATIVE_STATUS",
        agents_section,
    )

    log_text = "\n".join(
        [
            f"Step: {STEP_ID}",
            f"Status: {STATUS}",
            f"Created local: {NOW_LOCAL.isoformat()}",
            f"Selected method: {selected_method}",
            f"Selected WAPE: {selected_wape}",
            f"Selected MAE: {selected_mae}",
            f"Selected RMSE: {selected_rmse}",
            f"Selected absolute bias percentage: {selected_abs_bias}",
            f"Representative config: {representative_config_id}",
            (
                "Representative advanced-model blend weight: "
                f"{representative_blend_weight}"
            ),
            (
                "Representative calibration type: "
                f"{representative_calibration_type}"
            ),
            f"Total fitting seconds: {total_fitting_seconds}",
            f"Checkpoint SHA256: {checkpoint_sha256}",
            "March target vault opened: False",
            "Final production model fitted: False",
            "Final model artifact saved: False",
            "ND07 step lock created: False",
            "",
        ]
    )
    atomic_write_text(LOG_PATH, log_text)

except Exception:
    if STAGING_ROOT.exists():
        shutil.rmtree(STAGING_ROOT)
    raise


# =============================================================================
# FINAL CONSOLE OUTPUT
# =============================================================================

print("=" * 112)
print("EDEN NORMAL-DEMAND MODEL V2 — ND07 COMPLETE")
print("=" * 112)
print(f"Status: {STATUS}")
print(f"Local time: {NOW_LOCAL.isoformat()}")
print(f"ND07 root: {ND07_ROOT}")

print("\nINPUT VERIFICATION")
print(f"ND03 checkpoint SHA-256: {checkpoint_hashes['ND03']}")
print(f"ND04 checkpoint SHA-256: {checkpoint_hashes['ND04']}")
print(f"ND05 checkpoint SHA-256: {checkpoint_hashes['ND05']}")
print(f"ND06 checkpoint SHA-256: {checkpoint_hashes['ND06']}")
print(f"Main development rows: {len(main_model):,}")
print(f"Core predictors: {len(core_predictors)}")
print(f"Numeric predictors: {len(numeric_predictors)}")
print(f"Categorical predictors: {len(categorical_predictors)}")
print("March target vault opened: False")
print("Previous inputs modified: False")

print("\nNESTED CHRONOLOGICAL DESIGN")
print(f"Outer folds: {EXPECTED_OUTER_FOLDS}")
print(
    "Outer validation product-date rows per method: "
    f"{EXPECTED_OUTER_VALIDATION_ROWS:,}"
)
print(
    "Outer validation operating dates: "
    f"{EXPECTED_OUTER_VALIDATION_DATES}"
)
print(
    "Inner validation operating dates per outer fold: "
    f"{INNER_VALIDATION_OPERATING_DATES}"
)
print("Outer validation used for tuning: False")
print(f"Total fitting seconds: {total_fitting_seconds:.3f}")

print("\nACCEPTED BENCHMARK")
print(f"Method: {benchmark_method}")
print(f"WAPE: {benchmark_wape:.6f}%")
print(f"MAE: {benchmark_mae:.6f}")
print(f"RMSE: {benchmark_rmse:.6f}")
print(f"Total bias: {benchmark_total_bias:.6f}")
print(
    "Absolute bias percentage: "
    f"{benchmark_absolute_bias_percentage:.6f}%"
)

print("\nNESTED OUTER CANDIDATE RESULTS")
display_columns = [
    "DevelopmentRank",
    "CandidateMethod",
    "FeatureSet",
    "WAPEPercentage",
    "MAE",
    "RMSE",
    "TotalBias",
    "AbsoluteBiasPercentage",
    "WAPEImprovementPercentagePoints",
    "FoldsWonAgainstBenchmark",
    "StrictlyQualifiesAsDevelopmentMethod",
]
print(
    challenger_metrics[display_columns].to_string(index=False)
)

print("\nSELECTED DEVELOPMENT METHOD")
print(f"Method: {selected_method}")
print(f"WAPE: {selected_wape:.6f}%")
print(f"MAE: {selected_mae:.6f}")
print(f"RMSE: {selected_rmse:.6f}")
print(f"Total bias: {selected_bias:.6f}")
print(f"Absolute bias percentage: {selected_abs_bias:.6f}%")
print(
    "WAPE improvement versus benchmark: "
    f"{selected_improvement:.6f} percentage points"
)
print(
    "Folds won against benchmark: "
    f"{selected_folds_won}/{EXPECTED_OUTER_FOLDS}"
)
print(f"Strict qualification: {selected_qualified}")
print(f"Recommendation: {development_recommendation}")

print("\nREPRESENTATIVE SETTINGS FOR ND08")
print(f"Configuration: {representative_config_id}")
print(
    "Advanced-model blend weight: "
    f"{representative_blend_weight:.2f}"
)
print(
    "ROLLING_MEAN_5 blend weight: "
    f"{1.0 - representative_blend_weight:.2f}"
)
print(f"Calibration type: {representative_calibration_type}")
print(
    "Final calibration parameter: estimate from prequential "
    "out-of-fold predictions; not copied from outer folds."
)

print("\nSELECTED ROUTED SYSTEM")
print(
    routed_metrics[
        [
            "EvaluationLevel",
            "Observations",
            "WAPEPercentage",
            "MAE",
            "RMSE",
            "TotalBias",
        ]
    ].to_string(index=False)
)
print(
    "Forecast mode: DAILY_UPDATED_AGGREGATION_DIAGNOSTIC"
)
print("Week-start recursive forecast: False")

print("\nOUTPUTS")
print(f"- Inner trial metrics: {INNER_TRIAL_METRICS_PATH}")
print(f"- Outer selections: {OUTER_SELECTIONS_PATH}")
print(f"- Outer candidate predictions: {OUTER_PREDICTIONS_PATH}")
print(f"- Outer candidate metrics: {OUTER_CANDIDATE_METRICS_PATH}")
print(f"- Final selection: {FINAL_SELECTION_PATH}")
print(f"- Selected main predictions: {SELECTED_MAIN_PREDICTIONS_PATH}")
print(f"- Selected routed predictions: {SELECTED_ROUTED_PREDICTIONS_PATH}")
print(f"- Routed metrics: {ROUTED_METRICS_PATH}")
print(f"- Method contract: {FINAL_METHOD_CONTRACT_PATH}")
print(f"- Figures: {FIGURE_DIR}")
print(f"- Report summary: {REPORT_SUMMARY_PATH}")
print(f"- Validation: {VALIDATION_PATH}")
print(f"- Manifest: {MANIFEST_PATH}")
print(f"- Checkpoint: {TOP_LEVEL_CHECKPOINT_PATH}")
print(f"- Checkpoint SHA-256: {checkpoint_sha256}")
print(f"- Agent handoff: {ND07_HANDOFF_PATH}")

print("\nSAFETY")
print("- Hyperparameters selected using inner validation only: True")
print("- Outer validation used for tuning: False")
print("- March target vault opened: False")
print("- Final production model fitted: False")
print("- Final model artifact saved: False")
print("- Previous inputs modified: False")
print("- Existing model locks modified: False")
print("- ND07 step lock created: False")
print("- ND07 checkpoint and hashes created: True")

print("\nNEXT STEP")
print(
    "ND08 — genuine Monday-origin recursive daily-to-weekly "
    "evaluation using the selected development method."
)
print("=" * 112)

Outer fold 1/5: NESTED_TUNED_CATBOOST_RMSE_CORE53 selected CB_D6_LR003_I500_L25, blend=0.25, calibration=ADDITIVE, inner WAPE=61.807%
Outer fold 1/5: NESTED_TUNED_XGBOOST_SQUARED_CORE53 selected XGB_D4_LR003_I450_MC3, blend=0.50, calibration=ADDITIVE, inner WAPE=61.277%
Outer fold 1/5: NESTED_TUNED_XGBOOST_SQUARED_PRODUCT_AWARE selected XGBP_D4_LR003_I450_MC3, blend=0.50, calibration=ADDITIVE, inner WAPE=60.961%
Outer fold 2/5: NESTED_TUNED_CATBOOST_RMSE_CORE53 selected CB_D7_LR004_I400_L25, blend=0.25, calibration=MULTIPLICATIVE, inner WAPE=43.281%
Outer fold 2/5: NESTED_TUNED_XGBOOST_SQUARED_CORE53 selected XGB_D6_LR003_I450_MC3, blend=0.25, calibration=NONE, inner WAPE=43.404%
Outer fold 2/5: NESTED_TUNED_XGBOOST_SQUARED_PRODUCT_AWARE selected XGBP_D4_LR003_I450_MC3, blend=0.25, calibration=MULTIPLICATIVE, inner WAPE=43.592%
Outer fold 3/5: NESTED_TUNED_CATBOOST_RMSE_CORE53 selected CB_D6_LR003_I500_L25, blend=0.75, calibration=MULTIPLICATIVE, inner WAPE=34.321%
Outer fold 3/5: NEST

In [9]:
from __future__ import annotations

# EDEN NORMAL-DEMAND MODEL V2
# ND07E — Focused nested ensemble challenge
# Run this file as one complete Jupyter cell.

import hashlib
import itertools
import json
import math
import os
import platform
import shutil
import time
import uuid
from collections import Counter
from datetime import datetime, timezone
from pathlib import Path
from zoneinfo import ZoneInfo

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import sklearn
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OrdinalEncoder

try:
    import xgboost
    from xgboost import XGBRegressor
except Exception as exc:
    raise RuntimeError("ND07E requires xgboost from the ND06/ND07 environment.") from exc

try:
    import catboost
    from catboost import CatBoostRegressor
except Exception as exc:
    raise RuntimeError("ND07E requires catboost from the ND06/ND07 environment.") from exc

os.environ.setdefault("OMP_NUM_THREADS", "1")
os.environ.setdefault("OPENBLAS_NUM_THREADS", "1")
os.environ.setdefault("MKL_NUM_THREADS", "1")
os.environ.setdefault("NUMEXPR_NUM_THREADS", "1")

# -----------------------------------------------------------------------------
# Paths and constants
# -----------------------------------------------------------------------------
PROJECT_ROOT = Path("/Users/ryansmac/Desktop/Meng Project")
MODEL_ROOT = PROJECT_ROOT / "eden_datasets" / "eden_normal_demand_model_v2"
ND03 = MODEL_ROOT / "02_feature_engineering" / "ND03_normal_demand_features"
ND04 = MODEL_ROOT / "03_models" / "00_candidates" / "ND04_baseline_and_fallback_evaluation"
ND05 = MODEL_ROOT / "03_models" / "00_candidates" / "ND05_original_hurdle_model_rebuild"
ND06 = MODEL_ROOT / "03_models" / "00_candidates" / "ND06_expanded_model_challenge"
ND07 = MODEL_ROOT / "03_models" / "01_tuning" / "ND07_tuning_calibration_blending"

DEV_PATH = ND03 / "01_model_ready_datasets" / "ND03_pre_march_main_model_development_dataset.csv"
PREDICTOR_PATH = ND03 / "03_contracts" / "ND03_core_predictor_list.csv"
FOLD_PATH = ND04 / "03_audits" / "ND04_chronological_fold_definition.csv"
METHOD_PATH = ND04 / "02_metrics" / "ND04_selected_route_methods.csv"
ROUTED_PATH = ND04 / "01_predictions" / "ND04_selected_routed_system_predictions.csv"
HURDLE_METRICS_PATH = ND05 / "02_metrics" / "ND05_main_candidate_metrics.csv"
ND06_METRICS_PATH = ND06 / "02_metrics" / "ND06_candidate_metrics.csv"
ND07_SELECTIONS_PATH = ND07 / "02_metrics" / "ND07_outer_fold_selected_configurations.csv"
ND07_SPLITS_PATH = ND07 / "03_audits" / "ND07_nested_split_audit.csv"
ND07_FINAL_PATH = ND07 / "02_metrics" / "ND07_final_development_selection.csv"
ND07_MANIFEST_PATH = ND07 / "07_control" / "ND07_artifact_hash_manifest.csv"

CHECKPOINTS = {
    "ND03": MODEL_ROOT / "08_checkpoints" / "ND03_checkpoint.json",
    "ND04": MODEL_ROOT / "08_checkpoints" / "ND04_checkpoint.json",
    "ND05": MODEL_ROOT / "08_checkpoints" / "ND05_checkpoint.json",
    "ND06": MODEL_ROOT / "08_checkpoints" / "ND06_checkpoint.json",
    "ND07": MODEL_ROOT / "08_checkpoints" / "ND07_checkpoint.json",
}
EXPECTED_CP = {
    "ND03": "0845af89a5b459ca13ae6ffd99dde444f5010f6c0fb5ba4c34a4f091ac2e151c",
    "ND04": "2fdc5d2c64c38f85b2669ca942042884209d80111cc840261307da98b1e9cf54",
    "ND05": "ce3342c8b960aa5c4791a114ab09ae1a86d1dab2a3eebb648aa060579a1378ff",
    "ND06": "e3b7bb75a1b2968e426c9a4c1654e10420d683af4bd5cb2b75fa4b5f0f18f357",
    "ND07": "39077dd8c197561e5384a6943c3a4e153153e75fa4d03019f45005eb145cf18e",
}

OUT = MODEL_ROOT / "03_models" / "01_tuning" / "ND07E_focused_ensemble_challenge"
PRED_DIR = OUT / "01_predictions"
METRIC_DIR = OUT / "02_metrics"
AUDIT_DIR = OUT / "03_audits"
CONTRACT_DIR = OUT / "04_contracts"
FIG_DIR = OUT / "05_figures"
REPORT_DIR = OUT / "06_reports"
CONTROL_DIR = OUT / "07_control"
TOP_CP = MODEL_ROOT / "08_checkpoints" / "ND07E_checkpoint.json"
TOP_CP_SHA = MODEL_ROOT / "08_checkpoints" / "ND07E_checkpoint.sha256"
MEMORY = MODEL_ROOT / "00_project_memory"
HANDOFF = MEMORY / "ND07E_HANDOFF.md"
CURRENT_HANDOFF = MEMORY / "CURRENT_HANDOFF.md"
WORKFLOW = MEMORY / "WORKFLOW.md"
DECISIONS = MEMORY / "DECISIONS.md"
RESULTS_MEMORY = MEMORY / "METRICS_AND_RESULTS.md"
AGENTS = MODEL_ROOT / "AGENTS.md"
LOG = MODEL_ROOT / "09_logs" / "ND07E_ensemble_log.txt"

DATE = "Date"
PID = "CanonicalProductID"
PNAME = "CanonicalProductName"
TARGET = "NormalDemand"
ROUTE = "ForecastRoute"
FAMILY = "TierProductFamily"
DOW = "DayOfWeekNumber"
BASELINE_COL = "PastNormalDemandRollingMean_5"

EXPECTED_ROWS = 10_976
EXPECTED_VALID_ROWS = 4_124
EXPECTED_VALID_DATES = 100
EXPECTED_FOLDS = 5
EXPECTED_CORE = 53
EXPECTED_NUMERIC = 45
EXPECTED_CATEGORICAL = 8
EXPECTED_ND07_METHOD = "ROLLING_MEAN_5_BENCHMARK"

BENCH_WAPE = 41.331256
BENCH_MAE = 4.760912
BENCH_RMSE = 8.663054
BENCH_BIAS = -154.4
BENCH_ABS_BIAS = 0.325025

COMPONENTS = [
    "ROLLING_MEAN_5",
    "CATBOOST_CORE53",
    "XGBOOST_CORE53",
    "XGBOOST_PRODUCT_AWARE",
]
FAMILY_TO_COMPONENT = {
    "NESTED_TUNED_CATBOOST_RMSE_CORE53": "CATBOOST_CORE53",
    "NESTED_TUNED_XGBOOST_SQUARED_CORE53": "XGBOOST_CORE53",
    "NESTED_TUNED_XGBOOST_SQUARED_PRODUCT_AWARE": "XGBOOST_PRODUCT_AWARE",
}
EXPECTED_FAMILIES = set(FAMILY_TO_COMPONENT)
WEIGHT_STEP = 0.05
WEIGHT_UNITS = 20
INNER_MAX_ABS_BIAS = 5.0
INNER_MAX_MAE_RATIO = 1.05
MIN_WAPE_GAIN = 0.50
MAX_MAE_RATIO = 1.02
MAX_RMSE_RATIO = 1.10
MAX_ABS_BIAS = 3.0
MIN_FOLDS_WON = 3
SEED = 42
THREADS = max(1, min(4, os.cpu_count() or 1))
ALLOW_OVERWRITE = False
STATUS = "ND07E_FOCUSED_ENSEMBLE_CHALLENGE_COMPLETED_READY_FOR_ND08"
NOW_UTC = datetime.now(timezone.utc)
NOW_LOCAL = NOW_UTC.astimezone(ZoneInfo("Europe/Dublin"))

# -----------------------------------------------------------------------------
# Helpers
# -----------------------------------------------------------------------------
def sha(path: Path) -> str:
    h = hashlib.sha256()
    with path.open("rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()


def wcsv(path: Path, df: pd.DataFrame) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(path, index=False)


def wjson(path: Path, payload: dict) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(payload, indent=2, ensure_ascii=False, default=str) + "\n", encoding="utf-8")


def wtext(path: Path, text: str) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(text, encoding="utf-8")


def atomic_text(path: Path, text: str) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_name(f".{path.name}.{uuid.uuid4().hex}.tmp")
    tmp.write_text(text, encoding="utf-8")
    os.replace(tmp, path)


def append_once(path: Path, marker: str, text: str) -> None:
    existing = path.read_text(encoding="utf-8") if path.is_file() else ""
    if marker in existing:
        return
    sep = "\n" if existing.endswith("\n") else "\n\n"
    atomic_text(path, existing + sep + text.strip() + "\n")


def safe_wape(actual, pred) -> float:
    a = np.asarray(actual, dtype=float)
    p = np.asarray(pred, dtype=float)
    d = float(a.sum())
    return float("nan") if d == 0 else float(100.0 * np.abs(p - a).sum() / d)


def metrics(actual, pred, level: str) -> dict:
    a = np.asarray(actual, dtype=float)
    p = np.asarray(pred, dtype=float)
    e = p - a
    total = float(a.sum())
    bias = float(e.sum())
    return {
        "EvaluationLevel": level,
        "Observations": int(len(a)),
        "ActualTotal": total,
        "PredictedTotal": float(p.sum()),
        "WAPEPercentage": safe_wape(a, p),
        "MAE": float(np.abs(e).mean()),
        "RMSE": float(np.sqrt(np.square(e).mean())),
        "MeanBias": float(e.mean()),
        "TotalBias": bias,
        "AbsoluteBiasPercentage": float(100.0 * abs(bias) / total) if total else float("nan"),
    }


def make_preprocessor(num_cols, cat_cols, include_pid: bool):
    cats = list(cat_cols) + ([PID] if include_pid else [])
    return ColumnTransformer(
        [
            ("num", Pipeline([("imp", SimpleImputer(strategy="median"))]), num_cols),
            (
                "cat",
                Pipeline(
                    [
                        ("imp", SimpleImputer(strategy="most_frequent")),
                        (
                            "ord",
                            OrdinalEncoder(
                                handle_unknown="use_encoded_value",
                                unknown_value=-1,
                                encoded_missing_value=-2,
                            ),
                        ),
                    ]
                ),
                cats,
            ),
        ],
        remainder="drop",
        sparse_threshold=0.0,
        verbose_feature_names_out=False,
    )


def prep(df, num_cols, cat_cols, include_pid: bool):
    cols = list(num_cols) + list(cat_cols) + ([PID] if include_pid else [])
    out = df[cols].copy()
    for c in num_cols:
        out[c] = pd.to_numeric(out[c], errors="coerce")
    for c in cat_cols:
        out[c] = out[c].astype("string").fillna("__MISSING__").astype(str)
    if include_pid:
        out[PID] = out[PID].astype("string").fillna("__MISSING_PRODUCT__").astype(str)
    return out


def build_model(family_method: str, config: dict):
    params = {k: v for k, v in config.items() if k != "ConfigID"}
    if family_method == "NESTED_TUNED_CATBOOST_RMSE_CORE53":
        return CatBoostRegressor(
            loss_function="RMSE",
            random_seed=SEED,
            verbose=False,
            allow_writing_files=False,
            thread_count=THREADS,
            **params,
        )
    return XGBRegressor(
        objective="reg:squarederror",
        subsample=0.85,
        colsample_bytree=0.85,
        reg_lambda=2.0,
        reg_alpha=0.0,
        n_jobs=THREADS,
        random_state=SEED,
        verbosity=0,
        **params,
    )


def fit_predict(train, valid, family_method, config, num_cols, cat_cols):
    include_pid = family_method.endswith("PRODUCT_AWARE")
    pre = make_preprocessor(num_cols, cat_cols, include_pid)
    xtr = pre.fit_transform(prep(train, num_cols, cat_cols, include_pid))
    xva = pre.transform(prep(valid, num_cols, cat_cols, include_pid))
    model = build_model(family_method, config)
    start = time.perf_counter()
    model.fit(xtr, train[TARGET].to_numpy(dtype=float))
    pred = np.clip(np.asarray(model.predict(xva), dtype=float), 0.0, None)
    sec = float(time.perf_counter() - start)
    if not np.isfinite(pred).all():
        raise AssertionError(f"{family_method} produced non-finite predictions")
    return pred, sec, int(xtr.shape[1])


def compositions(total: int, parts: int):
    if parts == 1:
        yield (total,)
    else:
        for first in range(total + 1):
            for rest in compositions(total - first, parts - 1):
                yield (first,) + rest


def weight_grid() -> pd.DataFrame:
    rows = []
    for units in compositions(WEIGHT_UNITS, len(COMPONENTS)):
        count = sum(x > 0 for x in units)
        if count not in {2, 3, 4}:
            continue
        weights = [x / WEIGHT_UNITS for x in units]
        row = {
            "ModelCount": count,
            "ActiveComponents": "|".join(c for c, w in zip(COMPONENTS, weights) if w > 0),
            "WeightTuple": "|".join(f"{w:.2f}" for w in weights),
        }
        row.update({f"Weight_{c}": float(w) for c, w in zip(COMPONENTS, weights)})
        rows.append(row)
    return pd.DataFrame(rows)


def choose(trials: pd.DataFrame) -> pd.Series:
    q = trials.loc[trials["InnerQualified"]]
    pool = q if not q.empty else trials
    return pool.sort_values(
        ["WAPEPercentage", "MAE", "RMSE", "AbsoluteBiasPercentage", "Complexity", "SettingKey"]
    ).iloc[0]


def mode_tiebreak(values, score_lookup):
    counts = Counter(values)
    max_count = max(counts.values())
    candidates = [v for v, c in counts.items() if c == max_count]
    return min(candidates, key=lambda v: (score_lookup.get(v, float("inf")), str(v)))


def savefig(path: Path):
    path.parent.mkdir(parents=True, exist_ok=True)
    plt.tight_layout()
    plt.savefig(path, dpi=300, bbox_inches="tight")
    plt.close()

# -----------------------------------------------------------------------------
# Preflight
# -----------------------------------------------------------------------------
required = [
    DEV_PATH, PREDICTOR_PATH, FOLD_PATH, METHOD_PATH, ROUTED_PATH,
    HURDLE_METRICS_PATH, ND06_METRICS_PATH, ND07_SELECTIONS_PATH,
    ND07_SPLITS_PATH, ND07_FINAL_PATH, ND07_MANIFEST_PATH,
    *CHECKPOINTS.values(), AGENTS, CURRENT_HANDOFF, WORKFLOW, DECISIONS,
    RESULTS_MEMORY,
]
missing = [p for p in required if not p.is_file()]
if missing:
    raise FileNotFoundError("Missing ND07E inputs:\n" + "\n".join(f"- {p}" for p in missing))

cp_hashes = {k: sha(v) for k, v in CHECKPOINTS.items()}
for step, expected in EXPECTED_CP.items():
    if cp_hashes[step] != expected:
        raise AssertionError(f"{step} checkpoint mismatch\nExpected: {expected}\nActual:   {cp_hashes[step]}")

if OUT.exists() and not ALLOW_OVERWRITE:
    raise FileExistsError(f"ND07E output already exists; nothing changed:\n{OUT}")
if TOP_CP.exists() and not ALLOW_OVERWRITE:
    raise FileExistsError(f"ND07E checkpoint already exists; nothing changed:\n{TOP_CP}")
if ALLOW_OVERWRITE:
    shutil.rmtree(OUT, ignore_errors=True)
    for p in [TOP_CP, TOP_CP_SHA]:
        if p.exists():
            p.unlink()

STAGE = OUT.parent / f".ND07E_staging_{uuid.uuid4().hex}"
STAGE.mkdir(parents=True, exist_ok=False)

try:
    dev = pd.read_csv(DEV_PATH, low_memory=False)
    preg = pd.read_csv(PREDICTOR_PATH, low_memory=False).sort_values("PredictorOrder")
    folds = pd.read_csv(FOLD_PATH, low_memory=False)
    methods = pd.read_csv(METHOD_PATH, low_memory=False)
    routed_base = pd.read_csv(ROUTED_PATH, low_memory=False)
    hurdle_metrics = pd.read_csv(HURDLE_METRICS_PATH, low_memory=False)
    nd06_metrics = pd.read_csv(ND06_METRICS_PATH, low_memory=False)
    selections = pd.read_csv(ND07_SELECTIONS_PATH, low_memory=False)
    splits = pd.read_csv(ND07_SPLITS_PATH, low_memory=False)
    nd07_final = pd.read_csv(ND07_FINAL_PATH, low_memory=False)
    nd07_manifest = pd.read_csv(ND07_MANIFEST_PATH, low_memory=False)

    dev[DATE] = pd.to_datetime(dev[DATE], errors="raise")
    dev[TARGET] = pd.to_numeric(dev[TARGET], errors="raise").astype(float)
    routed_base[DATE] = pd.to_datetime(routed_base[DATE], errors="raise")
    for c in ["TrainStart", "TrainEnd", "ValidationStart", "ValidationEnd"]:
        folds[c] = pd.to_datetime(folds[c], errors="raise")
    for c in [
        "InnerTrainStart", "InnerTrainEnd", "InnerValidationStart", "InnerValidationEnd",
        "OuterTrainStart", "OuterTrainEnd", "OuterValidationStart", "OuterValidationEnd",
    ]:
        splits[c] = pd.to_datetime(splits[c], errors="raise")

    if len(dev) != EXPECTED_ROWS or set(dev[ROUTE].astype(str).unique()) != {"MAIN_MODEL"}:
        raise AssertionError("Unexpected ND03 main-model development dataset")
    if dev["IsOpenedMarchDiagnosticPeriod"].astype(bool).any():
        raise AssertionError("March rows found in ND07E development data")

    core = preg["Predictor"].astype(str).tolist()
    num_cols = preg.loc[preg["PredictorType"].astype(str) == "NUMERIC", "Predictor"].astype(str).tolist()
    cat_cols = preg.loc[preg["PredictorType"].astype(str) == "CATEGORICAL", "Predictor"].astype(str).tolist()
    if (len(core), len(num_cols), len(cat_cols)) != (EXPECTED_CORE, EXPECTED_NUMERIC, EXPECTED_CATEGORICAL):
        raise AssertionError("Unexpected ND03 predictor counts")
    forbidden = {TARGET, "BulkDemand", "TotalDemand", "IsObservedProductDate", "IsZeroDemandRow"}
    if forbidden & set(core):
        raise AssertionError("Forbidden fields found in predictor list")
    if set(core) - set(dev.columns):
        raise AssertionError("Core predictors missing from development data")

    main_method = methods.loc[methods["ForecastRoute"].astype(str) == "MAIN_MODEL"]
    if len(main_method) != 1 or str(main_method.iloc[0]["SelectedMethod"]) != "ROLLING_MEAN_5":
        raise AssertionError("ND04 benchmark method mismatch")
    for actual, expected in [
        (float(main_method.iloc[0]["WAPEPercentage"]), BENCH_WAPE),
        (float(main_method.iloc[0]["MAE"]), BENCH_MAE),
        (float(main_method.iloc[0]["RMSE"]), BENCH_RMSE),
        (float(main_method.iloc[0]["TotalBias"]), BENCH_BIAS),
        (float(main_method.iloc[0]["AbsoluteBiasPercentage"]), BENCH_ABS_BIAS),
    ]:
        if not math.isclose(actual, expected, rel_tol=0.0, abs_tol=1e-6):
            raise AssertionError("ND04 benchmark metric mismatch")

    if len(nd07_final) != 1 or str(nd07_final.iloc[0]["CandidateMethod"]) != EXPECTED_ND07_METHOD:
        raise AssertionError("ND07 accepted method mismatch")
    if set(selections["CandidateMethod"].astype(str).unique()) != EXPECTED_FAMILIES:
        raise AssertionError("ND07 selected-family set mismatch")
    if len(splits) != EXPECTED_FOLDS or len(selections) != EXPECTED_FOLDS * len(EXPECTED_FAMILIES):
        raise AssertionError("ND07 nested split/configuration count mismatch")

    # Hash-check the three ND07 decision inputs against the ND07 manifest.
    manifest_lookup = {str(r["RelativePath"]): str(r["SHA256"]) for _, r in nd07_manifest.iterrows()}
    input_hash_rows = []
    for p in [ND07_SELECTIONS_PATH, ND07_SPLITS_PATH, ND07_FINAL_PATH]:
        rel = str(p.relative_to(ND07))
        actual = sha(p)
        if manifest_lookup.get(rel) != actual:
            raise AssertionError(f"ND07 manifest mismatch for {rel}")
        input_hash_rows.append({"SourceStep": "ND07", "InputPath": str(p), "RelativePath": rel, "SHA256": actual, "MatchesManifest": True})
    input_hash_audit = pd.DataFrame(input_hash_rows)

    protected_before = {str(p): sha(p) for p in required if p.is_file()}
    grid = weight_grid()

    inner_base_parts, outer_base_parts, outer_candidate_parts = [], [], []
    trial_rows, selection_rows, config_rows, replay_rows = [], [], [], []
    start_all = time.perf_counter()

    for s in splits.sort_values("OuterFold").itertuples(index=False):
        fold = int(s.OuterFold)
        inner_train = dev.loc[dev[DATE].between(s.InnerTrainStart, s.InnerTrainEnd, inclusive="both")].copy()
        inner_val = dev.loc[dev[DATE].between(s.InnerValidationStart, s.InnerValidationEnd, inclusive="both")].copy()
        outer_train = dev.loc[dev[DATE].between(s.OuterTrainStart, s.OuterTrainEnd, inclusive="both")].copy()
        outer_val = dev.loc[dev[DATE].between(s.OuterValidationStart, s.OuterValidationEnd, inclusive="both")].copy()
        if min(len(inner_train), len(inner_val), len(outer_train), len(outer_val)) == 0:
            raise AssertionError(f"Empty nested split in fold {fold}")
        if not (inner_train[DATE].max() < inner_val[DATE].min() <= inner_val[DATE].max() < outer_val[DATE].min()):
            raise AssertionError(f"Nested chronology failed in fold {fold}")

        replay_rows.append({
            "OuterFold": fold,
            "InnerTrainStart": inner_train[DATE].min(), "InnerTrainEnd": inner_train[DATE].max(),
            "InnerValidationStart": inner_val[DATE].min(), "InnerValidationEnd": inner_val[DATE].max(),
            "OuterTrainStart": outer_train[DATE].min(), "OuterTrainEnd": outer_train[DATE].max(),
            "OuterValidationStart": outer_val[DATE].min(), "OuterValidationEnd": outer_val[DATE].max(),
            "InnerTrainRows": len(inner_train), "InnerValidationRows": len(inner_val),
            "OuterTrainRows": len(outer_train), "OuterValidationRows": len(outer_val),
            "ChronologyPassed": True,
        })

        ia = inner_val[TARGET].to_numpy(dtype=float)
        oa = outer_val[TARGET].to_numpy(dtype=float)
        inner_comp = {"ROLLING_MEAN_5": np.clip(inner_val[BASELINE_COL].to_numpy(dtype=float), 0.0, None)}
        outer_comp = {"ROLLING_MEAN_5": np.clip(outer_val[BASELINE_COL].to_numpy(dtype=float), 0.0, None)}

        fold_sel = selections.loc[selections["OuterFold"] == fold]
        if set(fold_sel["CandidateMethod"].astype(str)) != EXPECTED_FAMILIES:
            raise AssertionError(f"Missing ND07 base selection in fold {fold}")

        for r in fold_sel.itertuples(index=False):
            fam = str(r.CandidateMethod)
            comp = FAMILY_TO_COMPONENT[fam]
            cfg = json.loads(str(r.SelectedConfigJSON))
            ip, isec, ifeats = fit_predict(inner_train, inner_val, fam, cfg, num_cols, cat_cols)
            op, osec, ofeats = fit_predict(outer_train, outer_val, fam, cfg, num_cols, cat_cols)
            inner_comp[comp], outer_comp[comp] = ip, op
            config_rows.append({
                "OuterFold": fold, "CandidateMethod": fam, "Component": comp,
                "SelectedConfigID": str(r.SelectedConfigID), "SelectedConfigJSON": str(r.SelectedConfigJSON),
                "ConfigSelectedByND07InnerValidation": True,
                "InnerFitSeconds": isec, "OuterFitSeconds": osec,
                "InnerFeatureCount": ifeats, "OuterFeatureCount": ofeats,
            })

        imat = np.column_stack([inner_comp[c] for c in COMPONENTS])
        omat = np.column_stack([outer_comp[c] for c in COMPONENTS])
        inner_bench = metrics(ia, inner_comp["ROLLING_MEAN_5"], "INNER_BENCHMARK")

        for comp in COMPONENTS:
            inner_base_parts.append(pd.DataFrame({
                DATE: inner_val[DATE].to_numpy(), PID: inner_val[PID].astype(str).to_numpy(),
                "OuterFold": fold, "Component": comp, "ActualNormalDemand": ia,
                "PredictedNormalDemand": inner_comp[comp],
            }))
            outer_base_parts.append(pd.DataFrame({
                DATE: outer_val[DATE].to_numpy(), PID: outer_val[PID].astype(str).to_numpy(),
                "OuterFold": fold, "Component": comp, "ActualNormalDemand": oa,
                "PredictedNormalDemand": outer_comp[comp],
            }))

        # Benchmark outer candidate.
        outer_candidate_parts.append(pd.DataFrame({
            DATE: outer_val[DATE].to_numpy(), PID: outer_val[PID].astype(str).to_numpy(),
            PNAME: outer_val[PNAME].astype(str).to_numpy(), FAMILY: outer_val[FAMILY].astype(str).to_numpy(),
            DOW: outer_val[DOW].to_numpy(), "OuterFold": fold,
            "CandidateMethod": "ROLLING_MEAN_5_BENCHMARK", "EnsembleType": "BASELINE",
            "ModelCount": 1, "ActiveComponents": "ROLLING_MEAN_5", "SettingKey": "1.00|0.00|0.00|0.00",
            "Weight_ROLLING_MEAN_5": 1.0, "Weight_CATBOOST_CORE53": 0.0,
            "Weight_XGBOOST_CORE53": 0.0, "Weight_XGBOOST_PRODUCT_AWARE": 0.0,
            "ActualNormalDemand": oa, "PredictedNormalDemand": outer_comp["ROLLING_MEAN_5"],
        }))

        # Weighted ensembles by exact model count.
        for model_count in [2, 3, 4]:
            method = f"NESTED_WEIGHTED_ENSEMBLE_{model_count}_MODELS"
            local = []
            for g in grid.loc[grid["ModelCount"] == model_count].itertuples(index=False):
                weights = np.asarray([getattr(g, f"Weight_{c}") for c in COMPONENTS], dtype=float)
                pred = imat @ weights
                rec = metrics(ia, pred, "INNER_WEIGHT_TRIAL")
                rec.update({
                    "OuterFold": fold, "CandidateMethod": method, "EnsembleType": "WEIGHTED_MEAN",
                    "ModelCount": model_count, "ActiveComponents": g.ActiveComponents,
                    "SettingKey": g.WeightTuple, "Complexity": model_count,
                    **{f"Weight_{c}": float(w) for c, w in zip(COMPONENTS, weights)},
                    "InnerBenchmarkWAPEPercentage": inner_bench["WAPEPercentage"],
                    "InnerBenchmarkMAE": inner_bench["MAE"],
                    "InnerQualified": bool(
                        rec["AbsoluteBiasPercentage"] <= INNER_MAX_ABS_BIAS
                        and rec["MAE"] <= INNER_MAX_MAE_RATIO * inner_bench["MAE"]
                    ),
                })
                local.append(rec)
                trial_rows.append(rec)
            chosen = choose(pd.DataFrame(local))
            weights = np.asarray([chosen[f"Weight_{c}"] for c in COMPONENTS], dtype=float)
            opred = omat @ weights
            outer_candidate_parts.append(pd.DataFrame({
                DATE: outer_val[DATE].to_numpy(), PID: outer_val[PID].astype(str).to_numpy(),
                PNAME: outer_val[PNAME].astype(str).to_numpy(), FAMILY: outer_val[FAMILY].astype(str).to_numpy(),
                DOW: outer_val[DOW].to_numpy(), "OuterFold": fold,
                "CandidateMethod": method, "EnsembleType": "WEIGHTED_MEAN", "ModelCount": model_count,
                "ActiveComponents": chosen["ActiveComponents"], "SettingKey": chosen["SettingKey"],
                **{f"Weight_{c}": chosen[f"Weight_{c}"] for c in COMPONENTS},
                "ActualNormalDemand": oa, "PredictedNormalDemand": opred,
            }))
            selection_rows.append({
                "OuterFold": fold, "CandidateMethod": method, "EnsembleType": "WEIGHTED_MEAN",
                "ModelCount": model_count, "ActiveComponents": chosen["ActiveComponents"],
                "SettingKey": chosen["SettingKey"],
                **{f"Weight_{c}": chosen[f"Weight_{c}"] for c in COMPONENTS},
                "SelectedInnerWAPEPercentage": chosen["WAPEPercentage"], "SelectedInnerMAE": chosen["MAE"],
                "SelectedInnerRMSE": chosen["RMSE"], "SelectedInnerTotalBias": chosen["TotalBias"],
                "SelectedInnerAbsoluteBiasPercentage": chosen["AbsoluteBiasPercentage"],
                "SelectedInnerQualified": chosen["InnerQualified"],
            })

        # Median subsets of size 2, 3, and 4.
        local = []
        for subset_size in [2, 3, 4]:
            for subset in itertools.combinations(COMPONENTS, subset_size):
                idx = [COMPONENTS.index(c) for c in subset]
                pred = np.median(imat[:, idx], axis=1)
                rec = metrics(ia, pred, "INNER_MEDIAN_TRIAL")
                setting = "|".join(subset)
                rec.update({
                    "OuterFold": fold, "CandidateMethod": "NESTED_MEDIAN_ENSEMBLE",
                    "EnsembleType": "MEDIAN", "ModelCount": subset_size,
                    "ActiveComponents": setting, "SettingKey": setting, "Complexity": subset_size,
                    **{f"Weight_{c}": np.nan for c in COMPONENTS},
                    "InnerBenchmarkWAPEPercentage": inner_bench["WAPEPercentage"],
                    "InnerBenchmarkMAE": inner_bench["MAE"],
                    "InnerQualified": bool(
                        rec["AbsoluteBiasPercentage"] <= INNER_MAX_ABS_BIAS
                        and rec["MAE"] <= INNER_MAX_MAE_RATIO * inner_bench["MAE"]
                    ),
                })
                local.append(rec)
                trial_rows.append(rec)
        chosen = choose(pd.DataFrame(local))
        subset = str(chosen["SettingKey"]).split("|")
        idx = [COMPONENTS.index(c) for c in subset]
        opred = np.median(omat[:, idx], axis=1)
        outer_candidate_parts.append(pd.DataFrame({
            DATE: outer_val[DATE].to_numpy(), PID: outer_val[PID].astype(str).to_numpy(),
            PNAME: outer_val[PNAME].astype(str).to_numpy(), FAMILY: outer_val[FAMILY].astype(str).to_numpy(),
            DOW: outer_val[DOW].to_numpy(), "OuterFold": fold,
            "CandidateMethod": "NESTED_MEDIAN_ENSEMBLE", "EnsembleType": "MEDIAN",
            "ModelCount": int(chosen["ModelCount"]), "ActiveComponents": chosen["ActiveComponents"],
            "SettingKey": chosen["SettingKey"], **{f"Weight_{c}": np.nan for c in COMPONENTS},
            "ActualNormalDemand": oa, "PredictedNormalDemand": opred,
        }))
        selection_rows.append({
            "OuterFold": fold, "CandidateMethod": "NESTED_MEDIAN_ENSEMBLE", "EnsembleType": "MEDIAN",
            "ModelCount": int(chosen["ModelCount"]), "ActiveComponents": chosen["ActiveComponents"],
            "SettingKey": chosen["SettingKey"], **{f"Weight_{c}": np.nan for c in COMPONENTS},
            "SelectedInnerWAPEPercentage": chosen["WAPEPercentage"], "SelectedInnerMAE": chosen["MAE"],
            "SelectedInnerRMSE": chosen["RMSE"], "SelectedInnerTotalBias": chosen["TotalBias"],
            "SelectedInnerAbsoluteBiasPercentage": chosen["AbsoluteBiasPercentage"],
            "SelectedInnerQualified": chosen["InnerQualified"],
        })
        print(f"Outer fold {fold}/{EXPECTED_FOLDS}: ensemble settings selected using inner validation only")

    fitting_seconds = float(time.perf_counter() - start_all)
    inner_base = pd.concat(inner_base_parts, ignore_index=True)
    outer_base = pd.concat(outer_base_parts, ignore_index=True)
    outer_predictions = pd.concat(outer_candidate_parts, ignore_index=True)
    trials = pd.DataFrame(trial_rows)
    outer_selections = pd.DataFrame(selection_rows)
    config_audit = pd.DataFrame(config_rows)
    replay_audit = pd.DataFrame(replay_rows)
    outer_predictions["ForecastError"] = outer_predictions["PredictedNormalDemand"] - outer_predictions["ActualNormalDemand"]
    outer_predictions["AbsoluteError"] = outer_predictions["ForecastError"].abs()

    # Overall and fold metrics.
    overall_rows, fold_rows = [], []
    for method, frame in outer_predictions.groupby("CandidateMethod", sort=False):
        rec = metrics(frame["ActualNormalDemand"], frame["PredictedNormalDemand"], "OUTER_DAILY_PRODUCT_MAIN_ROUTE")
        rec.update({
            "CandidateMethod": method, "EnsembleType": frame["EnsembleType"].iloc[0],
            "OuterFoldsCompleted": frame["OuterFold"].nunique(),
            "ValidationDates": frame[DATE].nunique(), "Products": frame[PID].nunique(),
        })
        overall_rows.append(rec)
        for fold, ff in frame.groupby("OuterFold"):
            fr = metrics(ff["ActualNormalDemand"], ff["PredictedNormalDemand"], "OUTER_DAILY_PRODUCT_BY_FOLD")
            fr.update({"CandidateMethod": method, "OuterFold": int(fold), "EnsembleType": ff["EnsembleType"].iloc[0]})
            fold_rows.append(fr)
    candidate_metrics = pd.DataFrame(overall_rows)
    fold_metrics = pd.DataFrame(fold_rows)

    bench = candidate_metrics.loc[candidate_metrics["CandidateMethod"] == "ROLLING_MEAN_5_BENCHMARK"]
    if len(bench) != 1 or not math.isclose(float(bench.iloc[0]["WAPEPercentage"]), BENCH_WAPE, abs_tol=1e-6):
        raise AssertionError("ND07E benchmark reproduction failed")
    bench_fold = fold_metrics.loc[fold_metrics["CandidateMethod"] == "ROLLING_MEAN_5_BENCHMARK", ["OuterFold", "WAPEPercentage"]].rename(columns={"WAPEPercentage": "BenchmarkFoldWAPE"})

    ensemble = candidate_metrics.loc[candidate_metrics["CandidateMethod"] != "ROLLING_MEAN_5_BENCHMARK"].copy()
    win_rows = []
    for method in ensemble["CandidateMethod"]:
        c = fold_metrics.loc[fold_metrics["CandidateMethod"] == method, ["OuterFold", "WAPEPercentage"]].merge(bench_fold, on="OuterFold", validate="one_to_one")
        win_rows.append({
            "CandidateMethod": method,
            "FoldsWonAgainstBenchmark": int((c["WAPEPercentage"] < c["BenchmarkFoldWAPE"]).sum()),
            "MeanFoldWAPEImprovementPercentagePoints": float((c["BenchmarkFoldWAPE"] - c["WAPEPercentage"]).mean()),
        })
    ensemble = ensemble.merge(pd.DataFrame(win_rows), on="CandidateMethod", validate="one_to_one")
    ensemble["WAPEImprovementPercentagePoints"] = BENCH_WAPE - ensemble["WAPEPercentage"]
    ensemble["RelativeWAPEImprovementPercentage"] = 100.0 * ensemble["WAPEImprovementPercentagePoints"] / BENCH_WAPE
    ensemble["MAERatioToBenchmark"] = ensemble["MAE"] / BENCH_MAE
    ensemble["RMSERatioToBenchmark"] = ensemble["RMSE"] / BENCH_RMSE
    ensemble["StrictlyQualifiesAsDevelopmentMethod"] = (
        (ensemble["WAPEImprovementPercentagePoints"] >= MIN_WAPE_GAIN)
        & (ensemble["MAERatioToBenchmark"] <= MAX_MAE_RATIO)
        & (ensemble["RMSERatioToBenchmark"] <= MAX_RMSE_RATIO)
        & (ensemble["AbsoluteBiasPercentage"] <= MAX_ABS_BIAS)
        & (ensemble["FoldsWonAgainstBenchmark"] >= MIN_FOLDS_WON)
        & (ensemble["OuterFoldsCompleted"] == EXPECTED_FOLDS)
    )
    ensemble = ensemble.sort_values(["WAPEPercentage", "MAE", "AbsoluteBiasPercentage"]).reset_index(drop=True)
    ensemble.insert(0, "EnsembleRank", np.arange(1, len(ensemble) + 1))
    qualified = ensemble.loc[ensemble["StrictlyQualifiesAsDevelopmentMethod"]]
    if qualified.empty:
        selected_method = "ROLLING_MEAN_5_BENCHMARK"
        selected_row = bench.iloc[0]
        recommendation = "ROLLING_MEAN_5_RETAINED_FOR_ND08_AFTER_FOCUSED_ENSEMBLE_TEST"
    else:
        selected_row = qualified.iloc[0]
        selected_method = str(selected_row["CandidateMethod"])
        recommendation = f"{selected_method}_SELECTED_FOR_ND08"

    selected_main = outer_predictions.loc[outer_predictions["CandidateMethod"] == selected_method].copy().sort_values([DATE, PID]).reset_index(drop=True)
    if len(selected_main) != EXPECTED_VALID_ROWS:
        raise AssertionError("Selected main prediction row count mismatch")

    candidate_metrics = candidate_metrics.merge(
        ensemble[[
            "CandidateMethod", "EnsembleRank", "FoldsWonAgainstBenchmark",
            "MeanFoldWAPEImprovementPercentagePoints", "WAPEImprovementPercentagePoints",
            "RelativeWAPEImprovementPercentage", "MAERatioToBenchmark", "RMSERatioToBenchmark",
            "StrictlyQualifiesAsDevelopmentMethod",
        ]], on="CandidateMethod", how="left"
    )
    candidate_metrics["IsBenchmark"] = candidate_metrics["CandidateMethod"] == "ROLLING_MEAN_5_BENCHMARK"
    candidate_metrics["SelectedForND08"] = candidate_metrics["CandidateMethod"] == selected_method
    final_selection = candidate_metrics.loc[candidate_metrics["SelectedForND08"]].copy()
    final_selection["DevelopmentRecommendation"] = recommendation
    final_selection["SelectionUsesMarch2026"] = False
    final_selection["FinalUnbiasedEvaluationStillRequired"] = True
    final_selection["FinalProductionModelFitted"] = False
    final_selection["WeekStartRecursiveEvaluationCompleted"] = False

    # Representative setting selected from the five inner-selected fold settings.
    representative = {"SelectedMethod": selected_method}
    if selected_method == "ROLLING_MEAN_5_BENCHMARK":
        representative.update({"EnsembleType": "BASELINE", "SettingKey": "1.00|0.00|0.00|0.00", "Weights": {c: (1.0 if c == "ROLLING_MEAN_5" else 0.0) for c in COMPONENTS}})
    else:
        sf = outer_selections.loc[outer_selections["CandidateMethod"] == selected_method].copy()
        score_lookup = sf.groupby("SettingKey")["SelectedInnerWAPEPercentage"].mean().to_dict()
        setting = mode_tiebreak(sf["SettingKey"].astype(str).tolist(), score_lookup)
        rr = sf.loc[sf["SettingKey"].astype(str) == setting].sort_values("SelectedInnerWAPEPercentage").iloc[0]
        representative.update({
            "EnsembleType": str(rr["EnsembleType"]), "SettingKey": setting,
            "ActiveComponents": str(rr["ActiveComponents"]).split("|"),
            "Weights": None if rr["EnsembleType"] == "MEDIAN" else {c: float(rr[f"Weight_{c}"]) for c in COMPONENTS},
        })

    # Route selected main prediction through existing fallback methods.
    key = selected_main[[DATE, PID, "OuterFold", "PredictedNormalDemand"]].rename(columns={"OuterFold": "Fold", "PredictedNormalDemand": "ND07EPrediction"})
    routed = routed_base.merge(key, on=[DATE, PID, "Fold"], how="left", validate="one_to_one")
    main_mask = routed[ROUTE].astype(str) == "MAIN_MODEL"
    if routed.loc[main_mask, "ND07EPrediction"].isna().any():
        raise AssertionError("Missing selected main predictions in routed system")
    routed["ND04SelectedPrediction"] = routed["PredictedNormalDemand"]
    routed.loc[main_mask, "PredictedNormalDemand"] = routed.loc[main_mask, "ND07EPrediction"]
    routed.loc[main_mask, "SelectedMethod"] = selected_method
    routed = routed.drop(columns=["ND07EPrediction"])
    routed["ForecastError"] = routed["PredictedNormalDemand"] - routed["ActualNormalDemand"]
    routed["AbsoluteError"] = routed["ForecastError"].abs()
    routed["ForecastMode"] = "DAILY_UPDATED_ONE_STEP_VALIDATION"
    routed["WeeklyInterpretation"] = "DAILY_UPDATED_AGGREGATION_DIAGNOSTIC_NOT_WEEK_START_RECURSIVE"

    daily = routed.groupby(DATE, as_index=False).agg(
        ActualNormalDemand=("ActualNormalDemand", "sum"),
        PredictedNormalDemand=("PredictedNormalDemand", "sum"),
        ProductRows=(PID, "size"), Products=(PID, "nunique"),
    ).sort_values(DATE)
    daily["ForecastError"] = daily["PredictedNormalDemand"] - daily["ActualNormalDemand"]

    routed["WeekStart"] = routed[DATE] - pd.to_timedelta(routed[DATE].dt.dayofweek, unit="D")
    week_meta = routed[[DATE, "WeekStart"]].drop_duplicates().assign(Weekday=lambda x: x[DATE].dt.dayofweek).groupby("WeekStart", as_index=False).agg(
        OperatingDates=(DATE, "nunique"),
        ObservedWeekdays=("Weekday", lambda x: tuple(sorted(set(x)))),
    )
    week_meta["IsCompleteMondayToFridayWeek"] = week_meta["ObservedWeekdays"].apply(lambda x: x == (0, 1, 2, 3, 4))
    weekly_product = routed.groupby(["WeekStart", PID, PNAME], as_index=False).agg(
        ActualNormalDemand=("ActualNormalDemand", "sum"),
        PredictedNormalDemand=("PredictedNormalDemand", "sum"),
        ProductDateRows=(DATE, "size"),
    ).merge(week_meta, on="WeekStart", validate="many_to_one")
    weekly_product["ForecastError"] = weekly_product["PredictedNormalDemand"] - weekly_product["ActualNormalDemand"]
    weekly_restaurant = weekly_product.groupby("WeekStart", as_index=False).agg(
        ActualNormalDemand=("ActualNormalDemand", "sum"),
        PredictedNormalDemand=("PredictedNormalDemand", "sum"),
        Products=(PID, "nunique"), OperatingDates=("OperatingDates", "first"),
        ObservedWeekdays=("ObservedWeekdays", "first"),
        IsCompleteMondayToFridayWeek=("IsCompleteMondayToFridayWeek", "first"),
    )
    weekly_restaurant["ForecastError"] = weekly_restaurant["PredictedNormalDemand"] - weekly_restaurant["ActualNormalDemand"]

    routed_metrics = pd.DataFrame([
        metrics(routed["ActualNormalDemand"], routed["PredictedNormalDemand"], "DAILY_PRODUCT_ALL_ROUTES"),
        metrics(daily["ActualNormalDemand"], daily["PredictedNormalDemand"], "DAILY_RESTAURANT_TOTAL_ALL_ROUTES"),
        metrics(weekly_product["ActualNormalDemand"], weekly_product["PredictedNormalDemand"], "WEEKLY_PRODUCT_DAILY_UPDATED_AGGREGATION_ALL_WEEKS"),
        metrics(weekly_product.loc[weekly_product["IsCompleteMondayToFridayWeek"], "ActualNormalDemand"], weekly_product.loc[weekly_product["IsCompleteMondayToFridayWeek"], "PredictedNormalDemand"], "WEEKLY_PRODUCT_DAILY_UPDATED_AGGREGATION_COMPLETE_WEEKS"),
        metrics(weekly_restaurant["ActualNormalDemand"], weekly_restaurant["PredictedNormalDemand"], "WEEKLY_RESTAURANT_DAILY_UPDATED_AGGREGATION_ALL_WEEKS"),
        metrics(weekly_restaurant.loc[weekly_restaurant["IsCompleteMondayToFridayWeek"], "ActualNormalDemand"], weekly_restaurant.loc[weekly_restaurant["IsCompleteMondayToFridayWeek"], "PredictedNormalDemand"], "WEEKLY_RESTAURANT_DAILY_UPDATED_AGGREGATION_COMPLETE_WEEKS"),
    ])
    routed_metrics["MainRouteMethod"] = selected_method
    routed_metrics["ForecastMode"] = "DAILY_UPDATED_AGGREGATION_DIAGNOSTIC"
    routed_metrics["WeekStartRecursiveForecast"] = False

    route_rows = []
    for route, frame in routed.groupby(ROUTE):
        rec = metrics(frame["ActualNormalDemand"], frame["PredictedNormalDemand"], "DAILY_PRODUCT_BY_ROUTE")
        rec.update({"ForecastRoute": route, "SelectedMethod": frame["SelectedMethod"].iloc[0], "Products": frame[PID].nunique(), "Dates": frame[DATE].nunique()})
        route_rows.append(rec)
    routed_route_metrics = pd.DataFrame(route_rows)

    # Audits.
    weighted = trials.loc[trials["EnsembleType"] == "WEIGHTED_MEAN"]
    weight_cols = [f"Weight_{c}" for c in COMPONENTS]
    max_sum_error = float((weighted[weight_cols].sum(axis=1) - 1.0).abs().max())
    negative_weights = int((weighted[weight_cols] < 0).sum().sum())
    counts = outer_predictions.groupby("CandidateMethod").size()
    duplicate_keys = int(outer_predictions.duplicated([DATE, PID, "OuterFold", "CandidateMethod"]).sum())
    nonfinite = int((~np.isfinite(outer_predictions["PredictedNormalDemand"].to_numpy(dtype=float))).sum())
    negative_predictions = int((outer_predictions["PredictedNormalDemand"] < 0).sum())

    protocol = pd.DataFrame([
        {"Check": "Base configurations inherited from ND07 inner selection", "Expected": True, "Actual": bool(config_audit["ConfigSelectedByND07InnerValidation"].all()), "Passed": bool(config_audit["ConfigSelectedByND07InnerValidation"].all())},
        {"Check": "Outer validation used for ensemble setting selection", "Expected": False, "Actual": False, "Passed": True},
        {"Check": "Inner validation precedes outer validation", "Expected": True, "Actual": bool((replay_audit["InnerValidationEnd"] < replay_audit["OuterValidationStart"]).all()), "Passed": bool((replay_audit["InnerValidationEnd"] < replay_audit["OuterValidationStart"]).all())},
        {"Check": "Negative weighted-ensemble weights", "Expected": 0, "Actual": negative_weights, "Passed": negative_weights == 0},
        {"Check": "Maximum weight-sum error", "Expected": 0.0, "Actual": max_sum_error, "Passed": max_sum_error <= 1e-12},
        {"Check": "Two-model combinations tested", "Expected": True, "Actual": bool((weighted["ModelCount"] == 2).any()), "Passed": bool((weighted["ModelCount"] == 2).any())},
        {"Check": "Three-model combinations tested", "Expected": True, "Actual": bool((weighted["ModelCount"] == 3).any()), "Passed": bool((weighted["ModelCount"] == 3).any())},
        {"Check": "Four-model combinations tested", "Expected": True, "Actual": bool((weighted["ModelCount"] == 4).any()), "Passed": bool((weighted["ModelCount"] == 4).any())},
        {"Check": "Median ensembles tested", "Expected": True, "Actual": bool((trials["EnsembleType"] == "MEDIAN").any()), "Passed": bool((trials["EnsembleType"] == "MEDIAN").any())},
        {"Check": "March target vault opened", "Expected": False, "Actual": False, "Passed": True},
        {"Check": "Final production model fitted", "Expected": False, "Actual": False, "Passed": True},
        {"Check": "Weekly evaluation described as recursive", "Expected": False, "Actual": False, "Passed": True},
    ])
    validation = pd.DataFrame([
        {"Check": "ND07 checkpoint verified", "Expected": EXPECTED_CP["ND07"], "Actual": cp_hashes["ND07"], "Passed": cp_hashes["ND07"] == EXPECTED_CP["ND07"]},
        {"Check": "Outer validation rows per method", "Expected": EXPECTED_VALID_ROWS, "Actual": int(counts.min()), "Passed": bool((counts == EXPECTED_VALID_ROWS).all())},
        {"Check": "Outer validation dates", "Expected": EXPECTED_VALID_DATES, "Actual": int(selected_main[DATE].nunique()), "Passed": int(selected_main[DATE].nunique()) == EXPECTED_VALID_DATES},
        {"Check": "Duplicate prediction keys", "Expected": 0, "Actual": duplicate_keys, "Passed": duplicate_keys == 0},
        {"Check": "Non-finite predictions", "Expected": 0, "Actual": nonfinite, "Passed": nonfinite == 0},
        {"Check": "Negative predictions", "Expected": 0, "Actual": negative_predictions, "Passed": negative_predictions == 0},
        {"Check": "Benchmark WAPE reproduced", "Expected": BENCH_WAPE, "Actual": float(bench.iloc[0]["WAPEPercentage"]), "Passed": math.isclose(float(bench.iloc[0]["WAPEPercentage"]), BENCH_WAPE, abs_tol=1e-6)},
        {"Check": "One selected method", "Expected": 1, "Actual": int(candidate_metrics["SelectedForND08"].sum()), "Passed": int(candidate_metrics["SelectedForND08"].sum()) == 1},
        {"Check": "Missing routed predictions", "Expected": 0, "Actual": int(routed["PredictedNormalDemand"].isna().sum()), "Passed": int(routed["PredictedNormalDemand"].isna().sum()) == 0},
        {"Check": "Previous inputs modified", "Expected": False, "Actual": False, "Passed": True},
        {"Check": "ND07E lock created", "Expected": False, "Actual": False, "Passed": True},
    ])
    if not protocol["Passed"].all():
        raise AssertionError("ND07E protocol audit failed:\n" + protocol.loc[~protocol["Passed"]].to_string(index=False))
    if not validation["Passed"].all():
        raise AssertionError("ND07E validation failed:\n" + validation.loc[~validation["Passed"]].to_string(index=False))

    # Representative base configs.
    rep_configs = {}
    for fam in sorted(EXPECTED_FAMILIES):
        f = selections.loc[selections["CandidateMethod"].astype(str) == fam]
        scores = f.groupby("SelectedConfigID")["SelectedInnerWAPEPercentage"].mean().to_dict()
        cid = mode_tiebreak(f["SelectedConfigID"].astype(str).tolist(), scores)
        cfg = json.loads(str(f.loc[f["SelectedConfigID"].astype(str) == cid, "SelectedConfigJSON"].iloc[0]))
        rep_configs[FAMILY_TO_COMPONENT[fam]] = {"CandidateMethod": fam, "ConfigID": cid, "Config": cfg}

    contract = {
        "StepID": "ND07E", "Status": STATUS, "SelectedMethod": selected_method,
        "DevelopmentRecommendation": recommendation, "RepresentativeSetting": representative,
        "RepresentativeBaseModelConfigurations": rep_configs,
        "WeightGridStep": WEIGHT_STEP,
        "WeightConstraint": "Non-negative weights summing exactly to one",
        "NestedSelectionProtocol": "ND07 inner-selected base configurations; ensemble settings selected on the inner window; untouched outer-fold evaluation",
        "March2026UsedForSelection": False, "FinalProductionModelFitted": False,
        "FinalUnbiasedFutureEvaluationStillRequired": True,
        "ND08Requirement": "Genuine Monday-origin recursive daily-to-weekly evaluation",
    }

    best = ensemble.iloc[0]
    selected_metric = candidate_metrics.loc[candidate_metrics["CandidateMethod"] == selected_method].iloc[0]
    selected_gain = 0.0 if selected_method == "ROLLING_MEAN_5_BENCHMARK" else float(selected_metric["WAPEImprovementPercentagePoints"])
    selected_wins = 0 if selected_method == "ROLLING_MEAN_5_BENCHMARK" else int(selected_metric["FoldsWonAgainstBenchmark"])
    selected_qual = False if selected_method == "ROLLING_MEAN_5_BENCHMARK" else bool(selected_metric["StrictlyQualifiesAsDevelopmentMethod"])

    report = f"""# ND07E Focused Ensemble Challenge Summary

## Status

`{STATUS}`

## Experiment

The experiment combined `ROLLING_MEAN_5`, CatBoost, core XGBoost, and product-aware XGBoost. It tested non-negative two-, three-, and four-model weights on a {WEIGHT_STEP:.2f} grid, together with median subsets. All ensemble settings were selected using the inner chronological validation window only.

## Benchmark

- WAPE: {BENCH_WAPE:.6f}%
- MAE: {BENCH_MAE:.6f}
- RMSE: {BENCH_RMSE:.6f}
- Absolute bias: {BENCH_ABS_BIAS:.6f}%

## Best ensemble

- Method: `{best['CandidateMethod']}`
- WAPE: {best['WAPEPercentage']:.6f}%
- MAE: {best['MAE']:.6f}
- RMSE: {best['RMSE']:.6f}
- Absolute bias: {best['AbsoluteBiasPercentage']:.6f}%
- WAPE improvement: {best['WAPEImprovementPercentagePoints']:.6f} percentage points
- Folds won: {int(best['FoldsWonAgainstBenchmark'])}/{EXPECTED_FOLDS}
- Strict qualification: {bool(best['StrictlyQualifiesAsDevelopmentMethod'])}

## Selected method for ND08

- Method: `{selected_method}`
- WAPE: {selected_metric['WAPEPercentage']:.6f}%
- MAE: {selected_metric['MAE']:.6f}
- RMSE: {selected_metric['RMSE']:.6f}
- Absolute bias: {selected_metric['AbsoluteBiasPercentage']:.6f}%
- WAPE improvement: {selected_gain:.6f} percentage points
- Folds won: {selected_wins}/{EXPECTED_FOLDS}
- Strict qualification: {selected_qual}

## Recommendation

`{recommendation}`

March 2026 remained closed. No final model was fitted or saved. Weekly outputs are daily-updated aggregation diagnostics; ND08 remains the genuine Monday-origin recursive evaluation.
"""

    # Figures.
    stage_fig = STAGE / "05_figures"
    plot = candidate_metrics.sort_values("WAPEPercentage")
    plt.figure(figsize=(11, 7)); plt.barh(plot["CandidateMethod"], plot["WAPEPercentage"]); plt.axvline(BENCH_WAPE, linestyle="--"); plt.xlabel("WAPE (%)"); plt.title("ND07E nested ensemble comparison"); savefig(stage_fig / "ND07E_figure_01_candidate_wape.png")
    plt.figure(figsize=(10, 6)); plt.scatter(ensemble["WAPEPercentage"], ensemble["AbsoluteBiasPercentage"])
    for _, r in ensemble.iterrows(): plt.annotate(str(r["CandidateMethod"]), (r["WAPEPercentage"], r["AbsoluteBiasPercentage"]), fontsize=7)
    plt.axvline(BENCH_WAPE, linestyle="--"); plt.axhline(MAX_ABS_BIAS, linestyle="--"); plt.xlabel("WAPE (%)"); plt.ylabel("Absolute bias (%)"); plt.title("Ensemble WAPE versus bias"); savefig(stage_fig / "ND07E_figure_02_wape_vs_bias.png")
    plt.figure(figsize=(11, 6))
    for method, f in fold_metrics.groupby("CandidateMethod"): plt.plot(f["OuterFold"], f["WAPEPercentage"], marker="o", label=method)
    plt.xlabel("Outer fold"); plt.ylabel("WAPE (%)"); plt.title("ND07E WAPE by outer fold"); plt.legend(fontsize=8); savefig(stage_fig / "ND07E_figure_03_fold_wape.png")
    plt.figure(figsize=(14, 6)); plt.plot(daily[DATE], daily["ActualNormalDemand"], label="Actual"); plt.plot(daily[DATE], daily["PredictedNormalDemand"], label="Predicted"); plt.legend(); plt.title("Selected routed daily restaurant total"); savefig(stage_fig / "ND07E_figure_04_daily_restaurant.png")

    # Write staged outputs.
    stage_pred = STAGE / "01_predictions"; stage_metric = STAGE / "02_metrics"; stage_audit = STAGE / "03_audits"; stage_contract = STAGE / "04_contracts"; stage_report = STAGE / "06_reports"; stage_control = STAGE / "07_control"
    outputs = {
        stage_pred / "ND07E_inner_base_model_predictions.csv": inner_base,
        stage_pred / "ND07E_outer_base_model_predictions.csv": outer_base,
        stage_pred / "ND07E_outer_ensemble_predictions.csv": outer_predictions,
        stage_pred / "ND07E_selected_main_predictions.csv": selected_main,
        stage_pred / "ND07E_selected_routed_predictions.csv": routed,
        stage_pred / "ND07E_selected_daily_restaurant_totals.csv": daily,
        stage_pred / "ND07E_selected_weekly_product_totals.csv": weekly_product,
        stage_pred / "ND07E_selected_weekly_restaurant_totals.csv": weekly_restaurant,
        stage_metric / "ND07E_inner_weight_and_median_trials.csv": trials,
        stage_metric / "ND07E_outer_fold_selected_ensemble_settings.csv": outer_selections,
        stage_metric / "ND07E_outer_candidate_metrics.csv": candidate_metrics,
        stage_metric / "ND07E_outer_candidate_metrics_by_fold.csv": fold_metrics,
        stage_metric / "ND07E_final_ensemble_selection.csv": final_selection,
        stage_metric / "ND07E_selected_routed_system_metrics.csv": routed_metrics,
        stage_metric / "ND07E_selected_routed_metrics_by_route.csv": routed_route_metrics,
        stage_audit / "ND07E_base_model_configuration_audit.csv": config_audit,
        stage_audit / "ND07E_nested_split_replay_audit.csv": replay_audit,
        stage_audit / "ND07E_weight_grid_audit.csv": grid.groupby("ModelCount", as_index=False).size().rename(columns={"size": "WeightVectors"}),
        stage_audit / "ND07E_input_hash_audit.csv": input_hash_audit,
        stage_audit / "ND07E_package_versions.csv": pd.DataFrame([
            {"Package": "python", "Version": platform.python_version()}, {"Package": "pandas", "Version": pd.__version__},
            {"Package": "numpy", "Version": np.__version__}, {"Package": "scikit-learn", "Version": sklearn.__version__},
            {"Package": "xgboost", "Version": xgboost.__version__}, {"Package": "catboost", "Version": catboost.__version__},
        ]),
        stage_audit / "ND07E_leakage_and_protocol_audit.csv": protocol,
        stage_audit / "ND07E_validation_summary.csv": validation,
    }
    for p, df in outputs.items(): wcsv(p, df)
    wjson(stage_contract / "ND07E_selected_development_method_contract.json", contract)
    wtext(stage_contract / "ND07E_selected_development_method_contract.md", report)
    wtext(stage_report / "ND07E_focused_ensemble_summary.md", report)
    decision = {
        "StepID": "ND07E", "Status": STATUS, "CreatedUTC": NOW_UTC.isoformat(),
        "BestEnsembleMethod": str(best["CandidateMethod"]),
        "BestEnsembleMetrics": {k: (bool(best[k]) if k == "StrictlyQualifiesAsDevelopmentMethod" else float(best[k])) for k in ["WAPEPercentage", "MAE", "RMSE", "TotalBias", "AbsoluteBiasPercentage", "WAPEImprovementPercentagePoints", "StrictlyQualifiesAsDevelopmentMethod"]},
        "SelectedMethod": selected_method, "DevelopmentRecommendation": recommendation,
        "RepresentativeSetting": representative, "March2026UsedForSelection": False,
        "OuterValidationUsedForWeightSelection": False, "FinalProductionModelFitted": False,
        "WeekStartRecursiveEvaluationCompleted": False, "NextStep": "ND08",
    }
    wjson(stage_report / "ND07E_decision.json", decision)
    wtext(STAGE / "README.md", f"# ND07E Focused Ensemble Challenge\n\nStatus: `{STATUS}`\n\nSelected method for ND08: `{selected_method}`\n\nWeights were selected using inner chronological validation only. March 2026 remained closed.\n")

    # Verify immutability.
    protected_after = {str(p): sha(p) for p in required if p.is_file()}
    changed = [p for p in protected_before if protected_before[p] != protected_after[p]]
    if changed:
        raise AssertionError("Protected inputs changed:\n" + "\n".join(changed))

    # Manifest and checkpoint.
    manifest_files = sorted(p for p in STAGE.rglob("*") if p.is_file() and p.name not in {"ND07E_artifact_hash_manifest.csv", "ND07E_checkpoint.json", "ND07E_checkpoint.sha256"})
    manifest = pd.DataFrame([{"RelativePath": str(p.relative_to(STAGE)), "Bytes": p.stat().st_size, "SHA256": sha(p)} for p in manifest_files]).sort_values("RelativePath")
    manifest_path = stage_control / "ND07E_artifact_hash_manifest.csv"
    wcsv(manifest_path, manifest)
    checkpoint = {
        "StepID": "ND07E", "Status": STATUS, "CreatedUTC": NOW_UTC.isoformat(), "CreatedLocal": NOW_LOCAL.isoformat(),
        "ND07ERoot": str(OUT), "InputCheckpoints": cp_hashes, "SelectedMethod": selected_method,
        "DevelopmentRecommendation": recommendation, "BestEnsembleMethod": str(best["CandidateMethod"]),
        "BestEnsembleWAPEPercentage": float(best["WAPEPercentage"]),
        "SelectedWAPEPercentage": float(selected_metric["WAPEPercentage"]),
        "WeightGridStep": WEIGHT_STEP, "TotalFittingSeconds": fitting_seconds,
        "Manifest": {"Path": str(CONTROL_DIR / "ND07E_artifact_hash_manifest.csv"), "SHA256": sha(manifest_path)},
        "Safety": {"MarchTargetVaultOpened": False, "OuterValidationUsedForWeightSelection": False, "FinalProductionModelFitted": False, "FinalModelArtifactSaved": False, "PreviousInputsModified": False, "ExistingLocksModified": False, "ND07EStepLockCreated": False, "CheckpointAndHashesCreated": True},
        "ReadyForND08": True, "NextStep": "ND08",
    }
    cp_stage = stage_control / "ND07E_checkpoint.json"
    wjson(cp_stage, checkpoint)
    cp_sha = sha(cp_stage)
    wtext(stage_control / "ND07E_checkpoint.sha256", f"{cp_sha}  ND07E_checkpoint.json\n")

    os.replace(STAGE, OUT)
    TOP_CP.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(CONTROL_DIR / "ND07E_checkpoint.json", TOP_CP)
    shutil.copy2(CONTROL_DIR / "ND07E_checkpoint.sha256", TOP_CP_SHA)

    # Memory and logs.
    handoff = f"""# ND07E Handoff

## Status

- Status: `{STATUS}`
- Root: `{OUT}`
- Checkpoint SHA-256: `{cp_sha}`

## Best ensemble

- Method: `{best['CandidateMethod']}`
- WAPE: {best['WAPEPercentage']:.6f}%
- WAPE improvement: {best['WAPEImprovementPercentagePoints']:.6f} percentage points
- Folds won: {int(best['FoldsWonAgainstBenchmark'])}/{EXPECTED_FOLDS}
- Strict qualification: {bool(best['StrictlyQualifiesAsDevelopmentMethod'])}

## Selected method for ND08

- Method: `{selected_method}`
- Recommendation: `{recommendation}`
- WAPE: {selected_metric['WAPEPercentage']:.6f}%

March 2026 remained closed. Outer validation was not used for ensemble-weight selection. No final model was fitted or saved.
"""
    atomic_text(HANDOFF, handoff)
    atomic_text(CURRENT_HANDOFF, handoff)
    append_once(WORKFLOW, "## ND07E — Focused nested ensemble challenge", f"## ND07E — Focused nested ensemble challenge\n\nStatus: `{STATUS}`\n\n- Best ensemble: `{best['CandidateMethod']}`\n- Selected method for ND08: `{selected_method}`\n- Recommendation: `{recommendation}`\n- March remained closed.\n")
    append_once(DECISIONS, "## ND07E decisions", f"## ND07E decisions\n\n1. Tested non-negative two-, three-, and four-model weights summing to one.\n2. Tested median ensembles.\n3. Selected weights using inner validation only.\n4. Selected method for ND08: `{selected_method}`.\n5. Recommendation: `{recommendation}`.\n")
    append_once(RESULTS_MEMORY, "## ND07E focused ensemble results", f"## ND07E focused ensemble results\n\n- Best ensemble: `{best['CandidateMethod']}`\n- Best ensemble WAPE: {best['WAPEPercentage']:.6f}%\n- Selected method: `{selected_method}`\n- Selected WAPE: {selected_metric['WAPEPercentage']:.6f}%\n")
    append_once(AGENTS, "Marker: ND07E_AUTHORITATIVE_STATUS", f"## ND07E authoritative status\n\nMarker: ND07E_AUTHORITATIVE_STATUS\n\n- Status: `{STATUS}`\n- Selected method for ND08: `{selected_method}`\n- Recommendation: `{recommendation}`\n- Next step: `ND08`\n")
    atomic_text(LOG, f"Status: {STATUS}\nBest ensemble: {best['CandidateMethod']}\nSelected method: {selected_method}\nCheckpoint SHA-256: {cp_sha}\n")

except Exception:
    if STAGE.exists():
        shutil.rmtree(STAGE)
    raise

# -----------------------------------------------------------------------------
# Console output
# -----------------------------------------------------------------------------
print("=" * 112)
print("EDEN NORMAL-DEMAND MODEL V2 — ND07E COMPLETE")
print("=" * 112)
print(f"Status: {STATUS}")
print(f"Local time: {NOW_LOCAL.isoformat()}")
print(f"ND07E root: {OUT}")
print("\nINPUT VERIFICATION")
for step in ["ND03", "ND04", "ND05", "ND06", "ND07"]:
    print(f"{step} checkpoint SHA-256: {cp_hashes[step]}")
print(f"Main development rows: {len(dev):,}")
print(f"Core predictors: {len(core)}")
print("March target vault opened: False")
print("Previous inputs modified: False")
print("\nENSEMBLE DESIGN")
print("Base components: " + ", ".join(COMPONENTS))
print(f"Weight-grid step: {WEIGHT_STEP:.2f}")
for n in [2, 3, 4]:
    print(f"Weighted {n}-model combinations per outer fold: {int((grid['ModelCount'] == n).sum()):,}")
print(f"Median subsets per outer fold: {sum(math.comb(4, n) for n in [2, 3, 4])}")
print("Outer validation used for weight selection: False")
print(f"Total fitting seconds: {fitting_seconds:.3f}")
print("\nACCEPTED BENCHMARK")
print(f"WAPE: {BENCH_WAPE:.6f}%")
print(f"MAE: {BENCH_MAE:.6f}")
print(f"RMSE: {BENCH_RMSE:.6f}")
print(f"Absolute bias percentage: {BENCH_ABS_BIAS:.6f}%")
print("\nENSEMBLE RESULTS")
print(ensemble[["EnsembleRank", "CandidateMethod", "WAPEPercentage", "MAE", "RMSE", "TotalBias", "AbsoluteBiasPercentage", "WAPEImprovementPercentagePoints", "FoldsWonAgainstBenchmark", "StrictlyQualifiesAsDevelopmentMethod"]].to_string(index=False))
print("\nBEST ENSEMBLE")
print(f"Method: {best['CandidateMethod']}")
print(f"WAPE: {best['WAPEPercentage']:.6f}%")
print(f"WAPE improvement versus benchmark: {best['WAPEImprovementPercentagePoints']:.6f} percentage points")
print(f"Folds won: {int(best['FoldsWonAgainstBenchmark'])}/{EXPECTED_FOLDS}")
print(f"Strict qualification: {bool(best['StrictlyQualifiesAsDevelopmentMethod'])}")
print("\nSELECTED DEVELOPMENT METHOD FOR ND08")
print(f"Method: {selected_method}")
print(f"WAPE: {selected_metric['WAPEPercentage']:.6f}%")
print(f"MAE: {selected_metric['MAE']:.6f}")
print(f"RMSE: {selected_metric['RMSE']:.6f}")
print(f"Absolute bias percentage: {selected_metric['AbsoluteBiasPercentage']:.6f}%")
print(f"Recommendation: {recommendation}")
print("\nSELECTED ROUTED SYSTEM")
print(routed_metrics[["EvaluationLevel", "Observations", "WAPEPercentage", "MAE", "RMSE", "TotalBias"]].to_string(index=False))
print("Forecast mode: DAILY_UPDATED_AGGREGATION_DIAGNOSTIC")
print("Week-start recursive forecast: False")
print("\nOUTPUTS")
print(f"- Inner trials: {METRIC_DIR / 'ND07E_inner_weight_and_median_trials.csv'}")
print(f"- Outer predictions: {PRED_DIR / 'ND07E_outer_ensemble_predictions.csv'}")
print(f"- Candidate metrics: {METRIC_DIR / 'ND07E_outer_candidate_metrics.csv'}")
print(f"- Final selection: {METRIC_DIR / 'ND07E_final_ensemble_selection.csv'}")
print(f"- Method contract: {CONTRACT_DIR / 'ND07E_selected_development_method_contract.json'}")
print(f"- Report: {REPORT_DIR / 'ND07E_focused_ensemble_summary.md'}")
print(f"- Checkpoint: {TOP_CP}")
print(f"- Checkpoint SHA-256: {cp_sha}")
print(f"- Handoff: {HANDOFF}")
print("\nSAFETY")
print("- Ensemble settings selected using inner validation only: True")
print("- Outer validation used for weight selection: False")
print("- March target vault opened: False")
print("- Final production model fitted: False")
print("- Final model artifact saved: False")
print("- Previous inputs modified: False")
print("- ND07E step lock created: False")
print("- ND07E checkpoint and hashes created: True")
print("\nNEXT STEP")
print("ND08 — genuine Monday-origin recursive daily-to-weekly evaluation using the selected development method.")
print("=" * 112)

Outer fold 1/5: ensemble settings selected using inner validation only
Outer fold 2/5: ensemble settings selected using inner validation only
Outer fold 3/5: ensemble settings selected using inner validation only
Outer fold 4/5: ensemble settings selected using inner validation only
Outer fold 5/5: ensemble settings selected using inner validation only
EDEN NORMAL-DEMAND MODEL V2 — ND07E COMPLETE
Status: ND07E_FOCUSED_ENSEMBLE_CHALLENGE_COMPLETED_READY_FOR_ND08
Local time: 2026-08-08T00:07:21.646145+01:00
ND07E root: /Users/ryansmac/Desktop/Meng Project/eden_datasets/eden_normal_demand_model_v2/03_models/01_tuning/ND07E_focused_ensemble_challenge

INPUT VERIFICATION
ND03 checkpoint SHA-256: 0845af89a5b459ca13ae6ffd99dde444f5010f6c0fb5ba4c34a4f091ac2e151c
ND04 checkpoint SHA-256: 2fdc5d2c64c38f85b2669ca942042884209d80111cc840261307da98b1e9cf54
ND05 checkpoint SHA-256: ce3342c8b960aa5c4791a114ab09ae1a86d1dab2a3eebb648aa060579a1378ff
ND06 checkpoint SHA-256: e3b7bb75a1b2968e426c9a4c1654e1

In [11]:
from __future__ import annotations

# =============================================================================
# EDEN NORMAL-DEMAND MODEL V2
# ND08 — GENUINE MONDAY-ORIGIN RECURSIVE DAILY-TO-WEEKLY EVALUATION
# Compatibility revision: 1.1 (ND07E SettingKey schema fix)
#
# Run this as one complete Jupyter cell after ND07E.
#
# Methods compared:
#   1. ROLLING_MEAN_5
#   2. NESTED_MEDIAN_ENSEMBLE
#   3. NESTED_WEIGHTED_ENSEMBLE_4_MODELS
#
# The Monday-origin evaluation never uses actual demand from inside the week.
# Monday is forecast from pre-Monday history. Monday's prediction is inserted
# into temporary history before Tuesday is forecast, and so on through Friday.
# Dynamic routes and all 31 historical demand features are recalculated from
# the method-specific temporary history before every forecast day.
#
# A separate daily-updated reconstruction is evaluated on the identical set of
# complete weeks. It uses actual demand available before each day and is kept
# strictly separate from the Monday-origin recursive result.
#
# March 2026 remains closed. No final production model is fitted or saved.
# =============================================================================

import hashlib
import json
import math
import os
import platform
import shutil
import time
import uuid
import warnings
from collections import Counter
from datetime import datetime, timezone
from pathlib import Path
from zoneinfo import ZoneInfo

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import sklearn
from matplotlib.dates import DateFormatter
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OrdinalEncoder

try:
    import xgboost
    from xgboost import XGBRegressor
except Exception as error:
    raise RuntimeError(
        "ND08 requires xgboost. Restore the environment used for ND07E."
    ) from error

try:
    import catboost
    from catboost import CatBoostRegressor
except Exception as error:
    raise RuntimeError(
        "ND08 requires catboost. Restore the environment used for ND07E."
    ) from error

warnings.filterwarnings(
    "ignore",
    message="X does not have valid feature names",
)

os.environ.setdefault("OMP_NUM_THREADS", "1")
os.environ.setdefault("OPENBLAS_NUM_THREADS", "1")
os.environ.setdefault("MKL_NUM_THREADS", "1")
os.environ.setdefault("NUMEXPR_NUM_THREADS", "1")


# =============================================================================
# CONFIGURATION
# =============================================================================

PROJECT_ROOT = Path("/Users/ryansmac/Desktop/Meng Project")
EDEN_ROOT = PROJECT_ROOT / "eden_datasets"
MODEL_ROOT = EDEN_ROOT / "eden_normal_demand_model_v2"

ND03_ROOT = (
    MODEL_ROOT
    / "02_feature_engineering"
    / "ND03_normal_demand_features"
)

ND04_ROOT = (
    MODEL_ROOT
    / "03_models"
    / "00_candidates"
    / "ND04_baseline_and_fallback_evaluation"
)

ND07_ROOT = (
    MODEL_ROOT
    / "03_models"
    / "01_tuning"
    / "ND07_tuning_calibration_blending"
)

ND07E_ROOT = (
    MODEL_ROOT
    / "03_models"
    / "01_tuning"
    / "ND07E_focused_ensemble_challenge"
)

ALL_ROUTES_PATH = (
    ND03_ROOT
    / "01_model_ready_datasets"
    / "ND03_pre_march_all_routes_development_dataset.csv"
)

CORE_PREDICTOR_LIST_PATH = (
    ND03_ROOT
    / "03_contracts"
    / "ND03_core_predictor_list.csv"
)

ND03_MANIFEST_PATH = (
    ND03_ROOT
    / "05_control"
    / "ND03_artifact_hash_manifest.csv"
)

FOLD_DEFINITION_PATH = (
    ND04_ROOT
    / "03_audits"
    / "ND04_chronological_fold_definition.csv"
)

ND04_SELECTED_METHODS_PATH = (
    ND04_ROOT
    / "02_metrics"
    / "ND04_selected_route_methods.csv"
)

ND04_ROUTED_PREDICTIONS_PATH = (
    ND04_ROOT
    / "01_predictions"
    / "ND04_selected_routed_system_predictions.csv"
)

ND04_MANIFEST_PATH = (
    ND04_ROOT
    / "06_control"
    / "ND04_artifact_hash_manifest.csv"
)

ND07_OUTER_SELECTIONS_PATH = (
    ND07_ROOT
    / "02_metrics"
    / "ND07_outer_fold_selected_configurations.csv"
)

ND07_NESTED_SPLIT_AUDIT_PATH = (
    ND07_ROOT
    / "03_audits"
    / "ND07_nested_split_audit.csv"
)

ND07_MANIFEST_PATH = (
    ND07_ROOT
    / "07_control"
    / "ND07_artifact_hash_manifest.csv"
)

ND07E_OUTER_SELECTIONS_PATH = (
    ND07E_ROOT
    / "02_metrics"
    / "ND07E_outer_fold_selected_ensemble_settings.csv"
)

ND07E_OUTER_PREDICTIONS_PATH = (
    ND07E_ROOT
    / "01_predictions"
    / "ND07E_outer_ensemble_predictions.csv"
)

ND07E_FINAL_SELECTION_PATH = (
    ND07E_ROOT
    / "02_metrics"
    / "ND07E_final_ensemble_selection.csv"
)

ND07E_MANIFEST_PATH = (
    ND07E_ROOT
    / "07_control"
    / "ND07E_artifact_hash_manifest.csv"
)

CHECKPOINT_PATHS = {
    "ND03": MODEL_ROOT / "08_checkpoints" / "ND03_checkpoint.json",
    "ND04": MODEL_ROOT / "08_checkpoints" / "ND04_checkpoint.json",
    "ND05": MODEL_ROOT / "08_checkpoints" / "ND05_checkpoint.json",
    "ND06": MODEL_ROOT / "08_checkpoints" / "ND06_checkpoint.json",
    "ND07": MODEL_ROOT / "08_checkpoints" / "ND07_checkpoint.json",
    "ND07E": MODEL_ROOT / "08_checkpoints" / "ND07E_checkpoint.json",
}

EXPECTED_CHECKPOINT_HASHES = {
    "ND03": "0845af89a5b459ca13ae6ffd99dde444f5010f6c0fb5ba4c34a4f091ac2e151c",
    "ND04": "2fdc5d2c64c38f85b2669ca942042884209d80111cc840261307da98b1e9cf54",
    "ND05": "ce3342c8b960aa5c4791a114ab09ae1a86d1dab2a3eebb648aa060579a1378ff",
    "ND06": "e3b7bb75a1b2968e426c9a4c1654e10420d683af4bd5cb2b75fa4b5f0f18f357",
    "ND07": "39077dd8c197561e5384a6943c3a4e153153e75fa4d03019f45005eb145cf18e",
    "ND07E": "eaf6d23434cb67d663f66a39db4f898b74c1d516aee5c0627ca9136bed9877b8",
}

DATE_COLUMN = "Date"
PRODUCT_ID_COLUMN = "CanonicalProductID"
PRODUCT_NAME_COLUMN = "CanonicalProductName"
TARGET_COLUMN = "NormalDemand"
ROUTE_COLUMN = "ForecastRoute"
FAMILY_COLUMN = "TierProductFamily"
DAY_OF_WEEK_COLUMN = "DayOfWeekNumber"
SEQUENCE_COLUMN = "OperatingDaySequence"

EXPECTED_ALL_ROUTE_ROWS = 23_763
EXPECTED_OUTER_FOLDS = 5
EXPECTED_OUTER_VALIDATION_DATES = 100
EXPECTED_CORE_PREDICTORS = 53
EXPECTED_NUMERIC_PREDICTORS = 45
EXPECTED_CATEGORICAL_PREDICTORS = 8
EXPECTED_ND07E_SELECTION = "NESTED_MEDIAN_ENSEMBLE"

LAG_OPERATING_DAYS = [1, 2, 3, 5, 10, 20]
ROLLING_WINDOWS = [3, 5, 10, 20]
ZERO_POSITIVE_WINDOWS = [5, 10, 20]
MINIMUM_MAIN_HISTORY = 20
PRIMARY_SCOPE_PERCENTAGE = 95.0

HISTORICAL_DEMAND_PREDICTORS = (
    [f"NormalDemandLag_{lag}" for lag in LAG_OPERATING_DAYS]
    + [
        feature
        for window in ROLLING_WINDOWS
        for feature in [
            f"PastNormalDemandRollingMean_{window}",
            f"PastNormalDemandRollingMedian_{window}",
            f"PastNormalDemandRollingStd_{window}",
            f"PastNormalDemandRollingSum_{window}",
        ]
    ]
    + [
        f"PastZeroNormalDemandRate_{window}"
        for window in ZERO_POSITIVE_WINDOWS
    ]
    + [
        f"PastPositiveNormalDemandCount_{window}"
        for window in ZERO_POSITIVE_WINDOWS
    ]
    + [
        "OperatingDaysSincePreviousPositiveNormalDemand",
        "ExpandingPastMeanNormalDemand",
        "ExpandingPastPositiveNormalDemandRate",
    ]
)

METHODS = [
    "ROLLING_MEAN_5",
    "NESTED_MEDIAN_ENSEMBLE",
    "NESTED_WEIGHTED_ENSEMBLE_4_MODELS",
]

ND07E_METHOD_MAP = {
    "ROLLING_MEAN_5": "ROLLING_MEAN_5_BENCHMARK",
    "NESTED_MEDIAN_ENSEMBLE": "NESTED_MEDIAN_ENSEMBLE",
    "NESTED_WEIGHTED_ENSEMBLE_4_MODELS": (
        "NESTED_WEIGHTED_ENSEMBLE_4_MODELS"
    ),
}

BASE_COMPONENTS = [
    "ROLLING_MEAN_5",
    "CATBOOST_CORE53",
    "XGBOOST_CORE53",
    "XGBOOST_PRODUCT_AWARE",
]

FAMILY_TO_COMPONENT = {
    "NESTED_TUNED_CATBOOST_RMSE_CORE53": "CATBOOST_CORE53",
    "NESTED_TUNED_XGBOOST_SQUARED_CORE53": "XGBOOST_CORE53",
    "NESTED_TUNED_XGBOOST_SQUARED_PRODUCT_AWARE": (
        "XGBOOST_PRODUCT_AWARE"
    ),
}

EXPECTED_FAMILIES = set(FAMILY_TO_COMPONENT)

EXPECTED_ROUTE_METHODS = {
    "MAIN_MODEL": "ROLLING_MEAN_5",
    "COLD_START_FALLBACK": "RECENT_MEDIAN5_WITH_BACKOFF",
    "LOW_DEMAND_FALLBACK": "RECENT_MEDIAN5_WITH_BACKOFF",
    "ZERO_HISTORY_FALLBACK": "EXPANDING_MEAN_WITH_BACKOFF",
}

MINIMUM_WAPE_IMPROVEMENT_PP = 0.50
MAXIMUM_MAE_RATIO = 1.02
MAXIMUM_RMSE_RATIO = 1.10
MAXIMUM_ABSOLUTE_BIAS_PERCENTAGE = 3.00
MINIMUM_FOLDS_WON = 3

RANDOM_SEED = 42
MAX_THREADS = max(1, min(4, os.cpu_count() or 1))

ND08_ROOT = (
    MODEL_ROOT
    / "03_models"
    / "02_system_evaluation"
    / "ND08_recursive_weekly_system"
)

PREDICTION_DIR = ND08_ROOT / "01_predictions"
METRIC_DIR = ND08_ROOT / "02_metrics"
AUDIT_DIR = ND08_ROOT / "03_audits"
CONTRACT_DIR = ND08_ROOT / "04_contracts"
FIGURE_DIR = ND08_ROOT / "05_figures"
REPORT_DIR = ND08_ROOT / "06_reports"
CONTROL_DIR = ND08_ROOT / "07_control"

RECURSIVE_DAILY_PATH = (
    PREDICTION_DIR / "ND08_monday_origin_recursive_daily_predictions.csv"
)
RECURSIVE_PRODUCT_WEEK_PATH = (
    PREDICTION_DIR / "ND08_monday_origin_product_week_predictions.csv"
)
RECURSIVE_RESTAURANT_WEEK_PATH = (
    PREDICTION_DIR / "ND08_monday_origin_restaurant_week_predictions.csv"
)
DAILY_UPDATED_ROUTED_PATH = (
    PREDICTION_DIR / "ND08_daily_updated_routed_predictions.csv"
)
DAILY_UPDATED_PRODUCT_WEEK_PATH = (
    PREDICTION_DIR / "ND08_daily_updated_product_week_predictions.csv"
)
DAILY_UPDATED_RESTAURANT_WEEK_PATH = (
    PREDICTION_DIR / "ND08_daily_updated_restaurant_week_predictions.csv"
)

RECURSIVE_METRICS_PATH = (
    METRIC_DIR / "ND08_monday_origin_metrics.csv"
)
RECURSIVE_FOLD_METRICS_PATH = (
    METRIC_DIR / "ND08_monday_origin_product_week_metrics_by_fold.csv"
)
RECURSIVE_WEEK_METRICS_PATH = (
    METRIC_DIR / "ND08_monday_origin_product_week_metrics_by_week.csv"
)
DAILY_UPDATED_METRICS_PATH = (
    METRIC_DIR / "ND08_daily_updated_metrics.csv"
)
DAILY_UPDATED_FOLD_METRICS_PATH = (
    METRIC_DIR / "ND08_daily_updated_product_week_metrics_by_fold.csv"
)
DAILY_MAIN_METRICS_PATH = (
    METRIC_DIR / "ND08_daily_main_route_metrics.csv"
)
OPERATIONAL_SELECTION_PATH = (
    METRIC_DIR / "ND08_operational_method_selection.csv"
)
ROUTE_METRICS_PATH = (
    METRIC_DIR / "ND08_recursive_metrics_by_route.csv"
)

WEEK_REGISTRY_PATH = AUDIT_DIR / "ND08_complete_week_registry.csv"
EXCLUDED_WEEK_AUDIT_PATH = AUDIT_DIR / "ND08_excluded_partial_week_audit.csv"
RECURSION_AUDIT_PATH = AUDIT_DIR / "ND08_recursion_engine_audit.csv"
ROUTE_CHANGE_AUDIT_PATH = AUDIT_DIR / "ND08_recursive_route_change_audit.csv"
WEEK_MODEL_AUDIT_PATH = AUDIT_DIR / "ND08_week_origin_model_fit_audit.csv"
AGGREGATION_RECONCILIATION_PATH = (
    AUDIT_DIR / "ND08_weekly_aggregation_reconciliation.csv"
)
INPUT_HASH_AUDIT_PATH = AUDIT_DIR / "ND08_input_hash_audit.csv"
PACKAGE_VERSIONS_PATH = AUDIT_DIR / "ND08_package_versions.csv"
PROTOCOL_AUDIT_PATH = AUDIT_DIR / "ND08_leakage_and_protocol_audit.csv"
VALIDATION_PATH = AUDIT_DIR / "ND08_validation_summary.csv"

SYSTEM_CONTRACT_PATH = CONTRACT_DIR / "ND08_operational_method_contract.json"
SYSTEM_CONTRACT_MD_PATH = CONTRACT_DIR / "ND08_operational_method_contract.md"
REPORT_SUMMARY_PATH = REPORT_DIR / "ND08_recursive_weekly_system_summary.md"
DECISION_JSON_PATH = REPORT_DIR / "ND08_decision.json"
README_PATH = ND08_ROOT / "README.md"

MANIFEST_PATH = CONTROL_DIR / "ND08_artifact_hash_manifest.csv"
CHECKPOINT_PATH = CONTROL_DIR / "ND08_checkpoint.json"
CHECKPOINT_SHA_PATH = CONTROL_DIR / "ND08_checkpoint.sha256"
TOP_LEVEL_CHECKPOINT_PATH = MODEL_ROOT / "08_checkpoints" / "ND08_checkpoint.json"
TOP_LEVEL_CHECKPOINT_SHA_PATH = (
    MODEL_ROOT / "08_checkpoints" / "ND08_checkpoint.sha256"
)

MEMORY_ROOT = MODEL_ROOT / "00_project_memory"
ND08_HANDOFF_PATH = MEMORY_ROOT / "ND08_HANDOFF.md"
CURRENT_HANDOFF_PATH = MEMORY_ROOT / "CURRENT_HANDOFF.md"
WORKFLOW_PATH = MEMORY_ROOT / "WORKFLOW.md"
DECISIONS_PATH = MEMORY_ROOT / "DECISIONS.md"
METRICS_AND_RESULTS_PATH = MEMORY_ROOT / "METRICS_AND_RESULTS.md"
AGENTS_PATH = MODEL_ROOT / "AGENTS.md"
LOG_PATH = MODEL_ROOT / "09_logs" / "ND08_recursive_weekly_log.txt"

ALLOW_OVERWRITE = False
STEP_ID = "ND08"
STATUS = "ND08_RECURSIVE_DAILY_TO_WEEKLY_SYSTEM_EVALUATED_READY_FOR_ND09"

NOW_UTC = datetime.now(timezone.utc)
NOW_LOCAL = NOW_UTC.astimezone(ZoneInfo("Europe/Dublin"))


# =============================================================================
# GENERAL HELPERS
# =============================================================================

def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


def write_csv(path: Path, frame: pd.DataFrame) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    frame.to_csv(path, index=False)


def write_json(path: Path, payload: dict) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(
        json.dumps(payload, indent=2, ensure_ascii=False, default=str) + "\n",
        encoding="utf-8",
    )


def write_text(path: Path, text: str) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(text, encoding="utf-8")


def atomic_write_text(path: Path, text: str) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_name(f".{path.name}.{uuid.uuid4().hex}.tmp")
    temporary.write_text(text, encoding="utf-8")
    os.replace(temporary, path)


def append_marked_section(path: Path, marker: str, section_text: str) -> None:
    existing = path.read_text(encoding="utf-8") if path.is_file() else ""
    if marker in existing:
        return
    separator = "\n" if existing.endswith("\n") else "\n\n"
    atomic_write_text(path, existing + separator + section_text.strip() + "\n")


def validate_required_columns(
    frame: pd.DataFrame,
    columns: set[str],
    name: str,
) -> None:
    missing = sorted(columns - set(frame.columns))
    if missing:
        raise AssertionError(
            f"{name} is missing required columns:\n"
            + "\n".join(f"- {column}" for column in missing)
        )


def safe_wape(actual, predicted) -> float:
    actual_array = np.asarray(actual, dtype=float)
    predicted_array = np.asarray(predicted, dtype=float)
    denominator = float(actual_array.sum())
    if denominator == 0:
        return float("nan")
    return float(
        100.0
        * np.abs(predicted_array - actual_array).sum()
        / denominator
    )


def metric_record(actual, predicted, evaluation_level: str) -> dict:
    actual_array = np.asarray(actual, dtype=float)
    predicted_array = np.asarray(predicted, dtype=float)
    errors = predicted_array - actual_array
    actual_total = float(actual_array.sum())
    total_bias = float(errors.sum())
    return {
        "EvaluationLevel": evaluation_level,
        "Observations": int(len(actual_array)),
        "ActualTotal": actual_total,
        "PredictedTotal": float(predicted_array.sum()),
        "WAPEPercentage": safe_wape(actual_array, predicted_array),
        "MAE": float(np.abs(errors).mean()),
        "RMSE": float(np.sqrt(np.square(errors).mean())),
        "MeanBias": float(errors.mean()),
        "TotalBias": total_bias,
        "AbsoluteBiasPercentage": (
            float(100.0 * abs(total_bias) / actual_total)
            if actual_total != 0
            else float("nan")
        ),
    }


def save_figure(path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    plt.tight_layout()
    plt.savefig(path, dpi=300, bbox_inches="tight")
    plt.close()


def normalize_family(values: pd.Series) -> pd.Series:
    return values.astype("string").fillna("__MISSING_FAMILY__").astype(str)


# =============================================================================
# FEATURE AND ROUTING HELPERS
# =============================================================================

def historical_features_from_history(
    demand_history: np.ndarray,
    sequence_history: np.ndarray,
    current_sequence: float,
) -> dict[str, float]:
    demand_history = np.asarray(demand_history, dtype=float)
    sequence_history = np.asarray(sequence_history, dtype=float)
    result: dict[str, float] = {}

    for lag in LAG_OPERATING_DAYS:
        result[f"NormalDemandLag_{lag}"] = (
            float(demand_history[-lag])
            if len(demand_history) >= lag
            else float("nan")
        )

    for window in ROLLING_WINDOWS:
        values = demand_history[-window:]
        if len(values) == 0:
            mean_value = median_value = std_value = sum_value = float("nan")
        else:
            mean_value = float(np.mean(values))
            median_value = float(np.median(values))
            std_value = float(np.std(values, ddof=0))
            sum_value = float(np.sum(values))

        result[f"PastNormalDemandRollingMean_{window}"] = mean_value
        result[f"PastNormalDemandRollingMedian_{window}"] = median_value
        result[f"PastNormalDemandRollingStd_{window}"] = std_value
        result[f"PastNormalDemandRollingSum_{window}"] = sum_value

    for window in ZERO_POSITIVE_WINDOWS:
        values = demand_history[-window:]
        if len(values) == 0:
            zero_rate = positive_count = float("nan")
        else:
            zero_rate = float(np.mean(values == 0))
            positive_count = float(np.sum(values > 0))

        result[f"PastZeroNormalDemandRate_{window}"] = zero_rate
        result[f"PastPositiveNormalDemandCount_{window}"] = positive_count

    positive_indices = np.flatnonzero(demand_history > 0)
    if len(positive_indices) == 0:
        days_since_positive = float("nan")
    else:
        last_positive_sequence = float(sequence_history[int(positive_indices[-1])])
        days_since_positive = float(current_sequence - last_positive_sequence)

    result["OperatingDaysSincePreviousPositiveNormalDemand"] = (
        days_since_positive
    )

    if len(demand_history) == 0:
        expanding_mean = float("nan")
        expanding_positive_rate = float("nan")
    else:
        expanding_mean = float(np.mean(demand_history))
        expanding_positive_rate = float(np.mean(demand_history > 0))

    result["ExpandingPastMeanNormalDemand"] = expanding_mean
    result["ExpandingPastPositiveNormalDemandRate"] = expanding_positive_rate
    return result


def build_history_state(training_frame: pd.DataFrame) -> dict[str, dict[str, list]]:
    state: dict[str, dict[str, list]] = {}

    ordered = training_frame.sort_values(
        [PRODUCT_ID_COLUMN, DATE_COLUMN, SEQUENCE_COLUMN],
        kind="mergesort",
    )

    for product_id, product_frame in ordered.groupby(
        PRODUCT_ID_COLUMN,
        sort=False,
    ):
        product_key = str(product_id)
        state[product_key] = {
            "demand": product_frame[TARGET_COLUMN].astype(float).tolist(),
            "sequence": product_frame[SEQUENCE_COLUMN].astype(float).tolist(),
        }

    return state


def copy_history_state(
    base_state: dict[str, dict[str, list]],
) -> dict[str, dict[str, list]]:
    return {
        product_id: {
            "demand": values["demand"].copy(),
            "sequence": values["sequence"].copy(),
        }
        for product_id, values in base_state.items()
    }


def append_predictions_to_state(
    state: dict[str, dict[str, list]],
    rows: pd.DataFrame,
    predictions: np.ndarray,
) -> None:
    for product_id, sequence, prediction in zip(
        rows[PRODUCT_ID_COLUMN].astype(str),
        rows[SEQUENCE_COLUMN].astype(float),
        np.asarray(predictions, dtype=float),
    ):
        if product_id not in state:
            state[product_id] = {"demand": [], "sequence": []}

        if state[product_id]["sequence"]:
            if sequence <= state[product_id]["sequence"][-1]:
                raise AssertionError(
                    f"Non-increasing operating sequence for {product_id}."
                )

        state[product_id]["demand"].append(float(prediction))
        state[product_id]["sequence"].append(float(sequence))


def build_recursive_feature_frame(
    score_rows: pd.DataFrame,
    state: dict[str, dict[str, list]],
) -> pd.DataFrame:
    frame = score_rows.copy().reset_index(drop=True)
    feature_records = []
    prior_records = []

    for row in frame.itertuples(index=False):
        product_id = str(getattr(row, PRODUCT_ID_COLUMN))
        current_sequence = float(getattr(row, SEQUENCE_COLUMN))
        product_state = state.get(
            product_id,
            {"demand": [], "sequence": []},
        )

        demand_history = np.asarray(product_state["demand"], dtype=float)
        sequence_history = np.asarray(product_state["sequence"], dtype=float)

        feature_records.append(
            historical_features_from_history(
                demand_history,
                sequence_history,
                current_sequence,
            )
        )

        prior_count = int(len(demand_history))
        prior_cumulative = float(demand_history.sum())
        prior_positive = int(np.sum(demand_history > 0))

        prior_records.append(
            {
                "PriorOperatingDayCount": prior_count,
                "PriorCumulativeNormalDemand": prior_cumulative,
                "PriorPositiveNormalDemandDays": prior_positive,
                "PriorZeroNormalDemandDays": prior_count - prior_positive,
                "HasSufficientHistory20": prior_count >= MINIMUM_MAIN_HISTORY,
                "ColdStartFlag": prior_count < MINIMUM_MAIN_HISTORY,
                "ZeroPriorNormalDemandFlag": prior_cumulative <= 0,
            }
        )

    feature_values = pd.DataFrame(feature_records)
    prior_values = pd.DataFrame(prior_records)

    for column in HISTORICAL_DEMAND_PREDICTORS:
        frame[column] = feature_values[column].to_numpy()

    for column in prior_values.columns:
        frame[column] = prior_values[column].to_numpy()

    ranked = frame.sort_values(
        [
            "PriorCumulativeNormalDemand",
            "PriorPositiveNormalDemandDays",
            PRODUCT_ID_COLUMN,
        ],
        ascending=[False, False, True],
        kind="mergesort",
    ).copy()

    universe_total = float(ranked["PriorCumulativeNormalDemand"].sum())

    ranked["PriorDemandRank"] = np.arange(1, len(ranked) + 1, dtype=int)
    ranked["PriorDemandSharePercentage"] = 0.0
    ranked["PriorCumulativeDemandShareBeforePercentage"] = 0.0
    ranked["PriorCumulativeDemandSharePercentage"] = 0.0
    ranked["InPrior95DemandScope"] = False

    if universe_total > 0:
        share = 100.0 * ranked["PriorCumulativeNormalDemand"] / universe_total
        cumulative = share.cumsum()
        cumulative_before = cumulative - share
        ranked["PriorDemandSharePercentage"] = share
        ranked["PriorCumulativeDemandShareBeforePercentage"] = cumulative_before
        ranked["PriorCumulativeDemandSharePercentage"] = cumulative
        ranked["InPrior95DemandScope"] = (
            (ranked["PriorCumulativeNormalDemand"] > 0)
            & (cumulative_before < PRIMARY_SCOPE_PERCENTAGE)
        )

    ranked["EligibleForMainModel"] = (
        ranked["HasSufficientHistory20"]
        & ranked["InPrior95DemandScope"]
        & ~ranked["ZeroPriorNormalDemandFlag"]
    )

    ranked["RecursiveForecastRoute"] = np.select(
        [
            ranked["ZeroPriorNormalDemandFlag"],
            ranked["ColdStartFlag"],
            ranked["EligibleForMainModel"],
        ],
        [
            "ZERO_HISTORY_FALLBACK",
            "COLD_START_FALLBACK",
            "MAIN_MODEL",
        ],
        default="LOW_DEMAND_FALLBACK",
    )

    ranked["PriorDemandVolumeSegment"] = np.select(
        [
            universe_total <= 0,
            ranked["ZeroPriorNormalDemandFlag"],
            ranked["PriorCumulativeDemandShareBeforePercentage"] < 80.0,
            ranked["InPrior95DemandScope"],
        ],
        [
            "NO_PRIOR_DEMAND_UNIVERSE",
            "ZERO_PRIOR_DEMAND",
            "HIGH_DEMAND",
            "MODERATE_DEMAND",
        ],
        default="LOW_DEMAND",
    )

    ranking_columns = [
        PRODUCT_ID_COLUMN,
        "PriorDemandRank",
        "PriorDemandSharePercentage",
        "PriorCumulativeDemandShareBeforePercentage",
        "PriorCumulativeDemandSharePercentage",
        "InPrior95DemandScope",
        "EligibleForMainModel",
        "RecursiveForecastRoute",
        "PriorDemandVolumeSegment",
    ]

    frame = frame.drop(
        columns=[
            column
            for column in ranking_columns[1:]
            if column in frame.columns
        ],
        errors="ignore",
    ).merge(
        ranked[ranking_columns],
        on=PRODUCT_ID_COLUMN,
        how="left",
        validate="one_to_one",
    )

    return frame


# =============================================================================
# FALLBACK AND MODEL HELPERS
# =============================================================================

def build_hierarchy_statistics(training_frame: pd.DataFrame) -> dict:
    train = training_frame.copy()
    train[PRODUCT_ID_COLUMN] = train[PRODUCT_ID_COLUMN].astype(str)
    train["_FamilyKey"] = normalize_family(train[FAMILY_COLUMN])
    train[DAY_OF_WEEK_COLUMN] = pd.to_numeric(
        train[DAY_OF_WEEK_COLUMN],
        errors="raise",
    ).astype(int)

    return {
        "global_mean": float(train[TARGET_COLUMN].mean()),
        "product_weekday": train.groupby(
            [PRODUCT_ID_COLUMN, DAY_OF_WEEK_COLUMN],
            dropna=False,
        )[TARGET_COLUMN].mean(),
        "product_mean": train.groupby(
            PRODUCT_ID_COLUMN,
            dropna=False,
        )[TARGET_COLUMN].mean(),
        "family_weekday": train.groupby(
            ["_FamilyKey", DAY_OF_WEEK_COLUMN],
            dropna=False,
        )[TARGET_COLUMN].mean(),
        "family_mean": train.groupby(
            "_FamilyKey",
            dropna=False,
        )[TARGET_COLUMN].mean(),
        "global_weekday": train.groupby(
            DAY_OF_WEEK_COLUMN,
            dropna=False,
        )[TARGET_COLUMN].mean(),
    }


def hierarchy_predictions(
    score_frame: pd.DataFrame,
    statistics: dict,
) -> tuple[np.ndarray, np.ndarray]:
    score = score_frame.copy()
    product_ids = score[PRODUCT_ID_COLUMN].astype(str)
    families = normalize_family(score[FAMILY_COLUMN])
    weekdays = pd.to_numeric(
        score[DAY_OF_WEEK_COLUMN],
        errors="raise",
    ).astype(int)

    product_weekday_values = np.asarray(
        [
            statistics["product_weekday"].get((product_id, weekday), np.nan)
            for product_id, weekday in zip(product_ids, weekdays)
        ],
        dtype=float,
    )
    product_values = product_ids.map(statistics["product_mean"]).to_numpy(
        dtype=float
    )
    family_weekday_values = np.asarray(
        [
            statistics["family_weekday"].get((family, weekday), np.nan)
            for family, weekday in zip(families, weekdays)
        ],
        dtype=float,
    )
    family_values = families.map(statistics["family_mean"]).to_numpy(dtype=float)
    global_weekday_values = weekdays.map(statistics["global_weekday"]).to_numpy(
        dtype=float
    )
    global_values = np.full(len(score), statistics["global_mean"], dtype=float)
    zero_values = np.zeros(len(score), dtype=float)

    def coalesce(*arrays) -> np.ndarray:
        output = np.full(len(score), np.nan, dtype=float)
        for values in arrays:
            candidate = np.asarray(values, dtype=float)
            missing = ~np.isfinite(output)
            output[missing] = candidate[missing]
        return np.clip(np.nan_to_num(output, nan=0.0), 0.0, None)

    product_hierarchy = coalesce(
        product_weekday_values,
        product_values,
        family_weekday_values,
        family_values,
        global_weekday_values,
        global_values,
        zero_values,
    )

    family_hierarchy = coalesce(
        family_weekday_values,
        family_values,
        global_weekday_values,
        global_values,
        zero_values,
    )

    return product_hierarchy, family_hierarchy


def fallback_predictions(
    feature_frame: pd.DataFrame,
    hierarchy_statistics: dict,
) -> np.ndarray:
    product_hierarchy, family_hierarchy = hierarchy_predictions(
        feature_frame,
        hierarchy_statistics,
    )

    def values(column: str) -> np.ndarray:
        return pd.to_numeric(
            feature_frame[column],
            errors="coerce",
        ).to_numpy(dtype=float)

    def coalesce(*arrays) -> np.ndarray:
        output = np.full(len(feature_frame), np.nan, dtype=float)
        for array in arrays:
            candidate = np.asarray(array, dtype=float)
            missing = ~np.isfinite(output)
            output[missing] = candidate[missing]
        return np.clip(np.nan_to_num(output, nan=0.0), 0.0, None)

    recent_median5 = coalesce(
        values("PastNormalDemandRollingMedian_5"),
        values("PastNormalDemandRollingMedian_3"),
        values("NormalDemandLag_1"),
        values("ExpandingPastMeanNormalDemand"),
        family_hierarchy,
        np.zeros(len(feature_frame)),
    )

    expanding_mean = coalesce(
        values("ExpandingPastMeanNormalDemand"),
        product_hierarchy,
        family_hierarchy,
        np.zeros(len(feature_frame)),
    )

    routes = feature_frame["RecursiveForecastRoute"].astype(str).to_numpy()
    prediction = np.full(len(feature_frame), np.nan, dtype=float)

    median_mask = np.isin(
        routes,
        ["COLD_START_FALLBACK", "LOW_DEMAND_FALLBACK"],
    )
    zero_mask = routes == "ZERO_HISTORY_FALLBACK"

    prediction[median_mask] = recent_median5[median_mask]
    prediction[zero_mask] = expanding_mean[zero_mask]
    return prediction


def make_preprocessor(
    numeric_predictors: list[str],
    categorical_predictors: list[str],
    include_product_id: bool,
) -> ColumnTransformer:
    categorical_columns = list(categorical_predictors)
    if include_product_id:
        categorical_columns.append(PRODUCT_ID_COLUMN)

    return ColumnTransformer(
        transformers=[
            (
                "numeric",
                Pipeline(
                    steps=[
                        ("imputer", SimpleImputer(strategy="median")),
                    ]
                ),
                numeric_predictors,
            ),
            (
                "categorical",
                Pipeline(
                    steps=[
                        ("imputer", SimpleImputer(strategy="most_frequent")),
                        (
                            "ordinal",
                            OrdinalEncoder(
                                handle_unknown="use_encoded_value",
                                unknown_value=-1,
                                encoded_missing_value=-2,
                            ),
                        ),
                    ]
                ),
                categorical_columns,
            ),
        ],
        remainder="drop",
        sparse_threshold=0.0,
        verbose_feature_names_out=False,
    )


def prepare_source_frame(
    frame: pd.DataFrame,
    numeric_predictors: list[str],
    categorical_predictors: list[str],
    include_product_id: bool,
) -> pd.DataFrame:
    columns = (
        numeric_predictors
        + categorical_predictors
        + ([PRODUCT_ID_COLUMN] if include_product_id else [])
    )
    prepared = frame[columns].copy()

    for column in numeric_predictors:
        prepared[column] = pd.to_numeric(prepared[column], errors="coerce")

    for column in categorical_predictors:
        prepared[column] = (
            prepared[column]
            .astype("string")
            .fillna("__MISSING__")
            .astype(str)
        )

    if include_product_id:
        prepared[PRODUCT_ID_COLUMN] = (
            prepared[PRODUCT_ID_COLUMN]
            .astype("string")
            .fillna("__MISSING_PRODUCT__")
            .astype(str)
        )

    return prepared


def instantiate_model(family_method: str, config: dict):
    parameters = {
        key: value
        for key, value in config.items()
        if key != "ConfigID"
    }

    if family_method == "NESTED_TUNED_CATBOOST_RMSE_CORE53":
        return CatBoostRegressor(
            loss_function="RMSE",
            random_seed=RANDOM_SEED,
            verbose=False,
            allow_writing_files=False,
            thread_count=MAX_THREADS,
            **parameters,
        )

    if family_method in {
        "NESTED_TUNED_XGBOOST_SQUARED_CORE53",
        "NESTED_TUNED_XGBOOST_SQUARED_PRODUCT_AWARE",
    }:
        return XGBRegressor(
            objective="reg:squarederror",
            subsample=0.85,
            colsample_bytree=0.85,
            reg_lambda=2.0,
            reg_alpha=0.0,
            n_jobs=MAX_THREADS,
            random_state=RANDOM_SEED,
            verbosity=0,
            **parameters,
        )

    raise ValueError(f"Unsupported family method: {family_method}")


def fit_week_origin_models(
    training_main: pd.DataFrame,
    fold_configurations: dict[str, dict],
    numeric_predictors: list[str],
    categorical_predictors: list[str],
) -> tuple[dict, list[dict]]:
    fitted: dict = {}
    audit_records = []

    core_preprocessor = make_preprocessor(
        numeric_predictors,
        categorical_predictors,
        include_product_id=False,
    )
    X_core = core_preprocessor.fit_transform(
        prepare_source_frame(
            training_main,
            numeric_predictors,
            categorical_predictors,
            include_product_id=False,
        )
    )

    product_preprocessor = make_preprocessor(
        numeric_predictors,
        categorical_predictors,
        include_product_id=True,
    )
    X_product = product_preprocessor.fit_transform(
        prepare_source_frame(
            training_main,
            numeric_predictors,
            categorical_predictors,
            include_product_id=True,
        )
    )

    y_train = training_main[TARGET_COLUMN].to_numpy(dtype=float)

    for family_method, configuration in fold_configurations.items():
        model = instantiate_model(family_method, configuration)
        if family_method == "NESTED_TUNED_XGBOOST_SQUARED_PRODUCT_AWARE":
            X_train = X_product
            preprocessor = product_preprocessor
        else:
            X_train = X_core
            preprocessor = core_preprocessor

        start = time.perf_counter()
        model.fit(X_train, y_train)
        seconds = float(time.perf_counter() - start)
        component = FAMILY_TO_COMPONENT[family_method]
        fitted[component] = {
            "model": model,
            "preprocessor": preprocessor,
            "include_product_id": (
                family_method
                == "NESTED_TUNED_XGBOOST_SQUARED_PRODUCT_AWARE"
            ),
        }
        audit_records.append(
            {
                "CandidateMethod": family_method,
                "Component": component,
                "ConfigID": configuration.get("ConfigID", ""),
                "ConfigJSON": json.dumps(configuration, sort_keys=True),
                "TrainingRows": int(len(training_main)),
                "TransformedFeatureCount": int(X_train.shape[1]),
                "FitSeconds": seconds,
            }
        )

    return fitted, audit_records


def predict_base_components(
    main_features: pd.DataFrame,
    fitted_models: dict,
    numeric_predictors: list[str],
    categorical_predictors: list[str],
) -> dict[str, np.ndarray]:
    components = {
        "ROLLING_MEAN_5": np.clip(
            pd.to_numeric(
                main_features["PastNormalDemandRollingMean_5"],
                errors="coerce",
            ).to_numpy(dtype=float),
            0.0,
            None,
        )
    }

    transformed_cache = {}

    for component, fitted in fitted_models.items():
        cache_key = "PRODUCT" if fitted["include_product_id"] else "CORE"
        if cache_key not in transformed_cache:
            transformed_cache[cache_key] = fitted["preprocessor"].transform(
                prepare_source_frame(
                    main_features,
                    numeric_predictors,
                    categorical_predictors,
                    fitted["include_product_id"],
                )
            )

        prediction = np.asarray(
            fitted["model"].predict(transformed_cache[cache_key]),
            dtype=float,
        )
        components[component] = np.clip(prediction, 0.0, None)

    if set(components) != set(BASE_COMPONENTS):
        raise AssertionError(
            "The recursive base-component prediction set is incomplete."
        )

    return components


# =============================================================================
# AGGREGATION AND SELECTION HELPERS
# =============================================================================

def create_week_aggregates(
    daily_predictions: pd.DataFrame,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    product_week = (
        daily_predictions.groupby(
            [
                "ForecastMode",
                "CandidateMethod",
                "OuterFold",
                "WeekStart",
                PRODUCT_ID_COLUMN,
                PRODUCT_NAME_COLUMN,
            ],
            as_index=False,
        )
        .agg(
            ActualNormalDemand=("ActualNormalDemand", "sum"),
            PredictedNormalDemand=("PredictedNormalDemand", "sum"),
            ProductDateRows=(DATE_COLUMN, "size"),
        )
    )
    product_week["ForecastError"] = (
        product_week["PredictedNormalDemand"]
        - product_week["ActualNormalDemand"]
    )
    product_week["AbsoluteError"] = product_week["ForecastError"].abs()

    restaurant_week = (
        product_week.groupby(
            [
                "ForecastMode",
                "CandidateMethod",
                "OuterFold",
                "WeekStart",
            ],
            as_index=False,
        )
        .agg(
            ActualNormalDemand=("ActualNormalDemand", "sum"),
            PredictedNormalDemand=("PredictedNormalDemand", "sum"),
            Products=(PRODUCT_ID_COLUMN, "nunique"),
        )
    )
    restaurant_week["ForecastError"] = (
        restaurant_week["PredictedNormalDemand"]
        - restaurant_week["ActualNormalDemand"]
    )
    restaurant_week["AbsoluteError"] = restaurant_week["ForecastError"].abs()
    return product_week, restaurant_week


def build_level_metrics(
    daily_predictions: pd.DataFrame,
    product_week: pd.DataFrame,
    restaurant_week: pd.DataFrame,
    mode_label: str,
) -> pd.DataFrame:
    records = []

    for method, frame in daily_predictions.groupby("CandidateMethod", sort=False):
        record = metric_record(
            frame["ActualNormalDemand"],
            frame["PredictedNormalDemand"],
            "DAILY_PRODUCT_ALL_ROUTES",
        )
        record.update({"ForecastMode": mode_label, "CandidateMethod": method})
        records.append(record)

        daily_restaurant = (
            frame.groupby(DATE_COLUMN, as_index=False)
            .agg(
                ActualNormalDemand=("ActualNormalDemand", "sum"),
                PredictedNormalDemand=("PredictedNormalDemand", "sum"),
            )
        )
        record = metric_record(
            daily_restaurant["ActualNormalDemand"],
            daily_restaurant["PredictedNormalDemand"],
            "DAILY_RESTAURANT_TOTAL",
        )
        record.update({"ForecastMode": mode_label, "CandidateMethod": method})
        records.append(record)

    for method, frame in product_week.groupby("CandidateMethod", sort=False):
        record = metric_record(
            frame["ActualNormalDemand"],
            frame["PredictedNormalDemand"],
            "PRODUCT_WEEK",
        )
        record.update({"ForecastMode": mode_label, "CandidateMethod": method})
        records.append(record)

    for method, frame in restaurant_week.groupby("CandidateMethod", sort=False):
        record = metric_record(
            frame["ActualNormalDemand"],
            frame["PredictedNormalDemand"],
            "RESTAURANT_WEEK",
        )
        record.update({"ForecastMode": mode_label, "CandidateMethod": method})
        records.append(record)

    return pd.DataFrame(records)


def qualify_and_select(
    overall_metrics: pd.DataFrame,
    fold_metrics: pd.DataFrame,
    restaurant_metrics: pd.DataFrame,
    selection_context: str,
) -> tuple[pd.DataFrame, str]:
    benchmark = overall_metrics.loc[
        overall_metrics["CandidateMethod"] == "ROLLING_MEAN_5"
    ]
    if len(benchmark) != 1:
        raise AssertionError(
            f"{selection_context}: expected one ROLLING_MEAN_5 benchmark row."
        )

    benchmark_row = benchmark.iloc[0]
    benchmark_fold = (
        fold_metrics.loc[
            fold_metrics["CandidateMethod"] == "ROLLING_MEAN_5",
            ["OuterFold", "WAPEPercentage"],
        ]
        .rename(columns={"WAPEPercentage": "BenchmarkFoldWAPEPercentage"})
    )

    candidate_rows = []

    for _, row in overall_metrics.iterrows():
        method = str(row["CandidateMethod"])
        if method == "ROLLING_MEAN_5":
            candidate_rows.append(
                {
                    **row.to_dict(),
                    "WAPEImprovementPercentagePoints": 0.0,
                    "RelativeWAPEImprovementPercentage": 0.0,
                    "MAERatioToBenchmark": 1.0,
                    "RMSERatioToBenchmark": 1.0,
                    "FoldsWonAgainstBenchmark": 0,
                    "StrictlyQualifies": False,
                }
            )
            continue

        comparison = (
            fold_metrics.loc[
                fold_metrics["CandidateMethod"] == method,
                ["OuterFold", "WAPEPercentage"],
            ]
            .merge(
                benchmark_fold,
                on="OuterFold",
                how="left",
                validate="one_to_one",
            )
        )

        folds_won = int(
            (
                comparison["WAPEPercentage"]
                < comparison["BenchmarkFoldWAPEPercentage"]
            ).sum()
        )
        improvement = float(
            benchmark_row["WAPEPercentage"] - row["WAPEPercentage"]
        )
        mae_ratio = float(row["MAE"] / benchmark_row["MAE"])
        rmse_ratio = float(row["RMSE"] / benchmark_row["RMSE"])
        qualifies = bool(
            improvement >= MINIMUM_WAPE_IMPROVEMENT_PP
            and mae_ratio <= MAXIMUM_MAE_RATIO
            and rmse_ratio <= MAXIMUM_RMSE_RATIO
            and row["AbsoluteBiasPercentage"]
            <= MAXIMUM_ABSOLUTE_BIAS_PERCENTAGE
            and folds_won >= MINIMUM_FOLDS_WON
            and len(comparison) == EXPECTED_OUTER_FOLDS
        )

        candidate_rows.append(
            {
                **row.to_dict(),
                "WAPEImprovementPercentagePoints": improvement,
                "RelativeWAPEImprovementPercentage": (
                    100.0 * improvement / benchmark_row["WAPEPercentage"]
                ),
                "MAERatioToBenchmark": mae_ratio,
                "RMSERatioToBenchmark": rmse_ratio,
                "FoldsWonAgainstBenchmark": folds_won,
                "StrictlyQualifies": qualifies,
            }
        )

    decision = pd.DataFrame(candidate_rows)
    restaurant_lookup = restaurant_metrics.set_index("CandidateMethod")[
        "WAPEPercentage"
    ].to_dict()
    decision["RestaurantWAPEPercentage"] = decision["CandidateMethod"].map(
        restaurant_lookup
    )
    decision["SelectionContext"] = selection_context

    qualified = decision.loc[decision["StrictlyQualifies"]].copy()
    if qualified.empty:
        selected_method = "ROLLING_MEAN_5"
    else:
        selected_method = str(
            qualified.sort_values(
                [
                    "WAPEPercentage",
                    "RestaurantWAPEPercentage",
                    "RMSE",
                    "AbsoluteBiasPercentage",
                ]
            ).iloc[0]["CandidateMethod"]
        )

    decision["Selected"] = decision["CandidateMethod"] == selected_method
    return decision, selected_method


def score_by_method(frame: pd.DataFrame, evaluation_level: str) -> pd.DataFrame:
    records = []
    for method, method_frame in frame.groupby("CandidateMethod", sort=False):
        record = metric_record(
            method_frame["ActualNormalDemand"],
            method_frame["PredictedNormalDemand"],
            evaluation_level,
        )
        record["CandidateMethod"] = method
        records.append(record)
    return pd.DataFrame(records)


def score_by_method_and_fold(
    frame: pd.DataFrame,
    evaluation_level: str,
) -> pd.DataFrame:
    records = []
    for (method, fold), fold_frame in frame.groupby(
        ["CandidateMethod", "OuterFold"],
        sort=True,
    ):
        record = metric_record(
            fold_frame["ActualNormalDemand"],
            fold_frame["PredictedNormalDemand"],
            evaluation_level,
        )
        record.update({"CandidateMethod": method, "OuterFold": int(fold)})
        records.append(record)
    return pd.DataFrame(records)


# =============================================================================
# PREFLIGHT
# =============================================================================

required_inputs = [
    ALL_ROUTES_PATH,
    CORE_PREDICTOR_LIST_PATH,
    ND03_MANIFEST_PATH,
    FOLD_DEFINITION_PATH,
    ND04_SELECTED_METHODS_PATH,
    ND04_ROUTED_PREDICTIONS_PATH,
    ND04_MANIFEST_PATH,
    ND07_OUTER_SELECTIONS_PATH,
    ND07_NESTED_SPLIT_AUDIT_PATH,
    ND07_MANIFEST_PATH,
    ND07E_OUTER_SELECTIONS_PATH,
    ND07E_OUTER_PREDICTIONS_PATH,
    ND07E_FINAL_SELECTION_PATH,
    ND07E_MANIFEST_PATH,
    AGENTS_PATH,
    CURRENT_HANDOFF_PATH,
    WORKFLOW_PATH,
    DECISIONS_PATH,
    METRICS_AND_RESULTS_PATH,
    *CHECKPOINT_PATHS.values(),
]

missing_inputs = [path for path in required_inputs if not path.is_file()]
if missing_inputs:
    raise FileNotFoundError(
        "ND08 required inputs are missing:\n"
        + "\n".join(f"- {path}" for path in missing_inputs)
    )

checkpoint_hashes = {
    step: sha256_file(path)
    for step, path in CHECKPOINT_PATHS.items()
}

for step, expected_hash in EXPECTED_CHECKPOINT_HASHES.items():
    if checkpoint_hashes[step] != expected_hash:
        raise AssertionError(
            f"{step} checkpoint hash mismatch.\n"
            f"Expected: {expected_hash}\n"
            f"Actual:   {checkpoint_hashes[step]}"
        )

if ND08_ROOT.exists():
    if not ALLOW_OVERWRITE:
        raise FileExistsError(
            "ND08 output already exists. No files were changed:\n"
            f"{ND08_ROOT}"
        )
    shutil.rmtree(ND08_ROOT)

for path in [TOP_LEVEL_CHECKPOINT_PATH, TOP_LEVEL_CHECKPOINT_SHA_PATH]:
    if path.exists():
        if not ALLOW_OVERWRITE:
            raise FileExistsError(
                "An ND08 top-level checkpoint already exists. "
                f"No files were changed:\n{path}"
            )
        path.unlink()

STAGING_ROOT = ND08_ROOT.parent / f".ND08_staging_{uuid.uuid4().hex}"
STAGING_ROOT.mkdir(parents=True, exist_ok=False)


# =============================================================================
# LOAD, VERIFY, AND BUILD THE COMPLETE-WEEK REGISTRY
# =============================================================================

try:
    all_routes = pd.read_csv(ALL_ROUTES_PATH, low_memory=False)
    predictor_register = pd.read_csv(
        CORE_PREDICTOR_LIST_PATH,
        low_memory=False,
    )
    fold_definition = pd.read_csv(FOLD_DEFINITION_PATH, low_memory=False)
    selected_route_methods = pd.read_csv(
        ND04_SELECTED_METHODS_PATH,
        low_memory=False,
    )
    nd04_routed_predictions = pd.read_csv(
        ND04_ROUTED_PREDICTIONS_PATH,
        low_memory=False,
    )
    nd07_outer_selections = pd.read_csv(
        ND07_OUTER_SELECTIONS_PATH,
        low_memory=False,
    )
    nd07_nested_split_audit = pd.read_csv(
        ND07_NESTED_SPLIT_AUDIT_PATH,
        low_memory=False,
    )
    nd07e_outer_selections = pd.read_csv(
        ND07E_OUTER_SELECTIONS_PATH,
        low_memory=False,
    )
    nd07e_outer_predictions = pd.read_csv(
        ND07E_OUTER_PREDICTIONS_PATH,
        low_memory=False,
    )
    nd07e_final_selection = pd.read_csv(
        ND07E_FINAL_SELECTION_PATH,
        low_memory=False,
    )

    # ND07E's authoritative output schema stores both weighted settings and
    # median subsets in `SettingKey`. Earlier development drafts used separate
    # `WeightTuple` and `MedianSubset` columns. Normalise either schema into one
    # in-memory compatibility field without modifying the source CSV.
    required_nd07e_selection_columns = {
        "OuterFold",
        "CandidateMethod",
        "EnsembleType",
        "ActiveComponents",
        "SelectedInnerWAPEPercentage",
        *{f"Weight_{component}" for component in BASE_COMPONENTS},
    }
    missing_nd07e_selection_columns = sorted(
        required_nd07e_selection_columns
        - set(nd07e_outer_selections.columns)
    )
    if missing_nd07e_selection_columns:
        raise AssertionError(
            "ND07E outer-selection file is missing required columns:\n"
            + "\n".join(
                f"- {column}"
                for column in missing_nd07e_selection_columns
            )
        )

    if "SettingKey" in nd07e_outer_selections.columns:
        nd07e_outer_selections["EnsembleSettingKey"] = (
            nd07e_outer_selections["SettingKey"].astype(str)
        )
        nd07e_selection_source_schema = "SETTING_KEY"
    elif {"MedianSubset", "WeightTuple"}.issubset(
        nd07e_outer_selections.columns
    ):
        nd07e_outer_selections["EnsembleSettingKey"] = np.where(
            nd07e_outer_selections["EnsembleType"].astype(str) == "MEDIAN",
            nd07e_outer_selections["MedianSubset"].astype(str),
            nd07e_outer_selections["WeightTuple"].astype(str),
        )
        nd07e_selection_source_schema = "LEGACY_SPLIT_COLUMNS"
    else:
        raise AssertionError(
            "ND07E outer-selection file has no supported ensemble-setting "
            "column. Expected `SettingKey`, or both `MedianSubset` and "
            "`WeightTuple`."
        )

    for frame in [all_routes, nd04_routed_predictions, nd07e_outer_predictions]:
        frame[DATE_COLUMN] = pd.to_datetime(frame[DATE_COLUMN], errors="raise")

    for column in [
        "TrainStart",
        "TrainEnd",
        "ValidationStart",
        "ValidationEnd",
    ]:
        fold_definition[column] = pd.to_datetime(
            fold_definition[column],
            errors="raise",
        )

    validate_required_columns(
        all_routes,
        {
            DATE_COLUMN,
            PRODUCT_ID_COLUMN,
            PRODUCT_NAME_COLUMN,
            TARGET_COLUMN,
            ROUTE_COLUMN,
            FAMILY_COLUMN,
            DAY_OF_WEEK_COLUMN,
            SEQUENCE_COLUMN,
            "IsOpenedMarchDiagnosticPeriod",
        },
        "ND03 pre-March all-route dataset",
    )

    if len(all_routes) != EXPECTED_ALL_ROUTE_ROWS:
        raise AssertionError(
            f"Expected {EXPECTED_ALL_ROUTE_ROWS:,} all-route rows; "
            f"found {len(all_routes):,}."
        )

    if all_routes["IsOpenedMarchDiagnosticPeriod"].astype(bool).any():
        raise AssertionError("Opened March rows found in ND08 input data.")

    all_routes[PRODUCT_ID_COLUMN] = all_routes[PRODUCT_ID_COLUMN].astype(str)
    all_routes[TARGET_COLUMN] = pd.to_numeric(
        all_routes[TARGET_COLUMN],
        errors="raise",
    ).astype(float)
    all_routes[SEQUENCE_COLUMN] = pd.to_numeric(
        all_routes[SEQUENCE_COLUMN],
        errors="raise",
    ).astype(float)

    predictor_register = predictor_register.sort_values(
        "PredictorOrder"
    ).reset_index(drop=True)
    core_predictors = predictor_register["Predictor"].astype(str).tolist()
    numeric_predictors = (
        predictor_register.loc[
            predictor_register["PredictorType"].astype(str) == "NUMERIC",
            "Predictor",
        ]
        .astype(str)
        .tolist()
    )
    categorical_predictors = (
        predictor_register.loc[
            predictor_register["PredictorType"].astype(str) == "CATEGORICAL",
            "Predictor",
        ]
        .astype(str)
        .tolist()
    )

    if len(core_predictors) != EXPECTED_CORE_PREDICTORS:
        raise AssertionError("Unexpected core predictor count.")
    if len(numeric_predictors) != EXPECTED_NUMERIC_PREDICTORS:
        raise AssertionError("Unexpected numeric predictor count.")
    if len(categorical_predictors) != EXPECTED_CATEGORICAL_PREDICTORS:
        raise AssertionError("Unexpected categorical predictor count.")
    if set(HISTORICAL_DEMAND_PREDICTORS) - set(core_predictors):
        raise AssertionError("The recursive historical feature contract is incomplete.")

    route_method_map = dict(
        zip(
            selected_route_methods["ForecastRoute"].astype(str),
            selected_route_methods["SelectedMethod"].astype(str),
        )
    )
    if route_method_map != EXPECTED_ROUTE_METHODS:
        raise AssertionError(
            "ND04 selected route methods differ from the accepted routing contract.\n"
            f"Expected: {EXPECTED_ROUTE_METHODS}\n"
            f"Actual:   {route_method_map}"
        )

    if len(nd07e_final_selection) != 1:
        raise AssertionError("Expected one ND07E final-selection row.")
    if (
        str(nd07e_final_selection.iloc[0]["CandidateMethod"])
        != EXPECTED_ND07E_SELECTION
    ):
        raise AssertionError("ND07E selected method differs from the accepted result.")

    # Verify critical files against their step manifests.
    manifest_specs = {
        "ND03": (ND03_ROOT, pd.read_csv(ND03_MANIFEST_PATH)),
        "ND04": (ND04_ROOT, pd.read_csv(ND04_MANIFEST_PATH)),
        "ND07": (ND07_ROOT, pd.read_csv(ND07_MANIFEST_PATH)),
        "ND07E": (ND07E_ROOT, pd.read_csv(ND07E_MANIFEST_PATH)),
    }
    manifest_lookups = {
        step: {
            str(row["RelativePath"]): str(row["SHA256"])
            for _, row in manifest.iterrows()
        }
        for step, (_, manifest) in manifest_specs.items()
    }

    hash_input_specs = [
        ("ND03", ALL_ROUTES_PATH),
        ("ND03", CORE_PREDICTOR_LIST_PATH),
        ("ND04", FOLD_DEFINITION_PATH),
        ("ND04", ND04_SELECTED_METHODS_PATH),
        ("ND04", ND04_ROUTED_PREDICTIONS_PATH),
        ("ND07", ND07_OUTER_SELECTIONS_PATH),
        ("ND07", ND07_NESTED_SPLIT_AUDIT_PATH),
        ("ND07E", ND07E_OUTER_SELECTIONS_PATH),
        ("ND07E", ND07E_OUTER_PREDICTIONS_PATH),
        ("ND07E", ND07E_FINAL_SELECTION_PATH),
    ]
    input_hash_records = []

    for step, path in hash_input_specs:
        root = manifest_specs[step][0]
        relative = str(path.relative_to(root))
        expected_hash = manifest_lookups[step].get(relative)
        actual_hash = sha256_file(path)
        if expected_hash is None:
            raise AssertionError(f"{step} manifest does not list {relative}.")
        if actual_hash != expected_hash:
            raise AssertionError(f"{step} input hash mismatch for {relative}.")
        input_hash_records.append(
            {
                "SourceStep": step,
                "InputPath": str(path),
                "RelativePath": relative,
                "SHA256": actual_hash,
                "MatchesManifest": True,
            }
        )

    input_hash_audit = pd.DataFrame(input_hash_records)
    protected_hashes_before = {
        str(path): sha256_file(path)
        for path in required_inputs
        if path.is_file()
    }

    # Build date-to-fold mapping using the exact outer validation intervals.
    date_to_fold: dict[pd.Timestamp, int] = {}
    week_candidates = []

    for fold_row in fold_definition.itertuples(index=False):
        fold = int(fold_row.Fold)
        validation_dates = sorted(
            all_routes.loc[
                all_routes[DATE_COLUMN].between(
                    pd.Timestamp(fold_row.ValidationStart),
                    pd.Timestamp(fold_row.ValidationEnd),
                    inclusive="both",
                ),
                DATE_COLUMN,
            ].drop_duplicates()
        )

        for date_value in validation_dates:
            date_to_fold[pd.Timestamp(date_value)] = fold

        validation_calendar = pd.DataFrame({DATE_COLUMN: validation_dates})
        validation_calendar["WeekStart"] = (
            validation_calendar[DATE_COLUMN]
            - pd.to_timedelta(
                validation_calendar[DATE_COLUMN].dt.dayofweek,
                unit="D",
            )
        )
        validation_calendar["Weekday"] = validation_calendar[
            DATE_COLUMN
        ].dt.dayofweek

        for week_start, week_frame in validation_calendar.groupby(
            "WeekStart",
            sort=True,
        ):
            observed_weekdays = tuple(sorted(set(week_frame["Weekday"])))
            dates = tuple(sorted(week_frame[DATE_COLUMN]))
            is_complete = observed_weekdays == (0, 1, 2, 3, 4) and len(dates) == 5
            week_candidates.append(
                {
                    "OuterFold": fold,
                    "WeekStart": pd.Timestamp(week_start),
                    "WeekEnd": pd.Timestamp(week_start) + pd.Timedelta(days=4),
                    "ObservedDates": len(dates),
                    "ObservedWeekdays": str(observed_weekdays),
                    "IsCompleteMondayToFridayWeek": is_complete,
                    "DatesJSON": json.dumps([date.isoformat() for date in dates]),
                }
            )

    week_candidate_frame = pd.DataFrame(week_candidates)
    week_registry = (
        week_candidate_frame.loc[
            week_candidate_frame["IsCompleteMondayToFridayWeek"]
        ]
        .sort_values(["OuterFold", "WeekStart"])
        .reset_index(drop=True)
    )
    excluded_week_audit = (
        week_candidate_frame.loc[
            ~week_candidate_frame["IsCompleteMondayToFridayWeek"]
        ]
        .sort_values(["OuterFold", "WeekStart"])
        .reset_index(drop=True)
    )

    if week_registry.empty:
        raise AssertionError("No complete Monday-to-Friday evaluation weeks were found.")
    if week_registry["OuterFold"].nunique() != EXPECTED_OUTER_FOLDS:
        raise AssertionError(
            "Every outer fold must contribute at least one complete evaluation week."
        )

    evaluation_dates = set()
    for week_start in week_registry["WeekStart"]:
        evaluation_dates.update(
            pd.date_range(week_start, periods=5, freq="D").tolist()
        )

    # Fold-specific base-model configurations selected in ND07.
    fold_configurations: dict[int, dict[str, dict]] = {}
    for fold, fold_rows in nd07_outer_selections.groupby("OuterFold", sort=True):
        if set(fold_rows["CandidateMethod"].astype(str)) != EXPECTED_FAMILIES:
            raise AssertionError(f"Fold {fold} is missing an ND07 base configuration.")
        fold_configurations[int(fold)] = {
            str(row.CandidateMethod): json.loads(str(row.SelectedConfigJSON))
            for row in fold_rows.itertuples(index=False)
        }

    # Fold-specific ensemble settings selected in ND07E.
    fold_ensemble_settings: dict[int, dict] = {}
    for fold in sorted(week_registry["OuterFold"].unique()):
        fold_rows = nd07e_outer_selections.loc[
            nd07e_outer_selections["OuterFold"] == fold
        ]
        median_row = fold_rows.loc[
            fold_rows["CandidateMethod"] == "NESTED_MEDIAN_ENSEMBLE"
        ]
        weighted_row = fold_rows.loc[
            fold_rows["CandidateMethod"]
            == "NESTED_WEIGHTED_ENSEMBLE_4_MODELS"
        ]
        if len(median_row) != 1 or len(weighted_row) != 1:
            raise AssertionError(
                f"Fold {fold} is missing median or weighted-four settings."
            )

        median_subset = str(median_row.iloc[0]["EnsembleSettingKey"]).split("|")
        weights = {
            component: float(weighted_row.iloc[0][f"Weight_{component}"])
            for component in BASE_COMPONENTS
        }
        if set(median_subset) - set(BASE_COMPONENTS):
            raise AssertionError(f"Fold {fold} median subset is invalid.")
        if not math.isclose(sum(weights.values()), 1.0, abs_tol=1e-12):
            raise AssertionError(f"Fold {fold} weighted ensemble does not sum to one.")
        if any(weight <= 0 for weight in weights.values()):
            raise AssertionError(
                f"Fold {fold} four-model ensemble must use every component."
            )

        fold_ensemble_settings[int(fold)] = {
            "MedianSubset": median_subset,
            "Weighted4Weights": weights,
        }

    # =========================================================================
    # MONDAY-ORIGIN RECURSIVE EVALUATION
    # =========================================================================

    recursive_prediction_parts = []
    recursion_audit_records = []
    route_change_records = []
    week_model_audit_records = []

    total_start = time.perf_counter()

    for week_row in week_registry.itertuples(index=False):
        outer_fold = int(week_row.OuterFold)
        week_start = pd.Timestamp(week_row.WeekStart)
        week_dates = list(pd.date_range(week_start, periods=5, freq="D"))

        training_all = all_routes.loc[
            all_routes[DATE_COLUMN] < week_start
        ].copy()
        training_main = training_all.loc[
            training_all[ROUTE_COLUMN].astype(str) == "MAIN_MODEL"
        ].copy()

        if training_main.empty:
            raise AssertionError(f"Week {week_start.date()} has no main training rows.")

        fitted_models, fit_audit = fit_week_origin_models(
            training_main,
            fold_configurations[outer_fold],
            numeric_predictors,
            categorical_predictors,
        )
        for record in fit_audit:
            record.update(
                {
                    "OuterFold": outer_fold,
                    "WeekStart": week_start,
                    "TrainingEnd": training_all[DATE_COLUMN].max(),
                    "TrainingAllRouteRows": int(len(training_all)),
                }
            )
            week_model_audit_records.append(record)

        hierarchy_statistics = build_hierarchy_statistics(training_all)
        base_state = build_history_state(training_all)
        method_states = {
            method: copy_history_state(base_state)
            for method in METHODS
        }

        for recursive_day_number, forecast_date in enumerate(week_dates, start=1):
            score_rows = (
                all_routes.loc[all_routes[DATE_COLUMN] == forecast_date]
                .copy()
                .sort_values(PRODUCT_ID_COLUMN)
                .reset_index(drop=True)
            )
            if score_rows.empty:
                raise AssertionError(
                    f"Complete week {week_start.date()} is missing {forecast_date.date()}."
                )

            for method in METHODS:
                recursive_frame = build_recursive_feature_frame(
                    score_rows,
                    method_states[method],
                )

                fallback_prediction = fallback_predictions(
                    recursive_frame,
                    hierarchy_statistics,
                )
                prediction = fallback_prediction.copy()
                main_mask = (
                    recursive_frame["RecursiveForecastRoute"].astype(str)
                    == "MAIN_MODEL"
                ).to_numpy()

                component_columns = {
                    component: np.full(len(recursive_frame), np.nan, dtype=float)
                    for component in BASE_COMPONENTS
                }

                if main_mask.any():
                    main_features = recursive_frame.loc[main_mask].copy()
                    components = predict_base_components(
                        main_features,
                        fitted_models,
                        numeric_predictors,
                        categorical_predictors,
                    )
                    for component, component_prediction in components.items():
                        component_columns[component][main_mask] = component_prediction

                    if method == "ROLLING_MEAN_5":
                        main_prediction = components["ROLLING_MEAN_5"]
                    elif method == "NESTED_MEDIAN_ENSEMBLE":
                        subset = fold_ensemble_settings[outer_fold]["MedianSubset"]
                        main_prediction = np.median(
                            np.column_stack([components[name] for name in subset]),
                            axis=1,
                        )
                    elif method == "NESTED_WEIGHTED_ENSEMBLE_4_MODELS":
                        weights = fold_ensemble_settings[outer_fold][
                            "Weighted4Weights"
                        ]
                        main_prediction = sum(
                            weights[name] * components[name]
                            for name in BASE_COMPONENTS
                        )
                    else:
                        raise AssertionError(f"Unsupported method: {method}")

                    prediction[main_mask] = np.clip(main_prediction, 0.0, None)

                if not np.isfinite(prediction).all():
                    raise AssertionError(
                        f"Non-finite recursive prediction for {method} on {forecast_date}."
                    )

                output = pd.DataFrame(
                    {
                        DATE_COLUMN: recursive_frame[DATE_COLUMN].to_numpy(),
                        PRODUCT_ID_COLUMN: recursive_frame[
                            PRODUCT_ID_COLUMN
                        ].astype(str).to_numpy(),
                        PRODUCT_NAME_COLUMN: recursive_frame[
                            PRODUCT_NAME_COLUMN
                        ].astype(str).to_numpy(),
                        FAMILY_COLUMN: recursive_frame[FAMILY_COLUMN].to_numpy(),
                        DAY_OF_WEEK_COLUMN: recursive_frame[
                            DAY_OF_WEEK_COLUMN
                        ].to_numpy(),
                        "OuterFold": outer_fold,
                        "WeekStart": week_start,
                        "ForecastOrigin": week_start,
                        "RecursiveDayNumber": recursive_day_number,
                        "ForecastMode": "MONDAY_ORIGIN_RECURSIVE",
                        "CandidateMethod": method,
                        "ObservedHistoryRoute": recursive_frame[
                            ROUTE_COLUMN
                        ].astype(str).to_numpy(),
                        "RecursiveForecastRoute": recursive_frame[
                            "RecursiveForecastRoute"
                        ].astype(str).to_numpy(),
                        "PriorOperatingDayCount": recursive_frame[
                            "PriorOperatingDayCount"
                        ].to_numpy(),
                        "PriorCumulativeNormalDemand": recursive_frame[
                            "PriorCumulativeNormalDemand"
                        ].to_numpy(),
                        "WithinWeekActualDemandUsed": False,
                        "WithinWeekPredictedDaysAvailable": (
                            recursive_day_number - 1
                        ),
                        "ActualNormalDemand": recursive_frame[
                            TARGET_COLUMN
                        ].astype(float).to_numpy(),
                        "PredictedNormalDemand": prediction,
                    }
                )

                for component in BASE_COMPONENTS:
                    output[f"Component_{component}"] = component_columns[component]

                output["ForecastError"] = (
                    output["PredictedNormalDemand"]
                    - output["ActualNormalDemand"]
                )
                output["AbsoluteError"] = output["ForecastError"].abs()
                recursive_prediction_parts.append(output)

                if recursive_day_number == 1:
                    maximum_difference = 0.0
                    mismatched_features = 0
                    for feature in HISTORICAL_DEMAND_PREDICTORS:
                        expected = pd.to_numeric(
                            score_rows[feature],
                            errors="coerce",
                        ).to_numpy(dtype=float)
                        actual = pd.to_numeric(
                            recursive_frame[feature],
                            errors="coerce",
                        ).to_numpy(dtype=float)
                        comparable = np.isfinite(expected) & np.isfinite(actual)
                        if comparable.any():
                            feature_difference = float(
                                np.max(np.abs(expected[comparable] - actual[comparable]))
                            )
                            maximum_difference = max(
                                maximum_difference,
                                feature_difference,
                            )
                        if not np.array_equal(np.isnan(expected), np.isnan(actual)):
                            mismatched_features += 1

                    route_mismatches = int(
                        (
                            recursive_frame[ROUTE_COLUMN].astype(str)
                            != recursive_frame["RecursiveForecastRoute"].astype(str)
                        ).sum()
                    )
                    recursion_audit_records.append(
                        {
                            "OuterFold": outer_fold,
                            "WeekStart": week_start,
                            "CandidateMethod": method,
                            "MondayRows": int(len(recursive_frame)),
                            "HistoricalFeatureCount": len(
                                HISTORICAL_DEMAND_PREDICTORS
                            ),
                            "MaximumMondayHistoricalFeatureDifference": (
                                maximum_difference
                            ),
                            "MondayNaNPatternMismatchFeatures": mismatched_features,
                            "MondayRouteMismatches": route_mismatches,
                            "MondayReplayPassed": bool(
                                maximum_difference <= 1e-9
                                and mismatched_features == 0
                                and route_mismatches == 0
                            ),
                            "WithinWeekActualDemandUsed": False,
                        }
                    )

                route_change_records.append(
                    {
                        "OuterFold": outer_fold,
                        "WeekStart": week_start,
                        DATE_COLUMN: forecast_date,
                        "RecursiveDayNumber": recursive_day_number,
                        "CandidateMethod": method,
                        "Rows": int(len(recursive_frame)),
                        "RouteChangesVersusActualHistory": int(
                            (
                                recursive_frame[ROUTE_COLUMN].astype(str)
                                != recursive_frame[
                                    "RecursiveForecastRoute"
                                ].astype(str)
                            ).sum()
                        ),
                    }
                )

                append_predictions_to_state(
                    method_states[method],
                    recursive_frame,
                    prediction,
                )

        print(
            f"Week {week_start.date()} | fold {outer_fold}: "
            "completed genuine Monday-origin recursion for all three methods."
        )

    recursive_daily = pd.concat(recursive_prediction_parts, ignore_index=True)
    recursion_audit = pd.DataFrame(recursion_audit_records)
    route_change_audit = pd.DataFrame(route_change_records)
    week_model_audit = pd.DataFrame(week_model_audit_records)

    recursive_product_week, recursive_restaurant_week = create_week_aggregates(
        recursive_daily
    )
    recursive_metrics = build_level_metrics(
        recursive_daily,
        recursive_product_week,
        recursive_restaurant_week,
        "MONDAY_ORIGIN_RECURSIVE",
    )
    recursive_fold_metrics = score_by_method_and_fold(
        recursive_product_week,
        "MONDAY_ORIGIN_PRODUCT_WEEK_BY_FOLD",
    )
    recursive_week_metrics = []
    for (method, fold, week_start), week_frame in recursive_product_week.groupby(
        ["CandidateMethod", "OuterFold", "WeekStart"],
        sort=True,
    ):
        record = metric_record(
            week_frame["ActualNormalDemand"],
            week_frame["PredictedNormalDemand"],
            "MONDAY_ORIGIN_PRODUCT_WEEK_BY_WEEK",
        )
        record.update(
            {
                "CandidateMethod": method,
                "OuterFold": int(fold),
                "WeekStart": week_start,
            }
        )
        recursive_week_metrics.append(record)
    recursive_week_metrics = pd.DataFrame(recursive_week_metrics)

    # =========================================================================
    # DAILY-UPDATED RECONSTRUCTION ON THE IDENTICAL COMPLETE-WEEK REGISTRY
    # =========================================================================

    nd04_routed_predictions[PRODUCT_ID_COLUMN] = nd04_routed_predictions[
        PRODUCT_ID_COLUMN
    ].astype(str)
    nd07e_outer_predictions[PRODUCT_ID_COLUMN] = nd07e_outer_predictions[
        PRODUCT_ID_COLUMN
    ].astype(str)

    complete_week_date_frame = pd.concat(
        [
            pd.DataFrame(
                {
                    DATE_COLUMN: pd.date_range(
                        pd.Timestamp(row.WeekStart),
                        periods=5,
                        freq="D",
                    ),
                    "WeekStart": pd.Timestamp(row.WeekStart),
                    "OuterFold": int(row.OuterFold),
                }
            )
            for row in week_registry.itertuples(index=False)
        ],
        ignore_index=True,
    )

    daily_updated_parts = []

    for method in METHODS:
        source_method = ND07E_METHOD_MAP[method]
        main_predictions = (
            nd07e_outer_predictions.loc[
                nd07e_outer_predictions["CandidateMethod"].astype(str)
                == source_method,
                [
                    DATE_COLUMN,
                    PRODUCT_ID_COLUMN,
                    "OuterFold",
                    "PredictedNormalDemand",
                ],
            ]
            .rename(
                columns={
                    "OuterFold": "Fold",
                    "PredictedNormalDemand": "MainCandidatePrediction",
                }
            )
        )

        routed = nd04_routed_predictions.merge(
            main_predictions,
            on=[DATE_COLUMN, PRODUCT_ID_COLUMN, "Fold"],
            how="left",
            validate="one_to_one",
        )
        main_mask = routed[ROUTE_COLUMN].astype(str) == "MAIN_MODEL"
        if routed.loc[main_mask, "MainCandidatePrediction"].isna().any():
            raise AssertionError(
                f"Daily-updated main predictions are missing for {method}."
            )

        routed.loc[main_mask, "PredictedNormalDemand"] = routed.loc[
            main_mask,
            "MainCandidatePrediction",
        ]
        routed = routed.drop(columns=["MainCandidatePrediction"])
        routed = routed.merge(
            complete_week_date_frame,
            left_on=[DATE_COLUMN, "Fold"],
            right_on=[DATE_COLUMN, "OuterFold"],
            how="inner",
            validate="many_to_one",
        )
        routed["CandidateMethod"] = method
        routed["ForecastMode"] = "DAILY_UPDATED_ACTUAL_HISTORY"
        routed["ActualNormalDemand"] = pd.to_numeric(
            routed["ActualNormalDemand"],
            errors="raise",
        )
        routed["PredictedNormalDemand"] = pd.to_numeric(
            routed["PredictedNormalDemand"],
            errors="raise",
        )
        routed["ForecastError"] = (
            routed["PredictedNormalDemand"] - routed["ActualNormalDemand"]
        )
        routed["AbsoluteError"] = routed["ForecastError"].abs()
        daily_updated_parts.append(routed)

    daily_updated_routed = pd.concat(daily_updated_parts, ignore_index=True)
    daily_updated_product_week, daily_updated_restaurant_week = (
        create_week_aggregates(daily_updated_routed)
    )
    daily_updated_metrics = build_level_metrics(
        daily_updated_routed,
        daily_updated_product_week,
        daily_updated_restaurant_week,
        "DAILY_UPDATED_ACTUAL_HISTORY",
    )
    daily_updated_fold_metrics = score_by_method_and_fold(
        daily_updated_product_week,
        "DAILY_UPDATED_PRODUCT_WEEK_BY_FOLD",
    )

    # Daily main-route metrics use all exact ND07E outer rows, not only complete weeks.
    daily_main_records = []
    daily_main_fold_records = []
    for method in METHODS:
        source_method = ND07E_METHOD_MAP[method]
        frame = nd07e_outer_predictions.loc[
            nd07e_outer_predictions["CandidateMethod"].astype(str)
            == source_method
        ].copy()
        record = metric_record(
            frame["ActualNormalDemand"],
            frame["PredictedNormalDemand"],
            "DAILY_PRODUCT_MAIN_ROUTE_ALL_OUTER_DATES",
        )
        record["CandidateMethod"] = method
        daily_main_records.append(record)

        for fold, fold_frame in frame.groupby("OuterFold", sort=True):
            fold_record = metric_record(
                fold_frame["ActualNormalDemand"],
                fold_frame["PredictedNormalDemand"],
                "DAILY_PRODUCT_MAIN_ROUTE_BY_FOLD",
            )
            fold_record.update(
                {"CandidateMethod": method, "OuterFold": int(fold)}
            )
            daily_main_fold_records.append(fold_record)

    daily_main_metrics = pd.DataFrame(daily_main_records)
    daily_main_fold_metrics = pd.DataFrame(daily_main_fold_records)

    # =========================================================================
    # OPERATIONAL METHOD SELECTIONS
    # =========================================================================

    daily_main_restaurant = daily_main_metrics[[
        "CandidateMethod",
        "WAPEPercentage",
    ]].copy()
    daily_main_restaurant["WAPEPercentage"] = np.nan

    daily_decision, daily_product_method = qualify_and_select(
        daily_main_metrics,
        daily_main_fold_metrics,
        daily_main_restaurant,
        "BEST_DAILY_PRODUCT_METHOD",
    )

    recursive_product_metrics = recursive_metrics.loc[
        recursive_metrics["EvaluationLevel"] == "PRODUCT_WEEK"
    ].copy()
    recursive_restaurant_metrics = recursive_metrics.loc[
        recursive_metrics["EvaluationLevel"] == "RESTAURANT_WEEK"
    ].copy()
    week_start_decision, week_start_method = qualify_and_select(
        recursive_product_metrics,
        recursive_fold_metrics,
        recursive_restaurant_metrics,
        "BEST_MONDAY_ORIGIN_WEEKLY_PLANNING_METHOD",
    )

    daily_updated_product_metrics = daily_updated_metrics.loc[
        daily_updated_metrics["EvaluationLevel"] == "PRODUCT_WEEK"
    ].copy()
    daily_updated_restaurant_metrics = daily_updated_metrics.loc[
        daily_updated_metrics["EvaluationLevel"] == "RESTAURANT_WEEK"
    ].copy()
    rolling_decision, rolling_updated_method = qualify_and_select(
        daily_updated_product_metrics,
        daily_updated_fold_metrics,
        daily_updated_restaurant_metrics,
        "BEST_ROLLING_DAILY_UPDATED_WEEKLY_METHOD",
    )

    operational_selection = pd.concat(
        [daily_decision, week_start_decision, rolling_decision],
        ignore_index=True,
        sort=False,
    )

    best_recursive_restaurant_method = str(
        recursive_restaurant_metrics.sort_values(
            ["WAPEPercentage", "RMSE", "AbsoluteBiasPercentage"]
        ).iloc[0]["CandidateMethod"]
    )
    best_daily_updated_restaurant_method = str(
        daily_updated_restaurant_metrics.sort_values(
            ["WAPEPercentage", "RMSE", "AbsoluteBiasPercentage"]
        ).iloc[0]["CandidateMethod"]
    )

    route_metric_records = []
    for (method, route), route_frame in recursive_daily.groupby(
        ["CandidateMethod", "RecursiveForecastRoute"],
        sort=True,
    ):
        record = metric_record(
            route_frame["ActualNormalDemand"],
            route_frame["PredictedNormalDemand"],
            "MONDAY_ORIGIN_DAILY_PRODUCT_BY_RECURSIVE_ROUTE",
        )
        record.update(
            {
                "CandidateMethod": method,
                "RecursiveForecastRoute": route,
                "Products": int(route_frame[PRODUCT_ID_COLUMN].nunique()),
                "Dates": int(route_frame[DATE_COLUMN].nunique()),
            }
        )
        route_metric_records.append(record)
    route_metrics = pd.DataFrame(route_metric_records)

    # Reconciliation: reconstruct the rolling benchmark directly from the same
    # filtered ND04 rows and verify it matches the ND08 daily-updated output.
    direct_rolling = daily_updated_routed.loc[
        daily_updated_routed["CandidateMethod"] == "ROLLING_MEAN_5"
    ]
    direct_product_week, direct_restaurant_week = create_week_aggregates(
        direct_rolling
    )
    recorded_product_week = daily_updated_product_week.loc[
        daily_updated_product_week["CandidateMethod"] == "ROLLING_MEAN_5"
    ]
    recorded_restaurant_week = daily_updated_restaurant_week.loc[
        daily_updated_restaurant_week["CandidateMethod"] == "ROLLING_MEAN_5"
    ]

    aggregation_reconciliation = pd.DataFrame(
        [
            {
                "Check": "Rolling product-week row count",
                "Direct": int(len(direct_product_week)),
                "Recorded": int(len(recorded_product_week)),
                "Difference": int(len(direct_product_week) - len(recorded_product_week)),
                "Passed": len(direct_product_week) == len(recorded_product_week),
            },
            {
                "Check": "Rolling product-week predicted total",
                "Direct": float(direct_product_week["PredictedNormalDemand"].sum()),
                "Recorded": float(
                    recorded_product_week["PredictedNormalDemand"].sum()
                ),
                "Difference": float(
                    direct_product_week["PredictedNormalDemand"].sum()
                    - recorded_product_week["PredictedNormalDemand"].sum()
                ),
                "Passed": math.isclose(
                    float(direct_product_week["PredictedNormalDemand"].sum()),
                    float(recorded_product_week["PredictedNormalDemand"].sum()),
                    abs_tol=1e-9,
                ),
            },
            {
                "Check": "Rolling restaurant-week row count",
                "Direct": int(len(direct_restaurant_week)),
                "Recorded": int(len(recorded_restaurant_week)),
                "Difference": int(
                    len(direct_restaurant_week) - len(recorded_restaurant_week)
                ),
                "Passed": len(direct_restaurant_week)
                == len(recorded_restaurant_week),
            },
        ]
    )

    # =========================================================================
    # VALIDATION AND AUDITS
    # =========================================================================

    package_versions = pd.DataFrame(
        [
            {"Package": "python", "Version": platform.python_version()},
            {"Package": "pandas", "Version": pd.__version__},
            {"Package": "numpy", "Version": np.__version__},
            {"Package": "scikit-learn", "Version": sklearn.__version__},
            {"Package": "xgboost", "Version": xgboost.__version__},
            {"Package": "catboost", "Version": catboost.__version__},
        ]
    )

    expected_recursive_rows_per_method = int(
        all_routes[DATE_COLUMN].isin(evaluation_dates).sum()
    )
    recursive_counts = recursive_daily.groupby("CandidateMethod").size()
    actual_totals_by_method = recursive_daily.groupby("CandidateMethod")[
        "ActualNormalDemand"
    ].sum()

    protocol_audit = pd.DataFrame(
        [
            {
                "Check": "ND07E ensemble-setting schema normalised",
                "Expected": True,
                "Actual": bool(
                    nd07e_outer_selections["EnsembleSettingKey"]
                    .astype(str)
                    .str.len()
                    .gt(0)
                    .all()
                ),
                "Passed": bool(
                    nd07e_outer_selections["EnsembleSettingKey"]
                    .astype(str)
                    .str.len()
                    .gt(0)
                    .all()
                ),
            },
            {
                "Check": "Monday-origin forecasts use within-week actual demand",
                "Expected": False,
                "Actual": bool(recursive_daily["WithinWeekActualDemandUsed"].any()),
                "Passed": not bool(
                    recursive_daily["WithinWeekActualDemandUsed"].any()
                ),
            },
            {
                "Check": "Monday historical feature replay",
                "Expected": True,
                "Actual": bool(recursion_audit["MondayReplayPassed"].all()),
                "Passed": bool(recursion_audit["MondayReplayPassed"].all()),
            },
            {
                "Check": "Base-model fitting ends before week origin",
                "Expected": True,
                "Actual": bool(
                    (
                        pd.to_datetime(week_model_audit["TrainingEnd"])
                        < pd.to_datetime(week_model_audit["WeekStart"])
                    ).all()
                ),
                "Passed": bool(
                    (
                        pd.to_datetime(week_model_audit["TrainingEnd"])
                        < pd.to_datetime(week_model_audit["WeekStart"])
                    ).all()
                ),
            },
            {
                "Check": "Ensemble settings selected before outer evaluation",
                "Expected": True,
                "Actual": True,
                "Passed": True,
            },
            {
                "Check": "March target vault opened",
                "Expected": False,
                "Actual": False,
                "Passed": True,
            },
            {
                "Check": "Final production model fitted",
                "Expected": False,
                "Actual": False,
                "Passed": True,
            },
            {
                "Check": "Final model artifact saved",
                "Expected": False,
                "Actual": False,
                "Passed": True,
            },
        ]
    )

    validation = pd.DataFrame(
        [
            {
                "Check": "ND07E checkpoint verified",
                "Expected": EXPECTED_CHECKPOINT_HASHES["ND07E"],
                "Actual": checkpoint_hashes["ND07E"],
                "Passed": checkpoint_hashes["ND07E"]
                == EXPECTED_CHECKPOINT_HASHES["ND07E"],
            },
            {
                "Check": "Complete evaluation folds",
                "Expected": EXPECTED_OUTER_FOLDS,
                "Actual": int(week_registry["OuterFold"].nunique()),
                "Passed": int(week_registry["OuterFold"].nunique())
                == EXPECTED_OUTER_FOLDS,
            },
            {
                "Check": "Recursive rows per method",
                "Expected": expected_recursive_rows_per_method,
                "Actual": int(recursive_counts.min()),
                "Passed": bool(
                    (recursive_counts == expected_recursive_rows_per_method).all()
                ),
            },
            {
                "Check": "Actual totals identical across recursive methods",
                "Expected": 1,
                "Actual": int(actual_totals_by_method.nunique()),
                "Passed": int(actual_totals_by_method.nunique()) == 1,
            },
            {
                "Check": "Recursive duplicate keys",
                "Expected": 0,
                "Actual": int(
                    recursive_daily.duplicated(
                        [
                            DATE_COLUMN,
                            PRODUCT_ID_COLUMN,
                            "CandidateMethod",
                        ]
                    ).sum()
                ),
                "Passed": int(
                    recursive_daily.duplicated(
                        [
                            DATE_COLUMN,
                            PRODUCT_ID_COLUMN,
                            "CandidateMethod",
                        ]
                    ).sum()
                )
                == 0,
            },
            {
                "Check": "Non-finite recursive predictions",
                "Expected": 0,
                "Actual": int(
                    (~np.isfinite(recursive_daily["PredictedNormalDemand"])).sum()
                ),
                "Passed": int(
                    (~np.isfinite(recursive_daily["PredictedNormalDemand"])).sum()
                )
                == 0,
            },
            {
                "Check": "Negative recursive predictions",
                "Expected": 0,
                "Actual": int(
                    (recursive_daily["PredictedNormalDemand"] < 0).sum()
                ),
                "Passed": int(
                    (recursive_daily["PredictedNormalDemand"] < 0).sum()
                )
                == 0,
            },
            {
                "Check": "Aggregation reconciliation",
                "Expected": True,
                "Actual": bool(aggregation_reconciliation["Passed"].all()),
                "Passed": bool(aggregation_reconciliation["Passed"].all()),
            },
            {
                "Check": "One daily product method selected",
                "Expected": 1,
                "Actual": int(
                    daily_decision.loc[daily_decision["Selected"]].shape[0]
                ),
                "Passed": int(
                    daily_decision.loc[daily_decision["Selected"]].shape[0]
                )
                == 1,
            },
            {
                "Check": "One Monday-origin planning method selected",
                "Expected": 1,
                "Actual": int(
                    week_start_decision.loc[
                        week_start_decision["Selected"]
                    ].shape[0]
                ),
                "Passed": int(
                    week_start_decision.loc[
                        week_start_decision["Selected"]
                    ].shape[0]
                )
                == 1,
            },
            {
                "Check": "One rolling updated method selected",
                "Expected": 1,
                "Actual": int(
                    rolling_decision.loc[rolling_decision["Selected"]].shape[0]
                ),
                "Passed": int(
                    rolling_decision.loc[rolling_decision["Selected"]].shape[0]
                )
                == 1,
            },
            {
                "Check": "Previous inputs modified",
                "Expected": False,
                "Actual": False,
                "Passed": True,
            },
            {
                "Check": "ND08 step lock created",
                "Expected": False,
                "Actual": False,
                "Passed": True,
            },
        ]
    )

    if not protocol_audit["Passed"].all():
        raise AssertionError(
            "ND08 protocol audit failed:\n"
            + protocol_audit.loc[~protocol_audit["Passed"]].to_string(index=False)
        )
    if not validation["Passed"].all():
        raise AssertionError(
            "ND08 validation failed:\n"
            + validation.loc[~validation["Passed"]].to_string(index=False)
        )

    total_fitting_seconds = float(time.perf_counter() - total_start)

    # =========================================================================
    # REPRESENTATIVE SETTINGS AND OPERATIONAL CONTRACT
    # =========================================================================

    def representative_median_subset() -> list[str]:
        rows = nd07e_outer_selections.loc[
            nd07e_outer_selections["CandidateMethod"]
            == "NESTED_MEDIAN_ENSEMBLE"
        ].copy()
        counts = Counter(rows["EnsembleSettingKey"].astype(str))
        maximum = max(counts.values())
        candidates = [value for value, count in counts.items() if count == maximum]
        mean_scores = rows.groupby("EnsembleSettingKey")[
            "SelectedInnerWAPEPercentage"
        ].mean()
        selected = min(candidates, key=lambda value: (mean_scores[value], value))
        return selected.split("|")

    def representative_weighted4() -> dict[str, float]:
        rows = nd07e_outer_selections.loc[
            nd07e_outer_selections["CandidateMethod"]
            == "NESTED_WEIGHTED_ENSEMBLE_4_MODELS"
        ].copy()
        counts = Counter(rows["EnsembleSettingKey"].astype(str))
        maximum = max(counts.values())
        candidates = [value for value, count in counts.items() if count == maximum]
        mean_scores = rows.groupby("EnsembleSettingKey")[
            "SelectedInnerWAPEPercentage"
        ].mean()
        selected_tuple = min(
            candidates,
            key=lambda value: (mean_scores[value], value),
        )
        selected_row = rows.loc[
            rows["EnsembleSettingKey"].astype(str) == selected_tuple
        ].sort_values("SelectedInnerWAPEPercentage").iloc[0]
        return {
            component: float(selected_row[f"Weight_{component}"])
            for component in BASE_COMPONENTS
        }

    representative_settings = {
        "ROLLING_MEAN_5": {
            "Type": "ROLLING_MEAN",
            "HistoryOperatingDays": 5,
        },
        "NESTED_MEDIAN_ENSEMBLE": {
            "Type": "MEDIAN",
            "Components": representative_median_subset(),
        },
        "NESTED_WEIGHTED_ENSEMBLE_4_MODELS": {
            "Type": "WEIGHTED_MEAN",
            "Weights": representative_weighted4(),
        },
    }

    system_contract = {
        "StepID": STEP_ID,
        "Status": STATUS,
        "DailyProductMethod": daily_product_method,
        "MondayOriginWeeklyPlanningMethod": week_start_method,
        "RollingDailyUpdatedWeeklyMethod": rolling_updated_method,
        "BestMondayOriginRestaurantWeekMethod": (
            best_recursive_restaurant_method
        ),
        "BestDailyUpdatedRestaurantWeekMethod": (
            best_daily_updated_restaurant_method
        ),
        "MethodsCompared": METHODS,
        "RepresentativeSettings": representative_settings,
        "RecursiveForecastProtocol": {
            "ForecastOrigin": "Before Monday",
            "DaysForecast": ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday"],
            "WithinWeekActualDemandUsed": False,
            "HistoricalFeaturesRecomputedDaily": True,
            "DynamicRoutesRecomputedDaily": True,
            "PredictionsInsertedIntoTemporaryHistory": True,
        },
        "DailyUpdatedProtocol": {
            "ActualHistoryAvailableBeforeEachDay": True,
            "WeekRegistryIdenticalToMondayOriginEvaluation": True,
        },
        "March2026Used": False,
        "FinalProductionModelFitted": False,
        "FinalUnbiasedFutureEvaluationStillRequired": True,
        "NextStep": "ND09",
    }

    contract_markdown = f"""# ND08 Operational Method Contract

## Selected methods

- Best daily product method: `{daily_product_method}`
- Best Monday-origin weekly planning method: `{week_start_method}`
- Best rolling daily-updated weekly method: `{rolling_updated_method}`
- Best Monday-origin restaurant-week method: `{best_recursive_restaurant_method}`
- Best daily-updated restaurant-week method: `{best_daily_updated_restaurant_method}`

Different methods may be selected for different operating modes because daily product error, week-start product planning error, and aggregate restaurant error are distinct objectives.

## Monday-origin protocol

The full Monday-to-Friday forecast is generated before Monday. No actual demand from inside the forecast week is used. Every predicted day is inserted into temporary history before the following day is forecast. All historical demand features and dynamic routes are recalculated before each forecast day.

## Daily-updated protocol

The daily-updated comparison uses actual demand available before each day. It is evaluated on exactly the same complete weeks as the Monday-origin comparison.

## Restrictions

- March 2026 was not used.
- No final production model was fitted or saved.
- A new untouched future period remains required for final unbiased evaluation.
"""

    # =========================================================================
    # STAGED DIRECTORIES AND FIGURES
    # =========================================================================

    staged_prediction_dir = STAGING_ROOT / PREDICTION_DIR.relative_to(ND08_ROOT)
    staged_metric_dir = STAGING_ROOT / METRIC_DIR.relative_to(ND08_ROOT)
    staged_audit_dir = STAGING_ROOT / AUDIT_DIR.relative_to(ND08_ROOT)
    staged_contract_dir = STAGING_ROOT / CONTRACT_DIR.relative_to(ND08_ROOT)
    staged_figure_dir = STAGING_ROOT / FIGURE_DIR.relative_to(ND08_ROOT)
    staged_report_dir = STAGING_ROOT / REPORT_DIR.relative_to(ND08_ROOT)
    staged_control_dir = STAGING_ROOT / CONTROL_DIR.relative_to(ND08_ROOT)

    recursive_product_metric_plot = recursive_product_metrics.sort_values(
        "WAPEPercentage"
    )
    plt.figure(figsize=(10, 6))
    plt.bar(
        recursive_product_metric_plot["CandidateMethod"],
        recursive_product_metric_plot["WAPEPercentage"],
    )
    plt.title("Monday-origin recursive product-week WAPE")
    plt.xlabel("Method")
    plt.ylabel("WAPE (%)")
    plt.xticks(rotation=25, ha="right")
    plt.grid(axis="y", alpha=0.3)
    save_figure(staged_figure_dir / "ND08_figure_01_recursive_product_week_wape.png")

    plt.figure(figsize=(10, 6))
    plt.bar(
        recursive_restaurant_metrics["CandidateMethod"],
        recursive_restaurant_metrics["WAPEPercentage"],
    )
    plt.title("Monday-origin recursive restaurant-week WAPE")
    plt.xlabel("Method")
    plt.ylabel("WAPE (%)")
    plt.xticks(rotation=25, ha="right")
    plt.grid(axis="y", alpha=0.3)
    save_figure(staged_figure_dir / "ND08_figure_02_recursive_restaurant_week_wape.png")

    plt.figure(figsize=(10, 6))
    plt.bar(
        daily_updated_product_metrics["CandidateMethod"],
        daily_updated_product_metrics["WAPEPercentage"],
    )
    plt.title("Daily-updated product-week WAPE on identical weeks")
    plt.xlabel("Method")
    plt.ylabel("WAPE (%)")
    plt.xticks(rotation=25, ha="right")
    plt.grid(axis="y", alpha=0.3)
    save_figure(staged_figure_dir / "ND08_figure_03_daily_updated_product_week_wape.png")

    plt.figure(figsize=(10, 6))
    plt.bar(
        daily_main_metrics["CandidateMethod"],
        daily_main_metrics["WAPEPercentage"],
    )
    plt.title("Daily main-route WAPE")
    plt.xlabel("Method")
    plt.ylabel("WAPE (%)")
    plt.xticks(rotation=25, ha="right")
    plt.grid(axis="y", alpha=0.3)
    save_figure(staged_figure_dir / "ND08_figure_04_daily_main_wape.png")

    comparison_plot = pd.concat(
        [
            recursive_product_metrics[["CandidateMethod", "WAPEPercentage"]].assign(
                ForecastMode="Monday-origin"
            ),
            daily_updated_product_metrics[[
                "CandidateMethod",
                "WAPEPercentage",
            ]].assign(ForecastMode="Daily-updated"),
        ],
        ignore_index=True,
    )
    pivot = comparison_plot.pivot(
        index="CandidateMethod",
        columns="ForecastMode",
        values="WAPEPercentage",
    )
    plt.figure(figsize=(11, 6))
    x = np.arange(len(pivot.index))
    width = 0.35
    plt.bar(x - width / 2, pivot["Monday-origin"], width, label="Monday-origin")
    plt.bar(x + width / 2, pivot["Daily-updated"], width, label="Daily-updated")
    plt.xticks(x, pivot.index, rotation=25, ha="right")
    plt.ylabel("Product-week WAPE (%)")
    plt.title("Monday-origin versus daily-updated product-week WAPE")
    plt.grid(axis="y", alpha=0.3)
    plt.legend()
    save_figure(staged_figure_dir / "ND08_figure_05_origin_vs_updated_wape.png")

    plt.figure(figsize=(13, 6))
    for method, frame in recursive_week_metrics.groupby("CandidateMethod"):
        plt.plot(
            frame["WeekStart"],
            frame["WAPEPercentage"],
            marker="o",
            label=method,
        )
    plt.title("Monday-origin product-week WAPE by week")
    plt.xlabel("Week starting")
    plt.ylabel("WAPE (%)")
    plt.gca().xaxis.set_major_formatter(DateFormatter("%Y-%m-%d"))
    plt.xticks(rotation=45, ha="right")
    plt.grid(alpha=0.3)
    plt.legend(fontsize=8)
    save_figure(staged_figure_dir / "ND08_figure_06_recursive_week_wape_series.png")

    selected_restaurant = recursive_restaurant_week.loc[
        recursive_restaurant_week["CandidateMethod"] == week_start_method
    ].sort_values("WeekStart")
    plt.figure(figsize=(13, 6))
    plt.plot(
        selected_restaurant["WeekStart"],
        selected_restaurant["ActualNormalDemand"],
        marker="o",
        label="Actual",
    )
    plt.plot(
        selected_restaurant["WeekStart"],
        selected_restaurant["PredictedNormalDemand"],
        marker="o",
        label="Predicted",
    )
    plt.title(f"Selected Monday-origin method: restaurant-week totals")
    plt.xlabel("Week starting")
    plt.ylabel("Normal-demand units")
    plt.gca().xaxis.set_major_formatter(DateFormatter("%Y-%m-%d"))
    plt.xticks(rotation=45, ha="right")
    plt.grid(alpha=0.3)
    plt.legend()
    save_figure(staged_figure_dir / "ND08_figure_07_selected_restaurant_week.png")

    selection_summary = pd.DataFrame(
        [
            {
                "OperatingObjective": "Daily product",
                "SelectedMethod": daily_product_method,
            },
            {
                "OperatingObjective": "Monday-origin weekly planning",
                "SelectedMethod": week_start_method,
            },
            {
                "OperatingObjective": "Rolling daily-updated week",
                "SelectedMethod": rolling_updated_method,
            },
        ]
    )
    selection_counts = selection_summary.groupby("SelectedMethod").size()
    plt.figure(figsize=(9, 5))
    plt.bar(selection_counts.index, selection_counts.values)
    plt.title("Number of operating objectives assigned to each method")
    plt.xlabel("Method")
    plt.ylabel("Objectives selected")
    plt.xticks(rotation=25, ha="right")
    plt.grid(axis="y", alpha=0.3)
    save_figure(staged_figure_dir / "ND08_figure_08_operational_selections.png")

    # =========================================================================
    # REPORTS
    # =========================================================================

    def metric_lookup(
        metrics: pd.DataFrame,
        method: str,
        level: str,
    ) -> pd.Series:
        row = metrics.loc[
            (metrics["CandidateMethod"] == method)
            & (metrics["EvaluationLevel"] == level)
        ]
        if len(row) != 1:
            raise AssertionError(f"Metric lookup failed for {method} / {level}.")
        return row.iloc[0]

    selected_origin_product = metric_lookup(
        recursive_metrics,
        week_start_method,
        "PRODUCT_WEEK",
    )
    selected_origin_restaurant = metric_lookup(
        recursive_metrics,
        week_start_method,
        "RESTAURANT_WEEK",
    )
    selected_updated_product = metric_lookup(
        daily_updated_metrics,
        rolling_updated_method,
        "PRODUCT_WEEK",
    )
    selected_updated_restaurant = metric_lookup(
        daily_updated_metrics,
        rolling_updated_method,
        "RESTAURANT_WEEK",
    )

    report_text = f"""# ND08 Genuine Monday-Origin Recursive Weekly System

## Status

`{STATUS}`

## Evaluation scope

- Complete Monday-to-Friday weeks: {len(week_registry)}
- Outer chronological folds represented: {week_registry['OuterFold'].nunique()}
- March 2026 used: no
- Methods compared: {', '.join(METHODS)}

The week registry contains only weeks fully contained inside one outer validation fold and containing all five Monday-to-Friday operating dates. The same registry is used for the Monday-origin and daily-updated comparisons.

## Selected operating methods

- Best daily product method: `{daily_product_method}`
- Best Monday-origin weekly planning method: `{week_start_method}`
- Best rolling daily-updated weekly method: `{rolling_updated_method}`
- Best Monday-origin restaurant-week method: `{best_recursive_restaurant_method}`
- Best daily-updated restaurant-week method: `{best_daily_updated_restaurant_method}`

## Selected Monday-origin planning performance

- Product-week WAPE: {selected_origin_product['WAPEPercentage']:.6f}%
- Product-week MAE: {selected_origin_product['MAE']:.6f}
- Product-week RMSE: {selected_origin_product['RMSE']:.6f}
- Product-week total bias: {selected_origin_product['TotalBias']:.6f}
- Restaurant-week WAPE: {selected_origin_restaurant['WAPEPercentage']:.6f}%
- Restaurant-week total bias: {selected_origin_restaurant['TotalBias']:.6f}

## Selected rolling daily-updated performance

- Product-week WAPE: {selected_updated_product['WAPEPercentage']:.6f}%
- Product-week MAE: {selected_updated_product['MAE']:.6f}
- Product-week RMSE: {selected_updated_product['RMSE']:.6f}
- Product-week total bias: {selected_updated_product['TotalBias']:.6f}
- Restaurant-week WAPE: {selected_updated_restaurant['WAPEPercentage']:.6f}%
- Restaurant-week total bias: {selected_updated_restaurant['TotalBias']:.6f}

## Recursive protocol

Every week is forecast from an origin before Monday. Monday's forecast is inserted into method-specific temporary history before Tuesday is generated. The process continues through Friday. All 31 historical demand features and the dynamic forecast route are recomputed before every day. Actual demand from inside the week is never used.

## Interpretation

Daily product forecasting, initial week planning, rolling week updates, and restaurant-total forecasting are different objectives. ND08 therefore permits different methods to be selected for each objective when the chronological evidence supports that distinction.

## Safety

- March target vault opened: no
- Same-week actual demand used in Monday-origin forecast: no
- Final production model fitted: no
- Final model artifact saved: no
- Existing inputs modified: no
- ND08 step lock created: no
"""

    decision_payload = {
        "StepID": STEP_ID,
        "Status": STATUS,
        "CreatedUTC": NOW_UTC.isoformat(),
        "CompleteWeeks": int(len(week_registry)),
        "MethodsCompared": METHODS,
        "Selections": {
            "DailyProductMethod": daily_product_method,
            "MondayOriginWeeklyPlanningMethod": week_start_method,
            "RollingDailyUpdatedWeeklyMethod": rolling_updated_method,
            "BestMondayOriginRestaurantWeekMethod": (
                best_recursive_restaurant_method
            ),
            "BestDailyUpdatedRestaurantWeekMethod": (
                best_daily_updated_restaurant_method
            ),
        },
        "MondayOriginSelectedMetrics": selected_origin_product.to_dict(),
        "DailyUpdatedSelectedMetrics": selected_updated_product.to_dict(),
        "March2026Used": False,
        "WithinWeekActualDemandUsedForMondayOrigin": False,
        "FinalProductionModelFitted": False,
        "FinalUnbiasedFutureEvaluationStillRequired": True,
        "NextStep": "ND09",
    }

    readme_text = f"""# ND08 Genuine Monday-Origin Recursive Weekly System

Status: `{STATUS}`

Methods compared:

- ROLLING_MEAN_5
- NESTED_MEDIAN_ENSEMBLE
- NESTED_WEIGHTED_ENSEMBLE_4_MODELS

The Monday-origin evaluation forecasts Monday through Friday recursively without using actual demand from inside the week. Dynamic routes and historical demand features are recalculated after each predicted day.

Selected daily product method: `{daily_product_method}`
Selected Monday-origin planning method: `{week_start_method}`
Selected rolling daily-updated method: `{rolling_updated_method}`

March 2026 remained closed. No final production model was fitted or saved.
"""

    # =========================================================================
    # WRITE STAGED OUTPUTS
    # =========================================================================

    output_frames = {
        staged_prediction_dir / RECURSIVE_DAILY_PATH.name: recursive_daily,
        staged_prediction_dir
        / RECURSIVE_PRODUCT_WEEK_PATH.name: recursive_product_week,
        staged_prediction_dir
        / RECURSIVE_RESTAURANT_WEEK_PATH.name: recursive_restaurant_week,
        staged_prediction_dir
        / DAILY_UPDATED_ROUTED_PATH.name: daily_updated_routed,
        staged_prediction_dir
        / DAILY_UPDATED_PRODUCT_WEEK_PATH.name: daily_updated_product_week,
        staged_prediction_dir
        / DAILY_UPDATED_RESTAURANT_WEEK_PATH.name: daily_updated_restaurant_week,
        staged_metric_dir / RECURSIVE_METRICS_PATH.name: recursive_metrics,
        staged_metric_dir
        / RECURSIVE_FOLD_METRICS_PATH.name: recursive_fold_metrics,
        staged_metric_dir
        / RECURSIVE_WEEK_METRICS_PATH.name: recursive_week_metrics,
        staged_metric_dir / DAILY_UPDATED_METRICS_PATH.name: daily_updated_metrics,
        staged_metric_dir
        / DAILY_UPDATED_FOLD_METRICS_PATH.name: daily_updated_fold_metrics,
        staged_metric_dir / DAILY_MAIN_METRICS_PATH.name: daily_main_metrics,
        staged_metric_dir
        / OPERATIONAL_SELECTION_PATH.name: operational_selection,
        staged_metric_dir / ROUTE_METRICS_PATH.name: route_metrics,
        staged_audit_dir / WEEK_REGISTRY_PATH.name: week_registry,
        staged_audit_dir / EXCLUDED_WEEK_AUDIT_PATH.name: excluded_week_audit,
        staged_audit_dir / RECURSION_AUDIT_PATH.name: recursion_audit,
        staged_audit_dir / ROUTE_CHANGE_AUDIT_PATH.name: route_change_audit,
        staged_audit_dir / WEEK_MODEL_AUDIT_PATH.name: week_model_audit,
        staged_audit_dir
        / AGGREGATION_RECONCILIATION_PATH.name: aggregation_reconciliation,
        staged_audit_dir / INPUT_HASH_AUDIT_PATH.name: input_hash_audit,
        staged_audit_dir / PACKAGE_VERSIONS_PATH.name: package_versions,
        staged_audit_dir / PROTOCOL_AUDIT_PATH.name: protocol_audit,
        staged_audit_dir / VALIDATION_PATH.name: validation,
    }

    for path, frame in output_frames.items():
        write_csv(path, frame)

    write_json(
        staged_contract_dir / SYSTEM_CONTRACT_PATH.name,
        system_contract,
    )
    write_text(
        staged_contract_dir / SYSTEM_CONTRACT_MD_PATH.name,
        contract_markdown,
    )
    write_text(
        staged_report_dir / REPORT_SUMMARY_PATH.name,
        report_text,
    )
    write_json(
        staged_report_dir / DECISION_JSON_PATH.name,
        decision_payload,
    )
    write_text(STAGING_ROOT / README_PATH.name, readme_text)

    # =========================================================================
    # INPUT IMMUTABILITY, MANIFEST, AND CHECKPOINT
    # =========================================================================

    protected_hashes_after = {
        str(path): sha256_file(path)
        for path in required_inputs
        if path.is_file()
    }
    changed_inputs = [
        path
        for path in protected_hashes_before
        if protected_hashes_before[path] != protected_hashes_after[path]
    ]
    if changed_inputs:
        raise AssertionError(
            "Protected inputs changed during ND08:\n"
            + "\n".join(f"- {path}" for path in changed_inputs)
        )

    excluded_names = {
        MANIFEST_PATH.name,
        CHECKPOINT_PATH.name,
        CHECKPOINT_SHA_PATH.name,
    }
    files_for_manifest = sorted(
        path
        for path in STAGING_ROOT.rglob("*")
        if path.is_file() and path.name not in excluded_names
    )
    manifest = pd.DataFrame(
        [
            {
                "RelativePath": str(path.relative_to(STAGING_ROOT)),
                "Bytes": int(path.stat().st_size),
                "SHA256": sha256_file(path),
            }
            for path in files_for_manifest
        ]
    ).sort_values("RelativePath").reset_index(drop=True)

    staged_manifest_path = staged_control_dir / MANIFEST_PATH.name
    write_csv(staged_manifest_path, manifest)
    manifest_sha256 = sha256_file(staged_manifest_path)

    checkpoint_payload = {
        "StepID": STEP_ID,
        "Status": STATUS,
        "CreatedUTC": NOW_UTC.isoformat(),
        "CreatedLocal": NOW_LOCAL.isoformat(),
        "ND08Root": str(ND08_ROOT),
        "InputCheckpoints": checkpoint_hashes,
        "CompleteWeeks": int(len(week_registry)),
        "MethodsCompared": METHODS,
        "Selections": decision_payload["Selections"],
        "MondayOriginSelectedMetrics": decision_payload[
            "MondayOriginSelectedMetrics"
        ],
        "DailyUpdatedSelectedMetrics": decision_payload[
            "DailyUpdatedSelectedMetrics"
        ],
        "TotalFittingSeconds": total_fitting_seconds,
        "Manifest": {
            "Path": str(MANIFEST_PATH),
            "SHA256": manifest_sha256,
        },
        "Safety": {
            "MarchTargetVaultOpened": False,
            "WithinWeekActualDemandUsedForMondayOrigin": False,
            "FinalProductionModelFitted": False,
            "FinalModelArtifactSaved": False,
            "PreviousInputsModified": False,
            "ExistingLocksModified": False,
            "ND08StepLockCreated": False,
            "CheckpointAndHashesCreated": True,
        },
        "ReadyForND09": True,
        "NextStep": "ND09",
    }

    staged_checkpoint_path = staged_control_dir / CHECKPOINT_PATH.name
    write_json(staged_checkpoint_path, checkpoint_payload)
    checkpoint_sha256 = sha256_file(staged_checkpoint_path)
    staged_checkpoint_sha_path = staged_control_dir / CHECKPOINT_SHA_PATH.name
    write_text(
        staged_checkpoint_sha_path,
        f"{checkpoint_sha256}  {CHECKPOINT_PATH.name}\n",
    )

    required_outputs = [
        STAGING_ROOT / README_PATH.name,
        staged_prediction_dir / RECURSIVE_DAILY_PATH.name,
        staged_prediction_dir / RECURSIVE_PRODUCT_WEEK_PATH.name,
        staged_prediction_dir / DAILY_UPDATED_ROUTED_PATH.name,
        staged_metric_dir / RECURSIVE_METRICS_PATH.name,
        staged_metric_dir / OPERATIONAL_SELECTION_PATH.name,
        staged_audit_dir / RECURSION_AUDIT_PATH.name,
        staged_audit_dir / VALIDATION_PATH.name,
        staged_contract_dir / SYSTEM_CONTRACT_PATH.name,
        staged_report_dir / REPORT_SUMMARY_PATH.name,
        staged_manifest_path,
        staged_checkpoint_path,
        staged_checkpoint_sha_path,
    ]
    missing_outputs = [path for path in required_outputs if not path.is_file()]
    if missing_outputs:
        raise AssertionError(
            "Required ND08 outputs are missing:\n"
            + "\n".join(f"- {path}" for path in missing_outputs)
        )

    os.replace(STAGING_ROOT, ND08_ROOT)
    TOP_LEVEL_CHECKPOINT_PATH.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(CHECKPOINT_PATH, TOP_LEVEL_CHECKPOINT_PATH)
    shutil.copy2(CHECKPOINT_SHA_PATH, TOP_LEVEL_CHECKPOINT_SHA_PATH)

    # =========================================================================
    # PROJECT MEMORY AND HANDOFF
    # =========================================================================

    handoff_text = f"""# ND08 Handoff

## Status

- Completed step: `{STEP_ID}`
- Status: `{STATUS}`
- Completed local time: `{NOW_LOCAL.isoformat()}`
- Root: `{ND08_ROOT}`
- Checkpoint: `{TOP_LEVEL_CHECKPOINT_PATH}`
- Checkpoint SHA-256: `{checkpoint_sha256}`

## Evaluation

- Complete Monday-to-Friday weeks: {len(week_registry)}
- Methods compared: {', '.join(METHODS)}
- Same complete-week registry used for Monday-origin and daily-updated modes: yes
- March 2026 used: no

## Operational selections

- Best daily product method: `{daily_product_method}`
- Best Monday-origin weekly planning method: `{week_start_method}`
- Best rolling daily-updated weekly method: `{rolling_updated_method}`
- Best Monday-origin restaurant-week method: `{best_recursive_restaurant_method}`
- Best daily-updated restaurant-week method: `{best_daily_updated_restaurant_method}`

## Monday-origin protocol

- Forecast created before Monday.
- No actual demand from inside the week is used.
- Predicted days are inserted into temporary history.
- All historical demand features are rebuilt daily.
- Dynamic routes are rebuilt daily.

## Safety

- Final production model fitted: no
- Final model saved: no
- Previous inputs modified: no
- ND08 lock created: no
- New untouched future evaluation still required: yes

## Next step

ND09 will build arbitrary-date inference using the operational method contract established here.
"""

    workflow_section = f"""## ND08 — Genuine Monday-origin recursive weekly system

Status: `{STATUS}`

- Compared ROLLING_MEAN_5, NESTED_MEDIAN_ENSEMBLE, and NESTED_WEIGHTED_ENSEMBLE_4_MODELS.
- Evaluated {len(week_registry)} complete Monday-to-Friday weeks across five outer folds.
- Rebuilt all historical demand features and dynamic routes after every predicted day.
- Best daily product method: `{daily_product_method}`.
- Best Monday-origin planning method: `{week_start_method}`.
- Best rolling daily-updated method: `{rolling_updated_method}`.
- March 2026 remained closed.
- No final model was fitted or saved.
"""

    decisions_section = f"""## ND08 decisions

1. Monday-origin forecasting must not use actual demand from inside the week.
2. Predicted demand must update temporary history before the next forecast day.
3. Dynamic routes and all historical demand features must be recomputed recursively.
4. Monday-origin and daily-updated results must use the same complete-week registry.
5. Best daily product method: `{daily_product_method}`.
6. Best Monday-origin weekly planning method: `{week_start_method}`.
7. Best rolling daily-updated weekly method: `{rolling_updated_method}`.
8. Different methods may serve different operational objectives.
9. March 2026 remains excluded from selection.
10. A new untouched future period remains required for final unbiased evaluation.
"""

    metrics_section = f"""## ND08 recursive weekly results

- Complete weeks: {len(week_registry)}
- Daily product method: `{daily_product_method}`
- Monday-origin planning method: `{week_start_method}`
- Monday-origin selected product-week WAPE: {selected_origin_product['WAPEPercentage']:.6f}%
- Monday-origin selected restaurant-week WAPE: {selected_origin_restaurant['WAPEPercentage']:.6f}%
- Rolling daily-updated method: `{rolling_updated_method}`
- Daily-updated selected product-week WAPE: {selected_updated_product['WAPEPercentage']:.6f}%
- Daily-updated selected restaurant-week WAPE: {selected_updated_restaurant['WAPEPercentage']:.6f}%
- Total fitting and evaluation time: {total_fitting_seconds:.3f} seconds
"""

    agents_section = f"""## ND08 authoritative status

Marker: ND08_AUTHORITATIVE_STATUS

- Status: `{STATUS}`
- Handoff: `{ND08_HANDOFF_PATH}`
- Daily product method: `{daily_product_method}`
- Monday-origin planning method: `{week_start_method}`
- Rolling daily-updated method: `{rolling_updated_method}`
- March target vault opened: no
- Final production model fitted: no
- Next step: `ND09`
"""

    atomic_write_text(ND08_HANDOFF_PATH, handoff_text)
    atomic_write_text(CURRENT_HANDOFF_PATH, handoff_text)
    append_marked_section(
        WORKFLOW_PATH,
        "## ND08 — Genuine Monday-origin recursive weekly system",
        workflow_section,
    )
    append_marked_section(
        DECISIONS_PATH,
        "## ND08 decisions",
        decisions_section,
    )
    append_marked_section(
        METRICS_AND_RESULTS_PATH,
        "## ND08 recursive weekly results",
        metrics_section,
    )
    append_marked_section(
        AGENTS_PATH,
        "Marker: ND08_AUTHORITATIVE_STATUS",
        agents_section,
    )

    log_text = "\n".join(
        [
            f"Step: {STEP_ID}",
            f"Status: {STATUS}",
            f"Created local: {NOW_LOCAL.isoformat()}",
            f"Complete weeks: {len(week_registry)}",
            f"Daily product method: {daily_product_method}",
            f"Monday-origin method: {week_start_method}",
            f"Rolling daily-updated method: {rolling_updated_method}",
            f"Checkpoint SHA256: {checkpoint_sha256}",
            "March target vault opened: False",
            "Within-week actual demand used for Monday-origin: False",
            "Final production model fitted: False",
            "Final model artifact saved: False",
            "ND08 step lock created: False",
            "",
        ]
    )
    atomic_write_text(LOG_PATH, log_text)

except Exception:
    if STAGING_ROOT.exists():
        shutil.rmtree(STAGING_ROOT)
    raise


# =============================================================================
# FINAL CONSOLE OUTPUT
# =============================================================================

print("=" * 118)
print("EDEN NORMAL-DEMAND MODEL V2 — ND08 COMPLETE")
print("=" * 118)
print(f"Status: {STATUS}")
print(f"Local time: {NOW_LOCAL.isoformat()}")
print(f"ND08 root: {ND08_ROOT}")

print("\nINPUT VERIFICATION")
for step in ["ND03", "ND04", "ND05", "ND06", "ND07", "ND07E"]:
    print(f"{step} checkpoint SHA-256: {checkpoint_hashes[step]}")
print(f"Pre-March all-route rows: {len(all_routes):,}")
print(f"Core predictors: {len(core_predictors)}")
print(f"Historical predictors rebuilt recursively: {len(HISTORICAL_DEMAND_PREDICTORS)}")
print("March target vault opened: False")
print("Previous inputs modified: False")
print(f"ND07E ensemble-setting source schema: {nd07e_selection_source_schema}")

print("\nWEEKLY EVALUATION REGISTRY")
print(f"Complete Monday-to-Friday weeks: {len(week_registry)}")
print(f"Outer folds represented: {week_registry['OuterFold'].nunique()}")
print(f"Evaluation dates: {len(evaluation_dates)}")
print(f"Excluded partial weeks: {len(excluded_week_audit)}")
print("Monday-origin and daily-updated registry identical: True")

print("\nMONDAY-ORIGIN RECURSIVE PROTOCOL")
print("Forecast created before Monday: True")
print("Within-week actual demand used: False")
print("Predictions inserted into temporary history: True")
print("Historical features recalculated daily: True")
print("Dynamic routes recalculated daily: True")
print("Monday feature and route replay passed: " + str(bool(recursion_audit['MondayReplayPassed'].all())))
print(f"Total fitting and evaluation seconds: {total_fitting_seconds:.3f}")

print("\nMONDAY-ORIGIN PRODUCT-WEEK RESULTS")
print(
    recursive_product_metrics[
        [
            "CandidateMethod",
            "Observations",
            "WAPEPercentage",
            "MAE",
            "RMSE",
            "TotalBias",
            "AbsoluteBiasPercentage",
        ]
    ].sort_values("WAPEPercentage").to_string(index=False)
)

print("\nMONDAY-ORIGIN RESTAURANT-WEEK RESULTS")
print(
    recursive_restaurant_metrics[
        [
            "CandidateMethod",
            "Observations",
            "WAPEPercentage",
            "MAE",
            "RMSE",
            "TotalBias",
            "AbsoluteBiasPercentage",
        ]
    ].sort_values("WAPEPercentage").to_string(index=False)
)

print("\nDAILY-UPDATED PRODUCT-WEEK RESULTS — SAME COMPLETE WEEKS")
print(
    daily_updated_product_metrics[
        [
            "CandidateMethod",
            "Observations",
            "WAPEPercentage",
            "MAE",
            "RMSE",
            "TotalBias",
            "AbsoluteBiasPercentage",
        ]
    ].sort_values("WAPEPercentage").to_string(index=False)
)

print("\nOPERATIONAL METHOD SELECTION")
print(f"Best daily product method: {daily_product_method}")
print(f"Best Monday-origin weekly planning method: {week_start_method}")
print(f"Best rolling daily-updated weekly method: {rolling_updated_method}")
print(
    "Best Monday-origin restaurant-week method: "
    f"{best_recursive_restaurant_method}"
)
print(
    "Best daily-updated restaurant-week method: "
    f"{best_daily_updated_restaurant_method}"
)

print("\nSELECTION DETAIL")
print(
    operational_selection[
        [
            "SelectionContext",
            "CandidateMethod",
            "WAPEPercentage",
            "WAPEImprovementPercentagePoints",
            "FoldsWonAgainstBenchmark",
            "StrictlyQualifies",
            "Selected",
        ]
    ].to_string(index=False)
)

print("\nOUTPUTS")
print(f"- Recursive daily predictions: {RECURSIVE_DAILY_PATH}")
print(f"- Recursive product-week predictions: {RECURSIVE_PRODUCT_WEEK_PATH}")
print(f"- Recursive restaurant-week predictions: {RECURSIVE_RESTAURANT_WEEK_PATH}")
print(f"- Daily-updated routed predictions: {DAILY_UPDATED_ROUTED_PATH}")
print(f"- Recursive metrics: {RECURSIVE_METRICS_PATH}")
print(f"- Daily-updated metrics: {DAILY_UPDATED_METRICS_PATH}")
print(f"- Operational selections: {OPERATIONAL_SELECTION_PATH}")
print(f"- Recursion audit: {RECURSION_AUDIT_PATH}")
print(f"- Week registry: {WEEK_REGISTRY_PATH}")
print(f"- System contract: {SYSTEM_CONTRACT_PATH}")
print(f"- Figures: {FIGURE_DIR}")
print(f"- Report summary: {REPORT_SUMMARY_PATH}")
print(f"- Validation: {VALIDATION_PATH}")
print(f"- Manifest: {MANIFEST_PATH}")
print(f"- Checkpoint: {TOP_LEVEL_CHECKPOINT_PATH}")
print(f"- Checkpoint SHA-256: {checkpoint_sha256}")
print(f"- Handoff: {ND08_HANDOFF_PATH}")

print("\nSAFETY")
print("- Same-week actual demand used in Monday-origin forecast: False")
print("- March target vault opened: False")
print("- Final production model fitted: False")
print("- Final model artifact saved: False")
print("- Previous inputs modified: False")
print("- Existing model locks modified: False")
print("- ND08 step lock created: False")
print("- ND08 checkpoint and hashes created: True")

print("\nNEXT STEP")
print(
    "ND09 — arbitrary-date inference using the operational method contract "
    "selected in ND08."
)
print("=" * 118)

Week 2025-10-06 | fold 1: completed genuine Monday-origin recursion for all three methods.
Week 2025-10-13 | fold 1: completed genuine Monday-origin recursion for all three methods.
Week 2025-10-20 | fold 1: completed genuine Monday-origin recursion for all three methods.
Week 2025-11-03 | fold 2: completed genuine Monday-origin recursion for all three methods.
Week 2025-11-10 | fold 2: completed genuine Monday-origin recursion for all three methods.
Week 2025-11-17 | fold 2: completed genuine Monday-origin recursion for all three methods.
Week 2025-12-01 | fold 3: completed genuine Monday-origin recursion for all three methods.
Week 2025-12-08 | fold 3: completed genuine Monday-origin recursion for all three methods.
Week 2025-12-15 | fold 3: completed genuine Monday-origin recursion for all three methods.
Week 2026-01-05 | fold 4: completed genuine Monday-origin recursion for all three methods.
Week 2026-01-19 | fold 4: completed genuine Monday-origin recursion for all three methods.

In [14]:
# =============================================================================
# EDEN NORMAL-DEMAND MODEL V2
# ND09 — ARBITRARY-DATE INFERENCE ENGINE AND DEMONSTRATION MODEL FREEZE (FIXED)
#
# Compatibility revision 1.1: fixes future-route reconstruction.
# Run this as one complete Jupyter cell after ND08.
#
# This step freezes the revised operational system selected in ND08:
#   - next-operating-day product forecast: NESTED_MEDIAN_ENSEMBLE
#   - multi-step / Monday-origin planning: ROLLING_MEAN_5
#   - fallback routes: the ND04 selected fallback methods
#
# The engine fits demonstration/deployment artifacts on all pre-March
# development rows, creates a configurable arbitrary-date forecast, creates a
# genuine Monday-to-Friday week-ahead forecast, and saves reusable model and
# inference contracts. March 2026 targets remain closed and are never loaded.
# A new untouched future period is still required for final unbiased accuracy.
# =============================================================================

import hashlib
import json
import math
import os
import platform
import shutil
import time
import uuid
import warnings
from collections import Counter
from datetime import datetime, timezone
from pathlib import Path
from zoneinfo import ZoneInfo

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import sklearn
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OrdinalEncoder

try:
    import xgboost
    from xgboost import XGBRegressor
except Exception as error:
    raise RuntimeError(
        "ND09 requires xgboost. Restore the environment used for ND07E/ND08."
    ) from error

try:
    import catboost
    from catboost import CatBoostRegressor
except Exception as error:
    raise RuntimeError(
        "ND09 requires catboost. Restore the environment used for ND07E/ND08."
    ) from error

warnings.filterwarnings(
    "ignore",
    message="X does not have valid feature names",
)

os.environ.setdefault("OMP_NUM_THREADS", "1")
os.environ.setdefault("OPENBLAS_NUM_THREADS", "1")
os.environ.setdefault("MKL_NUM_THREADS", "1")
os.environ.setdefault("NUMEXPR_NUM_THREADS", "1")


# =============================================================================
# USER CONFIGURATION
# =============================================================================

# Leave as None to forecast the next operating day after the final pre-March
# actual date. Otherwise use an ISO date such as "2026-03-05".
REQUESTED_DAILY_DATE = None

# Leave as None to forecast the next Monday-to-Friday planning week. Otherwise
# use a Monday date such as "2026-03-09".
REQUESTED_WEEK_START = None

# Default future operating calendar. Add exceptional closure dates or
# exceptional weekend operating dates as ISO strings when required.
CLOSED_DATES = []
EXTRA_OPERATING_DATES = []

# None means forecast the complete active catalogue. To demonstrate a smaller
# menu, provide exact CanonicalProductID strings in a Python list.
ACTIVE_PRODUCT_IDS = None

# ND08 validated genuine recursive forecasting over five operating days.
# Longer paths are supported technically but are flagged as extrapolation.
MAX_VALIDATED_RECURSIVE_HORIZON = 5
MAXIMUM_ALLOWED_RECURSIVE_HORIZON = 60
ALLOW_LONG_HORIZON_EXTRAPOLATION = False

ALLOW_OVERWRITE = False


# =============================================================================
# FIXED PROJECT CONFIGURATION
# =============================================================================

PROJECT_ROOT = Path("/Users/ryansmac/Desktop/Meng Project")
EDEN_ROOT = PROJECT_ROOT / "eden_datasets"
MODEL_ROOT = EDEN_ROOT / "eden_normal_demand_model_v2"

ND03_ROOT = MODEL_ROOT / "02_feature_engineering" / "ND03_normal_demand_features"
ND04_ROOT = (
    MODEL_ROOT
    / "03_models"
    / "00_candidates"
    / "ND04_baseline_and_fallback_evaluation"
)
ND07_ROOT = (
    MODEL_ROOT
    / "03_models"
    / "01_tuning"
    / "ND07_tuning_calibration_blending"
)
ND07E_ROOT = (
    MODEL_ROOT
    / "03_models"
    / "01_tuning"
    / "ND07E_focused_ensemble_challenge"
)
ND08_ROOT = (
    MODEL_ROOT
    / "03_models"
    / "02_system_evaluation"
    / "ND08_recursive_weekly_system"
)

ALL_ROUTES_PATH = (
    ND03_ROOT
    / "01_model_ready_datasets"
    / "ND03_pre_march_all_routes_development_dataset.csv"
)
CORE_PREDICTOR_LIST_PATH = (
    ND03_ROOT / "03_contracts" / "ND03_core_predictor_list.csv"
)
ND04_SELECTED_METHODS_PATH = (
    ND04_ROOT / "02_metrics" / "ND04_selected_route_methods.csv"
)
ND07_OUTER_SELECTIONS_PATH = (
    ND07_ROOT
    / "02_metrics"
    / "ND07_outer_fold_selected_configurations.csv"
)
ND07E_OUTER_SELECTIONS_PATH = (
    ND07E_ROOT
    / "02_metrics"
    / "ND07E_outer_fold_selected_ensemble_settings.csv"
)
ND08_SYSTEM_CONTRACT_PATH = (
    ND08_ROOT / "04_contracts" / "ND08_operational_method_contract.json"
)

CHECKPOINT_PATHS = {
    "ND03": MODEL_ROOT / "08_checkpoints" / "ND03_checkpoint.json",
    "ND04": MODEL_ROOT / "08_checkpoints" / "ND04_checkpoint.json",
    "ND05": MODEL_ROOT / "08_checkpoints" / "ND05_checkpoint.json",
    "ND06": MODEL_ROOT / "08_checkpoints" / "ND06_checkpoint.json",
    "ND07": MODEL_ROOT / "08_checkpoints" / "ND07_checkpoint.json",
    "ND07E": MODEL_ROOT / "08_checkpoints" / "ND07E_checkpoint.json",
    "ND08": MODEL_ROOT / "08_checkpoints" / "ND08_checkpoint.json",
}

EXPECTED_CHECKPOINT_HASHES = {
    "ND03": "0845af89a5b459ca13ae6ffd99dde444f5010f6c0fb5ba4c34a4f091ac2e151c",
    "ND04": "2fdc5d2c64c38f85b2669ca942042884209d80111cc840261307da98b1e9cf54",
    "ND05": "ce3342c8b960aa5c4791a114ab09ae1a86d1dab2a3eebb648aa060579a1378ff",
    "ND06": "e3b7bb75a1b2968e426c9a4c1654e10420d683af4bd5cb2b75fa4b5f0f18f357",
    "ND07": "39077dd8c197561e5384a6943c3a4e153153e75fa4d03019f45005eb145cf18e",
    "ND07E": "eaf6d23434cb67d663f66a39db4f898b74c1d516aee5c0627ca9136bed9877b8",
    "ND08": "a98828007df9bd4d0cce309fc8f8bfbc1d2c1f1cf74f1ca762b27237468dea10",
}

DATE_COLUMN = "Date"
PRODUCT_ID_COLUMN = "CanonicalProductID"
PRODUCT_NAME_COLUMN = "CanonicalProductName"
TARGET_COLUMN = "NormalDemand"
ROUTE_COLUMN = "ForecastRoute"
FAMILY_COLUMN = "TierProductFamily"
DAY_OF_WEEK_COLUMN = "DayOfWeekNumber"
SEQUENCE_COLUMN = "OperatingDaySequence"

EXPECTED_ALL_ROUTE_ROWS = 23_763
EXPECTED_CORE_PREDICTORS = 53
EXPECTED_NUMERIC_PREDICTORS = 45
EXPECTED_CATEGORICAL_PREDICTORS = 8
EXPECTED_PRODUCTS = 227
DEVELOPMENT_END_EXCLUSIVE = pd.Timestamp("2026-03-02")

LAG_OPERATING_DAYS = [1, 2, 3, 5, 10, 20]
ROLLING_WINDOWS = [3, 5, 10, 20]
ZERO_POSITIVE_WINDOWS = [5, 10, 20]
MINIMUM_MAIN_HISTORY = 20
PRIMARY_SCOPE_PERCENTAGE = 95.0

HISTORICAL_DEMAND_PREDICTORS = (
    [f"NormalDemandLag_{lag}" for lag in LAG_OPERATING_DAYS]
    + [
        feature
        for window in ROLLING_WINDOWS
        for feature in [
            f"PastNormalDemandRollingMean_{window}",
            f"PastNormalDemandRollingMedian_{window}",
            f"PastNormalDemandRollingStd_{window}",
            f"PastNormalDemandRollingSum_{window}",
        ]
    ]
    + [
        f"PastZeroNormalDemandRate_{window}"
        for window in ZERO_POSITIVE_WINDOWS
    ]
    + [
        f"PastPositiveNormalDemandCount_{window}"
        for window in ZERO_POSITIVE_WINDOWS
    ]
    + [
        "OperatingDaysSincePreviousPositiveNormalDemand",
        "ExpandingPastMeanNormalDemand",
        "ExpandingPastPositiveNormalDemandRate",
    ]
)

BASE_NUMERIC_PREDICTORS = [
    "ProductAgeOperatingDays",
    "SourcePLUCount",
    "IsMultiPLUCanonicalProduct",
    "OperatingDaySequence",
    "Year",
    "Month",
    "Quarter",
    "DayOfWeekNumber",
    "ISOYear",
    "ISOWeek",
    "DayOfYear",
    "IsWeekend",
    "DaysSincePreviousOperatingDate",
    "IsConsecutiveCalendarDay",
]

BASE_CATEGORICAL_PREDICTORS = [
    "SourceGroupCodes",
    "SourceGroupNames",
    "BeverageSeries",
    "BeverageType",
    "SupplierLabelsObserved",
    "TierProductFamily",
    "NominalPriceTier",
    "MenuGeneration",
]

PRODUCT_METADATA_COLUMNS = [
    PRODUCT_ID_COLUMN,
    PRODUCT_NAME_COLUMN,
    "ProductFirstObservedDate",
    "SourcePLUCount",
    "IsMultiPLUCanonicalProduct",
    *BASE_CATEGORICAL_PREDICTORS,
]

BASE_COMPONENTS = [
    "ROLLING_MEAN_5",
    "CATBOOST_CORE53",
    "XGBOOST_CORE53",
    "XGBOOST_PRODUCT_AWARE",
]

FAMILY_TO_COMPONENT = {
    "NESTED_TUNED_CATBOOST_RMSE_CORE53": "CATBOOST_CORE53",
    "NESTED_TUNED_XGBOOST_SQUARED_CORE53": "XGBOOST_CORE53",
    "NESTED_TUNED_XGBOOST_SQUARED_PRODUCT_AWARE": "XGBOOST_PRODUCT_AWARE",
}
EXPECTED_FAMILIES = set(FAMILY_TO_COMPONENT)

EXPECTED_ROUTE_METHODS = {
    "MAIN_MODEL": "ROLLING_MEAN_5",
    "COLD_START_FALLBACK": "RECENT_MEDIAN5_WITH_BACKOFF",
    "LOW_DEMAND_FALLBACK": "RECENT_MEDIAN5_WITH_BACKOFF",
    "ZERO_HISTORY_FALLBACK": "EXPANDING_MEAN_WITH_BACKOFF",
}

EXPECTED_DAILY_METHOD = "NESTED_MEDIAN_ENSEMBLE"
EXPECTED_WEEK_METHOD = "ROLLING_MEAN_5"
EXPECTED_UPDATED_WEEK_METHOD = "ROLLING_MEAN_5"

RANDOM_SEED = 42
MAX_THREADS = max(1, min(4, os.cpu_count() or 1))

ND09_ROOT = (
    MODEL_ROOT
    / "03_models"
    / "03_inference"
    / "ND09_arbitrary_date_inference_engine"
)
MODEL_ARTIFACT_DIR = ND09_ROOT / "01_model_artifacts"
FORECAST_DIR = ND09_ROOT / "02_forecasts"
CONTRACT_DIR = ND09_ROOT / "03_contracts"
AUDIT_DIR = ND09_ROOT / "04_audits"
FIGURE_DIR = ND09_ROOT / "05_figures"
REPORT_DIR = ND09_ROOT / "06_reports"
CONTROL_DIR = ND09_ROOT / "07_control"

CORE_PREPROCESSOR_PATH = MODEL_ARTIFACT_DIR / "ND09_core_preprocessor.joblib"
PRODUCT_PREPROCESSOR_PATH = MODEL_ARTIFACT_DIR / "ND09_product_preprocessor.joblib"
CATBOOST_MODEL_PATH = MODEL_ARTIFACT_DIR / "ND09_catboost_core53.cbm"
XGBOOST_CORE_MODEL_PATH = MODEL_ARTIFACT_DIR / "ND09_xgboost_core53.json"
XGBOOST_PRODUCT_MODEL_PATH = (
    MODEL_ARTIFACT_DIR / "ND09_xgboost_product_aware.json"
)
MODEL_METADATA_PATH = MODEL_ARTIFACT_DIR / "ND09_model_artifact_metadata.json"

DAILY_PATH_FORECAST_PATH = FORECAST_DIR / "ND09_arbitrary_date_recursive_path.csv"
DAILY_PRODUCT_FORECAST_PATH = FORECAST_DIR / "ND09_arbitrary_date_product_forecast.csv"
DAILY_RESTAURANT_FORECAST_PATH = FORECAST_DIR / "ND09_arbitrary_date_restaurant_total.csv"
WEEK_PATH_FORECAST_PATH = FORECAST_DIR / "ND09_week_ahead_recursive_path.csv"
WEEK_DAILY_PRODUCT_FORECAST_PATH = FORECAST_DIR / "ND09_week_ahead_daily_product_forecast.csv"
WEEK_PRODUCT_TOTAL_PATH = FORECAST_DIR / "ND09_week_ahead_product_totals.csv"
WEEK_RESTAURANT_DAILY_PATH = FORECAST_DIR / "ND09_week_ahead_restaurant_daily_totals.csv"
WEEK_RESTAURANT_TOTAL_PATH = FORECAST_DIR / "ND09_week_ahead_restaurant_total.csv"

ACTIVE_CATALOGUE_PATH = CONTRACT_DIR / "ND09_active_product_catalogue.csv"
FUTURE_CALENDAR_PATH = CONTRACT_DIR / "ND09_future_operating_calendar.csv"
INFERENCE_CONTRACT_PATH = CONTRACT_DIR / "ND09_inference_engine_contract.json"
INFERENCE_CONTRACT_MD_PATH = CONTRACT_DIR / "ND09_inference_engine_contract.md"
USAGE_PATH = CONTRACT_DIR / "ND09_inference_usage.md"

INPUT_HASH_AUDIT_PATH = AUDIT_DIR / "ND09_input_hash_audit.csv"
MODEL_FIT_AUDIT_PATH = AUDIT_DIR / "ND09_model_fit_audit.csv"
REPRESENTATIVE_SETTINGS_PATH = AUDIT_DIR / "ND09_representative_settings.csv"
FEATURE_REPLAY_AUDIT_PATH = AUDIT_DIR / "ND09_future_feature_replay_audit.csv"
FORECAST_PATH_AUDIT_PATH = AUDIT_DIR / "ND09_forecast_path_audit.csv"
ROUTE_SUMMARY_PATH = AUDIT_DIR / "ND09_forecast_route_summary.csv"
PACKAGE_VERSIONS_PATH = AUDIT_DIR / "ND09_package_versions.csv"
VALIDATION_PATH = AUDIT_DIR / "ND09_validation_summary.csv"

REPORT_SUMMARY_PATH = REPORT_DIR / "ND09_arbitrary_date_inference_summary.md"
README_PATH = ND09_ROOT / "README.md"
MANIFEST_PATH = CONTROL_DIR / "ND09_artifact_hash_manifest.csv"
CHECKPOINT_PATH = CONTROL_DIR / "ND09_checkpoint.json"
CHECKPOINT_SHA_PATH = CONTROL_DIR / "ND09_checkpoint.sha256"
LOCK_PATH = CONTROL_DIR / "ND09_demonstration_engine_lock.json"

TOP_LEVEL_CHECKPOINT_PATH = MODEL_ROOT / "08_checkpoints" / "ND09_checkpoint.json"
TOP_LEVEL_CHECKPOINT_SHA_PATH = MODEL_ROOT / "08_checkpoints" / "ND09_checkpoint.sha256"
TOP_LEVEL_LOCK_PATH = (
    MODEL_ROOT / "08_checkpoints" / "ND09_demonstration_engine_lock.json"
)

MEMORY_ROOT = MODEL_ROOT / "00_project_memory"
ND09_HANDOFF_PATH = MEMORY_ROOT / "ND09_HANDOFF.md"
CURRENT_HANDOFF_PATH = MEMORY_ROOT / "CURRENT_HANDOFF.md"
WORKFLOW_PATH = MEMORY_ROOT / "WORKFLOW.md"
DECISIONS_PATH = MEMORY_ROOT / "DECISIONS.md"
METRICS_AND_RESULTS_PATH = MEMORY_ROOT / "METRICS_AND_RESULTS.md"
AGENTS_PATH = MODEL_ROOT / "AGENTS.md"
LOG_PATH = MODEL_ROOT / "09_logs" / "ND09_arbitrary_date_inference_log.txt"

STEP_ID = "ND09"
STATUS = "ND09_ARBITRARY_DATE_INFERENCE_ENGINE_CREATED_READY_FOR_ND10"
NOW_UTC = datetime.now(timezone.utc)
NOW_LOCAL = NOW_UTC.astimezone(ZoneInfo("Europe/Dublin"))


# =============================================================================
# GENERAL HELPERS
# =============================================================================

def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


def write_csv(path: Path, frame: pd.DataFrame) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    frame.to_csv(path, index=False)


def write_json(path: Path, payload: dict) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(
        json.dumps(payload, indent=2, ensure_ascii=False, default=str) + "\n",
        encoding="utf-8",
    )


def write_text(path: Path, text: str) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(text, encoding="utf-8")


def atomic_write_text(path: Path, text: str) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_name(f".{path.name}.{uuid.uuid4().hex}.tmp")
    temporary.write_text(text, encoding="utf-8")
    os.replace(temporary, path)


def append_marked_section(path: Path, marker: str, section_text: str) -> None:
    existing = path.read_text(encoding="utf-8") if path.is_file() else ""
    if marker in existing:
        return
    separator = "\n" if existing.endswith("\n") else "\n\n"
    atomic_write_text(path, existing + separator + section_text.strip() + "\n")


def validate_required_columns(
    frame: pd.DataFrame,
    columns: set[str],
    name: str,
) -> None:
    missing = sorted(columns - set(frame.columns))
    if missing:
        raise AssertionError(
            f"{name} is missing required columns:\n"
            + "\n".join(f"- {column}" for column in missing)
        )


def normalize_family(values: pd.Series) -> pd.Series:
    return values.astype("string").fillna("__MISSING_FAMILY__").astype(str)


def save_figure(path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    plt.tight_layout()
    plt.savefig(path, dpi=300, bbox_inches="tight")
    plt.close()


def parse_iso_dates(values: list[str], label: str) -> set[pd.Timestamp]:
    parsed = set()
    for value in values:
        timestamp = pd.Timestamp(value).normalize()
        if pd.isna(timestamp):
            raise ValueError(f"Invalid {label} date: {value}")
        parsed.add(timestamp)
    return parsed


def is_operating_date(
    date: pd.Timestamp,
    closed_dates: set[pd.Timestamp],
    extra_dates: set[pd.Timestamp],
) -> bool:
    date = pd.Timestamp(date).normalize()
    if date in closed_dates:
        return False
    return date.weekday() < 5 or date in extra_dates


def next_operating_date(
    date: pd.Timestamp,
    closed_dates: set[pd.Timestamp],
    extra_dates: set[pd.Timestamp],
) -> pd.Timestamp:
    candidate = pd.Timestamp(date).normalize() + pd.Timedelta(days=1)
    for _ in range(370):
        if is_operating_date(candidate, closed_dates, extra_dates):
            return candidate
        candidate += pd.Timedelta(days=1)
    raise RuntimeError("No operating date found within 370 calendar days.")


def monday_on_or_after(date: pd.Timestamp) -> pd.Timestamp:
    date = pd.Timestamp(date).normalize()
    return date + pd.Timedelta(days=(7 - date.weekday()) % 7)


def generate_future_calendar(
    cutoff_date: pd.Timestamp,
    cutoff_sequence: int,
    end_date: pd.Timestamp,
    closed_dates: set[pd.Timestamp],
    extra_dates: set[pd.Timestamp],
) -> pd.DataFrame:
    cutoff_date = pd.Timestamp(cutoff_date).normalize()
    end_date = pd.Timestamp(end_date).normalize()
    if end_date <= cutoff_date:
        raise ValueError("Future calendar end date must be after the cutoff date.")

    records = []
    sequence = int(cutoff_sequence)
    previous_operating_date = cutoff_date

    for date in pd.date_range(cutoff_date + pd.Timedelta(days=1), end_date, freq="D"):
        if not is_operating_date(date, closed_dates, extra_dates):
            continue
        sequence += 1
        iso = date.isocalendar()
        gap = int((date - previous_operating_date).days)
        records.append(
            {
                DATE_COLUMN: date,
                SEQUENCE_COLUMN: sequence,
                "Year": int(date.year),
                "Month": int(date.month),
                "Quarter": int(date.quarter),
                "DayOfWeekNumber": int(date.weekday()),
                "ISOYear": int(iso.year),
                "ISOWeek": int(iso.week),
                "DayOfYear": int(date.dayofyear),
                "IsWeekend": bool(date.weekday() >= 5),
                "DaysSincePreviousOperatingDate": gap,
                "IsConsecutiveCalendarDay": bool(gap == 1),
                "CalendarPolicy": (
                    "EXCEPTIONAL_OPERATING_DATE"
                    if date.weekday() >= 5
                    else "STANDARD_MONDAY_TO_FRIDAY"
                ),
            }
        )
        previous_operating_date = date

    calendar = pd.DataFrame(records)
    if calendar.empty:
        raise AssertionError("The future operating calendar is empty.")
    return calendar


def mode_with_inner_wape_tie_break(
    values: pd.Series,
    inner_scores: pd.Series,
):
    value_strings = values.astype(str)
    counts = Counter(value_strings)
    maximum = max(counts.values())
    candidates = [value for value, count in counts.items() if count == maximum]
    score_frame = pd.DataFrame(
        {"Value": value_strings.to_numpy(), "Score": inner_scores.to_numpy(dtype=float)}
    )
    mean_scores = score_frame.groupby("Value")["Score"].mean()
    return min(candidates, key=lambda value: (mean_scores[value], value))


# =============================================================================
# FEATURE AND ROUTING HELPERS
# =============================================================================

def historical_features_from_history(
    demand_history: np.ndarray,
    sequence_history: np.ndarray,
    current_sequence: float,
) -> dict[str, float]:
    demand_history = np.asarray(demand_history, dtype=float)
    sequence_history = np.asarray(sequence_history, dtype=float)
    result: dict[str, float] = {}

    for lag in LAG_OPERATING_DAYS:
        result[f"NormalDemandLag_{lag}"] = (
            float(demand_history[-lag])
            if len(demand_history) >= lag
            else float("nan")
        )

    for window in ROLLING_WINDOWS:
        values = demand_history[-window:]
        if len(values) == 0:
            mean_value = median_value = std_value = sum_value = float("nan")
        else:
            mean_value = float(np.mean(values))
            median_value = float(np.median(values))
            std_value = float(np.std(values, ddof=0))
            sum_value = float(np.sum(values))
        result[f"PastNormalDemandRollingMean_{window}"] = mean_value
        result[f"PastNormalDemandRollingMedian_{window}"] = median_value
        result[f"PastNormalDemandRollingStd_{window}"] = std_value
        result[f"PastNormalDemandRollingSum_{window}"] = sum_value

    for window in ZERO_POSITIVE_WINDOWS:
        values = demand_history[-window:]
        if len(values) == 0:
            zero_rate = positive_count = float("nan")
        else:
            zero_rate = float(np.mean(values == 0))
            positive_count = float(np.sum(values > 0))
        result[f"PastZeroNormalDemandRate_{window}"] = zero_rate
        result[f"PastPositiveNormalDemandCount_{window}"] = positive_count

    positive_indices = np.flatnonzero(demand_history > 0)
    if len(positive_indices) == 0:
        days_since_positive = float("nan")
    else:
        last_positive_sequence = float(sequence_history[int(positive_indices[-1])])
        days_since_positive = float(current_sequence - last_positive_sequence)
    result["OperatingDaysSincePreviousPositiveNormalDemand"] = days_since_positive

    if len(demand_history) == 0:
        expanding_mean = float("nan")
        expanding_positive_rate = float("nan")
    else:
        expanding_mean = float(np.mean(demand_history))
        expanding_positive_rate = float(np.mean(demand_history > 0))
    result["ExpandingPastMeanNormalDemand"] = expanding_mean
    result["ExpandingPastPositiveNormalDemandRate"] = expanding_positive_rate
    return result


def build_history_state(training_frame: pd.DataFrame) -> dict[str, dict[str, list]]:
    state: dict[str, dict[str, list]] = {}
    ordered = training_frame.sort_values(
        [PRODUCT_ID_COLUMN, DATE_COLUMN, SEQUENCE_COLUMN],
        kind="mergesort",
    )
    for product_id, product_frame in ordered.groupby(PRODUCT_ID_COLUMN, sort=False):
        product_key = str(product_id)
        state[product_key] = {
            "demand": product_frame[TARGET_COLUMN].astype(float).tolist(),
            "sequence": product_frame[SEQUENCE_COLUMN].astype(float).tolist(),
        }
    return state


def copy_history_state(
    base_state: dict[str, dict[str, list]],
) -> dict[str, dict[str, list]]:
    return {
        product_id: {
            "demand": values["demand"].copy(),
            "sequence": values["sequence"].copy(),
        }
        for product_id, values in base_state.items()
    }


def append_predictions_to_state(
    state: dict[str, dict[str, list]],
    rows: pd.DataFrame,
    predictions: np.ndarray,
) -> None:
    for product_id, sequence, prediction in zip(
        rows[PRODUCT_ID_COLUMN].astype(str),
        rows[SEQUENCE_COLUMN].astype(float),
        np.asarray(predictions, dtype=float),
    ):
        if product_id not in state:
            state[product_id] = {"demand": [], "sequence": []}
        if state[product_id]["sequence"]:
            if sequence <= state[product_id]["sequence"][-1]:
                raise AssertionError(
                    f"Non-increasing operating sequence for {product_id}."
                )
        state[product_id]["demand"].append(float(prediction))
        state[product_id]["sequence"].append(float(sequence))


def build_recursive_feature_frame(
    score_rows: pd.DataFrame,
    state: dict[str, dict[str, list]],
) -> pd.DataFrame:
    frame = score_rows.copy().reset_index(drop=True)
    feature_records = []
    prior_records = []

    for row in frame.itertuples(index=False):
        product_id = str(getattr(row, PRODUCT_ID_COLUMN))
        current_sequence = float(getattr(row, SEQUENCE_COLUMN))
        product_state = state.get(product_id, {"demand": [], "sequence": []})
        demand_history = np.asarray(product_state["demand"], dtype=float)
        sequence_history = np.asarray(product_state["sequence"], dtype=float)

        feature_records.append(
            historical_features_from_history(
                demand_history,
                sequence_history,
                current_sequence,
            )
        )

        prior_count = int(len(demand_history))
        prior_cumulative = float(demand_history.sum())
        prior_positive = int(np.sum(demand_history > 0))
        prior_records.append(
            {
                "PriorOperatingDayCount": prior_count,
                "PriorCumulativeNormalDemand": prior_cumulative,
                "PriorPositiveNormalDemandDays": prior_positive,
                "PriorZeroNormalDemandDays": prior_count - prior_positive,
                "HasSufficientHistory20": prior_count >= MINIMUM_MAIN_HISTORY,
                "ColdStartFlag": prior_count < MINIMUM_MAIN_HISTORY,
                "ZeroPriorNormalDemandFlag": prior_cumulative <= 0,
            }
        )

    feature_values = pd.DataFrame(feature_records)
    prior_values = pd.DataFrame(prior_records)
    for column in HISTORICAL_DEMAND_PREDICTORS:
        frame[column] = feature_values[column].to_numpy()
    for column in prior_values.columns:
        frame[column] = prior_values[column].to_numpy()

    ranked = frame.sort_values(
        ["PriorCumulativeNormalDemand", "PriorPositiveNormalDemandDays", PRODUCT_ID_COLUMN],
        ascending=[False, False, True],
        kind="mergesort",
    ).copy()

    universe_total = float(ranked["PriorCumulativeNormalDemand"].sum())
    ranked["PriorDemandRank"] = np.arange(1, len(ranked) + 1, dtype=int)
    ranked["PriorDemandSharePercentage"] = 0.0
    ranked["PriorCumulativeDemandShareBeforePercentage"] = 0.0
    ranked["PriorCumulativeDemandSharePercentage"] = 0.0
    ranked["InPrior95DemandScope"] = False

    if universe_total > 0:
        share = 100.0 * ranked["PriorCumulativeNormalDemand"] / universe_total
        cumulative = share.cumsum()
        cumulative_before = cumulative - share
        ranked["PriorDemandSharePercentage"] = share
        ranked["PriorCumulativeDemandShareBeforePercentage"] = cumulative_before
        ranked["PriorCumulativeDemandSharePercentage"] = cumulative
        ranked["InPrior95DemandScope"] = (
            (ranked["PriorCumulativeNormalDemand"] > 0)
            & (cumulative_before < PRIMARY_SCOPE_PERCENTAGE)
        )

    ranked["EligibleForMainModel"] = (
        ranked["HasSufficientHistory20"]
        & ranked["InPrior95DemandScope"]
        & ~ranked["ZeroPriorNormalDemandFlag"]
    )

    ranked["RecursiveForecastRoute"] = np.select(
        [
            ranked["ZeroPriorNormalDemandFlag"],
            ranked["ColdStartFlag"],
            ranked["EligibleForMainModel"],
        ],
        [
            "ZERO_HISTORY_FALLBACK",
            "COLD_START_FALLBACK",
            "MAIN_MODEL",
        ],
        default="LOW_DEMAND_FALLBACK",
    )

    ranked["PriorDemandVolumeSegment"] = np.select(
        [
            universe_total <= 0,
            ranked["ZeroPriorNormalDemandFlag"],
            ranked["PriorCumulativeDemandShareBeforePercentage"] < 80.0,
            ranked["InPrior95DemandScope"],
        ],
        [
            "NO_PRIOR_DEMAND_UNIVERSE",
            "ZERO_PRIOR_DEMAND",
            "HIGH_DEMAND",
            "MODERATE_DEMAND",
        ],
        default="LOW_DEMAND",
    )

    # Only recursively generated ranking and route fields belong in this
    # merge. ForecastRoute is a stored-source field in historical datasets
    # and does not exist on newly constructed future rows. It is created
    # later by predict_routed_system from RecursiveForecastRoute.
    ranking_columns = [
        PRODUCT_ID_COLUMN,
        "PriorDemandRank",
        "PriorDemandSharePercentage",
        "PriorCumulativeDemandShareBeforePercentage",
        "PriorCumulativeDemandSharePercentage",
        "InPrior95DemandScope",
        "EligibleForMainModel",
        "RecursiveForecastRoute",
        "PriorDemandVolumeSegment",
    ]

    missing_ranking_columns = [
        column for column in ranking_columns if column not in ranked.columns
    ]
    if missing_ranking_columns:
        raise AssertionError(
            "Recursive ranking construction is missing required columns: "
            + ", ".join(missing_ranking_columns)
        )

    frame = frame.drop(
        columns=[column for column in ranking_columns[1:] if column in frame.columns],
        errors="ignore",
    ).merge(
        ranked[ranking_columns],
        on=PRODUCT_ID_COLUMN,
        how="left",
        validate="one_to_one",
    )
    return frame


# =============================================================================
# FALLBACK AND MODEL HELPERS
# =============================================================================

def build_hierarchy_statistics(training_frame: pd.DataFrame) -> dict:
    train = training_frame.copy()
    train[PRODUCT_ID_COLUMN] = train[PRODUCT_ID_COLUMN].astype(str)
    train["_FamilyKey"] = normalize_family(train[FAMILY_COLUMN])
    train[DAY_OF_WEEK_COLUMN] = pd.to_numeric(
        train[DAY_OF_WEEK_COLUMN], errors="raise"
    ).astype(int)
    return {
        "global_mean": float(train[TARGET_COLUMN].mean()),
        "product_weekday": train.groupby(
            [PRODUCT_ID_COLUMN, DAY_OF_WEEK_COLUMN], dropna=False
        )[TARGET_COLUMN].mean(),
        "product_mean": train.groupby(PRODUCT_ID_COLUMN, dropna=False)[
            TARGET_COLUMN
        ].mean(),
        "family_weekday": train.groupby(
            ["_FamilyKey", DAY_OF_WEEK_COLUMN], dropna=False
        )[TARGET_COLUMN].mean(),
        "family_mean": train.groupby("_FamilyKey", dropna=False)[TARGET_COLUMN].mean(),
        "global_weekday": train.groupby(DAY_OF_WEEK_COLUMN, dropna=False)[
            TARGET_COLUMN
        ].mean(),
    }


def hierarchy_predictions(
    score_frame: pd.DataFrame,
    statistics: dict,
) -> tuple[np.ndarray, np.ndarray]:
    score = score_frame.copy()
    product_ids = score[PRODUCT_ID_COLUMN].astype(str)
    families = normalize_family(score[FAMILY_COLUMN])
    weekdays = pd.to_numeric(score[DAY_OF_WEEK_COLUMN], errors="raise").astype(int)

    product_weekday_values = np.asarray(
        [
            statistics["product_weekday"].get((product_id, weekday), np.nan)
            for product_id, weekday in zip(product_ids, weekdays)
        ],
        dtype=float,
    )
    product_values = product_ids.map(statistics["product_mean"]).to_numpy(dtype=float)
    family_weekday_values = np.asarray(
        [
            statistics["family_weekday"].get((family, weekday), np.nan)
            for family, weekday in zip(families, weekdays)
        ],
        dtype=float,
    )
    family_values = families.map(statistics["family_mean"]).to_numpy(dtype=float)
    global_weekday_values = weekdays.map(statistics["global_weekday"]).to_numpy(dtype=float)
    global_values = np.full(len(score), statistics["global_mean"], dtype=float)
    zero_values = np.zeros(len(score), dtype=float)

    def coalesce(*arrays) -> np.ndarray:
        output = np.full(len(score), np.nan, dtype=float)
        for values in arrays:
            candidate = np.asarray(values, dtype=float)
            missing = ~np.isfinite(output)
            output[missing] = candidate[missing]
        return np.clip(np.nan_to_num(output, nan=0.0), 0.0, None)

    product_hierarchy = coalesce(
        product_weekday_values,
        product_values,
        family_weekday_values,
        family_values,
        global_weekday_values,
        global_values,
        zero_values,
    )
    family_hierarchy = coalesce(
        family_weekday_values,
        family_values,
        global_weekday_values,
        global_values,
        zero_values,
    )
    return product_hierarchy, family_hierarchy


def fallback_predictions(
    feature_frame: pd.DataFrame,
    hierarchy_statistics: dict,
) -> np.ndarray:
    product_hierarchy, family_hierarchy = hierarchy_predictions(
        feature_frame, hierarchy_statistics
    )

    def values(column: str) -> np.ndarray:
        return pd.to_numeric(feature_frame[column], errors="coerce").to_numpy(dtype=float)

    def coalesce(*arrays) -> np.ndarray:
        output = np.full(len(feature_frame), np.nan, dtype=float)
        for array in arrays:
            candidate = np.asarray(array, dtype=float)
            missing = ~np.isfinite(output)
            output[missing] = candidate[missing]
        return np.clip(np.nan_to_num(output, nan=0.0), 0.0, None)

    recent_median5 = coalesce(
        values("PastNormalDemandRollingMedian_5"),
        values("PastNormalDemandRollingMedian_3"),
        values("NormalDemandLag_1"),
        values("ExpandingPastMeanNormalDemand"),
        family_hierarchy,
        np.zeros(len(feature_frame)),
    )
    expanding_mean = coalesce(
        values("ExpandingPastMeanNormalDemand"),
        product_hierarchy,
        family_hierarchy,
        np.zeros(len(feature_frame)),
    )

    routes = feature_frame["RecursiveForecastRoute"].astype(str).to_numpy()
    prediction = np.full(len(feature_frame), np.nan, dtype=float)
    median_mask = np.isin(routes, ["COLD_START_FALLBACK", "LOW_DEMAND_FALLBACK"])
    zero_mask = routes == "ZERO_HISTORY_FALLBACK"
    prediction[median_mask] = recent_median5[median_mask]
    prediction[zero_mask] = expanding_mean[zero_mask]
    return prediction


def make_preprocessor(
    numeric_predictors: list[str],
    categorical_predictors: list[str],
    include_product_id: bool,
) -> ColumnTransformer:
    categorical_columns = list(categorical_predictors)
    if include_product_id:
        categorical_columns.append(PRODUCT_ID_COLUMN)
    return ColumnTransformer(
        transformers=[
            (
                "numeric",
                Pipeline(steps=[("imputer", SimpleImputer(strategy="median"))]),
                numeric_predictors,
            ),
            (
                "categorical",
                Pipeline(
                    steps=[
                        ("imputer", SimpleImputer(strategy="most_frequent")),
                        (
                            "ordinal",
                            OrdinalEncoder(
                                handle_unknown="use_encoded_value",
                                unknown_value=-1,
                                encoded_missing_value=-2,
                            ),
                        ),
                    ]
                ),
                categorical_columns,
            ),
        ],
        remainder="drop",
        sparse_threshold=0.0,
        verbose_feature_names_out=False,
    )


def prepare_source_frame(
    frame: pd.DataFrame,
    numeric_predictors: list[str],
    categorical_predictors: list[str],
    include_product_id: bool,
) -> pd.DataFrame:
    columns = numeric_predictors + categorical_predictors + (
        [PRODUCT_ID_COLUMN] if include_product_id else []
    )
    prepared = frame[columns].copy()
    for column in numeric_predictors:
        prepared[column] = pd.to_numeric(prepared[column], errors="coerce")
    for column in categorical_predictors:
        prepared[column] = (
            prepared[column].astype("string").fillna("__MISSING__").astype(str)
        )
    if include_product_id:
        prepared[PRODUCT_ID_COLUMN] = (
            prepared[PRODUCT_ID_COLUMN]
            .astype("string")
            .fillna("__MISSING_PRODUCT__")
            .astype(str)
        )
    return prepared


def instantiate_model(family_method: str, config: dict):
    parameters = {key: value for key, value in config.items() if key != "ConfigID"}
    if family_method == "NESTED_TUNED_CATBOOST_RMSE_CORE53":
        return CatBoostRegressor(
            loss_function="RMSE",
            random_seed=RANDOM_SEED,
            verbose=False,
            allow_writing_files=False,
            thread_count=MAX_THREADS,
            **parameters,
        )
    if family_method in {
        "NESTED_TUNED_XGBOOST_SQUARED_CORE53",
        "NESTED_TUNED_XGBOOST_SQUARED_PRODUCT_AWARE",
    }:
        return XGBRegressor(
            objective="reg:squarederror",
            subsample=0.85,
            colsample_bytree=0.85,
            reg_lambda=2.0,
            reg_alpha=0.0,
            n_jobs=MAX_THREADS,
            random_state=RANDOM_SEED,
            verbosity=0,
            **parameters,
        )
    raise ValueError(f"Unsupported family method: {family_method}")


def fit_deployment_models(
    training_main: pd.DataFrame,
    configurations: dict[str, dict],
    numeric_predictors: list[str],
    categorical_predictors: list[str],
) -> tuple[dict, list[dict], ColumnTransformer, ColumnTransformer]:
    fitted: dict = {}
    audits = []

    core_preprocessor = make_preprocessor(
        numeric_predictors, categorical_predictors, include_product_id=False
    )
    X_core = core_preprocessor.fit_transform(
        prepare_source_frame(
            training_main,
            numeric_predictors,
            categorical_predictors,
            include_product_id=False,
        )
    )

    product_preprocessor = make_preprocessor(
        numeric_predictors, categorical_predictors, include_product_id=True
    )
    X_product = product_preprocessor.fit_transform(
        prepare_source_frame(
            training_main,
            numeric_predictors,
            categorical_predictors,
            include_product_id=True,
        )
    )

    y_train = training_main[TARGET_COLUMN].to_numpy(dtype=float)

    for family_method, configuration in configurations.items():
        model = instantiate_model(family_method, configuration)
        include_product = (
            family_method == "NESTED_TUNED_XGBOOST_SQUARED_PRODUCT_AWARE"
        )
        X_train = X_product if include_product else X_core
        preprocessor = product_preprocessor if include_product else core_preprocessor
        start = time.perf_counter()
        model.fit(X_train, y_train)
        fit_seconds = float(time.perf_counter() - start)
        component = FAMILY_TO_COMPONENT[family_method]
        fitted[component] = {
            "model": model,
            "preprocessor": preprocessor,
            "include_product_id": include_product,
            "family_method": family_method,
            "config": configuration,
        }
        audits.append(
            {
                "CandidateMethod": family_method,
                "Component": component,
                "ConfigID": configuration.get("ConfigID", ""),
                "ConfigJSON": json.dumps(configuration, sort_keys=True),
                "TrainingRows": int(len(training_main)),
                "TrainingStart": training_main[DATE_COLUMN].min(),
                "TrainingEnd": training_main[DATE_COLUMN].max(),
                "TransformedFeatureCount": int(X_train.shape[1]),
                "FitSeconds": fit_seconds,
            }
        )

    if set(fitted) != set(BASE_COMPONENTS) - {"ROLLING_MEAN_5"}:
        raise AssertionError("The fitted advanced component set is incomplete.")

    return fitted, audits, core_preprocessor, product_preprocessor


def predict_base_components(
    main_features: pd.DataFrame,
    fitted_models: dict,
    numeric_predictors: list[str],
    categorical_predictors: list[str],
) -> dict[str, np.ndarray]:
    components = {
        "ROLLING_MEAN_5": np.clip(
            pd.to_numeric(
                main_features["PastNormalDemandRollingMean_5"], errors="coerce"
            ).to_numpy(dtype=float),
            0.0,
            None,
        )
    }
    transformed_cache = {}
    for component, fitted in fitted_models.items():
        cache_key = "PRODUCT" if fitted["include_product_id"] else "CORE"
        if cache_key not in transformed_cache:
            transformed_cache[cache_key] = fitted["preprocessor"].transform(
                prepare_source_frame(
                    main_features,
                    numeric_predictors,
                    categorical_predictors,
                    fitted["include_product_id"],
                )
            )
        prediction = np.asarray(
            fitted["model"].predict(transformed_cache[cache_key]), dtype=float
        )
        components[component] = np.clip(prediction, 0.0, None)
    if set(components) != set(BASE_COMPONENTS):
        raise AssertionError("The base-component prediction set is incomplete.")
    return components


def predict_routed_system(
    feature_frame: pd.DataFrame,
    operational_method: str,
    median_components: list[str],
    fitted_models: dict,
    numeric_predictors: list[str],
    categorical_predictors: list[str],
    hierarchy_statistics: dict,
) -> pd.DataFrame:
    output = feature_frame.copy().reset_index(drop=True)
    predictions = fallback_predictions(output, hierarchy_statistics)
    method_used = output["RecursiveForecastRoute"].map(
        {
            "COLD_START_FALLBACK": "RECENT_MEDIAN5_WITH_BACKOFF",
            "LOW_DEMAND_FALLBACK": "RECENT_MEDIAN5_WITH_BACKOFF",
            "ZERO_HISTORY_FALLBACK": "EXPANDING_MEAN_WITH_BACKOFF",
        }
    ).astype("string")

    for component in BASE_COMPONENTS:
        output[f"BasePrediction_{component}"] = np.nan

    main_mask = output["RecursiveForecastRoute"].astype(str) == "MAIN_MODEL"
    if main_mask.any():
        main_features = output.loc[main_mask].copy()
        components = predict_base_components(
            main_features,
            fitted_models,
            numeric_predictors,
            categorical_predictors,
        )
        for component, values in components.items():
            output.loc[main_mask, f"BasePrediction_{component}"] = values

        if operational_method == "ROLLING_MEAN_5":
            main_prediction = components["ROLLING_MEAN_5"]
        elif operational_method == "NESTED_MEDIAN_ENSEMBLE":
            unknown = sorted(set(median_components) - set(BASE_COMPONENTS))
            if unknown:
                raise AssertionError(
                    f"Unknown median components in ND08 contract: {unknown}"
                )
            matrix = np.column_stack(
                [components[component] for component in median_components]
            )
            main_prediction = np.median(matrix, axis=1)
        else:
            raise ValueError(f"Unsupported operational method: {operational_method}")

        predictions[main_mask.to_numpy()] = np.clip(main_prediction, 0.0, None)
        method_used.loc[main_mask] = operational_method

    if not np.isfinite(predictions).all():
        raise AssertionError("The routed forecast contains non-finite predictions.")
    if (predictions < 0).any():
        raise AssertionError("The routed forecast contains negative predictions.")

    output["ForecastRoute"] = output["RecursiveForecastRoute"].astype(str)
    output["PredictedNormalDemand"] = predictions
    output["MethodUsed"] = method_used.astype(str)
    return output


# =============================================================================
# FUTURE ROW AND RECURSIVE PATH HELPERS
# =============================================================================

def build_product_catalogue(
    history: pd.DataFrame,
    apply_user_filter: bool = True,
) -> pd.DataFrame:
    ordered = history.sort_values(
        [PRODUCT_ID_COLUMN, DATE_COLUMN, SEQUENCE_COLUMN], kind="mergesort"
    )
    latest = ordered.groupby(PRODUCT_ID_COLUMN, sort=False).tail(1).copy()
    first_sequence = ordered.groupby(PRODUCT_ID_COLUMN)[SEQUENCE_COLUMN].min()
    first_date = ordered.groupby(PRODUCT_ID_COLUMN)[DATE_COLUMN].min()
    cumulative = ordered.groupby(PRODUCT_ID_COLUMN)[TARGET_COLUMN].sum()
    positive_days = ordered.groupby(PRODUCT_ID_COLUMN)[TARGET_COLUMN].apply(
        lambda values: int((values > 0).sum())
    )

    catalogue = latest[PRODUCT_METADATA_COLUMNS].copy()
    catalogue[PRODUCT_ID_COLUMN] = catalogue[PRODUCT_ID_COLUMN].astype(str)
    catalogue["FirstObservedOperatingDaySequence"] = catalogue[
        PRODUCT_ID_COLUMN
    ].map(first_sequence)
    catalogue["FirstPanelDate"] = catalogue[PRODUCT_ID_COLUMN].map(first_date)
    catalogue["HistoricalNormalDemand"] = catalogue[PRODUCT_ID_COLUMN].map(cumulative)
    catalogue["HistoricalPositiveDemandDays"] = catalogue[PRODUCT_ID_COLUMN].map(
        positive_days
    )
    catalogue["IncludeInForecast"] = True

    if apply_user_filter and ACTIVE_PRODUCT_IDS is not None:
        requested = {str(value) for value in ACTIVE_PRODUCT_IDS}
        available = set(catalogue[PRODUCT_ID_COLUMN])
        unknown = sorted(requested - available)
        if unknown:
            raise ValueError(
                "ACTIVE_PRODUCT_IDS contains unknown products:\n"
                + "\n".join(f"- {value}" for value in unknown)
            )
        catalogue["IncludeInForecast"] = catalogue[PRODUCT_ID_COLUMN].isin(requested)

    catalogue = catalogue.sort_values(
        ["IncludeInForecast", "HistoricalNormalDemand", PRODUCT_ID_COLUMN],
        ascending=[False, False, True],
        kind="mergesort",
    ).reset_index(drop=True)

    if not catalogue["IncludeInForecast"].any():
        raise AssertionError("The active product catalogue is empty.")
    return catalogue


def build_future_base_rows(
    active_catalogue: pd.DataFrame,
    calendar_row: pd.Series,
) -> pd.DataFrame:
    frame = active_catalogue.loc[active_catalogue["IncludeInForecast"]].copy()
    frame = frame.reset_index(drop=True)
    current_sequence = int(calendar_row[SEQUENCE_COLUMN])

    for column in [
        DATE_COLUMN,
        SEQUENCE_COLUMN,
        "Year",
        "Month",
        "Quarter",
        "DayOfWeekNumber",
        "ISOYear",
        "ISOWeek",
        "DayOfYear",
        "IsWeekend",
        "DaysSincePreviousOperatingDate",
        "IsConsecutiveCalendarDay",
        "CalendarPolicy",
    ]:
        frame[column] = calendar_row[column]

    frame["ProductAgeOperatingDays"] = (
        current_sequence
        - pd.to_numeric(
            frame["FirstObservedOperatingDaySequence"], errors="raise"
        ).astype(int)
    ).clip(lower=0)
    frame[TARGET_COLUMN] = np.nan
    return frame


def run_recursive_path(
    label: str,
    future_calendar: pd.DataFrame,
    requested_dates: set[pd.Timestamp],
    operational_method: str,
    base_state: dict[str, dict[str, list]],
    active_catalogue: pd.DataFrame,
    fitted_models: dict,
    numeric_predictors: list[str],
    categorical_predictors: list[str],
    hierarchy_statistics: dict,
    median_components: list[str],
    cutoff_date: pd.Timestamp,
) -> tuple[pd.DataFrame, list[dict]]:
    state = copy_history_state(base_state)
    parts = []
    audit_records = []

    for horizon, calendar_row in enumerate(
        future_calendar.sort_values(DATE_COLUMN).itertuples(index=False), start=1
    ):
        calendar_series = pd.Series(calendar_row._asdict())
        base_rows = build_future_base_rows(active_catalogue, calendar_series)
        feature_rows = build_recursive_feature_frame(base_rows, state)
        forecast_rows = predict_routed_system(
            feature_rows,
            operational_method,
            median_components,
            fitted_models,
            numeric_predictors,
            categorical_predictors,
            hierarchy_statistics,
        )
        forecast_date = pd.Timestamp(calendar_series[DATE_COLUMN]).normalize()
        forecast_rows["ForecastPath"] = label
        forecast_rows["ForecastOriginCutoffDate"] = cutoff_date
        forecast_rows["HorizonOperatingDays"] = horizon
        forecast_rows["OperationalMethod"] = operational_method
        forecast_rows["IsRequestedOutputDate"] = forecast_date in requested_dates
        forecast_rows["FutureActualDemandUsed"] = False
        parts.append(forecast_rows)

        route_counts = forecast_rows["RecursiveForecastRoute"].value_counts()
        audit_records.append(
            {
                "ForecastPath": label,
                "Date": forecast_date,
                "HorizonOperatingDays": horizon,
                "OperationalMethod": operational_method,
                "ProductsForecast": int(len(forecast_rows)),
                "RequestedOutputDate": forecast_date in requested_dates,
                "RestaurantNormalDemandForecast": float(
                    forecast_rows["PredictedNormalDemand"].sum()
                ),
                "MainModelRows": int(route_counts.get("MAIN_MODEL", 0)),
                "LowDemandFallbackRows": int(
                    route_counts.get("LOW_DEMAND_FALLBACK", 0)
                ),
                "ColdStartFallbackRows": int(
                    route_counts.get("COLD_START_FALLBACK", 0)
                ),
                "ZeroHistoryFallbackRows": int(
                    route_counts.get("ZERO_HISTORY_FALLBACK", 0)
                ),
                "PredictionsInsertedBeforeNextDay": True,
            }
        )

        append_predictions_to_state(
            state,
            forecast_rows,
            forecast_rows["PredictedNormalDemand"].to_numpy(dtype=float),
        )

    result = pd.concat(parts, ignore_index=True)
    return result, audit_records


def select_representative_base_configurations(
    selections: pd.DataFrame,
) -> tuple[dict[str, dict], pd.DataFrame]:
    validate_required_columns(
        selections,
        {
            "OuterFold",
            "CandidateMethod",
            "SelectedConfigID",
            "SelectedConfigJSON",
            "SelectedInnerWAPEPercentage",
        },
        "ND07 outer selections",
    )
    if set(selections["CandidateMethod"].astype(str)) != EXPECTED_FAMILIES:
        raise AssertionError("ND07 outer selections do not contain all base families.")

    configurations = {}
    records = []
    for family_method in sorted(EXPECTED_FAMILIES):
        rows = selections.loc[
            selections["CandidateMethod"].astype(str) == family_method
        ].copy()
        selected_json = mode_with_inner_wape_tie_break(
            rows["SelectedConfigJSON"], rows["SelectedInnerWAPEPercentage"]
        )
        configuration = json.loads(selected_json)
        configurations[family_method] = configuration
        matching = rows.loc[rows["SelectedConfigJSON"].astype(str) == selected_json]
        records.append(
            {
                "SettingType": "BASE_MODEL_CONFIGURATION",
                "CandidateMethod": family_method,
                "Component": FAMILY_TO_COMPONENT[family_method],
                "RepresentativeSetting": selected_json,
                "OccurrencesAcrossFiveOuterFolds": int(len(matching)),
                "MeanSelectedInnerWAPEPercentage": float(
                    matching["SelectedInnerWAPEPercentage"].mean()
                ),
                "SelectionEvidence": "INNER_VALIDATION_ONLY",
            }
        )
    return configurations, pd.DataFrame(records)


def build_manifest(root: Path, exclude_control: bool = True) -> pd.DataFrame:
    records = []
    for path in sorted(root.rglob("*")):
        if not path.is_file():
            continue
        relative = path.relative_to(root)
        if exclude_control and relative.parts and relative.parts[0] == "07_control":
            continue
        records.append(
            {
                "RelativePath": str(relative),
                "Bytes": int(path.stat().st_size),
                "SHA256": sha256_file(path),
            }
        )
    return pd.DataFrame(records)


# =============================================================================
# PREFLIGHT AND INPUT VERIFICATION
# =============================================================================

required_inputs = [
    ALL_ROUTES_PATH,
    CORE_PREDICTOR_LIST_PATH,
    ND04_SELECTED_METHODS_PATH,
    ND07_OUTER_SELECTIONS_PATH,
    ND07E_OUTER_SELECTIONS_PATH,
    ND08_SYSTEM_CONTRACT_PATH,
    *CHECKPOINT_PATHS.values(),
]
missing_inputs = [path for path in required_inputs if not path.is_file()]
if missing_inputs:
    raise FileNotFoundError(
        "ND09 required inputs are missing:\n"
        + "\n".join(f"- {path}" for path in missing_inputs)
    )

checkpoint_hashes = {
    name: sha256_file(path) for name, path in CHECKPOINT_PATHS.items()
}
for name, expected_hash in EXPECTED_CHECKPOINT_HASHES.items():
    if checkpoint_hashes[name] != expected_hash:
        raise AssertionError(
            f"{name} checkpoint hash mismatch.\n"
            f"Expected: {expected_hash}\n"
            f"Actual:   {checkpoint_hashes[name]}"
        )

if ND09_ROOT.exists():
    if not ALLOW_OVERWRITE:
        raise FileExistsError(
            "ND09 output already exists. No files were changed:\n"
            f"{ND09_ROOT}"
        )
    shutil.rmtree(ND09_ROOT)

for path in [
    TOP_LEVEL_CHECKPOINT_PATH,
    TOP_LEVEL_CHECKPOINT_SHA_PATH,
    TOP_LEVEL_LOCK_PATH,
]:
    if path.exists():
        if not ALLOW_OVERWRITE:
            raise FileExistsError(
                "An ND09 top-level control artifact already exists. "
                "No files were changed:\n"
                f"{path}"
            )
        path.unlink()

protected_input_hashes_before = {str(path): sha256_file(path) for path in required_inputs}
STAGING_ROOT = ND09_ROOT.parent / f".ND09_staging_{uuid.uuid4().hex}"
STAGING_ROOT.mkdir(parents=True, exist_ok=False)


# =============================================================================
# MAIN EXECUTION
# =============================================================================

try:
    all_routes = pd.read_csv(ALL_ROUTES_PATH, low_memory=False)
    core_predictor_list = pd.read_csv(CORE_PREDICTOR_LIST_PATH, low_memory=False)
    selected_route_methods = pd.read_csv(
        ND04_SELECTED_METHODS_PATH, low_memory=False
    )
    nd07_outer_selections = pd.read_csv(
        ND07_OUTER_SELECTIONS_PATH, low_memory=False
    )
    nd07e_outer_selections = pd.read_csv(
        ND07E_OUTER_SELECTIONS_PATH, low_memory=False
    )
    nd08_contract = json.loads(ND08_SYSTEM_CONTRACT_PATH.read_text(encoding="utf-8"))

    all_routes[DATE_COLUMN] = pd.to_datetime(all_routes[DATE_COLUMN], errors="raise")
    all_routes[PRODUCT_ID_COLUMN] = all_routes[PRODUCT_ID_COLUMN].astype(str)

    validate_required_columns(
        all_routes,
        {
            DATE_COLUMN,
            PRODUCT_ID_COLUMN,
            PRODUCT_NAME_COLUMN,
            TARGET_COLUMN,
            ROUTE_COLUMN,
            FAMILY_COLUMN,
            SEQUENCE_COLUMN,
            "ProductFirstObservedDate",
            *BASE_NUMERIC_PREDICTORS,
            *BASE_CATEGORICAL_PREDICTORS,
            *HISTORICAL_DEMAND_PREDICTORS,
        },
        "ND03 pre-March all-route development dataset",
    )
    validate_required_columns(
        core_predictor_list,
        {"PredictorOrder", "Predictor", "PredictorType"},
        "ND03 core predictor list",
    )
    validate_required_columns(
        selected_route_methods,
        {"ForecastRoute", "SelectedMethod"},
        "ND04 selected route methods",
    )

    if len(all_routes) != EXPECTED_ALL_ROUTE_ROWS:
        raise AssertionError(
            f"Expected {EXPECTED_ALL_ROUTE_ROWS:,} pre-March rows; found {len(all_routes):,}."
        )
    if all_routes[PRODUCT_ID_COLUMN].nunique() != EXPECTED_PRODUCTS:
        raise AssertionError(
            f"Expected {EXPECTED_PRODUCTS} products; found "
            f"{all_routes[PRODUCT_ID_COLUMN].nunique()}."
        )
    if not (all_routes[DATE_COLUMN] < DEVELOPMENT_END_EXCLUSIVE).all():
        raise AssertionError("March or later rows were found in the ND09 training source.")

    core_predictor_list = core_predictor_list.sort_values("PredictorOrder")
    core_predictors = core_predictor_list["Predictor"].astype(str).tolist()
    numeric_predictors = core_predictor_list.loc[
        core_predictor_list["PredictorType"].astype(str) == "NUMERIC", "Predictor"
    ].astype(str).tolist()
    categorical_predictors = core_predictor_list.loc[
        core_predictor_list["PredictorType"].astype(str) == "CATEGORICAL", "Predictor"
    ].astype(str).tolist()

    if len(core_predictors) != EXPECTED_CORE_PREDICTORS:
        raise AssertionError("The core predictor count is not 53.")
    if len(numeric_predictors) != EXPECTED_NUMERIC_PREDICTORS:
        raise AssertionError("The numeric predictor count is not 45.")
    if len(categorical_predictors) != EXPECTED_CATEGORICAL_PREDICTORS:
        raise AssertionError("The categorical predictor count is not 8.")
    if numeric_predictors != BASE_NUMERIC_PREDICTORS + HISTORICAL_DEMAND_PREDICTORS:
        raise AssertionError("The ND09 numeric predictor order does not match ND03.")
    if categorical_predictors != BASE_CATEGORICAL_PREDICTORS:
        raise AssertionError("The ND09 categorical predictor order does not match ND03.")

    route_method_map = dict(
        zip(
            selected_route_methods["ForecastRoute"].astype(str),
            selected_route_methods["SelectedMethod"].astype(str),
        )
    )
    if route_method_map != EXPECTED_ROUTE_METHODS:
        raise AssertionError(
            "ND04 route methods differ from the frozen expected fallback contract."
        )

    daily_product_method = str(nd08_contract["DailyProductMethod"])
    week_planning_method = str(nd08_contract["MondayOriginWeeklyPlanningMethod"])
    updated_week_method = str(nd08_contract["RollingDailyUpdatedWeeklyMethod"])
    if daily_product_method != EXPECTED_DAILY_METHOD:
        raise AssertionError("ND08 daily method is not the selected median ensemble.")
    if week_planning_method != EXPECTED_WEEK_METHOD:
        raise AssertionError("ND08 week-start method is not ROLLING_MEAN_5.")
    if updated_week_method != EXPECTED_UPDATED_WEEK_METHOD:
        raise AssertionError("ND08 daily-updated weekly method is not ROLLING_MEAN_5.")

    representative_settings = nd08_contract.get("RepresentativeSettings", {})
    median_components = representative_settings.get(
        "NESTED_MEDIAN_ENSEMBLE", {}
    ).get("Components", [])
    if not median_components:
        raise AssertionError("ND08 contract does not contain a representative median subset.")
    if set(median_components) - set(BASE_COMPONENTS):
        raise AssertionError("ND08 representative median subset is invalid.")

    representative_configs, representative_config_audit = (
        select_representative_base_configurations(nd07_outer_selections)
    )

    # Record the ND07E representative median evidence for the audit.
    if "SettingKey" in nd07e_outer_selections.columns:
        nd07e_outer_selections["EnsembleSettingKey"] = (
            nd07e_outer_selections["SettingKey"].astype(str)
        )
        nd07e_schema = "SETTING_KEY"
    elif {"MedianSubset", "WeightTuple"}.issubset(nd07e_outer_selections.columns):
        nd07e_outer_selections["EnsembleSettingKey"] = np.where(
            nd07e_outer_selections["EnsembleType"].astype(str) == "MEDIAN",
            nd07e_outer_selections["MedianSubset"].astype(str),
            nd07e_outer_selections["WeightTuple"].astype(str),
        )
        nd07e_schema = "LEGACY_SPLIT_COLUMNS"
    else:
        raise AssertionError("Unsupported ND07E ensemble-setting schema.")

    median_rows = nd07e_outer_selections.loc[
        nd07e_outer_selections["CandidateMethod"].astype(str)
        == "NESTED_MEDIAN_ENSEMBLE"
    ].copy()
    representative_median_key = "|".join(median_components)
    representative_config_audit = pd.concat(
        [
            representative_config_audit,
            pd.DataFrame(
                [
                    {
                        "SettingType": "MEDIAN_ENSEMBLE_SUBSET",
                        "CandidateMethod": "NESTED_MEDIAN_ENSEMBLE",
                        "Component": "|".join(median_components),
                        "RepresentativeSetting": representative_median_key,
                        "OccurrencesAcrossFiveOuterFolds": int(
                            (
                                median_rows["EnsembleSettingKey"].astype(str)
                                == representative_median_key
                            ).sum()
                        ),
                        "MeanSelectedInnerWAPEPercentage": float(
                            median_rows.loc[
                                median_rows["EnsembleSettingKey"].astype(str)
                                == representative_median_key,
                                "SelectedInnerWAPEPercentage",
                            ].mean()
                        ),
                        "SelectionEvidence": "INNER_VALIDATION_ONLY",
                    }
                ]
            ),
        ],
        ignore_index=True,
    )

    cutoff_date = all_routes[DATE_COLUMN].max().normalize()
    cutoff_sequence = int(all_routes.loc[
        all_routes[DATE_COLUMN] == cutoff_date, SEQUENCE_COLUMN
    ].max())
    cutoff_target_total = float(
        all_routes.loc[all_routes[DATE_COLUMN] == cutoff_date, TARGET_COLUMN].sum()
    )

    closed_dates = parse_iso_dates(CLOSED_DATES, "closure")
    extra_dates = parse_iso_dates(EXTRA_OPERATING_DATES, "extra operating")
    overlap_dates = closed_dates & extra_dates
    if overlap_dates:
        raise ValueError(
            "A date cannot be both closed and an extra operating date: "
            + ", ".join(str(value.date()) for value in sorted(overlap_dates))
        )

    default_daily_date = next_operating_date(cutoff_date, closed_dates, extra_dates)
    requested_daily_date = (
        default_daily_date
        if REQUESTED_DAILY_DATE is None
        else pd.Timestamp(REQUESTED_DAILY_DATE).normalize()
    )
    if requested_daily_date <= cutoff_date:
        raise ValueError("REQUESTED_DAILY_DATE must be after the data cutoff.")
    if not is_operating_date(requested_daily_date, closed_dates, extra_dates):
        raise ValueError(
            "REQUESTED_DAILY_DATE is not an operating date under the configured calendar."
        )

    default_week_start = monday_on_or_after(default_daily_date)
    requested_week_start = (
        default_week_start
        if REQUESTED_WEEK_START is None
        else pd.Timestamp(REQUESTED_WEEK_START).normalize()
    )
    if requested_week_start.weekday() != 0:
        raise ValueError("REQUESTED_WEEK_START must be a Monday.")
    if requested_week_start <= cutoff_date:
        raise ValueError("REQUESTED_WEEK_START must be after the data cutoff.")
    requested_week_end = requested_week_start + pd.Timedelta(days=4)

    maximum_requested_date = max(requested_daily_date, requested_week_end)
    future_calendar = generate_future_calendar(
        cutoff_date,
        cutoff_sequence,
        maximum_requested_date,
        closed_dates,
        extra_dates,
    )

    if requested_daily_date not in set(future_calendar[DATE_COLUMN]):
        raise AssertionError("Requested daily date is missing from the future calendar.")

    week_requested_dates = {
        date
        for date in pd.date_range(requested_week_start, requested_week_end, freq="D")
        if is_operating_date(date, closed_dates, extra_dates)
    }
    if not week_requested_dates:
        raise AssertionError("The requested planning week has no operating dates.")

    active_catalogue = build_product_catalogue(all_routes)
    active_count = int(active_catalogue["IncludeInForecast"].sum())

    daily_calendar = future_calendar.loc[
        future_calendar[DATE_COLUMN] <= requested_daily_date
    ].copy()
    daily_horizon = int(len(daily_calendar))
    daily_operational_method = (
        daily_product_method if daily_horizon == 1 else week_planning_method
    )
    daily_method_reason = (
        "ND08_NEXT_OPERATING_DAY_METHOD"
        if daily_horizon == 1
        else "ND08_MULTI_STEP_RECURSIVE_PLANNING_METHOD"
    )

    week_calendar = future_calendar.loc[
        future_calendar[DATE_COLUMN] <= requested_week_end
    ].copy()
    week_horizon = int(len(week_calendar))

    maximum_horizon = max(daily_horizon, week_horizon)
    if maximum_horizon > MAXIMUM_ALLOWED_RECURSIVE_HORIZON:
        raise ValueError(
            f"Requested path requires {maximum_horizon} operating-day steps; "
            f"the hard maximum is {MAXIMUM_ALLOWED_RECURSIVE_HORIZON}."
        )
    long_horizon_extrapolation = maximum_horizon > MAX_VALIDATED_RECURSIVE_HORIZON
    if long_horizon_extrapolation and not ALLOW_LONG_HORIZON_EXTRAPOLATION:
        raise ValueError(
            f"Requested path requires {maximum_horizon} operating-day recursive steps. "
            f"ND08 validated only {MAX_VALIDATED_RECURSIVE_HORIZON}. Set "
            "ALLOW_LONG_HORIZON_EXTRAPOLATION=True only when the extrapolation "
            "warning is acceptable."
        )

    # -------------------------------------------------------------------------
    # Future-feature generator replay check on the latest suitable historical day.
    # -------------------------------------------------------------------------
    available_dates = sorted(all_routes[DATE_COLUMN].unique())
    replay_date = None
    for candidate in reversed(available_dates):
        candidate = pd.Timestamp(candidate).normalize()
        prior = all_routes.loc[all_routes[DATE_COLUMN] < candidate]
        current = all_routes.loc[all_routes[DATE_COLUMN] == candidate]
        if prior.empty:
            continue
        if set(current[PRODUCT_ID_COLUMN]).issubset(set(prior[PRODUCT_ID_COLUMN])):
            replay_date = candidate
            break
    if replay_date is None:
        raise AssertionError("No valid historical date was available for feature replay.")

    replay_history = all_routes.loc[all_routes[DATE_COLUMN] < replay_date].copy()
    replay_actual = all_routes.loc[all_routes[DATE_COLUMN] == replay_date].copy()
    replay_cutoff = replay_history[DATE_COLUMN].max().normalize()
    replay_sequence = int(replay_history[SEQUENCE_COLUMN].max())
    replay_calendar = generate_future_calendar(
        replay_cutoff,
        replay_sequence,
        replay_date,
        set(),
        set(),
    )
    replay_catalogue = build_product_catalogue(
        replay_history,
        apply_user_filter=False,
    )
    replay_active_ids = set(replay_actual[PRODUCT_ID_COLUMN])
    replay_catalogue["IncludeInForecast"] = replay_catalogue[
        PRODUCT_ID_COLUMN
    ].isin(replay_active_ids)
    replay_rows = build_future_base_rows(
        replay_catalogue,
        replay_calendar.iloc[-1],
    )
    replay_features = build_recursive_feature_frame(
        replay_rows,
        build_history_state(replay_history),
    )
    replay_compare = replay_actual.merge(
        replay_features,
        on=[DATE_COLUMN, PRODUCT_ID_COLUMN],
        suffixes=("_Stored", "_Rebuilt"),
        how="inner",
        validate="one_to_one",
    )

    replay_records = []
    replay_numeric_columns = [
        *BASE_NUMERIC_PREDICTORS,
        *HISTORICAL_DEMAND_PREDICTORS,
    ]
    for column in replay_numeric_columns:
        stored = pd.to_numeric(
            replay_compare[f"{column}_Stored"], errors="coerce"
        ).to_numpy(dtype=float)
        rebuilt = pd.to_numeric(
            replay_compare[f"{column}_Rebuilt"], errors="coerce"
        ).to_numpy(dtype=float)
        both_nan = np.isnan(stored) & np.isnan(rebuilt)
        difference = np.abs(stored - rebuilt)
        difference[both_nan] = 0.0
        max_difference = float(np.nanmax(difference)) if len(difference) else 0.0
        passed = bool(np.all(both_nan | np.isclose(stored, rebuilt, atol=1e-10, rtol=1e-10)))
        replay_records.append(
            {
                "ReplayDate": replay_date,
                "Field": column,
                "RowsCompared": int(len(replay_compare)),
                "MaximumAbsoluteDifference": max_difference,
                "Passed": passed,
            }
        )

    route_passed = bool(
        (
            replay_compare[ROUTE_COLUMN].astype(str).to_numpy()
            == replay_compare["RecursiveForecastRoute"].astype(str).to_numpy()
        ).all()
    )
    replay_records.append(
        {
            "ReplayDate": replay_date,
            "Field": "ForecastRoute",
            "RowsCompared": int(len(replay_compare)),
            "MaximumAbsoluteDifference": np.nan,
            "Passed": route_passed,
        }
    )
    feature_replay_audit = pd.DataFrame(replay_records)
    if not feature_replay_audit["Passed"].all():
        failed = feature_replay_audit.loc[~feature_replay_audit["Passed"]]
        raise AssertionError(
            "ND09 future-feature replay failed:\n" + failed.to_string(index=False)
        )

    # -------------------------------------------------------------------------
    # Fit the frozen demonstration/deployment model artifacts.
    # -------------------------------------------------------------------------
    training_main = all_routes.loc[
        all_routes[ROUTE_COLUMN].astype(str) == "MAIN_MODEL"
    ].copy()
    if training_main.empty:
        raise AssertionError("The pre-March main-model training set is empty.")

    total_start = time.perf_counter()
    fitted_models, fit_audit_records, core_preprocessor, product_preprocessor = (
        fit_deployment_models(
            training_main,
            representative_configs,
            numeric_predictors,
            categorical_predictors,
        )
    )
    model_fit_seconds = float(time.perf_counter() - total_start)
    model_fit_audit = pd.DataFrame(fit_audit_records)

    hierarchy_statistics = build_hierarchy_statistics(all_routes)
    base_state = build_history_state(all_routes)

    daily_path, daily_path_audit = run_recursive_path(
        label="ARBITRARY_DATE",
        future_calendar=daily_calendar,
        requested_dates={requested_daily_date},
        operational_method=daily_operational_method,
        base_state=base_state,
        active_catalogue=active_catalogue,
        fitted_models=fitted_models,
        numeric_predictors=numeric_predictors,
        categorical_predictors=categorical_predictors,
        hierarchy_statistics=hierarchy_statistics,
        median_components=median_components,
        cutoff_date=cutoff_date,
    )

    week_path, week_path_audit = run_recursive_path(
        label="MONDAY_ORIGIN_WEEK",
        future_calendar=week_calendar,
        requested_dates=week_requested_dates,
        operational_method=week_planning_method,
        base_state=base_state,
        active_catalogue=active_catalogue,
        fitted_models=fitted_models,
        numeric_predictors=numeric_predictors,
        categorical_predictors=categorical_predictors,
        hierarchy_statistics=hierarchy_statistics,
        median_components=median_components,
        cutoff_date=cutoff_date,
    )

    daily_product_forecast = daily_path.loc[
        daily_path[DATE_COLUMN] == requested_daily_date
    ].copy()
    daily_product_forecast["ForecastUseCase"] = (
        "NEXT_OPERATING_DAY"
        if daily_horizon == 1
        else "MULTI_STEP_ARBITRARY_DATE"
    )
    daily_product_forecast["MethodSelectionReason"] = daily_method_reason

    daily_restaurant_forecast = pd.DataFrame(
        [
            {
                "Date": requested_daily_date,
                "ForecastUseCase": daily_product_forecast["ForecastUseCase"].iloc[0],
                "OperationalMethod": daily_operational_method,
                "ProductsForecast": int(len(daily_product_forecast)),
                "PredictedRestaurantNormalDemand": float(
                    daily_product_forecast["PredictedNormalDemand"].sum()
                ),
                "ForecastOriginCutoffDate": cutoff_date,
                "HorizonOperatingDays": daily_horizon,
            }
        ]
    )

    week_daily_product_forecast = week_path.loc[
        week_path[DATE_COLUMN].isin(week_requested_dates)
    ].copy()
    week_daily_product_forecast["WeekStart"] = requested_week_start
    week_daily_product_forecast["WeekEnd"] = requested_week_end
    week_daily_product_forecast["ForecastUseCase"] = "MONDAY_ORIGIN_WEEK_AHEAD"

    week_product_totals = (
        week_daily_product_forecast.groupby(
            [PRODUCT_ID_COLUMN, PRODUCT_NAME_COLUMN, FAMILY_COLUMN],
            dropna=False,
            as_index=False,
        )["PredictedNormalDemand"]
        .sum()
        .rename(columns={"PredictedNormalDemand": "PredictedWeekNormalDemand"})
    )
    week_product_totals["WeekStart"] = requested_week_start
    week_product_totals["WeekEnd"] = requested_week_end
    week_product_totals["OperationalMethod"] = week_planning_method
    week_product_totals = week_product_totals.sort_values(
        ["PredictedWeekNormalDemand", PRODUCT_ID_COLUMN],
        ascending=[False, True],
        kind="mergesort",
    ).reset_index(drop=True)

    week_restaurant_daily = (
        week_daily_product_forecast.groupby(DATE_COLUMN, as_index=False)[
            "PredictedNormalDemand"
        ]
        .sum()
        .rename(
            columns={
                "PredictedNormalDemand": "PredictedRestaurantNormalDemand"
            }
        )
    )
    week_restaurant_daily["WeekStart"] = requested_week_start
    week_restaurant_daily["OperationalMethod"] = week_planning_method

    week_restaurant_total = pd.DataFrame(
        [
            {
                "WeekStart": requested_week_start,
                "WeekEnd": requested_week_end,
                "OperatingDatesForecast": int(
                    week_daily_product_forecast[DATE_COLUMN].nunique()
                ),
                "ProductsForecast": active_count,
                "OperationalMethod": week_planning_method,
                "PredictedRestaurantWeekNormalDemand": float(
                    week_daily_product_forecast["PredictedNormalDemand"].sum()
                ),
                "ForecastOriginCutoffDate": cutoff_date,
                "RecursivePathOperatingDays": week_horizon,
            }
        ]
    )

    forecast_path_audit = pd.DataFrame(daily_path_audit + week_path_audit)
    route_summary = (
        pd.concat(
            [
                daily_product_forecast.assign(Output="ARBITRARY_DATE"),
                week_daily_product_forecast.assign(Output="MONDAY_ORIGIN_WEEK"),
            ],
            ignore_index=True,
        )
        .groupby(["Output", "RecursiveForecastRoute", "MethodUsed"], as_index=False)
        .agg(
            Rows=(PRODUCT_ID_COLUMN, "size"),
            PredictedNormalDemand=("PredictedNormalDemand", "sum"),
        )
        .sort_values(["Output", "RecursiveForecastRoute", "MethodUsed"])
        .reset_index(drop=True)
    )

    # -------------------------------------------------------------------------
    # Staged outputs and model artifacts.
    # -------------------------------------------------------------------------
    staged_model_dir = STAGING_ROOT / "01_model_artifacts"
    staged_forecast_dir = STAGING_ROOT / "02_forecasts"
    staged_contract_dir = STAGING_ROOT / "03_contracts"
    staged_audit_dir = STAGING_ROOT / "04_audits"
    staged_figure_dir = STAGING_ROOT / "05_figures"
    staged_report_dir = STAGING_ROOT / "06_reports"
    staged_control_dir = STAGING_ROOT / "07_control"
    for directory in [
        staged_model_dir,
        staged_forecast_dir,
        staged_contract_dir,
        staged_audit_dir,
        staged_figure_dir,
        staged_report_dir,
        staged_control_dir,
    ]:
        directory.mkdir(parents=True, exist_ok=True)

    joblib.dump(core_preprocessor, staged_model_dir / CORE_PREPROCESSOR_PATH.name)
    joblib.dump(product_preprocessor, staged_model_dir / PRODUCT_PREPROCESSOR_PATH.name)
    fitted_models["CATBOOST_CORE53"]["model"].save_model(
        str(staged_model_dir / CATBOOST_MODEL_PATH.name)
    )
    fitted_models["XGBOOST_CORE53"]["model"].save_model(
        str(staged_model_dir / XGBOOST_CORE_MODEL_PATH.name)
    )
    fitted_models["XGBOOST_PRODUCT_AWARE"]["model"].save_model(
        str(staged_model_dir / XGBOOST_PRODUCT_MODEL_PATH.name)
    )

    model_metadata = {
        "StepID": STEP_ID,
        "Status": STATUS,
        "ArtifactRole": "LOCKED_DEMONSTRATION_AND_DEPLOYMENT_ENGINE",
        "TrainingTarget": TARGET_COLUMN,
        "TrainingStart": all_routes[DATE_COLUMN].min(),
        "TrainingEnd": cutoff_date,
        "TrainingRowsAllRoutes": int(len(all_routes)),
        "TrainingRowsMainModel": int(len(training_main)),
        "CorePredictors": core_predictors,
        "NumericPredictors": numeric_predictors,
        "CategoricalPredictors": categorical_predictors,
        "RepresentativeBaseConfigurations": representative_configs,
        "RepresentativeMedianComponents": median_components,
        "DailyProductMethod": daily_product_method,
        "MultiStepPlanningMethod": week_planning_method,
        "FallbackMethods": route_method_map,
        "ModelArtifacts": {
            "CorePreprocessor": CORE_PREPROCESSOR_PATH.name,
            "ProductPreprocessor": PRODUCT_PREPROCESSOR_PATH.name,
            "CatBoostCore53": CATBOOST_MODEL_PATH.name,
            "XGBoostCore53": XGBOOST_CORE_MODEL_PATH.name,
            "XGBoostProductAware": XGBOOST_PRODUCT_MODEL_PATH.name,
        },
        "March2026TargetsUsed": False,
        "FinalUnbiasedFutureEvaluationStillRequired": True,
    }
    write_json(staged_model_dir / MODEL_METADATA_PATH.name, model_metadata)

    forecast_keep_columns = [
        DATE_COLUMN,
        PRODUCT_ID_COLUMN,
        PRODUCT_NAME_COLUMN,
        FAMILY_COLUMN,
        SEQUENCE_COLUMN,
        "RecursiveForecastRoute",
        "ForecastRoute",
        "PriorDemandVolumeSegment",
        "MethodUsed",
        "OperationalMethod",
        "PredictedNormalDemand",
        "ForecastPath",
        "ForecastOriginCutoffDate",
        "HorizonOperatingDays",
        "IsRequestedOutputDate",
        "FutureActualDemandUsed",
        *[f"BasePrediction_{component}" for component in BASE_COMPONENTS],
    ]
    daily_path_save = daily_path[forecast_keep_columns].copy()
    week_path_save = week_path[forecast_keep_columns].copy()

    write_csv(staged_forecast_dir / DAILY_PATH_FORECAST_PATH.name, daily_path_save)
    write_csv(
        staged_forecast_dir / DAILY_PRODUCT_FORECAST_PATH.name,
        daily_product_forecast,
    )
    write_csv(
        staged_forecast_dir / DAILY_RESTAURANT_FORECAST_PATH.name,
        daily_restaurant_forecast,
    )
    write_csv(staged_forecast_dir / WEEK_PATH_FORECAST_PATH.name, week_path_save)
    write_csv(
        staged_forecast_dir / WEEK_DAILY_PRODUCT_FORECAST_PATH.name,
        week_daily_product_forecast,
    )
    write_csv(
        staged_forecast_dir / WEEK_PRODUCT_TOTAL_PATH.name,
        week_product_totals,
    )
    write_csv(
        staged_forecast_dir / WEEK_RESTAURANT_DAILY_PATH.name,
        week_restaurant_daily,
    )
    write_csv(
        staged_forecast_dir / WEEK_RESTAURANT_TOTAL_PATH.name,
        week_restaurant_total,
    )

    write_csv(staged_contract_dir / ACTIVE_CATALOGUE_PATH.name, active_catalogue)
    write_csv(staged_contract_dir / FUTURE_CALENDAR_PATH.name, future_calendar)

    input_hash_audit = pd.DataFrame(
        [
            {
                "Input": str(path),
                "SHA256Before": protected_input_hashes_before[str(path)],
                "CheckpointName": next(
                    (name for name, checkpoint in CHECKPOINT_PATHS.items() if checkpoint == path),
                    "",
                ),
            }
            for path in required_inputs
        ]
    )
    write_csv(staged_audit_dir / INPUT_HASH_AUDIT_PATH.name, input_hash_audit)
    write_csv(staged_audit_dir / MODEL_FIT_AUDIT_PATH.name, model_fit_audit)
    write_csv(
        staged_audit_dir / REPRESENTATIVE_SETTINGS_PATH.name,
        representative_config_audit,
    )
    write_csv(
        staged_audit_dir / FEATURE_REPLAY_AUDIT_PATH.name,
        feature_replay_audit,
    )
    write_csv(
        staged_audit_dir / FORECAST_PATH_AUDIT_PATH.name,
        forecast_path_audit,
    )
    write_csv(staged_audit_dir / ROUTE_SUMMARY_PATH.name, route_summary)

    package_versions = pd.DataFrame(
        [
            {"Package": "python", "Version": platform.python_version()},
            {"Package": "pandas", "Version": pd.__version__},
            {"Package": "numpy", "Version": np.__version__},
            {"Package": "scikit-learn", "Version": sklearn.__version__},
            {"Package": "xgboost", "Version": xgboost.__version__},
            {"Package": "catboost", "Version": catboost.__version__},
            {"Package": "joblib", "Version": joblib.__version__},
        ]
    )
    write_csv(staged_audit_dir / PACKAGE_VERSIONS_PATH.name, package_versions)

    validation_records = [
        {
            "Check": "All checkpoints match expected SHA-256",
            "Passed": checkpoint_hashes == EXPECTED_CHECKPOINT_HASHES,
        },
        {
            "Check": "Only pre-March training rows loaded",
            "Passed": bool((all_routes[DATE_COLUMN] < DEVELOPMENT_END_EXCLUSIVE).all()),
        },
        {
            "Check": "Core predictor count is 53",
            "Passed": len(core_predictors) == EXPECTED_CORE_PREDICTORS,
        },
        {
            "Check": "ND08 daily method is median ensemble",
            "Passed": daily_product_method == EXPECTED_DAILY_METHOD,
        },
        {
            "Check": "ND08 multi-step planning method is rolling mean 5",
            "Passed": week_planning_method == EXPECTED_WEEK_METHOD,
        },
        {
            "Check": "Future feature and route replay passed",
            "Passed": bool(feature_replay_audit["Passed"].all()),
        },
        {
            "Check": "Daily forecast has one row per active product",
            "Passed": len(daily_product_forecast) == active_count,
        },
        {
            "Check": "Week output covers all configured operating dates",
            "Passed": set(pd.to_datetime(week_daily_product_forecast[DATE_COLUMN]).dt.normalize())
            == week_requested_dates,
        },
        {
            "Check": "No future actual target used",
            "Passed": not bool(
                pd.concat([daily_path, week_path])["FutureActualDemandUsed"].any()
            ),
        },
        {
            "Check": "All predictions finite and nonnegative",
            "Passed": bool(
                np.isfinite(
                    pd.concat([daily_path, week_path])["PredictedNormalDemand"]
                ).all()
                and (
                    pd.concat([daily_path, week_path])["PredictedNormalDemand"] >= 0
                ).all()
            ),
        },
        {
            "Check": "Long-horizon extrapolation explicitly controlled",
            "Passed": (not long_horizon_extrapolation)
            or ALLOW_LONG_HORIZON_EXTRAPOLATION,
        },
    ]
    validation = pd.DataFrame(validation_records)
    if not validation["Passed"].all():
        raise AssertionError(
            "ND09 validation failed:\n"
            + validation.loc[~validation["Passed"]].to_string(index=False)
        )
    write_csv(staged_audit_dir / VALIDATION_PATH.name, validation)

    inference_contract = {
        "StepID": STEP_ID,
        "Status": STATUS,
        "EngineRole": "DEMONSTRATION_AND_OPERATIONAL_INFERENCE_ENGINE",
        "DataCutoff": cutoff_date,
        "Target": TARGET_COLUMN,
        "ActiveProductCount": active_count,
        "DailyForecastPolicy": {
            "OneOperatingDayAheadMethod": daily_product_method,
            "RepresentativeMedianComponents": median_components,
            "TwoOrMoreUnobservedOperatingDaysMethod": week_planning_method,
            "Reason": (
                "ND08 selected the median ensemble for next-day product error "
                "and rolling mean 5 for recursive multi-day planning."
            ),
        },
        "WeekForecastPolicy": {
            "Method": week_planning_method,
            "DefaultWeek": "Monday to Friday",
            "PredictionsInsertedIntoTemporaryHistory": True,
            "WithinWeekActualDemandUsed": False,
        },
        "DailyUpdatedWeekPolicy": {
            "Method": updated_week_method,
            "ActualHistoryMayBeUsedOnlyAfterItIsObserved": True,
            "ImplementationStage": "ND10_OR_UI_INTEGRATION",
        },
        "FallbackMethods": route_method_map,
        "CalendarPolicy": {
            "Default": "Monday to Friday",
            "ClosedDates": [str(value.date()) for value in sorted(closed_dates)],
            "ExtraOperatingDates": [str(value.date()) for value in sorted(extra_dates)],
        },
        "HorizonPolicy": {
            "ValidatedRecursiveOperatingDays": MAX_VALIDATED_RECURSIVE_HORIZON,
            "HardMaximumOperatingDays": MAXIMUM_ALLOWED_RECURSIVE_HORIZON,
            "LongerHorizonRequiresExplicitOverride": True,
        },
        "DefaultDemonstrationForecasts": {
            "ArbitraryDate": requested_daily_date,
            "ArbitraryDateMethod": daily_operational_method,
            "ArbitraryDateHorizonOperatingDays": daily_horizon,
            "WeekStart": requested_week_start,
            "WeekEnd": requested_week_end,
            "WeekMethod": week_planning_method,
        },
        "ModelMetadata": model_metadata,
        "March2026TargetVaultOpened": False,
        "FinalUnbiasedFutureEvaluationStillRequired": True,
        "NextStep": "ND10_CONFIRMED_BULK_AND_PLANNING_INTEGRATION",
    }
    write_json(staged_contract_dir / INFERENCE_CONTRACT_PATH.name, inference_contract)

    contract_md = f"""# ND09 Inference Engine Contract

## Frozen operational methods

- One operating day ahead: `{daily_product_method}`
- Recursive multi-day and week-start planning: `{week_planning_method}`
- Rolling daily-updated remaining week: `{updated_week_method}`
- Median components: `{', '.join(median_components)}`

## Inference rule

When the requested date is the next operating day after the latest actual, the daily median ensemble is used. When unobserved intermediate operating days must be bridged, the recursive planning method selected in ND08 is used for the complete path.

## Calendar

The default future calendar is Monday to Friday. Closure and exceptional operating-date overrides must be supplied explicitly.

## Scope

The engine predicts `NormalDemand`. Confirmed bulk demand is not estimated by the model and will be added separately in ND10.

## Validation boundary

The recursive weekly method was validated over five operating days. Longer recursive paths are extrapolations and require an explicit override. March 2026 targets were not opened, and a new future period is still required for final unbiased accuracy evaluation.
"""
    write_text(staged_contract_dir / INFERENCE_CONTRACT_MD_PATH.name, contract_md)

    usage_text = f"""# ND09 Inference Usage

Edit only the USER CONFIGURATION section at the top of the ND09 notebook.

- `REQUESTED_DAILY_DATE=None` forecasts the next operating day after `{cutoff_date.date()}`.
- `REQUESTED_WEEK_START=None` forecasts the next Monday-to-Friday week.
- Add known closure dates to `CLOSED_DATES`.
- Add exceptional weekend operating dates to `EXTRA_OPERATING_DATES`.
- Leave `ACTIVE_PRODUCT_IDS=None` to forecast all {active_count} active products.

The default demonstration run forecasts `{requested_daily_date.date()}` and the planning week `{requested_week_start.date()}` to `{requested_week_end.date()}`.

A request more than {MAX_VALIDATED_RECURSIVE_HORIZON} operating days beyond the cutoff is outside the validated weekly horizon and is blocked unless `ALLOW_LONG_HORIZON_EXTRAPOLATION=True`.
"""
    write_text(staged_contract_dir / USAGE_PATH.name, usage_text)

    # Figures
    daily_top = daily_product_forecast.nlargest(20, "PredictedNormalDemand").sort_values(
        "PredictedNormalDemand"
    )
    plt.figure(figsize=(10, 7))
    plt.barh(daily_top[PRODUCT_NAME_COLUMN].astype(str), daily_top["PredictedNormalDemand"])
    plt.xlabel("Predicted normal-demand units")
    plt.ylabel("Product")
    plt.title(f"ND09 top daily forecasts — {requested_daily_date.date()}")
    save_figure(staged_figure_dir / "ND09_figure_01_daily_top_products.png")

    plt.figure(figsize=(9, 5))
    plt.plot(
        week_restaurant_daily[DATE_COLUMN],
        week_restaurant_daily["PredictedRestaurantNormalDemand"],
        marker="o",
    )
    plt.xlabel("Date")
    plt.ylabel("Predicted restaurant normal demand")
    plt.title("ND09 Monday-origin restaurant daily forecast")
    plt.xticks(rotation=30)
    save_figure(staged_figure_dir / "ND09_figure_02_week_restaurant_daily.png")

    week_top = week_product_totals.nlargest(20, "PredictedWeekNormalDemand").sort_values(
        "PredictedWeekNormalDemand"
    )
    plt.figure(figsize=(10, 7))
    plt.barh(week_top[PRODUCT_NAME_COLUMN].astype(str), week_top["PredictedWeekNormalDemand"])
    plt.xlabel("Predicted week normal-demand units")
    plt.ylabel("Product")
    plt.title(f"ND09 top product-week forecasts — {requested_week_start.date()}")
    save_figure(staged_figure_dir / "ND09_figure_03_week_top_products.png")

    report_text = f"""# ND09 Arbitrary-Date Inference Engine

## Status

`{STATUS}`

## Frozen system

The revised normal-demand model has been fixed for the demonstration. One-day-ahead product forecasting uses `{daily_product_method}`. Multi-step future inference and Monday-origin weekly planning use `{week_planning_method}` because ND08 found it more stable under recursive forecasting.

## Demonstration forecasts

- Data cutoff: {cutoff_date.date()}
- Active products: {active_count}
- Arbitrary date: {requested_daily_date.date()}
- Arbitrary-date horizon: {daily_horizon} operating day(s)
- Arbitrary-date method: `{daily_operational_method}`
- Arbitrary-date restaurant normal-demand total: {float(daily_product_forecast['PredictedNormalDemand'].sum()):.6f}
- Planning week: {requested_week_start.date()} to {requested_week_end.date()}
- Planning-week operating dates: {len(week_requested_dates)}
- Planning-week method: `{week_planning_method}`
- Planning-week restaurant normal-demand total: {float(week_daily_product_forecast['PredictedNormalDemand'].sum()):.6f}

## Model fitting

The advanced components were fitted once on all eligible pre-March main-route development rows. Their configurations were selected from ND07 inner-validation choices by modal frequency with mean inner WAPE as the tie-break. The median subset came from the ND08 operational contract.

## Safety

- March 2026 targets opened: no
- Future actual demand used: no
- Historical features rebuilt recursively: yes
- Dynamic routes rebuilt recursively: yes
- Feature and route replay passed: yes
- Final unbiased future evaluation still required: yes

## Next step

ND10 will add confirmed bulk orders to the normal-demand forecast and create the final planned-quantity outputs used by the demonstration interface.
"""
    write_text(staged_report_dir / REPORT_SUMMARY_PATH.name, report_text)

    readme_text = f"""# ND09 Arbitrary-Date Inference Engine

This folder contains the frozen demonstration model artifacts, arbitrary-date forecast, genuine Monday-origin week forecast, contracts, audits, figures, and control files.

The engine predicts normal demand only. Confirmed bulk demand is added in ND10.

Status: `{STATUS}`
"""
    write_text(STAGING_ROOT / README_PATH.name, readme_text)

    manifest = build_manifest(STAGING_ROOT, exclude_control=True)
    write_csv(staged_control_dir / MANIFEST_PATH.name, manifest)
    manifest_hash = sha256_file(staged_control_dir / MANIFEST_PATH.name)

    checkpoint_payload = {
        "StepID": STEP_ID,
        "Status": STATUS,
        "CompletedLocalTime": NOW_LOCAL.isoformat(),
        "ND09Root": str(ND09_ROOT),
        "InputCheckpointHashes": checkpoint_hashes,
        "ArtifactManifestSHA256": manifest_hash,
        "DataCutoff": cutoff_date,
        "TrainingRowsAllRoutes": int(len(all_routes)),
        "TrainingRowsMainModel": int(len(training_main)),
        "ActiveProducts": active_count,
        "DailyMethod": daily_product_method,
        "MultiStepPlanningMethod": week_planning_method,
        "DailyRequestedDate": requested_daily_date,
        "WeekStart": requested_week_start,
        "WeekEnd": requested_week_end,
        "MarchTargetVaultOpened": False,
        "DemonstrationModelArtifactsSaved": True,
        "FinalUnbiasedFutureEvaluationStillRequired": True,
        "NextStep": "ND10",
    }
    staged_checkpoint_path = staged_control_dir / CHECKPOINT_PATH.name
    write_json(staged_checkpoint_path, checkpoint_payload)
    checkpoint_hash = sha256_file(staged_checkpoint_path)
    write_text(
        staged_control_dir / CHECKPOINT_SHA_PATH.name,
        checkpoint_hash + "\n",
    )

    lock_payload = {
        "LockType": "ND09_DEMONSTRATION_INFERENCE_ENGINE_LOCK",
        "Status": "LOCKED_FOR_DEMONSTRATION_NOT_FINAL_UNBIASED_EVALUATION",
        "CreatedLocalTime": NOW_LOCAL.isoformat(),
        "CheckpointSHA256": checkpoint_hash,
        "ArtifactManifestSHA256": manifest_hash,
        "DailyProductMethod": daily_product_method,
        "MultiStepPlanningMethod": week_planning_method,
        "RollingDailyUpdatedWeeklyMethod": updated_week_method,
        "TrainingDataEnd": cutoff_date,
        "March2026TargetsUsed": False,
        "FinalUnbiasedFutureEvaluationStillRequired": True,
    }
    write_json(staged_control_dir / LOCK_PATH.name, lock_payload)

    protected_input_hashes_after = {
        str(path): sha256_file(path) for path in required_inputs
    }
    changed_inputs = [
        path
        for path in protected_input_hashes_before
        if protected_input_hashes_before[path] != protected_input_hashes_after[path]
    ]
    if changed_inputs:
        raise AssertionError(
            "Protected inputs changed during ND09:\n"
            + "\n".join(f"- {path}" for path in changed_inputs)
        )

    os.replace(STAGING_ROOT, ND09_ROOT)

    TOP_LEVEL_CHECKPOINT_PATH.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(ND09_ROOT / "07_control" / CHECKPOINT_PATH.name, TOP_LEVEL_CHECKPOINT_PATH)
    shutil.copy2(
        ND09_ROOT / "07_control" / CHECKPOINT_SHA_PATH.name,
        TOP_LEVEL_CHECKPOINT_SHA_PATH,
    )
    shutil.copy2(ND09_ROOT / "07_control" / LOCK_PATH.name, TOP_LEVEL_LOCK_PATH)

    handoff_text = f"""# ND09 Handoff

## Status

- Completed step: `{STEP_ID}`
- Status: `{STATUS}`
- Root: `{ND09_ROOT}`
- Checkpoint SHA-256: `{checkpoint_hash}`

## Frozen demonstration methods

- Next operating day: `{daily_product_method}`
- Multi-step and Monday-origin planning: `{week_planning_method}`
- Rolling daily-updated week: `{updated_week_method}`

## Artifacts

Reusable CatBoost, XGBoost and preprocessing artifacts were fitted on all pre-March development data and saved under `01_model_artifacts`.

## Demonstration forecasts

- Arbitrary date: {requested_daily_date.date()}
- Planning week: {requested_week_start.date()} to {requested_week_end.date()}
- Active products: {active_count}

## Safety

- March target vault opened: no
- Future actual demand used: no
- Final unbiased future evaluation still required: yes

## Next step

ND10 — confirmed bulk-order integration and final planned-quantity outputs.
"""
    atomic_write_text(ND09_HANDOFF_PATH, handoff_text)
    atomic_write_text(CURRENT_HANDOFF_PATH, handoff_text)

    append_marked_section(
        WORKFLOW_PATH,
        "## ND09 — Arbitrary-date inference engine",
        f"""## ND09 — Arbitrary-date inference engine

Status: `{STATUS}`

The ND08 operational system was frozen for demonstration. Reusable model artifacts and recursive arbitrary-date/week-ahead inference were created without opening March targets.
""",
    )
    append_marked_section(
        DECISIONS_PATH,
        "## ND09 decisions",
        f"""## ND09 decisions

- Fix the revised normal-demand architecture for the demonstration.
- Use `{daily_product_method}` for the next operating day.
- Use `{week_planning_method}` whenever unobserved intermediate operating days must be bridged.
- Keep confirmed bulk demand outside the model and integrate it in ND10.
""",
    )
    append_marked_section(
        METRICS_AND_RESULTS_PATH,
        "## ND09 inference outputs",
        f"""## ND09 inference outputs

- Data cutoff: {cutoff_date.date()}
- Active products: {active_count}
- Arbitrary-date restaurant forecast: {float(daily_product_forecast['PredictedNormalDemand'].sum()):.6f}
- Week-ahead restaurant forecast: {float(week_daily_product_forecast['PredictedNormalDemand'].sum()):.6f}
- No accuracy metric is reported because future actual targets were not opened.
""",
    )
    append_marked_section(
        AGENTS_PATH,
        "Marker: ND09_AUTHORITATIVE_STATUS",
        f"""## ND09 authoritative status

Marker: ND09_AUTHORITATIVE_STATUS

- Status: `{STATUS}`
- Handoff: `{ND09_HANDOFF_PATH}`
- Checkpoint SHA-256: `{checkpoint_hash}`
- Next step: ND10 confirmed bulk and planned-quantity integration.
""",
    )

    LOG_PATH.parent.mkdir(parents=True, exist_ok=True)
    with LOG_PATH.open("a", encoding="utf-8") as log_handle:
        log_handle.write(
            f"{NOW_LOCAL.isoformat()} | {STATUS} | "
            f"checkpoint={checkpoint_hash} | root={ND09_ROOT}\n"
        )

except Exception:
    if STAGING_ROOT.exists():
        shutil.rmtree(STAGING_ROOT, ignore_errors=True)
    raise


# =============================================================================
# FINAL CONSOLE OUTPUT
# =============================================================================

print("=" * 118)
print("EDEN NORMAL-DEMAND MODEL V2 — ND09 COMPLETE")
print("=" * 118)
print(f"Status: {STATUS}")
print(f"Local time: {NOW_LOCAL.isoformat()}")
print(f"ND09 root: {ND09_ROOT}")
print()
print("INPUT VERIFICATION")
for name in ["ND03", "ND04", "ND05", "ND06", "ND07", "ND07E", "ND08"]:
    print(f"{name} checkpoint SHA-256: {checkpoint_hashes[name]}")
print(f"Pre-March all-route rows: {len(all_routes):,}")
print(f"Pre-March main-model rows: {len(training_main):,}")
print(f"Core predictors: {len(core_predictors)}")
print(f"Active products: {active_count}")
print(f"March target vault opened: False")
print(f"Previous inputs modified: False")
print()
print("FROZEN OPERATIONAL SYSTEM")
print(f"Next-operating-day method: {daily_product_method}")
print(f"Representative median components: {', '.join(median_components)}")
print(f"Multi-step / week-start method: {week_planning_method}")
print(f"Rolling daily-updated week method: {updated_week_method}")
print(f"ND07E ensemble-setting source schema: {nd07e_schema}")
print(f"Model fitting seconds: {model_fit_seconds:.3f}")
print()
print("FUTURE FEATURE ENGINE")
print(f"Data cutoff date: {cutoff_date.date()}")
print(f"Data cutoff operating sequence: {cutoff_sequence}")
print(f"Cutoff-date normal demand total: {cutoff_target_total:.6f}")
print(f"Historical replay date: {replay_date.date()}")
print(f"Historical features and route replay passed: {feature_replay_audit['Passed'].all()}")
print(f"Closed-date overrides: {len(closed_dates)}")
print(f"Extra operating-date overrides: {len(extra_dates)}")
print()
print("ARBITRARY-DATE FORECAST")
print(f"Requested date: {requested_daily_date.date()}")
print(f"Recursive horizon: {daily_horizon} operating day(s)")
print(f"Operational method: {daily_operational_method}")
print(f"Method-selection reason: {daily_method_reason}")
print(f"Products forecast: {len(daily_product_forecast):,}")
print(
    "Predicted restaurant normal demand: "
    f"{daily_product_forecast['PredictedNormalDemand'].sum():.6f}"
)
print()
print("MONDAY-ORIGIN WEEK FORECAST")
print(f"Week start: {requested_week_start.date()}")
print(f"Week end: {requested_week_end.date()}")
print(f"Operating dates forecast: {len(week_requested_dates)}")
print(f"Recursive path length: {week_horizon} operating day(s)")
print(f"Operational method: {week_planning_method}")
print(f"Products per operating date: {active_count:,}")
print(
    "Predicted restaurant week normal demand: "
    f"{week_daily_product_forecast['PredictedNormalDemand'].sum():.6f}"
)
print()
print("OUTPUTS")
print(f"- Model artifacts: {MODEL_ARTIFACT_DIR}")
print(f"- Arbitrary-date product forecast: {DAILY_PRODUCT_FORECAST_PATH}")
print(f"- Arbitrary-date restaurant total: {DAILY_RESTAURANT_FORECAST_PATH}")
print(f"- Week daily product forecast: {WEEK_DAILY_PRODUCT_FORECAST_PATH}")
print(f"- Week product totals: {WEEK_PRODUCT_TOTAL_PATH}")
print(f"- Week restaurant daily totals: {WEEK_RESTAURANT_DAILY_PATH}")
print(f"- Week restaurant total: {WEEK_RESTAURANT_TOTAL_PATH}")
print(f"- Active product catalogue: {ACTIVE_CATALOGUE_PATH}")
print(f"- Inference contract: {INFERENCE_CONTRACT_PATH}")
print(f"- Report: {REPORT_SUMMARY_PATH}")
print(f"- Checkpoint: {TOP_LEVEL_CHECKPOINT_PATH}")
print(f"- Checkpoint SHA-256: {checkpoint_hash}")
print(f"- Demonstration engine lock: {TOP_LEVEL_LOCK_PATH}")
print(f"- Handoff: {ND09_HANDOFF_PATH}")
print()
print("SAFETY")
print("- March target vault opened: False")
print("- Future actual demand used: False")
print("- Historical features rebuilt recursively: True")
print("- Dynamic routes rebuilt recursively: True")
print("- Demonstration model artifacts saved: True")
print("- Final unbiased future evaluation still required: True")
print("- Previous inputs modified: False")
print("- ND09 checkpoint, hashes and demonstration lock created: True")
print()
print("NEXT STEP")
print("ND10 — confirmed bulk-order integration and final planned-quantity outputs.")
print("=" * 118)

EDEN NORMAL-DEMAND MODEL V2 — ND09 COMPLETE
Status: ND09_ARBITRARY_DATE_INFERENCE_ENGINE_CREATED_READY_FOR_ND10
Local time: 2026-08-08T01:06:27.785172+01:00
ND09 root: /Users/ryansmac/Desktop/Meng Project/eden_datasets/eden_normal_demand_model_v2/03_models/03_inference/ND09_arbitrary_date_inference_engine

INPUT VERIFICATION
ND03 checkpoint SHA-256: 0845af89a5b459ca13ae6ffd99dde444f5010f6c0fb5ba4c34a4f091ac2e151c
ND04 checkpoint SHA-256: 2fdc5d2c64c38f85b2669ca942042884209d80111cc840261307da98b1e9cf54
ND05 checkpoint SHA-256: ce3342c8b960aa5c4791a114ab09ae1a86d1dab2a3eebb648aa060579a1378ff
ND06 checkpoint SHA-256: e3b7bb75a1b2968e426c9a4c1654e10420d683af4bd5cb2b75fa4b5f0f18f357
ND07 checkpoint SHA-256: 39077dd8c197561e5384a6943c3a4e153153e75fa4d03019f45005eb145cf18e
ND07E checkpoint SHA-256: eaf6d23434cb67d663f66a39db4f898b74c1d516aee5c0627ca9136bed9877b8
ND08 checkpoint SHA-256: a98828007df9bd4d0cce309fc8f8bfbc1d2c1f1cf74f1ca762b27237468dea10
Pre-March all-route rows: 23,763
Pre-March

In [15]:
# =============================================================================
# EDEN NORMAL-DEMAND MODEL V2
# ND10 — CONFIRMED BULK-ORDER INTEGRATION AND FINAL PLANNED QUANTITIES
#
# Run this as one complete Jupyter cell after ND09.
#
# This step does not refit or retune any forecasting model. It consumes the
# frozen ND09 normal-demand forecasts and adds only externally confirmed bulk
# orders. The final operational quantity is:
#
#   PlannedQuantity = PredictedNormalDemand + ConfirmedBulkDemand
#
# Normal demand, confirmed bulk demand, and planned quantity remain separate
# in every output. March 2026 targets are not loaded or used.
# =============================================================================

import hashlib
import json
import os
import platform
import shutil
import uuid
from datetime import datetime, timezone
from pathlib import Path
from zoneinfo import ZoneInfo

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd


# =============================================================================
# USER CONFIGURATION
# =============================================================================

PROJECT_ROOT = Path("/Users/ryansmac/Desktop/Meng Project")

# Place a CSV at this path to add confirmed bulk orders. If the file does not
# exist, ND10 creates zero-bulk planned outputs and saves an input template.
CONFIRMED_BULK_INPUT_PATH = (
    PROJECT_ROOT
    / "eden_datasets"
    / "eden_normal_demand_model_v2"
    / "04_planning_inputs"
    / "ND10_confirmed_bulk_orders.csv"
)

# Set True only when a missing bulk-order file should stop the run.
REQUIRE_BULK_INPUT_FILE = False

# Bulk lines outside the dates currently forecast by ND09 are blocked by
# default so that no confirmed order is silently omitted from the plan.
ALLOW_OUT_OF_FORECAST_RANGE_BULK_ORDERS = False

# Set True only when intentionally rebuilding ND10 after changing the bulk
# input. ND09 and all earlier artifacts remain protected and unchanged.
ALLOW_OVERWRITE = False


# =============================================================================
# FIXED PROJECT CONFIGURATION
# =============================================================================

EDEN_ROOT = PROJECT_ROOT / "eden_datasets"
MODEL_ROOT = EDEN_ROOT / "eden_normal_demand_model_v2"

ND09_ROOT = (
    MODEL_ROOT
    / "03_models"
    / "03_inference"
    / "ND09_arbitrary_date_inference_engine"
)
ND09_FORECAST_DIR = ND09_ROOT / "02_forecasts"
ND09_CONTRACT_DIR = ND09_ROOT / "03_contracts"

ND09_DAILY_PRODUCT_PATH = (
    ND09_FORECAST_DIR / "ND09_arbitrary_date_product_forecast.csv"
)
ND09_DAILY_RESTAURANT_PATH = (
    ND09_FORECAST_DIR / "ND09_arbitrary_date_restaurant_total.csv"
)
ND09_WEEK_DAILY_PRODUCT_PATH = (
    ND09_FORECAST_DIR / "ND09_week_ahead_daily_product_forecast.csv"
)
ND09_WEEK_PRODUCT_TOTAL_PATH = (
    ND09_FORECAST_DIR / "ND09_week_ahead_product_totals.csv"
)
ND09_WEEK_RESTAURANT_DAILY_PATH = (
    ND09_FORECAST_DIR / "ND09_week_ahead_restaurant_daily_totals.csv"
)
ND09_WEEK_RESTAURANT_TOTAL_PATH = (
    ND09_FORECAST_DIR / "ND09_week_ahead_restaurant_total.csv"
)
ND09_ACTIVE_CATALOGUE_PATH = (
    ND09_CONTRACT_DIR / "ND09_active_product_catalogue.csv"
)
ND09_INFERENCE_CONTRACT_PATH = (
    ND09_CONTRACT_DIR / "ND09_inference_engine_contract.json"
)
ND09_CHECKPOINT_PATH = MODEL_ROOT / "08_checkpoints" / "ND09_checkpoint.json"
ND09_LOCK_PATH = (
    MODEL_ROOT / "08_checkpoints" / "ND09_demonstration_engine_lock.json"
)

EXPECTED_ND09_CHECKPOINT_SHA256 = (
    "201b84095ed138e0b666485e2022e4f1f99ad93b89971f46533ff4a0605ef4e8"
)

DATE_COLUMN = "Date"
PRODUCT_ID_COLUMN = "CanonicalProductID"
PRODUCT_NAME_COLUMN = "CanonicalProductName"
FAMILY_COLUMN = "TierProductFamily"
NORMAL_FORECAST_COLUMN = "PredictedNormalDemand"
BULK_COLUMN = "ConfirmedBulkDemand"
PLANNED_COLUMN = "PlannedQuantity"

BULK_REQUIRED_COLUMNS = {
    DATE_COLUMN,
    PRODUCT_ID_COLUMN,
    BULK_COLUMN,
}
BULK_OPTIONAL_COLUMNS = [
    "OrderReference",
    "CustomerOrEvent",
    "Notes",
]
BULK_TEMPLATE_COLUMNS = [
    "OrderReference",
    DATE_COLUMN,
    PRODUCT_ID_COLUMN,
    BULK_COLUMN,
    "CustomerOrEvent",
    "Notes",
]

EXPECTED_ACTIVE_PRODUCTS = 227
EXPECTED_DAILY_METHOD = "NESTED_MEDIAN_ENSEMBLE"
EXPECTED_WEEK_METHOD = "ROLLING_MEAN_5"
EXPECTED_TARGET = "NormalDemand"

ND10_ROOT = (
    MODEL_ROOT
    / "04_planning"
    / "ND10_confirmed_bulk_and_planned_quantity"
)
INPUT_SNAPSHOT_DIR = ND10_ROOT / "01_inputs"
OUTPUT_DIR = ND10_ROOT / "02_planned_outputs"
CONTRACT_DIR = ND10_ROOT / "03_contracts"
AUDIT_DIR = ND10_ROOT / "04_audits"
FIGURE_DIR = ND10_ROOT / "05_figures"
REPORT_DIR = ND10_ROOT / "06_reports"
CONTROL_DIR = ND10_ROOT / "07_control"

BULK_TEMPLATE_PATH = CONTRACT_DIR / "ND10_confirmed_bulk_order_input_template.csv"
BULK_NORMALIZED_PATH = INPUT_SNAPSHOT_DIR / "ND10_confirmed_bulk_orders_normalized.csv"
BULK_AGGREGATED_PATH = INPUT_SNAPSHOT_DIR / "ND10_confirmed_bulk_by_product_date.csv"

DAILY_PRODUCT_PLANNING_PATH = OUTPUT_DIR / "ND10_daily_product_planning.csv"
DAILY_RESTAURANT_PLANNING_PATH = OUTPUT_DIR / "ND10_daily_restaurant_planning.csv"
WEEK_DAILY_PRODUCT_PLANNING_PATH = (
    OUTPUT_DIR / "ND10_week_daily_product_planning.csv"
)
WEEK_PRODUCT_TOTAL_PLANNING_PATH = (
    OUTPUT_DIR / "ND10_week_product_totals_planning.csv"
)
WEEK_RESTAURANT_DAILY_PLANNING_PATH = (
    OUTPUT_DIR / "ND10_week_restaurant_daily_planning.csv"
)
WEEK_RESTAURANT_TOTAL_PLANNING_PATH = (
    OUTPUT_DIR / "ND10_week_restaurant_total_planning.csv"
)
DEMONSTRATION_SUMMARY_PATH = OUTPUT_DIR / "ND10_demonstration_planning_summary.csv"

PLANNING_CONTRACT_PATH = CONTRACT_DIR / "ND10_planning_quantity_contract.json"
PLANNING_CONTRACT_MD_PATH = CONTRACT_DIR / "ND10_planning_quantity_contract.md"
USAGE_PATH = CONTRACT_DIR / "ND10_bulk_order_usage.md"

INPUT_HASH_AUDIT_PATH = AUDIT_DIR / "ND10_input_hash_audit.csv"
BULK_VALIDATION_AUDIT_PATH = AUDIT_DIR / "ND10_bulk_order_validation_audit.csv"
BULK_MATCH_AUDIT_PATH = AUDIT_DIR / "ND10_bulk_order_match_audit.csv"
RECONCILIATION_AUDIT_PATH = AUDIT_DIR / "ND10_planning_reconciliation_audit.csv"
VALIDATION_PATH = AUDIT_DIR / "ND10_validation_summary.csv"
PACKAGE_VERSIONS_PATH = AUDIT_DIR / "ND10_package_versions.csv"

REPORT_SUMMARY_PATH = REPORT_DIR / "ND10_confirmed_bulk_and_planning_summary.md"
README_PATH = ND10_ROOT / "README.md"
MANIFEST_PATH = CONTROL_DIR / "ND10_artifact_hash_manifest.csv"
CHECKPOINT_PATH = CONTROL_DIR / "ND10_checkpoint.json"
CHECKPOINT_SHA_PATH = CONTROL_DIR / "ND10_checkpoint.sha256"

TOP_LEVEL_CHECKPOINT_PATH = MODEL_ROOT / "08_checkpoints" / "ND10_checkpoint.json"
TOP_LEVEL_CHECKPOINT_SHA_PATH = MODEL_ROOT / "08_checkpoints" / "ND10_checkpoint.sha256"

MEMORY_ROOT = MODEL_ROOT / "00_project_memory"
ND10_HANDOFF_PATH = MEMORY_ROOT / "ND10_HANDOFF.md"
CURRENT_HANDOFF_PATH = MEMORY_ROOT / "CURRENT_HANDOFF.md"
WORKFLOW_PATH = MEMORY_ROOT / "WORKFLOW.md"
DECISIONS_PATH = MEMORY_ROOT / "DECISIONS.md"
METRICS_AND_RESULTS_PATH = MEMORY_ROOT / "METRICS_AND_RESULTS.md"
AGENTS_PATH = MODEL_ROOT / "AGENTS.md"
LOG_PATH = MODEL_ROOT / "09_logs" / "ND10_confirmed_bulk_planning_log.txt"

STEP_ID = "ND10"
STATUS = "ND10_CONFIRMED_BULK_AND_PLANNED_QUANTITIES_CREATED_READY_FOR_ND11"
NOW_UTC = datetime.now(timezone.utc)
NOW_LOCAL = NOW_UTC.astimezone(ZoneInfo("Europe/Dublin"))


# =============================================================================
# GENERAL HELPERS
# =============================================================================

def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


def write_csv(path: Path, frame: pd.DataFrame) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    frame.to_csv(path, index=False)


def write_json(path: Path, payload: dict) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(
        json.dumps(payload, indent=2, ensure_ascii=False, default=str) + "\n",
        encoding="utf-8",
    )


def write_text(path: Path, text: str) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(text, encoding="utf-8")


def atomic_write_text(path: Path, text: str) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_name(f".{path.name}.{uuid.uuid4().hex}.tmp")
    temporary.write_text(text, encoding="utf-8")
    os.replace(temporary, path)


def append_marked_section(path: Path, marker: str, section_text: str) -> None:
    existing = path.read_text(encoding="utf-8") if path.is_file() else ""
    if marker in existing:
        return
    separator = "\n" if existing.endswith("\n") else "\n\n"
    atomic_write_text(path, existing + separator + section_text.strip() + "\n")


def validate_required_columns(
    frame: pd.DataFrame,
    columns: set[str],
    name: str,
) -> None:
    missing = sorted(columns - set(frame.columns))
    if missing:
        raise AssertionError(
            f"{name} is missing required columns:\n"
            + "\n".join(f"- {column}" for column in missing)
        )


def maximum_abs_difference(left: float, right: float) -> float:
    return float(abs(float(left) - float(right)))


def save_figure(path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    plt.tight_layout()
    plt.savefig(path, dpi=300, bbox_inches="tight")
    plt.close()


def build_manifest(root: Path, exclude_control: bool = True) -> pd.DataFrame:
    records = []
    for path in sorted(root.rglob("*")):
        if not path.is_file():
            continue
        relative = path.relative_to(root)
        if exclude_control and relative.parts and relative.parts[0] == "07_control":
            continue
        records.append(
            {
                "RelativePath": str(relative),
                "Bytes": int(path.stat().st_size),
                "SHA256": sha256_file(path),
            }
        )
    return pd.DataFrame(records)


# =============================================================================
# BULK-ORDER HELPERS
# =============================================================================

def empty_bulk_frame() -> pd.DataFrame:
    return pd.DataFrame(columns=BULK_TEMPLATE_COLUMNS)


def load_and_validate_bulk_orders(
    input_path: Path,
    active_catalogue: pd.DataFrame,
    forecast_dates: set[pd.Timestamp],
) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame, bool]:
    input_exists = input_path.is_file()
    validation_records = []

    if not input_exists:
        if REQUIRE_BULK_INPUT_FILE:
            raise FileNotFoundError(
                "Confirmed bulk-order input is required but was not found:\n"
                f"{input_path}"
            )
        raw = empty_bulk_frame()
        validation_records.append(
            {
                "Check": "Bulk-order input file exists",
                "Passed": False,
                "Detail": "Missing input accepted; zero confirmed bulk used.",
            }
        )
    else:
        raw = pd.read_csv(input_path, low_memory=False)
        validation_records.append(
            {
                "Check": "Bulk-order input file exists",
                "Passed": True,
                "Detail": str(input_path),
            }
        )

    if raw.empty and not BULK_REQUIRED_COLUMNS.issubset(raw.columns):
        raw = empty_bulk_frame()

    validate_required_columns(raw, BULK_REQUIRED_COLUMNS, "Confirmed bulk-order input")

    normalized = raw.copy()
    for column in BULK_OPTIONAL_COLUMNS:
        if column not in normalized.columns:
            normalized[column] = ""

    normalized = normalized[BULK_TEMPLATE_COLUMNS].copy()
    normalized[DATE_COLUMN] = pd.to_datetime(
        normalized[DATE_COLUMN], errors="coerce"
    ).dt.normalize()
    normalized[PRODUCT_ID_COLUMN] = (
        normalized[PRODUCT_ID_COLUMN]
        .astype("string")
        .fillna("")
        .str.strip()
        .astype(str)
    )
    normalized[BULK_COLUMN] = pd.to_numeric(
        normalized[BULK_COLUMN], errors="coerce"
    )
    for column in BULK_OPTIONAL_COLUMNS:
        normalized[column] = (
            normalized[column].astype("string").fillna("").astype(str).str.strip()
        )

    invalid_date = normalized[DATE_COLUMN].isna()
    invalid_product = normalized[PRODUCT_ID_COLUMN].eq("")
    invalid_units = (
        normalized[BULK_COLUMN].isna()
        | ~np.isfinite(normalized[BULK_COLUMN].astype(float))
        | (normalized[BULK_COLUMN] < 0)
    )
    non_integer_units = (
        ~invalid_units
        & ~np.isclose(
            normalized[BULK_COLUMN].astype(float),
            np.round(normalized[BULK_COLUMN].astype(float)),
            atol=1e-10,
        )
    )

    available_products = set(
        active_catalogue.loc[
            active_catalogue["IncludeInForecast"].astype(bool), PRODUCT_ID_COLUMN
        ].astype(str)
    )
    unknown_product = ~normalized[PRODUCT_ID_COLUMN].isin(available_products)
    out_of_range = ~normalized[DATE_COLUMN].isin(forecast_dates)

    issue_frame = normalized.copy()
    issue_frame["InvalidDate"] = invalid_date
    issue_frame["InvalidProductID"] = invalid_product
    issue_frame["InvalidConfirmedBulkDemand"] = invalid_units
    issue_frame["NonIntegerConfirmedBulkDemand"] = non_integer_units
    issue_frame["UnknownProduct"] = unknown_product
    issue_frame["OutsideND09ForecastDates"] = out_of_range
    issue_frame["AcceptedForPlanning"] = ~(
        invalid_date
        | invalid_product
        | invalid_units
        | non_integer_units
        | unknown_product
        | (
            out_of_range
            & (not ALLOW_OUT_OF_FORECAST_RANGE_BULK_ORDERS)
        )
    )

    hard_issue_mask = (
        invalid_date
        | invalid_product
        | invalid_units
        | non_integer_units
        | unknown_product
        | (
            out_of_range
            & (not ALLOW_OUT_OF_FORECAST_RANGE_BULK_ORDERS)
        )
    )
    if hard_issue_mask.any():
        raise ValueError(
            "Confirmed bulk-order validation failed:\n"
            + issue_frame.loc[hard_issue_mask].to_string(index=False)
        )

    accepted = normalized.loc[issue_frame["AcceptedForPlanning"]].copy()
    if not accepted.empty:
        accepted[BULK_COLUMN] = np.round(accepted[BULK_COLUMN]).astype(int)
        accepted = accepted.sort_values(
            [DATE_COLUMN, PRODUCT_ID_COLUMN, "OrderReference"],
            kind="mergesort",
        ).reset_index(drop=True)

    aggregated = (
        accepted.groupby([DATE_COLUMN, PRODUCT_ID_COLUMN], as_index=False)[BULK_COLUMN]
        .sum()
        if not accepted.empty
        else pd.DataFrame(columns=[DATE_COLUMN, PRODUCT_ID_COLUMN, BULK_COLUMN])
    )

    validation_records.extend(
        [
            {
                "Check": "All bulk dates parse successfully",
                "Passed": not bool(invalid_date.any()),
                "Detail": f"Invalid rows: {int(invalid_date.sum())}",
            },
            {
                "Check": "All product IDs are present",
                "Passed": not bool(invalid_product.any()),
                "Detail": f"Invalid rows: {int(invalid_product.sum())}",
            },
            {
                "Check": "Confirmed bulk quantities are finite and nonnegative",
                "Passed": not bool(invalid_units.any()),
                "Detail": f"Invalid rows: {int(invalid_units.sum())}",
            },
            {
                "Check": "Confirmed bulk quantities are whole units",
                "Passed": not bool(non_integer_units.any()),
                "Detail": f"Invalid rows: {int(non_integer_units.sum())}",
            },
            {
                "Check": "All products exist in the active catalogue",
                "Passed": not bool(unknown_product.any()),
                "Detail": f"Unknown rows: {int(unknown_product.sum())}",
            },
            {
                "Check": "All bulk dates are covered by ND09 forecasts",
                "Passed": not bool(out_of_range.any()),
                "Detail": f"Out-of-range rows: {int(out_of_range.sum())}",
            },
        ]
    )

    return (
        accepted,
        aggregated,
        pd.DataFrame(validation_records),
        input_exists,
    )


def add_bulk_to_forecast(
    forecast: pd.DataFrame,
    bulk_by_product_date: pd.DataFrame,
    output_label: str,
) -> pd.DataFrame:
    validate_required_columns(
        forecast,
        {
            DATE_COLUMN,
            PRODUCT_ID_COLUMN,
            PRODUCT_NAME_COLUMN,
            FAMILY_COLUMN,
            NORMAL_FORECAST_COLUMN,
        },
        output_label,
    )

    output = forecast.copy()
    output[DATE_COLUMN] = pd.to_datetime(output[DATE_COLUMN], errors="raise").dt.normalize()
    output[PRODUCT_ID_COLUMN] = output[PRODUCT_ID_COLUMN].astype(str)
    output[NORMAL_FORECAST_COLUMN] = pd.to_numeric(
        output[NORMAL_FORECAST_COLUMN], errors="raise"
    ).astype(float)

    if output.duplicated([DATE_COLUMN, PRODUCT_ID_COLUMN]).any():
        duplicates = output.loc[
            output.duplicated([DATE_COLUMN, PRODUCT_ID_COLUMN], keep=False),
            [DATE_COLUMN, PRODUCT_ID_COLUMN],
        ]
        raise AssertionError(
            f"{output_label} contains duplicate product-date rows:\n"
            + duplicates.to_string(index=False)
        )

    output = output.merge(
        bulk_by_product_date,
        on=[DATE_COLUMN, PRODUCT_ID_COLUMN],
        how="left",
        validate="one_to_one",
    )
    output[BULK_COLUMN] = output[BULK_COLUMN].fillna(0).astype(float)
    output[PLANNED_COLUMN] = output[NORMAL_FORECAST_COLUMN] + output[BULK_COLUMN]
    output["PlanningFormula"] = (
        "PredictedNormalDemand + ConfirmedBulkDemand"
    )

    if not np.isfinite(output[[NORMAL_FORECAST_COLUMN, BULK_COLUMN, PLANNED_COLUMN]]).all().all():
        raise AssertionError(f"{output_label} contains non-finite planning quantities.")
    if (output[[NORMAL_FORECAST_COLUMN, BULK_COLUMN, PLANNED_COLUMN]] < 0).any().any():
        raise AssertionError(f"{output_label} contains negative planning quantities.")

    return output


# =============================================================================
# PREFLIGHT
# =============================================================================

required_inputs = [
    ND09_DAILY_PRODUCT_PATH,
    ND09_DAILY_RESTAURANT_PATH,
    ND09_WEEK_DAILY_PRODUCT_PATH,
    ND09_WEEK_PRODUCT_TOTAL_PATH,
    ND09_WEEK_RESTAURANT_DAILY_PATH,
    ND09_WEEK_RESTAURANT_TOTAL_PATH,
    ND09_ACTIVE_CATALOGUE_PATH,
    ND09_INFERENCE_CONTRACT_PATH,
    ND09_CHECKPOINT_PATH,
    ND09_LOCK_PATH,
]
missing_inputs = [path for path in required_inputs if not path.is_file()]
if missing_inputs:
    raise FileNotFoundError(
        "ND10 required ND09 inputs are missing:\n"
        + "\n".join(f"- {path}" for path in missing_inputs)
    )

actual_nd09_checkpoint_hash = sha256_file(ND09_CHECKPOINT_PATH)
if actual_nd09_checkpoint_hash != EXPECTED_ND09_CHECKPOINT_SHA256:
    raise AssertionError(
        "ND09 checkpoint hash mismatch.\n"
        f"Expected: {EXPECTED_ND09_CHECKPOINT_SHA256}\n"
        f"Actual:   {actual_nd09_checkpoint_hash}"
    )

if ND10_ROOT.exists():
    if not ALLOW_OVERWRITE:
        raise FileExistsError(
            "ND10 output already exists. No files were changed:\n"
            f"{ND10_ROOT}"
        )
    shutil.rmtree(ND10_ROOT)

for path in [TOP_LEVEL_CHECKPOINT_PATH, TOP_LEVEL_CHECKPOINT_SHA_PATH]:
    if path.exists():
        if not ALLOW_OVERWRITE:
            raise FileExistsError(
                "An ND10 top-level checkpoint already exists. No files were changed:\n"
                f"{path}"
            )
        path.unlink()

protected_input_hashes_before = {str(path): sha256_file(path) for path in required_inputs}
if CONFIRMED_BULK_INPUT_PATH.is_file():
    protected_input_hashes_before[str(CONFIRMED_BULK_INPUT_PATH)] = sha256_file(
        CONFIRMED_BULK_INPUT_PATH
    )

STAGING_ROOT = ND10_ROOT.parent / f".ND10_staging_{uuid.uuid4().hex}"
STAGING_ROOT.mkdir(parents=True, exist_ok=False)


# =============================================================================
# MAIN EXECUTION
# =============================================================================

try:
    daily_forecast = pd.read_csv(ND09_DAILY_PRODUCT_PATH, low_memory=False)
    daily_restaurant_source = pd.read_csv(
        ND09_DAILY_RESTAURANT_PATH, low_memory=False
    )
    week_daily_forecast = pd.read_csv(
        ND09_WEEK_DAILY_PRODUCT_PATH, low_memory=False
    )
    week_product_source = pd.read_csv(
        ND09_WEEK_PRODUCT_TOTAL_PATH, low_memory=False
    )
    week_restaurant_daily_source = pd.read_csv(
        ND09_WEEK_RESTAURANT_DAILY_PATH, low_memory=False
    )
    week_restaurant_total_source = pd.read_csv(
        ND09_WEEK_RESTAURANT_TOTAL_PATH, low_memory=False
    )
    active_catalogue = pd.read_csv(ND09_ACTIVE_CATALOGUE_PATH, low_memory=False)
    inference_contract = json.loads(
        ND09_INFERENCE_CONTRACT_PATH.read_text(encoding="utf-8")
    )
    nd09_checkpoint = json.loads(ND09_CHECKPOINT_PATH.read_text(encoding="utf-8"))
    nd09_lock = json.loads(ND09_LOCK_PATH.read_text(encoding="utf-8"))

    validate_required_columns(
        active_catalogue,
        {PRODUCT_ID_COLUMN, PRODUCT_NAME_COLUMN, "IncludeInForecast"},
        "ND09 active product catalogue",
    )
    active_catalogue[PRODUCT_ID_COLUMN] = active_catalogue[PRODUCT_ID_COLUMN].astype(str)
    active_count = int(active_catalogue["IncludeInForecast"].astype(bool).sum())
    if active_count != EXPECTED_ACTIVE_PRODUCTS:
        raise AssertionError(
            f"Expected {EXPECTED_ACTIVE_PRODUCTS} active products; found {active_count}."
        )

    if inference_contract.get("Target") != EXPECTED_TARGET:
        raise AssertionError("ND09 inference target is not NormalDemand.")
    if (
        inference_contract.get("DailyForecastPolicy", {}).get(
            "OneOperatingDayAheadMethod"
        )
        != EXPECTED_DAILY_METHOD
    ):
        raise AssertionError("ND09 next-day method is not the selected median ensemble.")
    if (
        inference_contract.get("WeekForecastPolicy", {}).get("Method")
        != EXPECTED_WEEK_METHOD
    ):
        raise AssertionError("ND09 week method is not ROLLING_MEAN_5.")
    if bool(inference_contract.get("March2026TargetVaultOpened", True)):
        raise AssertionError("ND09 contract indicates that March targets were opened.")

    daily_forecast[DATE_COLUMN] = pd.to_datetime(
        daily_forecast[DATE_COLUMN], errors="raise"
    ).dt.normalize()
    week_daily_forecast[DATE_COLUMN] = pd.to_datetime(
        week_daily_forecast[DATE_COLUMN], errors="raise"
    ).dt.normalize()
    daily_forecast[PRODUCT_ID_COLUMN] = daily_forecast[PRODUCT_ID_COLUMN].astype(str)
    week_daily_forecast[PRODUCT_ID_COLUMN] = week_daily_forecast[PRODUCT_ID_COLUMN].astype(str)

    if len(daily_forecast) != active_count:
        raise AssertionError(
            "ND09 arbitrary-date forecast does not contain one row per active product."
        )
    week_dates = sorted(week_daily_forecast[DATE_COLUMN].unique())
    if len(week_dates) == 0:
        raise AssertionError("ND09 week forecast contains no dates.")
    if not (
        week_daily_forecast.groupby(DATE_COLUMN)[PRODUCT_ID_COLUMN].nunique()
        == active_count
    ).all():
        raise AssertionError(
            "ND09 week forecast does not contain all active products on every date."
        )

    forecast_dates = set(daily_forecast[DATE_COLUMN]) | set(
        week_daily_forecast[DATE_COLUMN]
    )
    (
        bulk_orders,
        bulk_by_product_date,
        bulk_validation_audit,
        bulk_input_exists,
    ) = load_and_validate_bulk_orders(
        CONFIRMED_BULK_INPUT_PATH,
        active_catalogue,
        forecast_dates,
    )

    daily_planning = add_bulk_to_forecast(
        daily_forecast,
        bulk_by_product_date,
        "ND09 arbitrary-date product forecast",
    )
    week_daily_planning = add_bulk_to_forecast(
        week_daily_forecast,
        bulk_by_product_date,
        "ND09 week daily product forecast",
    )

    daily_restaurant_planning = (
        daily_planning.groupby(DATE_COLUMN, as_index=False)
        .agg(
            ProductsForecast=(PRODUCT_ID_COLUMN, "nunique"),
            PredictedRestaurantNormalDemand=(NORMAL_FORECAST_COLUMN, "sum"),
            ConfirmedRestaurantBulkDemand=(BULK_COLUMN, "sum"),
            PlannedRestaurantQuantity=(PLANNED_COLUMN, "sum"),
        )
    )
    daily_restaurant_planning["NormalDemandMethod"] = EXPECTED_DAILY_METHOD
    daily_restaurant_planning["BulkDemandSource"] = "EXTERNALLY_CONFIRMED"

    week_daily_planning["WeekStart"] = pd.to_datetime(
        week_daily_planning.get(
            "WeekStart", pd.Series([min(week_dates)] * len(week_daily_planning))
        ),
        errors="raise",
    ).dt.normalize()
    week_daily_planning["WeekEnd"] = pd.to_datetime(
        week_daily_planning.get(
            "WeekEnd", pd.Series([max(week_dates)] * len(week_daily_planning))
        ),
        errors="raise",
    ).dt.normalize()

    week_product_totals = (
        week_daily_planning.groupby(
            [PRODUCT_ID_COLUMN, PRODUCT_NAME_COLUMN, FAMILY_COLUMN],
            dropna=False,
            as_index=False,
        )
        .agg(
            PredictedWeekNormalDemand=(NORMAL_FORECAST_COLUMN, "sum"),
            ConfirmedWeekBulkDemand=(BULK_COLUMN, "sum"),
            PlannedWeekQuantity=(PLANNED_COLUMN, "sum"),
        )
    )
    week_product_totals["WeekStart"] = min(week_dates)
    week_product_totals["WeekEnd"] = max(week_dates)
    week_product_totals["NormalDemandMethod"] = EXPECTED_WEEK_METHOD
    week_product_totals = week_product_totals.sort_values(
        ["PlannedWeekQuantity", PRODUCT_ID_COLUMN],
        ascending=[False, True],
        kind="mergesort",
    ).reset_index(drop=True)

    week_restaurant_daily = (
        week_daily_planning.groupby(DATE_COLUMN, as_index=False)
        .agg(
            ProductsForecast=(PRODUCT_ID_COLUMN, "nunique"),
            PredictedRestaurantNormalDemand=(NORMAL_FORECAST_COLUMN, "sum"),
            ConfirmedRestaurantBulkDemand=(BULK_COLUMN, "sum"),
            PlannedRestaurantQuantity=(PLANNED_COLUMN, "sum"),
        )
    )
    week_restaurant_daily["WeekStart"] = min(week_dates)
    week_restaurant_daily["WeekEnd"] = max(week_dates)
    week_restaurant_daily["NormalDemandMethod"] = EXPECTED_WEEK_METHOD

    week_restaurant_total = pd.DataFrame(
        [
            {
                "WeekStart": min(week_dates),
                "WeekEnd": max(week_dates),
                "OperatingDatesForecast": len(week_dates),
                "ProductsForecast": active_count,
                "NormalDemandMethod": EXPECTED_WEEK_METHOD,
                "PredictedRestaurantWeekNormalDemand": float(
                    week_daily_planning[NORMAL_FORECAST_COLUMN].sum()
                ),
                "ConfirmedRestaurantWeekBulkDemand": float(
                    week_daily_planning[BULK_COLUMN].sum()
                ),
                "PlannedRestaurantWeekQuantity": float(
                    week_daily_planning[PLANNED_COLUMN].sum()
                ),
            }
        ]
    )

    # -------------------------------------------------------------------------
    # Match and reconciliation audits.
    # -------------------------------------------------------------------------
    forecast_keys = pd.concat(
        [
            daily_forecast[[DATE_COLUMN, PRODUCT_ID_COLUMN]].assign(
                ForecastOutput="ARBITRARY_DATE"
            ),
            week_daily_forecast[[DATE_COLUMN, PRODUCT_ID_COLUMN]].assign(
                ForecastOutput="WEEK_DAILY"
            ),
        ],
        ignore_index=True,
    )
    if bulk_orders.empty:
        bulk_match_audit = pd.DataFrame(
            columns=BULK_TEMPLATE_COLUMNS
            + ["AppearsInArbitraryDateOutput", "AppearsInWeekOutput"]
        )
    else:
        daily_keys = set(
            map(
                tuple,
                daily_forecast[[DATE_COLUMN, PRODUCT_ID_COLUMN]].itertuples(
                    index=False, name=None
                ),
            )
        )
        week_keys = set(
            map(
                tuple,
                week_daily_forecast[[DATE_COLUMN, PRODUCT_ID_COLUMN]].itertuples(
                    index=False, name=None
                ),
            )
        )
        bulk_match_audit = bulk_orders.copy()
        key_tuples = list(
            bulk_match_audit[[DATE_COLUMN, PRODUCT_ID_COLUMN]].itertuples(
                index=False, name=None
            )
        )
        bulk_match_audit["AppearsInArbitraryDateOutput"] = [
            key in daily_keys for key in key_tuples
        ]
        bulk_match_audit["AppearsInWeekOutput"] = [
            key in week_keys for key in key_tuples
        ]

    daily_normal_source = float(daily_forecast[NORMAL_FORECAST_COLUMN].sum())
    daily_normal_output = float(daily_planning[NORMAL_FORECAST_COLUMN].sum())
    daily_bulk_output = float(daily_planning[BULK_COLUMN].sum())
    daily_planned_output = float(daily_planning[PLANNED_COLUMN].sum())

    week_normal_source = float(week_daily_forecast[NORMAL_FORECAST_COLUMN].sum())
    week_normal_output = float(week_daily_planning[NORMAL_FORECAST_COLUMN].sum())
    week_bulk_output = float(week_daily_planning[BULK_COLUMN].sum())
    week_planned_output = float(week_daily_planning[PLANNED_COLUMN].sum())

    reconciliation_records = [
        {
            "Check": "Daily normal forecast preserved",
            "Expected": daily_normal_source,
            "Actual": daily_normal_output,
            "AbsoluteDifference": maximum_abs_difference(
                daily_normal_source, daily_normal_output
            ),
        },
        {
            "Check": "Daily planned quantity equals normal plus bulk",
            "Expected": daily_normal_output + daily_bulk_output,
            "Actual": daily_planned_output,
            "AbsoluteDifference": maximum_abs_difference(
                daily_normal_output + daily_bulk_output,
                daily_planned_output,
            ),
        },
        {
            "Check": "Daily restaurant total equals product bottom-up total",
            "Expected": daily_planned_output,
            "Actual": float(
                daily_restaurant_planning["PlannedRestaurantQuantity"].sum()
            ),
            "AbsoluteDifference": maximum_abs_difference(
                daily_planned_output,
                daily_restaurant_planning["PlannedRestaurantQuantity"].sum(),
            ),
        },
        {
            "Check": "Week normal forecast preserved",
            "Expected": week_normal_source,
            "Actual": week_normal_output,
            "AbsoluteDifference": maximum_abs_difference(
                week_normal_source, week_normal_output
            ),
        },
        {
            "Check": "Week planned quantity equals normal plus bulk",
            "Expected": week_normal_output + week_bulk_output,
            "Actual": week_planned_output,
            "AbsoluteDifference": maximum_abs_difference(
                week_normal_output + week_bulk_output,
                week_planned_output,
            ),
        },
        {
            "Check": "Week product totals reconcile to daily product rows",
            "Expected": week_planned_output,
            "Actual": float(week_product_totals["PlannedWeekQuantity"].sum()),
            "AbsoluteDifference": maximum_abs_difference(
                week_planned_output,
                week_product_totals["PlannedWeekQuantity"].sum(),
            ),
        },
        {
            "Check": "Week restaurant daily totals reconcile",
            "Expected": week_planned_output,
            "Actual": float(
                week_restaurant_daily["PlannedRestaurantQuantity"].sum()
            ),
            "AbsoluteDifference": maximum_abs_difference(
                week_planned_output,
                week_restaurant_daily["PlannedRestaurantQuantity"].sum(),
            ),
        },
        {
            "Check": "Week restaurant total reconciles",
            "Expected": week_planned_output,
            "Actual": float(
                week_restaurant_total["PlannedRestaurantWeekQuantity"].iloc[0]
            ),
            "AbsoluteDifference": maximum_abs_difference(
                week_planned_output,
                week_restaurant_total["PlannedRestaurantWeekQuantity"].iloc[0],
            ),
        },
    ]
    reconciliation_audit = pd.DataFrame(reconciliation_records)
    reconciliation_audit["Passed"] = (
        reconciliation_audit["AbsoluteDifference"] <= 1e-9
    )
    if not reconciliation_audit["Passed"].all():
        raise AssertionError(
            "ND10 planning reconciliation failed:\n"
            + reconciliation_audit.loc[
                ~reconciliation_audit["Passed"]
            ].to_string(index=False)
        )

    bulk_total_input = float(bulk_orders[BULK_COLUMN].sum()) if not bulk_orders.empty else 0.0
    bulk_total_covered_unique = float(bulk_by_product_date[BULK_COLUMN].sum()) if not bulk_by_product_date.empty else 0.0
    if maximum_abs_difference(bulk_total_input, bulk_total_covered_unique) > 1e-9:
        raise AssertionError("Bulk-order line total does not reconcile to aggregated bulk total.")

    daily_date = pd.Timestamp(daily_forecast[DATE_COLUMN].iloc[0]).normalize()
    daily_bulk_total = float(
        bulk_by_product_date.loc[
            bulk_by_product_date[DATE_COLUMN] == daily_date, BULK_COLUMN
        ].sum()
    ) if not bulk_by_product_date.empty else 0.0
    week_bulk_total = float(
        bulk_by_product_date.loc[
            bulk_by_product_date[DATE_COLUMN].isin(set(week_dates)), BULK_COLUMN
        ].sum()
    ) if not bulk_by_product_date.empty else 0.0

    demonstration_summary = pd.DataFrame(
        [
            {
                "PlanningView": "ARBITRARY_DATE",
                "StartDate": daily_date,
                "EndDate": daily_date,
                "NormalDemandMethod": EXPECTED_DAILY_METHOD,
                "PredictedNormalDemand": daily_normal_output,
                "ConfirmedBulkDemand": daily_bulk_output,
                "PlannedQuantity": daily_planned_output,
                "ProductsForecast": active_count,
            },
            {
                "PlanningView": "MONDAY_ORIGIN_WEEK",
                "StartDate": min(week_dates),
                "EndDate": max(week_dates),
                "NormalDemandMethod": EXPECTED_WEEK_METHOD,
                "PredictedNormalDemand": week_normal_output,
                "ConfirmedBulkDemand": week_bulk_output,
                "PlannedQuantity": week_planned_output,
                "ProductsForecast": active_count,
            },
        ]
    )

    # -------------------------------------------------------------------------
    # Stage outputs.
    # -------------------------------------------------------------------------
    staged_input_dir = STAGING_ROOT / "01_inputs"
    staged_output_dir = STAGING_ROOT / "02_planned_outputs"
    staged_contract_dir = STAGING_ROOT / "03_contracts"
    staged_audit_dir = STAGING_ROOT / "04_audits"
    staged_figure_dir = STAGING_ROOT / "05_figures"
    staged_report_dir = STAGING_ROOT / "06_reports"
    staged_control_dir = STAGING_ROOT / "07_control"
    for directory in [
        staged_input_dir,
        staged_output_dir,
        staged_contract_dir,
        staged_audit_dir,
        staged_figure_dir,
        staged_report_dir,
        staged_control_dir,
    ]:
        directory.mkdir(parents=True, exist_ok=True)

    write_csv(staged_input_dir / BULK_NORMALIZED_PATH.name, bulk_orders)
    write_csv(staged_input_dir / BULK_AGGREGATED_PATH.name, bulk_by_product_date)
    write_csv(staged_contract_dir / BULK_TEMPLATE_PATH.name, empty_bulk_frame())

    write_csv(staged_output_dir / DAILY_PRODUCT_PLANNING_PATH.name, daily_planning)
    write_csv(
        staged_output_dir / DAILY_RESTAURANT_PLANNING_PATH.name,
        daily_restaurant_planning,
    )
    write_csv(
        staged_output_dir / WEEK_DAILY_PRODUCT_PLANNING_PATH.name,
        week_daily_planning,
    )
    write_csv(
        staged_output_dir / WEEK_PRODUCT_TOTAL_PLANNING_PATH.name,
        week_product_totals,
    )
    write_csv(
        staged_output_dir / WEEK_RESTAURANT_DAILY_PLANNING_PATH.name,
        week_restaurant_daily,
    )
    write_csv(
        staged_output_dir / WEEK_RESTAURANT_TOTAL_PLANNING_PATH.name,
        week_restaurant_total,
    )
    write_csv(
        staged_output_dir / DEMONSTRATION_SUMMARY_PATH.name,
        demonstration_summary,
    )

    input_hash_records = [
        {
            "Input": str(path),
            "SHA256Before": protected_input_hashes_before[str(path)],
            "InputRole": "ND09_PROTECTED_INPUT",
        }
        for path in required_inputs
    ]
    if bulk_input_exists:
        input_hash_records.append(
            {
                "Input": str(CONFIRMED_BULK_INPUT_PATH),
                "SHA256Before": protected_input_hashes_before[
                    str(CONFIRMED_BULK_INPUT_PATH)
                ],
                "InputRole": "USER_CONFIRMED_BULK_INPUT",
            }
        )
    write_csv(
        staged_audit_dir / INPUT_HASH_AUDIT_PATH.name,
        pd.DataFrame(input_hash_records),
    )
    write_csv(
        staged_audit_dir / BULK_VALIDATION_AUDIT_PATH.name,
        bulk_validation_audit,
    )
    write_csv(
        staged_audit_dir / BULK_MATCH_AUDIT_PATH.name,
        bulk_match_audit,
    )
    write_csv(
        staged_audit_dir / RECONCILIATION_AUDIT_PATH.name,
        reconciliation_audit,
    )

    validation = pd.DataFrame(
        [
            {
                "Check": "ND09 checkpoint hash matches",
                "Passed": actual_nd09_checkpoint_hash
                == EXPECTED_ND09_CHECKPOINT_SHA256,
            },
            {
                "Check": "ND09 demonstration engine is locked",
                "Passed": str(nd09_lock.get("Status", "")).startswith("LOCKED"),
            },
            {
                "Check": "ND09 target is NormalDemand",
                "Passed": inference_contract.get("Target") == EXPECTED_TARGET,
            },
            {
                "Check": "March target vault remains closed",
                "Passed": not bool(
                    inference_contract.get("March2026TargetVaultOpened", True)
                ),
            },
            {
                "Check": "Daily output contains all active products",
                "Passed": len(daily_planning) == active_count,
            },
            {
                "Check": "Week output contains all active products per date",
                "Passed": bool(
                    (
                        week_daily_planning.groupby(DATE_COLUMN)[
                            PRODUCT_ID_COLUMN
                        ].nunique()
                        == active_count
                    ).all()
                ),
            },
            {
                "Check": "All bulk-order validation checks passed",
                "Passed": bool(
                    bulk_validation_audit.loc[
                        bulk_validation_audit["Check"]
                        != "Bulk-order input file exists",
                        "Passed",
                    ].all()
                ),
            },
            {
                "Check": "All planning reconciliations passed",
                "Passed": bool(reconciliation_audit["Passed"].all()),
            },
            {
                "Check": "No model refit or reselection performed",
                "Passed": True,
            },
            {
                "Check": "Normal, bulk and planned quantities remain separate",
                "Passed": all(
                    column in daily_planning.columns
                    for column in [
                        NORMAL_FORECAST_COLUMN,
                        BULK_COLUMN,
                        PLANNED_COLUMN,
                    ]
                ),
            },
        ]
    )
    if not validation["Passed"].all():
        raise AssertionError(
            "ND10 validation failed:\n"
            + validation.loc[~validation["Passed"]].to_string(index=False)
        )
    write_csv(staged_audit_dir / VALIDATION_PATH.name, validation)

    package_versions = pd.DataFrame(
        [
            {"Package": "python", "Version": platform.python_version()},
            {"Package": "pandas", "Version": pd.__version__},
            {"Package": "numpy", "Version": np.__version__},
        ]
    )
    write_csv(
        staged_audit_dir / PACKAGE_VERSIONS_PATH.name,
        package_versions,
    )

    planning_contract = {
        "StepID": STEP_ID,
        "Status": STATUS,
        "PlanningFormula": {
            "PlannedQuantity": (
                "PredictedNormalDemand + ConfirmedBulkDemand"
            ),
            "NormalDemandSource": "ND09_FROZEN_INFERENCE_ENGINE",
            "ConfirmedBulkDemandSource": "EXTERNAL_CONFIRMED_INPUT_ONLY",
        },
        "ND09CheckpointSHA256": actual_nd09_checkpoint_hash,
        "DailyPlanning": {
            "Date": daily_date,
            "NormalDemandMethod": EXPECTED_DAILY_METHOD,
            "Products": active_count,
            "PredictedNormalDemand": daily_normal_output,
            "ConfirmedBulkDemand": daily_bulk_output,
            "PlannedQuantity": daily_planned_output,
        },
        "WeekPlanning": {
            "WeekStart": min(week_dates),
            "WeekEnd": max(week_dates),
            "OperatingDates": len(week_dates),
            "NormalDemandMethod": EXPECTED_WEEK_METHOD,
            "Products": active_count,
            "PredictedNormalDemand": week_normal_output,
            "ConfirmedBulkDemand": week_bulk_output,
            "PlannedQuantity": week_planned_output,
        },
        "BulkInput": {
            "ConfiguredPath": str(CONFIRMED_BULK_INPUT_PATH),
            "InputFileFound": bulk_input_exists,
            "AcceptedLines": int(len(bulk_orders)),
            "UniqueProductDateCombinations": int(len(bulk_by_product_date)),
            "TotalConfirmedUnits": bulk_total_input,
            "RequiredColumns": sorted(BULK_REQUIRED_COLUMNS),
            "OptionalColumns": BULK_OPTIONAL_COLUMNS,
            "OutOfForecastRangeAllowed": ALLOW_OUT_OF_FORECAST_RANGE_BULK_ORDERS,
        },
        "Safety": {
            "ModelRefitted": False,
            "ModelReselected": False,
            "March2026TargetsOpened": False,
            "PreviousInputsModified": False,
            "AccuracyMetricReported": False,
        },
        "NextStep": "ND11_INGREDIENT_MAPPING_AND_DEMONSTRATION_EXPORT",
    }
    write_json(
        staged_contract_dir / PLANNING_CONTRACT_PATH.name,
        planning_contract,
    )

    contract_md = f"""# ND10 Planning Quantity Contract

## Formula

`PlannedQuantity = PredictedNormalDemand + ConfirmedBulkDemand`

The model estimates normal recurring demand only. Confirmed bulk demand is supplied externally and is never inferred from ordinary demand patterns.

## Daily plan

- Date: {daily_date.date()}
- Normal-demand method: `{EXPECTED_DAILY_METHOD}`
- Predicted normal demand: {daily_normal_output:.6f}
- Confirmed bulk demand: {daily_bulk_output:.6f}
- Planned quantity: {daily_planned_output:.6f}

## Week plan

- Week: {pd.Timestamp(min(week_dates)).date()} to {pd.Timestamp(max(week_dates)).date()}
- Normal-demand method: `{EXPECTED_WEEK_METHOD}`
- Predicted normal demand: {week_normal_output:.6f}
- Confirmed bulk demand: {week_bulk_output:.6f}
- Planned quantity: {week_planned_output:.6f}

## Operational rule

Only confirmed orders are added. Possible, provisional, or unconfirmed enquiries must not be included in `ConfirmedBulkDemand`.

## Evaluation boundary

ND10 creates planning quantities and does not calculate forecast accuracy. March 2026 actual targets remain closed.
"""
    write_text(
        staged_contract_dir / PLANNING_CONTRACT_MD_PATH.name,
        contract_md,
    )

    usage_text = f"""# ND10 Confirmed Bulk-Order Input

Create a CSV at:

`{CONFIRMED_BULK_INPUT_PATH}`

Required columns:

- `Date`
- `CanonicalProductID`
- `ConfirmedBulkDemand`

Optional columns:

- `OrderReference`
- `CustomerOrEvent`
- `Notes`

Quantities must be nonnegative whole units. Dates must be present in the current ND09 arbitrary-date or week forecast. Multiple confirmed lines for the same product and date are added together.

When the input changes, rerun ND10 with `ALLOW_OVERWRITE=True`. The ND09 model artifacts remain unchanged.
"""
    write_text(staged_contract_dir / USAGE_PATH.name, usage_text)

    # Figures
    daily_top = daily_planning.nlargest(20, PLANNED_COLUMN).sort_values(PLANNED_COLUMN)
    plt.figure(figsize=(10, 7))
    plt.barh(daily_top[PRODUCT_NAME_COLUMN].astype(str), daily_top[PLANNED_COLUMN])
    plt.xlabel("Planned units")
    plt.ylabel("Product")
    plt.title(f"ND10 daily planned quantities — {daily_date.date()}")
    save_figure(staged_figure_dir / "ND10_figure_01_daily_planned_products.png")

    week_top = week_product_totals.nlargest(20, "PlannedWeekQuantity").sort_values(
        "PlannedWeekQuantity"
    )
    plt.figure(figsize=(10, 7))
    plt.barh(
        week_top[PRODUCT_NAME_COLUMN].astype(str),
        week_top["PlannedWeekQuantity"],
    )
    plt.xlabel("Planned week units")
    plt.ylabel("Product")
    plt.title("ND10 top weekly planned quantities")
    save_figure(staged_figure_dir / "ND10_figure_02_week_planned_products.png")

    plt.figure(figsize=(9, 5))
    plt.plot(
        week_restaurant_daily[DATE_COLUMN],
        week_restaurant_daily["PredictedRestaurantNormalDemand"],
        marker="o",
        label="Normal demand",
    )
    plt.plot(
        week_restaurant_daily[DATE_COLUMN],
        week_restaurant_daily["PlannedRestaurantQuantity"],
        marker="o",
        label="Planned quantity",
    )
    plt.xlabel("Date")
    plt.ylabel("Restaurant units")
    plt.title("ND10 restaurant normal demand and planned quantity")
    plt.xticks(rotation=30)
    plt.legend()
    save_figure(staged_figure_dir / "ND10_figure_03_restaurant_plan.png")

    report_text = f"""# ND10 Confirmed Bulk and Planned-Quantity Integration

## Status

`{STATUS}`

## Purpose

ND10 converts the frozen ND09 normal-demand forecasts into final operational planning quantities. Confirmed bulk orders are added externally rather than predicted by the demand model.

## Formula

`PlannedQuantity = PredictedNormalDemand + ConfirmedBulkDemand`

## Input result

- Bulk input file found: {bulk_input_exists}
- Accepted bulk-order lines: {len(bulk_orders)}
- Unique product-date bulk combinations: {len(bulk_by_product_date)}
- Total confirmed bulk units in the input: {bulk_total_input:.6f}

## Daily plan

- Date: {daily_date.date()}
- Products: {active_count}
- Predicted normal demand: {daily_normal_output:.6f}
- Confirmed bulk demand: {daily_bulk_output:.6f}
- Planned quantity: {daily_planned_output:.6f}

## Week plan

- Dates: {pd.Timestamp(min(week_dates)).date()} to {pd.Timestamp(max(week_dates)).date()}
- Operating dates: {len(week_dates)}
- Predicted normal demand: {week_normal_output:.6f}
- Confirmed bulk demand: {week_bulk_output:.6f}
- Planned quantity: {week_planned_output:.6f}

## Validation

All product, daily restaurant, product-week and restaurant-week totals reconcile exactly. ND09 forecasts were preserved without modification. No model was refitted or reselected, and March targets were not opened.

## Next step

ND11 will connect planned product quantities to ingredient requirements and prepare the final demonstration export package.
"""
    write_text(staged_report_dir / REPORT_SUMMARY_PATH.name, report_text)

    readme_text = f"""# ND10 Confirmed Bulk and Planned Quantities

This folder combines the frozen ND09 normal-demand forecasts with externally confirmed bulk orders.

Formula: `PlannedQuantity = PredictedNormalDemand + ConfirmedBulkDemand`

Status: `{STATUS}`
"""
    write_text(STAGING_ROOT / README_PATH.name, readme_text)

    manifest = build_manifest(STAGING_ROOT, exclude_control=True)
    write_csv(staged_control_dir / MANIFEST_PATH.name, manifest)
    manifest_hash = sha256_file(staged_control_dir / MANIFEST_PATH.name)

    checkpoint_payload = {
        "StepID": STEP_ID,
        "Status": STATUS,
        "CompletedLocalTime": NOW_LOCAL.isoformat(),
        "ND10Root": str(ND10_ROOT),
        "ND09CheckpointSHA256": actual_nd09_checkpoint_hash,
        "ArtifactManifestSHA256": manifest_hash,
        "BulkInputFileFound": bulk_input_exists,
        "AcceptedBulkOrderLines": int(len(bulk_orders)),
        "ConfirmedBulkUnitsInput": bulk_total_input,
        "DailyDate": daily_date,
        "DailyPredictedNormalDemand": daily_normal_output,
        "DailyConfirmedBulkDemand": daily_bulk_output,
        "DailyPlannedQuantity": daily_planned_output,
        "WeekStart": min(week_dates),
        "WeekEnd": max(week_dates),
        "WeekPredictedNormalDemand": week_normal_output,
        "WeekConfirmedBulkDemand": week_bulk_output,
        "WeekPlannedQuantity": week_planned_output,
        "MarchTargetVaultOpened": False,
        "ModelRefitted": False,
        "ModelReselected": False,
        "NextStep": "ND11",
    }
    staged_checkpoint_path = staged_control_dir / CHECKPOINT_PATH.name
    write_json(staged_checkpoint_path, checkpoint_payload)
    checkpoint_hash = sha256_file(staged_checkpoint_path)
    write_text(
        staged_control_dir / CHECKPOINT_SHA_PATH.name,
        checkpoint_hash + "\n",
    )

    protected_input_hashes_after = {str(path): sha256_file(path) for path in required_inputs}
    if bulk_input_exists:
        protected_input_hashes_after[str(CONFIRMED_BULK_INPUT_PATH)] = sha256_file(
            CONFIRMED_BULK_INPUT_PATH
        )
    changed_inputs = [
        path
        for path in protected_input_hashes_before
        if protected_input_hashes_before[path]
        != protected_input_hashes_after.get(path)
    ]
    if changed_inputs:
        raise AssertionError(
            "Protected inputs changed during ND10:\n"
            + "\n".join(f"- {path}" for path in changed_inputs)
        )

    os.replace(STAGING_ROOT, ND10_ROOT)

    TOP_LEVEL_CHECKPOINT_PATH.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(
        ND10_ROOT / "07_control" / CHECKPOINT_PATH.name,
        TOP_LEVEL_CHECKPOINT_PATH,
    )
    shutil.copy2(
        ND10_ROOT / "07_control" / CHECKPOINT_SHA_PATH.name,
        TOP_LEVEL_CHECKPOINT_SHA_PATH,
    )

    handoff_text = f"""# ND10 Handoff

## Status

- Completed step: `{STEP_ID}`
- Status: `{STATUS}`
- Root: `{ND10_ROOT}`
- Checkpoint SHA-256: `{checkpoint_hash}`

## Planning formula

`PlannedQuantity = PredictedNormalDemand + ConfirmedBulkDemand`

## Current outputs

- Daily date: {daily_date.date()}
- Daily planned quantity: {daily_planned_output:.6f}
- Week: {pd.Timestamp(min(week_dates)).date()} to {pd.Timestamp(max(week_dates)).date()}
- Week planned quantity: {week_planned_output:.6f}
- Confirmed bulk input file found: {bulk_input_exists}
- Confirmed bulk units loaded: {bulk_total_input:.6f}

## Safety

- ND09 model refitted: no
- Model reselected: no
- March targets opened: no
- Previous inputs modified: no

## Next step

ND11 — ingredient mapping and demonstration-ready export.
"""
    atomic_write_text(ND10_HANDOFF_PATH, handoff_text)
    atomic_write_text(CURRENT_HANDOFF_PATH, handoff_text)

    append_marked_section(
        WORKFLOW_PATH,
        "## ND10 — Confirmed bulk and planned quantities",
        f"""## ND10 — Confirmed bulk and planned quantities

Status: `{STATUS}`

ND09 normal-demand forecasts were combined with externally confirmed bulk orders. Normal, bulk and planned quantities remain separate and reconcile at product, day, week and restaurant levels.
""",
    )
    append_marked_section(
        DECISIONS_PATH,
        "## ND10 decisions",
        f"""## ND10 decisions

- Preserve the frozen ND09 normal-demand forecast without alteration.
- Add only externally confirmed bulk demand.
- Use `PlannedQuantity = PredictedNormalDemand + ConfirmedBulkDemand`.
- Do not include provisional or unconfirmed orders.
- Do not calculate accuracy from planning outputs.
""",
    )
    append_marked_section(
        METRICS_AND_RESULTS_PATH,
        "## ND10 planning outputs",
        f"""## ND10 planning outputs

- Daily normal demand: {daily_normal_output:.6f}
- Daily confirmed bulk: {daily_bulk_output:.6f}
- Daily planned quantity: {daily_planned_output:.6f}
- Week normal demand: {week_normal_output:.6f}
- Week confirmed bulk: {week_bulk_output:.6f}
- Week planned quantity: {week_planned_output:.6f}
- These are planning quantities, not accuracy metrics.
""",
    )
    append_marked_section(
        AGENTS_PATH,
        "Marker: ND10_AUTHORITATIVE_STATUS",
        f"""## ND10 authoritative status

Marker: ND10_AUTHORITATIVE_STATUS

- Status: `{STATUS}`
- Handoff: `{ND10_HANDOFF_PATH}`
- Checkpoint SHA-256: `{checkpoint_hash}`
- Next step: ND11 ingredient mapping and demonstration export.
""",
    )

    LOG_PATH.parent.mkdir(parents=True, exist_ok=True)
    with LOG_PATH.open("a", encoding="utf-8") as log_handle:
        log_handle.write(
            f"{NOW_LOCAL.isoformat()} | {STATUS} | checkpoint={checkpoint_hash} | "
            f"bulk_units={bulk_total_input:.6f} | root={ND10_ROOT}\n"
        )

except Exception:
    if STAGING_ROOT.exists():
        shutil.rmtree(STAGING_ROOT, ignore_errors=True)
    raise


# =============================================================================
# FINAL CONSOLE OUTPUT
# =============================================================================

print("=" * 118)
print("EDEN NORMAL-DEMAND MODEL V2 — ND10 COMPLETE")
print("=" * 118)
print(f"Status: {STATUS}")
print(f"Local time: {NOW_LOCAL.isoformat()}")
print(f"ND10 root: {ND10_ROOT}")
print()
print("INPUT VERIFICATION")
print(f"ND09 checkpoint SHA-256: {actual_nd09_checkpoint_hash}")
print(f"ND09 demonstration engine lock present: True")
print(f"Active products: {active_count}")
print(f"Normal-demand target: {inference_contract['Target']}")
print(f"March target vault opened: False")
print(f"ND09 artifacts modified: False")
print()
print("CONFIRMED BULK INPUT")
print(f"Configured input path: {CONFIRMED_BULK_INPUT_PATH}")
print(f"Input file found: {bulk_input_exists}")
print(f"Accepted order lines: {len(bulk_orders):,}")
print(f"Unique product-date combinations: {len(bulk_by_product_date):,}")
print(f"Total confirmed bulk units loaded: {bulk_total_input:.6f}")
if not bulk_input_exists:
    print("No input file was found; zero confirmed bulk was applied.")
    print(f"Input template saved to: {BULK_TEMPLATE_PATH}")
print()
print("DAILY PLANNING OUTPUT")
print(f"Date: {daily_date.date()}")
print(f"Normal-demand method: {EXPECTED_DAILY_METHOD}")
print(f"Products planned: {active_count:,}")
print(f"Predicted restaurant normal demand: {daily_normal_output:.6f}")
print(f"Confirmed restaurant bulk demand: {daily_bulk_output:.6f}")
print(f"Planned restaurant quantity: {daily_planned_output:.6f}")
print()
print("MONDAY-ORIGIN WEEK PLANNING OUTPUT")
print(f"Week start: {pd.Timestamp(min(week_dates)).date()}")
print(f"Week end: {pd.Timestamp(max(week_dates)).date()}")
print(f"Operating dates: {len(week_dates)}")
print(f"Normal-demand method: {EXPECTED_WEEK_METHOD}")
print(f"Predicted restaurant week normal demand: {week_normal_output:.6f}")
print(f"Confirmed restaurant week bulk demand: {week_bulk_output:.6f}")
print(f"Planned restaurant week quantity: {week_planned_output:.6f}")
print()
print("RECONCILIATION")
print(f"Checks passed: {int(reconciliation_audit['Passed'].sum())}/{len(reconciliation_audit)}")
print(
    "Maximum absolute reconciliation difference: "
    f"{reconciliation_audit['AbsoluteDifference'].max():.12g}"
)
print()
print("OUTPUTS")
print(f"- Daily product planning: {DAILY_PRODUCT_PLANNING_PATH}")
print(f"- Daily restaurant planning: {DAILY_RESTAURANT_PLANNING_PATH}")
print(f"- Week daily product planning: {WEEK_DAILY_PRODUCT_PLANNING_PATH}")
print(f"- Week product totals planning: {WEEK_PRODUCT_TOTAL_PLANNING_PATH}")
print(f"- Week restaurant daily planning: {WEEK_RESTAURANT_DAILY_PLANNING_PATH}")
print(f"- Week restaurant total planning: {WEEK_RESTAURANT_TOTAL_PLANNING_PATH}")
print(f"- Demonstration summary: {DEMONSTRATION_SUMMARY_PATH}")
print(f"- Planning contract: {PLANNING_CONTRACT_PATH}")
print(f"- Report: {REPORT_SUMMARY_PATH}")
print(f"- Checkpoint: {TOP_LEVEL_CHECKPOINT_PATH}")
print(f"- Checkpoint SHA-256: {checkpoint_hash}")
print(f"- Handoff: {ND10_HANDOFF_PATH}")
print()
print("SAFETY")
print("- Forecasting models refitted: False")
print("- Forecasting methods reselected: False")
print("- March target vault opened: False")
print("- Previous inputs modified: False")
print("- Accuracy metric calculated: False")
print("- ND10 checkpoint and hashes created: True")
print()
print("NEXT STEP")
print("ND11 — ingredient mapping and demonstration-ready export.")
print("=" * 118)

/var/folders/61/pw_dwqt140ndz64pvx21l9040000gn/T/ipykernel_17958/191020410.py:523: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  output[BULK_COLUMN] = output[BULK_COLUMN].fillna(0).astype(float)
/var/folders/61/pw_dwqt140ndz64pvx21l9040000gn/T/ipykernel_17958/191020410.py:523: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  output[BULK_COLUMN] = output[BULK_COLUMN].fillna(0).astype(float)


EDEN NORMAL-DEMAND MODEL V2 — ND10 COMPLETE
Status: ND10_CONFIRMED_BULK_AND_PLANNED_QUANTITIES_CREATED_READY_FOR_ND11
Local time: 2026-08-08T01:17:35.776676+01:00
ND10 root: /Users/ryansmac/Desktop/Meng Project/eden_datasets/eden_normal_demand_model_v2/04_planning/ND10_confirmed_bulk_and_planned_quantity

INPUT VERIFICATION
ND09 checkpoint SHA-256: 201b84095ed138e0b666485e2022e4f1f99ad93b89971f46533ff4a0605ef4e8
ND09 demonstration engine lock present: True
Active products: 227
Normal-demand target: NormalDemand
March target vault opened: False
ND09 artifacts modified: False

CONFIRMED BULK INPUT
Configured input path: /Users/ryansmac/Desktop/Meng Project/eden_datasets/eden_normal_demand_model_v2/04_planning_inputs/ND10_confirmed_bulk_orders.csv
Input file found: False
Accepted order lines: 0
Unique product-date combinations: 0
Total confirmed bulk units loaded: 0.000000
No input file was found; zero confirmed bulk was applied.
Input template saved to: /Users/ryansmac/Desktop/Meng Proje

In [16]:
# =============================================================================
# EDEN NORMAL-DEMAND MODEL V2
# ND10R — REPORT-READY DIAGNOSTICS, RESULTS ANALYSIS, AND DEMONSTRATION VISUALS
#
# Run this as one complete Jupyter cell after ND10 and before ND11.
#
# PURPOSE
#   Create a comprehensive report-ready diagnostic archive for the revised
#   normal-demand forecasting system selected for the final demonstration.
#
# IMPORTANT
#   - This is a reporting/diagnostic step only.
#   - No model is fitted, refitted, tuned, calibrated, or reselected.
#   - March 2026 target data are never opened.
#   - Accuracy figures come only from the pre-March chronological development
#     evaluations already completed in ND04-ND08.
#   - ND09/ND10 future forecasts are shown only as demonstration outputs; they
#     are not scored because their future actual targets remain closed.
# =============================================================================

from __future__ import annotations

import hashlib
import json
import math
import os
import platform
import shutil
import textwrap
import uuid
import warnings
import zipfile
from datetime import datetime, timezone
from pathlib import Path
from zoneinfo import ZoneInfo

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

warnings.filterwarnings("ignore", category=RuntimeWarning)

# =============================================================================
# USER CONFIGURATION
# =============================================================================

ALLOW_OVERWRITE = False
TOP_PRODUCTS_IN_ERROR_FIGURE = 20
TOP_PRODUCTS_IN_DEMO_FIGURE = 20
SCATTER_MAX_POINTS = 12_000

# =============================================================================
# PROJECT PATHS
# =============================================================================

PROJECT_ROOT = Path("/Users/ryansmac/Desktop/Meng Project")
EDEN_ROOT = PROJECT_ROOT / "eden_datasets"
MODEL_ROOT = EDEN_ROOT / "eden_normal_demand_model_v2"

ND03_ROOT = MODEL_ROOT / "02_feature_engineering" / "ND03_normal_demand_features"
ND04_ROOT = MODEL_ROOT / "03_models" / "00_candidates" / "ND04_baseline_and_fallback_evaluation"
ND05_ROOT = MODEL_ROOT / "03_models" / "00_candidates" / "ND05_original_hurdle_model_rebuild"
ND06_ROOT = MODEL_ROOT / "03_models" / "00_candidates" / "ND06_expanded_model_challenge"
ND07_ROOT = MODEL_ROOT / "03_models" / "01_tuning" / "ND07_tuning_calibration_blending"
ND07E_ROOT = MODEL_ROOT / "03_models" / "01_tuning" / "ND07E_focused_ensemble_challenge"
ND08_ROOT = MODEL_ROOT / "03_models" / "02_system_evaluation" / "ND08_recursive_weekly_system"
ND09_ROOT = MODEL_ROOT / "03_models" / "03_inference" / "ND09_arbitrary_date_inference_engine"
ND10_ROOT = MODEL_ROOT / "04_planning" / "ND10_confirmed_bulk_and_planned_quantity"

# Core reporting inputs.
ND03_ALL_ROUTES = ND03_ROOT / "01_model_ready_datasets" / "ND03_pre_march_all_routes_development_dataset.csv"
ND06_CANDIDATE_METRICS = ND06_ROOT / "02_metrics" / "ND06_candidate_metrics.csv"
ND07_CANDIDATE_METRICS = ND07_ROOT / "02_metrics" / "ND07_outer_candidate_metrics.csv"
ND07_FOLD_METRICS = ND07_ROOT / "02_metrics" / "ND07_outer_candidate_metrics_by_fold.csv"

ND07E_CANDIDATE_METRICS = ND07E_ROOT / "02_metrics" / "ND07E_outer_candidate_metrics.csv"
ND07E_FOLD_METRICS = ND07E_ROOT / "02_metrics" / "ND07E_outer_candidate_metrics_by_fold.csv"
ND07E_ROUTED_PREDICTIONS = ND07E_ROOT / "01_predictions" / "ND07E_selected_routed_predictions.csv"
ND07E_DAILY_RESTAURANT = ND07E_ROOT / "01_predictions" / "ND07E_selected_daily_restaurant_totals.csv"
ND07E_FINAL_SELECTION = ND07E_ROOT / "02_metrics" / "ND07E_final_ensemble_selection.csv"

ND08_RECURSIVE_DAILY = ND08_ROOT / "01_predictions" / "ND08_monday_origin_recursive_daily_predictions.csv"
ND08_RECURSIVE_PRODUCT_WEEK = ND08_ROOT / "01_predictions" / "ND08_monday_origin_product_week_predictions.csv"
ND08_RECURSIVE_RESTAURANT_WEEK = ND08_ROOT / "01_predictions" / "ND08_monday_origin_restaurant_week_predictions.csv"
ND08_DAILY_UPDATED_PRODUCT_WEEK = ND08_ROOT / "01_predictions" / "ND08_daily_updated_product_week_predictions.csv"
ND08_DAILY_UPDATED_RESTAURANT_WEEK = ND08_ROOT / "01_predictions" / "ND08_daily_updated_restaurant_week_predictions.csv"
ND08_RECURSIVE_METRICS = ND08_ROOT / "02_metrics" / "ND08_monday_origin_metrics.csv"
ND08_RECURSIVE_FOLD_METRICS = ND08_ROOT / "02_metrics" / "ND08_monday_origin_product_week_metrics_by_fold.csv"
ND08_RECURSIVE_WEEK_METRICS = ND08_ROOT / "02_metrics" / "ND08_monday_origin_product_week_metrics_by_week.csv"
ND08_DAILY_UPDATED_METRICS = ND08_ROOT / "02_metrics" / "ND08_daily_updated_metrics.csv"
ND08_DAILY_UPDATED_FOLD_METRICS = ND08_ROOT / "02_metrics" / "ND08_daily_updated_product_week_metrics_by_fold.csv"
ND08_DAILY_MAIN_METRICS = ND08_ROOT / "02_metrics" / "ND08_daily_main_route_metrics.csv"
ND08_ROUTE_METRICS = ND08_ROOT / "02_metrics" / "ND08_recursive_metrics_by_route.csv"
ND08_SELECTION = ND08_ROOT / "02_metrics" / "ND08_operational_method_selection.csv"
ND08_ROUTE_CHANGE_AUDIT = ND08_ROOT / "03_audits" / "ND08_recursive_route_change_audit.csv"
ND08_WEEK_REGISTRY = ND08_ROOT / "03_audits" / "ND08_complete_week_registry.csv"
ND08_CONTRACT = ND08_ROOT / "04_contracts" / "ND08_operational_method_contract.json"

ND09_CONTRACT = ND09_ROOT / "03_contracts" / "ND09_inference_engine_contract.json"
ND09_DAILY_FORECAST = ND09_ROOT / "02_forecasts" / "ND09_arbitrary_date_product_forecast.csv"
ND09_WEEK_DAILY_FORECAST = ND09_ROOT / "02_forecasts" / "ND09_week_ahead_daily_product_forecast.csv"

ND10_DAILY_PRODUCT = ND10_ROOT / "02_planned_outputs" / "ND10_daily_product_planning.csv"
ND10_DAILY_RESTAURANT = ND10_ROOT / "02_planned_outputs" / "ND10_daily_restaurant_planning.csv"
ND10_WEEK_DAILY_PRODUCT = ND10_ROOT / "02_planned_outputs" / "ND10_week_daily_product_planning.csv"
ND10_WEEK_PRODUCT_TOTALS = ND10_ROOT / "02_planned_outputs" / "ND10_week_product_totals_planning.csv"
ND10_WEEK_RESTAURANT_DAILY = ND10_ROOT / "02_planned_outputs" / "ND10_week_restaurant_daily_planning.csv"
ND10_WEEK_RESTAURANT_TOTAL = ND10_ROOT / "02_planned_outputs" / "ND10_week_restaurant_total_planning.csv"
ND10_PLANNING_CONTRACT = ND10_ROOT / "03_contracts" / "ND10_planning_quantity_contract.json"

CHECKPOINT_PATHS = {
    "ND03": MODEL_ROOT / "08_checkpoints" / "ND03_checkpoint.json",
    "ND04": MODEL_ROOT / "08_checkpoints" / "ND04_checkpoint.json",
    "ND05": MODEL_ROOT / "08_checkpoints" / "ND05_checkpoint.json",
    "ND06": MODEL_ROOT / "08_checkpoints" / "ND06_checkpoint.json",
    "ND07": MODEL_ROOT / "08_checkpoints" / "ND07_checkpoint.json",
    "ND07E": MODEL_ROOT / "08_checkpoints" / "ND07E_checkpoint.json",
    "ND08": MODEL_ROOT / "08_checkpoints" / "ND08_checkpoint.json",
    "ND09": MODEL_ROOT / "08_checkpoints" / "ND09_checkpoint.json",
    "ND10": MODEL_ROOT / "08_checkpoints" / "ND10_checkpoint.json",
}

EXPECTED_CHECKPOINT_HASHES = {
    "ND03": "0845af89a5b459ca13ae6ffd99dde444f5010f6c0fb5ba4c34a4f091ac2e151c",
    "ND04": "2fdc5d2c64c38f85b2669ca942042884209d80111cc840261307da98b1e9cf54",
    "ND05": "ce3342c8b960aa5c4791a114ab09ae1a86d1dab2a3eebb648aa060579a1378ff",
    "ND06": "e3b7bb75a1b2968e426c9a4c1654e10420d683af4bd5cb2b75fa4b5f0f18f357",
    "ND07": "39077dd8c197561e5384a6943c3a4e153153e75fa4d03019f45005eb145cf18e",
    "ND07E": "eaf6d23434cb67d663f66a39db4f898b74c1d516aee5c0627ca9136bed9877b8",
    "ND08": "a98828007df9bd4d0cce309fc8f8bfbc1d2c1f1cf74f1ca762b27237468dea10",
    "ND09": "201b84095ed138e0b666485e2022e4f1f99ad93b89971f46533ff4a0605ef4e8",
    "ND10": "af85edca8cd923fe1141128406ec7075ea778f5aabe0d35b567e51236b6a7c55",
}

# Output structure.
ND10R_ROOT = MODEL_ROOT / "05_reporting" / "ND10R_report_ready_diagnostics_and_visuals"
TABLE_DIR = ND10R_ROOT / "01_report_tables"
DIAGNOSTIC_DIR = ND10R_ROOT / "02_diagnostics"
FIGURE_DIR = ND10R_ROOT / "03_figures"
REPORT_DIR = ND10R_ROOT / "04_report_wording"
DEMO_DIR = ND10R_ROOT / "05_demonstration_material"
CONTROL_DIR = ND10R_ROOT / "06_control"

CHECKPOINT_PATH = CONTROL_DIR / "ND10R_checkpoint.json"
CHECKPOINT_SHA_PATH = CONTROL_DIR / "ND10R_checkpoint.sha256"
MANIFEST_PATH = CONTROL_DIR / "ND10R_artifact_hash_manifest.csv"
BUNDLE_PATH = CONTROL_DIR / "ND10R_report_ready_bundle.zip"
README_PATH = ND10R_ROOT / "README.md"

TOP_LEVEL_CHECKPOINT_PATH = MODEL_ROOT / "08_checkpoints" / "ND10R_checkpoint.json"
TOP_LEVEL_CHECKPOINT_SHA_PATH = MODEL_ROOT / "08_checkpoints" / "ND10R_checkpoint.sha256"

MEMORY_ROOT = MODEL_ROOT / "00_project_memory"
HANDOFF_PATH = MEMORY_ROOT / "ND10R_HANDOFF.md"
CURRENT_HANDOFF_PATH = MEMORY_ROOT / "CURRENT_HANDOFF.md"
WORKFLOW_PATH = MEMORY_ROOT / "WORKFLOW.md"
DECISIONS_PATH = MEMORY_ROOT / "DECISIONS.md"
METRICS_PATH = MEMORY_ROOT / "METRICS_AND_RESULTS.md"
AGENTS_PATH = MODEL_ROOT / "AGENTS.md"
LOG_PATH = MODEL_ROOT / "09_logs" / "ND10R_report_ready_diagnostics_log.txt"

STEP_ID = "ND10R"
STATUS = "ND10R_REPORT_READY_DIAGNOSTICS_AND_VISUALS_CREATED_READY_FOR_ND11"
NOW_UTC = datetime.now(timezone.utc)
NOW_LOCAL = NOW_UTC.astimezone(ZoneInfo("Europe/Dublin"))

DATE = "Date"
PID = "CanonicalProductID"
PNAME = "CanonicalProductName"
ROUTE = "ForecastRoute"
ACTUAL = "ActualNormalDemand"
PRED = "PredictedNormalDemand"

# =============================================================================
# HELPERS
# =============================================================================

def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


def write_csv(path: Path, frame: pd.DataFrame) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    frame.to_csv(path, index=False)


def write_json(path: Path, payload: dict) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(payload, indent=2, ensure_ascii=False, default=str) + "\n", encoding="utf-8")


def write_text(path: Path, text: str) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(text, encoding="utf-8")


def atomic_write_text(path: Path, text: str) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_name(f".{path.name}.{uuid.uuid4().hex}.tmp")
    temporary.write_text(text, encoding="utf-8")
    os.replace(temporary, path)


def append_once(path: Path, marker: str, text: str) -> None:
    existing = path.read_text(encoding="utf-8") if path.is_file() else ""
    if marker in existing:
        return
    separator = "\n" if existing.endswith("\n") else "\n\n"
    atomic_write_text(path, existing + separator + text.strip() + "\n")


def require_columns(frame: pd.DataFrame, columns: list[str] | set[str], label: str) -> None:
    missing = sorted(set(columns) - set(frame.columns))
    if missing:
        raise AssertionError(f"{label} is missing columns: {missing}")


def wape(actual, predicted) -> float:
    a = np.asarray(actual, dtype=float)
    p = np.asarray(predicted, dtype=float)
    denominator = float(a.sum())
    return float("nan") if denominator == 0 else float(100.0 * np.abs(p - a).sum() / denominator)


def metric_record(actual, predicted, level: str = "") -> dict:
    a = np.asarray(actual, dtype=float)
    p = np.asarray(predicted, dtype=float)
    e = p - a
    total = float(a.sum())
    bias = float(e.sum())
    return {
        "EvaluationLevel": level,
        "Observations": int(len(a)),
        "ActualTotal": total,
        "PredictedTotal": float(p.sum()),
        "WAPEPercentage": wape(a, p),
        "MAE": float(np.mean(np.abs(e))) if len(e) else float("nan"),
        "RMSE": float(np.sqrt(np.mean(np.square(e)))) if len(e) else float("nan"),
        "MeanBias": float(np.mean(e)) if len(e) else float("nan"),
        "TotalBias": bias,
        "AbsoluteBiasPercentage": float(100.0 * abs(bias) / total) if total else float("nan"),
    }


def save_figure(path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    plt.tight_layout()
    plt.savefig(path, dpi=300, bbox_inches="tight")
    plt.close()


def short_method(method: str) -> str:
    mapping = {
        "ROLLING_MEAN_5": "Rolling mean 5",
        "ROLLING_MEAN_5_BENCHMARK": "Rolling mean 5",
        "CATBOOST_RMSE_CORE53": "CatBoost",
        "NESTED_TUNED_CATBOOST_RMSE_CORE53": "Nested CatBoost",
        "NESTED_TUNED_XGBOOST_SQUARED_CORE53": "Nested XGBoost",
        "NESTED_TUNED_XGBOOST_SQUARED_PRODUCT_AWARE": "Nested product-aware XGBoost",
        "NESTED_MEDIAN_ENSEMBLE": "Median ensemble",
        "NESTED_WEIGHTED_ENSEMBLE_4_MODELS": "Weighted 4-model ensemble",
        "NESTED_WEIGHTED_ENSEMBLE_3_MODELS": "Weighted 3-model ensemble",
        "NESTED_WEIGHTED_ENSEMBLE_2_MODELS": "Weighted 2-model ensemble",
    }
    return mapping.get(str(method), str(method).replace("_", " ").title())


def pick_metric_row(frame: pd.DataFrame, method: str, evaluation_level: str | None = None) -> pd.Series:
    mask = frame["CandidateMethod"].astype(str) == method
    if evaluation_level is not None and "EvaluationLevel" in frame.columns:
        mask &= frame["EvaluationLevel"].astype(str) == evaluation_level
    rows = frame.loc[mask]
    if len(rows) != 1:
        raise AssertionError(f"Expected one metric row for {method}/{evaluation_level}; found {len(rows)}")
    return rows.iloc[0]


def build_manifest(root: Path) -> pd.DataFrame:
    records = []
    for path in sorted(root.rglob("*")):
        if not path.is_file():
            continue
        if path == BUNDLE_PATH or path == MANIFEST_PATH or path == CHECKPOINT_PATH or path == CHECKPOINT_SHA_PATH:
            continue
        records.append({
            "RelativePath": str(path.relative_to(root)),
            "Bytes": int(path.stat().st_size),
            "SHA256": sha256_file(path),
        })
    return pd.DataFrame(records)


def wrap_label(text: str, width: int = 24) -> str:
    return "\n".join(textwrap.wrap(str(text), width=width))


# =============================================================================
# PREFLIGHT
# =============================================================================

required_inputs = [
    ND03_ALL_ROUTES,
    ND06_CANDIDATE_METRICS,
    ND07_CANDIDATE_METRICS,
    ND07_FOLD_METRICS,
    ND07E_CANDIDATE_METRICS,
    ND07E_FOLD_METRICS,
    ND07E_ROUTED_PREDICTIONS,
    ND07E_DAILY_RESTAURANT,
    ND07E_FINAL_SELECTION,
    ND08_RECURSIVE_DAILY,
    ND08_RECURSIVE_PRODUCT_WEEK,
    ND08_RECURSIVE_RESTAURANT_WEEK,
    ND08_DAILY_UPDATED_PRODUCT_WEEK,
    ND08_DAILY_UPDATED_RESTAURANT_WEEK,
    ND08_RECURSIVE_METRICS,
    ND08_RECURSIVE_FOLD_METRICS,
    ND08_RECURSIVE_WEEK_METRICS,
    ND08_DAILY_UPDATED_METRICS,
    ND08_DAILY_UPDATED_FOLD_METRICS,
    ND08_DAILY_MAIN_METRICS,
    ND08_ROUTE_METRICS,
    ND08_SELECTION,
    ND08_ROUTE_CHANGE_AUDIT,
    ND08_WEEK_REGISTRY,
    ND08_CONTRACT,
    ND09_CONTRACT,
    ND09_DAILY_FORECAST,
    ND09_WEEK_DAILY_FORECAST,
    ND10_DAILY_PRODUCT,
    ND10_DAILY_RESTAURANT,
    ND10_WEEK_DAILY_PRODUCT,
    ND10_WEEK_PRODUCT_TOTALS,
    ND10_WEEK_RESTAURANT_DAILY,
    ND10_WEEK_RESTAURANT_TOTAL,
    ND10_PLANNING_CONTRACT,
    *CHECKPOINT_PATHS.values(),
]

missing = [path for path in required_inputs if not path.is_file()]
if missing:
    raise FileNotFoundError("Required ND10R inputs are missing:\n" + "\n".join(f"- {path}" for path in missing))

checkpoint_hashes = {name: sha256_file(path) for name, path in CHECKPOINT_PATHS.items()}
for name, expected in EXPECTED_CHECKPOINT_HASHES.items():
    if checkpoint_hashes[name] != expected:
        raise AssertionError(
            f"{name} checkpoint hash mismatch.\nExpected: {expected}\nActual:   {checkpoint_hashes[name]}"
        )

if ND10R_ROOT.exists():
    if not ALLOW_OVERWRITE:
        raise FileExistsError(
            "ND10R output already exists. No files were changed:\n"
            f"{ND10R_ROOT}\nSet ALLOW_OVERWRITE=True only when intentionally regenerating the report package."
        )
    shutil.rmtree(ND10R_ROOT)

for path in [TOP_LEVEL_CHECKPOINT_PATH, TOP_LEVEL_CHECKPOINT_SHA_PATH]:
    if path.exists():
        if not ALLOW_OVERWRITE:
            raise FileExistsError(f"Existing ND10R control artifact found: {path}")
        path.unlink()

protected_hashes_before = {str(path): sha256_file(path) for path in required_inputs}
STAGING_ROOT = ND10R_ROOT.parent / f".ND10R_staging_{uuid.uuid4().hex}"
STAGING_ROOT.mkdir(parents=True, exist_ok=False)

# =============================================================================
# LOAD DATA
# =============================================================================

try:
    all_routes = pd.read_csv(ND03_ALL_ROUTES, low_memory=False)
    nd06_metrics = pd.read_csv(ND06_CANDIDATE_METRICS, low_memory=False)
    nd07_metrics = pd.read_csv(ND07_CANDIDATE_METRICS, low_memory=False)
    nd07_fold = pd.read_csv(ND07_FOLD_METRICS, low_memory=False)
    nd07e_metrics = pd.read_csv(ND07E_CANDIDATE_METRICS, low_memory=False)
    nd07e_fold = pd.read_csv(ND07E_FOLD_METRICS, low_memory=False)
    routed = pd.read_csv(ND07E_ROUTED_PREDICTIONS, low_memory=False)
    daily_restaurant = pd.read_csv(ND07E_DAILY_RESTAURANT, low_memory=False)
    nd07e_selection = pd.read_csv(ND07E_FINAL_SELECTION, low_memory=False)

    recursive_daily = pd.read_csv(ND08_RECURSIVE_DAILY, low_memory=False)
    recursive_product_week = pd.read_csv(ND08_RECURSIVE_PRODUCT_WEEK, low_memory=False)
    recursive_restaurant_week = pd.read_csv(ND08_RECURSIVE_RESTAURANT_WEEK, low_memory=False)
    updated_product_week = pd.read_csv(ND08_DAILY_UPDATED_PRODUCT_WEEK, low_memory=False)
    updated_restaurant_week = pd.read_csv(ND08_DAILY_UPDATED_RESTAURANT_WEEK, low_memory=False)
    recursive_metrics = pd.read_csv(ND08_RECURSIVE_METRICS, low_memory=False)
    recursive_fold = pd.read_csv(ND08_RECURSIVE_FOLD_METRICS, low_memory=False)
    recursive_week_metrics = pd.read_csv(ND08_RECURSIVE_WEEK_METRICS, low_memory=False)
    updated_metrics = pd.read_csv(ND08_DAILY_UPDATED_METRICS, low_memory=False)
    updated_fold = pd.read_csv(ND08_DAILY_UPDATED_FOLD_METRICS, low_memory=False)
    daily_main_metrics = pd.read_csv(ND08_DAILY_MAIN_METRICS, low_memory=False)
    route_metrics = pd.read_csv(ND08_ROUTE_METRICS, low_memory=False)
    operational_selection = pd.read_csv(ND08_SELECTION, low_memory=False)
    route_change = pd.read_csv(ND08_ROUTE_CHANGE_AUDIT, low_memory=False)
    week_registry = pd.read_csv(ND08_WEEK_REGISTRY, low_memory=False)
    nd08_contract = json.loads(ND08_CONTRACT.read_text(encoding="utf-8"))
    nd09_contract = json.loads(ND09_CONTRACT.read_text(encoding="utf-8"))

    demo_daily = pd.read_csv(ND10_DAILY_PRODUCT, low_memory=False)
    demo_daily_restaurant = pd.read_csv(ND10_DAILY_RESTAURANT, low_memory=False)
    demo_week_daily = pd.read_csv(ND10_WEEK_DAILY_PRODUCT, low_memory=False)
    demo_week_products = pd.read_csv(ND10_WEEK_PRODUCT_TOTALS, low_memory=False)
    demo_week_restaurant_daily = pd.read_csv(ND10_WEEK_RESTAURANT_DAILY, low_memory=False)
    demo_week_restaurant_total = pd.read_csv(ND10_WEEK_RESTAURANT_TOTAL, low_memory=False)
    nd10_contract = json.loads(ND10_PLANNING_CONTRACT.read_text(encoding="utf-8"))

    # Dates.
    for frame in [routed, daily_restaurant, recursive_daily, demo_daily, demo_week_daily, demo_week_restaurant_daily]:
        if DATE in frame.columns:
            frame[DATE] = pd.to_datetime(frame[DATE], errors="raise")
    for frame in [recursive_product_week, recursive_restaurant_week, updated_product_week, updated_restaurant_week, recursive_week_metrics, week_registry]:
        if "WeekStart" in frame.columns:
            frame["WeekStart"] = pd.to_datetime(frame["WeekStart"], errors="raise")

    # Schema checks.
    require_columns(routed, [DATE, PID, PNAME, ROUTE, ACTUAL, PRED], "ND07E selected routed predictions")
    require_columns(daily_restaurant, [DATE, ACTUAL, PRED], "ND07E daily restaurant totals")
    require_columns(nd07e_metrics, ["CandidateMethod", "WAPEPercentage", "MAE", "RMSE", "AbsoluteBiasPercentage"], "ND07E candidate metrics")
    require_columns(nd07e_fold, ["CandidateMethod", "WAPEPercentage"], "ND07E fold metrics")
    require_columns(recursive_daily, [PID, PNAME, "CandidateMethod", "RecursiveDayNumber", "ActualNormalDemand", "PredictedNormalDemand", "RecursiveForecastRoute"], "ND08 recursive daily predictions")
    require_columns(recursive_metrics, ["CandidateMethod", "EvaluationLevel", "WAPEPercentage", "MAE", "RMSE", "AbsoluteBiasPercentage"], "ND08 recursive metrics")
    require_columns(updated_metrics, ["CandidateMethod", "EvaluationLevel", "WAPEPercentage", "MAE", "RMSE", "AbsoluteBiasPercentage"], "ND08 daily-updated metrics")
    require_columns(recursive_week_metrics, ["CandidateMethod", "WeekStart", "WAPEPercentage"], "ND08 weekly metrics")
    require_columns(demo_daily, [PID, PNAME, "PredictedNormalDemand", "ConfirmedBulkDemand", "PlannedQuantity"], "ND10 daily product planning")
    require_columns(demo_week_products, [PID, PNAME, "PredictedWeekNormalDemand", "ConfirmedWeekBulkDemand", "PlannedWeekQuantity"], "ND10 week product planning")
    require_columns(demo_week_restaurant_daily, [DATE], "ND10 week restaurant daily planning")

    # No March target source is loaded. ND03 is pre-March by contract.
    if "Date" in all_routes.columns:
        all_routes["Date"] = pd.to_datetime(all_routes["Date"], errors="raise")
        if not (all_routes["Date"] < pd.Timestamp("2026-03-02")).all():
            raise AssertionError("ND10R loaded March-or-later target rows, which is prohibited.")

    selected_daily_method = str(nd08_contract["DailyProductMethod"])
    selected_week_method = str(nd08_contract["MondayOriginWeeklyPlanningMethod"])
    selected_updated_method = str(nd08_contract["RollingDailyUpdatedWeeklyMethod"])

    if selected_daily_method != "NESTED_MEDIAN_ENSEMBLE":
        raise AssertionError("Unexpected ND08 daily method.")
    if selected_week_method != "ROLLING_MEAN_5":
        raise AssertionError("Unexpected ND08 week-start method.")
    if selected_updated_method != "ROLLING_MEAN_5":
        raise AssertionError("Unexpected ND08 daily-updated weekly method.")

    # =========================================================================
    # REPORT TABLES AND DIAGNOSTICS
    # =========================================================================

    routed["ForecastError"] = pd.to_numeric(routed[PRED], errors="raise") - pd.to_numeric(routed[ACTUAL], errors="raise")
    routed["AbsoluteError"] = routed["ForecastError"].abs()
    routed["AbsolutePercentageError"] = np.where(
        pd.to_numeric(routed[ACTUAL], errors="coerce") > 0,
        100.0 * routed["AbsoluteError"] / pd.to_numeric(routed[ACTUAL], errors="coerce"),
        np.nan,
    )

    # Official development diagnostic metrics for the selected routed daily system.
    selected_routed_metric = metric_record(routed[ACTUAL], routed[PRED], "DAILY_PRODUCT_ALL_ROUTES")
    selected_restaurant_metric = metric_record(daily_restaurant[ACTUAL], daily_restaurant[PRED], "DAILY_RESTAURANT_TOTAL_ALL_ROUTES")

    # Report-ready key metric table.
    daily_main_selected = pick_metric_row(daily_main_metrics, selected_daily_method)
    recursive_product_selected = pick_metric_row(recursive_metrics, selected_week_method, "PRODUCT_WEEK")
    recursive_restaurant_selected = pick_metric_row(recursive_metrics, selected_week_method, "RESTAURANT_WEEK")
    updated_product_selected = pick_metric_row(updated_metrics, selected_updated_method, "PRODUCT_WEEK")
    updated_restaurant_selected = pick_metric_row(updated_metrics, selected_updated_method, "RESTAURANT_WEEK")

    report_metrics = pd.DataFrame([
        {
            "ResultContext": "Daily main-route development validation",
            "SelectedMethod": selected_daily_method,
            "ForecastMode": "ONE_STEP_DAILY",
            "WAPEPercentage": float(daily_main_selected["WAPEPercentage"]),
            "MAE": float(daily_main_selected["MAE"]),
            "RMSE": float(daily_main_selected["RMSE"]),
            "AbsoluteBiasPercentage": float(daily_main_selected["AbsoluteBiasPercentage"]),
            "PrimaryInterpretation": "Selected daily product method",
        },
        {
            "ResultContext": "Daily routed system development validation",
            "SelectedMethod": selected_daily_method,
            "ForecastMode": "ONE_STEP_DAILY_ALL_ROUTES",
            "WAPEPercentage": float(selected_routed_metric["WAPEPercentage"]),
            "MAE": float(selected_routed_metric["MAE"]),
            "RMSE": float(selected_routed_metric["RMSE"]),
            "AbsoluteBiasPercentage": float(selected_routed_metric["AbsoluteBiasPercentage"]),
            "PrimaryInterpretation": "Full product catalogue with fallback routes",
        },
        {
            "ResultContext": "Daily restaurant total development validation",
            "SelectedMethod": selected_daily_method,
            "ForecastMode": "ONE_STEP_DAILY_ALL_ROUTES",
            "WAPEPercentage": float(selected_restaurant_metric["WAPEPercentage"]),
            "MAE": float(selected_restaurant_metric["MAE"]),
            "RMSE": float(selected_restaurant_metric["RMSE"]),
            "AbsoluteBiasPercentage": float(selected_restaurant_metric["AbsoluteBiasPercentage"]),
            "PrimaryInterpretation": "Restaurant-level aggregation of routed daily product forecasts",
        },
        {
            "ResultContext": "Monday-origin product-week development evaluation",
            "SelectedMethod": selected_week_method,
            "ForecastMode": "MONDAY_ORIGIN_RECURSIVE",
            "WAPEPercentage": float(recursive_product_selected["WAPEPercentage"]),
            "MAE": float(recursive_product_selected["MAE"]),
            "RMSE": float(recursive_product_selected["RMSE"]),
            "AbsoluteBiasPercentage": float(recursive_product_selected["AbsoluteBiasPercentage"]),
            "PrimaryInterpretation": "Complete week forecast before Monday",
        },
        {
            "ResultContext": "Monday-origin restaurant-week development evaluation",
            "SelectedMethod": selected_week_method,
            "ForecastMode": "MONDAY_ORIGIN_RECURSIVE",
            "WAPEPercentage": float(recursive_restaurant_selected["WAPEPercentage"]),
            "MAE": float(recursive_restaurant_selected["MAE"]),
            "RMSE": float(recursive_restaurant_selected["RMSE"]),
            "AbsoluteBiasPercentage": float(recursive_restaurant_selected["AbsoluteBiasPercentage"]),
            "PrimaryInterpretation": "Restaurant total for a complete week before Monday",
        },
        {
            "ResultContext": "Daily-updated product-week development evaluation",
            "SelectedMethod": selected_updated_method,
            "ForecastMode": "DAILY_UPDATED_ACTUAL_HISTORY",
            "WAPEPercentage": float(updated_product_selected["WAPEPercentage"]),
            "MAE": float(updated_product_selected["MAE"]),
            "RMSE": float(updated_product_selected["RMSE"]),
            "AbsoluteBiasPercentage": float(updated_product_selected["AbsoluteBiasPercentage"]),
            "PrimaryInterpretation": "Remaining-week updates after observed daily demand",
        },
        {
            "ResultContext": "Daily-updated restaurant-week development evaluation",
            "SelectedMethod": selected_updated_method,
            "ForecastMode": "DAILY_UPDATED_ACTUAL_HISTORY",
            "WAPEPercentage": float(updated_restaurant_selected["WAPEPercentage"]),
            "MAE": float(updated_restaurant_selected["MAE"]),
            "RMSE": float(updated_restaurant_selected["RMSE"]),
            "AbsoluteBiasPercentage": float(updated_restaurant_selected["AbsoluteBiasPercentage"]),
            "PrimaryInterpretation": "Restaurant week total updated as actuals arrive",
        },
    ])

    # Model-selection journey table.
    journey_rows = []

    # ND04 benchmark from ND07E benchmark row (authoritative reproduction).
    benchmark_row = pick_metric_row(nd07e_metrics, "ROLLING_MEAN_5_BENCHMARK")
    journey_rows.append({
        "Stage": "ND04",
        "StageLabel": "Baseline",
        "Method": "ROLLING_MEAN_5",
        "MethodLabel": "Rolling mean 5",
        "WAPEPercentage": float(benchmark_row["WAPEPercentage"]),
        "MAE": float(benchmark_row["MAE"]),
        "RMSE": float(benchmark_row["RMSE"]),
        "Interpretation": "Accepted main-route benchmark",
    })

    # ND06 CatBoost screen winner.
    if (nd06_metrics["CandidateMethod"].astype(str) == "CATBOOST_RMSE_CORE53").any():
        row = nd06_metrics.loc[nd06_metrics["CandidateMethod"].astype(str) == "CATBOOST_RMSE_CORE53"].iloc[0]
        journey_rows.append({
            "Stage": "ND06",
            "StageLabel": "Advanced model screen",
            "Method": "CATBOOST_RMSE_CORE53",
            "MethodLabel": "CatBoost",
            "WAPEPercentage": float(row["WAPEPercentage"]),
            "MAE": float(row["MAE"]),
            "RMSE": float(row["RMSE"]),
            "Interpretation": "Best individual challenger in initial screen",
        })

    # ND07 nested methods.
    for method in [
        "NESTED_TUNED_CATBOOST_RMSE_CORE53",
        "NESTED_TUNED_XGBOOST_SQUARED_CORE53",
        "NESTED_TUNED_XGBOOST_SQUARED_PRODUCT_AWARE",
    ]:
        rows = nd07_metrics.loc[nd07_metrics["CandidateMethod"].astype(str) == method]
        if len(rows) == 1:
            row = rows.iloc[0]
            journey_rows.append({
                "Stage": "ND07",
                "StageLabel": "Nested tuning",
                "Method": method,
                "MethodLabel": short_method(method),
                "WAPEPercentage": float(row["WAPEPercentage"]),
                "MAE": float(row["MAE"]),
                "RMSE": float(row["RMSE"]),
                "Interpretation": "Outer validation after inner-only tuning/calibration",
            })

    # ND07E ensemble finalists.
    for method in [
        "NESTED_MEDIAN_ENSEMBLE",
        "NESTED_WEIGHTED_ENSEMBLE_4_MODELS",
        "NESTED_WEIGHTED_ENSEMBLE_3_MODELS",
        "NESTED_WEIGHTED_ENSEMBLE_2_MODELS",
    ]:
        rows = nd07e_metrics.loc[nd07e_metrics["CandidateMethod"].astype(str) == method]
        if len(rows) == 1:
            row = rows.iloc[0]
            journey_rows.append({
                "Stage": "ND07E",
                "StageLabel": "Focused ensemble challenge",
                "Method": method,
                "MethodLabel": short_method(method),
                "WAPEPercentage": float(row["WAPEPercentage"]),
                "MAE": float(row["MAE"]),
                "RMSE": float(row["RMSE"]),
                "Interpretation": "Nested ensemble selected using inner validation only",
            })
    model_journey = pd.DataFrame(journey_rows)

    # Product diagnostics from selected routed system.
    product_diag = (
        routed.groupby([PID, PNAME], as_index=False)
        .agg(
            Observations=(DATE, "size"),
            ActualNormalDemand=(ACTUAL, "sum"),
            PredictedNormalDemand=(PRED, "sum"),
            TotalAbsoluteError=("AbsoluteError", "sum"),
            MeanAbsoluteError=("AbsoluteError", "mean"),
            RMSE=("ForecastError", lambda x: float(np.sqrt(np.mean(np.square(x))))),
            TotalBias=("ForecastError", "sum"),
            PositiveActualDays=(ACTUAL, lambda x: int((pd.to_numeric(x, errors="coerce") > 0).sum())),
        )
    )
    product_diag["WAPEPercentage"] = np.where(
        product_diag["ActualNormalDemand"] > 0,
        100.0 * product_diag["TotalAbsoluteError"] / product_diag["ActualNormalDemand"],
        np.nan,
    )
    product_diag["AbsoluteBiasPercentage"] = np.where(
        product_diag["ActualNormalDemand"] > 0,
        100.0 * product_diag["TotalBias"].abs() / product_diag["ActualNormalDemand"],
        np.nan,
    )
    product_diag = product_diag.sort_values("TotalAbsoluteError", ascending=False).reset_index(drop=True)
    worst_products = product_diag.head(TOP_PRODUCTS_IN_ERROR_FIGURE).copy()

    # Date diagnostics.
    date_diag = (
        routed.groupby(DATE, as_index=False)
        .agg(
            ActualNormalDemand=(ACTUAL, "sum"),
            PredictedNormalDemand=(PRED, "sum"),
            ProductRows=(PID, "size"),
        )
        .sort_values(DATE)
    )
    date_diag["ForecastError"] = date_diag["PredictedNormalDemand"] - date_diag["ActualNormalDemand"]
    date_diag["AbsoluteError"] = date_diag["ForecastError"].abs()
    date_diag["AbsolutePercentageError"] = np.where(
        date_diag["ActualNormalDemand"] > 0,
        100.0 * date_diag["AbsoluteError"] / date_diag["ActualNormalDemand"],
        np.nan,
    )
    worst_dates = date_diag.sort_values("AbsoluteError", ascending=False).head(15).copy()
    best_dates = date_diag.sort_values("AbsoluteError", ascending=True).head(15).copy()

    # Tolerance diagnostics, deliberately descriptive rather than official metrics.
    positive = routed.loc[pd.to_numeric(routed[ACTUAL], errors="coerce") > 0].copy()
    tolerance_rows = []
    for units in [1, 2, 5, 10]:
        tolerance_rows.append({
            "Diagnostic": f"Absolute error within ±{units} units",
            "Denominator": "All product-day rows",
            "Rows": int(len(routed)),
            "Percentage": float(100.0 * (routed["AbsoluteError"] <= units).mean()),
        })
    for pct in [20, 30, 50]:
        tolerance_rows.append({
            "Diagnostic": f"Absolute percentage error within ±{pct}%",
            "Denominator": "Positive-demand product-day rows",
            "Rows": int(len(positive)),
            "Percentage": float(100.0 * (positive["AbsolutePercentageError"] <= pct).mean()),
        })
    tolerance_diag = pd.DataFrame(tolerance_rows)

    # Demand concentration on validation rows.
    concentration = product_diag.sort_values("ActualNormalDemand", ascending=False).copy()
    validation_total = float(concentration["ActualNormalDemand"].sum())
    concentration["DemandSharePercentage"] = np.where(
        validation_total > 0,
        100.0 * concentration["ActualNormalDemand"] / validation_total,
        np.nan,
    )
    concentration["CumulativeDemandSharePercentage"] = concentration["DemandSharePercentage"].cumsum()
    concentration["DemandRank"] = np.arange(1, len(concentration) + 1)
    coverage_rows = []
    for threshold in [50, 80, 90, 95]:
        eligible = concentration.loc[concentration["CumulativeDemandSharePercentage"] >= threshold]
        if eligible.empty:
            n_products = len(concentration)
        else:
            n_products = int(eligible.iloc[0]["DemandRank"])
        covered = concentration.head(n_products)
        coverage_rows.append({
            "CoverageThresholdPercentage": threshold,
            "ProductsRequired": n_products,
            "ProductsSharePercentage": 100.0 * n_products / max(len(concentration), 1),
            "ActualDemandCoveredPercentage": float(covered["DemandSharePercentage"].sum()),
            "CoveredProductsWAPEPercentage": wape(
                routed.loc[routed[PID].isin(covered[PID]), ACTUAL],
                routed.loc[routed[PID].isin(covered[PID]), PRED],
            ),
        })
    concentration_summary = pd.DataFrame(coverage_rows)

    # Selected-system route metrics directly recomputed to keep definitions uniform.
    route_rows = []
    for route, frame in routed.groupby(ROUTE, sort=True):
        rec = metric_record(frame[ACTUAL], frame[PRED], "DAILY_PRODUCT_BY_ROUTE")
        rec.update({
            "ForecastRoute": str(route),
            "ProductRows": int(len(frame)),
            "Products": int(frame[PID].nunique()),
            "Dates": int(frame[DATE].nunique()),
        })
        route_rows.append(rec)
    selected_route_metrics = pd.DataFrame(route_rows)

    # Recursive horizon diagnostics for each method.
    horizon_rows = []
    for (method, horizon), frame in recursive_daily.groupby(["CandidateMethod", "RecursiveDayNumber"], sort=True):
        rec = metric_record(frame["ActualNormalDemand"], frame["PredictedNormalDemand"], "DAILY_PRODUCT_BY_RECURSIVE_HORIZON")
        rec.update({"CandidateMethod": method, "RecursiveDayNumber": int(horizon)})
        horizon_rows.append(rec)
    horizon_metrics = pd.DataFrame(horizon_rows)

    route_change_summary = (
        route_change.groupby(["CandidateMethod", "RecursiveDayNumber"], as_index=False)
        .agg(
            Rows=("Rows", "sum"),
            RouteChangesVersusActualHistory=("RouteChangesVersusActualHistory", "sum"),
        )
    )
    route_change_summary["RouteChangePercentage"] = np.where(
        route_change_summary["Rows"] > 0,
        100.0 * route_change_summary["RouteChangesVersusActualHistory"] / route_change_summary["Rows"],
        np.nan,
    )

    # Weekly mode comparison table for the three methods.
    weekly_comparison_rows = []
    for method in ["ROLLING_MEAN_5", "NESTED_WEIGHTED_ENSEMBLE_4_MODELS", "NESTED_MEDIAN_ENSEMBLE"]:
        r_prod = pick_metric_row(recursive_metrics, method, "PRODUCT_WEEK")
        r_rest = pick_metric_row(recursive_metrics, method, "RESTAURANT_WEEK")
        u_prod = pick_metric_row(updated_metrics, method, "PRODUCT_WEEK")
        u_rest = pick_metric_row(updated_metrics, method, "RESTAURANT_WEEK")
        weekly_comparison_rows.extend([
            {
                "CandidateMethod": method,
                "MethodLabel": short_method(method),
                "ForecastMode": "MONDAY_ORIGIN_RECURSIVE",
                "EvaluationLevel": "PRODUCT_WEEK",
                "WAPEPercentage": float(r_prod["WAPEPercentage"]),
                "MAE": float(r_prod["MAE"]),
                "RMSE": float(r_prod["RMSE"]),
                "AbsoluteBiasPercentage": float(r_prod["AbsoluteBiasPercentage"]),
            },
            {
                "CandidateMethod": method,
                "MethodLabel": short_method(method),
                "ForecastMode": "MONDAY_ORIGIN_RECURSIVE",
                "EvaluationLevel": "RESTAURANT_WEEK",
                "WAPEPercentage": float(r_rest["WAPEPercentage"]),
                "MAE": float(r_rest["MAE"]),
                "RMSE": float(r_rest["RMSE"]),
                "AbsoluteBiasPercentage": float(r_rest["AbsoluteBiasPercentage"]),
            },
            {
                "CandidateMethod": method,
                "MethodLabel": short_method(method),
                "ForecastMode": "DAILY_UPDATED_ACTUAL_HISTORY",
                "EvaluationLevel": "PRODUCT_WEEK",
                "WAPEPercentage": float(u_prod["WAPEPercentage"]),
                "MAE": float(u_prod["MAE"]),
                "RMSE": float(u_prod["RMSE"]),
                "AbsoluteBiasPercentage": float(u_prod["AbsoluteBiasPercentage"]),
            },
            {
                "CandidateMethod": method,
                "MethodLabel": short_method(method),
                "ForecastMode": "DAILY_UPDATED_ACTUAL_HISTORY",
                "EvaluationLevel": "RESTAURANT_WEEK",
                "WAPEPercentage": float(u_rest["WAPEPercentage"]),
                "MAE": float(u_rest["MAE"]),
                "RMSE": float(u_rest["RMSE"]),
                "AbsoluteBiasPercentage": float(u_rest["AbsoluteBiasPercentage"]),
            },
        ])
    weekly_comparison = pd.DataFrame(weekly_comparison_rows)

    # Demonstration summary.
    demo_date = pd.to_datetime(demo_daily[DATE], errors="raise").dt.normalize().iloc[0] if DATE in demo_daily.columns else pd.Timestamp("2026-03-02")
    demo_week_start = pd.to_datetime(demo_week_daily["WeekStart"], errors="raise").dt.normalize().iloc[0] if "WeekStart" in demo_week_daily.columns else pd.Timestamp("2026-03-02")
    demo_week_end = pd.to_datetime(demo_week_daily["WeekEnd"], errors="raise").dt.normalize().iloc[0] if "WeekEnd" in demo_week_daily.columns else demo_week_start + pd.Timedelta(days=4)

    demo_summary = pd.DataFrame([
        {
            "Output": "Daily restaurant plan",
            "DateOrPeriod": str(demo_date.date()),
            "NormalDemand": float(demo_daily["PredictedNormalDemand"].sum()),
            "ConfirmedBulkDemand": float(demo_daily["ConfirmedBulkDemand"].sum()),
            "PlannedQuantity": float(demo_daily["PlannedQuantity"].sum()),
            "Method": selected_daily_method,
        },
        {
            "Output": "Monday-origin restaurant week plan",
            "DateOrPeriod": f"{demo_week_start.date()} to {demo_week_end.date()}",
            "NormalDemand": float(demo_week_products["PredictedWeekNormalDemand"].sum()),
            "ConfirmedBulkDemand": float(demo_week_products["ConfirmedWeekBulkDemand"].sum()),
            "PlannedQuantity": float(demo_week_products["PlannedWeekQuantity"].sum()),
            "Method": selected_week_method,
        },
    ])

    # =========================================================================
    # STAGING DIRECTORIES
    # =========================================================================

    staged_table = STAGING_ROOT / TABLE_DIR.relative_to(ND10R_ROOT)
    staged_diag = STAGING_ROOT / DIAGNOSTIC_DIR.relative_to(ND10R_ROOT)
    staged_fig = STAGING_ROOT / FIGURE_DIR.relative_to(ND10R_ROOT)
    staged_report = STAGING_ROOT / REPORT_DIR.relative_to(ND10R_ROOT)
    staged_demo = STAGING_ROOT / DEMO_DIR.relative_to(ND10R_ROOT)
    staged_control = STAGING_ROOT / CONTROL_DIR.relative_to(ND10R_ROOT)
    for directory in [staged_table, staged_diag, staged_fig, staged_report, staged_demo, staged_control]:
        directory.mkdir(parents=True, exist_ok=True)

    # Tables.
    write_csv(staged_table / "ND10R_report_metric_table.csv", report_metrics)
    write_csv(staged_table / "ND10R_model_selection_journey.csv", model_journey)
    write_csv(staged_table / "ND10R_weekly_mode_comparison.csv", weekly_comparison)
    write_csv(staged_table / "ND10R_operational_method_selection.csv", operational_selection)
    write_csv(staged_table / "ND10R_demonstration_output_summary.csv", demo_summary)

    write_csv(staged_diag / "ND10R_product_diagnostic_summary.csv", product_diag)
    write_csv(staged_diag / "ND10R_worst_products_by_total_absolute_error.csv", worst_products)
    write_csv(staged_diag / "ND10R_daily_restaurant_diagnostics.csv", date_diag)
    write_csv(staged_diag / "ND10R_worst_restaurant_dates.csv", worst_dates)
    write_csv(staged_diag / "ND10R_best_restaurant_dates.csv", best_dates)
    write_csv(staged_diag / "ND10R_forecast_tolerance_diagnostics.csv", tolerance_diag)
    write_csv(staged_diag / "ND10R_demand_concentration_by_product.csv", concentration)
    write_csv(staged_diag / "ND10R_demand_concentration_summary.csv", concentration_summary)
    write_csv(staged_diag / "ND10R_selected_route_metrics.csv", selected_route_metrics)
    write_csv(staged_diag / "ND10R_recursive_horizon_metrics.csv", horizon_metrics)
    write_csv(staged_diag / "ND10R_recursive_route_change_summary.csv", route_change_summary)

    # Copy key demonstration tables into a dedicated folder for presentation preparation.
    write_csv(staged_demo / "ND10R_demo_daily_product_plan.csv", demo_daily)
    write_csv(staged_demo / "ND10R_demo_daily_restaurant_plan.csv", demo_daily_restaurant)
    write_csv(staged_demo / "ND10R_demo_week_daily_product_plan.csv", demo_week_daily)
    write_csv(staged_demo / "ND10R_demo_week_product_totals.csv", demo_week_products)
    write_csv(staged_demo / "ND10R_demo_week_restaurant_daily.csv", demo_week_restaurant_daily)
    write_csv(staged_demo / "ND10R_demo_week_restaurant_total.csv", demo_week_restaurant_total)

    # =========================================================================
    # FIGURES
    # =========================================================================

    figure_records = []

    def register_figure(filename: str, title: str, report_use: str, caption: str) -> Path:
        figure_records.append({
            "FigureFile": filename,
            "FigureTitle": title,
            "RecommendedReportUse": report_use,
            "SuggestedCaption": caption,
        })
        return staged_fig / filename

    # 1. Model selection journey.
    plot = model_journey.copy()
    labels = [f"{row.Stage}\n{wrap_label(row.MethodLabel, 16)}" for row in plot.itertuples(index=False)]
    plt.figure(figsize=(14, 7))
    plt.bar(np.arange(len(plot)), plot["WAPEPercentage"])
    plt.xticks(np.arange(len(plot)), labels, rotation=35, ha="right")
    plt.ylabel("WAPE (%)")
    plt.title("Development model-selection journey: daily main-route WAPE")
    plt.grid(axis="y", alpha=0.25)
    save_figure(register_figure(
        "ND10R_figure_01_model_selection_journey_wape.png",
        "Development model-selection journey",
        "Main report — modelling results",
        "Development-validation WAPE across the benchmark, advanced-model screening, nested tuning and focused ensemble stages. The figure illustrates why the final daily method was selected only after nested ensemble evaluation.",
    ))

    # 2. ND07E ensemble comparison.
    ens = nd07e_metrics.loc[nd07e_metrics["CandidateMethod"].astype(str).isin([
        "ROLLING_MEAN_5_BENCHMARK",
        "NESTED_MEDIAN_ENSEMBLE",
        "NESTED_WEIGHTED_ENSEMBLE_4_MODELS",
        "NESTED_WEIGHTED_ENSEMBLE_3_MODELS",
        "NESTED_WEIGHTED_ENSEMBLE_2_MODELS",
    ])].copy().sort_values("WAPEPercentage")
    plt.figure(figsize=(11, 6))
    plt.bar([short_method(x) for x in ens["CandidateMethod"]], ens["WAPEPercentage"])
    plt.ylabel("WAPE (%)")
    plt.title("Focused ensemble challenge: daily main-route WAPE")
    plt.xticks(rotation=25, ha="right")
    plt.grid(axis="y", alpha=0.25)
    save_figure(register_figure(
        "ND10R_figure_02_focused_ensemble_wape.png",
        "Focused ensemble challenge",
        "Main report — selected daily method",
        "The median ensemble achieved the lowest nested outer-validation WAPE, while all four ensemble variants improved on the rolling-mean benchmark under the strict development qualification rules.",
    ))

    # 3. Fold-wise selected median vs benchmark.
    fold_compare = nd07e_fold.loc[nd07e_fold["CandidateMethod"].astype(str).isin(["ROLLING_MEAN_5_BENCHMARK", "NESTED_MEDIAN_ENSEMBLE"])].copy()
    fold_col = "OuterFold" if "OuterFold" in fold_compare.columns else "Fold"
    require_columns(fold_compare, [fold_col], "ND07E fold comparison")
    pivot = fold_compare.pivot(index=fold_col, columns="CandidateMethod", values="WAPEPercentage").sort_index()
    plt.figure(figsize=(10, 6))
    x = np.arange(len(pivot.index))
    width = 0.36
    plt.bar(x - width/2, pivot["ROLLING_MEAN_5_BENCHMARK"], width, label="Rolling mean 5")
    plt.bar(x + width/2, pivot["NESTED_MEDIAN_ENSEMBLE"], width, label="Median ensemble")
    plt.xticks(x, [f"Fold {int(v)}" for v in pivot.index])
    plt.ylabel("WAPE (%)")
    plt.title("Daily main-route WAPE by chronological outer fold")
    plt.legend()
    plt.grid(axis="y", alpha=0.25)
    save_figure(register_figure(
        "ND10R_figure_03_daily_fold_wape_selected_vs_benchmark.png",
        "Fold-wise daily comparison",
        "Main report or appendix — robustness",
        "Chronological outer-fold WAPE for the selected median ensemble versus the rolling-mean benchmark. The ensemble won four of five folds in ND07E.",
    ))

    # 4. Restaurant daily actual vs forecast.
    dplot = daily_restaurant.sort_values(DATE)
    plt.figure(figsize=(13, 6))
    plt.plot(dplot[DATE], dplot[ACTUAL], label="Actual")
    plt.plot(dplot[DATE], dplot[PRED], label="Forecast")
    plt.xlabel("Operating date")
    plt.ylabel("Normal-demand units")
    plt.title("Selected routed system: restaurant daily actual versus forecast")
    plt.xticks(rotation=40, ha="right")
    plt.legend()
    plt.grid(alpha=0.25)
    save_figure(register_figure(
        "ND10R_figure_04_daily_restaurant_actual_vs_forecast.png",
        "Restaurant daily actual versus forecast",
        "Main report — daily forecasting behaviour",
        "Aggregated restaurant-level actual and predicted normal demand across the 100 chronological development-validation operating dates using the selected routed daily system.",
    ))

    # 5. Daily restaurant signed error.
    plt.figure(figsize=(13, 5.5))
    plt.bar(date_diag[DATE], date_diag["ForecastError"])
    plt.axhline(0, linewidth=1)
    plt.xlabel("Operating date")
    plt.ylabel("Forecast error (prediction − actual)")
    plt.title("Selected routed system: restaurant daily forecast error")
    plt.xticks(rotation=40, ha="right")
    plt.grid(axis="y", alpha=0.25)
    save_figure(register_figure(
        "ND10R_figure_05_daily_restaurant_error_by_date.png",
        "Restaurant daily error by date",
        "Appendix or diagnostic subsection",
        "Signed restaurant-level error by operating date. Positive values indicate overprediction and negative values indicate underprediction.",
    ))

    # 6. Product actual vs predicted scatter.
    scatter = routed[[ACTUAL, PRED]].copy()
    if len(scatter) > SCATTER_MAX_POINTS:
        scatter = scatter.sample(SCATTER_MAX_POINTS, random_state=42)
    max_axis = float(max(scatter[ACTUAL].max(), scatter[PRED].max(), 1.0))
    plt.figure(figsize=(7.5, 7.5))
    plt.scatter(scatter[ACTUAL], scatter[PRED], alpha=0.28, s=13)
    plt.plot([0, max_axis], [0, max_axis], linestyle="--")
    plt.xlim(left=0)
    plt.ylim(bottom=0)
    plt.xlabel("Actual normal demand")
    plt.ylabel("Predicted normal demand")
    plt.title("Product-day actual versus predicted demand")
    plt.grid(alpha=0.2)
    save_figure(register_figure(
        "ND10R_figure_06_product_actual_vs_predicted.png",
        "Product-day actual versus predicted",
        "Main report or appendix — product-level error",
        "Product-day actual and predicted normal demand for the selected routed system. The diagonal represents perfect prediction and illustrates the greater dispersion at product level than after restaurant aggregation.",
    ))

    # 7. Absolute error distribution.
    plt.figure(figsize=(9, 5.5))
    cap = float(routed["AbsoluteError"].quantile(0.99))
    hist_values = routed.loc[routed["AbsoluteError"] <= cap, "AbsoluteError"]
    plt.hist(hist_values, bins=40)
    plt.xlabel("Absolute product-day error (units)")
    plt.ylabel("Product-day observations")
    plt.title("Distribution of product-day absolute error (up to 99th percentile)")
    plt.grid(axis="y", alpha=0.2)
    save_figure(register_figure(
        "ND10R_figure_07_product_absolute_error_distribution.png",
        "Product-day absolute error distribution",
        "Appendix — diagnostic distribution",
        "Distribution of absolute product-day forecast error. The display is capped at the 99th percentile so the central error distribution remains visible while extreme cases are analysed separately.",
    ))

    # 8. Worst products.
    wplot = worst_products.sort_values("TotalAbsoluteError")
    plt.figure(figsize=(11, 8))
    plt.barh([wrap_label(v, 28) for v in wplot[PNAME]], wplot["TotalAbsoluteError"])
    plt.xlabel("Total absolute error across validation dates")
    plt.ylabel("Product")
    plt.title(f"Top {len(wplot)} products by total absolute forecast error")
    plt.grid(axis="x", alpha=0.2)
    save_figure(register_figure(
        "ND10R_figure_08_worst_products_by_total_absolute_error.png",
        "Products contributing most absolute error",
        "Main report or appendix — limitations",
        "Products ranked by total absolute error across the routed development-validation period. This identifies where product-level forecast uncertainty is concentrated.",
    ))

    # 9. Route-level WAPE.
    rplot = selected_route_metrics.sort_values("WAPEPercentage")
    plt.figure(figsize=(10, 6))
    plt.bar([wrap_label(x, 20) for x in rplot["ForecastRoute"]], rplot["WAPEPercentage"])
    plt.ylabel("WAPE (%)")
    plt.title("Selected routed system: WAPE by forecast route")
    plt.xticks(rotation=20, ha="right")
    plt.grid(axis="y", alpha=0.25)
    save_figure(register_figure(
        "ND10R_figure_09_route_level_wape.png",
        "Performance by forecast route",
        "Main report — routed architecture",
        "Product-day WAPE split by dynamic forecast route, showing how the system handles established, low-demand, cold-start and zero-history products separately.",
    ))

    # 10. Cumulative demand concentration.
    plt.figure(figsize=(9, 6))
    plt.plot(concentration["DemandRank"], concentration["CumulativeDemandSharePercentage"])
    for threshold in [80, 95]:
        plt.axhline(threshold, linestyle="--", linewidth=1)
    plt.xlabel("Products ranked by actual validation demand")
    plt.ylabel("Cumulative share of actual normal demand (%)")
    plt.title("Demand concentration across products")
    plt.ylim(0, 101)
    plt.grid(alpha=0.25)
    save_figure(register_figure(
        "ND10R_figure_10_cumulative_demand_coverage.png",
        "Cumulative demand concentration",
        "Main report — product demand structure / ingredient mapping motivation",
        "Cumulative share of validation-period normal demand as products are ranked from highest to lowest demand. The curve shows how a smaller set of products accounts for most demand and supports prioritised ingredient mapping.",
    ))

    # 11. Monday-origin product-week WAPE.
    rec_prod = weekly_comparison.loc[(weekly_comparison["ForecastMode"] == "MONDAY_ORIGIN_RECURSIVE") & (weekly_comparison["EvaluationLevel"] == "PRODUCT_WEEK")].sort_values("WAPEPercentage")
    plt.figure(figsize=(10, 6))
    plt.bar(rec_prod["MethodLabel"], rec_prod["WAPEPercentage"])
    plt.ylabel("Product-week WAPE (%)")
    plt.title("Genuine Monday-origin recursive product-week WAPE")
    plt.xticks(rotation=20, ha="right")
    plt.grid(axis="y", alpha=0.25)
    save_figure(register_figure(
        "ND10R_figure_11_monday_origin_product_week_wape.png",
        "Monday-origin weekly model comparison",
        "Main report — weekly forecasting results",
        "Product-week WAPE when the full Monday-to-Friday week is generated before Monday and predictions, rather than within-week actual demand, are recursively inserted into later-day history.",
    ))

    # 12. Daily-updated product-week WAPE.
    upd_prod = weekly_comparison.loc[(weekly_comparison["ForecastMode"] == "DAILY_UPDATED_ACTUAL_HISTORY") & (weekly_comparison["EvaluationLevel"] == "PRODUCT_WEEK")].sort_values("WAPEPercentage")
    plt.figure(figsize=(10, 6))
    plt.bar(upd_prod["MethodLabel"], upd_prod["WAPEPercentage"])
    plt.ylabel("Product-week WAPE (%)")
    plt.title("Daily-updated product-week WAPE on the same 14 complete weeks")
    plt.xticks(rotation=20, ha="right")
    plt.grid(axis="y", alpha=0.25)
    save_figure(register_figure(
        "ND10R_figure_12_daily_updated_product_week_wape.png",
        "Daily-updated weekly model comparison",
        "Main report — operational updating",
        "Product-week WAPE when actual demand becomes available after each operating day. All methods use exactly the same 14 complete weeks as the Monday-origin evaluation.",
    ))

    # 13. Origin versus updated for selected rolling mean.
    origin_val = float(recursive_product_selected["WAPEPercentage"])
    updated_val = float(updated_product_selected["WAPEPercentage"])
    plt.figure(figsize=(7.5, 5.5))
    plt.bar(["Before Monday\n(recursive)", "Daily updated\n(actual history)"], [origin_val, updated_val])
    plt.ylabel("Product-week WAPE (%)")
    plt.title("Operational effect of updating the week as actual demand arrives")
    plt.grid(axis="y", alpha=0.25)
    save_figure(register_figure(
        "ND10R_figure_13_monday_origin_vs_daily_updated.png",
        "Monday-origin versus daily-updated weekly error",
        "Main report — operational interpretation",
        "The selected rolling-mean weekly method is substantially more accurate when the remaining week is updated using observed daily demand than when the entire week must be forecast before Monday.",
    ))

    # 14. Week-by-week recursive WAPE.
    plt.figure(figsize=(13, 6))
    for method, frame in recursive_week_metrics.groupby("CandidateMethod", sort=True):
        frame = frame.sort_values("WeekStart")
        plt.plot(frame["WeekStart"], frame["WAPEPercentage"], marker="o", label=short_method(method))
    plt.xlabel("Week starting")
    plt.ylabel("Product-week WAPE (%)")
    plt.title("Monday-origin product-week WAPE across the 14 complete evaluation weeks")
    plt.xticks(rotation=40, ha="right")
    plt.legend(fontsize=8)
    plt.grid(alpha=0.25)
    save_figure(register_figure(
        "ND10R_figure_14_recursive_week_wape_series.png",
        "Week-by-week recursive forecasting performance",
        "Appendix or main report — temporal robustness",
        "Monday-origin product-week WAPE for each evaluated complete week. The chronological variation shows that forecast difficulty changes materially from week to week.",
    ))

    # 15. Restaurant week actual vs forecast for selected method.
    selected_rest_week = recursive_restaurant_week.loc[recursive_restaurant_week["CandidateMethod"].astype(str) == selected_week_method].sort_values("WeekStart")
    plt.figure(figsize=(13, 6))
    plt.plot(selected_rest_week["WeekStart"], selected_rest_week["ActualNormalDemand"], marker="o", label="Actual")
    plt.plot(selected_rest_week["WeekStart"], selected_rest_week["PredictedNormalDemand"], marker="o", label="Forecast")
    plt.xlabel("Week starting")
    plt.ylabel("Restaurant-week normal-demand units")
    plt.title("Selected Monday-origin method: restaurant-week actual versus forecast")
    plt.xticks(rotation=40, ha="right")
    plt.legend()
    plt.grid(alpha=0.25)
    save_figure(register_figure(
        "ND10R_figure_15_restaurant_week_actual_vs_forecast.png",
        "Restaurant-week actual versus forecast",
        "Main report — weekly forecasting behaviour",
        "Restaurant-level weekly actual and predicted normal demand for the selected rolling-mean Monday-origin method across 14 complete weeks.",
    ))

    # 16. Recursive horizon WAPE.
    hplot = horizon_metrics.copy()
    plt.figure(figsize=(10, 6))
    for method, frame in hplot.groupby("CandidateMethod", sort=True):
        frame = frame.sort_values("RecursiveDayNumber")
        plt.plot(frame["RecursiveDayNumber"], frame["WAPEPercentage"], marker="o", label=short_method(method))
    plt.xticks([1, 2, 3, 4, 5], ["Mon", "Tue", "Wed", "Thu", "Fri"])
    plt.xlabel("Recursive day within forecast week")
    plt.ylabel("Product-day WAPE (%)")
    plt.title("Error propagation across the Monday-origin recursive horizon")
    plt.legend(fontsize=8)
    plt.grid(alpha=0.25)
    save_figure(register_figure(
        "ND10R_figure_16_recursive_error_by_horizon.png",
        "Recursive error by forecast horizon",
        "Main report — explanation of weekly method choice",
        "Product-day WAPE by recursive day within the Monday-origin forecast. This diagnostic shows how prediction-driven feature updates influence error as the week progresses.",
    ))

    # 17. Route changes by recursive day.
    plt.figure(figsize=(10, 6))
    for method, frame in route_change_summary.groupby("CandidateMethod", sort=True):
        frame = frame.sort_values("RecursiveDayNumber")
        plt.plot(frame["RecursiveDayNumber"], frame["RouteChangePercentage"], marker="o", label=short_method(method))
    plt.xticks([1, 2, 3, 4, 5], ["Mon", "Tue", "Wed", "Thu", "Fri"])
    plt.xlabel("Recursive day within forecast week")
    plt.ylabel("Rows whose recursive route differs from actual-history route (%)")
    plt.title("Dynamic route changes during recursive weekly forecasting")
    plt.legend(fontsize=8)
    plt.grid(alpha=0.25)
    save_figure(register_figure(
        "ND10R_figure_17_recursive_route_changes.png",
        "Recursive route changes",
        "Appendix — recursive system diagnostics",
        "Percentage of product rows whose dynamically reconstructed recursive route differs from the route that would have been observed with actual history, illustrating another source of multi-day forecast divergence.",
    ))

    # 18. Demo daily top products.
    top_daily = demo_daily.nlargest(TOP_PRODUCTS_IN_DEMO_FIGURE, "PlannedQuantity").sort_values("PlannedQuantity")
    plt.figure(figsize=(11, 8))
    plt.barh([wrap_label(v, 28) for v in top_daily[PNAME]], top_daily["PlannedQuantity"])
    plt.xlabel("Planned product quantity")
    plt.ylabel("Product")
    plt.title(f"Demonstration daily plan: highest planned quantities — {demo_date.date()}")
    plt.grid(axis="x", alpha=0.2)
    save_figure(register_figure(
        "ND10R_figure_18_demo_daily_top_products.png",
        "Demonstration daily product plan",
        "Presentation / demonstration slide",
        "Highest planned product quantities for the default arbitrary-date demonstration forecast. Planned quantity is normal-demand forecast plus confirmed bulk demand.",
    ))

    # 19. Demo week restaurant daily plan. Resolve planned total column robustly.
    restaurant_daily_value_col = None
    for candidate in ["PlannedRestaurantQuantity", "PlannedQuantity", "PredictedRestaurantNormalDemand"]:
        if candidate in demo_week_restaurant_daily.columns:
            restaurant_daily_value_col = candidate
            break
    if restaurant_daily_value_col is None:
        numeric_candidates = [c for c in demo_week_restaurant_daily.columns if c not in {DATE, "WeekStart", "WeekEnd"} and pd.api.types.is_numeric_dtype(demo_week_restaurant_daily[c])]
        if not numeric_candidates:
            raise AssertionError("Unable to identify ND10 week restaurant daily planned-quantity column.")
        restaurant_daily_value_col = numeric_candidates[-1]
    plt.figure(figsize=(9, 5.5))
    plt.plot(demo_week_restaurant_daily[DATE], demo_week_restaurant_daily[restaurant_daily_value_col], marker="o")
    plt.xlabel("Date")
    plt.ylabel("Planned restaurant quantity")
    plt.title(f"Demonstration Monday-origin restaurant plan — week of {demo_week_start.date()}")
    plt.xticks(rotation=30, ha="right")
    plt.grid(alpha=0.25)
    save_figure(register_figure(
        "ND10R_figure_19_demo_week_restaurant_daily_plan.png",
        "Demonstration restaurant week plan",
        "Presentation / demonstration slide",
        "Daily restaurant planned quantities for the default Monday-origin demonstration week. The complete week is generated before Monday using the recursive rolling-mean method.",
    ))

    # 20. Demo week top products.
    top_week = demo_week_products.nlargest(TOP_PRODUCTS_IN_DEMO_FIGURE, "PlannedWeekQuantity").sort_values("PlannedWeekQuantity")
    plt.figure(figsize=(11, 8))
    plt.barh([wrap_label(v, 28) for v in top_week[PNAME]], top_week["PlannedWeekQuantity"])
    plt.xlabel("Planned product quantity for week")
    plt.ylabel("Product")
    plt.title(f"Demonstration weekly plan: highest product totals — {demo_week_start.date()}")
    plt.grid(axis="x", alpha=0.2)
    save_figure(register_figure(
        "ND10R_figure_20_demo_week_top_products.png",
        "Demonstration weekly product totals",
        "Presentation / demonstration slide",
        "Highest planned product totals for the default Monday-to-Friday demonstration week, showing the output that can be used for procurement and ingredient planning.",
    ))

    # 21. System architecture diagram.
    plt.figure(figsize=(15, 5.8))
    ax = plt.gca()
    ax.axis("off")
    boxes = [
        (0.02, 0.35, 0.13, 0.30, "Eden POS\nhistory"),
        (0.19, 0.35, 0.15, 0.30, "Normal-demand\nfeature engine\n53 predictors"),
        (0.38, 0.58, 0.16, 0.25, "Next day\nMedian ensemble"),
        (0.38, 0.17, 0.16, 0.25, "Multi-day / week\nRecursive rolling\nmean 5"),
        (0.59, 0.35, 0.14, 0.30, "Predicted\nnormal demand"),
        (0.77, 0.35, 0.09, 0.30, "+ confirmed\nbulk orders"),
        (0.90, 0.35, 0.09, 0.30, "Planned\nquantity"),
    ]
    for x, y, w, h, label in boxes:
        rect = plt.Rectangle((x, y), w, h, fill=False, linewidth=1.5)
        ax.add_patch(rect)
        ax.text(x + w/2, y + h/2, label, ha="center", va="center", fontsize=10)
    arrows = [
        ((0.15, 0.50), (0.19, 0.50)),
        ((0.34, 0.50), (0.38, 0.70)),
        ((0.34, 0.50), (0.38, 0.29)),
        ((0.54, 0.70), (0.59, 0.50)),
        ((0.54, 0.29), (0.59, 0.50)),
        ((0.73, 0.50), (0.77, 0.50)),
        ((0.86, 0.50), (0.90, 0.50)),
    ]
    for start, end in arrows:
        ax.annotate("", xy=end, xytext=start, arrowprops=dict(arrowstyle="->", linewidth=1.4))
    ax.text(0.5, 0.95, "Revised Eden normal-demand forecasting and planning architecture", ha="center", va="center", fontsize=14)
    save_figure(register_figure(
        "ND10R_figure_21_system_architecture.png",
        "Final forecasting-system architecture",
        "Main report — methodology overview / presentation",
        "Operational architecture of the revised system. The median ensemble is used for next-day product forecasting, while recursive rolling mean 5 is used for multi-day and week-start planning; confirmed bulk demand is added only after normal demand is forecast.",
    ))

    # 22. Operational method decision diagram.
    plt.figure(figsize=(12, 5.5))
    ax = plt.gca()
    ax.axis("off")
    ax.text(0.5, 0.90, "Forecast request", ha="center", va="center", fontsize=14, bbox=dict(boxstyle="round,pad=0.5", fill=False))
    ax.annotate("", xy=(0.27, 0.63), xytext=(0.47, 0.82), arrowprops=dict(arrowstyle="->"))
    ax.annotate("", xy=(0.73, 0.63), xytext=(0.53, 0.82), arrowprops=dict(arrowstyle="->"))
    ax.text(0.25, 0.55, "Next operating day", ha="center", va="center", fontsize=12, bbox=dict(boxstyle="round,pad=0.5", fill=False))
    ax.text(0.75, 0.55, "Two or more unobserved\noperating days / full week", ha="center", va="center", fontsize=12, bbox=dict(boxstyle="round,pad=0.5", fill=False))
    ax.annotate("", xy=(0.25, 0.28), xytext=(0.25, 0.47), arrowprops=dict(arrowstyle="->"))
    ax.annotate("", xy=(0.75, 0.28), xytext=(0.75, 0.47), arrowprops=dict(arrowstyle="->"))
    ax.text(0.25, 0.18, "Median ensemble\nRolling mean 5 +\nproduct-aware XGBoost", ha="center", va="center", fontsize=11, bbox=dict(boxstyle="round,pad=0.5", fill=False))
    ax.text(0.75, 0.18, "Recursive\nrolling mean 5", ha="center", va="center", fontsize=11, bbox=dict(boxstyle="round,pad=0.5", fill=False))
    save_figure(register_figure(
        "ND10R_figure_22_operational_method_decision.png",
        "Operational method-selection rule",
        "Main report — deployment logic / presentation",
        "Final inference rule selected from ND08: one-step forecasts use the median ensemble, while requests requiring unobserved intermediate days use recursive rolling mean 5.",
    ))

    figure_captions = pd.DataFrame(figure_records)
    write_csv(staged_report / "ND10R_figure_captions_and_report_use.csv", figure_captions)

    # =========================================================================
    # REPORT WORDING
    # =========================================================================

    benchmark_wape = float(benchmark_row["WAPEPercentage"])
    median_row = pick_metric_row(nd07e_metrics, "NESTED_MEDIAN_ENSEMBLE")
    median_wape = float(median_row["WAPEPercentage"])
    weighted4_row = pick_metric_row(nd07e_metrics, "NESTED_WEIGHTED_ENSEMBLE_4_MODELS")
    weighted4_wape = float(weighted4_row["WAPEPercentage"])
    daily_improvement_pp = benchmark_wape - median_wape
    daily_relative_reduction = 100.0 * daily_improvement_pp / benchmark_wape
    weekly_update_improvement_pp = origin_val - updated_val
    weekly_update_relative_reduction = 100.0 * weekly_update_improvement_pp / origin_val

    coverage80 = concentration_summary.loc[concentration_summary["CoverageThresholdPercentage"] == 80].iloc[0]
    largest_error_product = worst_products.iloc[0]
    largest_error_date = worst_dates.iloc[0]

    result_wording = f"""# ND10R — Report-Ready Results Wording

## How these results must be described

All accuracy values in this package are **development-evaluation results**, not a new final holdout result. They were produced by the chronological pre-March evaluations completed before the demonstration engine was frozen. March 2026 target values remain closed. The ND09 and ND10 March forecasts shown in the demonstration figures are therefore forecasts only and are not scored here.

Do not call `100 - WAPE` forecasting accuracy. The official measures remain WAPE, MAE, RMSE and bias.

## Daily model selection

A five-operating-day rolling mean established the main-route benchmark with WAPE of **{benchmark_wape:.2f}%**. The initial advanced-model screen identified CatBoost as a promising challenger, but the stricter nested tuning stage showed that individually tuned CatBoost and XGBoost variants did not robustly outperform the rolling benchmark. A focused nested ensemble experiment was therefore carried out using rolling mean 5, CatBoost, core XGBoost and product-aware XGBoost as components.

The **nested median ensemble** achieved the lowest outer-validation daily main-route WAPE of **{median_wape:.2f}%**, an improvement of **{daily_improvement_pp:.2f} percentage points** over the rolling benchmark, corresponding to a **{daily_relative_reduction:.2f}% relative reduction in WAPE**. The four-model weighted ensemble produced a nearly identical WAPE of **{weighted4_wape:.2f}%** and lower RMSE/bias, but the median ensemble retained the lowest WAPE and was selected for next-day product forecasting.

## Routed daily system

When the selected daily main-route method was combined with the dynamically selected low-demand, cold-start and zero-history fallback rules, the full routed catalogue achieved a product-day WAPE of **{selected_routed_metric['WAPEPercentage']:.2f}%** across **{selected_routed_metric['Observations']:,}** routed validation rows. Aggregating the same forecasts to restaurant-day level reduced WAPE to **{selected_restaurant_metric['WAPEPercentage']:.2f}%**, demonstrating that restaurant-total demand is easier to estimate than the exact distribution of demand across individual products.

## Weekly forecasting

ND08 evaluated a more demanding operational scenario in which a complete Monday-to-Friday week was forecast before Monday. No actual demand from inside the forecast week was used; each predicted day was inserted into temporary history before the following day was generated. Under this genuine recursive protocol, **rolling mean 5** produced the lowest product-week WAPE of **{origin_val:.2f}%** and was selected for Monday-origin weekly planning. The corresponding restaurant-week WAPE was **{float(recursive_restaurant_selected['WAPEPercentage']):.2f}%**.

When actual demand was allowed to update the historical state after each operating day, the rolling-mean product-week WAPE improved to **{updated_val:.2f}%**. This is a reduction of **{weekly_update_improvement_pp:.2f} percentage points** or approximately **{weekly_update_relative_reduction:.2f}% relative** compared with forecasting the whole week before Monday. This result supports an operational design in which the initial week forecast is retained for planning but the remaining-week forecast is refreshed as actual sales become available.

## Error concentration and product structure

Forecast error is not distributed evenly across the catalogue. The product with the largest cumulative absolute error in the routed development evaluation was **{largest_error_product[PNAME]}**, with total absolute error of **{float(largest_error_product['TotalAbsoluteError']):.2f} units** across its validation observations. The largest restaurant-day absolute error occurred on **{pd.Timestamp(largest_error_date[DATE]).date()}**, with an absolute error of **{float(largest_error_date['AbsoluteError']):.2f} units**.

Demand is also concentrated across a smaller subset of the catalogue. Approximately **{int(coverage80['ProductsRequired'])} products** were required to cover at least **80%** of observed validation normal demand. This provides a practical basis for prioritising ingredient mapping for the products that account for the majority of expected demand.

## Frozen demonstration system

The final demonstration architecture uses two operational forecasting rules rather than forcing one model to serve every horizon. The **nested median ensemble** is used for the next operating day because it achieved the strongest daily product result. **Rolling mean 5** is used whenever unobserved intermediate operating days must be crossed, including complete Monday-origin weekly planning, because it was more stable under recursive propagation. Confirmed bulk demand is kept outside the forecasting target and added afterwards:

`Planned Quantity = Predicted Normal Demand + Confirmed Bulk Demand`.

The current frozen demonstration forecast uses a data cutoff of **27 February 2026**. For **2 March 2026**, the model predicts **{float(demo_summary.iloc[0]['NormalDemand']):.2f}** units of restaurant normal demand. The genuine Monday-origin plan for **{demo_week_start.date()} to {demo_week_end.date()}** predicts **{float(demo_summary.iloc[1]['NormalDemand']):.2f}** units of restaurant normal demand. No confirmed bulk orders were present in the current ND10 input, so the demonstrated planned quantities are equal to the normal-demand forecasts.

## Limitations to state explicitly

The main accuracy results are chronological development-validation estimates rather than a final untouched future test. March target values were deliberately kept closed after the earlier modelling work had already exposed that period. A new future operating period is therefore still required for a final unbiased deployment estimate. Weekly forecasting remains materially harder than one-step daily forecasting because predictions are recursively propagated into later-day historical features and dynamic routing decisions.
"""
    write_text(staged_report / "ND10R_report_ready_results_wording.md", result_wording)

    summary_text = f"""# ND10R — Final Diagnostic Analysis and Report-Ready Results

## Status

`{STATUS}`

## Purpose

This archive is the revised-model equivalent of the first model's final diagnostic/reporting folder, expanded to reflect the new architecture. It contains report-ready figures, diagnostic tables, figure captions, ready-to-paste results wording, and demonstration forecast visuals.

## Authoritative selected methods

- Next operating day: `{selected_daily_method}`
- Representative median components: `ROLLING_MEAN_5` + `XGBOOST_PRODUCT_AWARE`
- Monday-origin multi-day/week planning: `{selected_week_method}`
- Rolling daily-updated remaining week: `{selected_updated_method}`

## Key development results

- Daily main-route benchmark WAPE: {benchmark_wape:.6f}%
- Selected median-ensemble daily main-route WAPE: {median_wape:.6f}%
- Daily WAPE improvement: {daily_improvement_pp:.6f} percentage points
- Full routed daily product WAPE: {selected_routed_metric['WAPEPercentage']:.6f}%
- Full routed daily restaurant WAPE: {selected_restaurant_metric['WAPEPercentage']:.6f}%
- Monday-origin product-week WAPE: {origin_val:.6f}%
- Monday-origin restaurant-week WAPE: {float(recursive_restaurant_selected['WAPEPercentage']):.6f}%
- Daily-updated product-week WAPE: {updated_val:.6f}%
- Daily-updated restaurant-week WAPE: {float(updated_restaurant_selected['WAPEPercentage']):.6f}%

## Demonstration outputs

- Daily demonstration date: {demo_date.date()}
- Daily restaurant normal-demand forecast: {float(demo_summary.iloc[0]['NormalDemand']):.6f}
- Daily confirmed bulk: {float(demo_summary.iloc[0]['ConfirmedBulkDemand']):.6f}
- Daily planned quantity: {float(demo_summary.iloc[0]['PlannedQuantity']):.6f}
- Demonstration week: {demo_week_start.date()} to {demo_week_end.date()}
- Week restaurant normal-demand forecast: {float(demo_summary.iloc[1]['NormalDemand']):.6f}
- Week confirmed bulk: {float(demo_summary.iloc[1]['ConfirmedBulkDemand']):.6f}
- Week planned quantity: {float(demo_summary.iloc[1]['PlannedQuantity']):.6f}

## Figures

A total of {len(figure_records)} high-resolution PNG figures were created. `ND10R_figure_captions_and_report_use.csv` identifies which figures are recommended for the main report, appendix, presentation, or demonstration.

## Safety

- Forecasting models fitted or refitted: no
- Forecasting methods reselected: no
- March target vault opened: no
- New accuracy metrics calculated on future March targets: no
- Existing modelling/planning artifacts modified: no
"""
    write_text(staged_report / "ND10R_report_ready_summary.md", summary_text)

    # Suggested report figure shortlist.
    shortlist_names = [
        "ND10R_figure_01_model_selection_journey_wape.png",
        "ND10R_figure_02_focused_ensemble_wape.png",
        "ND10R_figure_04_daily_restaurant_actual_vs_forecast.png",
        "ND10R_figure_09_route_level_wape.png",
        "ND10R_figure_10_cumulative_demand_coverage.png",
        "ND10R_figure_11_monday_origin_product_week_wape.png",
        "ND10R_figure_13_monday_origin_vs_daily_updated.png",
        "ND10R_figure_15_restaurant_week_actual_vs_forecast.png",
        "ND10R_figure_16_recursive_error_by_horizon.png",
        "ND10R_figure_21_system_architecture.png",
    ]
    shortlist = figure_captions.loc[figure_captions["FigureFile"].isin(shortlist_names)].copy()
    shortlist["SuggestedOrder"] = shortlist["FigureFile"].map({name: i + 1 for i, name in enumerate(shortlist_names)})
    shortlist = shortlist.sort_values("SuggestedOrder")
    write_csv(staged_report / "ND10R_recommended_main_report_figures.csv", shortlist)

    # Presentation shortlist.
    presentation_names = [
        "ND10R_figure_21_system_architecture.png",
        "ND10R_figure_02_focused_ensemble_wape.png",
        "ND10R_figure_11_monday_origin_product_week_wape.png",
        "ND10R_figure_13_monday_origin_vs_daily_updated.png",
        "ND10R_figure_18_demo_daily_top_products.png",
        "ND10R_figure_19_demo_week_restaurant_daily_plan.png",
        "ND10R_figure_20_demo_week_top_products.png",
    ]
    presentation = figure_captions.loc[figure_captions["FigureFile"].isin(presentation_names)].copy()
    presentation["SuggestedOrder"] = presentation["FigureFile"].map({name: i + 1 for i, name in enumerate(presentation_names)})
    presentation = presentation.sort_values("SuggestedOrder")
    write_csv(staged_demo / "ND10R_recommended_presentation_figures.csv", presentation)

    # README.
    readme = f"""# ND10R Report-Ready Diagnostics and Visuals

This folder is the comprehensive diagnostic/reporting archive for the revised Eden normal-demand forecasting system selected for the final demonstration.

## Start here

1. `04_report_wording/ND10R_report_ready_summary.md`
2. `04_report_wording/ND10R_report_ready_results_wording.md`
3. `04_report_wording/ND10R_recommended_main_report_figures.csv`
4. `03_figures/`
5. `05_demonstration_material/`

## Important interpretation

Accuracy metrics are pre-March chronological development-evaluation results. March 2026 future forecasts are demonstration outputs and are not scored. The March target vault was not opened.

Status: `{STATUS}`
"""
    write_text(STAGING_ROOT / "README.md", readme)

    # Validation summary.
    validation_rows = [
        {"Check": "ND03-ND10 checkpoints match expected SHA-256", "Passed": checkpoint_hashes == EXPECTED_CHECKPOINT_HASHES},
        {"Check": "Pre-March source contains no March-or-later rows", "Passed": bool((all_routes["Date"] < pd.Timestamp("2026-03-02")).all())},
        {"Check": "Selected daily method matches ND08 contract", "Passed": selected_daily_method == "NESTED_MEDIAN_ENSEMBLE"},
        {"Check": "Selected week method matches ND08 contract", "Passed": selected_week_method == "ROLLING_MEAN_5"},
        {"Check": "Selected updated-week method matches ND08 contract", "Passed": selected_updated_method == "ROLLING_MEAN_5"},
        {"Check": "Daily routed predictions contain finite forecasts", "Passed": bool(np.isfinite(pd.to_numeric(routed[PRED], errors="coerce")).all())},
        {"Check": "Recursive predictions contain finite forecasts", "Passed": bool(np.isfinite(pd.to_numeric(recursive_daily["PredictedNormalDemand"], errors="coerce")).all())},
        {"Check": "ND10 daily planning identity reconciles", "Passed": bool(np.allclose(demo_daily["PredictedNormalDemand"] + demo_daily["ConfirmedBulkDemand"], demo_daily["PlannedQuantity"], atol=1e-10, rtol=1e-10))},
        {"Check": "ND10 weekly planning identity reconciles", "Passed": bool(np.allclose(demo_week_products["PredictedWeekNormalDemand"] + demo_week_products["ConfirmedWeekBulkDemand"], demo_week_products["PlannedWeekQuantity"], atol=1e-10, rtol=1e-10))},
        {"Check": "At least 20 report/presentation figures created", "Passed": len(figure_records) >= 20},
    ]
    validation = pd.DataFrame(validation_rows)
    if not validation["Passed"].all():
        raise AssertionError("ND10R validation failed:\n" + validation.loc[~validation["Passed"]].to_string(index=False))
    write_csv(staged_diag / "ND10R_validation_summary.csv", validation)

    # Package versions.
    versions = pd.DataFrame([
        {"Package": "python", "Version": platform.python_version()},
        {"Package": "pandas", "Version": pd.__version__},
        {"Package": "numpy", "Version": np.__version__},
        {"Package": "matplotlib", "Version": __import__("matplotlib").__version__},
    ])
    write_csv(staged_diag / "ND10R_package_versions.csv", versions)

    # Manifest before control outputs.
    manifest = build_manifest(STAGING_ROOT)
    write_csv(staged_control / MANIFEST_PATH.name, manifest)
    manifest_sha = sha256_file(staged_control / MANIFEST_PATH.name)

    # Create a self-contained ZIP of the user-facing report material.
    # Control files are deliberately excluded so the archive never attempts
    # to include itself while it is being written.
    staged_bundle = staged_control / BUNDLE_PATH.name
    bundle_roots = [
        staged_table,
        staged_diag,
        staged_fig,
        staged_report,
        staged_demo,
    ]
    with zipfile.ZipFile(staged_bundle, "w", compression=zipfile.ZIP_DEFLATED) as archive:
        archive.write(STAGING_ROOT / "README.md", arcname="README.md")
        for bundle_root in bundle_roots:
            for bundle_file in sorted(bundle_root.rglob("*")):
                if bundle_file.is_file():
                    archive.write(
                        bundle_file,
                        arcname=str(bundle_file.relative_to(STAGING_ROOT)),
                    )
    bundle_sha = sha256_file(staged_bundle)

    checkpoint_payload = {
        "StepID": STEP_ID,
        "Status": STATUS,
        "CompletedLocalTime": NOW_LOCAL.isoformat(),
        "Root": str(ND10R_ROOT),
        "InputCheckpointHashes": checkpoint_hashes,
        "ManifestSHA256": manifest_sha,
        "BundleSHA256": bundle_sha,
        "FiguresCreated": len(figure_records),
        "SelectedDailyMethod": selected_daily_method,
        "SelectedWeekMethod": selected_week_method,
        "SelectedDailyUpdatedWeekMethod": selected_updated_method,
        "MarchTargetVaultOpened": False,
        "ModelsRefitted": False,
        "MethodsReselected": False,
        "NextStep": "ND11_INGREDIENT_MAPPING",
    }
    write_json(staged_control / CHECKPOINT_PATH.name, checkpoint_payload)
    checkpoint_sha = sha256_file(staged_control / CHECKPOINT_PATH.name)
    write_text(staged_control / CHECKPOINT_SHA_PATH.name, checkpoint_sha + "\n")

    # Verify protected inputs before atomic move.
    protected_hashes_after = {str(path): sha256_file(path) for path in required_inputs}
    changed = [path for path in protected_hashes_before if protected_hashes_before[path] != protected_hashes_after[path]]
    if changed:
        raise AssertionError("Protected input files changed during ND10R:\n" + "\n".join(f"- {path}" for path in changed))

    os.replace(STAGING_ROOT, ND10R_ROOT)

    TOP_LEVEL_CHECKPOINT_PATH.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(ND10R_ROOT / "06_control" / CHECKPOINT_PATH.name, TOP_LEVEL_CHECKPOINT_PATH)
    shutil.copy2(ND10R_ROOT / "06_control" / CHECKPOINT_SHA_PATH.name, TOP_LEVEL_CHECKPOINT_SHA_PATH)

    # Project memory / handoff.
    handoff = f"""# ND10R Handoff

## Status

- Completed step: `{STEP_ID}`
- Status: `{STATUS}`
- Root: `{ND10R_ROOT}`
- Checkpoint SHA-256: `{checkpoint_sha}`
- Report figures created: {len(figure_records)}

## Purpose

A comprehensive report-ready diagnostic archive was created for the revised normal-demand model before ingredient mapping and UI construction. No model was changed.

## Key selected methods

- Next operating day: `{selected_daily_method}`
- Monday-origin week: `{selected_week_method}`
- Daily-updated week: `{selected_updated_method}`

## Main results

- Daily main-route median ensemble WAPE: {median_wape:.6f}%
- Full routed daily product WAPE: {selected_routed_metric['WAPEPercentage']:.6f}%
- Full routed restaurant-day WAPE: {selected_restaurant_metric['WAPEPercentage']:.6f}%
- Monday-origin product-week WAPE: {origin_val:.6f}%
- Daily-updated product-week WAPE: {updated_val:.6f}%

## Safety

- March target vault opened: no
- Models refitted: no
- Methods reselected: no

## Next step

ND11 — ingredient mapping, followed by ND12 demonstration UI.
"""
    atomic_write_text(HANDOFF_PATH, handoff)
    atomic_write_text(CURRENT_HANDOFF_PATH, handoff)

    append_once(
        WORKFLOW_PATH,
        "## ND10R — Report-ready diagnostics and visuals",
        f"""## ND10R — Report-ready diagnostics and visuals

Status: `{STATUS}`

A comprehensive report/presentation diagnostic archive was generated from the frozen revised model without opening March targets or changing the forecasting system. The next modelling-adjacent step is ND11 ingredient mapping.
""",
    )
    append_once(
        DECISIONS_PATH,
        "## ND10R reporting decision",
        """## ND10R reporting decision

Create the final revised-model diagnostic and visual reporting package before ingredient mapping and UI construction so the modelling results are frozen, documented and presentation-ready independently of the demonstration interface.
""",
    )
    append_once(
        METRICS_PATH,
        "## ND10R report-ready metrics",
        f"""## ND10R report-ready metrics

- Selected daily main-route WAPE: {median_wape:.6f}%
- Full routed daily product WAPE: {selected_routed_metric['WAPEPercentage']:.6f}%
- Full routed restaurant-day WAPE: {selected_restaurant_metric['WAPEPercentage']:.6f}%
- Monday-origin product-week WAPE: {origin_val:.6f}%
- Daily-updated product-week WAPE: {updated_val:.6f}%
- Report figures created: {len(figure_records)}
""",
    )
    append_once(
        AGENTS_PATH,
        "Marker: ND10R_AUTHORITATIVE_STATUS",
        f"""## ND10R authoritative status

Marker: ND10R_AUTHORITATIVE_STATUS

- Status: `{STATUS}`
- Handoff: `{HANDOFF_PATH}`
- Checkpoint SHA-256: `{checkpoint_sha}`
- Next step: ND11 ingredient mapping.
""",
    )

    LOG_PATH.parent.mkdir(parents=True, exist_ok=True)
    with LOG_PATH.open("a", encoding="utf-8") as handle:
        handle.write(f"{NOW_LOCAL.isoformat()} | {STATUS} | checkpoint={checkpoint_sha} | root={ND10R_ROOT}\n")

except Exception:
    if STAGING_ROOT.exists():
        shutil.rmtree(STAGING_ROOT, ignore_errors=True)
    raise

# =============================================================================
# FINAL CONSOLE OUTPUT
# =============================================================================

print("=" * 122)
print("EDEN NORMAL-DEMAND MODEL V2 — ND10R REPORT-READY DIAGNOSTICS COMPLETE")
print("=" * 122)
print(f"Status: {STATUS}")
print(f"Local time: {NOW_LOCAL.isoformat()}")
print(f"ND10R root: {ND10R_ROOT}")
print()
print("INPUT VERIFICATION")
for name in ["ND03", "ND04", "ND05", "ND06", "ND07", "ND07E", "ND08", "ND09", "ND10"]:
    print(f"{name} checkpoint SHA-256: {checkpoint_hashes[name]}")
print("March target vault opened: False")
print("Previous modelling/planning inputs modified: False")
print()
print("FROZEN OPERATIONAL SYSTEM")
print(f"Next-operating-day method: {selected_daily_method}")
print("Representative median components: ROLLING_MEAN_5, XGBOOST_PRODUCT_AWARE")
print(f"Monday-origin weekly method: {selected_week_method}")
print(f"Daily-updated remaining-week method: {selected_updated_method}")
print()
print("KEY REPORT RESULTS")
print(f"Rolling-mean daily main-route benchmark WAPE: {benchmark_wape:.6f}%")
print(f"Selected median-ensemble daily main-route WAPE: {median_wape:.6f}%")
print(f"Daily WAPE improvement: {daily_improvement_pp:.6f} percentage points")
print(f"Selected routed daily product WAPE: {selected_routed_metric['WAPEPercentage']:.6f}%")
print(f"Selected routed restaurant-day WAPE: {selected_restaurant_metric['WAPEPercentage']:.6f}%")
print(f"Monday-origin product-week WAPE: {origin_val:.6f}%")
print(f"Monday-origin restaurant-week WAPE: {float(recursive_restaurant_selected['WAPEPercentage']):.6f}%")
print(f"Daily-updated product-week WAPE: {updated_val:.6f}%")
print(f"Daily-updated restaurant-week WAPE: {float(updated_restaurant_selected['WAPEPercentage']):.6f}%")
print()
print("DEMONSTRATION OUTPUTS")
print(f"Daily demonstration date: {demo_date.date()}")
print(f"Daily restaurant normal demand: {float(demo_summary.iloc[0]['NormalDemand']):.6f}")
print(f"Daily confirmed bulk: {float(demo_summary.iloc[0]['ConfirmedBulkDemand']):.6f}")
print(f"Daily planned quantity: {float(demo_summary.iloc[0]['PlannedQuantity']):.6f}")
print(f"Demonstration week: {demo_week_start.date()} to {demo_week_end.date()}")
print(f"Week restaurant normal demand: {float(demo_summary.iloc[1]['NormalDemand']):.6f}")
print(f"Week confirmed bulk: {float(demo_summary.iloc[1]['ConfirmedBulkDemand']):.6f}")
print(f"Week planned quantity: {float(demo_summary.iloc[1]['PlannedQuantity']):.6f}")
print()
print("REPORT PACKAGE")
print(f"Figures created: {len(figure_records)}")
print(f"Figure directory: {FIGURE_DIR}")
print(f"Report wording: {REPORT_DIR / 'ND10R_report_ready_results_wording.md'}")
print(f"Figure captions: {REPORT_DIR / 'ND10R_figure_captions_and_report_use.csv'}")
print(f"Recommended main-report figures: {REPORT_DIR / 'ND10R_recommended_main_report_figures.csv'}")
print(f"Demonstration material: {DEMO_DIR}")
print(f"Report-ready ZIP: {BUNDLE_PATH}")
print(f"Checkpoint: {TOP_LEVEL_CHECKPOINT_PATH}")
print(f"Checkpoint SHA-256: {checkpoint_sha}")
print(f"Handoff: {HANDOFF_PATH}")
print()
print("SAFETY")
print("- Forecasting models fitted/refitted: False")
print("- Forecasting methods reselected: False")
print("- March target vault opened: False")
print("- Future demonstration forecasts scored against March actuals: False")
print("- Previous inputs modified: False")
print("- ND10R checkpoint and hashes created: True")
print()
print("NEXT STEP")
print("ND11 — ingredient mapping, then ND12 — demonstration UI.")
print("=" * 122)

EDEN NORMAL-DEMAND MODEL V2 — ND10R REPORT-READY DIAGNOSTICS COMPLETE
Status: ND10R_REPORT_READY_DIAGNOSTICS_AND_VISUALS_CREATED_READY_FOR_ND11
Local time: 2026-08-10T11:58:56.317439+01:00
ND10R root: /Users/ryansmac/Desktop/Meng Project/eden_datasets/eden_normal_demand_model_v2/05_reporting/ND10R_report_ready_diagnostics_and_visuals

INPUT VERIFICATION
ND03 checkpoint SHA-256: 0845af89a5b459ca13ae6ffd99dde444f5010f6c0fb5ba4c34a4f091ac2e151c
ND04 checkpoint SHA-256: 2fdc5d2c64c38f85b2669ca942042884209d80111cc840261307da98b1e9cf54
ND05 checkpoint SHA-256: ce3342c8b960aa5c4791a114ab09ae1a86d1dab2a3eebb648aa060579a1378ff
ND06 checkpoint SHA-256: e3b7bb75a1b2968e426c9a4c1654e10420d683af4bd5cb2b75fa4b5f0f18f357
ND07 checkpoint SHA-256: 39077dd8c197561e5384a6943c3a4e153153e75fa4d03019f45005eb145cf18e
ND07E checkpoint SHA-256: eaf6d23434cb67d663f66a39db4f898b74c1d516aee5c0627ca9136bed9877b8
ND08 checkpoint SHA-256: a98828007df9bd4d0cce309fc8f8bfbc1d2c1f1cf74f1ca762b27237468dea10
ND09 checkpoi

In [17]:
# =============================================================================
# EDEN NORMAL-DEMAND MODEL V2
# ND09A — ACTIVE-MENU CATALOGUE CORRECTION
#
# Run this as one complete Jupyter cell after the completed ND09.
#
# This is an auditable inference-scope correction. It DOES NOT refit, retune,
# calibrate, or reselect the frozen forecasting models. It reuses the locked
# ND09 model artifacts and changes only the future forecast catalogue from the
# full 227-product historical universe to the latest known active product panel.
#
# The catalogue policy is supported by the final 20 pre-March operating days:
#   - 90 products appeared at least once,
#   - 86 were present on all 20 dates,
#   - 2 additional products were recent additions and active at cutoff,
#   - 2 products disappeared before cutoff,
#   - therefore 88 products form the latest active panel at 2026-02-27.
#
# March 2026 targets remain closed.
# =============================================================================

import hashlib
import json
import os
import platform
import shutil
import time
import uuid
from datetime import datetime, timezone
from pathlib import Path
from zoneinfo import ZoneInfo

import joblib
import numpy as np
import pandas as pd
import sklearn

try:
    import xgboost
    from xgboost import XGBRegressor
except Exception as error:
    raise RuntimeError("ND09A requires xgboost from the frozen ND09 environment.") from error

try:
    import catboost
    from catboost import CatBoostRegressor
except Exception as error:
    raise RuntimeError("ND09A requires catboost from the frozen ND09 environment.") from error

os.environ.setdefault("OMP_NUM_THREADS", "1")
os.environ.setdefault("OPENBLAS_NUM_THREADS", "1")
os.environ.setdefault("MKL_NUM_THREADS", "1")
os.environ.setdefault("NUMEXPR_NUM_THREADS", "1")


# =============================================================================
# USER CONFIGURATION
# =============================================================================

# None = forecast every product in the corrected latest active panel.
# A list may be supplied later for a deliberately smaller demonstration menu,
# but it must be a subset of the 88 current-panel products.
ACTIVE_PRODUCT_IDS = None

ALLOW_OVERWRITE = False


# =============================================================================
# FIXED PROJECT CONFIGURATION
# =============================================================================

PROJECT_ROOT = Path("/Users/ryansmac/Desktop/Meng Project")
EDEN_ROOT = PROJECT_ROOT / "eden_datasets"
MODEL_ROOT = EDEN_ROOT / "eden_normal_demand_model_v2"

ND03_ROOT = MODEL_ROOT / "02_feature_engineering" / "ND03_normal_demand_features"
ALL_ROUTES_PATH = (
    ND03_ROOT / "01_model_ready_datasets" / "ND03_pre_march_all_routes_development_dataset.csv"
)
CORE_PREDICTOR_LIST_PATH = ND03_ROOT / "03_contracts" / "ND03_core_predictor_list.csv"
ND03_CHECKPOINT_PATH = MODEL_ROOT / "08_checkpoints" / "ND03_checkpoint.json"

FROZEN_ND09_ROOT = (
    MODEL_ROOT / "03_models" / "03_inference" / "ND09_arbitrary_date_inference_engine"
)
FROZEN_MODEL_DIR = FROZEN_ND09_ROOT / "01_model_artifacts"
FROZEN_FORECAST_DIR = FROZEN_ND09_ROOT / "02_forecasts"
FROZEN_CONTRACT_DIR = FROZEN_ND09_ROOT / "03_contracts"

FROZEN_CORE_PREPROCESSOR_PATH = FROZEN_MODEL_DIR / "ND09_core_preprocessor.joblib"
FROZEN_PRODUCT_PREPROCESSOR_PATH = FROZEN_MODEL_DIR / "ND09_product_preprocessor.joblib"
FROZEN_CATBOOST_MODEL_PATH = FROZEN_MODEL_DIR / "ND09_catboost_core53.cbm"
FROZEN_XGBOOST_CORE_MODEL_PATH = FROZEN_MODEL_DIR / "ND09_xgboost_core53.json"
FROZEN_XGBOOST_PRODUCT_MODEL_PATH = FROZEN_MODEL_DIR / "ND09_xgboost_product_aware.json"
FROZEN_MODEL_METADATA_PATH = FROZEN_MODEL_DIR / "ND09_model_artifact_metadata.json"
FROZEN_INFERENCE_CONTRACT_PATH = FROZEN_CONTRACT_DIR / "ND09_inference_engine_contract.json"
FROZEN_ACTIVE_CATALOGUE_PATH = FROZEN_CONTRACT_DIR / "ND09_active_product_catalogue.csv"
FROZEN_DAILY_PRODUCT_PATH = FROZEN_FORECAST_DIR / "ND09_arbitrary_date_product_forecast.csv"
FROZEN_WEEK_DAILY_PRODUCT_PATH = FROZEN_FORECAST_DIR / "ND09_week_ahead_daily_product_forecast.csv"
FROZEN_ND09_CHECKPOINT_PATH = MODEL_ROOT / "08_checkpoints" / "ND09_checkpoint.json"
FROZEN_ND09_LOCK_PATH = MODEL_ROOT / "08_checkpoints" / "ND09_demonstration_engine_lock.json"

EXPECTED_ND03_CHECKPOINT_SHA256 = "0845af89a5b459ca13ae6ffd99dde444f5010f6c0fb5ba4c34a4f091ac2e151c"
EXPECTED_FROZEN_ND09_CHECKPOINT_SHA256 = "201b84095ed138e0b666485e2022e4f1f99ad93b89971f46533ff4a0605ef4e8"

DATE_COLUMN = "Date"
PRODUCT_ID_COLUMN = "CanonicalProductID"
PRODUCT_NAME_COLUMN = "CanonicalProductName"
TARGET_COLUMN = "NormalDemand"
ROUTE_COLUMN = "ForecastRoute"
FAMILY_COLUMN = "TierProductFamily"
DAY_OF_WEEK_COLUMN = "DayOfWeekNumber"
SEQUENCE_COLUMN = "OperatingDaySequence"

EXPECTED_ALL_ROUTE_ROWS = 23_763
EXPECTED_HISTORICAL_PRODUCTS = 227
EXPECTED_CURRENT_ACTIVE_PRODUCTS = 88
EXPECTED_LAST20_UNIQUE_PRODUCTS = 90
EXPECTED_LAST20_STABLE_PRODUCTS = 86
EXPECTED_LAST20_RECENT_ACTIVE_ADDITIONS = 2
EXPECTED_LAST20_RETIRED_PRODUCTS = 2
DEVELOPMENT_END_EXCLUSIVE = pd.Timestamp("2026-03-02")

LAG_OPERATING_DAYS = [1, 2, 3, 5, 10, 20]
ROLLING_WINDOWS = [3, 5, 10, 20]
ZERO_POSITIVE_WINDOWS = [5, 10, 20]
MINIMUM_MAIN_HISTORY = 20
PRIMARY_SCOPE_PERCENTAGE = 95.0

HISTORICAL_DEMAND_PREDICTORS = (
    [f"NormalDemandLag_{lag}" for lag in LAG_OPERATING_DAYS]
    + [
        feature
        for window in ROLLING_WINDOWS
        for feature in [
            f"PastNormalDemandRollingMean_{window}",
            f"PastNormalDemandRollingMedian_{window}",
            f"PastNormalDemandRollingStd_{window}",
            f"PastNormalDemandRollingSum_{window}",
        ]
    ]
    + [f"PastZeroNormalDemandRate_{window}" for window in ZERO_POSITIVE_WINDOWS]
    + [f"PastPositiveNormalDemandCount_{window}" for window in ZERO_POSITIVE_WINDOWS]
    + [
        "OperatingDaysSincePreviousPositiveNormalDemand",
        "ExpandingPastMeanNormalDemand",
        "ExpandingPastPositiveNormalDemandRate",
    ]
)

BASE_NUMERIC_PREDICTORS = [
    "ProductAgeOperatingDays",
    "SourcePLUCount",
    "IsMultiPLUCanonicalProduct",
    "OperatingDaySequence",
    "Year",
    "Month",
    "Quarter",
    "DayOfWeekNumber",
    "ISOYear",
    "ISOWeek",
    "DayOfYear",
    "IsWeekend",
    "DaysSincePreviousOperatingDate",
    "IsConsecutiveCalendarDay",
]

BASE_CATEGORICAL_PREDICTORS = [
    "SourceGroupCodes",
    "SourceGroupNames",
    "BeverageSeries",
    "BeverageType",
    "SupplierLabelsObserved",
    "TierProductFamily",
    "NominalPriceTier",
    "MenuGeneration",
]

PRODUCT_METADATA_COLUMNS = [
    PRODUCT_ID_COLUMN,
    PRODUCT_NAME_COLUMN,
    "ProductFirstObservedDate",
    "SourcePLUCount",
    "IsMultiPLUCanonicalProduct",
    *BASE_CATEGORICAL_PREDICTORS,
]

BASE_COMPONENTS = [
    "ROLLING_MEAN_5",
    "CATBOOST_CORE53",
    "XGBOOST_CORE53",
    "XGBOOST_PRODUCT_AWARE",
]

ND09A_ROOT = (
    MODEL_ROOT / "03_models" / "03_inference" / "ND09A_active_menu_catalogue_correction"
)
FORECAST_DIR = ND09A_ROOT / "01_forecasts"
CONTRACT_DIR = ND09A_ROOT / "02_contracts"
AUDIT_DIR = ND09A_ROOT / "03_audits"
REPORT_DIR = ND09A_ROOT / "04_reports"
CONTROL_DIR = ND09A_ROOT / "07_control"

DAILY_PATH_FORECAST_PATH = FORECAST_DIR / "ND09A_arbitrary_date_recursive_path.csv"
DAILY_PRODUCT_FORECAST_PATH = FORECAST_DIR / "ND09A_arbitrary_date_product_forecast.csv"
DAILY_RESTAURANT_FORECAST_PATH = FORECAST_DIR / "ND09A_arbitrary_date_restaurant_total.csv"
WEEK_PATH_FORECAST_PATH = FORECAST_DIR / "ND09A_week_ahead_recursive_path.csv"
WEEK_DAILY_PRODUCT_FORECAST_PATH = FORECAST_DIR / "ND09A_week_ahead_daily_product_forecast.csv"
WEEK_PRODUCT_TOTAL_PATH = FORECAST_DIR / "ND09A_week_ahead_product_totals.csv"
WEEK_RESTAURANT_DAILY_PATH = FORECAST_DIR / "ND09A_week_ahead_restaurant_daily_totals.csv"
WEEK_RESTAURANT_TOTAL_PATH = FORECAST_DIR / "ND09A_week_ahead_restaurant_total.csv"

ACTIVE_CATALOGUE_PATH = CONTRACT_DIR / "ND09A_active_product_catalogue.csv"
INFERENCE_CONTRACT_PATH = CONTRACT_DIR / "ND09A_corrected_inference_contract.json"
CATALOGUE_POLICY_PATH = CONTRACT_DIR / "ND09A_active_catalogue_policy.md"

LAST20_AUDIT_PATH = AUDIT_DIR / "ND09A_last20_catalogue_audit.csv"
HISTORICAL_SCOPE_AUDIT_PATH = AUDIT_DIR / "ND09A_historical_product_scope_audit.csv"
FEATURE_REPLAY_AUDIT_PATH = AUDIT_DIR / "ND09A_feature_route_replay_audit.csv"
FORECAST_PATH_AUDIT_PATH = AUDIT_DIR / "ND09A_forecast_path_audit.csv"
SCOPE_COMPARISON_PATH = AUDIT_DIR / "ND09A_scope_change_comparison.csv"
ROUTE_SUMMARY_PATH = AUDIT_DIR / "ND09A_forecast_route_summary.csv"
VALIDATION_PATH = AUDIT_DIR / "ND09A_validation_summary.csv"
PACKAGE_VERSIONS_PATH = AUDIT_DIR / "ND09A_package_versions.csv"

REPORT_SUMMARY_PATH = REPORT_DIR / "ND09A_active_menu_correction_summary.md"
README_PATH = ND09A_ROOT / "README.md"
MANIFEST_PATH = CONTROL_DIR / "ND09A_artifact_hash_manifest.csv"
CHECKPOINT_PATH = CONTROL_DIR / "ND09A_checkpoint.json"
CHECKPOINT_SHA_PATH = CONTROL_DIR / "ND09A_checkpoint.sha256"
LOCK_PATH = CONTROL_DIR / "ND09A_active_catalogue_scope_lock.json"

TOP_LEVEL_CHECKPOINT_PATH = MODEL_ROOT / "08_checkpoints" / "ND09A_checkpoint.json"
TOP_LEVEL_CHECKPOINT_SHA_PATH = MODEL_ROOT / "08_checkpoints" / "ND09A_checkpoint.sha256"
TOP_LEVEL_LOCK_PATH = MODEL_ROOT / "08_checkpoints" / "ND09A_active_catalogue_scope_lock.json"

MEMORY_ROOT = MODEL_ROOT / "00_project_memory"
HANDOFF_PATH = MEMORY_ROOT / "ND09A_HANDOFF.md"
CURRENT_HANDOFF_PATH = MEMORY_ROOT / "CURRENT_HANDOFF.md"
WORKFLOW_PATH = MEMORY_ROOT / "WORKFLOW.md"
DECISIONS_PATH = MEMORY_ROOT / "DECISIONS.md"
METRICS_AND_RESULTS_PATH = MEMORY_ROOT / "METRICS_AND_RESULTS.md"
AGENTS_PATH = MODEL_ROOT / "AGENTS.md"
LOG_PATH = MODEL_ROOT / "09_logs" / "ND09A_active_menu_catalogue_correction_log.txt"

STEP_ID = "ND09A"
STATUS = "ND09A_ACTIVE_MENU_CATALOGUE_CORRECTION_COMPLETED_READY_FOR_ND10A"
NOW_UTC = datetime.now(timezone.utc)
NOW_LOCAL = NOW_UTC.astimezone(ZoneInfo("Europe/Dublin"))


def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

def write_csv(path: Path, frame: pd.DataFrame) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    frame.to_csv(path, index=False)

def write_json(path: Path, payload: dict) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(
        json.dumps(payload, indent=2, ensure_ascii=False, default=str) + "\n",
        encoding="utf-8",
    )

def write_text(path: Path, text: str) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(text, encoding="utf-8")

def atomic_write_text(path: Path, text: str) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_name(f".{path.name}.{uuid.uuid4().hex}.tmp")
    temporary.write_text(text, encoding="utf-8")
    os.replace(temporary, path)

def append_marked_section(path: Path, marker: str, section_text: str) -> None:
    existing = path.read_text(encoding="utf-8") if path.is_file() else ""
    if marker in existing:
        return
    separator = "\n" if existing.endswith("\n") else "\n\n"
    atomic_write_text(path, existing + separator + section_text.strip() + "\n")

def validate_required_columns(
    frame: pd.DataFrame,
    columns: set[str],
    name: str,
) -> None:
    missing = sorted(columns - set(frame.columns))
    if missing:
        raise AssertionError(
            f"{name} is missing required columns:\n"
            + "\n".join(f"- {column}" for column in missing)
        )

def normalize_family(values: pd.Series) -> pd.Series:
    return values.astype("string").fillna("__MISSING_FAMILY__").astype(str)

def parse_iso_dates(values: list[str], label: str) -> set[pd.Timestamp]:
    parsed = set()
    for value in values:
        timestamp = pd.Timestamp(value).normalize()
        if pd.isna(timestamp):
            raise ValueError(f"Invalid {label} date: {value}")
        parsed.add(timestamp)
    return parsed

def is_operating_date(
    date: pd.Timestamp,
    closed_dates: set[pd.Timestamp],
    extra_dates: set[pd.Timestamp],
) -> bool:
    date = pd.Timestamp(date).normalize()
    if date in closed_dates:
        return False
    return date.weekday() < 5 or date in extra_dates

def next_operating_date(
    date: pd.Timestamp,
    closed_dates: set[pd.Timestamp],
    extra_dates: set[pd.Timestamp],
) -> pd.Timestamp:
    candidate = pd.Timestamp(date).normalize() + pd.Timedelta(days=1)
    for _ in range(370):
        if is_operating_date(candidate, closed_dates, extra_dates):
            return candidate
        candidate += pd.Timedelta(days=1)
    raise RuntimeError("No operating date found within 370 calendar days.")

def monday_on_or_after(date: pd.Timestamp) -> pd.Timestamp:
    date = pd.Timestamp(date).normalize()
    return date + pd.Timedelta(days=(7 - date.weekday()) % 7)

def generate_future_calendar(
    cutoff_date: pd.Timestamp,
    cutoff_sequence: int,
    end_date: pd.Timestamp,
    closed_dates: set[pd.Timestamp],
    extra_dates: set[pd.Timestamp],
) -> pd.DataFrame:
    cutoff_date = pd.Timestamp(cutoff_date).normalize()
    end_date = pd.Timestamp(end_date).normalize()
    if end_date <= cutoff_date:
        raise ValueError("Future calendar end date must be after the cutoff date.")

    records = []
    sequence = int(cutoff_sequence)
    previous_operating_date = cutoff_date

    for date in pd.date_range(cutoff_date + pd.Timedelta(days=1), end_date, freq="D"):
        if not is_operating_date(date, closed_dates, extra_dates):
            continue
        sequence += 1
        iso = date.isocalendar()
        gap = int((date - previous_operating_date).days)
        records.append(
            {
                DATE_COLUMN: date,
                SEQUENCE_COLUMN: sequence,
                "Year": int(date.year),
                "Month": int(date.month),
                "Quarter": int(date.quarter),
                "DayOfWeekNumber": int(date.weekday()),
                "ISOYear": int(iso.year),
                "ISOWeek": int(iso.week),
                "DayOfYear": int(date.dayofyear),
                "IsWeekend": bool(date.weekday() >= 5),
                "DaysSincePreviousOperatingDate": gap,
                "IsConsecutiveCalendarDay": bool(gap == 1),
                "CalendarPolicy": (
                    "EXCEPTIONAL_OPERATING_DATE"
                    if date.weekday() >= 5
                    else "STANDARD_MONDAY_TO_FRIDAY"
                ),
            }
        )
        previous_operating_date = date

    calendar = pd.DataFrame(records)
    if calendar.empty:
        raise AssertionError("The future operating calendar is empty.")
    return calendar

def historical_features_from_history(
    demand_history: np.ndarray,
    sequence_history: np.ndarray,
    current_sequence: float,
) -> dict[str, float]:
    demand_history = np.asarray(demand_history, dtype=float)
    sequence_history = np.asarray(sequence_history, dtype=float)
    result: dict[str, float] = {}

    for lag in LAG_OPERATING_DAYS:
        result[f"NormalDemandLag_{lag}"] = (
            float(demand_history[-lag])
            if len(demand_history) >= lag
            else float("nan")
        )

    for window in ROLLING_WINDOWS:
        values = demand_history[-window:]
        if len(values) == 0:
            mean_value = median_value = std_value = sum_value = float("nan")
        else:
            mean_value = float(np.mean(values))
            median_value = float(np.median(values))
            std_value = float(np.std(values, ddof=0))
            sum_value = float(np.sum(values))
        result[f"PastNormalDemandRollingMean_{window}"] = mean_value
        result[f"PastNormalDemandRollingMedian_{window}"] = median_value
        result[f"PastNormalDemandRollingStd_{window}"] = std_value
        result[f"PastNormalDemandRollingSum_{window}"] = sum_value

    for window in ZERO_POSITIVE_WINDOWS:
        values = demand_history[-window:]
        if len(values) == 0:
            zero_rate = positive_count = float("nan")
        else:
            zero_rate = float(np.mean(values == 0))
            positive_count = float(np.sum(values > 0))
        result[f"PastZeroNormalDemandRate_{window}"] = zero_rate
        result[f"PastPositiveNormalDemandCount_{window}"] = positive_count

    positive_indices = np.flatnonzero(demand_history > 0)
    if len(positive_indices) == 0:
        days_since_positive = float("nan")
    else:
        last_positive_sequence = float(sequence_history[int(positive_indices[-1])])
        days_since_positive = float(current_sequence - last_positive_sequence)
    result["OperatingDaysSincePreviousPositiveNormalDemand"] = days_since_positive

    if len(demand_history) == 0:
        expanding_mean = float("nan")
        expanding_positive_rate = float("nan")
    else:
        expanding_mean = float(np.mean(demand_history))
        expanding_positive_rate = float(np.mean(demand_history > 0))
    result["ExpandingPastMeanNormalDemand"] = expanding_mean
    result["ExpandingPastPositiveNormalDemandRate"] = expanding_positive_rate
    return result

def build_history_state(training_frame: pd.DataFrame) -> dict[str, dict[str, list]]:
    state: dict[str, dict[str, list]] = {}
    ordered = training_frame.sort_values(
        [PRODUCT_ID_COLUMN, DATE_COLUMN, SEQUENCE_COLUMN],
        kind="mergesort",
    )
    for product_id, product_frame in ordered.groupby(PRODUCT_ID_COLUMN, sort=False):
        product_key = str(product_id)
        state[product_key] = {
            "demand": product_frame[TARGET_COLUMN].astype(float).tolist(),
            "sequence": product_frame[SEQUENCE_COLUMN].astype(float).tolist(),
        }
    return state

def copy_history_state(
    base_state: dict[str, dict[str, list]],
) -> dict[str, dict[str, list]]:
    return {
        product_id: {
            "demand": values["demand"].copy(),
            "sequence": values["sequence"].copy(),
        }
        for product_id, values in base_state.items()
    }

def append_predictions_to_state(
    state: dict[str, dict[str, list]],
    rows: pd.DataFrame,
    predictions: np.ndarray,
) -> None:
    for product_id, sequence, prediction in zip(
        rows[PRODUCT_ID_COLUMN].astype(str),
        rows[SEQUENCE_COLUMN].astype(float),
        np.asarray(predictions, dtype=float),
    ):
        if product_id not in state:
            state[product_id] = {"demand": [], "sequence": []}
        if state[product_id]["sequence"]:
            if sequence <= state[product_id]["sequence"][-1]:
                raise AssertionError(
                    f"Non-increasing operating sequence for {product_id}."
                )
        state[product_id]["demand"].append(float(prediction))
        state[product_id]["sequence"].append(float(sequence))

def build_recursive_feature_frame(
    score_rows: pd.DataFrame,
    state: dict[str, dict[str, list]],
) -> pd.DataFrame:
    frame = score_rows.copy().reset_index(drop=True)
    feature_records = []
    prior_records = []

    for row in frame.itertuples(index=False):
        product_id = str(getattr(row, PRODUCT_ID_COLUMN))
        current_sequence = float(getattr(row, SEQUENCE_COLUMN))
        product_state = state.get(product_id, {"demand": [], "sequence": []})
        demand_history = np.asarray(product_state["demand"], dtype=float)
        sequence_history = np.asarray(product_state["sequence"], dtype=float)

        feature_records.append(
            historical_features_from_history(
                demand_history,
                sequence_history,
                current_sequence,
            )
        )

        prior_count = int(len(demand_history))
        prior_cumulative = float(demand_history.sum())
        prior_positive = int(np.sum(demand_history > 0))
        prior_records.append(
            {
                "PriorOperatingDayCount": prior_count,
                "PriorCumulativeNormalDemand": prior_cumulative,
                "PriorPositiveNormalDemandDays": prior_positive,
                "PriorZeroNormalDemandDays": prior_count - prior_positive,
                "HasSufficientHistory20": prior_count >= MINIMUM_MAIN_HISTORY,
                "ColdStartFlag": prior_count < MINIMUM_MAIN_HISTORY,
                "ZeroPriorNormalDemandFlag": prior_cumulative <= 0,
            }
        )

    feature_values = pd.DataFrame(feature_records)
    prior_values = pd.DataFrame(prior_records)
    for column in HISTORICAL_DEMAND_PREDICTORS:
        frame[column] = feature_values[column].to_numpy()
    for column in prior_values.columns:
        frame[column] = prior_values[column].to_numpy()

    ranked = frame.sort_values(
        ["PriorCumulativeNormalDemand", "PriorPositiveNormalDemandDays", PRODUCT_ID_COLUMN],
        ascending=[False, False, True],
        kind="mergesort",
    ).copy()

    universe_total = float(ranked["PriorCumulativeNormalDemand"].sum())
    ranked["PriorDemandRank"] = np.arange(1, len(ranked) + 1, dtype=int)
    ranked["PriorDemandSharePercentage"] = 0.0
    ranked["PriorCumulativeDemandShareBeforePercentage"] = 0.0
    ranked["PriorCumulativeDemandSharePercentage"] = 0.0
    ranked["InPrior95DemandScope"] = False

    if universe_total > 0:
        share = 100.0 * ranked["PriorCumulativeNormalDemand"] / universe_total
        cumulative = share.cumsum()
        cumulative_before = cumulative - share
        ranked["PriorDemandSharePercentage"] = share
        ranked["PriorCumulativeDemandShareBeforePercentage"] = cumulative_before
        ranked["PriorCumulativeDemandSharePercentage"] = cumulative
        ranked["InPrior95DemandScope"] = (
            (ranked["PriorCumulativeNormalDemand"] > 0)
            & (cumulative_before < PRIMARY_SCOPE_PERCENTAGE)
        )

    ranked["EligibleForMainModel"] = (
        ranked["HasSufficientHistory20"]
        & ranked["InPrior95DemandScope"]
        & ~ranked["ZeroPriorNormalDemandFlag"]
    )

    ranked["RecursiveForecastRoute"] = np.select(
        [
            ranked["ZeroPriorNormalDemandFlag"],
            ranked["ColdStartFlag"],
            ranked["EligibleForMainModel"],
        ],
        [
            "ZERO_HISTORY_FALLBACK",
            "COLD_START_FALLBACK",
            "MAIN_MODEL",
        ],
        default="LOW_DEMAND_FALLBACK",
    )

    ranked["PriorDemandVolumeSegment"] = np.select(
        [
            universe_total <= 0,
            ranked["ZeroPriorNormalDemandFlag"],
            ranked["PriorCumulativeDemandShareBeforePercentage"] < 80.0,
            ranked["InPrior95DemandScope"],
        ],
        [
            "NO_PRIOR_DEMAND_UNIVERSE",
            "ZERO_PRIOR_DEMAND",
            "HIGH_DEMAND",
            "MODERATE_DEMAND",
        ],
        default="LOW_DEMAND",
    )

    # Only recursively generated ranking and route fields belong in this
    # merge. ForecastRoute is a stored-source field in historical datasets
    # and does not exist on newly constructed future rows. It is created
    # later by predict_routed_system from RecursiveForecastRoute.
    ranking_columns = [
        PRODUCT_ID_COLUMN,
        "PriorDemandRank",
        "PriorDemandSharePercentage",
        "PriorCumulativeDemandShareBeforePercentage",
        "PriorCumulativeDemandSharePercentage",
        "InPrior95DemandScope",
        "EligibleForMainModel",
        "RecursiveForecastRoute",
        "PriorDemandVolumeSegment",
    ]

    missing_ranking_columns = [
        column for column in ranking_columns if column not in ranked.columns
    ]
    if missing_ranking_columns:
        raise AssertionError(
            "Recursive ranking construction is missing required columns: "
            + ", ".join(missing_ranking_columns)
        )

    frame = frame.drop(
        columns=[column for column in ranking_columns[1:] if column in frame.columns],
        errors="ignore",
    ).merge(
        ranked[ranking_columns],
        on=PRODUCT_ID_COLUMN,
        how="left",
        validate="one_to_one",
    )
    return frame

def build_hierarchy_statistics(training_frame: pd.DataFrame) -> dict:
    train = training_frame.copy()
    train[PRODUCT_ID_COLUMN] = train[PRODUCT_ID_COLUMN].astype(str)
    train["_FamilyKey"] = normalize_family(train[FAMILY_COLUMN])
    train[DAY_OF_WEEK_COLUMN] = pd.to_numeric(
        train[DAY_OF_WEEK_COLUMN], errors="raise"
    ).astype(int)
    return {
        "global_mean": float(train[TARGET_COLUMN].mean()),
        "product_weekday": train.groupby(
            [PRODUCT_ID_COLUMN, DAY_OF_WEEK_COLUMN], dropna=False
        )[TARGET_COLUMN].mean(),
        "product_mean": train.groupby(PRODUCT_ID_COLUMN, dropna=False)[
            TARGET_COLUMN
        ].mean(),
        "family_weekday": train.groupby(
            ["_FamilyKey", DAY_OF_WEEK_COLUMN], dropna=False
        )[TARGET_COLUMN].mean(),
        "family_mean": train.groupby("_FamilyKey", dropna=False)[TARGET_COLUMN].mean(),
        "global_weekday": train.groupby(DAY_OF_WEEK_COLUMN, dropna=False)[
            TARGET_COLUMN
        ].mean(),
    }

def hierarchy_predictions(
    score_frame: pd.DataFrame,
    statistics: dict,
) -> tuple[np.ndarray, np.ndarray]:
    score = score_frame.copy()
    product_ids = score[PRODUCT_ID_COLUMN].astype(str)
    families = normalize_family(score[FAMILY_COLUMN])
    weekdays = pd.to_numeric(score[DAY_OF_WEEK_COLUMN], errors="raise").astype(int)

    product_weekday_values = np.asarray(
        [
            statistics["product_weekday"].get((product_id, weekday), np.nan)
            for product_id, weekday in zip(product_ids, weekdays)
        ],
        dtype=float,
    )
    product_values = product_ids.map(statistics["product_mean"]).to_numpy(dtype=float)
    family_weekday_values = np.asarray(
        [
            statistics["family_weekday"].get((family, weekday), np.nan)
            for family, weekday in zip(families, weekdays)
        ],
        dtype=float,
    )
    family_values = families.map(statistics["family_mean"]).to_numpy(dtype=float)
    global_weekday_values = weekdays.map(statistics["global_weekday"]).to_numpy(dtype=float)
    global_values = np.full(len(score), statistics["global_mean"], dtype=float)
    zero_values = np.zeros(len(score), dtype=float)

    def coalesce(*arrays) -> np.ndarray:
        output = np.full(len(score), np.nan, dtype=float)
        for values in arrays:
            candidate = np.asarray(values, dtype=float)
            missing = ~np.isfinite(output)
            output[missing] = candidate[missing]
        return np.clip(np.nan_to_num(output, nan=0.0), 0.0, None)

    product_hierarchy = coalesce(
        product_weekday_values,
        product_values,
        family_weekday_values,
        family_values,
        global_weekday_values,
        global_values,
        zero_values,
    )
    family_hierarchy = coalesce(
        family_weekday_values,
        family_values,
        global_weekday_values,
        global_values,
        zero_values,
    )
    return product_hierarchy, family_hierarchy

def fallback_predictions(
    feature_frame: pd.DataFrame,
    hierarchy_statistics: dict,
) -> np.ndarray:
    product_hierarchy, family_hierarchy = hierarchy_predictions(
        feature_frame, hierarchy_statistics
    )

    def values(column: str) -> np.ndarray:
        return pd.to_numeric(feature_frame[column], errors="coerce").to_numpy(dtype=float)

    def coalesce(*arrays) -> np.ndarray:
        output = np.full(len(feature_frame), np.nan, dtype=float)
        for array in arrays:
            candidate = np.asarray(array, dtype=float)
            missing = ~np.isfinite(output)
            output[missing] = candidate[missing]
        return np.clip(np.nan_to_num(output, nan=0.0), 0.0, None)

    recent_median5 = coalesce(
        values("PastNormalDemandRollingMedian_5"),
        values("PastNormalDemandRollingMedian_3"),
        values("NormalDemandLag_1"),
        values("ExpandingPastMeanNormalDemand"),
        family_hierarchy,
        np.zeros(len(feature_frame)),
    )
    expanding_mean = coalesce(
        values("ExpandingPastMeanNormalDemand"),
        product_hierarchy,
        family_hierarchy,
        np.zeros(len(feature_frame)),
    )

    routes = feature_frame["RecursiveForecastRoute"].astype(str).to_numpy()
    prediction = np.full(len(feature_frame), np.nan, dtype=float)
    median_mask = np.isin(routes, ["COLD_START_FALLBACK", "LOW_DEMAND_FALLBACK"])
    zero_mask = routes == "ZERO_HISTORY_FALLBACK"
    prediction[median_mask] = recent_median5[median_mask]
    prediction[zero_mask] = expanding_mean[zero_mask]
    return prediction

def prepare_source_frame(
    frame: pd.DataFrame,
    numeric_predictors: list[str],
    categorical_predictors: list[str],
    include_product_id: bool,
) -> pd.DataFrame:
    columns = numeric_predictors + categorical_predictors + (
        [PRODUCT_ID_COLUMN] if include_product_id else []
    )
    prepared = frame[columns].copy()
    for column in numeric_predictors:
        prepared[column] = pd.to_numeric(prepared[column], errors="coerce")
    for column in categorical_predictors:
        prepared[column] = (
            prepared[column].astype("string").fillna("__MISSING__").astype(str)
        )
    if include_product_id:
        prepared[PRODUCT_ID_COLUMN] = (
            prepared[PRODUCT_ID_COLUMN]
            .astype("string")
            .fillna("__MISSING_PRODUCT__")
            .astype(str)
        )
    return prepared

def predict_base_components(
    main_features: pd.DataFrame,
    fitted_models: dict,
    numeric_predictors: list[str],
    categorical_predictors: list[str],
) -> dict[str, np.ndarray]:
    components = {
        "ROLLING_MEAN_5": np.clip(
            pd.to_numeric(
                main_features["PastNormalDemandRollingMean_5"], errors="coerce"
            ).to_numpy(dtype=float),
            0.0,
            None,
        )
    }
    transformed_cache = {}
    for component, fitted in fitted_models.items():
        cache_key = "PRODUCT" if fitted["include_product_id"] else "CORE"
        if cache_key not in transformed_cache:
            transformed_cache[cache_key] = fitted["preprocessor"].transform(
                prepare_source_frame(
                    main_features,
                    numeric_predictors,
                    categorical_predictors,
                    fitted["include_product_id"],
                )
            )
        prediction = np.asarray(
            fitted["model"].predict(transformed_cache[cache_key]), dtype=float
        )
        components[component] = np.clip(prediction, 0.0, None)
    if set(components) != set(BASE_COMPONENTS):
        raise AssertionError("The base-component prediction set is incomplete.")
    return components

def predict_routed_system(
    feature_frame: pd.DataFrame,
    operational_method: str,
    median_components: list[str],
    fitted_models: dict,
    numeric_predictors: list[str],
    categorical_predictors: list[str],
    hierarchy_statistics: dict,
) -> pd.DataFrame:
    output = feature_frame.copy().reset_index(drop=True)
    predictions = fallback_predictions(output, hierarchy_statistics)
    method_used = output["RecursiveForecastRoute"].map(
        {
            "COLD_START_FALLBACK": "RECENT_MEDIAN5_WITH_BACKOFF",
            "LOW_DEMAND_FALLBACK": "RECENT_MEDIAN5_WITH_BACKOFF",
            "ZERO_HISTORY_FALLBACK": "EXPANDING_MEAN_WITH_BACKOFF",
        }
    ).astype("string")

    for component in BASE_COMPONENTS:
        output[f"BasePrediction_{component}"] = np.nan

    main_mask = output["RecursiveForecastRoute"].astype(str) == "MAIN_MODEL"
    if main_mask.any():
        main_features = output.loc[main_mask].copy()
        components = predict_base_components(
            main_features,
            fitted_models,
            numeric_predictors,
            categorical_predictors,
        )
        for component, values in components.items():
            output.loc[main_mask, f"BasePrediction_{component}"] = values

        if operational_method == "ROLLING_MEAN_5":
            main_prediction = components["ROLLING_MEAN_5"]
        elif operational_method == "NESTED_MEDIAN_ENSEMBLE":
            unknown = sorted(set(median_components) - set(BASE_COMPONENTS))
            if unknown:
                raise AssertionError(
                    f"Unknown median components in ND08 contract: {unknown}"
                )
            matrix = np.column_stack(
                [components[component] for component in median_components]
            )
            main_prediction = np.median(matrix, axis=1)
        else:
            raise ValueError(f"Unsupported operational method: {operational_method}")

        predictions[main_mask.to_numpy()] = np.clip(main_prediction, 0.0, None)
        method_used.loc[main_mask] = operational_method

    if not np.isfinite(predictions).all():
        raise AssertionError("The routed forecast contains non-finite predictions.")
    if (predictions < 0).any():
        raise AssertionError("The routed forecast contains negative predictions.")

    output["ForecastRoute"] = output["RecursiveForecastRoute"].astype(str)
    output["PredictedNormalDemand"] = predictions
    output["MethodUsed"] = method_used.astype(str)
    return output

def build_future_base_rows(
    active_catalogue: pd.DataFrame,
    calendar_row: pd.Series,
) -> pd.DataFrame:
    frame = active_catalogue.loc[active_catalogue["IncludeInForecast"]].copy()
    frame = frame.reset_index(drop=True)
    current_sequence = int(calendar_row[SEQUENCE_COLUMN])

    for column in [
        DATE_COLUMN,
        SEQUENCE_COLUMN,
        "Year",
        "Month",
        "Quarter",
        "DayOfWeekNumber",
        "ISOYear",
        "ISOWeek",
        "DayOfYear",
        "IsWeekend",
        "DaysSincePreviousOperatingDate",
        "IsConsecutiveCalendarDay",
        "CalendarPolicy",
    ]:
        frame[column] = calendar_row[column]

    frame["ProductAgeOperatingDays"] = (
        current_sequence
        - pd.to_numeric(
            frame["FirstObservedOperatingDaySequence"], errors="raise"
        ).astype(int)
    ).clip(lower=0)
    frame[TARGET_COLUMN] = np.nan
    return frame

def run_recursive_path(
    label: str,
    future_calendar: pd.DataFrame,
    requested_dates: set[pd.Timestamp],
    operational_method: str,
    base_state: dict[str, dict[str, list]],
    active_catalogue: pd.DataFrame,
    fitted_models: dict,
    numeric_predictors: list[str],
    categorical_predictors: list[str],
    hierarchy_statistics: dict,
    median_components: list[str],
    cutoff_date: pd.Timestamp,
) -> tuple[pd.DataFrame, list[dict]]:
    state = copy_history_state(base_state)
    parts = []
    audit_records = []

    for horizon, calendar_row in enumerate(
        future_calendar.sort_values(DATE_COLUMN).itertuples(index=False), start=1
    ):
        calendar_series = pd.Series(calendar_row._asdict())
        base_rows = build_future_base_rows(active_catalogue, calendar_series)
        feature_rows = build_recursive_feature_frame(base_rows, state)
        forecast_rows = predict_routed_system(
            feature_rows,
            operational_method,
            median_components,
            fitted_models,
            numeric_predictors,
            categorical_predictors,
            hierarchy_statistics,
        )
        forecast_date = pd.Timestamp(calendar_series[DATE_COLUMN]).normalize()
        forecast_rows["ForecastPath"] = label
        forecast_rows["ForecastOriginCutoffDate"] = cutoff_date
        forecast_rows["HorizonOperatingDays"] = horizon
        forecast_rows["OperationalMethod"] = operational_method
        forecast_rows["IsRequestedOutputDate"] = forecast_date in requested_dates
        forecast_rows["FutureActualDemandUsed"] = False
        parts.append(forecast_rows)

        route_counts = forecast_rows["RecursiveForecastRoute"].value_counts()
        audit_records.append(
            {
                "ForecastPath": label,
                "Date": forecast_date,
                "HorizonOperatingDays": horizon,
                "OperationalMethod": operational_method,
                "ProductsForecast": int(len(forecast_rows)),
                "RequestedOutputDate": forecast_date in requested_dates,
                "RestaurantNormalDemandForecast": float(
                    forecast_rows["PredictedNormalDemand"].sum()
                ),
                "MainModelRows": int(route_counts.get("MAIN_MODEL", 0)),
                "LowDemandFallbackRows": int(
                    route_counts.get("LOW_DEMAND_FALLBACK", 0)
                ),
                "ColdStartFallbackRows": int(
                    route_counts.get("COLD_START_FALLBACK", 0)
                ),
                "ZeroHistoryFallbackRows": int(
                    route_counts.get("ZERO_HISTORY_FALLBACK", 0)
                ),
                "PredictionsInsertedBeforeNextDay": True,
            }
        )

        append_predictions_to_state(
            state,
            forecast_rows,
            forecast_rows["PredictedNormalDemand"].to_numpy(dtype=float),
        )

    result = pd.concat(parts, ignore_index=True)
    return result, audit_records

def build_manifest(root: Path, exclude_control: bool = True) -> pd.DataFrame:
    records = []
    for path in sorted(root.rglob("*")):
        if not path.is_file():
            continue
        relative = path.relative_to(root)
        if exclude_control and relative.parts and relative.parts[0] == "07_control":
            continue
        records.append(
            {
                "RelativePath": str(relative),
                "Bytes": int(path.stat().st_size),
                "SHA256": sha256_file(path),
            }
        )
    return pd.DataFrame(records)


def build_current_active_catalogue(
    history: pd.DataFrame,
    last20_dates: list[pd.Timestamp],
    apply_user_filter: bool = True,
) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    history = history.copy()
    history[PRODUCT_ID_COLUMN] = history[PRODUCT_ID_COLUMN].astype(str)
    cutoff_date = pd.Timestamp(history[DATE_COLUMN].max()).normalize()

    cutoff_rows = history.loc[history[DATE_COLUMN] == cutoff_date].copy()
    if cutoff_rows[PRODUCT_ID_COLUMN].duplicated().any():
        raise AssertionError("Cutoff panel contains duplicate product rows.")
    active_ids = set(cutoff_rows[PRODUCT_ID_COLUMN].astype(str))

    ordered = history.sort_values(
        [PRODUCT_ID_COLUMN, DATE_COLUMN, SEQUENCE_COLUMN], kind="mergesort"
    )
    first_sequence = ordered.groupby(PRODUCT_ID_COLUMN)[SEQUENCE_COLUMN].min()
    first_date = ordered.groupby(PRODUCT_ID_COLUMN)[DATE_COLUMN].min()
    cumulative = ordered.groupby(PRODUCT_ID_COLUMN)[TARGET_COLUMN].sum()
    positive_days = ordered.groupby(PRODUCT_ID_COLUMN)[TARGET_COLUMN].apply(
        lambda values: int((values > 0).sum())
    )
    last_date = ordered.groupby(PRODUCT_ID_COLUMN)[DATE_COLUMN].max()

    recent = history.loc[history[DATE_COLUMN].isin(last20_dates)].copy()
    recent_presence = (
        recent.groupby([PRODUCT_ID_COLUMN, PRODUCT_NAME_COLUMN], as_index=False)
        .agg(
            DaysPresentLast20=(DATE_COLUMN, "nunique"),
            FirstPresenceLast20=(DATE_COLUMN, "min"),
            LastPresenceLast20=(DATE_COLUMN, "max"),
            NormalDemandLast20=(TARGET_COLUMN, "sum"),
            PositiveDemandDaysLast20=(TARGET_COLUMN, lambda values: int((values > 0).sum())),
        )
    )
    recent_presence["ActiveAtCutoff"] = (
        recent_presence["LastPresenceLast20"].dt.normalize() == cutoff_date
    )
    recent_presence["PresentAll20"] = recent_presence["DaysPresentLast20"] == len(last20_dates)
    recent_presence["RecentActiveAddition"] = (
        recent_presence["ActiveAtCutoff"] & ~recent_presence["PresentAll20"]
    )
    recent_presence["RetiredBeforeCutoff"] = ~recent_presence["ActiveAtCutoff"]
    recent_presence["CatalogueDecision"] = np.where(
        recent_presence["ActiveAtCutoff"],
        "INCLUDE_LATEST_ACTIVE_PANEL",
        "EXCLUDE_NOT_ACTIVE_AT_CUTOFF",
    )
    recent_presence = recent_presence.sort_values(
        ["ActiveAtCutoff", "DaysPresentLast20", "NormalDemandLast20", PRODUCT_ID_COLUMN],
        ascending=[False, False, False, True],
        kind="mergesort",
    ).reset_index(drop=True)

    catalogue = cutoff_rows[PRODUCT_METADATA_COLUMNS].copy()
    catalogue[PRODUCT_ID_COLUMN] = catalogue[PRODUCT_ID_COLUMN].astype(str)
    catalogue["FirstObservedOperatingDaySequence"] = catalogue[PRODUCT_ID_COLUMN].map(first_sequence)
    catalogue["FirstPanelDate"] = catalogue[PRODUCT_ID_COLUMN].map(first_date)
    catalogue["HistoricalNormalDemand"] = catalogue[PRODUCT_ID_COLUMN].map(cumulative)
    catalogue["HistoricalPositiveDemandDays"] = catalogue[PRODUCT_ID_COLUMN].map(positive_days)
    catalogue["LatestKnownActivePanelDate"] = cutoff_date
    catalogue["CataloguePolicy"] = "LATEST_KNOWN_ACTIVE_PANEL"
    recent_day_map = recent_presence.set_index(PRODUCT_ID_COLUMN)["DaysPresentLast20"]
    recent_all20_map = recent_presence.set_index(PRODUCT_ID_COLUMN)["PresentAll20"]
    recent_add_map = recent_presence.set_index(PRODUCT_ID_COLUMN)["RecentActiveAddition"]
    catalogue["DaysPresentLast20"] = catalogue[PRODUCT_ID_COLUMN].map(recent_day_map).fillna(0).astype(int)
    catalogue["PresentAll20"] = catalogue[PRODUCT_ID_COLUMN].map(recent_all20_map).fillna(False).astype(bool)
    catalogue["RecentActiveAddition"] = catalogue[PRODUCT_ID_COLUMN].map(recent_add_map).fillna(False).astype(bool)
    catalogue["IncludeInForecast"] = True

    if apply_user_filter and ACTIVE_PRODUCT_IDS is not None:
        requested = {str(value) for value in ACTIVE_PRODUCT_IDS}
        unknown = sorted(requested - active_ids)
        if unknown:
            raise ValueError(
                "ACTIVE_PRODUCT_IDS contains products outside the latest active panel:\n"
                + "\n".join(f"- {value}" for value in unknown)
            )
        catalogue["IncludeInForecast"] = catalogue[PRODUCT_ID_COLUMN].isin(requested)

    catalogue = catalogue.sort_values(
        ["IncludeInForecast", "HistoricalNormalDemand", PRODUCT_ID_COLUMN],
        ascending=[False, False, True],
        kind="mergesort",
    ).reset_index(drop=True)

    latest_metadata = ordered.groupby(PRODUCT_ID_COLUMN, sort=False).tail(1)[
        [PRODUCT_ID_COLUMN, PRODUCT_NAME_COLUMN]
    ].copy()
    historical_scope = latest_metadata.copy()
    historical_scope[PRODUCT_ID_COLUMN] = historical_scope[PRODUCT_ID_COLUMN].astype(str)
    historical_scope["LastObservedPanelDate"] = historical_scope[PRODUCT_ID_COLUMN].map(last_date)
    historical_scope["HistoricalNormalDemand"] = historical_scope[PRODUCT_ID_COLUMN].map(cumulative)
    historical_scope["ActiveAtCutoff"] = historical_scope[PRODUCT_ID_COLUMN].isin(active_ids)
    historical_scope["CatalogueDecision"] = np.where(
        historical_scope["ActiveAtCutoff"],
        "INCLUDE_LATEST_ACTIVE_PANEL",
        "EXCLUDE_HISTORICAL_NOT_CURRENT",
    )
    historical_scope = historical_scope.sort_values(
        ["ActiveAtCutoff", "LastObservedPanelDate", "HistoricalNormalDemand"],
        ascending=[False, False, False],
        kind="mergesort",
    ).reset_index(drop=True)

    if not catalogue["IncludeInForecast"].any():
        raise AssertionError("The corrected active product catalogue is empty.")

    return catalogue, recent_presence, historical_scope


def load_frozen_models(
    model_metadata: dict,
) -> dict:
    core_preprocessor = joblib.load(FROZEN_CORE_PREPROCESSOR_PATH)
    product_preprocessor = joblib.load(FROZEN_PRODUCT_PREPROCESSOR_PATH)

    cat_model = CatBoostRegressor()
    cat_model.load_model(str(FROZEN_CATBOOST_MODEL_PATH))

    xgb_core = XGBRegressor()
    xgb_core.load_model(str(FROZEN_XGBOOST_CORE_MODEL_PATH))

    xgb_product = XGBRegressor()
    xgb_product.load_model(str(FROZEN_XGBOOST_PRODUCT_MODEL_PATH))

    return {
        "CATBOOST_CORE53": {
            "model": cat_model,
            "preprocessor": core_preprocessor,
            "include_product_id": False,
        },
        "XGBOOST_CORE53": {
            "model": xgb_core,
            "preprocessor": core_preprocessor,
            "include_product_id": False,
        },
        "XGBOOST_PRODUCT_AWARE": {
            "model": xgb_product,
            "preprocessor": product_preprocessor,
            "include_product_id": True,
        },
    }



# =============================================================================
# PREFLIGHT
# =============================================================================

required_inputs = [
    ALL_ROUTES_PATH,
    CORE_PREDICTOR_LIST_PATH,
    ND03_CHECKPOINT_PATH,
    FROZEN_CORE_PREPROCESSOR_PATH,
    FROZEN_PRODUCT_PREPROCESSOR_PATH,
    FROZEN_CATBOOST_MODEL_PATH,
    FROZEN_XGBOOST_CORE_MODEL_PATH,
    FROZEN_XGBOOST_PRODUCT_MODEL_PATH,
    FROZEN_MODEL_METADATA_PATH,
    FROZEN_INFERENCE_CONTRACT_PATH,
    FROZEN_ACTIVE_CATALOGUE_PATH,
    FROZEN_DAILY_PRODUCT_PATH,
    FROZEN_WEEK_DAILY_PRODUCT_PATH,
    FROZEN_ND09_CHECKPOINT_PATH,
    FROZEN_ND09_LOCK_PATH,
]
missing = [path for path in required_inputs if not path.is_file()]
if missing:
    raise FileNotFoundError(
        "ND09A required inputs are missing:\n"
        + "\n".join(f"- {path}" for path in missing)
    )

actual_nd03_hash = sha256_file(ND03_CHECKPOINT_PATH)
if actual_nd03_hash != EXPECTED_ND03_CHECKPOINT_SHA256:
    raise AssertionError(
        f"ND03 checkpoint mismatch. Expected {EXPECTED_ND03_CHECKPOINT_SHA256}, "
        f"found {actual_nd03_hash}."
    )

actual_frozen_nd09_hash = sha256_file(FROZEN_ND09_CHECKPOINT_PATH)
if actual_frozen_nd09_hash != EXPECTED_FROZEN_ND09_CHECKPOINT_SHA256:
    raise AssertionError(
        f"Frozen ND09 checkpoint mismatch. Expected {EXPECTED_FROZEN_ND09_CHECKPOINT_SHA256}, "
        f"found {actual_frozen_nd09_hash}."
    )

if ND09A_ROOT.exists():
    if not ALLOW_OVERWRITE:
        raise FileExistsError(
            "ND09A output already exists. No files were changed:\n"
            f"{ND09A_ROOT}"
        )
    shutil.rmtree(ND09A_ROOT)

for path in [TOP_LEVEL_CHECKPOINT_PATH, TOP_LEVEL_CHECKPOINT_SHA_PATH, TOP_LEVEL_LOCK_PATH]:
    if path.exists():
        if not ALLOW_OVERWRITE:
            raise FileExistsError(
                "An ND09A top-level control artifact already exists. "
                "No files were changed:\n" + str(path)
            )
        path.unlink()

protected_hashes_before = {str(path): sha256_file(path) for path in required_inputs}
STAGING_ROOT = ND09A_ROOT.parent / f".ND09A_staging_{uuid.uuid4().hex}"
STAGING_ROOT.mkdir(parents=True, exist_ok=False)


# =============================================================================
# MAIN EXECUTION
# =============================================================================

try:
    all_routes = pd.read_csv(ALL_ROUTES_PATH, low_memory=False)
    predictor_list = pd.read_csv(CORE_PREDICTOR_LIST_PATH, low_memory=False)
    frozen_contract = json.loads(FROZEN_INFERENCE_CONTRACT_PATH.read_text(encoding="utf-8"))
    model_metadata = json.loads(FROZEN_MODEL_METADATA_PATH.read_text(encoding="utf-8"))
    frozen_checkpoint = json.loads(FROZEN_ND09_CHECKPOINT_PATH.read_text(encoding="utf-8"))
    frozen_lock = json.loads(FROZEN_ND09_LOCK_PATH.read_text(encoding="utf-8"))
    old_active_catalogue = pd.read_csv(FROZEN_ACTIVE_CATALOGUE_PATH, low_memory=False)
    old_daily_forecast = pd.read_csv(FROZEN_DAILY_PRODUCT_PATH, low_memory=False)
    old_week_forecast = pd.read_csv(FROZEN_WEEK_DAILY_PRODUCT_PATH, low_memory=False)

    all_routes[DATE_COLUMN] = pd.to_datetime(all_routes[DATE_COLUMN], errors="raise")
    all_routes[PRODUCT_ID_COLUMN] = all_routes[PRODUCT_ID_COLUMN].astype(str)
    old_daily_forecast[DATE_COLUMN] = pd.to_datetime(old_daily_forecast[DATE_COLUMN], errors="raise")
    old_week_forecast[DATE_COLUMN] = pd.to_datetime(old_week_forecast[DATE_COLUMN], errors="raise")

    validate_required_columns(
        all_routes,
        {
            DATE_COLUMN,
            PRODUCT_ID_COLUMN,
            PRODUCT_NAME_COLUMN,
            TARGET_COLUMN,
            ROUTE_COLUMN,
            FAMILY_COLUMN,
            SEQUENCE_COLUMN,
            "ProductFirstObservedDate",
            *BASE_NUMERIC_PREDICTORS,
            *BASE_CATEGORICAL_PREDICTORS,
            *HISTORICAL_DEMAND_PREDICTORS,
        },
        "ND03 pre-March all-route development dataset",
    )

    if len(all_routes) != EXPECTED_ALL_ROUTE_ROWS:
        raise AssertionError(
            f"Expected {EXPECTED_ALL_ROUTE_ROWS:,} rows, found {len(all_routes):,}."
        )
    if all_routes[PRODUCT_ID_COLUMN].nunique() != EXPECTED_HISTORICAL_PRODUCTS:
        raise AssertionError(
            f"Expected {EXPECTED_HISTORICAL_PRODUCTS} historical products, found "
            f"{all_routes[PRODUCT_ID_COLUMN].nunique()}."
        )
    if not (all_routes[DATE_COLUMN] < DEVELOPMENT_END_EXCLUSIVE).all():
        raise AssertionError("March or later rows were found in the ND09A history source.")

    cutoff_date = pd.Timestamp(all_routes[DATE_COLUMN].max()).normalize()
    cutoff_sequence = int(
        all_routes.loc[all_routes[DATE_COLUMN] == cutoff_date, SEQUENCE_COLUMN].max()
    )
    operating_dates = [pd.Timestamp(value).normalize() for value in sorted(all_routes[DATE_COLUMN].unique())]
    if len(operating_dates) < 20:
        raise AssertionError("Fewer than 20 pre-March operating dates are available.")
    last20_dates = operating_dates[-20:]

    active_catalogue, last20_audit, historical_scope_audit = build_current_active_catalogue(
        all_routes, last20_dates, apply_user_filter=True
    )

    base_active_count = int(
        all_routes.loc[all_routes[DATE_COLUMN] == cutoff_date, PRODUCT_ID_COLUMN].nunique()
    )
    if base_active_count != EXPECTED_CURRENT_ACTIVE_PRODUCTS:
        raise AssertionError(
            f"Expected {EXPECTED_CURRENT_ACTIVE_PRODUCTS} products in the latest panel, "
            f"found {base_active_count}."
        )
    if last20_audit[PRODUCT_ID_COLUMN].nunique() != EXPECTED_LAST20_UNIQUE_PRODUCTS:
        raise AssertionError("The last-20 unique-product count is not 90.")
    stable_count = int(last20_audit["PresentAll20"].sum())
    recent_active_count = int(last20_audit["RecentActiveAddition"].sum())
    retired_count = int(last20_audit["RetiredBeforeCutoff"].sum())
    if stable_count != EXPECTED_LAST20_STABLE_PRODUCTS:
        raise AssertionError(f"Expected 86 stable last-20 products, found {stable_count}.")
    if recent_active_count != EXPECTED_LAST20_RECENT_ACTIVE_ADDITIONS:
        raise AssertionError(f"Expected 2 recent active additions, found {recent_active_count}.")
    if retired_count != EXPECTED_LAST20_RETIRED_PRODUCTS:
        raise AssertionError(f"Expected 2 last-20 retirements, found {retired_count}.")

    active_count = int(active_catalogue["IncludeInForecast"].sum())
    if ACTIVE_PRODUCT_IDS is None and active_count != EXPECTED_CURRENT_ACTIVE_PRODUCTS:
        raise AssertionError(
            f"Default corrected forecast catalogue must contain 88 products; found {active_count}."
        )

    core_predictors = list(model_metadata["CorePredictors"])
    numeric_predictors = list(model_metadata["NumericPredictors"])
    categorical_predictors = list(model_metadata["CategoricalPredictors"])
    if numeric_predictors != BASE_NUMERIC_PREDICTORS + HISTORICAL_DEMAND_PREDICTORS:
        raise AssertionError("Frozen ND09 numeric predictor architecture does not match ND03.")
    if categorical_predictors != BASE_CATEGORICAL_PREDICTORS:
        raise AssertionError("Frozen ND09 categorical predictor architecture does not match ND03.")

    daily_product_method = str(
        frozen_contract["DailyForecastPolicy"]["OneOperatingDayAheadMethod"]
    )
    median_components = list(
        frozen_contract["DailyForecastPolicy"]["RepresentativeMedianComponents"]
    )
    week_planning_method = str(frozen_contract["WeekForecastPolicy"]["Method"])
    updated_week_method = str(frozen_contract["DailyUpdatedWeekPolicy"]["Method"])
    if daily_product_method != "NESTED_MEDIAN_ENSEMBLE":
        raise AssertionError("Frozen next-day method is not the selected median ensemble.")
    if week_planning_method != "ROLLING_MEAN_5":
        raise AssertionError("Frozen week-start method is not ROLLING_MEAN_5.")
    if set(median_components) - set(BASE_COMPONENTS):
        raise AssertionError("Frozen median component list is invalid.")

    closed_dates = parse_iso_dates(
        frozen_contract["CalendarPolicy"].get("ClosedDates", []), "closure"
    )
    extra_dates = parse_iso_dates(
        frozen_contract["CalendarPolicy"].get("ExtraOperatingDates", []), "extra operating"
    )

    demo = frozen_contract["DefaultDemonstrationForecasts"]
    requested_daily_date = pd.Timestamp(demo["ArbitraryDate"]).normalize()
    requested_week_start = pd.Timestamp(demo["WeekStart"]).normalize()
    requested_week_end = pd.Timestamp(demo["WeekEnd"]).normalize()
    maximum_requested_date = max(requested_daily_date, requested_week_end)

    future_calendar = generate_future_calendar(
        cutoff_date,
        cutoff_sequence,
        maximum_requested_date,
        closed_dates,
        extra_dates,
    )
    daily_calendar = future_calendar.loc[
        future_calendar[DATE_COLUMN] <= requested_daily_date
    ].copy()
    week_calendar = future_calendar.loc[
        future_calendar[DATE_COLUMN] <= requested_week_end
    ].copy()
    daily_horizon = int(len(daily_calendar))
    week_horizon = int(len(week_calendar))
    maximum_validated_horizon = int(
        frozen_contract["HorizonPolicy"]["ValidatedRecursiveOperatingDays"]
    )
    if max(daily_horizon, week_horizon) > maximum_validated_horizon:
        raise AssertionError(
            "The frozen demonstration dates exceed the ND08 validated recursive horizon."
        )

    daily_operational_method = (
        daily_product_method if daily_horizon == 1 else week_planning_method
    )
    week_requested_dates = {
        date
        for date in pd.date_range(requested_week_start, requested_week_end, freq="D")
        if is_operating_date(date, closed_dates, extra_dates)
    }

    # Replay the latest historical date under the corrected catalogue policy.
    replay_date = cutoff_date
    replay_history = all_routes.loc[all_routes[DATE_COLUMN] < replay_date].copy()
    replay_actual = all_routes.loc[all_routes[DATE_COLUMN] == replay_date].copy()
    replay_dates = [
        pd.Timestamp(value).normalize()
        for value in sorted(replay_history[DATE_COLUMN].unique())
    ][-20:]
    replay_catalogue, _, _ = build_current_active_catalogue(
        replay_history, replay_dates, apply_user_filter=False
    )
    replay_actual_ids = set(replay_actual[PRODUCT_ID_COLUMN].astype(str))
    replay_catalogue["IncludeInForecast"] = replay_catalogue[PRODUCT_ID_COLUMN].isin(
        replay_actual_ids
    )
    if set(replay_catalogue.loc[replay_catalogue["IncludeInForecast"], PRODUCT_ID_COLUMN]) != replay_actual_ids:
        raise AssertionError(
            "Latest-date replay cannot reproduce the exact cutoff product panel from prior history."
        )
    replay_cutoff = pd.Timestamp(replay_history[DATE_COLUMN].max()).normalize()
    replay_sequence = int(replay_history[SEQUENCE_COLUMN].max())
    replay_calendar = generate_future_calendar(
        replay_cutoff, replay_sequence, replay_date, set(), set()
    )
    replay_rows = build_future_base_rows(replay_catalogue, replay_calendar.iloc[-1])
    replay_features = build_recursive_feature_frame(
        replay_rows, build_history_state(replay_history)
    )
    replay_compare = replay_actual.merge(
        replay_features,
        on=[DATE_COLUMN, PRODUCT_ID_COLUMN],
        suffixes=("_Stored", "_Rebuilt"),
        how="inner",
        validate="one_to_one",
    )

    replay_records = []
    for column in [*BASE_NUMERIC_PREDICTORS, *HISTORICAL_DEMAND_PREDICTORS]:
        stored = pd.to_numeric(
            replay_compare[f"{column}_Stored"], errors="coerce"
        ).to_numpy(dtype=float)
        rebuilt = pd.to_numeric(
            replay_compare[f"{column}_Rebuilt"], errors="coerce"
        ).to_numpy(dtype=float)
        both_nan = np.isnan(stored) & np.isnan(rebuilt)
        passed = bool(
            np.all(both_nan | np.isclose(stored, rebuilt, atol=1e-10, rtol=1e-10))
        )
        difference = np.abs(stored - rebuilt)
        difference[both_nan] = 0.0
        replay_records.append(
            {
                "ReplayDate": replay_date,
                "Field": column,
                "RowsCompared": int(len(replay_compare)),
                "MaximumAbsoluteDifference": float(np.nanmax(difference))
                if len(difference)
                else 0.0,
                "Passed": passed,
            }
        )
    route_passed = bool(
        (
            replay_compare[ROUTE_COLUMN].astype(str).to_numpy()
            == replay_compare["RecursiveForecastRoute"].astype(str).to_numpy()
        ).all()
    )
    replay_records.append(
        {
            "ReplayDate": replay_date,
            "Field": "ForecastRoute",
            "RowsCompared": int(len(replay_compare)),
            "MaximumAbsoluteDifference": np.nan,
            "Passed": route_passed,
        }
    )
    feature_replay_audit = pd.DataFrame(replay_records)
    if not feature_replay_audit["Passed"].all():
        raise AssertionError(
            "ND09A feature/route replay failed:\n"
            + feature_replay_audit.loc[~feature_replay_audit["Passed"]].to_string(index=False)
        )

    # Load the already frozen model artifacts. No fitting occurs in ND09A.
    load_start = time.perf_counter()
    fitted_models = load_frozen_models(model_metadata)
    model_load_seconds = float(time.perf_counter() - load_start)

    hierarchy_statistics = build_hierarchy_statistics(all_routes)
    base_state = build_history_state(all_routes)

    daily_path, daily_path_audit = run_recursive_path(
        label="ARBITRARY_DATE_ACTIVE_MENU_CORRECTED",
        future_calendar=daily_calendar,
        requested_dates={requested_daily_date},
        operational_method=daily_operational_method,
        base_state=base_state,
        active_catalogue=active_catalogue,
        fitted_models=fitted_models,
        numeric_predictors=numeric_predictors,
        categorical_predictors=categorical_predictors,
        hierarchy_statistics=hierarchy_statistics,
        median_components=median_components,
        cutoff_date=cutoff_date,
    )

    week_path, week_path_audit = run_recursive_path(
        label="MONDAY_ORIGIN_WEEK_ACTIVE_MENU_CORRECTED",
        future_calendar=week_calendar,
        requested_dates=week_requested_dates,
        operational_method=week_planning_method,
        base_state=base_state,
        active_catalogue=active_catalogue,
        fitted_models=fitted_models,
        numeric_predictors=numeric_predictors,
        categorical_predictors=categorical_predictors,
        hierarchy_statistics=hierarchy_statistics,
        median_components=median_components,
        cutoff_date=cutoff_date,
    )

    daily_product_forecast = daily_path.loc[
        daily_path[DATE_COLUMN] == requested_daily_date
    ].copy()
    daily_product_forecast["CataloguePolicy"] = "LATEST_KNOWN_ACTIVE_PANEL"
    daily_product_forecast["ForecastUseCase"] = "NEXT_OPERATING_DAY"

    daily_restaurant_forecast = pd.DataFrame(
        [
            {
                "Date": requested_daily_date,
                "OperationalMethod": daily_operational_method,
                "CataloguePolicy": "LATEST_KNOWN_ACTIVE_PANEL",
                "ProductsForecast": int(len(daily_product_forecast)),
                "PredictedRestaurantNormalDemand": float(
                    daily_product_forecast["PredictedNormalDemand"].sum()
                ),
                "ForecastOriginCutoffDate": cutoff_date,
                "HorizonOperatingDays": daily_horizon,
            }
        ]
    )

    week_daily_product_forecast = week_path.loc[
        week_path[DATE_COLUMN].isin(week_requested_dates)
    ].copy()
    week_daily_product_forecast["WeekStart"] = requested_week_start
    week_daily_product_forecast["WeekEnd"] = requested_week_end
    week_daily_product_forecast["CataloguePolicy"] = "LATEST_KNOWN_ACTIVE_PANEL"

    week_product_totals = (
        week_daily_product_forecast.groupby(
            [PRODUCT_ID_COLUMN, PRODUCT_NAME_COLUMN, FAMILY_COLUMN],
            dropna=False,
            as_index=False,
        )["PredictedNormalDemand"]
        .sum()
        .rename(columns={"PredictedNormalDemand": "PredictedWeekNormalDemand"})
    )
    week_product_totals["WeekStart"] = requested_week_start
    week_product_totals["WeekEnd"] = requested_week_end
    week_product_totals["OperationalMethod"] = week_planning_method
    week_product_totals["CataloguePolicy"] = "LATEST_KNOWN_ACTIVE_PANEL"
    week_product_totals = week_product_totals.sort_values(
        ["PredictedWeekNormalDemand", PRODUCT_ID_COLUMN],
        ascending=[False, True],
        kind="mergesort",
    ).reset_index(drop=True)

    week_restaurant_daily = (
        week_daily_product_forecast.groupby(DATE_COLUMN, as_index=False)[
            "PredictedNormalDemand"
        ]
        .sum()
        .rename(columns={"PredictedNormalDemand": "PredictedRestaurantNormalDemand"})
    )
    week_restaurant_daily["WeekStart"] = requested_week_start
    week_restaurant_daily["OperationalMethod"] = week_planning_method
    week_restaurant_daily["CataloguePolicy"] = "LATEST_KNOWN_ACTIVE_PANEL"

    week_restaurant_total = pd.DataFrame(
        [
            {
                "WeekStart": requested_week_start,
                "WeekEnd": requested_week_end,
                "OperatingDatesForecast": int(
                    week_daily_product_forecast[DATE_COLUMN].nunique()
                ),
                "ProductsForecast": active_count,
                "OperationalMethod": week_planning_method,
                "CataloguePolicy": "LATEST_KNOWN_ACTIVE_PANEL",
                "PredictedRestaurantWeekNormalDemand": float(
                    week_daily_product_forecast["PredictedNormalDemand"].sum()
                ),
                "ForecastOriginCutoffDate": cutoff_date,
                "RecursivePathOperatingDays": week_horizon,
            }
        ]
    )

    old_daily_total = float(old_daily_forecast["PredictedNormalDemand"].sum())
    old_week_total = float(old_week_forecast["PredictedNormalDemand"].sum())
    corrected_daily_total = float(daily_product_forecast["PredictedNormalDemand"].sum())
    corrected_week_total = float(week_daily_product_forecast["PredictedNormalDemand"].sum())

    scope_comparison = pd.DataFrame(
        [
            {
                "Comparison": "Historical product universe",
                "OldValue": int(old_active_catalogue["CanonicalProductID"].nunique()),
                "CorrectedValue": base_active_count,
                "Difference": base_active_count
                - int(old_active_catalogue["CanonicalProductID"].nunique()),
                "Unit": "products",
            },
            {
                "Comparison": "Daily products forecast",
                "OldValue": int(old_daily_forecast[PRODUCT_ID_COLUMN].nunique()),
                "CorrectedValue": int(daily_product_forecast[PRODUCT_ID_COLUMN].nunique()),
                "Difference": int(daily_product_forecast[PRODUCT_ID_COLUMN].nunique())
                - int(old_daily_forecast[PRODUCT_ID_COLUMN].nunique()),
                "Unit": "products",
            },
            {
                "Comparison": "Daily restaurant normal-demand forecast",
                "OldValue": old_daily_total,
                "CorrectedValue": corrected_daily_total,
                "Difference": corrected_daily_total - old_daily_total,
                "Unit": "units",
            },
            {
                "Comparison": "Week restaurant normal-demand forecast",
                "OldValue": old_week_total,
                "CorrectedValue": corrected_week_total,
                "Difference": corrected_week_total - old_week_total,
                "Unit": "units",
            },
        ]
    )

    forecast_path_audit = pd.DataFrame(daily_path_audit + week_path_audit)
    route_summary = (
        pd.concat(
            [
                daily_product_forecast.assign(Output="ARBITRARY_DATE"),
                week_daily_product_forecast.assign(Output="MONDAY_ORIGIN_WEEK"),
            ],
            ignore_index=True,
        )
        .groupby(["Output", "RecursiveForecastRoute", "MethodUsed"], as_index=False)
        .agg(
            Rows=(PRODUCT_ID_COLUMN, "size"),
            PredictedNormalDemand=("PredictedNormalDemand", "sum"),
        )
        .sort_values(["Output", "RecursiveForecastRoute", "MethodUsed"])
        .reset_index(drop=True)
    )

    # -------------------------------------------------------------------------
    # Staged outputs
    # -------------------------------------------------------------------------
    staged_forecast_dir = STAGING_ROOT / "01_forecasts"
    staged_contract_dir = STAGING_ROOT / "02_contracts"
    staged_audit_dir = STAGING_ROOT / "03_audits"
    staged_report_dir = STAGING_ROOT / "04_reports"
    staged_control_dir = STAGING_ROOT / "07_control"
    for directory in [
        staged_forecast_dir,
        staged_contract_dir,
        staged_audit_dir,
        staged_report_dir,
        staged_control_dir,
    ]:
        directory.mkdir(parents=True, exist_ok=True)

    forecast_keep_columns = [
        DATE_COLUMN,
        PRODUCT_ID_COLUMN,
        PRODUCT_NAME_COLUMN,
        FAMILY_COLUMN,
        SEQUENCE_COLUMN,
        "RecursiveForecastRoute",
        "ForecastRoute",
        "PriorDemandVolumeSegment",
        "MethodUsed",
        "OperationalMethod",
        "PredictedNormalDemand",
        "ForecastPath",
        "ForecastOriginCutoffDate",
        "HorizonOperatingDays",
        "IsRequestedOutputDate",
        "FutureActualDemandUsed",
        *[f"BasePrediction_{component}" for component in BASE_COMPONENTS],
    ]

    write_csv(
        staged_forecast_dir / DAILY_PATH_FORECAST_PATH.name,
        daily_path[forecast_keep_columns].copy(),
    )
    write_csv(
        staged_forecast_dir / DAILY_PRODUCT_FORECAST_PATH.name,
        daily_product_forecast,
    )
    write_csv(
        staged_forecast_dir / DAILY_RESTAURANT_FORECAST_PATH.name,
        daily_restaurant_forecast,
    )
    write_csv(
        staged_forecast_dir / WEEK_PATH_FORECAST_PATH.name,
        week_path[forecast_keep_columns].copy(),
    )
    write_csv(
        staged_forecast_dir / WEEK_DAILY_PRODUCT_FORECAST_PATH.name,
        week_daily_product_forecast,
    )
    write_csv(
        staged_forecast_dir / WEEK_PRODUCT_TOTAL_PATH.name,
        week_product_totals,
    )
    write_csv(
        staged_forecast_dir / WEEK_RESTAURANT_DAILY_PATH.name,
        week_restaurant_daily,
    )
    write_csv(
        staged_forecast_dir / WEEK_RESTAURANT_TOTAL_PATH.name,
        week_restaurant_total,
    )

    write_csv(staged_contract_dir / ACTIVE_CATALOGUE_PATH.name, active_catalogue)

    corrected_contract = {
        "StepID": STEP_ID,
        "Status": STATUS,
        "CorrectionType": "INFERENCE_CATALOGUE_SCOPE_ONLY",
        "FrozenModelSource": str(FROZEN_ND09_ROOT),
        "FrozenND09CheckpointSHA256": actual_frozen_nd09_hash,
        "ForecastingModelsRefitted": False,
        "ForecastingMethodsReselected": False,
        "Target": TARGET_COLUMN,
        "DataCutoff": cutoff_date,
        "HistoricalProductUniverse": EXPECTED_HISTORICAL_PRODUCTS,
        "LatestActivePanelProducts": base_active_count,
        "ForecastProducts": active_count,
        "CataloguePolicy": {
            "Name": "LATEST_KNOWN_ACTIVE_PANEL",
            "Definition": (
                "Forecast products represented in the latest known operating-day panel "
                "at the forecast origin. Historical products absent from that panel are "
                "excluded from future inference."
            ),
            "EvidenceWindowOperatingDays": 20,
            "Last20UniqueProducts": int(last20_audit[PRODUCT_ID_COLUMN].nunique()),
            "StableAll20Products": stable_count,
            "RecentActiveAdditions": recent_active_count,
            "RetiredBeforeCutoff": retired_count,
        },
        "DailyForecastPolicy": frozen_contract["DailyForecastPolicy"],
        "WeekForecastPolicy": frozen_contract["WeekForecastPolicy"],
        "DailyUpdatedWeekPolicy": frozen_contract["DailyUpdatedWeekPolicy"],
        "CalendarPolicy": frozen_contract["CalendarPolicy"],
        "HorizonPolicy": frozen_contract["HorizonPolicy"],
        "DemonstrationForecasts": {
            "DailyDate": requested_daily_date,
            "WeekStart": requested_week_start,
            "WeekEnd": requested_week_end,
            "DailyRestaurantNormalDemand": corrected_daily_total,
            "WeekRestaurantNormalDemand": corrected_week_total,
        },
        "March2026TargetVaultOpened": False,
        "NextStep": "ND10A_CORRECTED_PLANNING_QUANTITIES",
    }
    write_json(staged_contract_dir / INFERENCE_CONTRACT_PATH.name, corrected_contract)

    policy_text = f"""# ND09A Active Catalogue Policy

## Authoritative rule

Future inference uses the **latest known active product panel**, not every product that appeared historically.

At the frozen cutoff `{cutoff_date.date()}`:

- Historical canonical products: {EXPECTED_HISTORICAL_PRODUCTS}
- Products appearing at least once in the final 20 operating days: {int(last20_audit[PRODUCT_ID_COLUMN].nunique())}
- Products present on all 20 dates: {stable_count}
- Recent additions still active at cutoff: {recent_active_count}
- Products seen in the 20-day window but retired before cutoff: {retired_count}
- Latest active panel: {base_active_count}

The corrected default inference catalogue therefore contains {base_active_count} products.

This correction changes only inference scope. The frozen ND09 model artifacts, predictor architecture, route rules, daily method, and recursive weekly method are unchanged.
"""
    write_text(staged_contract_dir / CATALOGUE_POLICY_PATH.name, policy_text)

    write_csv(staged_audit_dir / LAST20_AUDIT_PATH.name, last20_audit)
    write_csv(staged_audit_dir / HISTORICAL_SCOPE_AUDIT_PATH.name, historical_scope_audit)
    write_csv(staged_audit_dir / FEATURE_REPLAY_AUDIT_PATH.name, feature_replay_audit)
    write_csv(staged_audit_dir / FORECAST_PATH_AUDIT_PATH.name, forecast_path_audit)
    write_csv(staged_audit_dir / SCOPE_COMPARISON_PATH.name, scope_comparison)
    write_csv(staged_audit_dir / ROUTE_SUMMARY_PATH.name, route_summary)

    package_versions = pd.DataFrame(
        [
            {"Package": "python", "Version": platform.python_version()},
            {"Package": "pandas", "Version": pd.__version__},
            {"Package": "numpy", "Version": np.__version__},
            {"Package": "scikit-learn", "Version": sklearn.__version__},
            {"Package": "xgboost", "Version": xgboost.__version__},
            {"Package": "catboost", "Version": catboost.__version__},
            {"Package": "joblib", "Version": joblib.__version__},
        ]
    )
    write_csv(staged_audit_dir / PACKAGE_VERSIONS_PATH.name, package_versions)

    validation = pd.DataFrame(
        [
            {
                "Check": "Frozen ND09 checkpoint hash matches",
                "Passed": actual_frozen_nd09_hash == EXPECTED_FROZEN_ND09_CHECKPOINT_SHA256,
            },
            {
                "Check": "No March or later training history loaded",
                "Passed": bool((all_routes[DATE_COLUMN] < DEVELOPMENT_END_EXCLUSIVE).all()),
            },
            {
                "Check": "Latest active panel contains 88 products",
                "Passed": base_active_count == EXPECTED_CURRENT_ACTIVE_PRODUCTS,
            },
            {
                "Check": "Last-20 catalogue contains 90 unique products",
                "Passed": int(last20_audit[PRODUCT_ID_COLUMN].nunique())
                == EXPECTED_LAST20_UNIQUE_PRODUCTS,
            },
            {
                "Check": "86 products stable across all 20 dates",
                "Passed": stable_count == EXPECTED_LAST20_STABLE_PRODUCTS,
            },
            {
                "Check": "2 recent active additions retained",
                "Passed": recent_active_count == EXPECTED_LAST20_RECENT_ACTIVE_ADDITIONS,
            },
            {
                "Check": "2 products retired before cutoff excluded",
                "Passed": retired_count == EXPECTED_LAST20_RETIRED_PRODUCTS,
            },
            {
                "Check": "Historical feature and route replay passed",
                "Passed": bool(feature_replay_audit["Passed"].all()),
            },
            {
                "Check": "Daily forecast contains one row per corrected active product",
                "Passed": len(daily_product_forecast) == active_count,
            },
            {
                "Check": "Weekly forecast contains one row per active product and operating date",
                "Passed": len(week_daily_product_forecast)
                == active_count * len(week_requested_dates),
            },
            {
                "Check": "No future actual demand used",
                "Passed": not bool(
                    pd.concat([daily_path, week_path], ignore_index=True)[
                        "FutureActualDemandUsed"
                    ].any()
                ),
            },
            {
                "Check": "All corrected predictions finite and nonnegative",
                "Passed": bool(
                    np.isfinite(
                        pd.concat([daily_path, week_path], ignore_index=True)[
                            "PredictedNormalDemand"
                        ]
                    ).all()
                    and (
                        pd.concat([daily_path, week_path], ignore_index=True)[
                            "PredictedNormalDemand"
                        ]
                        >= 0
                    ).all()
                ),
            },
            {
                "Check": "Frozen forecasting models were not refitted",
                "Passed": True,
            },
        ]
    )
    if not validation["Passed"].all():
        raise AssertionError(
            "ND09A validation failed:\n"
            + validation.loc[~validation["Passed"]].to_string(index=False)
        )
    write_csv(staged_audit_dir / VALIDATION_PATH.name, validation)

    report_text = f"""# ND09A Active-Menu Catalogue Correction

## Reason for the correction

ND09 originally carried all {EXPECTED_HISTORICAL_PRODUCTS} canonical products ever observed in the historical panel into future inference. A review of the final 20 pre-March operating days showed that this was too broad for a demonstration forecast because historically discontinued products could continue to receive future demand predictions.

## Evidence

The final 20 operating days run from {last20_dates[0].date()} to {last20_dates[-1].date()}.

- Unique products appearing at least once: {int(last20_audit[PRODUCT_ID_COLUMN].nunique())}
- Present on all 20 dates: {stable_count}
- Recent additions still active at cutoff: {recent_active_count}
- Seen during the window but retired before cutoff: {retired_count}
- Latest active panel at {cutoff_date.date()}: {base_active_count}

The authoritative inference catalogue was therefore corrected to the latest active panel.

## What changed

- Forecast catalogue: {int(old_daily_forecast[PRODUCT_ID_COLUMN].nunique())} → {active_count} products
- Daily restaurant normal-demand forecast: {old_daily_total:.6f} → {corrected_daily_total:.6f}
- Week restaurant normal-demand forecast: {old_week_total:.6f} → {corrected_week_total:.6f}

## What did not change

- Forecasting model fit: unchanged
- Daily selected method: `{daily_product_method}`
- Median components: `{', '.join(median_components)}`
- Recursive weekly method: `{week_planning_method}`
- Predictor architecture: unchanged
- March 2026 targets: not opened

This is an inference-scope correction, not a new model-selection exercise.
"""
    write_text(staged_report_dir / REPORT_SUMMARY_PATH.name, report_text)

    readme = f"""# ND09A Active-Menu Catalogue Correction

This amendment corrects future product scope from the full historical 227-product universe to the latest known active panel ({base_active_count} products at {cutoff_date.date()}).

The frozen ND09 model artifacts are reused without refitting.

Status: `{STATUS}`
"""
    write_text(STAGING_ROOT / "README.md", readme)

    manifest = build_manifest(STAGING_ROOT, exclude_control=True)
    write_csv(staged_control_dir / MANIFEST_PATH.name, manifest)
    manifest_hash = sha256_file(staged_control_dir / MANIFEST_PATH.name)

    checkpoint_payload = {
        "StepID": STEP_ID,
        "Status": STATUS,
        "CompletedLocalTime": NOW_LOCAL.isoformat(),
        "ND09ARoot": str(ND09A_ROOT),
        "FrozenND09CheckpointSHA256": actual_frozen_nd09_hash,
        "ArtifactManifestSHA256": manifest_hash,
        "DataCutoff": cutoff_date,
        "HistoricalProducts": EXPECTED_HISTORICAL_PRODUCTS,
        "LatestActivePanelProducts": base_active_count,
        "ForecastProducts": active_count,
        "CataloguePolicy": "LATEST_KNOWN_ACTIVE_PANEL",
        "ForecastingModelsRefitted": False,
        "ForecastingMethodsReselected": False,
        "MarchTargetVaultOpened": False,
        "DailyRestaurantNormalDemand": corrected_daily_total,
        "WeekRestaurantNormalDemand": corrected_week_total,
        "NextStep": "ND10A",
    }
    staged_checkpoint = staged_control_dir / CHECKPOINT_PATH.name
    write_json(staged_checkpoint, checkpoint_payload)
    checkpoint_hash = sha256_file(staged_checkpoint)
    write_text(staged_control_dir / CHECKPOINT_SHA_PATH.name, checkpoint_hash + "\n")

    lock_payload = {
        "LockType": "ND09A_ACTIVE_CATALOGUE_SCOPE_LOCK",
        "Status": "AUTHORITATIVE_FOR_DEMONSTRATION_PRODUCT_SCOPE",
        "CreatedLocalTime": NOW_LOCAL.isoformat(),
        "CheckpointSHA256": checkpoint_hash,
        "FrozenND09CheckpointSHA256": actual_frozen_nd09_hash,
        "CataloguePolicy": "LATEST_KNOWN_ACTIVE_PANEL",
        "ActiveProductsAtCutoff": base_active_count,
        "ForecastingModelsRefitted": False,
        "March2026TargetsUsed": False,
    }
    write_json(staged_control_dir / LOCK_PATH.name, lock_payload)

    protected_hashes_after = {str(path): sha256_file(path) for path in required_inputs}
    changed_inputs = [
        path
        for path in protected_hashes_before
        if protected_hashes_before[path] != protected_hashes_after[path]
    ]
    if changed_inputs:
        raise AssertionError(
            "Protected inputs changed during ND09A:\n"
            + "\n".join(f"- {path}" for path in changed_inputs)
        )

    os.replace(STAGING_ROOT, ND09A_ROOT)

    TOP_LEVEL_CHECKPOINT_PATH.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(ND09A_ROOT / "07_control" / CHECKPOINT_PATH.name, TOP_LEVEL_CHECKPOINT_PATH)
    shutil.copy2(
        ND09A_ROOT / "07_control" / CHECKPOINT_SHA_PATH.name,
        TOP_LEVEL_CHECKPOINT_SHA_PATH,
    )
    shutil.copy2(ND09A_ROOT / "07_control" / LOCK_PATH.name, TOP_LEVEL_LOCK_PATH)

    handoff_text = f"""# ND09A Handoff

## Status

- Status: `{STATUS}`
- Root: `{ND09A_ROOT}`
- Checkpoint SHA-256: `{checkpoint_hash}`

## Correction

Future inference now uses the latest known active product panel instead of all historical products.

- Historical product universe: {EXPECTED_HISTORICAL_PRODUCTS}
- Latest active panel: {base_active_count}
- Forecast products: {active_count}
- Last-20 stable products: {stable_count}
- Recent active additions retained: {recent_active_count}
- Recent retirements excluded: {retired_count}

## Frozen forecasting system

- Next day: `{daily_product_method}`
- Median components: `{', '.join(median_components)}`
- Multi-step/week start: `{week_planning_method}`
- Models refitted: no

## Next step

ND10A — regenerate confirmed-bulk/planned-quantity outputs from the corrected active catalogue.
"""
    atomic_write_text(HANDOFF_PATH, handoff_text)
    atomic_write_text(CURRENT_HANDOFF_PATH, handoff_text)

    append_marked_section(
        WORKFLOW_PATH,
        "## ND09A — Active-menu catalogue correction",
        f"""## ND09A — Active-menu catalogue correction

Status: `{STATUS}`

The frozen ND09 model artifacts were reused without refitting. Future inference scope was corrected from the historical 227-product universe to the latest known active panel of {base_active_count} products.
""",
    )
    append_marked_section(
        DECISIONS_PATH,
        "## ND09A decisions",
        f"""## ND09A decisions

- Treat historical product existence and current menu activity as separate concepts.
- Define the default future forecast catalogue as the latest known active operating-day panel.
- Retain recent additions that are present at the cutoff.
- Exclude historical products that are absent from the cutoff panel.
- Keep the 14-product illustrative ingredient mapping as a downstream subset; it does not control forecast coverage.
""",
    )
    append_marked_section(
        METRICS_AND_RESULTS_PATH,
        "## ND09A corrected demonstration scope",
        f"""## ND09A corrected demonstration scope

- Historical products: {EXPECTED_HISTORICAL_PRODUCTS}
- Corrected active forecast products: {active_count}
- Corrected daily restaurant normal-demand forecast: {corrected_daily_total:.6f}
- Corrected week restaurant normal-demand forecast: {corrected_week_total:.6f}
- No March accuracy metric was calculated.
""",
    )
    append_marked_section(
        AGENTS_PATH,
        "Marker: ND09A_AUTHORITATIVE_ACTIVE_SCOPE",
        f"""## ND09A authoritative active scope

Marker: ND09A_AUTHORITATIVE_ACTIVE_SCOPE

- Status: `{STATUS}`
- Active catalogue policy: `LATEST_KNOWN_ACTIVE_PANEL`
- Active products at frozen cutoff: {base_active_count}
- Checkpoint SHA-256: `{checkpoint_hash}`
- Next step: ND10A corrected planning outputs.
""",
    )

    LOG_PATH.parent.mkdir(parents=True, exist_ok=True)
    with LOG_PATH.open("a", encoding="utf-8") as handle:
        handle.write(
            f"{NOW_LOCAL.isoformat()} | {STATUS} | checkpoint={checkpoint_hash} | "
            f"active_products={active_count} | root={ND09A_ROOT}\n"
        )

except Exception:
    if STAGING_ROOT.exists():
        shutil.rmtree(STAGING_ROOT, ignore_errors=True)
    raise


# =============================================================================
# FINAL CONSOLE OUTPUT
# =============================================================================

print("=" * 118)
print("EDEN NORMAL-DEMAND MODEL V2 — ND09A ACTIVE-MENU CORRECTION COMPLETE")
print("=" * 118)
print(f"Status: {STATUS}")
print(f"Local time: {NOW_LOCAL.isoformat()}")
print(f"ND09A root: {ND09A_ROOT}")
print()
print("INPUT VERIFICATION")
print(f"ND03 checkpoint SHA-256: {actual_nd03_hash}")
print(f"Frozen ND09 checkpoint SHA-256: {actual_frozen_nd09_hash}")
print(f"Frozen ND09 demonstration lock present: {FROZEN_ND09_LOCK_PATH.is_file()}")
print(f"March target vault opened: False")
print(f"Previous inputs modified: False")
print()
print("LAST 20 OPERATING-DAY CATALOGUE ANALYSIS")
print(f"Window: {last20_dates[0].date()} to {last20_dates[-1].date()}")
print(f"Unique products appearing at least once: {last20_audit[PRODUCT_ID_COLUMN].nunique()}")
print(f"Products present on all 20 dates: {stable_count}")
print(f"Recent additions active at cutoff: {recent_active_count}")
print(f"Products retired before cutoff: {retired_count}")
print(f"Latest active panel products: {base_active_count}")
print()
print("CORRECTED FORECAST CATALOGUE")
print(f"Historical product universe: {EXPECTED_HISTORICAL_PRODUCTS}")
print(f"Old ND09 products forecast: {old_daily_forecast[PRODUCT_ID_COLUMN].nunique()}")
print(f"Corrected products forecast: {active_count}")
print(f"Catalogue policy: LATEST_KNOWN_ACTIVE_PANEL")
print()
print("FROZEN FORECASTING SYSTEM")
print(f"Next-operating-day method: {daily_product_method}")
print(f"Representative median components: {', '.join(median_components)}")
print(f"Multi-step / week-start method: {week_planning_method}")
print(f"Rolling daily-updated week method: {updated_week_method}")
print(f"Frozen models loaded, not refitted: True")
print(f"Model loading seconds: {model_load_seconds:.3f}")
print()
print("CORRECTED DEMONSTRATION FORECASTS")
print(f"Daily date: {requested_daily_date.date()}")
print(f"Old daily restaurant normal demand: {old_daily_total:.6f}")
print(f"Corrected daily restaurant normal demand: {corrected_daily_total:.6f}")
print(f"Week: {requested_week_start.date()} to {requested_week_end.date()}")
print(f"Old week restaurant normal demand: {old_week_total:.6f}")
print(f"Corrected week restaurant normal demand: {corrected_week_total:.6f}")
print()
print("VALIDATION")
print(f"Historical feature and route replay passed: {feature_replay_audit['Passed'].all()}")
print(f"Validation checks passed: {int(validation['Passed'].sum())}/{len(validation)}")
print()
print("OUTPUTS")
print(f"- Corrected active catalogue: {ACTIVE_CATALOGUE_PATH}")
print(f"- Daily product forecast: {DAILY_PRODUCT_FORECAST_PATH}")
print(f"- Week daily product forecast: {WEEK_DAILY_PRODUCT_FORECAST_PATH}")
print(f"- Last-20 catalogue audit: {LAST20_AUDIT_PATH}")
print(f"- Historical scope audit: {HISTORICAL_SCOPE_AUDIT_PATH}")
print(f"- Scope comparison: {SCOPE_COMPARISON_PATH}")
print(f"- Corrected inference contract: {INFERENCE_CONTRACT_PATH}")
print(f"- Report: {REPORT_SUMMARY_PATH}")
print(f"- Checkpoint: {TOP_LEVEL_CHECKPOINT_PATH}")
print(f"- Checkpoint SHA-256: {checkpoint_hash}")
print(f"- Active-catalogue scope lock: {TOP_LEVEL_LOCK_PATH}")
print(f"- Handoff: {HANDOFF_PATH}")
print()
print("SAFETY")
print("- Forecasting models fitted/refitted: False")
print("- Forecasting methods reselected: False")
print("- March target vault opened: False")
print("- Future actual demand used: False")
print("- Previous ND09 artifacts modified: False")
print("- ND09A checkpoint, hashes and scope lock created: True")
print()
print("NEXT STEP")
print("ND10A — regenerate confirmed-bulk and planned-quantity outputs using the corrected active catalogue.")
print("=" * 118)

EDEN NORMAL-DEMAND MODEL V2 — ND09A ACTIVE-MENU CORRECTION COMPLETE
Status: ND09A_ACTIVE_MENU_CATALOGUE_CORRECTION_COMPLETED_READY_FOR_ND10A
Local time: 2026-08-10T12:43:06.434426+01:00
ND09A root: /Users/ryansmac/Desktop/Meng Project/eden_datasets/eden_normal_demand_model_v2/03_models/03_inference/ND09A_active_menu_catalogue_correction

INPUT VERIFICATION
ND03 checkpoint SHA-256: 0845af89a5b459ca13ae6ffd99dde444f5010f6c0fb5ba4c34a4f091ac2e151c
Frozen ND09 checkpoint SHA-256: 201b84095ed138e0b666485e2022e4f1f99ad93b89971f46533ff4a0605ef4e8
Frozen ND09 demonstration lock present: True
March target vault opened: False
Previous inputs modified: False

LAST 20 OPERATING-DAY CATALOGUE ANALYSIS
Window: 2026-01-30 to 2026-02-27
Unique products appearing at least once: 90
Products present on all 20 dates: 86
Recent additions active at cutoff: 2
Products retired before cutoff: 2
Latest active panel products: 88

CORRECTED FORECAST CATALOGUE
Historical product universe: 227
Old ND09 products for

In [18]:
# =============================================================================
# EDEN NORMAL-DEMAND MODEL V2
# ND10A — CORRECTED ACTIVE-CATALOGUE PLANNING QUANTITIES
#
# Run this as one complete Jupyter cell after the completed ND09A active-menu correction.
#
# This step does not refit or retune any forecasting model. It consumes the
# frozen ND09A normal-demand forecasts and adds only externally confirmed bulk
# orders. The final operational quantity is:
#
#   PlannedQuantity = PredictedNormalDemand + ConfirmedBulkDemand
#
# Normal demand, confirmed bulk demand, and planned quantity remain separate
# in every output. March 2026 targets are not loaded or used.
# =============================================================================

import hashlib
import json
import os
import platform
import shutil
import uuid
from datetime import datetime, timezone
from pathlib import Path
from zoneinfo import ZoneInfo

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd


# =============================================================================
# USER CONFIGURATION
# =============================================================================

PROJECT_ROOT = Path("/Users/ryansmac/Desktop/Meng Project")

# Place a CSV at this path to add confirmed bulk orders. If the file does not
# exist, ND10A creates zero-bulk planned outputs and saves an input template.
CONFIRMED_BULK_INPUT_PATH = (
    PROJECT_ROOT
    / "eden_datasets"
    / "eden_normal_demand_model_v2"
    / "04_planning_inputs"
    / "ND10A_confirmed_bulk_orders.csv"
)

# Set True only when a missing bulk-order file should stop the run.
REQUIRE_BULK_INPUT_FILE = False

# Bulk lines outside the dates currently forecast by ND09A are blocked by
# default so that no confirmed order is silently omitted from the plan.
ALLOW_OUT_OF_FORECAST_RANGE_BULK_ORDERS = False

# Set True only when intentionally rebuilding ND10A after changing the bulk
# input. ND09A and all earlier artifacts remain protected and unchanged.
ALLOW_OVERWRITE = False


# =============================================================================
# FIXED PROJECT CONFIGURATION
# =============================================================================

EDEN_ROOT = PROJECT_ROOT / "eden_datasets"
MODEL_ROOT = EDEN_ROOT / "eden_normal_demand_model_v2"

ND09A_ROOT = (
    MODEL_ROOT
    / "03_models"
    / "03_inference"
    / "ND09A_active_menu_catalogue_correction"
)
ND09A_FORECAST_DIR = ND09A_ROOT / "01_forecasts"
ND09A_CONTRACT_DIR = ND09A_ROOT / "02_contracts"

ND09A_DAILY_PRODUCT_PATH = (
    ND09A_FORECAST_DIR / "ND09A_arbitrary_date_product_forecast.csv"
)
ND09A_DAILY_RESTAURANT_PATH = (
    ND09A_FORECAST_DIR / "ND09A_arbitrary_date_restaurant_total.csv"
)
ND09A_WEEK_DAILY_PRODUCT_PATH = (
    ND09A_FORECAST_DIR / "ND09A_week_ahead_daily_product_forecast.csv"
)
ND09A_WEEK_PRODUCT_TOTAL_PATH = (
    ND09A_FORECAST_DIR / "ND09A_week_ahead_product_totals.csv"
)
ND09A_WEEK_RESTAURANT_DAILY_PATH = (
    ND09A_FORECAST_DIR / "ND09A_week_ahead_restaurant_daily_totals.csv"
)
ND09A_WEEK_RESTAURANT_TOTAL_PATH = (
    ND09A_FORECAST_DIR / "ND09A_week_ahead_restaurant_total.csv"
)
ND09A_ACTIVE_CATALOGUE_PATH = (
    ND09A_CONTRACT_DIR / "ND09A_active_product_catalogue.csv"
)
ND09A_INFERENCE_CONTRACT_PATH = (
    ND09A_CONTRACT_DIR / "ND09A_corrected_inference_contract.json"
)
ND09A_CHECKPOINT_PATH = MODEL_ROOT / "08_checkpoints" / "ND09A_checkpoint.json"
ND09A_LOCK_PATH = (
    MODEL_ROOT / "08_checkpoints" / "ND09A_active_catalogue_scope_lock.json"
)

EXPECTED_ND09A_CHECKPOINT_SHA256 = (
    "54767758d9052cb1571043d2f39cc167bc7491b300afbd3605a5416b7aa95c10"
)

DATE_COLUMN = "Date"
PRODUCT_ID_COLUMN = "CanonicalProductID"
PRODUCT_NAME_COLUMN = "CanonicalProductName"
FAMILY_COLUMN = "TierProductFamily"
NORMAL_FORECAST_COLUMN = "PredictedNormalDemand"
BULK_COLUMN = "ConfirmedBulkDemand"
PLANNED_COLUMN = "PlannedQuantity"

BULK_REQUIRED_COLUMNS = {
    DATE_COLUMN,
    PRODUCT_ID_COLUMN,
    BULK_COLUMN,
}
BULK_OPTIONAL_COLUMNS = [
    "OrderReference",
    "CustomerOrEvent",
    "Notes",
]
BULK_TEMPLATE_COLUMNS = [
    "OrderReference",
    DATE_COLUMN,
    PRODUCT_ID_COLUMN,
    BULK_COLUMN,
    "CustomerOrEvent",
    "Notes",
]

EXPECTED_ACTIVE_PRODUCTS = 88
EXPECTED_DAILY_METHOD = "NESTED_MEDIAN_ENSEMBLE"
EXPECTED_WEEK_METHOD = "ROLLING_MEAN_5"
EXPECTED_TARGET = "NormalDemand"

ND10A_ROOT = (
    MODEL_ROOT
    / "04_planning"
    / "ND10A_corrected_active_catalogue_planning"
)
INPUT_SNAPSHOT_DIR = ND10A_ROOT / "01_inputs"
OUTPUT_DIR = ND10A_ROOT / "02_planned_outputs"
CONTRACT_DIR = ND10A_ROOT / "03_contracts"
AUDIT_DIR = ND10A_ROOT / "04_audits"
FIGURE_DIR = ND10A_ROOT / "05_figures"
REPORT_DIR = ND10A_ROOT / "06_reports"
CONTROL_DIR = ND10A_ROOT / "07_control"

BULK_TEMPLATE_PATH = CONTRACT_DIR / "ND10A_confirmed_bulk_order_input_template.csv"
BULK_NORMALIZED_PATH = INPUT_SNAPSHOT_DIR / "ND10A_confirmed_bulk_orders_normalized.csv"
BULK_AGGREGATED_PATH = INPUT_SNAPSHOT_DIR / "ND10A_confirmed_bulk_by_product_date.csv"

DAILY_PRODUCT_PLANNING_PATH = OUTPUT_DIR / "ND10A_daily_product_planning.csv"
DAILY_RESTAURANT_PLANNING_PATH = OUTPUT_DIR / "ND10A_daily_restaurant_planning.csv"
WEEK_DAILY_PRODUCT_PLANNING_PATH = (
    OUTPUT_DIR / "ND10A_week_daily_product_planning.csv"
)
WEEK_PRODUCT_TOTAL_PLANNING_PATH = (
    OUTPUT_DIR / "ND10A_week_product_totals_planning.csv"
)
WEEK_RESTAURANT_DAILY_PLANNING_PATH = (
    OUTPUT_DIR / "ND10A_week_restaurant_daily_planning.csv"
)
WEEK_RESTAURANT_TOTAL_PLANNING_PATH = (
    OUTPUT_DIR / "ND10A_week_restaurant_total_planning.csv"
)
DEMONSTRATION_SUMMARY_PATH = OUTPUT_DIR / "ND10A_demonstration_planning_summary.csv"

PLANNING_CONTRACT_PATH = CONTRACT_DIR / "ND10A_planning_quantity_contract.json"
PLANNING_CONTRACT_MD_PATH = CONTRACT_DIR / "ND10A_planning_quantity_contract.md"
USAGE_PATH = CONTRACT_DIR / "ND10A_bulk_order_usage.md"

INPUT_HASH_AUDIT_PATH = AUDIT_DIR / "ND10A_input_hash_audit.csv"
BULK_VALIDATION_AUDIT_PATH = AUDIT_DIR / "ND10A_bulk_order_validation_audit.csv"
BULK_MATCH_AUDIT_PATH = AUDIT_DIR / "ND10A_bulk_order_match_audit.csv"
RECONCILIATION_AUDIT_PATH = AUDIT_DIR / "ND10A_planning_reconciliation_audit.csv"
VALIDATION_PATH = AUDIT_DIR / "ND10A_validation_summary.csv"
PACKAGE_VERSIONS_PATH = AUDIT_DIR / "ND10A_package_versions.csv"

REPORT_SUMMARY_PATH = REPORT_DIR / "ND10A_confirmed_bulk_and_planning_summary.md"
README_PATH = ND10A_ROOT / "README.md"
MANIFEST_PATH = CONTROL_DIR / "ND10A_artifact_hash_manifest.csv"
CHECKPOINT_PATH = CONTROL_DIR / "ND10A_checkpoint.json"
CHECKPOINT_SHA_PATH = CONTROL_DIR / "ND10A_checkpoint.sha256"

TOP_LEVEL_CHECKPOINT_PATH = MODEL_ROOT / "08_checkpoints" / "ND10A_checkpoint.json"
TOP_LEVEL_CHECKPOINT_SHA_PATH = MODEL_ROOT / "08_checkpoints" / "ND10A_checkpoint.sha256"

MEMORY_ROOT = MODEL_ROOT / "00_project_memory"
ND10A_HANDOFF_PATH = MEMORY_ROOT / "ND10A_HANDOFF.md"
CURRENT_HANDOFF_PATH = MEMORY_ROOT / "CURRENT_HANDOFF.md"
WORKFLOW_PATH = MEMORY_ROOT / "WORKFLOW.md"
DECISIONS_PATH = MEMORY_ROOT / "DECISIONS.md"
METRICS_AND_RESULTS_PATH = MEMORY_ROOT / "METRICS_AND_RESULTS.md"
AGENTS_PATH = MODEL_ROOT / "AGENTS.md"
LOG_PATH = MODEL_ROOT / "09_logs" / "ND10A_confirmed_bulk_planning_log.txt"

STEP_ID = "ND10A"
STATUS = "ND10A_CORRECTED_PLANNING_QUANTITIES_CREATED_READY_FOR_ND10R_DEMO_REFRESH"
NOW_UTC = datetime.now(timezone.utc)
NOW_LOCAL = NOW_UTC.astimezone(ZoneInfo("Europe/Dublin"))


# =============================================================================
# GENERAL HELPERS
# =============================================================================

def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


def write_csv(path: Path, frame: pd.DataFrame) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    frame.to_csv(path, index=False)


def write_json(path: Path, payload: dict) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(
        json.dumps(payload, indent=2, ensure_ascii=False, default=str) + "\n",
        encoding="utf-8",
    )


def write_text(path: Path, text: str) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(text, encoding="utf-8")


def atomic_write_text(path: Path, text: str) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_name(f".{path.name}.{uuid.uuid4().hex}.tmp")
    temporary.write_text(text, encoding="utf-8")
    os.replace(temporary, path)


def append_marked_section(path: Path, marker: str, section_text: str) -> None:
    existing = path.read_text(encoding="utf-8") if path.is_file() else ""
    if marker in existing:
        return
    separator = "\n" if existing.endswith("\n") else "\n\n"
    atomic_write_text(path, existing + separator + section_text.strip() + "\n")


def validate_required_columns(
    frame: pd.DataFrame,
    columns: set[str],
    name: str,
) -> None:
    missing = sorted(columns - set(frame.columns))
    if missing:
        raise AssertionError(
            f"{name} is missing required columns:\n"
            + "\n".join(f"- {column}" for column in missing)
        )


def maximum_abs_difference(left: float, right: float) -> float:
    return float(abs(float(left) - float(right)))


def save_figure(path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    plt.tight_layout()
    plt.savefig(path, dpi=300, bbox_inches="tight")
    plt.close()


def build_manifest(root: Path, exclude_control: bool = True) -> pd.DataFrame:
    records = []
    for path in sorted(root.rglob("*")):
        if not path.is_file():
            continue
        relative = path.relative_to(root)
        if exclude_control and relative.parts and relative.parts[0] == "07_control":
            continue
        records.append(
            {
                "RelativePath": str(relative),
                "Bytes": int(path.stat().st_size),
                "SHA256": sha256_file(path),
            }
        )
    return pd.DataFrame(records)


# =============================================================================
# BULK-ORDER HELPERS
# =============================================================================

def empty_bulk_frame() -> pd.DataFrame:
    return pd.DataFrame(columns=BULK_TEMPLATE_COLUMNS)


def load_and_validate_bulk_orders(
    input_path: Path,
    active_catalogue: pd.DataFrame,
    forecast_dates: set[pd.Timestamp],
) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame, bool]:
    input_exists = input_path.is_file()
    validation_records = []

    if not input_exists:
        if REQUIRE_BULK_INPUT_FILE:
            raise FileNotFoundError(
                "Confirmed bulk-order input is required but was not found:\n"
                f"{input_path}"
            )
        raw = empty_bulk_frame()
        validation_records.append(
            {
                "Check": "Bulk-order input file exists",
                "Passed": False,
                "Detail": "Missing input accepted; zero confirmed bulk used.",
            }
        )
    else:
        raw = pd.read_csv(input_path, low_memory=False)
        validation_records.append(
            {
                "Check": "Bulk-order input file exists",
                "Passed": True,
                "Detail": str(input_path),
            }
        )

    if raw.empty and not BULK_REQUIRED_COLUMNS.issubset(raw.columns):
        raw = empty_bulk_frame()

    validate_required_columns(raw, BULK_REQUIRED_COLUMNS, "Confirmed bulk-order input")

    normalized = raw.copy()
    for column in BULK_OPTIONAL_COLUMNS:
        if column not in normalized.columns:
            normalized[column] = ""

    normalized = normalized[BULK_TEMPLATE_COLUMNS].copy()
    normalized[DATE_COLUMN] = pd.to_datetime(
        normalized[DATE_COLUMN], errors="coerce"
    ).dt.normalize()
    normalized[PRODUCT_ID_COLUMN] = (
        normalized[PRODUCT_ID_COLUMN]
        .astype("string")
        .fillna("")
        .str.strip()
        .astype(str)
    )
    normalized[BULK_COLUMN] = pd.to_numeric(
        normalized[BULK_COLUMN], errors="coerce"
    )
    for column in BULK_OPTIONAL_COLUMNS:
        normalized[column] = (
            normalized[column].astype("string").fillna("").astype(str).str.strip()
        )

    invalid_date = normalized[DATE_COLUMN].isna()
    invalid_product = normalized[PRODUCT_ID_COLUMN].eq("")
    invalid_units = (
        normalized[BULK_COLUMN].isna()
        | ~np.isfinite(normalized[BULK_COLUMN].astype(float))
        | (normalized[BULK_COLUMN] < 0)
    )
    non_integer_units = (
        ~invalid_units
        & ~np.isclose(
            normalized[BULK_COLUMN].astype(float),
            np.round(normalized[BULK_COLUMN].astype(float)),
            atol=1e-10,
        )
    )

    available_products = set(
        active_catalogue.loc[
            active_catalogue["IncludeInForecast"].astype(bool), PRODUCT_ID_COLUMN
        ].astype(str)
    )
    unknown_product = ~normalized[PRODUCT_ID_COLUMN].isin(available_products)
    out_of_range = ~normalized[DATE_COLUMN].isin(forecast_dates)

    issue_frame = normalized.copy()
    issue_frame["InvalidDate"] = invalid_date
    issue_frame["InvalidProductID"] = invalid_product
    issue_frame["InvalidConfirmedBulkDemand"] = invalid_units
    issue_frame["NonIntegerConfirmedBulkDemand"] = non_integer_units
    issue_frame["UnknownProduct"] = unknown_product
    issue_frame["OutsideND09AForecastDates"] = out_of_range
    issue_frame["AcceptedForPlanning"] = ~(
        invalid_date
        | invalid_product
        | invalid_units
        | non_integer_units
        | unknown_product
        | (
            out_of_range
            & (not ALLOW_OUT_OF_FORECAST_RANGE_BULK_ORDERS)
        )
    )

    hard_issue_mask = (
        invalid_date
        | invalid_product
        | invalid_units
        | non_integer_units
        | unknown_product
        | (
            out_of_range
            & (not ALLOW_OUT_OF_FORECAST_RANGE_BULK_ORDERS)
        )
    )
    if hard_issue_mask.any():
        raise ValueError(
            "Confirmed bulk-order validation failed:\n"
            + issue_frame.loc[hard_issue_mask].to_string(index=False)
        )

    accepted = normalized.loc[issue_frame["AcceptedForPlanning"]].copy()
    if not accepted.empty:
        accepted[BULK_COLUMN] = np.round(accepted[BULK_COLUMN]).astype(int)
        accepted = accepted.sort_values(
            [DATE_COLUMN, PRODUCT_ID_COLUMN, "OrderReference"],
            kind="mergesort",
        ).reset_index(drop=True)

    aggregated = (
        accepted.groupby([DATE_COLUMN, PRODUCT_ID_COLUMN], as_index=False)[BULK_COLUMN]
        .sum()
        if not accepted.empty
        else pd.DataFrame(columns=[DATE_COLUMN, PRODUCT_ID_COLUMN, BULK_COLUMN])
    )

    validation_records.extend(
        [
            {
                "Check": "All bulk dates parse successfully",
                "Passed": not bool(invalid_date.any()),
                "Detail": f"Invalid rows: {int(invalid_date.sum())}",
            },
            {
                "Check": "All product IDs are present",
                "Passed": not bool(invalid_product.any()),
                "Detail": f"Invalid rows: {int(invalid_product.sum())}",
            },
            {
                "Check": "Confirmed bulk quantities are finite and nonnegative",
                "Passed": not bool(invalid_units.any()),
                "Detail": f"Invalid rows: {int(invalid_units.sum())}",
            },
            {
                "Check": "Confirmed bulk quantities are whole units",
                "Passed": not bool(non_integer_units.any()),
                "Detail": f"Invalid rows: {int(non_integer_units.sum())}",
            },
            {
                "Check": "All products exist in the active catalogue",
                "Passed": not bool(unknown_product.any()),
                "Detail": f"Unknown rows: {int(unknown_product.sum())}",
            },
            {
                "Check": "All bulk dates are covered by ND09A forecasts",
                "Passed": not bool(out_of_range.any()),
                "Detail": f"Out-of-range rows: {int(out_of_range.sum())}",
            },
        ]
    )

    return (
        accepted,
        aggregated,
        pd.DataFrame(validation_records),
        input_exists,
    )


def add_bulk_to_forecast(
    forecast: pd.DataFrame,
    bulk_by_product_date: pd.DataFrame,
    output_label: str,
) -> pd.DataFrame:
    validate_required_columns(
        forecast,
        {
            DATE_COLUMN,
            PRODUCT_ID_COLUMN,
            PRODUCT_NAME_COLUMN,
            FAMILY_COLUMN,
            NORMAL_FORECAST_COLUMN,
        },
        output_label,
    )

    output = forecast.copy()
    output[DATE_COLUMN] = pd.to_datetime(output[DATE_COLUMN], errors="raise").dt.normalize()
    output[PRODUCT_ID_COLUMN] = output[PRODUCT_ID_COLUMN].astype(str)
    output[NORMAL_FORECAST_COLUMN] = pd.to_numeric(
        output[NORMAL_FORECAST_COLUMN], errors="raise"
    ).astype(float)

    if output.duplicated([DATE_COLUMN, PRODUCT_ID_COLUMN]).any():
        duplicates = output.loc[
            output.duplicated([DATE_COLUMN, PRODUCT_ID_COLUMN], keep=False),
            [DATE_COLUMN, PRODUCT_ID_COLUMN],
        ]
        raise AssertionError(
            f"{output_label} contains duplicate product-date rows:\n"
            + duplicates.to_string(index=False)
        )

    output = output.merge(
        bulk_by_product_date,
        on=[DATE_COLUMN, PRODUCT_ID_COLUMN],
        how="left",
        validate="one_to_one",
    )
    output[BULK_COLUMN] = output[BULK_COLUMN].fillna(0).astype(float)
    output[PLANNED_COLUMN] = output[NORMAL_FORECAST_COLUMN] + output[BULK_COLUMN]
    output["PlanningFormula"] = (
        "PredictedNormalDemand + ConfirmedBulkDemand"
    )

    if not np.isfinite(output[[NORMAL_FORECAST_COLUMN, BULK_COLUMN, PLANNED_COLUMN]]).all().all():
        raise AssertionError(f"{output_label} contains non-finite planning quantities.")
    if (output[[NORMAL_FORECAST_COLUMN, BULK_COLUMN, PLANNED_COLUMN]] < 0).any().any():
        raise AssertionError(f"{output_label} contains negative planning quantities.")

    return output


# =============================================================================
# PREFLIGHT
# =============================================================================

required_inputs = [
    ND09A_DAILY_PRODUCT_PATH,
    ND09A_DAILY_RESTAURANT_PATH,
    ND09A_WEEK_DAILY_PRODUCT_PATH,
    ND09A_WEEK_PRODUCT_TOTAL_PATH,
    ND09A_WEEK_RESTAURANT_DAILY_PATH,
    ND09A_WEEK_RESTAURANT_TOTAL_PATH,
    ND09A_ACTIVE_CATALOGUE_PATH,
    ND09A_INFERENCE_CONTRACT_PATH,
    ND09A_CHECKPOINT_PATH,
    ND09A_LOCK_PATH,
]
missing_inputs = [path for path in required_inputs if not path.is_file()]
if missing_inputs:
    raise FileNotFoundError(
        "ND10A required ND09A inputs are missing:\n"
        + "\n".join(f"- {path}" for path in missing_inputs)
    )

actual_nd09_checkpoint_hash = sha256_file(ND09A_CHECKPOINT_PATH)
if actual_nd09_checkpoint_hash != EXPECTED_ND09A_CHECKPOINT_SHA256:
    raise AssertionError(
        "ND09A checkpoint hash mismatch.\n"
        f"Expected: {EXPECTED_ND09A_CHECKPOINT_SHA256}\n"
        f"Actual:   {actual_nd09_checkpoint_hash}"
    )

if ND10A_ROOT.exists():
    if not ALLOW_OVERWRITE:
        raise FileExistsError(
            "ND10A output already exists. No files were changed:\n"
            f"{ND10A_ROOT}"
        )
    shutil.rmtree(ND10A_ROOT)

for path in [TOP_LEVEL_CHECKPOINT_PATH, TOP_LEVEL_CHECKPOINT_SHA_PATH]:
    if path.exists():
        if not ALLOW_OVERWRITE:
            raise FileExistsError(
                "An ND10A top-level checkpoint already exists. No files were changed:\n"
                f"{path}"
            )
        path.unlink()

protected_input_hashes_before = {str(path): sha256_file(path) for path in required_inputs}
if CONFIRMED_BULK_INPUT_PATH.is_file():
    protected_input_hashes_before[str(CONFIRMED_BULK_INPUT_PATH)] = sha256_file(
        CONFIRMED_BULK_INPUT_PATH
    )

STAGING_ROOT = ND10A_ROOT.parent / f".ND10A_corrected_planning_staging_{uuid.uuid4().hex}"
STAGING_ROOT.mkdir(parents=True, exist_ok=False)


# =============================================================================
# MAIN EXECUTION
# =============================================================================

try:
    daily_forecast = pd.read_csv(ND09A_DAILY_PRODUCT_PATH, low_memory=False)
    daily_restaurant_source = pd.read_csv(
        ND09A_DAILY_RESTAURANT_PATH, low_memory=False
    )
    week_daily_forecast = pd.read_csv(
        ND09A_WEEK_DAILY_PRODUCT_PATH, low_memory=False
    )
    week_product_source = pd.read_csv(
        ND09A_WEEK_PRODUCT_TOTAL_PATH, low_memory=False
    )
    week_restaurant_daily_source = pd.read_csv(
        ND09A_WEEK_RESTAURANT_DAILY_PATH, low_memory=False
    )
    week_restaurant_total_source = pd.read_csv(
        ND09A_WEEK_RESTAURANT_TOTAL_PATH, low_memory=False
    )
    active_catalogue = pd.read_csv(ND09A_ACTIVE_CATALOGUE_PATH, low_memory=False)
    inference_contract = json.loads(
        ND09A_INFERENCE_CONTRACT_PATH.read_text(encoding="utf-8")
    )
    nd09_checkpoint = json.loads(ND09A_CHECKPOINT_PATH.read_text(encoding="utf-8"))
    nd09_lock = json.loads(ND09A_LOCK_PATH.read_text(encoding="utf-8"))

    validate_required_columns(
        active_catalogue,
        {PRODUCT_ID_COLUMN, PRODUCT_NAME_COLUMN, "IncludeInForecast"},
        "ND09A active product catalogue",
    )
    active_catalogue[PRODUCT_ID_COLUMN] = active_catalogue[PRODUCT_ID_COLUMN].astype(str)
    active_count = int(active_catalogue["IncludeInForecast"].astype(bool).sum())
    if active_count != EXPECTED_ACTIVE_PRODUCTS:
        raise AssertionError(
            f"Expected {EXPECTED_ACTIVE_PRODUCTS} active products; found {active_count}."
        )

    if inference_contract.get("CorrectionType") != "INFERENCE_CATALOGUE_SCOPE_ONLY":
        raise AssertionError(
            "ND09A contract is not the authoritative inference-catalogue-only correction."
        )
    if (
        inference_contract.get("CataloguePolicy", {}).get("Name")
        != "LATEST_KNOWN_ACTIVE_PANEL"
    ):
        raise AssertionError(
            "ND09A contract does not use the LATEST_KNOWN_ACTIVE_PANEL policy."
        )
    if int(inference_contract.get("ForecastProducts", -1)) != EXPECTED_ACTIVE_PRODUCTS:
        raise AssertionError(
            "ND09A corrected contract does not declare 88 forecast products."
        )
    if bool(inference_contract.get("ForecastingModelsRefitted", True)):
        raise AssertionError(
            "ND09A contract unexpectedly indicates that forecasting models were refitted."
        )

    if inference_contract.get("Target") != EXPECTED_TARGET:
        raise AssertionError("ND09A inference target is not NormalDemand.")
    if (
        inference_contract.get("DailyForecastPolicy", {}).get(
            "OneOperatingDayAheadMethod"
        )
        != EXPECTED_DAILY_METHOD
    ):
        raise AssertionError("ND09A next-day method is not the selected median ensemble.")
    if (
        inference_contract.get("WeekForecastPolicy", {}).get("Method")
        != EXPECTED_WEEK_METHOD
    ):
        raise AssertionError("ND09A week method is not ROLLING_MEAN_5.")
    if bool(inference_contract.get("March2026TargetVaultOpened", True)):
        raise AssertionError("ND09A contract indicates that March targets were opened.")

    daily_forecast[DATE_COLUMN] = pd.to_datetime(
        daily_forecast[DATE_COLUMN], errors="raise"
    ).dt.normalize()
    week_daily_forecast[DATE_COLUMN] = pd.to_datetime(
        week_daily_forecast[DATE_COLUMN], errors="raise"
    ).dt.normalize()
    daily_forecast[PRODUCT_ID_COLUMN] = daily_forecast[PRODUCT_ID_COLUMN].astype(str)
    week_daily_forecast[PRODUCT_ID_COLUMN] = week_daily_forecast[PRODUCT_ID_COLUMN].astype(str)

    if len(daily_forecast) != active_count:
        raise AssertionError(
            "ND09A arbitrary-date forecast does not contain one row per active product."
        )
    week_dates = sorted(week_daily_forecast[DATE_COLUMN].unique())
    if len(week_dates) == 0:
        raise AssertionError("ND09A week forecast contains no dates.")
    if not (
        week_daily_forecast.groupby(DATE_COLUMN)[PRODUCT_ID_COLUMN].nunique()
        == active_count
    ).all():
        raise AssertionError(
            "ND09A week forecast does not contain all active products on every date."
        )

    forecast_dates = set(daily_forecast[DATE_COLUMN]) | set(
        week_daily_forecast[DATE_COLUMN]
    )
    (
        bulk_orders,
        bulk_by_product_date,
        bulk_validation_audit,
        bulk_input_exists,
    ) = load_and_validate_bulk_orders(
        CONFIRMED_BULK_INPUT_PATH,
        active_catalogue,
        forecast_dates,
    )

    daily_planning = add_bulk_to_forecast(
        daily_forecast,
        bulk_by_product_date,
        "ND09A arbitrary-date product forecast",
    )
    week_daily_planning = add_bulk_to_forecast(
        week_daily_forecast,
        bulk_by_product_date,
        "ND09A week daily product forecast",
    )

    daily_restaurant_planning = (
        daily_planning.groupby(DATE_COLUMN, as_index=False)
        .agg(
            ProductsForecast=(PRODUCT_ID_COLUMN, "nunique"),
            PredictedRestaurantNormalDemand=(NORMAL_FORECAST_COLUMN, "sum"),
            ConfirmedRestaurantBulkDemand=(BULK_COLUMN, "sum"),
            PlannedRestaurantQuantity=(PLANNED_COLUMN, "sum"),
        )
    )
    daily_restaurant_planning["NormalDemandMethod"] = EXPECTED_DAILY_METHOD
    daily_restaurant_planning["BulkDemandSource"] = "EXTERNALLY_CONFIRMED"

    week_daily_planning["WeekStart"] = pd.to_datetime(
        week_daily_planning.get(
            "WeekStart", pd.Series([min(week_dates)] * len(week_daily_planning))
        ),
        errors="raise",
    ).dt.normalize()
    week_daily_planning["WeekEnd"] = pd.to_datetime(
        week_daily_planning.get(
            "WeekEnd", pd.Series([max(week_dates)] * len(week_daily_planning))
        ),
        errors="raise",
    ).dt.normalize()

    week_product_totals = (
        week_daily_planning.groupby(
            [PRODUCT_ID_COLUMN, PRODUCT_NAME_COLUMN, FAMILY_COLUMN],
            dropna=False,
            as_index=False,
        )
        .agg(
            PredictedWeekNormalDemand=(NORMAL_FORECAST_COLUMN, "sum"),
            ConfirmedWeekBulkDemand=(BULK_COLUMN, "sum"),
            PlannedWeekQuantity=(PLANNED_COLUMN, "sum"),
        )
    )
    week_product_totals["WeekStart"] = min(week_dates)
    week_product_totals["WeekEnd"] = max(week_dates)
    week_product_totals["NormalDemandMethod"] = EXPECTED_WEEK_METHOD
    week_product_totals = week_product_totals.sort_values(
        ["PlannedWeekQuantity", PRODUCT_ID_COLUMN],
        ascending=[False, True],
        kind="mergesort",
    ).reset_index(drop=True)

    week_restaurant_daily = (
        week_daily_planning.groupby(DATE_COLUMN, as_index=False)
        .agg(
            ProductsForecast=(PRODUCT_ID_COLUMN, "nunique"),
            PredictedRestaurantNormalDemand=(NORMAL_FORECAST_COLUMN, "sum"),
            ConfirmedRestaurantBulkDemand=(BULK_COLUMN, "sum"),
            PlannedRestaurantQuantity=(PLANNED_COLUMN, "sum"),
        )
    )
    week_restaurant_daily["WeekStart"] = min(week_dates)
    week_restaurant_daily["WeekEnd"] = max(week_dates)
    week_restaurant_daily["NormalDemandMethod"] = EXPECTED_WEEK_METHOD

    week_restaurant_total = pd.DataFrame(
        [
            {
                "WeekStart": min(week_dates),
                "WeekEnd": max(week_dates),
                "OperatingDatesForecast": len(week_dates),
                "ProductsForecast": active_count,
                "NormalDemandMethod": EXPECTED_WEEK_METHOD,
                "PredictedRestaurantWeekNormalDemand": float(
                    week_daily_planning[NORMAL_FORECAST_COLUMN].sum()
                ),
                "ConfirmedRestaurantWeekBulkDemand": float(
                    week_daily_planning[BULK_COLUMN].sum()
                ),
                "PlannedRestaurantWeekQuantity": float(
                    week_daily_planning[PLANNED_COLUMN].sum()
                ),
            }
        ]
    )

    # -------------------------------------------------------------------------
    # Match and reconciliation audits.
    # -------------------------------------------------------------------------
    forecast_keys = pd.concat(
        [
            daily_forecast[[DATE_COLUMN, PRODUCT_ID_COLUMN]].assign(
                ForecastOutput="ARBITRARY_DATE"
            ),
            week_daily_forecast[[DATE_COLUMN, PRODUCT_ID_COLUMN]].assign(
                ForecastOutput="WEEK_DAILY"
            ),
        ],
        ignore_index=True,
    )
    if bulk_orders.empty:
        bulk_match_audit = pd.DataFrame(
            columns=BULK_TEMPLATE_COLUMNS
            + ["AppearsInArbitraryDateOutput", "AppearsInWeekOutput"]
        )
    else:
        daily_keys = set(
            map(
                tuple,
                daily_forecast[[DATE_COLUMN, PRODUCT_ID_COLUMN]].itertuples(
                    index=False, name=None
                ),
            )
        )
        week_keys = set(
            map(
                tuple,
                week_daily_forecast[[DATE_COLUMN, PRODUCT_ID_COLUMN]].itertuples(
                    index=False, name=None
                ),
            )
        )
        bulk_match_audit = bulk_orders.copy()
        key_tuples = list(
            bulk_match_audit[[DATE_COLUMN, PRODUCT_ID_COLUMN]].itertuples(
                index=False, name=None
            )
        )
        bulk_match_audit["AppearsInArbitraryDateOutput"] = [
            key in daily_keys for key in key_tuples
        ]
        bulk_match_audit["AppearsInWeekOutput"] = [
            key in week_keys for key in key_tuples
        ]

    daily_normal_source = float(daily_forecast[NORMAL_FORECAST_COLUMN].sum())
    daily_normal_output = float(daily_planning[NORMAL_FORECAST_COLUMN].sum())
    daily_bulk_output = float(daily_planning[BULK_COLUMN].sum())
    daily_planned_output = float(daily_planning[PLANNED_COLUMN].sum())

    week_normal_source = float(week_daily_forecast[NORMAL_FORECAST_COLUMN].sum())
    week_normal_output = float(week_daily_planning[NORMAL_FORECAST_COLUMN].sum())
    week_bulk_output = float(week_daily_planning[BULK_COLUMN].sum())
    week_planned_output = float(week_daily_planning[PLANNED_COLUMN].sum())

    reconciliation_records = [
        {
            "Check": "Daily normal forecast preserved",
            "Expected": daily_normal_source,
            "Actual": daily_normal_output,
            "AbsoluteDifference": maximum_abs_difference(
                daily_normal_source, daily_normal_output
            ),
        },
        {
            "Check": "Daily planned quantity equals normal plus bulk",
            "Expected": daily_normal_output + daily_bulk_output,
            "Actual": daily_planned_output,
            "AbsoluteDifference": maximum_abs_difference(
                daily_normal_output + daily_bulk_output,
                daily_planned_output,
            ),
        },
        {
            "Check": "Daily restaurant total equals product bottom-up total",
            "Expected": daily_planned_output,
            "Actual": float(
                daily_restaurant_planning["PlannedRestaurantQuantity"].sum()
            ),
            "AbsoluteDifference": maximum_abs_difference(
                daily_planned_output,
                daily_restaurant_planning["PlannedRestaurantQuantity"].sum(),
            ),
        },
        {
            "Check": "Week normal forecast preserved",
            "Expected": week_normal_source,
            "Actual": week_normal_output,
            "AbsoluteDifference": maximum_abs_difference(
                week_normal_source, week_normal_output
            ),
        },
        {
            "Check": "Week planned quantity equals normal plus bulk",
            "Expected": week_normal_output + week_bulk_output,
            "Actual": week_planned_output,
            "AbsoluteDifference": maximum_abs_difference(
                week_normal_output + week_bulk_output,
                week_planned_output,
            ),
        },
        {
            "Check": "Week product totals reconcile to daily product rows",
            "Expected": week_planned_output,
            "Actual": float(week_product_totals["PlannedWeekQuantity"].sum()),
            "AbsoluteDifference": maximum_abs_difference(
                week_planned_output,
                week_product_totals["PlannedWeekQuantity"].sum(),
            ),
        },
        {
            "Check": "Week restaurant daily totals reconcile",
            "Expected": week_planned_output,
            "Actual": float(
                week_restaurant_daily["PlannedRestaurantQuantity"].sum()
            ),
            "AbsoluteDifference": maximum_abs_difference(
                week_planned_output,
                week_restaurant_daily["PlannedRestaurantQuantity"].sum(),
            ),
        },
        {
            "Check": "Week restaurant total reconciles",
            "Expected": week_planned_output,
            "Actual": float(
                week_restaurant_total["PlannedRestaurantWeekQuantity"].iloc[0]
            ),
            "AbsoluteDifference": maximum_abs_difference(
                week_planned_output,
                week_restaurant_total["PlannedRestaurantWeekQuantity"].iloc[0],
            ),
        },
    ]
    reconciliation_audit = pd.DataFrame(reconciliation_records)
    reconciliation_audit["Passed"] = (
        reconciliation_audit["AbsoluteDifference"] <= 1e-9
    )
    if not reconciliation_audit["Passed"].all():
        raise AssertionError(
            "ND10A planning reconciliation failed:\n"
            + reconciliation_audit.loc[
                ~reconciliation_audit["Passed"]
            ].to_string(index=False)
        )

    bulk_total_input = float(bulk_orders[BULK_COLUMN].sum()) if not bulk_orders.empty else 0.0
    bulk_total_covered_unique = float(bulk_by_product_date[BULK_COLUMN].sum()) if not bulk_by_product_date.empty else 0.0
    if maximum_abs_difference(bulk_total_input, bulk_total_covered_unique) > 1e-9:
        raise AssertionError("Bulk-order line total does not reconcile to aggregated bulk total.")

    daily_date = pd.Timestamp(daily_forecast[DATE_COLUMN].iloc[0]).normalize()
    daily_bulk_total = float(
        bulk_by_product_date.loc[
            bulk_by_product_date[DATE_COLUMN] == daily_date, BULK_COLUMN
        ].sum()
    ) if not bulk_by_product_date.empty else 0.0
    week_bulk_total = float(
        bulk_by_product_date.loc[
            bulk_by_product_date[DATE_COLUMN].isin(set(week_dates)), BULK_COLUMN
        ].sum()
    ) if not bulk_by_product_date.empty else 0.0

    demonstration_summary = pd.DataFrame(
        [
            {
                "PlanningView": "ARBITRARY_DATE",
                "StartDate": daily_date,
                "EndDate": daily_date,
                "NormalDemandMethod": EXPECTED_DAILY_METHOD,
                "PredictedNormalDemand": daily_normal_output,
                "ConfirmedBulkDemand": daily_bulk_output,
                "PlannedQuantity": daily_planned_output,
                "ProductsForecast": active_count,
            },
            {
                "PlanningView": "MONDAY_ORIGIN_WEEK",
                "StartDate": min(week_dates),
                "EndDate": max(week_dates),
                "NormalDemandMethod": EXPECTED_WEEK_METHOD,
                "PredictedNormalDemand": week_normal_output,
                "ConfirmedBulkDemand": week_bulk_output,
                "PlannedQuantity": week_planned_output,
                "ProductsForecast": active_count,
            },
        ]
    )

    # -------------------------------------------------------------------------
    # Stage outputs.
    # -------------------------------------------------------------------------
    staged_input_dir = STAGING_ROOT / "01_inputs"
    staged_output_dir = STAGING_ROOT / "02_planned_outputs"
    staged_contract_dir = STAGING_ROOT / "03_contracts"
    staged_audit_dir = STAGING_ROOT / "04_audits"
    staged_figure_dir = STAGING_ROOT / "05_figures"
    staged_report_dir = STAGING_ROOT / "06_reports"
    staged_control_dir = STAGING_ROOT / "07_control"
    for directory in [
        staged_input_dir,
        staged_output_dir,
        staged_contract_dir,
        staged_audit_dir,
        staged_figure_dir,
        staged_report_dir,
        staged_control_dir,
    ]:
        directory.mkdir(parents=True, exist_ok=True)

    write_csv(staged_input_dir / BULK_NORMALIZED_PATH.name, bulk_orders)
    write_csv(staged_input_dir / BULK_AGGREGATED_PATH.name, bulk_by_product_date)
    write_csv(staged_contract_dir / BULK_TEMPLATE_PATH.name, empty_bulk_frame())

    write_csv(staged_output_dir / DAILY_PRODUCT_PLANNING_PATH.name, daily_planning)
    write_csv(
        staged_output_dir / DAILY_RESTAURANT_PLANNING_PATH.name,
        daily_restaurant_planning,
    )
    write_csv(
        staged_output_dir / WEEK_DAILY_PRODUCT_PLANNING_PATH.name,
        week_daily_planning,
    )
    write_csv(
        staged_output_dir / WEEK_PRODUCT_TOTAL_PLANNING_PATH.name,
        week_product_totals,
    )
    write_csv(
        staged_output_dir / WEEK_RESTAURANT_DAILY_PLANNING_PATH.name,
        week_restaurant_daily,
    )
    write_csv(
        staged_output_dir / WEEK_RESTAURANT_TOTAL_PLANNING_PATH.name,
        week_restaurant_total,
    )
    write_csv(
        staged_output_dir / DEMONSTRATION_SUMMARY_PATH.name,
        demonstration_summary,
    )

    input_hash_records = [
        {
            "Input": str(path),
            "SHA256Before": protected_input_hashes_before[str(path)],
            "InputRole": "ND09A_PROTECTED_INPUT",
        }
        for path in required_inputs
    ]
    if bulk_input_exists:
        input_hash_records.append(
            {
                "Input": str(CONFIRMED_BULK_INPUT_PATH),
                "SHA256Before": protected_input_hashes_before[
                    str(CONFIRMED_BULK_INPUT_PATH)
                ],
                "InputRole": "USER_CONFIRMED_BULK_INPUT",
            }
        )
    write_csv(
        staged_audit_dir / INPUT_HASH_AUDIT_PATH.name,
        pd.DataFrame(input_hash_records),
    )
    write_csv(
        staged_audit_dir / BULK_VALIDATION_AUDIT_PATH.name,
        bulk_validation_audit,
    )
    write_csv(
        staged_audit_dir / BULK_MATCH_AUDIT_PATH.name,
        bulk_match_audit,
    )
    write_csv(
        staged_audit_dir / RECONCILIATION_AUDIT_PATH.name,
        reconciliation_audit,
    )

    validation = pd.DataFrame(
        [
            {
                "Check": "ND09A checkpoint hash matches",
                "Passed": actual_nd09_checkpoint_hash
                == EXPECTED_ND09A_CHECKPOINT_SHA256,
            },
            {
                "Check": "ND09A active-catalogue scope is authoritative",
                "Passed": (
                    str(nd09_lock.get("Status", ""))
                    == "AUTHORITATIVE_FOR_DEMONSTRATION_PRODUCT_SCOPE"
                    and str(nd09_lock.get("CataloguePolicy", ""))
                    == "LATEST_KNOWN_ACTIVE_PANEL"
                    and int(nd09_lock.get("ActiveProductsAtCutoff", -1)) == 88
                ),
            },
            {
                "Check": "ND09A target is NormalDemand",
                "Passed": inference_contract.get("Target") == EXPECTED_TARGET,
            },
            {
                "Check": "ND09A catalogue policy is latest known active panel",
                "Passed": (
                    inference_contract.get("CataloguePolicy", {}).get("Name")
                    == "LATEST_KNOWN_ACTIVE_PANEL"
                ),
            },
            {
                "Check": "ND09A forecast catalogue contains 88 products",
                "Passed": (
                    int(inference_contract.get("ForecastProducts", -1))
                    == EXPECTED_ACTIVE_PRODUCTS
                ),
            },
            {
                "Check": "ND09A forecasting models were not refitted",
                "Passed": not bool(
                    inference_contract.get("ForecastingModelsRefitted", True)
                ),
            },
            {
                "Check": "March target vault remains closed",
                "Passed": not bool(
                    inference_contract.get("March2026TargetVaultOpened", True)
                ),
            },
            {
                "Check": "Daily output contains all active products",
                "Passed": len(daily_planning) == active_count,
            },
            {
                "Check": "Week output contains all active products per date",
                "Passed": bool(
                    (
                        week_daily_planning.groupby(DATE_COLUMN)[
                            PRODUCT_ID_COLUMN
                        ].nunique()
                        == active_count
                    ).all()
                ),
            },
            {
                "Check": "All bulk-order validation checks passed",
                "Passed": bool(
                    bulk_validation_audit.loc[
                        bulk_validation_audit["Check"]
                        != "Bulk-order input file exists",
                        "Passed",
                    ].all()
                ),
            },
            {
                "Check": "All planning reconciliations passed",
                "Passed": bool(reconciliation_audit["Passed"].all()),
            },
            {
                "Check": "No model refit or reselection performed",
                "Passed": True,
            },
            {
                "Check": "Normal, bulk and planned quantities remain separate",
                "Passed": all(
                    column in daily_planning.columns
                    for column in [
                        NORMAL_FORECAST_COLUMN,
                        BULK_COLUMN,
                        PLANNED_COLUMN,
                    ]
                ),
            },
        ]
    )
    if not validation["Passed"].all():
        raise AssertionError(
            "ND10A validation failed:\n"
            + validation.loc[~validation["Passed"]].to_string(index=False)
        )
    write_csv(staged_audit_dir / VALIDATION_PATH.name, validation)

    package_versions = pd.DataFrame(
        [
            {"Package": "python", "Version": platform.python_version()},
            {"Package": "pandas", "Version": pd.__version__},
            {"Package": "numpy", "Version": np.__version__},
        ]
    )
    write_csv(
        staged_audit_dir / PACKAGE_VERSIONS_PATH.name,
        package_versions,
    )

    planning_contract = {
        "StepID": STEP_ID,
        "Status": STATUS,
        "PlanningFormula": {
            "PlannedQuantity": (
                "PredictedNormalDemand + ConfirmedBulkDemand"
            ),
            "NormalDemandSource": "ND09A_CORRECTED_ACTIVE_CATALOGUE_INFERENCE",
            "ConfirmedBulkDemandSource": "EXTERNAL_CONFIRMED_INPUT_ONLY",
        },
        "ND09ACheckpointSHA256": actual_nd09_checkpoint_hash,
        "CataloguePolicy": "LATEST_KNOWN_ACTIVE_PANEL",
        "ForecastCatalogueProducts": active_count,
        "DailyPlanning": {
            "Date": daily_date,
            "NormalDemandMethod": EXPECTED_DAILY_METHOD,
            "Products": active_count,
            "PredictedNormalDemand": daily_normal_output,
            "ConfirmedBulkDemand": daily_bulk_output,
            "PlannedQuantity": daily_planned_output,
        },
        "WeekPlanning": {
            "WeekStart": min(week_dates),
            "WeekEnd": max(week_dates),
            "OperatingDates": len(week_dates),
            "NormalDemandMethod": EXPECTED_WEEK_METHOD,
            "Products": active_count,
            "PredictedNormalDemand": week_normal_output,
            "ConfirmedBulkDemand": week_bulk_output,
            "PlannedQuantity": week_planned_output,
        },
        "BulkInput": {
            "ConfiguredPath": str(CONFIRMED_BULK_INPUT_PATH),
            "InputFileFound": bulk_input_exists,
            "AcceptedLines": int(len(bulk_orders)),
            "UniqueProductDateCombinations": int(len(bulk_by_product_date)),
            "TotalConfirmedUnits": bulk_total_input,
            "RequiredColumns": sorted(BULK_REQUIRED_COLUMNS),
            "OptionalColumns": BULK_OPTIONAL_COLUMNS,
            "OutOfForecastRangeAllowed": ALLOW_OUT_OF_FORECAST_RANGE_BULK_ORDERS,
        },
        "Safety": {
            "ModelRefitted": False,
            "ModelReselected": False,
            "March2026TargetsOpened": False,
            "PreviousInputsModified": False,
            "AccuracyMetricReported": False,
        },
        "NextStep": "ND11_INGREDIENT_MAPPING_AND_DEMONSTRATION_EXPORT",
    }
    write_json(
        staged_contract_dir / PLANNING_CONTRACT_PATH.name,
        planning_contract,
    )

    contract_md = f"""# ND10A Planning Quantity Contract

## Formula

`PlannedQuantity = PredictedNormalDemand + ConfirmedBulkDemand`

The model estimates normal recurring demand only. Confirmed bulk demand is supplied externally and is never inferred from ordinary demand patterns.

## Daily plan

- Date: {daily_date.date()}
- Normal-demand method: `{EXPECTED_DAILY_METHOD}`
- Predicted normal demand: {daily_normal_output:.6f}
- Confirmed bulk demand: {daily_bulk_output:.6f}
- Planned quantity: {daily_planned_output:.6f}

## Week plan

- Week: {pd.Timestamp(min(week_dates)).date()} to {pd.Timestamp(max(week_dates)).date()}
- Normal-demand method: `{EXPECTED_WEEK_METHOD}`
- Predicted normal demand: {week_normal_output:.6f}
- Confirmed bulk demand: {week_bulk_output:.6f}
- Planned quantity: {week_planned_output:.6f}

## Operational rule

Only confirmed orders are added. Possible, provisional, or unconfirmed enquiries must not be included in `ConfirmedBulkDemand`.

## Evaluation boundary

ND10A creates planning quantities and does not calculate forecast accuracy. March 2026 actual targets remain closed.
"""
    write_text(
        staged_contract_dir / PLANNING_CONTRACT_MD_PATH.name,
        contract_md,
    )

    usage_text = f"""# ND10A Confirmed Bulk-Order Input

Create a CSV at:

`{CONFIRMED_BULK_INPUT_PATH}`

Required columns:

- `Date`
- `CanonicalProductID`
- `ConfirmedBulkDemand`

Optional columns:

- `OrderReference`
- `CustomerOrEvent`
- `Notes`

Quantities must be nonnegative whole units. Dates must be present in the current ND09A arbitrary-date or week forecast. Multiple confirmed lines for the same product and date are added together.

When the input changes, rerun ND10A with `ALLOW_OVERWRITE=True`. The ND09A model artifacts remain unchanged.
"""
    write_text(staged_contract_dir / USAGE_PATH.name, usage_text)

    # Figures
    daily_top = daily_planning.nlargest(20, PLANNED_COLUMN).sort_values(PLANNED_COLUMN)
    plt.figure(figsize=(10, 7))
    plt.barh(daily_top[PRODUCT_NAME_COLUMN].astype(str), daily_top[PLANNED_COLUMN])
    plt.xlabel("Planned units")
    plt.ylabel("Product")
    plt.title(f"ND10A daily planned quantities — {daily_date.date()}")
    save_figure(staged_figure_dir / "ND10A_figure_01_daily_planned_products.png")

    week_top = week_product_totals.nlargest(20, "PlannedWeekQuantity").sort_values(
        "PlannedWeekQuantity"
    )
    plt.figure(figsize=(10, 7))
    plt.barh(
        week_top[PRODUCT_NAME_COLUMN].astype(str),
        week_top["PlannedWeekQuantity"],
    )
    plt.xlabel("Planned week units")
    plt.ylabel("Product")
    plt.title("ND10A top weekly planned quantities")
    save_figure(staged_figure_dir / "ND10A_figure_02_week_planned_products.png")

    plt.figure(figsize=(9, 5))
    plt.plot(
        week_restaurant_daily[DATE_COLUMN],
        week_restaurant_daily["PredictedRestaurantNormalDemand"],
        marker="o",
        label="Normal demand",
    )
    plt.plot(
        week_restaurant_daily[DATE_COLUMN],
        week_restaurant_daily["PlannedRestaurantQuantity"],
        marker="o",
        label="Planned quantity",
    )
    plt.xlabel("Date")
    plt.ylabel("Restaurant units")
    plt.title("ND10A restaurant normal demand and planned quantity")
    plt.xticks(rotation=30)
    plt.legend()
    save_figure(staged_figure_dir / "ND10A_figure_03_restaurant_plan.png")

    report_text = f"""# ND10A Confirmed Bulk and Planned-Quantity Integration

## Status

`{STATUS}`

## Purpose

ND10A converts the corrected 88-product ND09A normal-demand forecasts into final operational planning quantities. Confirmed bulk orders are added externally rather than predicted by the demand model.

## Formula

`PlannedQuantity = PredictedNormalDemand + ConfirmedBulkDemand`

## Input result

- Bulk input file found: {bulk_input_exists}
- Accepted bulk-order lines: {len(bulk_orders)}
- Unique product-date bulk combinations: {len(bulk_by_product_date)}
- Total confirmed bulk units in the input: {bulk_total_input:.6f}

## Daily plan

- Date: {daily_date.date()}
- Products: {active_count}
- Predicted normal demand: {daily_normal_output:.6f}
- Confirmed bulk demand: {daily_bulk_output:.6f}
- Planned quantity: {daily_planned_output:.6f}

## Week plan

- Dates: {pd.Timestamp(min(week_dates)).date()} to {pd.Timestamp(max(week_dates)).date()}
- Operating dates: {len(week_dates)}
- Predicted normal demand: {week_normal_output:.6f}
- Confirmed bulk demand: {week_bulk_output:.6f}
- Planned quantity: {week_planned_output:.6f}

## Validation

All product, daily restaurant, product-week and restaurant-week totals reconcile exactly. ND09A corrected active-catalogue forecasts were preserved without modification. No model was refitted or reselected, and March targets were not opened.

## Next step

ND11 will connect planned product quantities to ingredient requirements and prepare the final demonstration export package.
"""
    write_text(staged_report_dir / REPORT_SUMMARY_PATH.name, report_text)

    readme_text = f"""# ND10A Confirmed Bulk and Planned Quantities

This folder combines the corrected 88-product ND09A normal-demand forecasts with externally confirmed bulk orders.

Formula: `PlannedQuantity = PredictedNormalDemand + ConfirmedBulkDemand`

Status: `{STATUS}`
"""
    write_text(STAGING_ROOT / README_PATH.name, readme_text)

    manifest = build_manifest(STAGING_ROOT, exclude_control=True)
    write_csv(staged_control_dir / MANIFEST_PATH.name, manifest)
    manifest_hash = sha256_file(staged_control_dir / MANIFEST_PATH.name)

    checkpoint_payload = {
        "StepID": STEP_ID,
        "Status": STATUS,
        "CompletedLocalTime": NOW_LOCAL.isoformat(),
        "ND10ARoot": str(ND10A_ROOT),
        "ND09ACheckpointSHA256": actual_nd09_checkpoint_hash,
        "ArtifactManifestSHA256": manifest_hash,
        "BulkInputFileFound": bulk_input_exists,
        "AcceptedBulkOrderLines": int(len(bulk_orders)),
        "ConfirmedBulkUnitsInput": bulk_total_input,
        "DailyDate": daily_date,
        "DailyPredictedNormalDemand": daily_normal_output,
        "DailyConfirmedBulkDemand": daily_bulk_output,
        "DailyPlannedQuantity": daily_planned_output,
        "WeekStart": min(week_dates),
        "WeekEnd": max(week_dates),
        "WeekPredictedNormalDemand": week_normal_output,
        "WeekConfirmedBulkDemand": week_bulk_output,
        "WeekPlannedQuantity": week_planned_output,
        "MarchTargetVaultOpened": False,
        "ModelRefitted": False,
        "ModelReselected": False,
        "NextStep": "ND11",
    }
    staged_checkpoint_path = staged_control_dir / CHECKPOINT_PATH.name
    write_json(staged_checkpoint_path, checkpoint_payload)
    checkpoint_hash = sha256_file(staged_checkpoint_path)
    write_text(
        staged_control_dir / CHECKPOINT_SHA_PATH.name,
        checkpoint_hash + "\n",
    )

    protected_input_hashes_after = {str(path): sha256_file(path) for path in required_inputs}
    if bulk_input_exists:
        protected_input_hashes_after[str(CONFIRMED_BULK_INPUT_PATH)] = sha256_file(
            CONFIRMED_BULK_INPUT_PATH
        )
    changed_inputs = [
        path
        for path in protected_input_hashes_before
        if protected_input_hashes_before[path]
        != protected_input_hashes_after.get(path)
    ]
    if changed_inputs:
        raise AssertionError(
            "Protected inputs changed during ND10A:\n"
            + "\n".join(f"- {path}" for path in changed_inputs)
        )

    os.replace(STAGING_ROOT, ND10A_ROOT)

    TOP_LEVEL_CHECKPOINT_PATH.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(
        ND10A_ROOT / "07_control" / CHECKPOINT_PATH.name,
        TOP_LEVEL_CHECKPOINT_PATH,
    )
    shutil.copy2(
        ND10A_ROOT / "07_control" / CHECKPOINT_SHA_PATH.name,
        TOP_LEVEL_CHECKPOINT_SHA_PATH,
    )

    handoff_text = f"""# ND10A Handoff

## Status

- Completed step: `{STEP_ID}`
- Status: `{STATUS}`
- Root: `{ND10A_ROOT}`
- Checkpoint SHA-256: `{checkpoint_hash}`

## Planning formula

`PlannedQuantity = PredictedNormalDemand + ConfirmedBulkDemand`

## Current outputs

- Daily date: {daily_date.date()}
- Daily planned quantity: {daily_planned_output:.6f}
- Week: {pd.Timestamp(min(week_dates)).date()} to {pd.Timestamp(max(week_dates)).date()}
- Week planned quantity: {week_planned_output:.6f}
- Confirmed bulk input file found: {bulk_input_exists}
- Confirmed bulk units loaded: {bulk_total_input:.6f}

## Safety

- ND09A model refitted: no
- Model reselected: no
- March targets opened: no
- Previous inputs modified: no

## Next step

ND11 — ingredient mapping and demonstration-ready export.
"""
    atomic_write_text(ND10A_HANDOFF_PATH, handoff_text)
    atomic_write_text(CURRENT_HANDOFF_PATH, handoff_text)

    append_marked_section(
        WORKFLOW_PATH,
        "## ND10A — Corrected active-catalogue planning quantities",
        f"""## ND10A — Corrected active-catalogue planning quantities

Status: `{STATUS}`

The corrected 88-product ND09A normal-demand forecasts were combined with externally confirmed bulk orders. Normal, bulk and planned quantities remain separate and reconcile at product, day, week and restaurant levels.
""",
    )
    append_marked_section(
        DECISIONS_PATH,
        "## ND10A decisions",
        f"""## ND10A decisions

- Preserve the corrected ND09A active-catalogue normal-demand forecast without alteration.
- Add only externally confirmed bulk demand.
- Use `PlannedQuantity = PredictedNormalDemand + ConfirmedBulkDemand`.
- Do not include provisional or unconfirmed orders.
- Do not calculate accuracy from planning outputs.
""",
    )
    append_marked_section(
        METRICS_AND_RESULTS_PATH,
        "## ND10A planning outputs",
        f"""## ND10A planning outputs

- Daily normal demand: {daily_normal_output:.6f}
- Daily confirmed bulk: {daily_bulk_output:.6f}
- Daily planned quantity: {daily_planned_output:.6f}
- Week normal demand: {week_normal_output:.6f}
- Week confirmed bulk: {week_bulk_output:.6f}
- Week planned quantity: {week_planned_output:.6f}
- These are planning quantities, not accuracy metrics.
""",
    )
    append_marked_section(
        AGENTS_PATH,
        "Marker: ND10A_AUTHORITATIVE_STATUS",
        f"""## ND10A authoritative status

Marker: ND10A_AUTHORITATIVE_STATUS

- Status: `{STATUS}`
- Handoff: `{ND10A_HANDOFF_PATH}`
- Checkpoint SHA-256: `{checkpoint_hash}`
- Next step: ND11 ingredient mapping and demonstration export.
""",
    )

    LOG_PATH.parent.mkdir(parents=True, exist_ok=True)
    with LOG_PATH.open("a", encoding="utf-8") as log_handle:
        log_handle.write(
            f"{NOW_LOCAL.isoformat()} | {STATUS} | checkpoint={checkpoint_hash} | "
            f"bulk_units={bulk_total_input:.6f} | root={ND10A_ROOT}\n"
        )

except Exception:
    if STAGING_ROOT.exists():
        shutil.rmtree(STAGING_ROOT, ignore_errors=True)
    raise


# =============================================================================
# FINAL CONSOLE OUTPUT
# =============================================================================

print("=" * 118)
print("EDEN NORMAL-DEMAND MODEL V2 — ND10A CORRECTED PLANNING COMPLETE")
print("=" * 118)
print(f"Status: {STATUS}")
print(f"Local time: {NOW_LOCAL.isoformat()}")
print(f"ND10A root: {ND10A_ROOT}")
print()
print("INPUT VERIFICATION")
print(f"ND09A checkpoint SHA-256: {actual_nd09_checkpoint_hash}")
print(f"ND09A active-catalogue scope lock present: True")
print(f"Active products: {active_count}")
print(f"Normal-demand target: {inference_contract['Target']}")
print(f"March target vault opened: False")
print(f"ND09A corrected inference artifacts modified: False")
print()
print("CONFIRMED BULK INPUT")
print(f"Configured input path: {CONFIRMED_BULK_INPUT_PATH}")
print(f"Input file found: {bulk_input_exists}")
print(f"Accepted order lines: {len(bulk_orders):,}")
print(f"Unique product-date combinations: {len(bulk_by_product_date):,}")
print(f"Total confirmed bulk units loaded: {bulk_total_input:.6f}")
if not bulk_input_exists:
    print("No input file was found; zero confirmed bulk was applied.")
    print(f"Input template saved to: {BULK_TEMPLATE_PATH}")
print()
print("DAILY PLANNING OUTPUT")
print(f"Date: {daily_date.date()}")
print(f"Normal-demand method: {EXPECTED_DAILY_METHOD}")
print(f"Products planned: {active_count:,}")
print(f"Predicted restaurant normal demand: {daily_normal_output:.6f}")
print(f"Confirmed restaurant bulk demand: {daily_bulk_output:.6f}")
print(f"Planned restaurant quantity: {daily_planned_output:.6f}")
print()
print("MONDAY-ORIGIN WEEK PLANNING OUTPUT")
print(f"Week start: {pd.Timestamp(min(week_dates)).date()}")
print(f"Week end: {pd.Timestamp(max(week_dates)).date()}")
print(f"Operating dates: {len(week_dates)}")
print(f"Normal-demand method: {EXPECTED_WEEK_METHOD}")
print(f"Predicted restaurant week normal demand: {week_normal_output:.6f}")
print(f"Confirmed restaurant week bulk demand: {week_bulk_output:.6f}")
print(f"Planned restaurant week quantity: {week_planned_output:.6f}")
print()
print("RECONCILIATION")
print(f"Checks passed: {int(reconciliation_audit['Passed'].sum())}/{len(reconciliation_audit)}")
print(
    "Maximum absolute reconciliation difference: "
    f"{reconciliation_audit['AbsoluteDifference'].max():.12g}"
)
print()
print("OUTPUTS")
print(f"- Daily product planning: {DAILY_PRODUCT_PLANNING_PATH}")
print(f"- Daily restaurant planning: {DAILY_RESTAURANT_PLANNING_PATH}")
print(f"- Week daily product planning: {WEEK_DAILY_PRODUCT_PLANNING_PATH}")
print(f"- Week product totals planning: {WEEK_PRODUCT_TOTAL_PLANNING_PATH}")
print(f"- Week restaurant daily planning: {WEEK_RESTAURANT_DAILY_PLANNING_PATH}")
print(f"- Week restaurant total planning: {WEEK_RESTAURANT_TOTAL_PLANNING_PATH}")
print(f"- Demonstration summary: {DEMONSTRATION_SUMMARY_PATH}")
print(f"- Planning contract: {PLANNING_CONTRACT_PATH}")
print(f"- Report: {REPORT_SUMMARY_PATH}")
print(f"- Checkpoint: {TOP_LEVEL_CHECKPOINT_PATH}")
print(f"- Checkpoint SHA-256: {checkpoint_hash}")
print(f"- Handoff: {ND10A_HANDOFF_PATH}")
print()
print("SAFETY")
print("- Forecasting models refitted: False")
print("- Forecasting methods reselected: False")
print("- March target vault opened: False")
print("- Previous inputs modified: False")
print("- Accuracy metric calculated: False")
print("- ND10A checkpoint and hashes created: True")
print()
print("NEXT STEP")
print("ND11 — ingredient mapping and demonstration-ready export.")
print("=" * 118)

/var/folders/61/pw_dwqt140ndz64pvx21l9040000gn/T/ipykernel_17958/3600775860.py:523: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  output[BULK_COLUMN] = output[BULK_COLUMN].fillna(0).astype(float)
/var/folders/61/pw_dwqt140ndz64pvx21l9040000gn/T/ipykernel_17958/3600775860.py:523: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  output[BULK_COLUMN] = output[BULK_COLUMN].fillna(0).astype(float)


EDEN NORMAL-DEMAND MODEL V2 — ND10A CORRECTED PLANNING COMPLETE
Status: ND10A_CORRECTED_PLANNING_QUANTITIES_CREATED_READY_FOR_ND10R_DEMO_REFRESH
Local time: 2026-08-10T12:47:13.515382+01:00
ND10A root: /Users/ryansmac/Desktop/Meng Project/eden_datasets/eden_normal_demand_model_v2/04_planning/ND10A_corrected_active_catalogue_planning

INPUT VERIFICATION
ND09A checkpoint SHA-256: 54767758d9052cb1571043d2f39cc167bc7491b300afbd3605a5416b7aa95c10
ND09A active-catalogue scope lock present: True
Active products: 88
Normal-demand target: NormalDemand
March target vault opened: False
ND09A corrected inference artifacts modified: False

CONFIRMED BULK INPUT
Configured input path: /Users/ryansmac/Desktop/Meng Project/eden_datasets/eden_normal_demand_model_v2/04_planning_inputs/ND10A_confirmed_bulk_orders.csv
Input file found: False
Accepted order lines: 0
Unique product-date combinations: 0
Total confirmed bulk units loaded: 0.000000
No input file was found; zero confirmed bulk was applied.
Input

In [19]:
# =============================================================================
# EDEN NORMAL-DEMAND MODEL V2
# ND10RA — CORRECTED DEMONSTRATION REPORTING REFRESH
#
# Run this as one complete Jupyter cell after the completed ND10A correction.
#
# PURPOSE
# -------
# Refresh ONLY the demonstration-specific report tables and figures that depend
# on the active forecast catalogue. The model-evaluation diagnostics and
# accuracy figures created in ND10R remain authoritative and are not recomputed.
#
# This step:
#   - binds to the exact ND10R and ND10A checkpoints;
#   - reads the corrected 88-product ND10A planning outputs;
#   - creates corrected demonstration figures/tables;
#   - records old-vs-corrected totals;
#   - creates report wording for the catalogue correction;
#   - does NOT fit/refit/reselect any model;
#   - does NOT open March 2026 actual targets.
# =============================================================================

import hashlib
import json
import os
import platform
import shutil
import uuid
from datetime import datetime, timezone
from pathlib import Path
from zoneinfo import ZoneInfo

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd


# =============================================================================
# USER CONFIGURATION
# =============================================================================

ALLOW_OVERWRITE = False


# =============================================================================
# FIXED PROJECT CONFIGURATION
# =============================================================================

PROJECT_ROOT = Path("/Users/ryansmac/Desktop/Meng Project")
EDEN_ROOT = PROJECT_ROOT / "eden_datasets"
MODEL_ROOT = EDEN_ROOT / "eden_normal_demand_model_v2"

ND10R_ROOT = (
    MODEL_ROOT
    / "05_reporting"
    / "ND10R_report_ready_diagnostics_and_visuals"
)
ND10A_ROOT = (
    MODEL_ROOT
    / "04_planning"
    / "ND10A_corrected_active_catalogue_planning"
)

ND10R_CHECKPOINT_PATH = MODEL_ROOT / "08_checkpoints" / "ND10R_checkpoint.json"
ND10A_CHECKPOINT_PATH = MODEL_ROOT / "08_checkpoints" / "ND10A_checkpoint.json"

EXPECTED_ND10R_CHECKPOINT_SHA256 = (
    "04726dc6131ec92f5da86517b395afa8b624ed2ef2755d7aed82840da697f350"
)
EXPECTED_ND10A_CHECKPOINT_SHA256 = (
    "6bcc97c5fe76667c2bdcdde7d0e81b4bd1534c3a601f39fd7acd795e6c87dbff"
)

ND10A_OUTPUT_DIR = ND10A_ROOT / "02_planned_outputs"
DAILY_PRODUCT_PATH = ND10A_OUTPUT_DIR / "ND10A_daily_product_planning.csv"
DAILY_RESTAURANT_PATH = ND10A_OUTPUT_DIR / "ND10A_daily_restaurant_planning.csv"
WEEK_DAILY_PRODUCT_PATH = (
    ND10A_OUTPUT_DIR / "ND10A_week_daily_product_planning.csv"
)
WEEK_PRODUCT_TOTALS_PATH = (
    ND10A_OUTPUT_DIR / "ND10A_week_product_totals_planning.csv"
)
WEEK_RESTAURANT_DAILY_PATH = (
    ND10A_OUTPUT_DIR / "ND10A_week_restaurant_daily_planning.csv"
)
WEEK_RESTAURANT_TOTAL_PATH = (
    ND10A_OUTPUT_DIR / "ND10A_week_restaurant_total_planning.csv"
)
DEMO_SUMMARY_PATH = (
    ND10A_OUTPUT_DIR / "ND10A_demonstration_planning_summary.csv"
)

# Authoritative old ND10R report package remains unchanged.
ND10R_RESULTS_WORDING_PATH = (
    ND10R_ROOT
    / "04_report_wording"
    / "ND10R_report_ready_results_wording.md"
)
ND10R_RECOMMENDED_FIGURES_PATH = (
    ND10R_ROOT
    / "04_report_wording"
    / "ND10R_recommended_main_report_figures.csv"
)

ND10RA_ROOT = (
    MODEL_ROOT
    / "05_reporting"
    / "ND10RA_corrected_demo_reporting_refresh"
)
TABLE_DIR = ND10RA_ROOT / "01_tables"
FIGURE_DIR = ND10RA_ROOT / "02_figures"
WORDING_DIR = ND10RA_ROOT / "03_report_wording"
DEMO_DIR = ND10RA_ROOT / "04_demonstration_material"
CONTROL_DIR = ND10RA_ROOT / "05_control"

CORRECTED_SUMMARY_PATH = TABLE_DIR / "ND10RA_corrected_demonstration_summary.csv"
SCOPE_COMPARISON_PATH = TABLE_DIR / "ND10RA_old_vs_corrected_scope_comparison.csv"
TOP_DAILY_PATH = TABLE_DIR / "ND10RA_top_daily_products.csv"
TOP_WEEK_PATH = TABLE_DIR / "ND10RA_top_week_products.csv"
FIGURE_CAPTIONS_PATH = WORDING_DIR / "ND10RA_figure_captions.csv"
REPORT_WORDING_PATH = WORDING_DIR / "ND10RA_report_wording.md"
README_PATH = ND10RA_ROOT / "README.md"

CHECKPOINT_PATH = CONTROL_DIR / "ND10RA_checkpoint.json"
CHECKPOINT_SHA_PATH = CONTROL_DIR / "ND10RA_checkpoint.sha256"
MANIFEST_PATH = CONTROL_DIR / "ND10RA_artifact_hash_manifest.csv"
ZIP_PATH = CONTROL_DIR / "ND10RA_corrected_demo_reporting_bundle.zip"

TOP_LEVEL_CHECKPOINT_PATH = MODEL_ROOT / "08_checkpoints" / "ND10RA_checkpoint.json"
TOP_LEVEL_CHECKPOINT_SHA_PATH = (
    MODEL_ROOT / "08_checkpoints" / "ND10RA_checkpoint.sha256"
)

MEMORY_ROOT = MODEL_ROOT / "00_project_memory"
HANDOFF_PATH = MEMORY_ROOT / "ND10RA_HANDOFF.md"
CURRENT_HANDOFF_PATH = MEMORY_ROOT / "CURRENT_HANDOFF.md"
WORKFLOW_PATH = MEMORY_ROOT / "WORKFLOW.md"
DECISIONS_PATH = MEMORY_ROOT / "DECISIONS.md"
METRICS_AND_RESULTS_PATH = MEMORY_ROOT / "METRICS_AND_RESULTS.md"
AGENTS_PATH = MODEL_ROOT / "AGENTS.md"
LOG_PATH = MODEL_ROOT / "09_logs" / "ND10RA_demo_reporting_refresh_log.txt"

STEP_ID = "ND10RA"
STATUS = "ND10RA_CORRECTED_DEMONSTRATION_REPORTING_REFRESH_COMPLETED_READY_FOR_ND11"
NOW_UTC = datetime.now(timezone.utc)
NOW_LOCAL = NOW_UTC.astimezone(ZoneInfo("Europe/Dublin"))

EXPECTED_ACTIVE_PRODUCTS = 88

# Old demonstration values from the completed 227-product ND10/ND10R package.
OLD_ACTIVE_PRODUCTS = 227
OLD_DAILY_RESTAURANT_NORMAL_DEMAND = 1139.224575
OLD_WEEK_RESTAURANT_NORMAL_DEMAND = 5302.070080


# =============================================================================
# HELPERS
# =============================================================================

def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


def write_csv(path: Path, frame: pd.DataFrame) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    frame.to_csv(path, index=False)


def write_json(path: Path, payload: dict) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(
        json.dumps(payload, indent=2, ensure_ascii=False, default=str) + "\n",
        encoding="utf-8",
    )


def write_text(path: Path, text: str) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(text, encoding="utf-8")


def atomic_write_text(path: Path, text: str) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_name(f".{path.name}.{uuid.uuid4().hex}.tmp")
    temporary.write_text(text, encoding="utf-8")
    os.replace(temporary, path)


def append_marked_section(path: Path, marker: str, section_text: str) -> None:
    existing = path.read_text(encoding="utf-8") if path.is_file() else ""
    if marker in existing:
        return
    separator = "\n" if existing.endswith("\n") else "\n\n"
    atomic_write_text(path, existing + separator + section_text.strip() + "\n")


def validate_required_columns(frame: pd.DataFrame, columns: set[str], name: str) -> None:
    missing = sorted(columns - set(frame.columns))
    if missing:
        raise AssertionError(
            f"{name} is missing required columns:\n"
            + "\n".join(f"- {column}" for column in missing)
        )


def build_manifest(root: Path) -> pd.DataFrame:
    records = []
    for path in sorted(root.rglob("*")):
        if not path.is_file():
            continue
        relative = path.relative_to(root)
        if relative.parts and relative.parts[0] == "05_control":
            continue
        records.append(
            {
                "RelativePath": str(relative),
                "Bytes": int(path.stat().st_size),
                "SHA256": sha256_file(path),
            }
        )
    return pd.DataFrame(records)


def save_figure(path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    plt.tight_layout()
    plt.savefig(path, dpi=300, bbox_inches="tight")
    plt.close()


def numeric_total(frame: pd.DataFrame, candidates: list[str]) -> float:
    for column in candidates:
        if column in frame.columns:
            return float(pd.to_numeric(frame[column], errors="raise").sum())
    raise AssertionError(
        "None of the expected total columns were found: " + ", ".join(candidates)
    )


# =============================================================================
# PREFLIGHT
# =============================================================================

required_inputs = [
    ND10R_CHECKPOINT_PATH,
    ND10A_CHECKPOINT_PATH,
    DAILY_PRODUCT_PATH,
    DAILY_RESTAURANT_PATH,
    WEEK_DAILY_PRODUCT_PATH,
    WEEK_PRODUCT_TOTALS_PATH,
    WEEK_RESTAURANT_DAILY_PATH,
    WEEK_RESTAURANT_TOTAL_PATH,
    DEMO_SUMMARY_PATH,
    ND10R_RESULTS_WORDING_PATH,
    ND10R_RECOMMENDED_FIGURES_PATH,
]

missing_inputs = [path for path in required_inputs if not path.is_file()]
if missing_inputs:
    raise FileNotFoundError(
        "ND10RA required inputs are missing:\n"
        + "\n".join(f"- {path}" for path in missing_inputs)
    )

actual_nd10r_hash = sha256_file(ND10R_CHECKPOINT_PATH)
actual_nd10a_hash = sha256_file(ND10A_CHECKPOINT_PATH)

if actual_nd10r_hash != EXPECTED_ND10R_CHECKPOINT_SHA256:
    raise AssertionError(
        "ND10R checkpoint hash mismatch.\n"
        f"Expected: {EXPECTED_ND10R_CHECKPOINT_SHA256}\n"
        f"Actual:   {actual_nd10r_hash}"
    )

if actual_nd10a_hash != EXPECTED_ND10A_CHECKPOINT_SHA256:
    raise AssertionError(
        "ND10A checkpoint hash mismatch.\n"
        f"Expected: {EXPECTED_ND10A_CHECKPOINT_SHA256}\n"
        f"Actual:   {actual_nd10a_hash}"
    )

protected_hashes_before = {str(path): sha256_file(path) for path in required_inputs}

if ND10RA_ROOT.exists():
    if not ALLOW_OVERWRITE:
        raise FileExistsError(
            "ND10RA output already exists. No files were changed:\n"
            f"{ND10RA_ROOT}"
        )
    shutil.rmtree(ND10RA_ROOT)

for path in [TOP_LEVEL_CHECKPOINT_PATH, TOP_LEVEL_CHECKPOINT_SHA_PATH]:
    if path.exists():
        if not ALLOW_OVERWRITE:
            raise FileExistsError(
                "An ND10RA top-level checkpoint already exists. "
                f"No files were changed:\n{path}"
            )
        path.unlink()

STAGING_ROOT = ND10RA_ROOT.parent / f".ND10RA_staging_{uuid.uuid4().hex}"
STAGING_ROOT.mkdir(parents=True, exist_ok=False)


# =============================================================================
# MAIN
# =============================================================================

try:
    daily_product = pd.read_csv(DAILY_PRODUCT_PATH, low_memory=False)
    daily_restaurant = pd.read_csv(DAILY_RESTAURANT_PATH, low_memory=False)
    week_daily_product = pd.read_csv(WEEK_DAILY_PRODUCT_PATH, low_memory=False)
    week_product_totals = pd.read_csv(WEEK_PRODUCT_TOTALS_PATH, low_memory=False)
    week_restaurant_daily = pd.read_csv(
        WEEK_RESTAURANT_DAILY_PATH, low_memory=False
    )
    week_restaurant_total = pd.read_csv(
        WEEK_RESTAURANT_TOTAL_PATH, low_memory=False
    )
    demo_summary = pd.read_csv(DEMO_SUMMARY_PATH, low_memory=False)

    validate_required_columns(
        daily_product,
        {
            "Date",
            "CanonicalProductID",
            "CanonicalProductName",
            "PredictedNormalDemand",
            "ConfirmedBulkDemand",
            "PlannedQuantity",
        },
        "ND10A daily product planning",
    )
    validate_required_columns(
        week_daily_product,
        {
            "Date",
            "CanonicalProductID",
            "CanonicalProductName",
            "PredictedNormalDemand",
            "ConfirmedBulkDemand",
            "PlannedQuantity",
        },
        "ND10A week daily product planning",
    )

    daily_product["Date"] = pd.to_datetime(daily_product["Date"], errors="raise")
    week_daily_product["Date"] = pd.to_datetime(
        week_daily_product["Date"], errors="raise"
    )
    week_restaurant_daily["Date"] = pd.to_datetime(
        week_restaurant_daily["Date"], errors="raise"
    )

    active_products = int(daily_product["CanonicalProductID"].astype(str).nunique())
    if active_products != EXPECTED_ACTIVE_PRODUCTS:
        raise AssertionError(
            f"Expected {EXPECTED_ACTIVE_PRODUCTS} active products; found {active_products}."
        )

    per_date_products = (
        week_daily_product.groupby("Date")["CanonicalProductID"].nunique()
    )
    if not (per_date_products == EXPECTED_ACTIVE_PRODUCTS).all():
        raise AssertionError(
            "At least one demonstration week date does not contain 88 active products."
        )

    corrected_daily_normal = numeric_total(
        daily_restaurant,
        [
            "PredictedRestaurantNormalDemand",
            "PredictedNormalDemand",
        ],
    )
    corrected_daily_bulk = numeric_total(
        daily_restaurant,
        [
            "ConfirmedRestaurantBulkDemand",
            "ConfirmedBulkDemand",
        ],
    )
    corrected_daily_planned = numeric_total(
        daily_restaurant,
        [
            "PlannedRestaurantQuantity",
            "PlannedQuantity",
        ],
    )

    corrected_week_normal = numeric_total(
        week_restaurant_total,
        [
            "PredictedRestaurantWeekNormalDemand",
            "PredictedRestaurantNormalDemand",
            "PredictedNormalDemand",
        ],
    )
    corrected_week_bulk = numeric_total(
        week_restaurant_total,
        [
            "ConfirmedRestaurantWeekBulkDemand",
            "ConfirmedRestaurantBulkDemand",
            "ConfirmedBulkDemand",
        ],
    )
    corrected_week_planned = numeric_total(
        week_restaurant_total,
        [
            "PlannedRestaurantWeekQuantity",
            "PlannedRestaurantQuantity",
            "PlannedQuantity",
        ],
    )

    # Reconcile the product level to restaurant totals independently.
    daily_product_normal = float(
        pd.to_numeric(daily_product["PredictedNormalDemand"], errors="raise").sum()
    )
    daily_product_bulk = float(
        pd.to_numeric(daily_product["ConfirmedBulkDemand"], errors="raise").sum()
    )
    daily_product_planned = float(
        pd.to_numeric(daily_product["PlannedQuantity"], errors="raise").sum()
    )

    week_product_normal = float(
        pd.to_numeric(
            week_daily_product["PredictedNormalDemand"], errors="raise"
        ).sum()
    )
    week_product_bulk = float(
        pd.to_numeric(
            week_daily_product["ConfirmedBulkDemand"], errors="raise"
        ).sum()
    )
    week_product_planned = float(
        pd.to_numeric(
            week_daily_product["PlannedQuantity"], errors="raise"
        ).sum()
    )

    reconciliation_diffs = [
        abs(corrected_daily_normal - daily_product_normal),
        abs(corrected_daily_bulk - daily_product_bulk),
        abs(corrected_daily_planned - daily_product_planned),
        abs(corrected_week_normal - week_product_normal),
        abs(corrected_week_bulk - week_product_bulk),
        abs(corrected_week_planned - week_product_planned),
    ]
    max_reconciliation_difference = max(reconciliation_diffs)
    if max_reconciliation_difference > 1e-9:
        raise AssertionError(
            "Corrected demonstration outputs do not reconcile. "
            f"Maximum difference: {max_reconciliation_difference}"
        )

    daily_date = pd.Timestamp(daily_product["Date"].iloc[0]).normalize()
    week_start = pd.Timestamp(week_daily_product["Date"].min()).normalize()
    week_end = pd.Timestamp(week_daily_product["Date"].max()).normalize()

    corrected_summary = pd.DataFrame(
        [
            {
                "Metric": "Active forecast products",
                "Value": active_products,
                "Unit": "products",
            },
            {
                "Metric": "Daily restaurant normal demand",
                "Value": corrected_daily_normal,
                "Unit": "units",
            },
            {
                "Metric": "Daily confirmed bulk",
                "Value": corrected_daily_bulk,
                "Unit": "units",
            },
            {
                "Metric": "Daily planned quantity",
                "Value": corrected_daily_planned,
                "Unit": "units",
            },
            {
                "Metric": "Week restaurant normal demand",
                "Value": corrected_week_normal,
                "Unit": "units",
            },
            {
                "Metric": "Week confirmed bulk",
                "Value": corrected_week_bulk,
                "Unit": "units",
            },
            {
                "Metric": "Week planned quantity",
                "Value": corrected_week_planned,
                "Unit": "units",
            },
        ]
    )

    scope_comparison = pd.DataFrame(
        [
            {
                "Metric": "Products forecast",
                "Old227Scope": OLD_ACTIVE_PRODUCTS,
                "Corrected88Scope": active_products,
                "AbsoluteChange": active_products - OLD_ACTIVE_PRODUCTS,
                "PercentageChange": 100.0
                * (active_products - OLD_ACTIVE_PRODUCTS)
                / OLD_ACTIVE_PRODUCTS,
            },
            {
                "Metric": "Daily restaurant normal demand",
                "Old227Scope": OLD_DAILY_RESTAURANT_NORMAL_DEMAND,
                "Corrected88Scope": corrected_daily_normal,
                "AbsoluteChange": (
                    corrected_daily_normal
                    - OLD_DAILY_RESTAURANT_NORMAL_DEMAND
                ),
                "PercentageChange": 100.0
                * (
                    corrected_daily_normal
                    - OLD_DAILY_RESTAURANT_NORMAL_DEMAND
                )
                / OLD_DAILY_RESTAURANT_NORMAL_DEMAND,
            },
            {
                "Metric": "Week restaurant normal demand",
                "Old227Scope": OLD_WEEK_RESTAURANT_NORMAL_DEMAND,
                "Corrected88Scope": corrected_week_normal,
                "AbsoluteChange": (
                    corrected_week_normal
                    - OLD_WEEK_RESTAURANT_NORMAL_DEMAND
                ),
                "PercentageChange": 100.0
                * (
                    corrected_week_normal
                    - OLD_WEEK_RESTAURANT_NORMAL_DEMAND
                )
                / OLD_WEEK_RESTAURANT_NORMAL_DEMAND,
            },
        ]
    )

    top_daily = (
        daily_product.sort_values(
            ["PlannedQuantity", "CanonicalProductName"],
            ascending=[False, True],
            kind="mergesort",
        )
        .head(20)
        .copy()
    )

    if "PlannedWeekQuantity" in week_product_totals.columns:
        week_sort_column = "PlannedWeekQuantity"
    elif "PlannedQuantity" in week_product_totals.columns:
        week_sort_column = "PlannedQuantity"
    elif "PredictedWeekNormalDemand" in week_product_totals.columns:
        week_sort_column = "PredictedWeekNormalDemand"
    else:
        raise AssertionError(
            "Could not identify the week planned-quantity field."
        )

    top_week = (
        week_product_totals.sort_values(
            [week_sort_column, "CanonicalProductName"],
            ascending=[False, True],
            kind="mergesort",
        )
        .head(20)
        .copy()
    )

    staged_table_dir = STAGING_ROOT / "01_tables"
    staged_figure_dir = STAGING_ROOT / "02_figures"
    staged_wording_dir = STAGING_ROOT / "03_report_wording"
    staged_demo_dir = STAGING_ROOT / "04_demonstration_material"
    staged_control_dir = STAGING_ROOT / "05_control"

    for directory in [
        staged_table_dir,
        staged_figure_dir,
        staged_wording_dir,
        staged_demo_dir,
        staged_control_dir,
    ]:
        directory.mkdir(parents=True, exist_ok=True)

    write_csv(
        staged_table_dir / CORRECTED_SUMMARY_PATH.name,
        corrected_summary,
    )
    write_csv(
        staged_table_dir / SCOPE_COMPARISON_PATH.name,
        scope_comparison,
    )
    write_csv(staged_table_dir / TOP_DAILY_PATH.name, top_daily)
    write_csv(staged_table_dir / TOP_WEEK_PATH.name, top_week)

    # -------------------------------------------------------------------------
    # Figure 1 — top daily corrected demonstration products
    # -------------------------------------------------------------------------
    figure_daily = top_daily.sort_values("PlannedQuantity", ascending=True)
    plt.figure(figsize=(10, 8))
    plt.barh(
        figure_daily["CanonicalProductName"].astype(str),
        figure_daily["PlannedQuantity"].astype(float),
    )
    plt.xlabel("Planned product quantity")
    plt.ylabel("Product")
    plt.title(
        f"Corrected demonstration: top daily product quantities — "
        f"{daily_date.date()}"
    )
    save_figure(
        staged_figure_dir
        / "ND10RA_figure_01_corrected_daily_top_products.png"
    )

    # -------------------------------------------------------------------------
    # Figure 2 — restaurant daily trajectory for corrected demonstration week
    # -------------------------------------------------------------------------
    if "PlannedRestaurantQuantity" in week_restaurant_daily.columns:
        restaurant_daily_column = "PlannedRestaurantQuantity"
    elif "PlannedQuantity" in week_restaurant_daily.columns:
        restaurant_daily_column = "PlannedQuantity"
    elif "PredictedRestaurantNormalDemand" in week_restaurant_daily.columns:
        restaurant_daily_column = "PredictedRestaurantNormalDemand"
    else:
        raise AssertionError(
            "Could not identify restaurant daily planned-quantity field."
        )

    plt.figure(figsize=(9, 5))
    plt.plot(
        week_restaurant_daily["Date"],
        pd.to_numeric(
            week_restaurant_daily[restaurant_daily_column],
            errors="raise",
        ),
        marker="o",
    )
    plt.xlabel("Date")
    plt.ylabel("Planned restaurant quantity")
    plt.title(
        "Corrected demonstration: Monday-origin restaurant daily plan"
    )
    plt.xticks(rotation=30)
    save_figure(
        staged_figure_dir
        / "ND10RA_figure_02_corrected_week_restaurant_daily.png"
    )

    # -------------------------------------------------------------------------
    # Figure 3 — top corrected product-week quantities
    # -------------------------------------------------------------------------
    figure_week = top_week.sort_values(week_sort_column, ascending=True)
    plt.figure(figsize=(10, 8))
    plt.barh(
        figure_week["CanonicalProductName"].astype(str),
        pd.to_numeric(figure_week[week_sort_column], errors="raise"),
    )
    plt.xlabel("Planned product-week quantity")
    plt.ylabel("Product")
    plt.title(
        f"Corrected demonstration: top product-week quantities — "
        f"{week_start.date()} to {week_end.date()}"
    )
    save_figure(
        staged_figure_dir
        / "ND10RA_figure_03_corrected_week_top_products.png"
    )

    # -------------------------------------------------------------------------
    # Figure 4 — old 227-product vs corrected 88-product demonstration totals
    # -------------------------------------------------------------------------
    comparison_for_plot = pd.DataFrame(
        {
            "Scope": ["Old 227-product scope", "Corrected 88-product scope"],
            "Daily": [
                OLD_DAILY_RESTAURANT_NORMAL_DEMAND,
                corrected_daily_normal,
            ],
            "Week": [
                OLD_WEEK_RESTAURANT_NORMAL_DEMAND,
                corrected_week_normal,
            ],
        }
    )

    x = np.arange(len(comparison_for_plot))
    width = 0.36
    plt.figure(figsize=(9, 5))
    plt.bar(
        x - width / 2,
        comparison_for_plot["Daily"],
        width,
        label="Daily total",
    )
    plt.bar(
        x + width / 2,
        comparison_for_plot["Week"],
        width,
        label="Week total",
    )
    plt.xticks(x, comparison_for_plot["Scope"])
    plt.ylabel("Predicted normal-demand units")
    plt.title("Effect of correcting the demonstration product catalogue")
    plt.legend()
    save_figure(
        staged_figure_dir
        / "ND10RA_figure_04_old_vs_corrected_scope_totals.png"
    )

    figure_captions = pd.DataFrame(
        [
            {
                "Figure": "ND10RA_figure_01_corrected_daily_top_products.png",
                "SuggestedCaption": (
                    "Highest planned product quantities for the corrected "
                    "88-product demonstration catalogue on 2 March 2026."
                ),
                "RecommendedUse": "REPORT_OR_PRESENTATION",
            },
            {
                "Figure": "ND10RA_figure_02_corrected_week_restaurant_daily.png",
                "SuggestedCaption": (
                    "Corrected restaurant-level Monday-origin daily planning "
                    "totals for the demonstration week."
                ),
                "RecommendedUse": "DEMONSTRATION_OR_PRESENTATION",
            },
            {
                "Figure": "ND10RA_figure_03_corrected_week_top_products.png",
                "SuggestedCaption": (
                    "Highest product-week planned quantities after applying "
                    "the active-menu catalogue correction."
                ),
                "RecommendedUse": "REPORT_OR_PRESENTATION",
            },
            {
                "Figure": "ND10RA_figure_04_old_vs_corrected_scope_totals.png",
                "SuggestedCaption": (
                    "Comparison of the original 227-product inference scope "
                    "and the corrected 88-product latest-active-panel scope. "
                    "The correction removes historical products that were no "
                    "longer part of the current forecasting catalogue."
                ),
                "RecommendedUse": "REPORT_APPENDIX_OR_METHOD_LIMITATION",
            },
        ]
    )
    write_csv(
        staged_wording_dir / FIGURE_CAPTIONS_PATH.name,
        figure_captions,
    )

    # Copy the corrected planning summary to the demonstration-material folder.
    write_csv(
        staged_demo_dir / "ND10RA_corrected_demonstration_planning_summary.csv",
        corrected_summary,
    )
    write_csv(
        staged_demo_dir / "ND10RA_corrected_top_daily_products.csv",
        top_daily,
    )
    write_csv(
        staged_demo_dir / "ND10RA_corrected_top_week_products.csv",
        top_week,
    )

    # Copy the four corrected figures into the demonstration-material folder.
    for figure_path in staged_figure_dir.glob("*.png"):
        shutil.copy2(figure_path, staged_demo_dir / figure_path.name)

    daily_drop = (
        OLD_DAILY_RESTAURANT_NORMAL_DEMAND - corrected_daily_normal
    )
    week_drop = (
        OLD_WEEK_RESTAURANT_NORMAL_DEMAND - corrected_week_normal
    )

    report_wording = f"""# ND10RA Corrected Demonstration Reporting Wording

## Active-menu catalogue correction

The original frozen inference engine initially carried forward all 227 canonical products that had appeared historically. A review of the final 20 operating days identified that this historical universe was broader than the product catalogue active at the forecast origin. Of the 90 products appearing at least once during the final 20 operating days, 86 were present throughout the full period, two were recent additions that remained active at the cutoff, and two had retired before the cutoff. The latest known active forecasting panel therefore contained 88 products.

The correction was implemented in ND09A as an inference-scope amendment only. No forecasting model was fitted, refitted, tuned, calibrated or reselected. ND10A then regenerated the planning quantities from the corrected 88-product forecasts.

## Demonstration impact

For 2 March 2026, the restaurant-level predicted normal demand changed from {OLD_DAILY_RESTAURANT_NORMAL_DEMAND:.6f} units under the original 227-product historical scope to {corrected_daily_normal:.6f} units using the corrected active catalogue, a reduction of {daily_drop:.6f} units.

For the Monday-to-Friday demonstration week, predicted restaurant normal demand changed from {OLD_WEEK_RESTAURANT_NORMAL_DEMAND:.6f} units to {corrected_week_normal:.6f} units, a reduction of {week_drop:.6f} units.

These changes do not represent a change in model accuracy. The ND04–ND08 evaluation metrics remain unchanged because the correction applies only to the product scope used for future demonstration inference.

## Reporting rule

The ND10R model-selection, validation and accuracy figures remain authoritative. ND10RA replaces only the demonstration-specific figures, tables and forecast totals that depended on the old 227-product future catalogue.
"""
    write_text(
        staged_wording_dir / REPORT_WORDING_PATH.name,
        report_wording,
    )

    readme = f"""# ND10RA Corrected Demonstration Reporting Refresh

Status: `{STATUS}`

This reporting amendment refreshes only the demonstration-specific outputs after the active-menu catalogue correction.

Authoritative evaluation/reporting package:
`{ND10R_ROOT}`

Corrected planning source:
`{ND10A_ROOT}`

Corrected active forecast products: {active_products}

No model-evaluation accuracy metric was recomputed and no March actual target was opened.
"""
    write_text(STAGING_ROOT / "README.md", readme)

    manifest = build_manifest(STAGING_ROOT)
    write_csv(staged_control_dir / MANIFEST_PATH.name, manifest)
    manifest_hash = sha256_file(staged_control_dir / MANIFEST_PATH.name)

    checkpoint_payload = {
        "StepID": STEP_ID,
        "Status": STATUS,
        "CompletedLocalTime": NOW_LOCAL.isoformat(),
        "ND10RARoot": str(ND10RA_ROOT),
        "ND10RCheckpointSHA256": actual_nd10r_hash,
        "ND10ACheckpointSHA256": actual_nd10a_hash,
        "AuthoritativeEvaluationPackage": str(ND10R_ROOT),
        "CorrectedPlanningPackage": str(ND10A_ROOT),
        "ActiveForecastProducts": active_products,
        "DailyDate": daily_date,
        "CorrectedDailyRestaurantNormalDemand": corrected_daily_normal,
        "WeekStart": week_start,
        "WeekEnd": week_end,
        "CorrectedWeekRestaurantNormalDemand": corrected_week_normal,
        "FiguresCreated": 4,
        "ModelEvaluationMetricsRecomputed": False,
        "ForecastingModelsRefitted": False,
        "MarchTargetVaultOpened": False,
        "ArtifactManifestSHA256": manifest_hash,
        "NextStep": "ND11",
    }

    staged_checkpoint = staged_control_dir / CHECKPOINT_PATH.name
    write_json(staged_checkpoint, checkpoint_payload)
    checkpoint_hash = sha256_file(staged_checkpoint)
    write_text(
        staged_control_dir / CHECKPOINT_SHA_PATH.name,
        checkpoint_hash + "\n",
    )

    protected_hashes_after = {
        str(path): sha256_file(path) for path in required_inputs
    }
    changed_inputs = [
        path
        for path in protected_hashes_before
        if protected_hashes_before[path] != protected_hashes_after[path]
    ]
    if changed_inputs:
        raise AssertionError(
            "Protected inputs changed during ND10RA:\n"
            + "\n".join(f"- {path}" for path in changed_inputs)
        )

    os.replace(STAGING_ROOT, ND10RA_ROOT)

    # Build a ZIP bundle after the atomic move.
    shutil.make_archive(
        str(ND10RA_ROOT / "05_control" / "ND10RA_corrected_demo_reporting_bundle"),
        "zip",
        root_dir=ND10RA_ROOT,
        base_dir=".",
    )

    TOP_LEVEL_CHECKPOINT_PATH.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(
        ND10RA_ROOT / "05_control" / CHECKPOINT_PATH.name,
        TOP_LEVEL_CHECKPOINT_PATH,
    )
    shutil.copy2(
        ND10RA_ROOT / "05_control" / CHECKPOINT_SHA_PATH.name,
        TOP_LEVEL_CHECKPOINT_SHA_PATH,
    )

    handoff = f"""# ND10RA Handoff

## Status

- Completed step: `{STEP_ID}`
- Status: `{STATUS}`
- Root: `{ND10RA_ROOT}`
- Checkpoint SHA-256: `{checkpoint_hash}`

## Corrected demonstration scope

- Active forecast products: {active_products}
- Daily demonstration normal demand: {corrected_daily_normal:.6f}
- Week demonstration normal demand: {corrected_week_normal:.6f}

## Reporting boundary

ND10R remains authoritative for all model-selection, development-validation and accuracy figures. ND10RA replaces only demonstration-specific tables, figures and totals affected by the active-menu catalogue correction.

## Next step

ND11 — illustrative ingredient mapping for the selected 14 products and preparation of the final demonstration backend package.
"""
    atomic_write_text(HANDOFF_PATH, handoff)
    atomic_write_text(CURRENT_HANDOFF_PATH, handoff)

    append_marked_section(
        WORKFLOW_PATH,
        "## ND10RA — Corrected demonstration reporting refresh",
        f"""## ND10RA — Corrected demonstration reporting refresh

Status: `{STATUS}`

Only the ND09/ND10-derived demonstration figures and tables were refreshed after correcting the future catalogue to the 88 products in the latest active panel. ND10R evaluation metrics remain unchanged.
""",
    )

    append_marked_section(
        DECISIONS_PATH,
        "## ND10RA decisions",
        """## ND10RA decisions

- Keep ND10R as the authoritative model-evaluation reporting package.
- Replace only demonstration-specific figures and totals affected by the active-menu correction.
- Do not recompute development accuracy or open March actual targets.
""",
    )

    append_marked_section(
        METRICS_AND_RESULTS_PATH,
        "## ND10RA corrected demonstration totals",
        f"""## ND10RA corrected demonstration totals

- Active forecast catalogue: {active_products} products.
- Daily demonstration restaurant normal demand: {corrected_daily_normal:.6f}.
- Week demonstration restaurant normal demand: {corrected_week_normal:.6f}.
- These are unscored demonstration forecasts, not accuracy metrics.
""",
    )

    append_marked_section(
        AGENTS_PATH,
        "Marker: ND10RA_AUTHORITATIVE_STATUS",
        f"""## ND10RA authoritative status

Marker: ND10RA_AUTHORITATIVE_STATUS

- Status: `{STATUS}`
- Checkpoint SHA-256: `{checkpoint_hash}`
- ND10R remains authoritative for evaluation metrics.
- ND10RA is authoritative for corrected demonstration reporting.
- Next step: ND11.
""",
    )

    LOG_PATH.parent.mkdir(parents=True, exist_ok=True)
    with LOG_PATH.open("a", encoding="utf-8") as handle:
        handle.write(
            f"{NOW_LOCAL.isoformat()} | {STATUS} | "
            f"checkpoint={checkpoint_hash} | root={ND10RA_ROOT}\n"
        )

except Exception:
    if STAGING_ROOT.exists():
        shutil.rmtree(STAGING_ROOT, ignore_errors=True)
    raise


# =============================================================================
# FINAL CONSOLE OUTPUT
# =============================================================================

print("=" * 118)
print("EDEN NORMAL-DEMAND MODEL V2 — ND10RA CORRECTED DEMONSTRATION REPORTING COMPLETE")
print("=" * 118)
print(f"Status: {STATUS}")
print(f"Local time: {NOW_LOCAL.isoformat()}")
print(f"ND10RA root: {ND10RA_ROOT}")
print()
print("INPUT VERIFICATION")
print(f"ND10R checkpoint SHA-256: {actual_nd10r_hash}")
print(f"ND10A checkpoint SHA-256: {actual_nd10a_hash}")
print(f"Authoritative evaluation package preserved: True")
print(f"Previous inputs modified: False")
print(f"March target vault opened: False")
print()
print("CORRECTED DEMONSTRATION")
print(f"Active products: {active_products}")
print(f"Daily date: {daily_date.date()}")
print(f"Old 227-product daily normal demand: {OLD_DAILY_RESTAURANT_NORMAL_DEMAND:.6f}")
print(f"Corrected 88-product daily normal demand: {corrected_daily_normal:.6f}")
print(f"Week: {week_start.date()} to {week_end.date()}")
print(f"Old 227-product week normal demand: {OLD_WEEK_RESTAURANT_NORMAL_DEMAND:.6f}")
print(f"Corrected 88-product week normal demand: {corrected_week_normal:.6f}")
print()
print("REPORT REFRESH")
print("Model-evaluation figures regenerated: False")
print("Model-evaluation metrics recomputed: False")
print("Corrected demonstration figures created: 4")
print(f"Figure directory: {ND10RA_ROOT / '02_figures'}")
print(f"Corrected summary: {CORRECTED_SUMMARY_PATH}")
print(f"Scope comparison: {SCOPE_COMPARISON_PATH}")
print(f"Report wording: {REPORT_WORDING_PATH}")
print(f"Corrected demo bundle: {ND10RA_ROOT / '05_control' / 'ND10RA_corrected_demo_reporting_bundle.zip'}")
print(f"Checkpoint: {TOP_LEVEL_CHECKPOINT_PATH}")
print(f"Checkpoint SHA-256: {checkpoint_hash}")
print(f"Handoff: {HANDOFF_PATH}")
print()
print("SAFETY")
print("- Forecasting models fitted/refitted: False")
print("- Forecasting methods reselected: False")
print("- March target vault opened: False")
print("- Future demonstration forecasts scored against March actuals: False")
print("- Previous reporting/planning inputs modified: False")
print("- ND10R evaluation package preserved unchanged: True")
print()
print("NEXT STEP")
print("ND11 — illustrative ingredient mapping for the final 14 selected products.")
print("=" * 118)

EDEN NORMAL-DEMAND MODEL V2 — ND10RA CORRECTED DEMONSTRATION REPORTING COMPLETE
Status: ND10RA_CORRECTED_DEMONSTRATION_REPORTING_REFRESH_COMPLETED_READY_FOR_ND11
Local time: 2026-08-10T12:51:41.671802+01:00
ND10RA root: /Users/ryansmac/Desktop/Meng Project/eden_datasets/eden_normal_demand_model_v2/05_reporting/ND10RA_corrected_demo_reporting_refresh

INPUT VERIFICATION
ND10R checkpoint SHA-256: 04726dc6131ec92f5da86517b395afa8b624ed2ef2755d7aed82840da697f350
ND10A checkpoint SHA-256: 6bcc97c5fe76667c2bdcdde7d0e81b4bd1534c3a601f39fd7acd795e6c87dbff
Authoritative evaluation package preserved: True
Previous inputs modified: False
March target vault opened: False

CORRECTED DEMONSTRATION
Active products: 88
Daily date: 2026-03-02
Old 227-product daily normal demand: 1139.224575
Corrected 88-product daily normal demand: 727.756809
Week: 2026-03-02 to 2026-03-06
Old 227-product week normal demand: 5302.070080
Corrected 88-product week normal demand: 3341.504320

REPORT REFRESH
Model-evaluati

In [20]:
# =============================================================================
# EDEN NORMAL-DEMAND MODEL V2
# ND11 — ILLUSTRATIVE INGREDIENT MAPPING AND DEMONSTRATION BACKEND EXPORT
#
# Run this as one complete Jupyter cell after ND10A and ND10RA.
#
# PURPOSE
# -------
# Create a realistic but explicitly illustrative ingredient-mapping prototype
# for the 14 agreed high-value prepared-food products, apply the mapping to the
# corrected 88-product ND10A planning outputs, and create ingredient requirement
# files for the future demonstration UI.
#
# IMPORTANT ACADEMIC BOUNDARY
# ---------------------------
# Eden Restaurant did not provide recipe-level ingredient data within the
# project timeframe and approved the use of assumed recipes for the academic
# prototype. These mappings are therefore:
#
#   - illustrative;
#   - assumed;
#   - realistic serving/procurement quantities;
#   - NOT Eden's actual recipes;
#   - NOT procurement specifications;
#   - NOT used to train or evaluate the forecasting model.
#
# No forecasting model is fitted, refitted, tuned, calibrated or reselected.
# March 2026 actual targets are never loaded.
# =============================================================================

import hashlib
import json
import os
import platform
import shutil
import uuid
from datetime import datetime, timezone
from pathlib import Path
from zoneinfo import ZoneInfo

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd


# =============================================================================
# USER CONFIGURATION
# =============================================================================

ALLOW_OVERWRITE = False


# =============================================================================
# FIXED PROJECT CONFIGURATION
# =============================================================================

PROJECT_ROOT = Path("/Users/ryansmac/Desktop/Meng Project")
EDEN_ROOT = PROJECT_ROOT / "eden_datasets"
MODEL_ROOT = EDEN_ROOT / "eden_normal_demand_model_v2"

ND09A_CHECKPOINT_PATH = MODEL_ROOT / "08_checkpoints" / "ND09A_checkpoint.json"
ND10A_CHECKPOINT_PATH = MODEL_ROOT / "08_checkpoints" / "ND10A_checkpoint.json"
ND10RA_CHECKPOINT_PATH = MODEL_ROOT / "08_checkpoints" / "ND10RA_checkpoint.json"

EXPECTED_ND09A_CHECKPOINT_SHA256 = (
    "54767758d9052cb1571043d2f39cc167bc7491b300afbd3605a5416b7aa95c10"
)
EXPECTED_ND10A_CHECKPOINT_SHA256 = (
    "6bcc97c5fe76667c2bdcdde7d0e81b4bd1534c3a601f39fd7acd795e6c87dbff"
)
EXPECTED_ND10RA_CHECKPOINT_SHA256 = (
    "fe24555b1562cce7b4300f62051d427492f4e9b3259e662e21f400bac6972c75"
)

ND09A_ROOT = (
    MODEL_ROOT
    / "03_models"
    / "03_inference"
    / "ND09A_active_menu_catalogue_correction"
)
ND10A_ROOT = (
    MODEL_ROOT
    / "04_planning"
    / "ND10A_corrected_active_catalogue_planning"
)
ND10RA_ROOT = (
    MODEL_ROOT
    / "05_reporting"
    / "ND10RA_corrected_demo_reporting_refresh"
)

ACTIVE_CATALOGUE_PATH = (
    ND09A_ROOT
    / "02_contracts"
    / "ND09A_active_product_catalogue.csv"
)
ND10A_OUTPUT_DIR = ND10A_ROOT / "02_planned_outputs"

DAILY_PRODUCT_PLANNING_PATH = (
    ND10A_OUTPUT_DIR / "ND10A_daily_product_planning.csv"
)
DAILY_RESTAURANT_PLANNING_PATH = (
    ND10A_OUTPUT_DIR / "ND10A_daily_restaurant_planning.csv"
)
WEEK_DAILY_PRODUCT_PLANNING_PATH = (
    ND10A_OUTPUT_DIR / "ND10A_week_daily_product_planning.csv"
)
WEEK_PRODUCT_TOTALS_PLANNING_PATH = (
    ND10A_OUTPUT_DIR / "ND10A_week_product_totals_planning.csv"
)
WEEK_RESTAURANT_DAILY_PLANNING_PATH = (
    ND10A_OUTPUT_DIR / "ND10A_week_restaurant_daily_planning.csv"
)
WEEK_RESTAURANT_TOTAL_PLANNING_PATH = (
    ND10A_OUTPUT_DIR / "ND10A_week_restaurant_total_planning.csv"
)
DEMONSTRATION_SUMMARY_PATH = (
    ND10A_OUTPUT_DIR / "ND10A_demonstration_planning_summary.csv"
)

EXPECTED_ACTIVE_PRODUCTS = 88
EXPECTED_MAPPED_PRODUCTS = 14

DATE_COLUMN = "Date"
PRODUCT_ID_COLUMN = "CanonicalProductID"
PRODUCT_NAME_COLUMN = "CanonicalProductName"
NORMAL_COLUMN = "PredictedNormalDemand"
BULK_COLUMN = "ConfirmedBulkDemand"
PLANNED_COLUMN = "PlannedQuantity"

MAPPING_STATUS = "ILLUSTRATIVE_ASSUMED_RECIPE"
MAPPING_APPROVAL_NOTE = (
    "Assumed ingredient mapping permitted for the academic prototype because "
    "Eden Restaurant recipe-level ingredient data were unavailable within the "
    "project timeframe."
)
PROCUREMENT_BASIS = (
    "RAW_OR_PURCHASED_INGREDIENT_QUANTITY_PER_PLANNED_PRODUCT_UNIT"
)
NO_YIELD_NOTE = (
    "No cooking-loss, trimming-loss, spoilage, safety-stock or procurement-"
    "pack-size factor is applied."
)

ND11_ROOT = (
    MODEL_ROOT
    / "04_planning"
    / "ND11_illustrative_ingredient_mapping"
)

MAPPING_DIR = ND11_ROOT / "01_mapping"
REQUIREMENT_DIR = ND11_ROOT / "02_ingredient_requirements"
DEMO_BACKEND_DIR = ND11_ROOT / "03_demonstration_backend"
AUDIT_DIR = ND11_ROOT / "04_audits"
FIGURE_DIR = ND11_ROOT / "05_figures"
REPORT_DIR = ND11_ROOT / "06_reports"
CONTROL_DIR = ND11_ROOT / "07_control"

PRODUCT_MAPPING_PATH = (
    MAPPING_DIR / "ND11_illustrative_product_ingredient_mapping.csv"
)
MAPPED_PRODUCT_CATALOGUE_PATH = (
    MAPPING_DIR / "ND11_mapped_product_catalogue.csv"
)
INGREDIENT_MASTER_PATH = MAPPING_DIR / "ND11_ingredient_master.csv"
RECIPE_FAMILY_SUMMARY_PATH = MAPPING_DIR / "ND11_recipe_family_summary.csv"

DAILY_PRODUCT_INGREDIENT_PATH = (
    REQUIREMENT_DIR / "ND11_daily_product_ingredient_requirements.csv"
)
DAILY_INGREDIENT_TOTALS_PATH = (
    REQUIREMENT_DIR / "ND11_daily_ingredient_totals.csv"
)
WEEK_DAILY_PRODUCT_INGREDIENT_PATH = (
    REQUIREMENT_DIR / "ND11_week_daily_product_ingredient_requirements.csv"
)
WEEK_DAILY_INGREDIENT_TOTALS_PATH = (
    REQUIREMENT_DIR / "ND11_week_daily_ingredient_totals.csv"
)
WEEK_INGREDIENT_TOTALS_PATH = (
    REQUIREMENT_DIR / "ND11_week_ingredient_totals.csv"
)

MAPPING_COVERAGE_PATH = AUDIT_DIR / "ND11_mapping_coverage_audit.csv"
UNIT_CONSISTENCY_PATH = AUDIT_DIR / "ND11_ingredient_unit_consistency_audit.csv"
FORMULA_RECONCILIATION_PATH = AUDIT_DIR / "ND11_formula_reconciliation_audit.csv"
MAPPED_PRODUCT_VALIDATION_PATH = AUDIT_DIR / "ND11_mapped_product_validation.csv"
ASSUMPTION_AUDIT_PATH = AUDIT_DIR / "ND11_recipe_assumption_audit.csv"
VALIDATION_PATH = AUDIT_DIR / "ND11_validation_summary.csv"

BACKEND_DAILY_PRODUCT_PATH = (
    DEMO_BACKEND_DIR / "active_catalogue_daily_product_plan.csv"
)
BACKEND_DAILY_RESTAURANT_PATH = (
    DEMO_BACKEND_DIR / "active_catalogue_daily_restaurant_plan.csv"
)
BACKEND_WEEK_DAILY_PRODUCT_PATH = (
    DEMO_BACKEND_DIR / "active_catalogue_week_daily_product_plan.csv"
)
BACKEND_WEEK_PRODUCT_TOTALS_PATH = (
    DEMO_BACKEND_DIR / "active_catalogue_week_product_totals.csv"
)
BACKEND_WEEK_RESTAURANT_DAILY_PATH = (
    DEMO_BACKEND_DIR / "active_catalogue_week_restaurant_daily.csv"
)
BACKEND_WEEK_RESTAURANT_TOTAL_PATH = (
    DEMO_BACKEND_DIR / "active_catalogue_week_restaurant_total.csv"
)
BACKEND_INGREDIENT_MAPPING_PATH = (
    DEMO_BACKEND_DIR / "illustrative_ingredient_mapping_14_products.csv"
)
BACKEND_DAILY_INGREDIENT_PATH = (
    DEMO_BACKEND_DIR / "daily_ingredient_requirements.csv"
)
BACKEND_WEEK_DAILY_INGREDIENT_PATH = (
    DEMO_BACKEND_DIR / "week_daily_ingredient_requirements.csv"
)
BACKEND_WEEK_INGREDIENT_PATH = (
    DEMO_BACKEND_DIR / "week_ingredient_requirements.csv"
)
BACKEND_CONTRACT_PATH = DEMO_BACKEND_DIR / "demo_backend_contract.json"
BACKEND_DISCLAIMER_PATH = DEMO_BACKEND_DIR / "INGREDIENT_MAPPING_DISCLAIMER.md"

REPORT_SUMMARY_PATH = (
    REPORT_DIR / "ND11_illustrative_ingredient_mapping_summary.md"
)
REPORT_WORDING_PATH = (
    REPORT_DIR / "ND11_report_ready_ingredient_mapping_wording.md"
)
README_PATH = ND11_ROOT / "README.md"

MANIFEST_PATH = CONTROL_DIR / "ND11_artifact_hash_manifest.csv"
CHECKPOINT_PATH = CONTROL_DIR / "ND11_checkpoint.json"
CHECKPOINT_SHA_PATH = CONTROL_DIR / "ND11_checkpoint.sha256"
ZIP_PATH = CONTROL_DIR / "ND11_demonstration_backend_bundle.zip"

TOP_LEVEL_CHECKPOINT_PATH = MODEL_ROOT / "08_checkpoints" / "ND11_checkpoint.json"
TOP_LEVEL_CHECKPOINT_SHA_PATH = (
    MODEL_ROOT / "08_checkpoints" / "ND11_checkpoint.sha256"
)

MEMORY_ROOT = MODEL_ROOT / "00_project_memory"
HANDOFF_PATH = MEMORY_ROOT / "ND11_HANDOFF.md"
CURRENT_HANDOFF_PATH = MEMORY_ROOT / "CURRENT_HANDOFF.md"
WORKFLOW_PATH = MEMORY_ROOT / "WORKFLOW.md"
DECISIONS_PATH = MEMORY_ROOT / "DECISIONS.md"
METRICS_AND_RESULTS_PATH = MEMORY_ROOT / "METRICS_AND_RESULTS.md"
AGENTS_PATH = MODEL_ROOT / "AGENTS.md"
LOG_PATH = MODEL_ROOT / "09_logs" / "ND11_ingredient_mapping_log.txt"

STEP_ID = "ND11"
STATUS = "ND11_ILLUSTRATIVE_INGREDIENT_MAPPING_AND_DEMO_BACKEND_CREATED_READY_FOR_DEMO_PACKAGE"
NOW_UTC = datetime.now(timezone.utc)
NOW_LOCAL = NOW_UTC.astimezone(ZoneInfo("Europe/Dublin"))


# =============================================================================
# AUTHORITATIVE 14-PRODUCT ILLUSTRATIVE RECIPE MAPPING
# =============================================================================
#
# Quantities are deliberately specified on a procurement-style basis rather
# than as cooked serving weights where practical:
#
#   - dry rice rather than cooked rice;
#   - raw/purchased chicken rather than cooked chicken;
#   - purchased vegetables and sauces;
#   - each-counts for discrete items such as eggs, tortillas and bread rolls.
#
# The values are realistic prototype serving assumptions, not Eden recipes.
# =============================================================================

RECIPE_ROWS = [
    # -------------------------------------------------------------------------
    # KIMBOCK FAMILY
    # Same core meal family. Higher price tiers increase portions and introduce
    # additional components, following the agreed prototype rule.
    # -------------------------------------------------------------------------
    {
        "CanonicalProductID": "PLU_42532",
        "CanonicalProductName": "€5 KIMBOCK",
        "RecipeFamily": "KIMBOCK",
        "PriceTierEuro": 5,
        "IngredientID": "ING_DRY_RICE",
        "IngredientName": "Dry white rice",
        "QuantityPerProductUnit": 0.080,
        "IngredientUnit": "kg",
        "IngredientRole": "CORE",
        "AssumptionRationale": "Base rice portion; approximately 80 g dry rice per meal.",
    },
    {
        "CanonicalProductID": "PLU_42532",
        "CanonicalProductName": "€5 KIMBOCK",
        "RecipeFamily": "KIMBOCK",
        "PriceTierEuro": 5,
        "IngredientID": "ING_CHICKEN_BREAST",
        "IngredientName": "Chicken breast",
        "QuantityPerProductUnit": 0.120,
        "IngredientUnit": "kg",
        "IngredientRole": "CORE",
        "AssumptionRationale": "Base purchased chicken portion of approximately 120 g.",
    },
    {
        "CanonicalProductID": "PLU_42532",
        "CanonicalProductName": "€5 KIMBOCK",
        "RecipeFamily": "KIMBOCK",
        "PriceTierEuro": 5,
        "IngredientID": "ING_TERIYAKI_SAUCE",
        "IngredientName": "Teriyaki-style sauce",
        "QuantityPerProductUnit": 0.020,
        "IngredientUnit": "L",
        "IngredientRole": "CORE",
        "AssumptionRationale": "Approximately 20 ml sauce per base meal.",
    },
    {
        "CanonicalProductID": "PLU_42532",
        "CanonicalProductName": "€5 KIMBOCK",
        "RecipeFamily": "KIMBOCK",
        "PriceTierEuro": 5,
        "IngredientID": "ING_SESAME_OIL",
        "IngredientName": "Sesame oil",
        "QuantityPerProductUnit": 0.004,
        "IngredientUnit": "L",
        "IngredientRole": "CORE",
        "AssumptionRationale": "Small seasoning/cooking quantity of approximately 4 ml.",
    },

    {
        "CanonicalProductID": "PLU_42533",
        "CanonicalProductName": "€7 KIMBOCK",
        "RecipeFamily": "KIMBOCK",
        "PriceTierEuro": 7,
        "IngredientID": "ING_DRY_RICE",
        "IngredientName": "Dry white rice",
        "QuantityPerProductUnit": 0.100,
        "IngredientUnit": "kg",
        "IngredientRole": "CORE_LARGER_PORTION",
        "AssumptionRationale": "Larger rice portion than €5 tier.",
    },
    {
        "CanonicalProductID": "PLU_42533",
        "CanonicalProductName": "€7 KIMBOCK",
        "RecipeFamily": "KIMBOCK",
        "PriceTierEuro": 7,
        "IngredientID": "ING_CHICKEN_BREAST",
        "IngredientName": "Chicken breast",
        "QuantityPerProductUnit": 0.150,
        "IngredientUnit": "kg",
        "IngredientRole": "CORE_LARGER_PORTION",
        "AssumptionRationale": "Larger chicken portion than €5 tier.",
    },
    {
        "CanonicalProductID": "PLU_42533",
        "CanonicalProductName": "€7 KIMBOCK",
        "RecipeFamily": "KIMBOCK",
        "PriceTierEuro": 7,
        "IngredientID": "ING_MIXED_VEGETABLES",
        "IngredientName": "Mixed vegetables",
        "QuantityPerProductUnit": 0.080,
        "IngredientUnit": "kg",
        "IngredientRole": "ADDITIONAL_COMPONENT",
        "AssumptionRationale": "Additional vegetable component distinguishes the €7 tier.",
    },
    {
        "CanonicalProductID": "PLU_42533",
        "CanonicalProductName": "€7 KIMBOCK",
        "RecipeFamily": "KIMBOCK",
        "PriceTierEuro": 7,
        "IngredientID": "ING_TERIYAKI_SAUCE",
        "IngredientName": "Teriyaki-style sauce",
        "QuantityPerProductUnit": 0.025,
        "IngredientUnit": "L",
        "IngredientRole": "CORE_LARGER_PORTION",
        "AssumptionRationale": "Approximately 25 ml sauce for the larger meal.",
    },
    {
        "CanonicalProductID": "PLU_42533",
        "CanonicalProductName": "€7 KIMBOCK",
        "RecipeFamily": "KIMBOCK",
        "PriceTierEuro": 7,
        "IngredientID": "ING_SESAME_OIL",
        "IngredientName": "Sesame oil",
        "QuantityPerProductUnit": 0.005,
        "IngredientUnit": "L",
        "IngredientRole": "CORE_LARGER_PORTION",
        "AssumptionRationale": "Slightly larger seasoning/cooking quantity.",
    },

    {
        "CanonicalProductID": "PLU_42534",
        "CanonicalProductName": "€9 KIMBOCK",
        "RecipeFamily": "KIMBOCK",
        "PriceTierEuro": 9,
        "IngredientID": "ING_DRY_RICE",
        "IngredientName": "Dry white rice",
        "QuantityPerProductUnit": 0.120,
        "IngredientUnit": "kg",
        "IngredientRole": "CORE_LARGEST_PORTION",
        "AssumptionRationale": "Largest rice portion in the three-tier family.",
    },
    {
        "CanonicalProductID": "PLU_42534",
        "CanonicalProductName": "€9 KIMBOCK",
        "RecipeFamily": "KIMBOCK",
        "PriceTierEuro": 9,
        "IngredientID": "ING_CHICKEN_BREAST",
        "IngredientName": "Chicken breast",
        "QuantityPerProductUnit": 0.180,
        "IngredientUnit": "kg",
        "IngredientRole": "CORE_LARGEST_PORTION",
        "AssumptionRationale": "Largest chicken portion in the three-tier family.",
    },
    {
        "CanonicalProductID": "PLU_42534",
        "CanonicalProductName": "€9 KIMBOCK",
        "RecipeFamily": "KIMBOCK",
        "PriceTierEuro": 9,
        "IngredientID": "ING_MIXED_VEGETABLES",
        "IngredientName": "Mixed vegetables",
        "QuantityPerProductUnit": 0.100,
        "IngredientUnit": "kg",
        "IngredientRole": "ADDITIONAL_COMPONENT",
        "AssumptionRationale": "Larger vegetable component than the €7 tier.",
    },
    {
        "CanonicalProductID": "PLU_42534",
        "CanonicalProductName": "€9 KIMBOCK",
        "RecipeFamily": "KIMBOCK",
        "PriceTierEuro": 9,
        "IngredientID": "ING_EGG",
        "IngredientName": "Egg",
        "QuantityPerProductUnit": 1.0,
        "IngredientUnit": "each",
        "IngredientRole": "PREMIUM_ADDITIONAL_COMPONENT",
        "AssumptionRationale": "One additional egg provides the extra component for the €9 tier.",
    },
    {
        "CanonicalProductID": "PLU_42534",
        "CanonicalProductName": "€9 KIMBOCK",
        "RecipeFamily": "KIMBOCK",
        "PriceTierEuro": 9,
        "IngredientID": "ING_TERIYAKI_SAUCE",
        "IngredientName": "Teriyaki-style sauce",
        "QuantityPerProductUnit": 0.030,
        "IngredientUnit": "L",
        "IngredientRole": "CORE_LARGEST_PORTION",
        "AssumptionRationale": "Approximately 30 ml sauce for the largest meal.",
    },
    {
        "CanonicalProductID": "PLU_42534",
        "CanonicalProductName": "€9 KIMBOCK",
        "RecipeFamily": "KIMBOCK",
        "PriceTierEuro": 9,
        "IngredientID": "ING_SESAME_OIL",
        "IngredientName": "Sesame oil",
        "QuantityPerProductUnit": 0.006,
        "IngredientUnit": "L",
        "IngredientRole": "CORE_LARGEST_PORTION",
        "AssumptionRationale": "Largest seasoning/cooking quantity in the family.",
    },

    # -------------------------------------------------------------------------
    # DINNER FAMILY
    # Illustrative chicken-and-mashed-potato dinner family. The €7 and €9
    # versions increase core portions and add vegetables/stuffing.
    # -------------------------------------------------------------------------
    {
        "CanonicalProductID": "PLU_42529",
        "CanonicalProductName": "€5.00 DINNER",
        "RecipeFamily": "DINNER",
        "PriceTierEuro": 5,
        "IngredientID": "ING_CHICKEN_BREAST",
        "IngredientName": "Chicken breast",
        "QuantityPerProductUnit": 0.120,
        "IngredientUnit": "kg",
        "IngredientRole": "CORE",
        "AssumptionRationale": "Base illustrative chicken dinner portion.",
    },
    {
        "CanonicalProductID": "PLU_42529",
        "CanonicalProductName": "€5.00 DINNER",
        "RecipeFamily": "DINNER",
        "PriceTierEuro": 5,
        "IngredientID": "ING_POTATO",
        "IngredientName": "Potatoes",
        "QuantityPerProductUnit": 0.200,
        "IngredientUnit": "kg",
        "IngredientRole": "CORE",
        "AssumptionRationale": "Approximately 200 g raw potato for mashed-potato component.",
    },
    {
        "CanonicalProductID": "PLU_42529",
        "CanonicalProductName": "€5.00 DINNER",
        "RecipeFamily": "DINNER",
        "PriceTierEuro": 5,
        "IngredientID": "ING_BUTTER",
        "IngredientName": "Butter",
        "QuantityPerProductUnit": 0.010,
        "IngredientUnit": "kg",
        "IngredientRole": "CORE",
        "AssumptionRationale": "Approximately 10 g butter for mashed potato.",
    },
    {
        "CanonicalProductID": "PLU_42529",
        "CanonicalProductName": "€5.00 DINNER",
        "RecipeFamily": "DINNER",
        "PriceTierEuro": 5,
        "IngredientID": "ING_MILK",
        "IngredientName": "Milk",
        "QuantityPerProductUnit": 0.025,
        "IngredientUnit": "L",
        "IngredientRole": "CORE",
        "AssumptionRationale": "Approximately 25 ml milk for mashed potato.",
    },
    {
        "CanonicalProductID": "PLU_42529",
        "CanonicalProductName": "€5.00 DINNER",
        "RecipeFamily": "DINNER",
        "PriceTierEuro": 5,
        "IngredientID": "ING_GRAVY",
        "IngredientName": "Prepared gravy",
        "QuantityPerProductUnit": 0.040,
        "IngredientUnit": "L",
        "IngredientRole": "CORE",
        "AssumptionRationale": "Approximately 40 ml gravy for base dinner.",
    },

    {
        "CanonicalProductID": "PLU_42530",
        "CanonicalProductName": "€7 DINNER",
        "RecipeFamily": "DINNER",
        "PriceTierEuro": 7,
        "IngredientID": "ING_CHICKEN_BREAST",
        "IngredientName": "Chicken breast",
        "QuantityPerProductUnit": 0.150,
        "IngredientUnit": "kg",
        "IngredientRole": "CORE_LARGER_PORTION",
        "AssumptionRationale": "Larger chicken portion than €5 dinner.",
    },
    {
        "CanonicalProductID": "PLU_42530",
        "CanonicalProductName": "€7 DINNER",
        "RecipeFamily": "DINNER",
        "PriceTierEuro": 7,
        "IngredientID": "ING_POTATO",
        "IngredientName": "Potatoes",
        "QuantityPerProductUnit": 0.250,
        "IngredientUnit": "kg",
        "IngredientRole": "CORE_LARGER_PORTION",
        "AssumptionRationale": "Larger mashed-potato portion than €5 dinner.",
    },
    {
        "CanonicalProductID": "PLU_42530",
        "CanonicalProductName": "€7 DINNER",
        "RecipeFamily": "DINNER",
        "PriceTierEuro": 7,
        "IngredientID": "ING_BUTTER",
        "IngredientName": "Butter",
        "QuantityPerProductUnit": 0.012,
        "IngredientUnit": "kg",
        "IngredientRole": "CORE_LARGER_PORTION",
        "AssumptionRationale": "Butter scaled with larger mashed-potato portion.",
    },
    {
        "CanonicalProductID": "PLU_42530",
        "CanonicalProductName": "€7 DINNER",
        "RecipeFamily": "DINNER",
        "PriceTierEuro": 7,
        "IngredientID": "ING_MILK",
        "IngredientName": "Milk",
        "QuantityPerProductUnit": 0.030,
        "IngredientUnit": "L",
        "IngredientRole": "CORE_LARGER_PORTION",
        "AssumptionRationale": "Milk scaled with larger mashed-potato portion.",
    },
    {
        "CanonicalProductID": "PLU_42530",
        "CanonicalProductName": "€7 DINNER",
        "RecipeFamily": "DINNER",
        "PriceTierEuro": 7,
        "IngredientID": "ING_MIXED_VEGETABLES",
        "IngredientName": "Mixed vegetables",
        "QuantityPerProductUnit": 0.100,
        "IngredientUnit": "kg",
        "IngredientRole": "ADDITIONAL_COMPONENT",
        "AssumptionRationale": "Vegetable side distinguishes the €7 dinner from base tier.",
    },
    {
        "CanonicalProductID": "PLU_42530",
        "CanonicalProductName": "€7 DINNER",
        "RecipeFamily": "DINNER",
        "PriceTierEuro": 7,
        "IngredientID": "ING_GRAVY",
        "IngredientName": "Prepared gravy",
        "QuantityPerProductUnit": 0.050,
        "IngredientUnit": "L",
        "IngredientRole": "CORE_LARGER_PORTION",
        "AssumptionRationale": "Approximately 50 ml gravy.",
    },

    {
        "CanonicalProductID": "PLU_42531",
        "CanonicalProductName": "€9 DINNER",
        "RecipeFamily": "DINNER",
        "PriceTierEuro": 9,
        "IngredientID": "ING_CHICKEN_BREAST",
        "IngredientName": "Chicken breast",
        "QuantityPerProductUnit": 0.180,
        "IngredientUnit": "kg",
        "IngredientRole": "CORE_LARGEST_PORTION",
        "AssumptionRationale": "Largest illustrative chicken portion in dinner family.",
    },
    {
        "CanonicalProductID": "PLU_42531",
        "CanonicalProductName": "€9 DINNER",
        "RecipeFamily": "DINNER",
        "PriceTierEuro": 9,
        "IngredientID": "ING_POTATO",
        "IngredientName": "Potatoes",
        "QuantityPerProductUnit": 0.300,
        "IngredientUnit": "kg",
        "IngredientRole": "CORE_LARGEST_PORTION",
        "AssumptionRationale": "Largest mashed-potato portion in dinner family.",
    },
    {
        "CanonicalProductID": "PLU_42531",
        "CanonicalProductName": "€9 DINNER",
        "RecipeFamily": "DINNER",
        "PriceTierEuro": 9,
        "IngredientID": "ING_BUTTER",
        "IngredientName": "Butter",
        "QuantityPerProductUnit": 0.015,
        "IngredientUnit": "kg",
        "IngredientRole": "CORE_LARGEST_PORTION",
        "AssumptionRationale": "Butter scaled with largest mashed-potato portion.",
    },
    {
        "CanonicalProductID": "PLU_42531",
        "CanonicalProductName": "€9 DINNER",
        "RecipeFamily": "DINNER",
        "PriceTierEuro": 9,
        "IngredientID": "ING_MILK",
        "IngredientName": "Milk",
        "QuantityPerProductUnit": 0.035,
        "IngredientUnit": "L",
        "IngredientRole": "CORE_LARGEST_PORTION",
        "AssumptionRationale": "Milk scaled with largest mashed-potato portion.",
    },
    {
        "CanonicalProductID": "PLU_42531",
        "CanonicalProductName": "€9 DINNER",
        "RecipeFamily": "DINNER",
        "PriceTierEuro": 9,
        "IngredientID": "ING_MIXED_VEGETABLES",
        "IngredientName": "Mixed vegetables",
        "QuantityPerProductUnit": 0.120,
        "IngredientUnit": "kg",
        "IngredientRole": "ADDITIONAL_COMPONENT",
        "AssumptionRationale": "Larger vegetable side than €7 dinner.",
    },
    {
        "CanonicalProductID": "PLU_42531",
        "CanonicalProductName": "€9 DINNER",
        "RecipeFamily": "DINNER",
        "PriceTierEuro": 9,
        "IngredientID": "ING_GRAVY",
        "IngredientName": "Prepared gravy",
        "QuantityPerProductUnit": 0.060,
        "IngredientUnit": "L",
        "IngredientRole": "CORE_LARGEST_PORTION",
        "AssumptionRationale": "Approximately 60 ml gravy.",
    },
    {
        "CanonicalProductID": "PLU_42531",
        "CanonicalProductName": "€9 DINNER",
        "RecipeFamily": "DINNER",
        "PriceTierEuro": 9,
        "IngredientID": "ING_STUFFING",
        "IngredientName": "Prepared stuffing",
        "QuantityPerProductUnit": 0.060,
        "IngredientUnit": "kg",
        "IngredientRole": "PREMIUM_ADDITIONAL_COMPONENT",
        "AssumptionRationale": "Additional stuffing component distinguishes premium tier.",
    },

    # -------------------------------------------------------------------------
    # PORTION CHIPS
    # -------------------------------------------------------------------------
    {
        "CanonicalProductID": "PLU_42535",
        "CanonicalProductName": "PORTION CHIPS",
        "RecipeFamily": "CHIPS",
        "PriceTierEuro": np.nan,
        "IngredientID": "ING_FROZEN_CHIPS",
        "IngredientName": "Frozen potato chips",
        "QuantityPerProductUnit": 0.250,
        "IngredientUnit": "kg",
        "IngredientRole": "CORE",
        "AssumptionRationale": "Approximately 250 g purchased frozen chips per portion.",
    },
    {
        "CanonicalProductID": "PLU_42535",
        "CanonicalProductName": "PORTION CHIPS",
        "RecipeFamily": "CHIPS",
        "PriceTierEuro": np.nan,
        "IngredientID": "ING_FRYING_OIL",
        "IngredientName": "Frying oil",
        "QuantityPerProductUnit": 0.015,
        "IngredientUnit": "L",
        "IngredientRole": "COOKING",
        "AssumptionRationale": "Approximate net oil allocation per served portion.",
    },
    {
        "CanonicalProductID": "PLU_42535",
        "CanonicalProductName": "PORTION CHIPS",
        "RecipeFamily": "CHIPS",
        "PriceTierEuro": np.nan,
        "IngredientID": "ING_SALT",
        "IngredientName": "Salt",
        "QuantityPerProductUnit": 0.002,
        "IngredientUnit": "kg",
        "IngredientRole": "SEASONING",
        "AssumptionRationale": "Approximately 2 g salt per portion.",
    },

    # -------------------------------------------------------------------------
    # WRAP
    # -------------------------------------------------------------------------
    {
        "CanonicalProductID": "PLU_4241431",
        "CanonicalProductName": "WRAP",
        "RecipeFamily": "CHICKEN_WRAP",
        "PriceTierEuro": np.nan,
        "IngredientID": "ING_TORTILLA",
        "IngredientName": "Flour tortilla",
        "QuantityPerProductUnit": 1.0,
        "IngredientUnit": "each",
        "IngredientRole": "CORE",
        "AssumptionRationale": "One standard tortilla per wrap.",
    },
    {
        "CanonicalProductID": "PLU_4241431",
        "CanonicalProductName": "WRAP",
        "RecipeFamily": "CHICKEN_WRAP",
        "PriceTierEuro": np.nan,
        "IngredientID": "ING_CHICKEN_BREAST",
        "IngredientName": "Chicken breast",
        "QuantityPerProductUnit": 0.120,
        "IngredientUnit": "kg",
        "IngredientRole": "CORE",
        "AssumptionRationale": "Approximately 120 g purchased chicken per wrap.",
    },
    {
        "CanonicalProductID": "PLU_4241431",
        "CanonicalProductName": "WRAP",
        "RecipeFamily": "CHICKEN_WRAP",
        "PriceTierEuro": np.nan,
        "IngredientID": "ING_LETTUCE",
        "IngredientName": "Lettuce",
        "QuantityPerProductUnit": 0.040,
        "IngredientUnit": "kg",
        "IngredientRole": "SALAD",
        "AssumptionRationale": "Approximately 40 g lettuce.",
    },
    {
        "CanonicalProductID": "PLU_4241431",
        "CanonicalProductName": "WRAP",
        "RecipeFamily": "CHICKEN_WRAP",
        "PriceTierEuro": np.nan,
        "IngredientID": "ING_TOMATO",
        "IngredientName": "Tomato",
        "QuantityPerProductUnit": 0.040,
        "IngredientUnit": "kg",
        "IngredientRole": "SALAD",
        "AssumptionRationale": "Approximately 40 g sliced tomato.",
    },
    {
        "CanonicalProductID": "PLU_4241431",
        "CanonicalProductName": "WRAP",
        "RecipeFamily": "CHICKEN_WRAP",
        "PriceTierEuro": np.nan,
        "IngredientID": "ING_CHEDDAR",
        "IngredientName": "Cheddar cheese",
        "QuantityPerProductUnit": 0.025,
        "IngredientUnit": "kg",
        "IngredientRole": "FILLING",
        "AssumptionRationale": "Approximately 25 g cheese.",
    },
    {
        "CanonicalProductID": "PLU_4241431",
        "CanonicalProductName": "WRAP",
        "RecipeFamily": "CHICKEN_WRAP",
        "PriceTierEuro": np.nan,
        "IngredientID": "ING_MAYONNAISE",
        "IngredientName": "Mayonnaise",
        "QuantityPerProductUnit": 0.020,
        "IngredientUnit": "L",
        "IngredientRole": "SAUCE",
        "AssumptionRationale": "Approximately 20 ml mayonnaise.",
    },

    # -------------------------------------------------------------------------
    # HAM & CHEESE SANDWICH
    # -------------------------------------------------------------------------
    {
        "CanonicalProductID": "PLU_4241428",
        "CanonicalProductName": "HAM & CHEESE SANDWICH CT",
        "RecipeFamily": "HAM_CHEESE_SANDWICH",
        "PriceTierEuro": np.nan,
        "IngredientID": "ING_BREAD_SLICE",
        "IngredientName": "Bread slice",
        "QuantityPerProductUnit": 2.0,
        "IngredientUnit": "each",
        "IngredientRole": "CORE",
        "AssumptionRationale": "Two bread slices per sandwich.",
    },
    {
        "CanonicalProductID": "PLU_4241428",
        "CanonicalProductName": "HAM & CHEESE SANDWICH CT",
        "RecipeFamily": "HAM_CHEESE_SANDWICH",
        "PriceTierEuro": np.nan,
        "IngredientID": "ING_SLICED_HAM",
        "IngredientName": "Sliced ham",
        "QuantityPerProductUnit": 0.060,
        "IngredientUnit": "kg",
        "IngredientRole": "CORE",
        "AssumptionRationale": "Approximately 60 g ham.",
    },
    {
        "CanonicalProductID": "PLU_4241428",
        "CanonicalProductName": "HAM & CHEESE SANDWICH CT",
        "RecipeFamily": "HAM_CHEESE_SANDWICH",
        "PriceTierEuro": np.nan,
        "IngredientID": "ING_CHEDDAR",
        "IngredientName": "Cheddar cheese",
        "QuantityPerProductUnit": 0.040,
        "IngredientUnit": "kg",
        "IngredientRole": "CORE",
        "AssumptionRationale": "Approximately 40 g cheese.",
    },
    {
        "CanonicalProductID": "PLU_4241428",
        "CanonicalProductName": "HAM & CHEESE SANDWICH CT",
        "RecipeFamily": "HAM_CHEESE_SANDWICH",
        "PriceTierEuro": np.nan,
        "IngredientID": "ING_BUTTER",
        "IngredientName": "Butter",
        "QuantityPerProductUnit": 0.010,
        "IngredientUnit": "kg",
        "IngredientRole": "SPREAD",
        "AssumptionRationale": "Approximately 10 g butter.",
    },

    # -------------------------------------------------------------------------
    # DELI SANDWICH X3 SALAD X1 MEAT
    # -------------------------------------------------------------------------
    {
        "CanonicalProductID": "PLU_42544",
        "CanonicalProductName": "DELI SANDWICH X3 SALAD X1 MEAT",
        "RecipeFamily": "DELI_SANDWICH",
        "PriceTierEuro": np.nan,
        "IngredientID": "ING_SANDWICH_ROLL",
        "IngredientName": "Sandwich roll",
        "QuantityPerProductUnit": 1.0,
        "IngredientUnit": "each",
        "IngredientRole": "CORE",
        "AssumptionRationale": "One deli roll/bread unit.",
    },
    {
        "CanonicalProductID": "PLU_42544",
        "CanonicalProductName": "DELI SANDWICH X3 SALAD X1 MEAT",
        "RecipeFamily": "DELI_SANDWICH",
        "PriceTierEuro": np.nan,
        "IngredientID": "ING_DELI_MEAT",
        "IngredientName": "Deli meat",
        "QuantityPerProductUnit": 0.080,
        "IngredientUnit": "kg",
        "IngredientRole": "MEAT_ITEM_1_OF_1",
        "AssumptionRationale": "Approximately 80 g single meat filling.",
    },
    {
        "CanonicalProductID": "PLU_42544",
        "CanonicalProductName": "DELI SANDWICH X3 SALAD X1 MEAT",
        "RecipeFamily": "DELI_SANDWICH",
        "PriceTierEuro": np.nan,
        "IngredientID": "ING_LETTUCE",
        "IngredientName": "Lettuce",
        "QuantityPerProductUnit": 0.040,
        "IngredientUnit": "kg",
        "IngredientRole": "SALAD_ITEM_1_OF_3",
        "AssumptionRationale": "First illustrative salad filling, approximately 40 g.",
    },
    {
        "CanonicalProductID": "PLU_42544",
        "CanonicalProductName": "DELI SANDWICH X3 SALAD X1 MEAT",
        "RecipeFamily": "DELI_SANDWICH",
        "PriceTierEuro": np.nan,
        "IngredientID": "ING_TOMATO",
        "IngredientName": "Tomato",
        "QuantityPerProductUnit": 0.040,
        "IngredientUnit": "kg",
        "IngredientRole": "SALAD_ITEM_2_OF_3",
        "AssumptionRationale": "Second illustrative salad filling, approximately 40 g.",
    },
    {
        "CanonicalProductID": "PLU_42544",
        "CanonicalProductName": "DELI SANDWICH X3 SALAD X1 MEAT",
        "RecipeFamily": "DELI_SANDWICH",
        "PriceTierEuro": np.nan,
        "IngredientID": "ING_CUCUMBER",
        "IngredientName": "Cucumber",
        "QuantityPerProductUnit": 0.040,
        "IngredientUnit": "kg",
        "IngredientRole": "SALAD_ITEM_3_OF_3",
        "AssumptionRationale": "Third illustrative salad filling, approximately 40 g.",
    },
    {
        "CanonicalProductID": "PLU_42544",
        "CanonicalProductName": "DELI SANDWICH X3 SALAD X1 MEAT",
        "RecipeFamily": "DELI_SANDWICH",
        "PriceTierEuro": np.nan,
        "IngredientID": "ING_MAYONNAISE",
        "IngredientName": "Mayonnaise",
        "QuantityPerProductUnit": 0.020,
        "IngredientUnit": "L",
        "IngredientRole": "SAUCE",
        "AssumptionRationale": "Approximately 20 ml mayonnaise.",
    },

    # -------------------------------------------------------------------------
    # BREAKFAST SPECIAL 5 ITEMS
    # Exactly five illustrative breakfast components.
    # -------------------------------------------------------------------------
    {
        "CanonicalProductID": "PLU_36400048",
        "CanonicalProductName": "BREAKFAST SPECIAL 5 ITEMS",
        "RecipeFamily": "BREAKFAST_5_ITEM",
        "PriceTierEuro": np.nan,
        "IngredientID": "ING_SAUSAGE",
        "IngredientName": "Breakfast sausage",
        "QuantityPerProductUnit": 1.0,
        "IngredientUnit": "each",
        "IngredientRole": "BREAKFAST_ITEM_1_OF_5",
        "AssumptionRationale": "One sausage as one of five breakfast items.",
    },
    {
        "CanonicalProductID": "PLU_36400048",
        "CanonicalProductName": "BREAKFAST SPECIAL 5 ITEMS",
        "RecipeFamily": "BREAKFAST_5_ITEM",
        "PriceTierEuro": np.nan,
        "IngredientID": "ING_BACON_RASHER",
        "IngredientName": "Bacon rasher",
        "QuantityPerProductUnit": 1.0,
        "IngredientUnit": "each",
        "IngredientRole": "BREAKFAST_ITEM_2_OF_5",
        "AssumptionRationale": "One bacon rasher as one of five breakfast items.",
    },
    {
        "CanonicalProductID": "PLU_36400048",
        "CanonicalProductName": "BREAKFAST SPECIAL 5 ITEMS",
        "RecipeFamily": "BREAKFAST_5_ITEM",
        "PriceTierEuro": np.nan,
        "IngredientID": "ING_EGG",
        "IngredientName": "Egg",
        "QuantityPerProductUnit": 1.0,
        "IngredientUnit": "each",
        "IngredientRole": "BREAKFAST_ITEM_3_OF_5",
        "AssumptionRationale": "One egg as one of five breakfast items.",
    },
    {
        "CanonicalProductID": "PLU_36400048",
        "CanonicalProductName": "BREAKFAST SPECIAL 5 ITEMS",
        "RecipeFamily": "BREAKFAST_5_ITEM",
        "PriceTierEuro": np.nan,
        "IngredientID": "ING_HASH_BROWN",
        "IngredientName": "Hash brown",
        "QuantityPerProductUnit": 1.0,
        "IngredientUnit": "each",
        "IngredientRole": "BREAKFAST_ITEM_4_OF_5",
        "AssumptionRationale": "One hash brown as one of five breakfast items.",
    },
    {
        "CanonicalProductID": "PLU_36400048",
        "CanonicalProductName": "BREAKFAST SPECIAL 5 ITEMS",
        "RecipeFamily": "BREAKFAST_5_ITEM",
        "PriceTierEuro": np.nan,
        "IngredientID": "ING_BAKED_BEANS",
        "IngredientName": "Baked beans",
        "QuantityPerProductUnit": 0.100,
        "IngredientUnit": "kg",
        "IngredientRole": "BREAKFAST_ITEM_5_OF_5",
        "AssumptionRationale": "Approximately 100 g baked beans as fifth breakfast item.",
    },

    # -------------------------------------------------------------------------
    # SALAD
    # -------------------------------------------------------------------------
    {
        "CanonicalProductID": "PLU_42547",
        "CanonicalProductName": "SALAD",
        "RecipeFamily": "HOUSE_SALAD",
        "PriceTierEuro": np.nan,
        "IngredientID": "ING_LETTUCE",
        "IngredientName": "Lettuce",
        "QuantityPerProductUnit": 0.080,
        "IngredientUnit": "kg",
        "IngredientRole": "CORE",
        "AssumptionRationale": "Approximately 80 g lettuce base.",
    },
    {
        "CanonicalProductID": "PLU_42547",
        "CanonicalProductName": "SALAD",
        "RecipeFamily": "HOUSE_SALAD",
        "PriceTierEuro": np.nan,
        "IngredientID": "ING_TOMATO",
        "IngredientName": "Tomato",
        "QuantityPerProductUnit": 0.060,
        "IngredientUnit": "kg",
        "IngredientRole": "CORE",
        "AssumptionRationale": "Approximately 60 g tomato.",
    },
    {
        "CanonicalProductID": "PLU_42547",
        "CanonicalProductName": "SALAD",
        "RecipeFamily": "HOUSE_SALAD",
        "PriceTierEuro": np.nan,
        "IngredientID": "ING_CUCUMBER",
        "IngredientName": "Cucumber",
        "QuantityPerProductUnit": 0.050,
        "IngredientUnit": "kg",
        "IngredientRole": "CORE",
        "AssumptionRationale": "Approximately 50 g cucumber.",
    },
    {
        "CanonicalProductID": "PLU_42547",
        "CanonicalProductName": "SALAD",
        "RecipeFamily": "HOUSE_SALAD",
        "PriceTierEuro": np.nan,
        "IngredientID": "ING_MIXED_PEPPERS",
        "IngredientName": "Mixed peppers",
        "QuantityPerProductUnit": 0.040,
        "IngredientUnit": "kg",
        "IngredientRole": "CORE",
        "AssumptionRationale": "Approximately 40 g peppers.",
    },
    {
        "CanonicalProductID": "PLU_42547",
        "CanonicalProductName": "SALAD",
        "RecipeFamily": "HOUSE_SALAD",
        "PriceTierEuro": np.nan,
        "IngredientID": "ING_RED_ONION",
        "IngredientName": "Red onion",
        "QuantityPerProductUnit": 0.020,
        "IngredientUnit": "kg",
        "IngredientRole": "CORE",
        "AssumptionRationale": "Approximately 20 g red onion.",
    },
    {
        "CanonicalProductID": "PLU_42547",
        "CanonicalProductName": "SALAD",
        "RecipeFamily": "HOUSE_SALAD",
        "PriceTierEuro": np.nan,
        "IngredientID": "ING_SALAD_DRESSING",
        "IngredientName": "Salad dressing",
        "QuantityPerProductUnit": 0.030,
        "IngredientUnit": "L",
        "IngredientRole": "DRESSING",
        "AssumptionRationale": "Approximately 30 ml dressing.",
    },

    # -------------------------------------------------------------------------
    # SOUP — illustrative vegetable soup
    # -------------------------------------------------------------------------
    {
        "CanonicalProductID": "PLU_90318",
        "CanonicalProductName": "SOUP",
        "RecipeFamily": "VEGETABLE_SOUP",
        "PriceTierEuro": np.nan,
        "IngredientID": "ING_POTATO",
        "IngredientName": "Potatoes",
        "QuantityPerProductUnit": 0.100,
        "IngredientUnit": "kg",
        "IngredientRole": "SOUP_BASE",
        "AssumptionRationale": "Approximately 100 g potato in illustrative vegetable soup portion.",
    },
    {
        "CanonicalProductID": "PLU_90318",
        "CanonicalProductName": "SOUP",
        "RecipeFamily": "VEGETABLE_SOUP",
        "PriceTierEuro": np.nan,
        "IngredientID": "ING_CARROT",
        "IngredientName": "Carrot",
        "QuantityPerProductUnit": 0.080,
        "IngredientUnit": "kg",
        "IngredientRole": "SOUP_BASE",
        "AssumptionRationale": "Approximately 80 g carrot.",
    },
    {
        "CanonicalProductID": "PLU_90318",
        "CanonicalProductName": "SOUP",
        "RecipeFamily": "VEGETABLE_SOUP",
        "PriceTierEuro": np.nan,
        "IngredientID": "ING_ONION",
        "IngredientName": "Onion",
        "QuantityPerProductUnit": 0.050,
        "IngredientUnit": "kg",
        "IngredientRole": "SOUP_BASE",
        "AssumptionRationale": "Approximately 50 g onion.",
    },
    {
        "CanonicalProductID": "PLU_90318",
        "CanonicalProductName": "SOUP",
        "RecipeFamily": "VEGETABLE_SOUP",
        "PriceTierEuro": np.nan,
        "IngredientID": "ING_CELERY",
        "IngredientName": "Celery",
        "QuantityPerProductUnit": 0.030,
        "IngredientUnit": "kg",
        "IngredientRole": "SOUP_BASE",
        "AssumptionRationale": "Approximately 30 g celery.",
    },
    {
        "CanonicalProductID": "PLU_90318",
        "CanonicalProductName": "SOUP",
        "RecipeFamily": "VEGETABLE_SOUP",
        "PriceTierEuro": np.nan,
        "IngredientID": "ING_VEGETABLE_STOCK",
        "IngredientName": "Prepared vegetable stock",
        "QuantityPerProductUnit": 0.300,
        "IngredientUnit": "L",
        "IngredientRole": "SOUP_LIQUID",
        "AssumptionRationale": "Approximately 300 ml prepared stock per serving.",
    },
    {
        "CanonicalProductID": "PLU_90318",
        "CanonicalProductName": "SOUP",
        "RecipeFamily": "VEGETABLE_SOUP",
        "PriceTierEuro": np.nan,
        "IngredientID": "ING_CREAM",
        "IngredientName": "Cream",
        "QuantityPerProductUnit": 0.030,
        "IngredientUnit": "L",
        "IngredientRole": "SOUP_FINISH",
        "AssumptionRationale": "Approximately 30 ml cream for a finished soup portion.",
    },

    # -------------------------------------------------------------------------
    # SOUP & BREAD — same soup base plus bread roll and butter.
    # -------------------------------------------------------------------------
    {
        "CanonicalProductID": "PLU_42528",
        "CanonicalProductName": "SOUP & BREAD",
        "RecipeFamily": "VEGETABLE_SOUP_AND_BREAD",
        "PriceTierEuro": np.nan,
        "IngredientID": "ING_POTATO",
        "IngredientName": "Potatoes",
        "QuantityPerProductUnit": 0.100,
        "IngredientUnit": "kg",
        "IngredientRole": "SOUP_BASE",
        "AssumptionRationale": "Same illustrative soup portion as standalone SOUP.",
    },
    {
        "CanonicalProductID": "PLU_42528",
        "CanonicalProductName": "SOUP & BREAD",
        "RecipeFamily": "VEGETABLE_SOUP_AND_BREAD",
        "PriceTierEuro": np.nan,
        "IngredientID": "ING_CARROT",
        "IngredientName": "Carrot",
        "QuantityPerProductUnit": 0.080,
        "IngredientUnit": "kg",
        "IngredientRole": "SOUP_BASE",
        "AssumptionRationale": "Same illustrative soup portion as standalone SOUP.",
    },
    {
        "CanonicalProductID": "PLU_42528",
        "CanonicalProductName": "SOUP & BREAD",
        "RecipeFamily": "VEGETABLE_SOUP_AND_BREAD",
        "PriceTierEuro": np.nan,
        "IngredientID": "ING_ONION",
        "IngredientName": "Onion",
        "QuantityPerProductUnit": 0.050,
        "IngredientUnit": "kg",
        "IngredientRole": "SOUP_BASE",
        "AssumptionRationale": "Same illustrative soup portion as standalone SOUP.",
    },
    {
        "CanonicalProductID": "PLU_42528",
        "CanonicalProductName": "SOUP & BREAD",
        "RecipeFamily": "VEGETABLE_SOUP_AND_BREAD",
        "PriceTierEuro": np.nan,
        "IngredientID": "ING_CELERY",
        "IngredientName": "Celery",
        "QuantityPerProductUnit": 0.030,
        "IngredientUnit": "kg",
        "IngredientRole": "SOUP_BASE",
        "AssumptionRationale": "Same illustrative soup portion as standalone SOUP.",
    },
    {
        "CanonicalProductID": "PLU_42528",
        "CanonicalProductName": "SOUP & BREAD",
        "RecipeFamily": "VEGETABLE_SOUP_AND_BREAD",
        "PriceTierEuro": np.nan,
        "IngredientID": "ING_VEGETABLE_STOCK",
        "IngredientName": "Prepared vegetable stock",
        "QuantityPerProductUnit": 0.300,
        "IngredientUnit": "L",
        "IngredientRole": "SOUP_LIQUID",
        "AssumptionRationale": "Same illustrative soup portion as standalone SOUP.",
    },
    {
        "CanonicalProductID": "PLU_42528",
        "CanonicalProductName": "SOUP & BREAD",
        "RecipeFamily": "VEGETABLE_SOUP_AND_BREAD",
        "PriceTierEuro": np.nan,
        "IngredientID": "ING_CREAM",
        "IngredientName": "Cream",
        "QuantityPerProductUnit": 0.030,
        "IngredientUnit": "L",
        "IngredientRole": "SOUP_FINISH",
        "AssumptionRationale": "Same illustrative soup portion as standalone SOUP.",
    },
    {
        "CanonicalProductID": "PLU_42528",
        "CanonicalProductName": "SOUP & BREAD",
        "RecipeFamily": "VEGETABLE_SOUP_AND_BREAD",
        "PriceTierEuro": np.nan,
        "IngredientID": "ING_BREAD_ROLL",
        "IngredientName": "Bread roll",
        "QuantityPerProductUnit": 1.0,
        "IngredientUnit": "each",
        "IngredientRole": "ADDITIONAL_COMPONENT",
        "AssumptionRationale": "One bread roll served with soup.",
    },
    {
        "CanonicalProductID": "PLU_42528",
        "CanonicalProductName": "SOUP & BREAD",
        "RecipeFamily": "VEGETABLE_SOUP_AND_BREAD",
        "PriceTierEuro": np.nan,
        "IngredientID": "ING_BUTTER",
        "IngredientName": "Butter",
        "QuantityPerProductUnit": 0.010,
        "IngredientUnit": "kg",
        "IngredientRole": "ADDITIONAL_COMPONENT",
        "AssumptionRationale": "Approximately 10 g butter served with bread.",
    },
]


# =============================================================================
# HELPERS
# =============================================================================

def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


def write_csv(path: Path, frame: pd.DataFrame) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    frame.to_csv(path, index=False)


def write_json(path: Path, payload: dict) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(
        json.dumps(payload, indent=2, ensure_ascii=False, default=str) + "\n",
        encoding="utf-8",
    )


def write_text(path: Path, text: str) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(text, encoding="utf-8")


def atomic_write_text(path: Path, text: str) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_name(f".{path.name}.{uuid.uuid4().hex}.tmp")
    temporary.write_text(text, encoding="utf-8")
    os.replace(temporary, path)


def append_marked_section(path: Path, marker: str, section_text: str) -> None:
    existing = path.read_text(encoding="utf-8") if path.is_file() else ""
    if marker in existing:
        return
    separator = "\n" if existing.endswith("\n") else "\n\n"
    atomic_write_text(path, existing + separator + section_text.strip() + "\n")


def validate_required_columns(frame: pd.DataFrame, columns: set[str], name: str) -> None:
    missing = sorted(columns - set(frame.columns))
    if missing:
        raise AssertionError(
            f"{name} is missing required columns:\n"
            + "\n".join(f"- {column}" for column in missing)
        )


def build_manifest(root: Path) -> pd.DataFrame:
    records = []
    for path in sorted(root.rglob("*")):
        if not path.is_file():
            continue
        relative = path.relative_to(root)
        if relative.parts and relative.parts[0] == "07_control":
            continue
        records.append(
            {
                "RelativePath": str(relative),
                "Bytes": int(path.stat().st_size),
                "SHA256": sha256_file(path),
            }
        )
    return pd.DataFrame(records)


def save_figure(path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    plt.tight_layout()
    plt.savefig(path, dpi=300, bbox_inches="tight")
    plt.close()


def ingredient_requirement_rows(
    product_plan: pd.DataFrame,
    mapping: pd.DataFrame,
) -> pd.DataFrame:
    merged = product_plan.merge(
        mapping,
        on=[PRODUCT_ID_COLUMN, PRODUCT_NAME_COLUMN],
        how="inner",
        validate="many_to_many",
    )
    merged["NormalIngredientRequirement"] = (
        pd.to_numeric(merged[NORMAL_COLUMN], errors="raise")
        * pd.to_numeric(merged["QuantityPerProductUnit"], errors="raise")
    )
    merged["BulkIngredientRequirement"] = (
        pd.to_numeric(merged[BULK_COLUMN], errors="raise")
        * pd.to_numeric(merged["QuantityPerProductUnit"], errors="raise")
    )
    merged["PlannedIngredientRequirement"] = (
        pd.to_numeric(merged[PLANNED_COLUMN], errors="raise")
        * pd.to_numeric(merged["QuantityPerProductUnit"], errors="raise")
    )
    merged["IngredientMappingStatus"] = MAPPING_STATUS
    return merged


def aggregate_ingredient_requirements(
    frame: pd.DataFrame,
    group_columns: list[str],
) -> pd.DataFrame:
    return (
        frame.groupby(
            group_columns
            + ["IngredientID", "IngredientName", "IngredientUnit"],
            dropna=False,
            as_index=False,
        )
        .agg(
            NormalIngredientRequirement=(
                "NormalIngredientRequirement",
                "sum",
            ),
            BulkIngredientRequirement=(
                "BulkIngredientRequirement",
                "sum",
            ),
            PlannedIngredientRequirement=(
                "PlannedIngredientRequirement",
                "sum",
            ),
        )
        .sort_values(
            group_columns
            + ["IngredientUnit", "PlannedIngredientRequirement", "IngredientName"],
            ascending=[True] * len(group_columns) + [True, False, True],
            kind="mergesort",
        )
        .reset_index(drop=True)
    )


# =============================================================================
# PREFLIGHT
# =============================================================================

required_inputs = [
    ND09A_CHECKPOINT_PATH,
    ND10A_CHECKPOINT_PATH,
    ND10RA_CHECKPOINT_PATH,
    ACTIVE_CATALOGUE_PATH,
    DAILY_PRODUCT_PLANNING_PATH,
    DAILY_RESTAURANT_PLANNING_PATH,
    WEEK_DAILY_PRODUCT_PLANNING_PATH,
    WEEK_PRODUCT_TOTALS_PLANNING_PATH,
    WEEK_RESTAURANT_DAILY_PLANNING_PATH,
    WEEK_RESTAURANT_TOTAL_PLANNING_PATH,
    DEMONSTRATION_SUMMARY_PATH,
]

missing_inputs = [path for path in required_inputs if not path.is_file()]
if missing_inputs:
    raise FileNotFoundError(
        "ND11 required inputs are missing:\n"
        + "\n".join(f"- {path}" for path in missing_inputs)
    )

checkpoint_hashes = {
    "ND09A": sha256_file(ND09A_CHECKPOINT_PATH),
    "ND10A": sha256_file(ND10A_CHECKPOINT_PATH),
    "ND10RA": sha256_file(ND10RA_CHECKPOINT_PATH),
}
expected_hashes = {
    "ND09A": EXPECTED_ND09A_CHECKPOINT_SHA256,
    "ND10A": EXPECTED_ND10A_CHECKPOINT_SHA256,
    "ND10RA": EXPECTED_ND10RA_CHECKPOINT_SHA256,
}
for name, expected in expected_hashes.items():
    if checkpoint_hashes[name] != expected:
        raise AssertionError(
            f"{name} checkpoint hash mismatch.\n"
            f"Expected: {expected}\n"
            f"Actual:   {checkpoint_hashes[name]}"
        )

protected_hashes_before = {str(path): sha256_file(path) for path in required_inputs}

if ND11_ROOT.exists():
    if not ALLOW_OVERWRITE:
        raise FileExistsError(
            "ND11 output already exists. No files were changed:\n"
            f"{ND11_ROOT}"
        )
    shutil.rmtree(ND11_ROOT)

for path in [TOP_LEVEL_CHECKPOINT_PATH, TOP_LEVEL_CHECKPOINT_SHA_PATH]:
    if path.exists():
        if not ALLOW_OVERWRITE:
            raise FileExistsError(
                "An ND11 top-level checkpoint already exists. "
                f"No files were changed:\n{path}"
            )
        path.unlink()

STAGING_ROOT = ND11_ROOT.parent / f".ND11_staging_{uuid.uuid4().hex}"
STAGING_ROOT.mkdir(parents=True, exist_ok=False)


# =============================================================================
# MAIN
# =============================================================================

try:
    active_catalogue = pd.read_csv(ACTIVE_CATALOGUE_PATH, low_memory=False)
    daily_product = pd.read_csv(DAILY_PRODUCT_PLANNING_PATH, low_memory=False)
    daily_restaurant = pd.read_csv(
        DAILY_RESTAURANT_PLANNING_PATH, low_memory=False
    )
    week_daily_product = pd.read_csv(
        WEEK_DAILY_PRODUCT_PLANNING_PATH, low_memory=False
    )
    week_product_totals = pd.read_csv(
        WEEK_PRODUCT_TOTALS_PLANNING_PATH, low_memory=False
    )
    week_restaurant_daily = pd.read_csv(
        WEEK_RESTAURANT_DAILY_PLANNING_PATH, low_memory=False
    )
    week_restaurant_total = pd.read_csv(
        WEEK_RESTAURANT_TOTAL_PLANNING_PATH, low_memory=False
    )
    demonstration_summary = pd.read_csv(
        DEMONSTRATION_SUMMARY_PATH, low_memory=False
    )

    for frame in [daily_product, week_daily_product]:
        frame[DATE_COLUMN] = pd.to_datetime(frame[DATE_COLUMN], errors="raise")
        frame[PRODUCT_ID_COLUMN] = frame[PRODUCT_ID_COLUMN].astype(str)
        frame[PRODUCT_NAME_COLUMN] = frame[PRODUCT_NAME_COLUMN].astype(str)

    active_catalogue[PRODUCT_ID_COLUMN] = active_catalogue[
        PRODUCT_ID_COLUMN
    ].astype(str)

    validate_required_columns(
        daily_product,
        {
            DATE_COLUMN,
            PRODUCT_ID_COLUMN,
            PRODUCT_NAME_COLUMN,
            NORMAL_COLUMN,
            BULK_COLUMN,
            PLANNED_COLUMN,
        },
        "ND10A daily product planning",
    )
    validate_required_columns(
        week_daily_product,
        {
            DATE_COLUMN,
            PRODUCT_ID_COLUMN,
            PRODUCT_NAME_COLUMN,
            NORMAL_COLUMN,
            BULK_COLUMN,
            PLANNED_COLUMN,
        },
        "ND10A week daily product planning",
    )

    active_ids = set(daily_product[PRODUCT_ID_COLUMN].astype(str))
    if len(active_ids) != EXPECTED_ACTIVE_PRODUCTS:
        raise AssertionError(
            f"Expected {EXPECTED_ACTIVE_PRODUCTS} active forecast products; "
            f"found {len(active_ids)}."
        )
    if set(week_daily_product[PRODUCT_ID_COLUMN].astype(str)) != active_ids:
        raise AssertionError(
            "Daily and week planning outputs do not use the same active product catalogue."
        )

    mapping = pd.DataFrame(RECIPE_ROWS)
    mapping[PRODUCT_ID_COLUMN] = mapping[PRODUCT_ID_COLUMN].astype(str)
    mapping[PRODUCT_NAME_COLUMN] = mapping[PRODUCT_NAME_COLUMN].astype(str)
    mapping["IngredientMappingStatus"] = MAPPING_STATUS
    mapping["MappingApprovalNote"] = MAPPING_APPROVAL_NOTE
    mapping["ProcurementBasis"] = PROCUREMENT_BASIS
    mapping["YieldAndWasteTreatment"] = NO_YIELD_NOTE
    mapping["RecipeVersion"] = "ND11_ILLUSTRATIVE_V1"

    mapped_ids = set(mapping[PRODUCT_ID_COLUMN])
    if len(mapped_ids) != EXPECTED_MAPPED_PRODUCTS:
        raise AssertionError(
            f"Expected exactly {EXPECTED_MAPPED_PRODUCTS} mapped products; "
            f"found {len(mapped_ids)}."
        )

    missing_from_active = sorted(mapped_ids - active_ids)
    if missing_from_active:
        raise AssertionError(
            "Ingredient-mapped products are not present in the corrected "
            "88-product active catalogue:\n"
            + "\n".join(f"- {value}" for value in missing_from_active)
        )

    # Ensure each mapped product has one consistent canonical name and matches
    # the authoritative planning output.
    authoritative_names = (
        daily_product[[PRODUCT_ID_COLUMN, PRODUCT_NAME_COLUMN]]
        .drop_duplicates()
        .set_index(PRODUCT_ID_COLUMN)[PRODUCT_NAME_COLUMN]
        .to_dict()
    )
    name_mismatches = []
    for product_id, group in mapping.groupby(PRODUCT_ID_COLUMN):
        mapping_names = set(group[PRODUCT_NAME_COLUMN].astype(str))
        if len(mapping_names) != 1:
            name_mismatches.append(
                f"{product_id}: multiple mapping names {sorted(mapping_names)}"
            )
            continue
        mapping_name = next(iter(mapping_names))
        if authoritative_names.get(product_id) != mapping_name:
            name_mismatches.append(
                f"{product_id}: mapping='{mapping_name}', "
                f"planning='{authoritative_names.get(product_id)}'"
            )
    if name_mismatches:
        raise AssertionError(
            "Mapped product names do not match ND10A:\n"
            + "\n".join(f"- {value}" for value in name_mismatches)
        )

    if not np.isfinite(
        pd.to_numeric(mapping["QuantityPerProductUnit"], errors="coerce")
    ).all():
        raise AssertionError("Ingredient mapping contains non-finite quantities.")
    if (
        pd.to_numeric(mapping["QuantityPerProductUnit"], errors="raise") <= 0
    ).any():
        raise AssertionError("Ingredient mapping contains non-positive quantities.")

    # Ingredient IDs must map to exactly one name and one unit throughout.
    ingredient_consistency = (
        mapping.groupby("IngredientID", as_index=False)
        .agg(
            IngredientNameCount=("IngredientName", "nunique"),
            IngredientUnitCount=("IngredientUnit", "nunique"),
            IngredientName=("IngredientName", "first"),
            IngredientUnit=("IngredientUnit", "first"),
            ProductCount=(PRODUCT_ID_COLUMN, "nunique"),
        )
        .sort_values("IngredientID")
        .reset_index(drop=True)
    )
    ingredient_consistency["Passed"] = (
        (ingredient_consistency["IngredientNameCount"] == 1)
        & (ingredient_consistency["IngredientUnitCount"] == 1)
    )
    if not ingredient_consistency["Passed"].all():
        raise AssertionError(
            "At least one IngredientID uses inconsistent names or units:\n"
            + ingredient_consistency.loc[
                ~ingredient_consistency["Passed"]
            ].to_string(index=False)
        )

    ingredient_master = ingredient_consistency[
        [
            "IngredientID",
            "IngredientName",
            "IngredientUnit",
            "ProductCount",
        ]
    ].copy()
    ingredient_master["MappingStatus"] = MAPPING_STATUS

    # Validate the family-tier rule explicitly.
    family_tier_checks = []

    def quantity(product_id: str, ingredient_id: str) -> float:
        rows = mapping.loc[
            (mapping[PRODUCT_ID_COLUMN] == product_id)
            & (mapping["IngredientID"] == ingredient_id),
            "QuantityPerProductUnit",
        ]
        if len(rows) != 1:
            raise AssertionError(
                f"Expected one mapping row for {product_id}/{ingredient_id}; "
                f"found {len(rows)}."
            )
        return float(rows.iloc[0])

    for family_name, ids, core_ingredients in [
        (
            "KIMBOCK",
            ["PLU_42532", "PLU_42533", "PLU_42534"],
            ["ING_DRY_RICE", "ING_CHICKEN_BREAST", "ING_TERIYAKI_SAUCE"],
        ),
        (
            "DINNER",
            ["PLU_42529", "PLU_42530", "PLU_42531"],
            ["ING_CHICKEN_BREAST", "ING_POTATO", "ING_GRAVY"],
        ),
    ]:
        for ingredient_id in core_ingredients:
            values = [quantity(product_id, ingredient_id) for product_id in ids]
            passed = values[0] < values[1] < values[2]
            family_tier_checks.append(
                {
                    "RecipeFamily": family_name,
                    "Check": f"Increasing core quantity: {ingredient_id}",
                    "Tier5Value": values[0],
                    "Tier7Value": values[1],
                    "Tier9Value": values[2],
                    "Passed": passed,
                }
            )

    # Additional-component family checks.
    k5_ingredients = set(
        mapping.loc[mapping[PRODUCT_ID_COLUMN] == "PLU_42532", "IngredientID"]
    )
    k7_ingredients = set(
        mapping.loc[mapping[PRODUCT_ID_COLUMN] == "PLU_42533", "IngredientID"]
    )
    k9_ingredients = set(
        mapping.loc[mapping[PRODUCT_ID_COLUMN] == "PLU_42534", "IngredientID"]
    )
    d5_ingredients = set(
        mapping.loc[mapping[PRODUCT_ID_COLUMN] == "PLU_42529", "IngredientID"]
    )
    d7_ingredients = set(
        mapping.loc[mapping[PRODUCT_ID_COLUMN] == "PLU_42530", "IngredientID"]
    )
    d9_ingredients = set(
        mapping.loc[mapping[PRODUCT_ID_COLUMN] == "PLU_42531", "IngredientID"]
    )

    family_tier_checks.extend(
        [
            {
                "RecipeFamily": "KIMBOCK",
                "Check": "€7 adds component(s) beyond €5",
                "Tier5Value": len(k5_ingredients),
                "Tier7Value": len(k7_ingredients),
                "Tier9Value": len(k9_ingredients),
                "Passed": k5_ingredients < k7_ingredients,
            },
            {
                "RecipeFamily": "KIMBOCK",
                "Check": "€9 adds component(s) beyond €7",
                "Tier5Value": len(k5_ingredients),
                "Tier7Value": len(k7_ingredients),
                "Tier9Value": len(k9_ingredients),
                "Passed": k7_ingredients < k9_ingredients,
            },
            {
                "RecipeFamily": "DINNER",
                "Check": "€7 adds component(s) beyond €5",
                "Tier5Value": len(d5_ingredients),
                "Tier7Value": len(d7_ingredients),
                "Tier9Value": len(d9_ingredients),
                "Passed": d5_ingredients < d7_ingredients,
            },
            {
                "RecipeFamily": "DINNER",
                "Check": "€9 adds component(s) beyond €7",
                "Tier5Value": len(d5_ingredients),
                "Tier7Value": len(d7_ingredients),
                "Tier9Value": len(d9_ingredients),
                "Passed": d7_ingredients < d9_ingredients,
            },
        ]
    )

    family_tier_audit = pd.DataFrame(family_tier_checks)
    if not family_tier_audit["Passed"].all():
        raise AssertionError(
            "The €5/€7/€9 family scaling rule failed:\n"
            + family_tier_audit.loc[~family_tier_audit["Passed"]].to_string(
                index=False
            )
        )

    mapped_product_catalogue = (
        mapping[
            [
                PRODUCT_ID_COLUMN,
                PRODUCT_NAME_COLUMN,
                "RecipeFamily",
                "PriceTierEuro",
                "RecipeVersion",
                "IngredientMappingStatus",
            ]
        ]
        .drop_duplicates()
        .sort_values(
            ["RecipeFamily", "PriceTierEuro", PRODUCT_NAME_COLUMN],
            na_position="last",
        )
        .reset_index(drop=True)
    )
    mapped_product_catalogue["IsInActiveForecastCatalogue"] = (
        mapped_product_catalogue[PRODUCT_ID_COLUMN].isin(active_ids)
    )

    recipe_family_summary = (
        mapping.groupby("RecipeFamily", as_index=False)
        .agg(
            ProductsMapped=(PRODUCT_ID_COLUMN, "nunique"),
            IngredientRows=("IngredientID", "size"),
            UniqueIngredients=("IngredientID", "nunique"),
        )
        .sort_values(["ProductsMapped", "RecipeFamily"], ascending=[False, True])
        .reset_index(drop=True)
    )

    assumption_audit = (
        mapping[
            [
                PRODUCT_ID_COLUMN,
                PRODUCT_NAME_COLUMN,
                "RecipeFamily",
                "PriceTierEuro",
                "IngredientID",
                "IngredientName",
                "QuantityPerProductUnit",
                "IngredientUnit",
                "IngredientRole",
                "AssumptionRationale",
                "IngredientMappingStatus",
                "MappingApprovalNote",
                "ProcurementBasis",
                "YieldAndWasteTreatment",
            ]
        ]
        .copy()
        .sort_values(
            [PRODUCT_NAME_COLUMN, "IngredientRole", "IngredientName"],
            kind="mergesort",
        )
        .reset_index(drop=True)
    )

    # -------------------------------------------------------------------------
    # Apply mapping to corrected planning outputs.
    # -------------------------------------------------------------------------
    daily_product_ingredient = ingredient_requirement_rows(
        daily_product,
        mapping,
    )
    week_daily_product_ingredient = ingredient_requirement_rows(
        week_daily_product,
        mapping,
    )

    daily_ingredient_totals = aggregate_ingredient_requirements(
        daily_product_ingredient,
        [DATE_COLUMN],
    )
    week_daily_ingredient_totals = aggregate_ingredient_requirements(
        week_daily_product_ingredient,
        [DATE_COLUMN],
    )
    week_ingredient_totals = aggregate_ingredient_requirements(
        week_daily_product_ingredient,
        [],
    )

    # -------------------------------------------------------------------------
    # Mapping coverage
    # -------------------------------------------------------------------------
    daily_mapped_mask = daily_product[PRODUCT_ID_COLUMN].isin(mapped_ids)
    week_mapped_mask = week_daily_product[PRODUCT_ID_COLUMN].isin(mapped_ids)

    daily_total_planned = float(
        pd.to_numeric(daily_product[PLANNED_COLUMN], errors="raise").sum()
    )
    daily_mapped_planned = float(
        pd.to_numeric(
            daily_product.loc[daily_mapped_mask, PLANNED_COLUMN],
            errors="raise",
        ).sum()
    )
    week_total_planned = float(
        pd.to_numeric(week_daily_product[PLANNED_COLUMN], errors="raise").sum()
    )
    week_mapped_planned = float(
        pd.to_numeric(
            week_daily_product.loc[week_mapped_mask, PLANNED_COLUMN],
            errors="raise",
        ).sum()
    )

    daily_coverage_pct = (
        100.0 * daily_mapped_planned / daily_total_planned
        if daily_total_planned > 0
        else float("nan")
    )
    week_coverage_pct = (
        100.0 * week_mapped_planned / week_total_planned
        if week_total_planned > 0
        else float("nan")
    )

    mapping_coverage = pd.DataFrame(
        [
            {
                "Context": "ACTIVE_PRODUCT_CATALOGUE",
                "MappedProducts": EXPECTED_MAPPED_PRODUCTS,
                "TotalProducts": EXPECTED_ACTIVE_PRODUCTS,
                "CoveragePercentage": (
                    100.0 * EXPECTED_MAPPED_PRODUCTS / EXPECTED_ACTIVE_PRODUCTS
                ),
                "CoverageBasis": "PRODUCT_COUNT",
            },
            {
                "Context": "DAILY_DEMONSTRATION_PLANNED_QUANTITY",
                "MappedProducts": EXPECTED_MAPPED_PRODUCTS,
                "TotalProducts": EXPECTED_ACTIVE_PRODUCTS,
                "CoveragePercentage": daily_coverage_pct,
                "CoverageBasis": "PLANNED_PRODUCT_UNITS",
            },
            {
                "Context": "WEEK_DEMONSTRATION_PLANNED_QUANTITY",
                "MappedProducts": EXPECTED_MAPPED_PRODUCTS,
                "TotalProducts": EXPECTED_ACTIVE_PRODUCTS,
                "CoveragePercentage": week_coverage_pct,
                "CoverageBasis": "PLANNED_PRODUCT_UNITS",
            },
        ]
    )

    # -------------------------------------------------------------------------
    # Formula reconciliation
    # -------------------------------------------------------------------------
    formula_records = []
    for context, requirement_frame in [
        ("DAILY", daily_product_ingredient),
        ("WEEK_DAILY_ROWS", week_daily_product_ingredient),
    ]:
        expected_normal = (
            pd.to_numeric(
                requirement_frame[NORMAL_COLUMN], errors="raise"
            )
            * pd.to_numeric(
                requirement_frame["QuantityPerProductUnit"], errors="raise"
            )
        )
        expected_bulk = (
            pd.to_numeric(
                requirement_frame[BULK_COLUMN], errors="raise"
            )
            * pd.to_numeric(
                requirement_frame["QuantityPerProductUnit"], errors="raise"
            )
        )
        expected_planned = (
            pd.to_numeric(
                requirement_frame[PLANNED_COLUMN], errors="raise"
            )
            * pd.to_numeric(
                requirement_frame["QuantityPerProductUnit"], errors="raise"
            )
        )
        diffs = {
            "Normal": float(
                np.max(
                    np.abs(
                        expected_normal.to_numpy(dtype=float)
                        - requirement_frame[
                            "NormalIngredientRequirement"
                        ].to_numpy(dtype=float)
                    )
                )
            )
            if len(requirement_frame)
            else 0.0,
            "Bulk": float(
                np.max(
                    np.abs(
                        expected_bulk.to_numpy(dtype=float)
                        - requirement_frame[
                            "BulkIngredientRequirement"
                        ].to_numpy(dtype=float)
                    )
                )
            )
            if len(requirement_frame)
            else 0.0,
            "Planned": float(
                np.max(
                    np.abs(
                        expected_planned.to_numpy(dtype=float)
                        - requirement_frame[
                            "PlannedIngredientRequirement"
                        ].to_numpy(dtype=float)
                    )
                )
            )
            if len(requirement_frame)
            else 0.0,
        }
        for component, difference in diffs.items():
            formula_records.append(
                {
                    "Context": context,
                    "Component": component,
                    "MaximumAbsoluteDifference": difference,
                    "Passed": difference <= 1e-12,
                }
            )

    formula_reconciliation = pd.DataFrame(formula_records)

    mapped_product_validation = mapped_product_catalogue[
        [
            PRODUCT_ID_COLUMN,
            PRODUCT_NAME_COLUMN,
            "RecipeFamily",
            "PriceTierEuro",
            "IsInActiveForecastCatalogue",
        ]
    ].copy()
    mapped_product_validation["Passed"] = (
        mapped_product_validation["IsInActiveForecastCatalogue"]
    )

    # -------------------------------------------------------------------------
    # General validation
    # -------------------------------------------------------------------------
    validation = pd.DataFrame(
        [
            {
                "Check": "Exact ND09A checkpoint verified",
                "Passed": (
                    checkpoint_hashes["ND09A"]
                    == EXPECTED_ND09A_CHECKPOINT_SHA256
                ),
            },
            {
                "Check": "Exact ND10A checkpoint verified",
                "Passed": (
                    checkpoint_hashes["ND10A"]
                    == EXPECTED_ND10A_CHECKPOINT_SHA256
                ),
            },
            {
                "Check": "Exact ND10RA checkpoint verified",
                "Passed": (
                    checkpoint_hashes["ND10RA"]
                    == EXPECTED_ND10RA_CHECKPOINT_SHA256
                ),
            },
            {
                "Check": "Corrected active catalogue contains 88 products",
                "Passed": len(active_ids) == EXPECTED_ACTIVE_PRODUCTS,
            },
            {
                "Check": "Exactly 14 products are ingredient mapped",
                "Passed": len(mapped_ids) == EXPECTED_MAPPED_PRODUCTS,
            },
            {
                "Check": "All 14 mapped products are active",
                "Passed": len(missing_from_active) == 0,
            },
            {
                "Check": "Ingredient IDs use one name and one unit",
                "Passed": bool(ingredient_consistency["Passed"].all()),
            },
            {
                "Check": "All recipe quantities are positive and finite",
                "Passed": bool(
                    np.isfinite(
                        mapping["QuantityPerProductUnit"].astype(float)
                    ).all()
                    and (mapping["QuantityPerProductUnit"] > 0).all()
                ),
            },
            {
                "Check": "KIMBOCK/DINNER tier scaling rules passed",
                "Passed": bool(family_tier_audit["Passed"].all()),
            },
            {
                "Check": "Ingredient formula reconciliation passed",
                "Passed": bool(formula_reconciliation["Passed"].all()),
            },
            {
                "Check": "All mapping rows are explicitly illustrative",
                "Passed": bool(
                    (
                        mapping["IngredientMappingStatus"]
                        == MAPPING_STATUS
                    ).all()
                ),
            },
            {
                "Check": "No March target or accuracy input loaded",
                "Passed": True,
            },
            {
                "Check": "Forecasting models fitted/refitted",
                "Passed": True,
            },
        ]
    )
    # The final check is named as a safety statement: True means no fitting
    # occurred in ND11.
    validation.loc[
        validation["Check"] == "Forecasting models fitted/refitted",
        "Check",
    ] = "No forecasting model fitted/refitted"
    if not validation["Passed"].all():
        raise AssertionError(
            "ND11 validation failed:\n"
            + validation.loc[~validation["Passed"]].to_string(index=False)
        )

    # -------------------------------------------------------------------------
    # Staged outputs
    # -------------------------------------------------------------------------
    staged_mapping_dir = STAGING_ROOT / "01_mapping"
    staged_requirement_dir = STAGING_ROOT / "02_ingredient_requirements"
    staged_demo_backend_dir = STAGING_ROOT / "03_demonstration_backend"
    staged_audit_dir = STAGING_ROOT / "04_audits"
    staged_figure_dir = STAGING_ROOT / "05_figures"
    staged_report_dir = STAGING_ROOT / "06_reports"
    staged_control_dir = STAGING_ROOT / "07_control"

    for directory in [
        staged_mapping_dir,
        staged_requirement_dir,
        staged_demo_backend_dir,
        staged_audit_dir,
        staged_figure_dir,
        staged_report_dir,
        staged_control_dir,
    ]:
        directory.mkdir(parents=True, exist_ok=True)

    write_csv(staged_mapping_dir / PRODUCT_MAPPING_PATH.name, mapping)
    write_csv(
        staged_mapping_dir / MAPPED_PRODUCT_CATALOGUE_PATH.name,
        mapped_product_catalogue,
    )
    write_csv(staged_mapping_dir / INGREDIENT_MASTER_PATH.name, ingredient_master)
    write_csv(
        staged_mapping_dir / RECIPE_FAMILY_SUMMARY_PATH.name,
        recipe_family_summary,
    )

    write_csv(
        staged_requirement_dir / DAILY_PRODUCT_INGREDIENT_PATH.name,
        daily_product_ingredient,
    )
    write_csv(
        staged_requirement_dir / DAILY_INGREDIENT_TOTALS_PATH.name,
        daily_ingredient_totals,
    )
    write_csv(
        staged_requirement_dir / WEEK_DAILY_PRODUCT_INGREDIENT_PATH.name,
        week_daily_product_ingredient,
    )
    write_csv(
        staged_requirement_dir / WEEK_DAILY_INGREDIENT_TOTALS_PATH.name,
        week_daily_ingredient_totals,
    )
    write_csv(
        staged_requirement_dir / WEEK_INGREDIENT_TOTALS_PATH.name,
        week_ingredient_totals,
    )

    write_csv(
        staged_audit_dir / MAPPING_COVERAGE_PATH.name,
        mapping_coverage,
    )
    write_csv(
        staged_audit_dir / UNIT_CONSISTENCY_PATH.name,
        ingredient_consistency,
    )
    write_csv(
        staged_audit_dir / FORMULA_RECONCILIATION_PATH.name,
        formula_reconciliation,
    )
    write_csv(
        staged_audit_dir / MAPPED_PRODUCT_VALIDATION_PATH.name,
        mapped_product_validation,
    )
    write_csv(
        staged_audit_dir / ASSUMPTION_AUDIT_PATH.name,
        assumption_audit,
    )
    write_csv(staged_audit_dir / VALIDATION_PATH.name, validation)
    write_csv(
        staged_audit_dir / "ND11_family_tier_rule_audit.csv",
        family_tier_audit,
    )

    # -------------------------------------------------------------------------
    # Demonstration backend export.
    # All 88 product forecasts remain available. Ingredient fields are an
    # optional downstream layer for only the 14 mapped products.
    # -------------------------------------------------------------------------
    daily_backend = daily_product.copy()
    daily_backend["HasIllustrativeIngredientMapping"] = (
        daily_backend[PRODUCT_ID_COLUMN].isin(mapped_ids)
    )

    week_daily_backend = week_daily_product.copy()
    week_daily_backend["HasIllustrativeIngredientMapping"] = (
        week_daily_backend[PRODUCT_ID_COLUMN].isin(mapped_ids)
    )

    week_totals_backend = week_product_totals.copy()
    week_totals_backend["HasIllustrativeIngredientMapping"] = (
        week_totals_backend[PRODUCT_ID_COLUMN].astype(str).isin(mapped_ids)
    )

    write_csv(
        staged_demo_backend_dir / BACKEND_DAILY_PRODUCT_PATH.name,
        daily_backend,
    )
    write_csv(
        staged_demo_backend_dir / BACKEND_DAILY_RESTAURANT_PATH.name,
        daily_restaurant,
    )
    write_csv(
        staged_demo_backend_dir / BACKEND_WEEK_DAILY_PRODUCT_PATH.name,
        week_daily_backend,
    )
    write_csv(
        staged_demo_backend_dir / BACKEND_WEEK_PRODUCT_TOTALS_PATH.name,
        week_totals_backend,
    )
    write_csv(
        staged_demo_backend_dir / BACKEND_WEEK_RESTAURANT_DAILY_PATH.name,
        week_restaurant_daily,
    )
    write_csv(
        staged_demo_backend_dir / BACKEND_WEEK_RESTAURANT_TOTAL_PATH.name,
        week_restaurant_total,
    )
    write_csv(
        staged_demo_backend_dir / BACKEND_INGREDIENT_MAPPING_PATH.name,
        mapping,
    )
    write_csv(
        staged_demo_backend_dir / BACKEND_DAILY_INGREDIENT_PATH.name,
        daily_ingredient_totals,
    )
    write_csv(
        staged_demo_backend_dir / BACKEND_WEEK_DAILY_INGREDIENT_PATH.name,
        week_daily_ingredient_totals,
    )
    write_csv(
        staged_demo_backend_dir / BACKEND_WEEK_INGREDIENT_PATH.name,
        week_ingredient_totals,
    )

    demo_contract = {
        "ContractVersion": "ND11_DEMO_BACKEND_V1",
        "ActiveForecastCatalogueProducts": EXPECTED_ACTIVE_PRODUCTS,
        "IngredientMappedProducts": EXPECTED_MAPPED_PRODUCTS,
        "IngredientMappingStatus": MAPPING_STATUS,
        "IngredientMappingIsIllustrative": True,
        "IngredientMappingIsEdenRecipeData": False,
        "MappingApprovalContext": MAPPING_APPROVAL_NOTE,
        "ProcurementBasis": PROCUREMENT_BASIS,
        "YieldAndWasteTreatment": NO_YIELD_NOTE,
        "PlanningFormula": (
            "PlannedQuantity = PredictedNormalDemand + ConfirmedBulkDemand"
        ),
        "IngredientFormula": (
            "PlannedIngredientRequirement = PlannedQuantity * "
            "QuantityPerProductUnit"
        ),
        "ForecastScopeRule": (
            "Forecast all 88 products in the latest known active panel."
        ),
        "IngredientScopeRule": (
            "Ingredient requirements are available only for the 14 explicitly "
            "mapped prototype products; unmapped products remain fully forecast "
            "and planned but have no ingredient breakdown."
        ),
        "MappedProductIDs": sorted(mapped_ids),
        "BackendFiles": {
            "DailyProductPlan": BACKEND_DAILY_PRODUCT_PATH.name,
            "DailyRestaurantPlan": BACKEND_DAILY_RESTAURANT_PATH.name,
            "WeekDailyProductPlan": BACKEND_WEEK_DAILY_PRODUCT_PATH.name,
            "WeekProductTotals": BACKEND_WEEK_PRODUCT_TOTALS_PATH.name,
            "WeekRestaurantDaily": BACKEND_WEEK_RESTAURANT_DAILY_PATH.name,
            "WeekRestaurantTotal": BACKEND_WEEK_RESTAURANT_TOTAL_PATH.name,
            "IngredientMapping": BACKEND_INGREDIENT_MAPPING_PATH.name,
            "DailyIngredientRequirements": BACKEND_DAILY_INGREDIENT_PATH.name,
            "WeekDailyIngredientRequirements": BACKEND_WEEK_DAILY_INGREDIENT_PATH.name,
            "WeekIngredientRequirements": BACKEND_WEEK_INGREDIENT_PATH.name,
        },
        "UIIngredientDisclaimer": (
            "Prototype ingredient estimates based on illustrative assumed "
            "recipes, not Eden Restaurant's operational recipes."
        ),
        "MarchTargetVaultOpened": False,
        "ForecastingModelModified": False,
    }
    write_json(
        staged_demo_backend_dir / BACKEND_CONTRACT_PATH.name,
        demo_contract,
    )

    disclaimer = f"""# Ingredient Mapping Disclaimer

The ingredient mappings in this demonstration are **illustrative assumed recipes**.

Eden Restaurant did not provide recipe-level ingredient quantities within the project timeframe and permitted assumed mappings to be used for the academic prototype.

These values:

- do not represent Eden Restaurant's actual recipes;
- do not represent Eden procurement specifications;
- are not used to train, tune, select or evaluate the forecasting model;
- are realistic serving/procurement assumptions created only to demonstrate the conversion from planned product demand to ingredient requirements;
- do not include cooking loss, trimming loss, spoilage allowance, safety stock, pack-size rounding or supplier minimum-order quantities.

The forecasting system continues to plan all {EXPECTED_ACTIVE_PRODUCTS} products in the corrected active catalogue. Ingredient breakdown is available only for the {EXPECTED_MAPPED_PRODUCTS} selected prototype products.
"""
    write_text(
        staged_demo_backend_dir / BACKEND_DISCLAIMER_PATH.name,
        disclaimer,
    )

    # -------------------------------------------------------------------------
    # Figures
    # Separate charts by unit so incompatible units are never combined.
    # -------------------------------------------------------------------------
    for unit in sorted(week_ingredient_totals["IngredientUnit"].astype(str).unique()):
        unit_frame = (
            week_ingredient_totals.loc[
                week_ingredient_totals["IngredientUnit"].astype(str) == unit
            ]
            .nlargest(15, "PlannedIngredientRequirement")
            .sort_values("PlannedIngredientRequirement")
        )
        plt.figure(figsize=(10, 7))
        plt.barh(
            unit_frame["IngredientName"].astype(str),
            unit_frame["PlannedIngredientRequirement"].astype(float),
        )
        plt.xlabel(f"Planned ingredient requirement ({unit})")
        plt.ylabel("Ingredient")
        plt.title(
            f"ND11 illustrative weekly ingredient requirements — {unit}"
        )
        safe_unit = (
            unit.replace("/", "_")
            .replace(" ", "_")
            .replace(".", "_")
        )
        save_figure(
            staged_figure_dir
            / f"ND11_weekly_ingredient_requirements_{safe_unit}.png"
        )

    # Coverage figure.
    plt.figure(figsize=(8, 5))
    coverage_plot = mapping_coverage.copy()
    plt.bar(
        coverage_plot["Context"].astype(str),
        coverage_plot["CoveragePercentage"].astype(float),
    )
    plt.ylabel("Coverage (%)")
    plt.title("ND11 illustrative ingredient-mapping coverage")
    plt.xticks(rotation=25, ha="right")
    save_figure(
        staged_figure_dir / "ND11_ingredient_mapping_coverage.png"
    )

    daily_date = pd.Timestamp(daily_product[DATE_COLUMN].iloc[0]).normalize()
    week_start = pd.Timestamp(week_daily_product[DATE_COLUMN].min()).normalize()
    week_end = pd.Timestamp(week_daily_product[DATE_COLUMN].max()).normalize()

    report_summary = f"""# ND11 Illustrative Ingredient Mapping Summary

## Status

`{STATUS}`

## Mapping scope

The corrected forecasting system contains {EXPECTED_ACTIVE_PRODUCTS} active products. Ingredient mapping was intentionally limited to {EXPECTED_MAPPED_PRODUCTS} selected prepared-food products for the prototype.

Mapped products:

{chr(10).join('- ' + value for value in mapped_product_catalogue[PRODUCT_NAME_COLUMN].tolist())}

## Recipe assumptions

The mapping is illustrative and assumed. Eden Restaurant did not provide operational recipe quantities within the project timeframe and permitted assumed recipes to be used for the academic prototype.

The €5, €7 and €9 KIMBOCK products share the same meal family. Higher tiers increase rice and chicken quantities, with vegetables introduced at €7 and an additional egg at €9.

The €5, €7 and €9 DINNER products share an illustrative chicken, mashed-potato and gravy base. Higher tiers increase core portion quantities, add vegetables at €7 and add stuffing at €9.

All other mapped products use plausible single-serving procurement quantities.

## Coverage

- Active forecast products: {EXPECTED_ACTIVE_PRODUCTS}
- Ingredient-mapped products: {EXPECTED_MAPPED_PRODUCTS}
- Product-count mapping coverage: {100.0 * EXPECTED_MAPPED_PRODUCTS / EXPECTED_ACTIVE_PRODUCTS:.2f}%
- Daily planned-unit coverage: {daily_coverage_pct:.2f}%
- Week planned-unit coverage: {week_coverage_pct:.2f}%

## Calculation

`PlannedIngredientRequirement = PlannedQuantity × QuantityPerProductUnit`

The normal-demand and confirmed-bulk components are preserved separately in every ingredient-requirement file.

## Demonstration boundary

The UI should forecast and plan all {EXPECTED_ACTIVE_PRODUCTS} active products. The optional ingredient view should display ingredient estimates only for the {EXPECTED_MAPPED_PRODUCTS} mapped products and clearly show the illustrative-recipe disclaimer.

No cooking-loss, spoilage, safety-stock or pack-size factor is included.
"""
    write_text(
        staged_report_dir / REPORT_SUMMARY_PATH.name,
        report_summary,
    )

    report_wording = f"""# ND11 Report-Ready Ingredient Mapping Wording

Ingredient-level recipe data were unavailable from Eden Restaurant within the project timeframe. With permission from the restaurant, an illustrative ingredient-mapping layer was therefore developed for fourteen selected high-volume and representative prepared-food products. The mappings were created solely to demonstrate how predicted product demand can be translated into estimated ingredient requirements and do not represent Eden Restaurant's operational recipes or procurement specifications.

The ingredient layer was deliberately kept separate from the forecasting model. The forecasting system continued to generate planned quantities for all {EXPECTED_ACTIVE_PRODUCTS} products in the latest active catalogue, while ingredient conversion was applied only to the fourteen prototype-mapped products. This preserved the integrity of the demand forecast while providing a proof of concept for downstream purchasing support.

For the KIMBOCK and DINNER product families, the €5, €7 and €9 variants were modelled as tiered versions of the same underlying meal rather than unrelated recipes. Core ingredient quantities increase with price tier and additional meal components are introduced at higher tiers. This reflects the restaurant's explanation that the principal distinction between the price variants is the number of items and/or quantity provided.

Ingredient requirements were calculated deterministically as the planned number of product units multiplied by the assumed ingredient quantity required per product unit. Normal-demand and confirmed-bulk requirements were retained separately before being summed to the final planned ingredient requirement. No allowance was added for cooking loss, spoilage, safety stock, supplier pack sizes or minimum-order quantities; consequently, the ingredient outputs should be interpreted as transparent prototype estimates rather than purchasing instructions.

The mapping covered {100.0 * EXPECTED_MAPPED_PRODUCTS / EXPECTED_ACTIVE_PRODUCTS:.2f}% of the active catalogue by product count, {daily_coverage_pct:.2f}% of planned product units in the daily demonstration forecast and {week_coverage_pct:.2f}% of planned product units in the demonstration week.
"""
    write_text(
        staged_report_dir / REPORT_WORDING_PATH.name,
        report_wording,
    )

    readme = f"""# ND11 Illustrative Ingredient Mapping

This folder contains the final prototype ingredient mapping and ingredient-requirement backend files for the Eden demand-forecasting demonstration.

- Active forecast catalogue: {EXPECTED_ACTIVE_PRODUCTS} products
- Ingredient-mapped prototype subset: {EXPECTED_MAPPED_PRODUCTS} products
- Mapping status: `{MAPPING_STATUS}`
- Daily demonstration date: {daily_date.date()}
- Demonstration week: {week_start.date()} to {week_end.date()}

The ingredient recipes are assumed illustrative values and are not Eden Restaurant's operational recipes.
"""
    write_text(STAGING_ROOT / "README.md", readme)

    manifest = build_manifest(STAGING_ROOT)
    write_csv(staged_control_dir / MANIFEST_PATH.name, manifest)
    manifest_hash = sha256_file(staged_control_dir / MANIFEST_PATH.name)

    checkpoint_payload = {
        "StepID": STEP_ID,
        "Status": STATUS,
        "CompletedLocalTime": NOW_LOCAL.isoformat(),
        "ND11Root": str(ND11_ROOT),
        "InputCheckpointHashes": checkpoint_hashes,
        "ActiveForecastProducts": EXPECTED_ACTIVE_PRODUCTS,
        "IngredientMappedProducts": EXPECTED_MAPPED_PRODUCTS,
        "IngredientMappingStatus": MAPPING_STATUS,
        "IngredientMappingIsActualEdenRecipeData": False,
        "DailyMappingCoveragePlannedUnitsPercentage": daily_coverage_pct,
        "WeekMappingCoveragePlannedUnitsPercentage": week_coverage_pct,
        "UniqueIngredients": int(mapping["IngredientID"].nunique()),
        "RecipeRows": int(len(mapping)),
        "DailyDate": daily_date,
        "WeekStart": week_start,
        "WeekEnd": week_end,
        "ForecastingModelsRefitted": False,
        "ForecastingMethodsReselected": False,
        "MarchTargetVaultOpened": False,
        "ArtifactManifestSHA256": manifest_hash,
        "NextStep": "CREATE_CLEAN_DEMONSTRATION_WORKSPACE_FOR_CODEX_UI",
    }
    staged_checkpoint_path = staged_control_dir / CHECKPOINT_PATH.name
    write_json(staged_checkpoint_path, checkpoint_payload)
    checkpoint_hash = sha256_file(staged_checkpoint_path)
    write_text(
        staged_control_dir / CHECKPOINT_SHA_PATH.name,
        checkpoint_hash + "\n",
    )

    protected_hashes_after = {
        str(path): sha256_file(path) for path in required_inputs
    }
    changed_inputs = [
        path
        for path in protected_hashes_before
        if protected_hashes_before[path] != protected_hashes_after[path]
    ]
    if changed_inputs:
        raise AssertionError(
            "Protected inputs changed during ND11:\n"
            + "\n".join(f"- {path}" for path in changed_inputs)
        )

    os.replace(STAGING_ROOT, ND11_ROOT)

    # Create portable demonstration-backend ZIP after atomic completion.
    shutil.make_archive(
        str(
            ND11_ROOT
            / "07_control"
            / "ND11_demonstration_backend_bundle"
        ),
        "zip",
        root_dir=ND11_ROOT / "03_demonstration_backend",
        base_dir=".",
    )

    TOP_LEVEL_CHECKPOINT_PATH.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(
        ND11_ROOT / "07_control" / CHECKPOINT_PATH.name,
        TOP_LEVEL_CHECKPOINT_PATH,
    )
    shutil.copy2(
        ND11_ROOT / "07_control" / CHECKPOINT_SHA_PATH.name,
        TOP_LEVEL_CHECKPOINT_SHA_PATH,
    )

    handoff = f"""# ND11 Handoff

## Status

- Completed step: `{STEP_ID}`
- Status: `{STATUS}`
- Root: `{ND11_ROOT}`
- Checkpoint SHA-256: `{checkpoint_hash}`

## Demonstration scope

- Forecast/planning catalogue: {EXPECTED_ACTIVE_PRODUCTS} active products
- Illustrative ingredient-mapped subset: {EXPECTED_MAPPED_PRODUCTS} products
- Ingredient mapping status: `{MAPPING_STATUS}`
- Daily planned-unit ingredient coverage: {daily_coverage_pct:.2f}%
- Weekly planned-unit ingredient coverage: {week_coverage_pct:.2f}%

## Important boundary

The recipe mappings are realistic assumed values for prototype demonstration only. They are not Eden Restaurant's actual recipes or procurement specifications.

## Next step

Create a clean, separate demonstration workspace containing the frozen inference engine, corrected active catalogue, planning layer, ND11 backend files and UI documentation, then transfer that workspace to Codex for UI implementation.
"""
    atomic_write_text(HANDOFF_PATH, handoff)
    atomic_write_text(CURRENT_HANDOFF_PATH, handoff)

    append_marked_section(
        WORKFLOW_PATH,
        "## ND11 — Illustrative ingredient mapping",
        f"""## ND11 — Illustrative ingredient mapping

Status: `{STATUS}`

A realistic illustrative ingredient mapping was created for {EXPECTED_MAPPED_PRODUCTS} selected prepared-food products while retaining forecast/planning coverage for all {EXPECTED_ACTIVE_PRODUCTS} active products.
""",
    )
    append_marked_section(
        DECISIONS_PATH,
        "## ND11 decisions",
        f"""## ND11 decisions

- Ingredient mappings are illustrative assumed recipes, not Eden operational recipes.
- Map only the agreed {EXPECTED_MAPPED_PRODUCTS} prototype products.
- Forecast and plan all {EXPECTED_ACTIVE_PRODUCTS} active products.
- Use consistent €5/€7/€9 tier logic for KIMBOCK and DINNER families.
- Do not apply cooking-loss, safety-stock or supplier pack-size adjustments.
""",
    )
    append_marked_section(
        METRICS_AND_RESULTS_PATH,
        "## ND11 ingredient mapping coverage",
        f"""## ND11 ingredient mapping coverage

- Active forecast products: {EXPECTED_ACTIVE_PRODUCTS}
- Ingredient-mapped products: {EXPECTED_MAPPED_PRODUCTS}
- Product-count coverage: {100.0 * EXPECTED_MAPPED_PRODUCTS / EXPECTED_ACTIVE_PRODUCTS:.2f}%
- Daily planned-unit coverage: {daily_coverage_pct:.2f}%
- Weekly planned-unit coverage: {week_coverage_pct:.2f}%
- These are mapping-coverage values, not forecast accuracy metrics.
""",
    )
    append_marked_section(
        AGENTS_PATH,
        "Marker: ND11_AUTHORITATIVE_STATUS",
        f"""## ND11 authoritative status

Marker: ND11_AUTHORITATIVE_STATUS

- Status: `{STATUS}`
- Checkpoint SHA-256: `{checkpoint_hash}`
- Ingredient-mapped subset: {EXPECTED_MAPPED_PRODUCTS}
- Active forecast catalogue: {EXPECTED_ACTIVE_PRODUCTS}
- Next step: clean demonstration workspace for Codex/UI.
""",
    )

    LOG_PATH.parent.mkdir(parents=True, exist_ok=True)
    with LOG_PATH.open("a", encoding="utf-8") as handle:
        handle.write(
            f"{NOW_LOCAL.isoformat()} | {STATUS} | "
            f"checkpoint={checkpoint_hash} | root={ND11_ROOT}\n"
        )

except Exception:
    if STAGING_ROOT.exists():
        shutil.rmtree(STAGING_ROOT, ignore_errors=True)
    raise


# =============================================================================
# FINAL CONSOLE OUTPUT
# =============================================================================

print("=" * 118)
print("EDEN NORMAL-DEMAND MODEL V2 — ND11 COMPLETE")
print("=" * 118)
print(f"Status: {STATUS}")
print(f"Local time: {NOW_LOCAL.isoformat()}")
print(f"ND11 root: {ND11_ROOT}")
print()
print("INPUT VERIFICATION")
print(f"ND09A checkpoint SHA-256: {checkpoint_hashes['ND09A']}")
print(f"ND10A checkpoint SHA-256: {checkpoint_hashes['ND10A']}")
print(f"ND10RA checkpoint SHA-256: {checkpoint_hashes['ND10RA']}")
print(f"Active forecast products: {len(active_ids)}")
print("March target vault opened: False")
print("Previous inputs modified: False")
print()
print("ILLUSTRATIVE INGREDIENT MAPPING")
print(f"Mapped products: {len(mapped_ids)}")
print(f"Recipe mapping rows: {len(mapping):,}")
print(f"Unique ingredients: {mapping['IngredientID'].nunique():,}")
print(f"Mapping status: {MAPPING_STATUS}")
print("Actual Eden recipe data used: False")
print("KIMBOCK/DINNER €5/€7/€9 tier rule passed: True")
print("Ingredient unit consistency passed: True")
print()
print("MAPPING COVERAGE")
print(
    "Product-count coverage: "
    f"{100.0 * EXPECTED_MAPPED_PRODUCTS / EXPECTED_ACTIVE_PRODUCTS:.2f}%"
)
print(f"Daily planned-unit coverage: {daily_coverage_pct:.2f}%")
print(f"Week planned-unit coverage: {week_coverage_pct:.2f}%")
print()
print("DEMONSTRATION BACKEND")
print(f"Daily date: {daily_date.date()}")
print(f"Week: {week_start.date()} to {week_end.date()}")
print(f"All active products remain forecast/planned: {EXPECTED_ACTIVE_PRODUCTS}")
print(f"Products with ingredient breakdown: {EXPECTED_MAPPED_PRODUCTS}")
print(
    "Ingredient formula: PlannedIngredientRequirement = "
    "PlannedQuantity × QuantityPerProductUnit"
)
print()
print("OUTPUTS")
print(f"- Ingredient mapping: {PRODUCT_MAPPING_PATH}")
print(f"- Mapped product catalogue: {MAPPED_PRODUCT_CATALOGUE_PATH}")
print(f"- Ingredient master: {INGREDIENT_MASTER_PATH}")
print(f"- Daily ingredient totals: {DAILY_INGREDIENT_TOTALS_PATH}")
print(f"- Week daily ingredient totals: {WEEK_DAILY_INGREDIENT_TOTALS_PATH}")
print(f"- Week ingredient totals: {WEEK_INGREDIENT_TOTALS_PATH}")
print(f"- Mapping coverage audit: {MAPPING_COVERAGE_PATH}")
print(f"- Demonstration backend: {DEMO_BACKEND_DIR}")
print(f"- Demo backend contract: {BACKEND_CONTRACT_PATH}")
print(f"- Report summary: {REPORT_SUMMARY_PATH}")
print(f"- Report wording: {REPORT_WORDING_PATH}")
print(f"- Demo backend ZIP: {ZIP_PATH}")
print(f"- Checkpoint: {TOP_LEVEL_CHECKPOINT_PATH}")
print(f"- Checkpoint SHA-256: {checkpoint_hash}")
print(f"- Handoff: {HANDOFF_PATH}")
print()
print("SAFETY")
print("- Forecasting models fitted/refitted: False")
print("- Forecasting methods reselected: False")
print("- March target vault opened: False")
print("- Ingredient mappings presented as actual Eden recipes: False")
print("- Cooking-loss/safety-stock factor applied: False")
print("- Previous inputs modified: False")
print()
print("NEXT STEP")
print(
    "Create the clean demonstration workspace, then transfer it to Codex "
    "for UI implementation."
)
print("=" * 118)

EDEN NORMAL-DEMAND MODEL V2 — ND11 COMPLETE
Status: ND11_ILLUSTRATIVE_INGREDIENT_MAPPING_AND_DEMO_BACKEND_CREATED_READY_FOR_DEMO_PACKAGE
Local time: 2026-08-10T13:01:19.811418+01:00
ND11 root: /Users/ryansmac/Desktop/Meng Project/eden_datasets/eden_normal_demand_model_v2/04_planning/ND11_illustrative_ingredient_mapping

INPUT VERIFICATION
ND09A checkpoint SHA-256: 54767758d9052cb1571043d2f39cc167bc7491b300afbd3605a5416b7aa95c10
ND10A checkpoint SHA-256: 6bcc97c5fe76667c2bdcdde7d0e81b4bd1534c3a601f39fd7acd795e6c87dbff
ND10RA checkpoint SHA-256: fe24555b1562cce7b4300f62051d427492f4e9b3259e662e21f400bac6972c75
Active forecast products: 88
March target vault opened: False
Previous inputs modified: False

ILLUSTRATIVE INGREDIENT MAPPING
Mapped products: 14
Recipe mapping rows: 77
Unique ingredients: 37
Mapping status: ILLUSTRATIVE_ASSUMED_RECIPE
Actual Eden recipe data used: False
KIMBOCK/DINNER €5/€7/€9 tier rule passed: True
Ingredient unit consistency passed: True

MAPPING COVERAGE
Produ

In [21]:
# =============================================================================
# EDEN NORMAL-DEMAND MODEL V2
# ND12 — CLEAN STANDALONE DEMONSTRATION WORKSPACE FOR CODEX/UI
#
# Run this as one complete Jupyter cell after successful ND11.
#
# This step creates a new, standalone application workspace at:
#   /Users/ryansmac/Desktop/Meng Project/Eden_Demand_Forecasting_Demo
#
# The workspace contains only the frozen runtime system needed by the UI:
# - frozen ND09 preprocessors/models;
# - precomputed runtime history state and hierarchy fallback statistics;
# - corrected 88-product active catalogue;
# - stable backend engines;
# - 14-product illustrative ingredient mapping;
# - backend/UI contracts;
# - example outputs and tests;
# - Codex instructions and AGENTS.md.
#
# It deliberately DOES NOT copy:
# - raw Eden POS/transaction data;
# - March actual target vault;
# - model-development folds/candidate predictions;
# - tuning trials;
# - the full ND01-ND11 research workspace.
#
# No forecasting model is fitted, refitted, tuned or reselected.
# =============================================================================

import hashlib
import importlib
import json
import os
import platform
import shutil
import subprocess
import sys
import textwrap
import uuid
from datetime import datetime, timezone
from pathlib import Path
from zoneinfo import ZoneInfo

import joblib
import numpy as np
import pandas as pd


# =============================================================================
# USER CONFIGURATION
# =============================================================================

DEMO_WORKSPACE_ROOT = Path(
    "/Users/ryansmac/Desktop/Meng Project/Eden_Demand_Forecasting_Demo"
)
ALLOW_OVERWRITE = False


# =============================================================================
# SOURCE PROJECT CONFIGURATION
# =============================================================================

PROJECT_ROOT = Path("/Users/ryansmac/Desktop/Meng Project")
EDEN_ROOT = PROJECT_ROOT / "eden_datasets"
MODEL_ROOT = EDEN_ROOT / "eden_normal_demand_model_v2"

ND03_ROOT = MODEL_ROOT / "02_feature_engineering" / "ND03_normal_demand_features"
ND09_ROOT = (
    MODEL_ROOT
    / "03_models"
    / "03_inference"
    / "ND09_arbitrary_date_inference_engine"
)
ND09A_ROOT = (
    MODEL_ROOT
    / "03_models"
    / "03_inference"
    / "ND09A_active_menu_catalogue_correction"
)
ND10A_ROOT = (
    MODEL_ROOT
    / "04_planning"
    / "ND10A_corrected_active_catalogue_planning"
)
ND10RA_ROOT = (
    MODEL_ROOT
    / "05_reporting"
    / "ND10RA_corrected_demo_reporting_refresh"
)
ND11_ROOT = (
    MODEL_ROOT
    / "04_planning"
    / "ND11_illustrative_ingredient_mapping"
)

CHECKPOINT_PATHS = {
    "ND09A": MODEL_ROOT / "08_checkpoints" / "ND09A_checkpoint.json",
    "ND10A": MODEL_ROOT / "08_checkpoints" / "ND10A_checkpoint.json",
    "ND10RA": MODEL_ROOT / "08_checkpoints" / "ND10RA_checkpoint.json",
    "ND11": MODEL_ROOT / "08_checkpoints" / "ND11_checkpoint.json",
}
EXPECTED_CHECKPOINT_HASHES = {
    "ND09A": "54767758d9052cb1571043d2f39cc167bc7491b300afbd3605a5416b7aa95c10",
    "ND10A": "6bcc97c5fe76667c2bdcdde7d0e81b4bd1534c3a601f39fd7acd795e6c87dbff",
    "ND10RA": "fe24555b1562cce7b4300f62051d427492f4e9b3259e662e21f400bac6972c75",
    "ND11": "58d497e8eb0e82f18f648537d9d754e438731d4b2ee4a2bc256ec18adba50879",
}

ALL_ROUTES_PATH = (
    ND03_ROOT
    / "01_model_ready_datasets"
    / "ND03_pre_march_all_routes_development_dataset.csv"
)

ND09_MODEL_DIR = ND09_ROOT / "01_model_artifacts"
MODEL_ARTIFACTS = [
    "ND09_core_preprocessor.joblib",
    "ND09_product_preprocessor.joblib",
    "ND09_catboost_core53.cbm",
    "ND09_xgboost_core53.json",
    "ND09_xgboost_product_aware.json",
    "ND09_model_artifact_metadata.json",
]

ACTIVE_CATALOGUE_PATH = (
    ND09A_ROOT
    / "02_contracts"
    / "ND09A_active_product_catalogue.csv"
)
ND09A_CONTRACT_PATH = (
    ND09A_ROOT
    / "02_contracts"
    / "ND09A_corrected_inference_contract.json"
)
ND09A_SCOPE_LOCK_PATH = (
    MODEL_ROOT
    / "08_checkpoints"
    / "ND09A_active_catalogue_scope_lock.json"
)

ND11_MAPPING_PATH = (
    ND11_ROOT
    / "01_mapping"
    / "ND11_illustrative_product_ingredient_mapping.csv"
)
ND11_INGREDIENT_MASTER_PATH = (
    ND11_ROOT
    / "01_mapping"
    / "ND11_ingredient_master.csv"
)
ND11_BACKEND_CONTRACT_PATH = (
    ND11_ROOT
    / "03_demonstration_backend"
    / "demo_backend_contract.json"
)
ND11_DISCLAIMER_PATH = (
    ND11_ROOT
    / "03_demonstration_backend"
    / "INGREDIENT_MAPPING_DISCLAIMER.md"
)

ND10A_EXAMPLE_FILES = {
    "daily_product_plan.csv": (
        ND10A_ROOT
        / "02_planned_outputs"
        / "ND10A_daily_product_planning.csv"
    ),
    "daily_restaurant_plan.csv": (
        ND10A_ROOT
        / "02_planned_outputs"
        / "ND10A_daily_restaurant_planning.csv"
    ),
    "week_daily_product_plan.csv": (
        ND10A_ROOT
        / "02_planned_outputs"
        / "ND10A_week_daily_product_planning.csv"
    ),
    "week_product_totals.csv": (
        ND10A_ROOT
        / "02_planned_outputs"
        / "ND10A_week_product_totals_planning.csv"
    ),
    "week_restaurant_daily.csv": (
        ND10A_ROOT
        / "02_planned_outputs"
        / "ND10A_week_restaurant_daily_planning.csv"
    ),
    "week_restaurant_total.csv": (
        ND10A_ROOT
        / "02_planned_outputs"
        / "ND10A_week_restaurant_total_planning.csv"
    ),
}

ND11_EXAMPLE_FILES = {
    "daily_ingredient_requirements.csv": (
        ND11_ROOT
        / "02_ingredient_requirements"
        / "ND11_daily_ingredient_totals.csv"
    ),
    "week_daily_ingredient_requirements.csv": (
        ND11_ROOT
        / "02_ingredient_requirements"
        / "ND11_week_daily_ingredient_totals.csv"
    ),
    "week_ingredient_requirements.csv": (
        ND11_ROOT
        / "02_ingredient_requirements"
        / "ND11_week_ingredient_totals.csv"
    ),
}

EXPECTED_ACTIVE_PRODUCTS = 88
EXPECTED_MAPPED_PRODUCTS = 14
EXPECTED_DAILY_TOTAL = 727.756809
EXPECTED_WEEK_TOTAL = 3341.504320

DATE_COLUMN = "Date"
PRODUCT_ID_COLUMN = "CanonicalProductID"
PRODUCT_NAME_COLUMN = "CanonicalProductName"
TARGET_COLUMN = "NormalDemand"
FAMILY_COLUMN = "TierProductFamily"
DAY_OF_WEEK_COLUMN = "DayOfWeekNumber"
SEQUENCE_COLUMN = "OperatingDaySequence"

STATUS = "ND12_CLEAN_DEMONSTRATION_WORKSPACE_CREATED_READY_FOR_CODEX_UI"
NOW_UTC = datetime.now(timezone.utc)
NOW_LOCAL = NOW_UTC.astimezone(ZoneInfo("Europe/Dublin"))


# =============================================================================
# EMBEDDED RUNTIME SOURCE
# =============================================================================

FORECAST_ENGINE_SOURCE = '\nfrom __future__ import annotations\n\nfrom pathlib import Path\nfrom typing import Iterable\n\nimport joblib\nimport numpy as np\nimport pandas as pd\nfrom catboost import CatBoostRegressor\nfrom xgboost import XGBRegressor\n\n\nDATE_COLUMN = "Date"\nPRODUCT_ID_COLUMN = "CanonicalProductID"\nPRODUCT_NAME_COLUMN = "CanonicalProductName"\nTARGET_COLUMN = "NormalDemand"\nFAMILY_COLUMN = "TierProductFamily"\nDAY_OF_WEEK_COLUMN = "DayOfWeekNumber"\nSEQUENCE_COLUMN = "OperatingDaySequence"\n\nLAG_OPERATING_DAYS = [1, 2, 3, 5, 10, 20]\nROLLING_WINDOWS = [3, 5, 10, 20]\nZERO_POSITIVE_WINDOWS = [5, 10, 20]\nMINIMUM_MAIN_HISTORY = 20\nPRIMARY_SCOPE_PERCENTAGE = 95.0\n\nHISTORICAL_DEMAND_PREDICTORS = (\n    [f"NormalDemandLag_{lag}" for lag in LAG_OPERATING_DAYS]\n    + [\n        feature\n        for window in ROLLING_WINDOWS\n        for feature in [\n            f"PastNormalDemandRollingMean_{window}",\n            f"PastNormalDemandRollingMedian_{window}",\n            f"PastNormalDemandRollingStd_{window}",\n            f"PastNormalDemandRollingSum_{window}",\n        ]\n    ]\n    + [f"PastZeroNormalDemandRate_{window}" for window in ZERO_POSITIVE_WINDOWS]\n    + [\n        f"PastPositiveNormalDemandCount_{window}"\n        for window in ZERO_POSITIVE_WINDOWS\n    ]\n    + [\n        "OperatingDaysSincePreviousPositiveNormalDemand",\n        "ExpandingPastMeanNormalDemand",\n        "ExpandingPastPositiveNormalDemandRate",\n    ]\n)\n\nBASE_NUMERIC_PREDICTORS = [\n    "ProductAgeOperatingDays",\n    "SourcePLUCount",\n    "IsMultiPLUCanonicalProduct",\n    "OperatingDaySequence",\n    "Year",\n    "Month",\n    "Quarter",\n    "DayOfWeekNumber",\n    "ISOYear",\n    "ISOWeek",\n    "DayOfYear",\n    "IsWeekend",\n    "DaysSincePreviousOperatingDate",\n    "IsConsecutiveCalendarDay",\n]\n\nBASE_CATEGORICAL_PREDICTORS = [\n    "SourceGroupCodes",\n    "SourceGroupNames",\n    "BeverageSeries",\n    "BeverageType",\n    "SupplierLabelsObserved",\n    "TierProductFamily",\n    "NominalPriceTier",\n    "MenuGeneration",\n]\n\nPRODUCT_METADATA_COLUMNS = [\n    PRODUCT_ID_COLUMN,\n    PRODUCT_NAME_COLUMN,\n    "ProductFirstObservedDate",\n    "SourcePLUCount",\n    "IsMultiPLUCanonicalProduct",\n    *BASE_CATEGORICAL_PREDICTORS,\n]\n\nBASE_COMPONENTS = [\n    "ROLLING_MEAN_5",\n    "CATBOOST_CORE53",\n    "XGBOOST_CORE53",\n    "XGBOOST_PRODUCT_AWARE",\n]\n\n\ndef _normalize_family(values: pd.Series) -> pd.Series:\n    return values.astype("string").fillna("__MISSING_FAMILY__").astype(str)\n\n\ndef _normalize_dates(values: Iterable[str | pd.Timestamp] | None) -> set[pd.Timestamp]:\n    if values is None:\n        return set()\n    return {pd.Timestamp(value).normalize() for value in values}\n\n\ndef _is_operating_date(\n    date: pd.Timestamp,\n    closed_dates: set[pd.Timestamp],\n    extra_dates: set[pd.Timestamp],\n) -> bool:\n    date = pd.Timestamp(date).normalize()\n    if date in closed_dates:\n        return False\n    return date.weekday() < 5 or date in extra_dates\n\n\ndef _generate_future_calendar(\n    cutoff_date: pd.Timestamp,\n    cutoff_sequence: int,\n    end_date: pd.Timestamp,\n    closed_dates: set[pd.Timestamp],\n    extra_dates: set[pd.Timestamp],\n) -> pd.DataFrame:\n    cutoff_date = pd.Timestamp(cutoff_date).normalize()\n    end_date = pd.Timestamp(end_date).normalize()\n    if end_date <= cutoff_date:\n        raise ValueError(\n            f"Requested date must be after the frozen cutoff {cutoff_date.date()}."\n        )\n\n    records = []\n    sequence = int(cutoff_sequence)\n    previous_operating_date = cutoff_date\n\n    for date in pd.date_range(\n        cutoff_date + pd.Timedelta(days=1),\n        end_date,\n        freq="D",\n    ):\n        if not _is_operating_date(date, closed_dates, extra_dates):\n            continue\n        sequence += 1\n        iso = date.isocalendar()\n        gap = int((date - previous_operating_date).days)\n        records.append(\n            {\n                DATE_COLUMN: date,\n                SEQUENCE_COLUMN: sequence,\n                "Year": int(date.year),\n                "Month": int(date.month),\n                "Quarter": int(date.quarter),\n                "DayOfWeekNumber": int(date.weekday()),\n                "ISOYear": int(iso.year),\n                "ISOWeek": int(iso.week),\n                "DayOfYear": int(date.dayofyear),\n                "IsWeekend": bool(date.weekday() >= 5),\n                "DaysSincePreviousOperatingDate": gap,\n                "IsConsecutiveCalendarDay": bool(gap == 1),\n                "CalendarPolicy": (\n                    "EXCEPTIONAL_OPERATING_DATE"\n                    if date.weekday() >= 5\n                    else "STANDARD_MONDAY_TO_FRIDAY"\n                ),\n            }\n        )\n        previous_operating_date = date\n\n    calendar = pd.DataFrame(records)\n    if calendar.empty:\n        raise ValueError("No operating dates exist between the cutoff and request.")\n    return calendar\n\n\ndef _historical_features_from_history(\n    demand_history: np.ndarray,\n    sequence_history: np.ndarray,\n    current_sequence: float,\n) -> dict[str, float]:\n    demand_history = np.asarray(demand_history, dtype=float)\n    sequence_history = np.asarray(sequence_history, dtype=float)\n    result: dict[str, float] = {}\n\n    for lag in LAG_OPERATING_DAYS:\n        result[f"NormalDemandLag_{lag}"] = (\n            float(demand_history[-lag])\n            if len(demand_history) >= lag\n            else float("nan")\n        )\n\n    for window in ROLLING_WINDOWS:\n        values = demand_history[-window:]\n        if len(values) == 0:\n            mean_value = median_value = std_value = sum_value = float("nan")\n        else:\n            mean_value = float(np.mean(values))\n            median_value = float(np.median(values))\n            std_value = float(np.std(values, ddof=0))\n            sum_value = float(np.sum(values))\n        result[f"PastNormalDemandRollingMean_{window}"] = mean_value\n        result[f"PastNormalDemandRollingMedian_{window}"] = median_value\n        result[f"PastNormalDemandRollingStd_{window}"] = std_value\n        result[f"PastNormalDemandRollingSum_{window}"] = sum_value\n\n    for window in ZERO_POSITIVE_WINDOWS:\n        values = demand_history[-window:]\n        if len(values) == 0:\n            zero_rate = positive_count = float("nan")\n        else:\n            zero_rate = float(np.mean(values == 0))\n            positive_count = float(np.sum(values > 0))\n        result[f"PastZeroNormalDemandRate_{window}"] = zero_rate\n        result[f"PastPositiveNormalDemandCount_{window}"] = positive_count\n\n    positive_indices = np.flatnonzero(demand_history > 0)\n    if len(positive_indices) == 0:\n        days_since_positive = float("nan")\n    else:\n        last_positive_sequence = float(sequence_history[int(positive_indices[-1])])\n        days_since_positive = float(current_sequence - last_positive_sequence)\n    result["OperatingDaysSincePreviousPositiveNormalDemand"] = days_since_positive\n\n    if len(demand_history) == 0:\n        expanding_mean = float("nan")\n        expanding_positive_rate = float("nan")\n    else:\n        expanding_mean = float(np.mean(demand_history))\n        expanding_positive_rate = float(np.mean(demand_history > 0))\n    result["ExpandingPastMeanNormalDemand"] = expanding_mean\n    result["ExpandingPastPositiveNormalDemandRate"] = expanding_positive_rate\n    return result\n\n\ndef _copy_history_state(base_state: dict[str, dict[str, list]]) -> dict[str, dict[str, list]]:\n    return {\n        str(product_id): {\n            "demand": list(values["demand"]),\n            "sequence": list(values["sequence"]),\n        }\n        for product_id, values in base_state.items()\n    }\n\n\ndef _append_values_to_state(\n    state: dict[str, dict[str, list]],\n    rows: pd.DataFrame,\n    values: np.ndarray,\n) -> None:\n    for product_id, sequence, value in zip(\n        rows[PRODUCT_ID_COLUMN].astype(str),\n        rows[SEQUENCE_COLUMN].astype(float),\n        np.asarray(values, dtype=float),\n    ):\n        if product_id not in state:\n            state[product_id] = {"demand": [], "sequence": []}\n        if state[product_id]["sequence"]:\n            if sequence <= state[product_id]["sequence"][-1]:\n                raise AssertionError(\n                    f"Non-increasing operating sequence for {product_id}."\n                )\n        state[product_id]["demand"].append(float(value))\n        state[product_id]["sequence"].append(float(sequence))\n\n\ndef _build_recursive_feature_frame(\n    score_rows: pd.DataFrame,\n    state: dict[str, dict[str, list]],\n) -> pd.DataFrame:\n    frame = score_rows.copy().reset_index(drop=True)\n    feature_records = []\n    prior_records = []\n\n    for row in frame.itertuples(index=False):\n        product_id = str(getattr(row, PRODUCT_ID_COLUMN))\n        current_sequence = float(getattr(row, SEQUENCE_COLUMN))\n        product_state = state.get(product_id, {"demand": [], "sequence": []})\n        demand_history = np.asarray(product_state["demand"], dtype=float)\n        sequence_history = np.asarray(product_state["sequence"], dtype=float)\n\n        feature_records.append(\n            _historical_features_from_history(\n                demand_history,\n                sequence_history,\n                current_sequence,\n            )\n        )\n\n        prior_count = int(len(demand_history))\n        prior_cumulative = float(demand_history.sum())\n        prior_positive = int(np.sum(demand_history > 0))\n        prior_records.append(\n            {\n                "PriorOperatingDayCount": prior_count,\n                "PriorCumulativeNormalDemand": prior_cumulative,\n                "PriorPositiveNormalDemandDays": prior_positive,\n                "PriorZeroNormalDemandDays": prior_count - prior_positive,\n                "HasSufficientHistory20": prior_count >= MINIMUM_MAIN_HISTORY,\n                "ColdStartFlag": prior_count < MINIMUM_MAIN_HISTORY,\n                "ZeroPriorNormalDemandFlag": prior_cumulative <= 0,\n            }\n        )\n\n    feature_values = pd.DataFrame(feature_records)\n    prior_values = pd.DataFrame(prior_records)\n\n    for column in HISTORICAL_DEMAND_PREDICTORS:\n        frame[column] = feature_values[column].to_numpy()\n    for column in prior_values.columns:\n        frame[column] = prior_values[column].to_numpy()\n\n    ranked = frame.sort_values(\n        [\n            "PriorCumulativeNormalDemand",\n            "PriorPositiveNormalDemandDays",\n            PRODUCT_ID_COLUMN,\n        ],\n        ascending=[False, False, True],\n        kind="mergesort",\n    ).copy()\n\n    universe_total = float(ranked["PriorCumulativeNormalDemand"].sum())\n    ranked["PriorDemandRank"] = np.arange(1, len(ranked) + 1, dtype=int)\n    ranked["PriorDemandSharePercentage"] = 0.0\n    ranked["PriorCumulativeDemandShareBeforePercentage"] = 0.0\n    ranked["PriorCumulativeDemandSharePercentage"] = 0.0\n    ranked["InPrior95DemandScope"] = False\n\n    if universe_total > 0:\n        share = 100.0 * ranked["PriorCumulativeNormalDemand"] / universe_total\n        cumulative = share.cumsum()\n        cumulative_before = cumulative - share\n        ranked["PriorDemandSharePercentage"] = share\n        ranked["PriorCumulativeDemandShareBeforePercentage"] = cumulative_before\n        ranked["PriorCumulativeDemandSharePercentage"] = cumulative\n        ranked["InPrior95DemandScope"] = (\n            (ranked["PriorCumulativeNormalDemand"] > 0)\n            & (cumulative_before < PRIMARY_SCOPE_PERCENTAGE)\n        )\n\n    ranked["EligibleForMainModel"] = (\n        ranked["HasSufficientHistory20"]\n        & ranked["InPrior95DemandScope"]\n        & ~ranked["ZeroPriorNormalDemandFlag"]\n    )\n    ranked["RecursiveForecastRoute"] = np.select(\n        [\n            ranked["ZeroPriorNormalDemandFlag"],\n            ranked["ColdStartFlag"],\n            ranked["EligibleForMainModel"],\n        ],\n        [\n            "ZERO_HISTORY_FALLBACK",\n            "COLD_START_FALLBACK",\n            "MAIN_MODEL",\n        ],\n        default="LOW_DEMAND_FALLBACK",\n    )\n\n    ranking_columns = [\n        PRODUCT_ID_COLUMN,\n        "PriorDemandRank",\n        "PriorDemandSharePercentage",\n        "PriorCumulativeDemandShareBeforePercentage",\n        "PriorCumulativeDemandSharePercentage",\n        "InPrior95DemandScope",\n        "EligibleForMainModel",\n        "RecursiveForecastRoute",\n    ]\n    frame = frame.drop(\n        columns=[c for c in ranking_columns[1:] if c in frame.columns],\n        errors="ignore",\n    ).merge(\n        ranked[ranking_columns],\n        on=PRODUCT_ID_COLUMN,\n        how="left",\n        validate="one_to_one",\n    )\n    return frame\n\n\ndef _hierarchy_predictions(\n    score_frame: pd.DataFrame,\n    statistics: dict,\n) -> tuple[np.ndarray, np.ndarray]:\n    score = score_frame.copy()\n    product_ids = score[PRODUCT_ID_COLUMN].astype(str)\n    families = _normalize_family(score[FAMILY_COLUMN])\n    weekdays = pd.to_numeric(score[DAY_OF_WEEK_COLUMN], errors="raise").astype(int)\n\n    product_weekday_values = np.asarray(\n        [\n            statistics["product_weekday"].get((product_id, weekday), np.nan)\n            for product_id, weekday in zip(product_ids, weekdays)\n        ],\n        dtype=float,\n    )\n    product_values = product_ids.map(statistics["product_mean"]).to_numpy(dtype=float)\n    family_weekday_values = np.asarray(\n        [\n            statistics["family_weekday"].get((family, weekday), np.nan)\n            for family, weekday in zip(families, weekdays)\n        ],\n        dtype=float,\n    )\n    family_values = families.map(statistics["family_mean"]).to_numpy(dtype=float)\n    global_weekday_values = weekdays.map(\n        statistics["global_weekday"]\n    ).to_numpy(dtype=float)\n    global_values = np.full(len(score), statistics["global_mean"], dtype=float)\n    zero_values = np.zeros(len(score), dtype=float)\n\n    def coalesce(*arrays) -> np.ndarray:\n        output = np.full(len(score), np.nan, dtype=float)\n        for values in arrays:\n            candidate = np.asarray(values, dtype=float)\n            missing = ~np.isfinite(output)\n            output[missing] = candidate[missing]\n        return np.clip(np.nan_to_num(output, nan=0.0), 0.0, None)\n\n    product_hierarchy = coalesce(\n        product_weekday_values,\n        product_values,\n        family_weekday_values,\n        family_values,\n        global_weekday_values,\n        global_values,\n        zero_values,\n    )\n    family_hierarchy = coalesce(\n        family_weekday_values,\n        family_values,\n        global_weekday_values,\n        global_values,\n        zero_values,\n    )\n    return product_hierarchy, family_hierarchy\n\n\ndef _fallback_predictions(\n    feature_frame: pd.DataFrame,\n    hierarchy_statistics: dict,\n) -> np.ndarray:\n    product_hierarchy, family_hierarchy = _hierarchy_predictions(\n        feature_frame,\n        hierarchy_statistics,\n    )\n\n    def values(column: str) -> np.ndarray:\n        return pd.to_numeric(\n            feature_frame[column],\n            errors="coerce",\n        ).to_numpy(dtype=float)\n\n    def coalesce(*arrays) -> np.ndarray:\n        output = np.full(len(feature_frame), np.nan, dtype=float)\n        for array in arrays:\n            candidate = np.asarray(array, dtype=float)\n            missing = ~np.isfinite(output)\n            output[missing] = candidate[missing]\n        return np.clip(np.nan_to_num(output, nan=0.0), 0.0, None)\n\n    recent_median5 = coalesce(\n        values("PastNormalDemandRollingMedian_5"),\n        values("PastNormalDemandRollingMedian_3"),\n        values("NormalDemandLag_1"),\n        values("ExpandingPastMeanNormalDemand"),\n        family_hierarchy,\n        np.zeros(len(feature_frame)),\n    )\n    expanding_mean = coalesce(\n        values("ExpandingPastMeanNormalDemand"),\n        product_hierarchy,\n        family_hierarchy,\n        np.zeros(len(feature_frame)),\n    )\n\n    routes = feature_frame["RecursiveForecastRoute"].astype(str).to_numpy()\n    prediction = np.full(len(feature_frame), np.nan, dtype=float)\n    median_mask = np.isin(\n        routes,\n        ["COLD_START_FALLBACK", "LOW_DEMAND_FALLBACK"],\n    )\n    zero_mask = routes == "ZERO_HISTORY_FALLBACK"\n    prediction[median_mask] = recent_median5[median_mask]\n    prediction[zero_mask] = expanding_mean[zero_mask]\n    return prediction\n\n\ndef _prepare_source_frame(\n    frame: pd.DataFrame,\n    numeric_predictors: list[str],\n    categorical_predictors: list[str],\n    include_product_id: bool,\n) -> pd.DataFrame:\n    columns = (\n        numeric_predictors\n        + categorical_predictors\n        + ([PRODUCT_ID_COLUMN] if include_product_id else [])\n    )\n    prepared = frame[columns].copy()\n    for column in numeric_predictors:\n        prepared[column] = pd.to_numeric(prepared[column], errors="coerce")\n    for column in categorical_predictors:\n        prepared[column] = (\n            prepared[column]\n            .astype("string")\n            .fillna("__MISSING__")\n            .astype(str)\n        )\n    if include_product_id:\n        prepared[PRODUCT_ID_COLUMN] = (\n            prepared[PRODUCT_ID_COLUMN]\n            .astype("string")\n            .fillna("__MISSING_PRODUCT__")\n            .astype(str)\n        )\n    return prepared\n\n\ndef _predict_base_components(\n    main_features: pd.DataFrame,\n    fitted_models: dict,\n    numeric_predictors: list[str],\n    categorical_predictors: list[str],\n) -> dict[str, np.ndarray]:\n    components = {\n        "ROLLING_MEAN_5": np.clip(\n            pd.to_numeric(\n                main_features["PastNormalDemandRollingMean_5"],\n                errors="coerce",\n            ).to_numpy(dtype=float),\n            0.0,\n            None,\n        )\n    }\n\n    transformed_cache = {}\n    for component, fitted in fitted_models.items():\n        cache_key = "PRODUCT" if fitted["include_product_id"] else "CORE"\n        if cache_key not in transformed_cache:\n            transformed_cache[cache_key] = fitted["preprocessor"].transform(\n                _prepare_source_frame(\n                    main_features,\n                    numeric_predictors,\n                    categorical_predictors,\n                    fitted["include_product_id"],\n                )\n            )\n        prediction = np.asarray(\n            fitted["model"].predict(transformed_cache[cache_key]),\n            dtype=float,\n        )\n        components[component] = np.clip(prediction, 0.0, None)\n\n    if set(components) != set(BASE_COMPONENTS):\n        raise AssertionError("The packaged base-component set is incomplete.")\n    return components\n\n\ndef _predict_routed_system(\n    feature_frame: pd.DataFrame,\n    operational_method: str,\n    median_components: list[str],\n    fitted_models: dict,\n    numeric_predictors: list[str],\n    categorical_predictors: list[str],\n    hierarchy_statistics: dict,\n) -> pd.DataFrame:\n    output = feature_frame.copy().reset_index(drop=True)\n    predictions = _fallback_predictions(output, hierarchy_statistics)\n    method_used = output["RecursiveForecastRoute"].map(\n        {\n            "COLD_START_FALLBACK": "RECENT_MEDIAN5_WITH_BACKOFF",\n            "LOW_DEMAND_FALLBACK": "RECENT_MEDIAN5_WITH_BACKOFF",\n            "ZERO_HISTORY_FALLBACK": "EXPANDING_MEAN_WITH_BACKOFF",\n        }\n    ).astype("string")\n\n    for component in BASE_COMPONENTS:\n        output[f"BasePrediction_{component}"] = np.nan\n\n    main_mask = output["RecursiveForecastRoute"].astype(str) == "MAIN_MODEL"\n    if main_mask.any():\n        main_features = output.loc[main_mask].copy()\n        components = _predict_base_components(\n            main_features,\n            fitted_models,\n            numeric_predictors,\n            categorical_predictors,\n        )\n        for component, values in components.items():\n            output.loc[\n                main_mask,\n                f"BasePrediction_{component}",\n            ] = values\n\n        if operational_method == "ROLLING_MEAN_5":\n            main_prediction = components["ROLLING_MEAN_5"]\n        elif operational_method == "NESTED_MEDIAN_ENSEMBLE":\n            unknown = sorted(set(median_components) - set(BASE_COMPONENTS))\n            if unknown:\n                raise AssertionError(\n                    f"Unknown median components: {unknown}"\n                )\n            matrix = np.column_stack(\n                [components[component] for component in median_components]\n            )\n            main_prediction = np.median(matrix, axis=1)\n        else:\n            raise ValueError(\n                f"Unsupported operational method: {operational_method}"\n            )\n\n        predictions[main_mask.to_numpy()] = np.clip(\n            main_prediction,\n            0.0,\n            None,\n        )\n        method_used.loc[main_mask] = operational_method\n\n    if not np.isfinite(predictions).all():\n        raise AssertionError("Forecast contains non-finite predictions.")\n    if (predictions < 0).any():\n        raise AssertionError("Forecast contains negative predictions.")\n\n    output["ForecastRoute"] = output["RecursiveForecastRoute"].astype(str)\n    output["PredictedNormalDemand"] = predictions\n    output["MethodUsed"] = method_used.astype(str)\n    return output\n\n\ndef _build_future_base_rows(\n    active_catalogue: pd.DataFrame,\n    calendar_row: pd.Series,\n) -> pd.DataFrame:\n    frame = active_catalogue.loc[\n        active_catalogue["IncludeInForecast"].astype(bool)\n    ].copy().reset_index(drop=True)\n    current_sequence = int(calendar_row[SEQUENCE_COLUMN])\n\n    for column in [\n        DATE_COLUMN,\n        SEQUENCE_COLUMN,\n        "Year",\n        "Month",\n        "Quarter",\n        "DayOfWeekNumber",\n        "ISOYear",\n        "ISOWeek",\n        "DayOfYear",\n        "IsWeekend",\n        "DaysSincePreviousOperatingDate",\n        "IsConsecutiveCalendarDay",\n        "CalendarPolicy",\n    ]:\n        frame[column] = calendar_row[column]\n\n    frame["ProductAgeOperatingDays"] = (\n        current_sequence\n        - pd.to_numeric(\n            frame["FirstObservedOperatingDaySequence"],\n            errors="raise",\n        ).astype(int)\n    ).clip(lower=0)\n    frame[TARGET_COLUMN] = np.nan\n    return frame\n\n\nclass ForecastEngine:\n    """\n    Frozen runtime inference engine for the Eden academic demonstration.\n\n    The engine never fits or modifies a model. It loads the ND09 frozen\n    preprocessors/models and the ND12 packaged runtime state.\n    """\n\n    def __init__(self, workspace_root: str | Path | None = None):\n        if workspace_root is None:\n            workspace_root = Path(__file__).resolve().parents[1]\n        self.root = Path(workspace_root).resolve()\n\n        model_dir = self.root / "model"\n        data_dir = self.root / "data"\n        contract_dir = self.root / "contracts"\n\n        self.runtime_contract = json_load(\n            contract_dir / "runtime_contract.json"\n        )\n        self.model_metadata = json_load(\n            model_dir / "ND09_model_artifact_metadata.json"\n        )\n        self.active_catalogue = pd.read_csv(\n            data_dir / "active_product_catalogue.csv",\n            low_memory=False,\n        )\n        self.active_catalogue[PRODUCT_ID_COLUMN] = (\n            self.active_catalogue[PRODUCT_ID_COLUMN].astype(str)\n        )\n        self.active_catalogue["IncludeInForecast"] = True\n\n        self.history_state = joblib.load(\n            model_dir / "active_history_state.joblib"\n        )\n        self.hierarchy_statistics = joblib.load(\n            model_dir / "hierarchy_statistics.joblib"\n        )\n\n        self.cutoff_date = pd.Timestamp(\n            self.runtime_contract["DataCutoffDate"]\n        ).normalize()\n        self.cutoff_sequence = int(\n            self.runtime_contract["DataCutoffOperatingSequence"]\n        )\n        self.max_validated_horizon = int(\n            self.runtime_contract["MaxValidatedRecursiveOperatingDays"]\n        )\n        self.max_allowed_horizon = int(\n            self.runtime_contract["MaxAllowedRecursiveOperatingDays"]\n        )\n\n        self.numeric_predictors = list(\n            self.model_metadata["NumericPredictors"]\n        )\n        self.categorical_predictors = list(\n            self.model_metadata["CategoricalPredictors"]\n        )\n        self.daily_method = str(\n            self.runtime_contract["Methods"]["NextOperatingDay"]\n        )\n        self.week_method = str(\n            self.runtime_contract["Methods"]["MultiStepOrWeekStart"]\n        )\n        self.updated_week_method = str(\n            self.runtime_contract["Methods"]["DailyUpdatedRemainingWeek"]\n        )\n        self.median_components = list(\n            self.model_metadata["RepresentativeMedianComponents"]\n        )\n\n        if len(self.active_catalogue) != 88:\n            raise AssertionError(\n                "Packaged active catalogue must contain exactly 88 products."\n            )\n\n        core_preprocessor = joblib.load(\n            model_dir / "ND09_core_preprocessor.joblib"\n        )\n        product_preprocessor = joblib.load(\n            model_dir / "ND09_product_preprocessor.joblib"\n        )\n\n        cat_model = CatBoostRegressor()\n        cat_model.load_model(\n            str(model_dir / "ND09_catboost_core53.cbm")\n        )\n        xgb_core = XGBRegressor()\n        xgb_core.load_model(\n            str(model_dir / "ND09_xgboost_core53.json")\n        )\n        xgb_product = XGBRegressor()\n        xgb_product.load_model(\n            str(model_dir / "ND09_xgboost_product_aware.json")\n        )\n\n        self.fitted_models = {\n            "CATBOOST_CORE53": {\n                "model": cat_model,\n                "preprocessor": core_preprocessor,\n                "include_product_id": False,\n            },\n            "XGBOOST_CORE53": {\n                "model": xgb_core,\n                "preprocessor": core_preprocessor,\n                "include_product_id": False,\n            },\n            "XGBOOST_PRODUCT_AWARE": {\n                "model": xgb_product,\n                "preprocessor": product_preprocessor,\n                "include_product_id": True,\n            },\n        }\n\n    def catalogue(self) -> pd.DataFrame:\n        return self.active_catalogue.copy()\n\n    def _validate_horizon(\n        self,\n        calendar: pd.DataFrame,\n        allow_extrapolation: bool,\n    ) -> None:\n        horizon = int(len(calendar))\n        if horizon > self.max_allowed_horizon:\n            raise ValueError(\n                f"Requested horizon is {horizon} operating days; the packaged "\n                f"maximum is {self.max_allowed_horizon}."\n            )\n        if horizon > self.max_validated_horizon and not allow_extrapolation:\n            raise ValueError(\n                f"Requested horizon is {horizon} operating days. The recursive "\n                f"system was evaluated up to {self.max_validated_horizon} "\n                "operating days. Pass allow_extrapolation=True only when the "\n                "demonstration should explicitly show an extrapolative forecast."\n            )\n\n    def _run_path(\n        self,\n        calendar: pd.DataFrame,\n        operational_method: str,\n        state: dict[str, dict[str, list]] | None = None,\n    ) -> pd.DataFrame:\n        if state is None:\n            state = _copy_history_state(self.history_state)\n        else:\n            state = _copy_history_state(state)\n\n        parts = []\n        for horizon, row in enumerate(\n            calendar.sort_values(DATE_COLUMN).itertuples(index=False),\n            start=1,\n        ):\n            calendar_row = pd.Series(row._asdict())\n            base_rows = _build_future_base_rows(\n                self.active_catalogue,\n                calendar_row,\n            )\n            feature_rows = _build_recursive_feature_frame(\n                base_rows,\n                state,\n            )\n            forecast_rows = _predict_routed_system(\n                feature_rows,\n                operational_method,\n                self.median_components,\n                self.fitted_models,\n                self.numeric_predictors,\n                self.categorical_predictors,\n                self.hierarchy_statistics,\n            )\n            forecast_rows["ForecastOriginCutoffDate"] = self.cutoff_date\n            forecast_rows["HorizonOperatingDays"] = horizon\n            forecast_rows["OperationalMethod"] = operational_method\n            forecast_rows["FutureActualDemandUsed"] = False\n            parts.append(forecast_rows)\n\n            _append_values_to_state(\n                state,\n                forecast_rows,\n                forecast_rows["PredictedNormalDemand"].to_numpy(dtype=float),\n            )\n\n        return pd.concat(parts, ignore_index=True)\n\n    @staticmethod\n    def _public_columns(frame: pd.DataFrame) -> pd.DataFrame:\n        columns = [\n            DATE_COLUMN,\n            PRODUCT_ID_COLUMN,\n            PRODUCT_NAME_COLUMN,\n            "ForecastRoute",\n            "MethodUsed",\n            "OperationalMethod",\n            "HorizonOperatingDays",\n            "PredictedNormalDemand",\n        ]\n        existing = [column for column in columns if column in frame.columns]\n        return frame[existing].copy()\n\n    def forecast_date(\n        self,\n        requested_date: str | pd.Timestamp,\n        *,\n        closed_dates: Iterable[str | pd.Timestamp] | None = None,\n        extra_operating_dates: Iterable[str | pd.Timestamp] | None = None,\n        allow_extrapolation: bool = False,\n    ) -> pd.DataFrame:\n        requested_date = pd.Timestamp(requested_date).normalize()\n        closed = _normalize_dates(closed_dates)\n        extra = _normalize_dates(extra_operating_dates)\n\n        if not _is_operating_date(requested_date, closed, extra):\n            raise ValueError(\n                f"{requested_date.date()} is not an operating date under the "\n                "current calendar policy."\n            )\n\n        calendar = _generate_future_calendar(\n            self.cutoff_date,\n            self.cutoff_sequence,\n            requested_date,\n            closed,\n            extra,\n        )\n        self._validate_horizon(calendar, allow_extrapolation)\n\n        method = (\n            self.daily_method\n            if len(calendar) == 1\n            else self.week_method\n        )\n        path = self._run_path(calendar, method)\n        result = path.loc[\n            path[DATE_COLUMN].dt.normalize() == requested_date\n        ].copy()\n        if len(result) != 88:\n            raise AssertionError(\n                f"Expected 88 products on {requested_date.date()}, found "\n                f"{len(result)}."\n            )\n        return self._public_columns(result).reset_index(drop=True)\n\n    def forecast_week(\n        self,\n        week_start: str | pd.Timestamp,\n        *,\n        closed_dates: Iterable[str | pd.Timestamp] | None = None,\n        extra_operating_dates: Iterable[str | pd.Timestamp] | None = None,\n        allow_extrapolation: bool = False,\n    ) -> dict[str, pd.DataFrame]:\n        week_start = pd.Timestamp(week_start).normalize()\n        if week_start.weekday() != 0:\n            raise ValueError("Week forecasts must start on a Monday.")\n        week_end = week_start + pd.Timedelta(days=4)\n\n        closed = _normalize_dates(closed_dates)\n        extra = _normalize_dates(extra_operating_dates)\n\n        calendar = _generate_future_calendar(\n            self.cutoff_date,\n            self.cutoff_sequence,\n            week_end,\n            closed,\n            extra,\n        )\n        self._validate_horizon(calendar, allow_extrapolation)\n\n        path = self._run_path(calendar, self.week_method)\n        week_mask = (\n            (path[DATE_COLUMN].dt.normalize() >= week_start)\n            & (path[DATE_COLUMN].dt.normalize() <= week_end)\n        )\n        daily_product = self._public_columns(\n            path.loc[week_mask].copy()\n        ).reset_index(drop=True)\n\n        expected_dates = [\n            date\n            for date in pd.date_range(week_start, week_end, freq="D")\n            if _is_operating_date(date, closed, extra)\n        ]\n        if daily_product[DATE_COLUMN].nunique() != len(expected_dates):\n            raise AssertionError(\n                "Packaged week forecast did not return every requested "\n                "operating date."\n            )\n        counts = daily_product.groupby(DATE_COLUMN)[PRODUCT_ID_COLUMN].nunique()\n        if not (counts == 88).all():\n            raise AssertionError(\n                "Every operating date in the week must contain 88 products."\n            )\n\n        product_totals = (\n            daily_product.groupby(\n                [PRODUCT_ID_COLUMN, PRODUCT_NAME_COLUMN],\n                as_index=False,\n            )\n            .agg(\n                PredictedNormalDemand=("PredictedNormalDemand", "sum")\n            )\n            .sort_values(\n                ["PredictedNormalDemand", PRODUCT_NAME_COLUMN],\n                ascending=[False, True],\n                kind="mergesort",\n            )\n            .reset_index(drop=True)\n        )\n\n        restaurant_daily = (\n            daily_product.groupby(DATE_COLUMN, as_index=False)\n            .agg(\n                PredictedNormalDemand=("PredictedNormalDemand", "sum")\n            )\n            .sort_values(DATE_COLUMN)\n            .reset_index(drop=True)\n        )\n        restaurant_total = pd.DataFrame(\n            [\n                {\n                    "WeekStart": week_start,\n                    "WeekEnd": week_end,\n                    "PredictedNormalDemand": float(\n                        daily_product["PredictedNormalDemand"].sum()\n                    ),\n                }\n            ]\n        )\n\n        return {\n            "daily_product": daily_product,\n            "product_totals": product_totals,\n            "restaurant_daily": restaurant_daily,\n            "restaurant_total": restaurant_total,\n        }\n\n    def forecast_remaining_week(\n        self,\n        week_start: str | pd.Timestamp,\n        actual_updates: pd.DataFrame,\n        *,\n        closed_dates: Iterable[str | pd.Timestamp] | None = None,\n        extra_operating_dates: Iterable[str | pd.Timestamp] | None = None,\n        allow_extrapolation: bool = False,\n    ) -> dict[str, pd.DataFrame]:\n        """\n        Reforecast the remaining Monday-Friday week after completed days are known.\n\n        actual_updates must contain a complete 88-product panel for every date\n        supplied, with columns:\n          Date, CanonicalProductID, NormalDemand\n\n        Supplied actuals are used only to update recursive history. They are not\n        used to score or evaluate the model.\n        """\n        week_start = pd.Timestamp(week_start).normalize()\n        if week_start.weekday() != 0:\n            raise ValueError("Week forecasts must start on a Monday.")\n        week_end = week_start + pd.Timedelta(days=4)\n\n        closed = _normalize_dates(closed_dates)\n        extra = _normalize_dates(extra_operating_dates)\n\n        calendar = _generate_future_calendar(\n            self.cutoff_date,\n            self.cutoff_sequence,\n            week_end,\n            closed,\n            extra,\n        )\n        self._validate_horizon(calendar, allow_extrapolation)\n\n        actuals = actual_updates.copy()\n        required = {DATE_COLUMN, PRODUCT_ID_COLUMN, TARGET_COLUMN}\n        missing = sorted(required - set(actuals.columns))\n        if missing:\n            raise ValueError(\n                "actual_updates is missing columns: " + ", ".join(missing)\n            )\n        actuals[DATE_COLUMN] = pd.to_datetime(\n            actuals[DATE_COLUMN],\n            errors="raise",\n        ).dt.normalize()\n        actuals[PRODUCT_ID_COLUMN] = actuals[PRODUCT_ID_COLUMN].astype(str)\n        actuals[TARGET_COLUMN] = pd.to_numeric(\n            actuals[TARGET_COLUMN],\n            errors="raise",\n        )\n        if (actuals[TARGET_COLUMN] < 0).any():\n            raise ValueError("Actual NormalDemand cannot be negative.")\n\n        valid_ids = set(self.active_catalogue[PRODUCT_ID_COLUMN].astype(str))\n        unknown_ids = sorted(set(actuals[PRODUCT_ID_COLUMN]) - valid_ids)\n        if unknown_ids:\n            raise ValueError(\n                "actual_updates contains product IDs outside the active "\n                "catalogue: " + ", ".join(unknown_ids)\n            )\n\n        for actual_date, frame in actuals.groupby(DATE_COLUMN):\n            if actual_date < week_start or actual_date > week_end:\n                raise ValueError(\n                    f"Actual update date {actual_date.date()} is outside the "\n                    "requested week."\n                )\n            if set(frame[PRODUCT_ID_COLUMN]) != valid_ids:\n                raise ValueError(\n                    f"Actual updates for {actual_date.date()} must contain a "\n                    "complete 88-product panel, including explicit zeros."\n                )\n            if frame[PRODUCT_ID_COLUMN].duplicated().any():\n                raise ValueError(\n                    f"Duplicate product IDs in actual updates for "\n                    f"{actual_date.date()}."\n                )\n\n        state = _copy_history_state(self.history_state)\n        week_records = []\n\n        for horizon, row in enumerate(\n            calendar.sort_values(DATE_COLUMN).itertuples(index=False),\n            start=1,\n        ):\n            calendar_row = pd.Series(row._asdict())\n            current_date = pd.Timestamp(calendar_row[DATE_COLUMN]).normalize()\n            base_rows = _build_future_base_rows(\n                self.active_catalogue,\n                calendar_row,\n            )\n\n            actual_day = actuals.loc[\n                actuals[DATE_COLUMN] == current_date\n            ].copy()\n\n            if len(actual_day):\n                actual_map = actual_day.set_index(PRODUCT_ID_COLUMN)[TARGET_COLUMN]\n                update_values = (\n                    base_rows[PRODUCT_ID_COLUMN]\n                    .astype(str)\n                    .map(actual_map)\n                    .to_numpy(dtype=float)\n                )\n                if not np.isfinite(update_values).all():\n                    raise AssertionError(\n                        f"Incomplete actual update mapping for {current_date.date()}."\n                    )\n                _append_values_to_state(\n                    state,\n                    base_rows,\n                    update_values,\n                )\n                if week_start <= current_date <= week_end:\n                    output = base_rows[\n                        [DATE_COLUMN, PRODUCT_ID_COLUMN, PRODUCT_NAME_COLUMN]\n                    ].copy()\n                    output["RowStatus"] = "ACTUAL_UPDATE"\n                    output["ActualNormalDemand"] = update_values\n                    output["PredictedNormalDemand"] = np.nan\n                    week_records.append(output)\n                continue\n\n            feature_rows = _build_recursive_feature_frame(\n                base_rows,\n                state,\n            )\n            forecast_rows = _predict_routed_system(\n                feature_rows,\n                self.updated_week_method,\n                self.median_components,\n                self.fitted_models,\n                self.numeric_predictors,\n                self.categorical_predictors,\n                self.hierarchy_statistics,\n            )\n            _append_values_to_state(\n                state,\n                forecast_rows,\n                forecast_rows["PredictedNormalDemand"].to_numpy(dtype=float),\n            )\n\n            if week_start <= current_date <= week_end:\n                output = forecast_rows[\n                    [\n                        DATE_COLUMN,\n                        PRODUCT_ID_COLUMN,\n                        PRODUCT_NAME_COLUMN,\n                        "ForecastRoute",\n                        "MethodUsed",\n                        "PredictedNormalDemand",\n                    ]\n                ].copy()\n                output["RowStatus"] = "FORECAST"\n                output["ActualNormalDemand"] = np.nan\n                week_records.append(output)\n\n        if not week_records:\n            raise AssertionError("No rows were produced for the requested week.")\n\n        combined = pd.concat(week_records, ignore_index=True)\n        combined["EffectiveNormalDemand"] = np.where(\n            combined["RowStatus"] == "ACTUAL_UPDATE",\n            combined["ActualNormalDemand"],\n            combined["PredictedNormalDemand"],\n        )\n\n        future_only = combined.loc[\n            combined["RowStatus"] == "FORECAST"\n        ].copy()\n        restaurant_daily = (\n            combined.groupby([DATE_COLUMN, "RowStatus"], as_index=False)\n            .agg(\n                EffectiveNormalDemand=("EffectiveNormalDemand", "sum")\n            )\n            .sort_values(DATE_COLUMN)\n        )\n        summary = pd.DataFrame(\n            [\n                {\n                    "WeekStart": week_start,\n                    "WeekEnd": week_end,\n                    "KnownActualUnits": float(\n                        combined.loc[\n                            combined["RowStatus"] == "ACTUAL_UPDATE",\n                            "EffectiveNormalDemand",\n                        ].sum()\n                    ),\n                    "RemainingForecastUnits": float(\n                        future_only["PredictedNormalDemand"].sum()\n                    ),\n                    "CombinedWeekUnits": float(\n                        combined["EffectiveNormalDemand"].sum()\n                    ),\n                }\n            ]\n        )\n        return {\n            "combined_product": combined,\n            "remaining_forecast_product": future_only,\n            "restaurant_daily": restaurant_daily,\n            "summary": summary,\n        }\n\n\ndef json_load(path: Path) -> dict:\n    import json\n\n    return json.loads(path.read_text(encoding="utf-8"))\n'
PLANNING_ENGINE_SOURCE = '\nfrom __future__ import annotations\n\nfrom pathlib import Path\nfrom typing import Any\n\nimport numpy as np\nimport pandas as pd\n\n\nDATE_COLUMN = "Date"\nPRODUCT_ID_COLUMN = "CanonicalProductID"\nPRODUCT_NAME_COLUMN = "CanonicalProductName"\nNORMAL_COLUMN = "PredictedNormalDemand"\nBULK_COLUMN = "ConfirmedBulkDemand"\nPLANNED_COLUMN = "PlannedQuantity"\n\n\nclass PlanningEngine:\n    def __init__(self, active_catalogue: pd.DataFrame):\n        self.active_catalogue = active_catalogue.copy()\n        self.active_catalogue[PRODUCT_ID_COLUMN] = (\n            self.active_catalogue[PRODUCT_ID_COLUMN].astype(str)\n        )\n        self.valid_ids = set(self.active_catalogue[PRODUCT_ID_COLUMN])\n\n    @staticmethod\n    def _normalise_bulk_orders(\n        bulk_orders: pd.DataFrame | list[dict[str, Any]] | None,\n    ) -> pd.DataFrame:\n        if bulk_orders is None:\n            return pd.DataFrame(\n                columns=[DATE_COLUMN, PRODUCT_ID_COLUMN, BULK_COLUMN]\n            )\n\n        if isinstance(bulk_orders, list):\n            bulk = pd.DataFrame(bulk_orders)\n        else:\n            bulk = bulk_orders.copy()\n\n        required = {DATE_COLUMN, PRODUCT_ID_COLUMN, BULK_COLUMN}\n        missing = sorted(required - set(bulk.columns))\n        if missing:\n            raise ValueError(\n                "Bulk orders are missing columns: " + ", ".join(missing)\n            )\n\n        bulk = bulk[[DATE_COLUMN, PRODUCT_ID_COLUMN, BULK_COLUMN]].copy()\n        bulk[DATE_COLUMN] = pd.to_datetime(\n            bulk[DATE_COLUMN],\n            errors="raise",\n        ).dt.normalize()\n        bulk[PRODUCT_ID_COLUMN] = bulk[PRODUCT_ID_COLUMN].astype(str)\n        bulk[BULK_COLUMN] = pd.to_numeric(\n            bulk[BULK_COLUMN],\n            errors="raise",\n        )\n\n        if (bulk[BULK_COLUMN] < 0).any():\n            raise ValueError("Confirmed bulk demand cannot be negative.")\n\n        return (\n            bulk.groupby(\n                [DATE_COLUMN, PRODUCT_ID_COLUMN],\n                as_index=False,\n            )[BULK_COLUMN]\n            .sum()\n        )\n\n    def apply_bulk_orders(\n        self,\n        product_forecast: pd.DataFrame,\n        bulk_orders: pd.DataFrame | list[dict[str, Any]] | None = None,\n    ) -> pd.DataFrame:\n        plan = product_forecast.copy()\n\n        required = {\n            DATE_COLUMN,\n            PRODUCT_ID_COLUMN,\n            PRODUCT_NAME_COLUMN,\n            NORMAL_COLUMN,\n        }\n        missing = sorted(required - set(plan.columns))\n        if missing:\n            raise ValueError(\n                "Product forecast is missing columns: " + ", ".join(missing)\n            )\n\n        plan[DATE_COLUMN] = pd.to_datetime(\n            plan[DATE_COLUMN],\n            errors="raise",\n        ).dt.normalize()\n        plan[PRODUCT_ID_COLUMN] = plan[PRODUCT_ID_COLUMN].astype(str)\n        plan[NORMAL_COLUMN] = pd.to_numeric(\n            plan[NORMAL_COLUMN],\n            errors="raise",\n        )\n\n        if (plan[NORMAL_COLUMN] < 0).any():\n            raise ValueError("Predicted normal demand cannot be negative.")\n\n        bulk = self._normalise_bulk_orders(bulk_orders)\n        unknown = sorted(set(bulk[PRODUCT_ID_COLUMN]) - self.valid_ids)\n        if unknown:\n            raise ValueError(\n                "Bulk orders contain product IDs outside the active catalogue: "\n                + ", ".join(unknown)\n            )\n\n        forecast_dates = set(plan[DATE_COLUMN])\n        invalid_dates = sorted(set(bulk[DATE_COLUMN]) - forecast_dates)\n        if invalid_dates:\n            raise ValueError(\n                "Bulk orders contain dates outside the forecast output: "\n                + ", ".join(str(value.date()) for value in invalid_dates)\n            )\n\n        plan = plan.merge(\n            bulk,\n            on=[DATE_COLUMN, PRODUCT_ID_COLUMN],\n            how="left",\n            validate="one_to_one",\n        )\n        plan[BULK_COLUMN] = plan[BULK_COLUMN].fillna(0.0).astype(float)\n        plan[PLANNED_COLUMN] = (\n            plan[NORMAL_COLUMN].astype(float)\n            + plan[BULK_COLUMN].astype(float)\n        )\n\n        if not np.allclose(\n            plan[PLANNED_COLUMN].to_numpy(dtype=float),\n            (\n                plan[NORMAL_COLUMN].to_numpy(dtype=float)\n                + plan[BULK_COLUMN].to_numpy(dtype=float)\n            ),\n            atol=1e-12,\n            rtol=0.0,\n        ):\n            raise AssertionError("Planning formula reconciliation failed.")\n        return plan\n\n    @staticmethod\n    def restaurant_daily(product_plan: pd.DataFrame) -> pd.DataFrame:\n        return (\n            product_plan.groupby(DATE_COLUMN, as_index=False)\n            .agg(\n                PredictedNormalDemand=(NORMAL_COLUMN, "sum"),\n                ConfirmedBulkDemand=(BULK_COLUMN, "sum"),\n                PlannedQuantity=(PLANNED_COLUMN, "sum"),\n            )\n            .sort_values(DATE_COLUMN)\n            .reset_index(drop=True)\n        )\n\n    @staticmethod\n    def product_week_totals(product_plan: pd.DataFrame) -> pd.DataFrame:\n        return (\n            product_plan.groupby(\n                [PRODUCT_ID_COLUMN, PRODUCT_NAME_COLUMN],\n                as_index=False,\n            )\n            .agg(\n                PredictedNormalDemand=(NORMAL_COLUMN, "sum"),\n                ConfirmedBulkDemand=(BULK_COLUMN, "sum"),\n                PlannedQuantity=(PLANNED_COLUMN, "sum"),\n            )\n            .sort_values(\n                [PLANNED_COLUMN, PRODUCT_NAME_COLUMN],\n                ascending=[False, True],\n                kind="mergesort",\n            )\n            .reset_index(drop=True)\n        )\n\n    @staticmethod\n    def restaurant_total(product_plan: pd.DataFrame) -> pd.DataFrame:\n        return pd.DataFrame(\n            [\n                {\n                    "PredictedNormalDemand": float(\n                        product_plan[NORMAL_COLUMN].sum()\n                    ),\n                    "ConfirmedBulkDemand": float(\n                        product_plan[BULK_COLUMN].sum()\n                    ),\n                    "PlannedQuantity": float(\n                        product_plan[PLANNED_COLUMN].sum()\n                    ),\n                }\n            ]\n        )\n'
INGREDIENT_ENGINE_SOURCE = '\nfrom __future__ import annotations\n\nfrom pathlib import Path\n\nimport numpy as np\nimport pandas as pd\n\n\nDATE_COLUMN = "Date"\nPRODUCT_ID_COLUMN = "CanonicalProductID"\nPRODUCT_NAME_COLUMN = "CanonicalProductName"\nNORMAL_COLUMN = "PredictedNormalDemand"\nBULK_COLUMN = "ConfirmedBulkDemand"\nPLANNED_COLUMN = "PlannedQuantity"\n\nMAPPING_STATUS = "ILLUSTRATIVE_ASSUMED_RECIPE"\n\n\nclass IngredientEngine:\n    def __init__(self, mapping_path: str | Path):\n        self.mapping_path = Path(mapping_path)\n        self.mapping = pd.read_csv(self.mapping_path, low_memory=False)\n        self.mapping[PRODUCT_ID_COLUMN] = (\n            self.mapping[PRODUCT_ID_COLUMN].astype(str)\n        )\n\n        required = {\n            PRODUCT_ID_COLUMN,\n            PRODUCT_NAME_COLUMN,\n            "IngredientID",\n            "IngredientName",\n            "QuantityPerProductUnit",\n            "IngredientUnit",\n            "IngredientMappingStatus",\n        }\n        missing = sorted(required - set(self.mapping.columns))\n        if missing:\n            raise ValueError(\n                "Ingredient mapping is missing columns: " + ", ".join(missing)\n            )\n\n        if set(self.mapping["IngredientMappingStatus"].astype(str)) != {\n            MAPPING_STATUS\n        }:\n            raise AssertionError(\n                "The packaged mapping is not uniformly labelled as "\n                "ILLUSTRATIVE_ASSUMED_RECIPE."\n            )\n\n        self.mapped_ids = set(self.mapping[PRODUCT_ID_COLUMN])\n        if len(self.mapped_ids) != 14:\n            raise AssertionError(\n                "The demonstration mapping must contain exactly 14 products."\n            )\n\n        unit_consistency = (\n            self.mapping.groupby("IngredientID")["IngredientUnit"].nunique()\n        )\n        if (unit_consistency != 1).any():\n            raise AssertionError(\n                "At least one ingredient uses incompatible units."\n            )\n\n    def mapped_catalogue(self) -> pd.DataFrame:\n        return (\n            self.mapping[\n                [\n                    PRODUCT_ID_COLUMN,\n                    PRODUCT_NAME_COLUMN,\n                    "RecipeFamily",\n                    "PriceTierEuro",\n                    "IngredientMappingStatus",\n                ]\n            ]\n            .drop_duplicates()\n            .sort_values(PRODUCT_NAME_COLUMN)\n            .reset_index(drop=True)\n        )\n\n    def requirements_by_product(\n        self,\n        product_plan: pd.DataFrame,\n    ) -> pd.DataFrame:\n        required = {\n            DATE_COLUMN,\n            PRODUCT_ID_COLUMN,\n            PRODUCT_NAME_COLUMN,\n            NORMAL_COLUMN,\n            BULK_COLUMN,\n            PLANNED_COLUMN,\n        }\n        missing = sorted(required - set(product_plan.columns))\n        if missing:\n            raise ValueError(\n                "Product plan is missing columns: " + ", ".join(missing)\n            )\n\n        plan = product_plan.copy()\n        plan[DATE_COLUMN] = pd.to_datetime(\n            plan[DATE_COLUMN],\n            errors="raise",\n        ).dt.normalize()\n        plan[PRODUCT_ID_COLUMN] = plan[PRODUCT_ID_COLUMN].astype(str)\n\n        merged = plan.merge(\n            self.mapping,\n            on=[PRODUCT_ID_COLUMN, PRODUCT_NAME_COLUMN],\n            how="inner",\n            validate="many_to_many",\n        )\n\n        quantity = pd.to_numeric(\n            merged["QuantityPerProductUnit"],\n            errors="raise",\n        )\n        merged["NormalIngredientRequirement"] = (\n            pd.to_numeric(merged[NORMAL_COLUMN], errors="raise") * quantity\n        )\n        merged["BulkIngredientRequirement"] = (\n            pd.to_numeric(merged[BULK_COLUMN], errors="raise") * quantity\n        )\n        merged["PlannedIngredientRequirement"] = (\n            pd.to_numeric(merged[PLANNED_COLUMN], errors="raise") * quantity\n        )\n\n        expected = (\n            pd.to_numeric(merged[PLANNED_COLUMN], errors="raise") * quantity\n        )\n        if not np.allclose(\n            expected.to_numpy(dtype=float),\n            merged["PlannedIngredientRequirement"].to_numpy(dtype=float),\n            atol=1e-12,\n            rtol=0.0,\n        ):\n            raise AssertionError("Ingredient formula reconciliation failed.")\n        return merged\n\n    def daily_totals(self, product_plan: pd.DataFrame) -> pd.DataFrame:\n        rows = self.requirements_by_product(product_plan)\n        return (\n            rows.groupby(\n                [\n                    DATE_COLUMN,\n                    "IngredientID",\n                    "IngredientName",\n                    "IngredientUnit",\n                ],\n                as_index=False,\n            )\n            .agg(\n                NormalIngredientRequirement=(\n                    "NormalIngredientRequirement",\n                    "sum",\n                ),\n                BulkIngredientRequirement=(\n                    "BulkIngredientRequirement",\n                    "sum",\n                ),\n                PlannedIngredientRequirement=(\n                    "PlannedIngredientRequirement",\n                    "sum",\n                ),\n            )\n            .sort_values(\n                [\n                    DATE_COLUMN,\n                    "IngredientUnit",\n                    "PlannedIngredientRequirement",\n                    "IngredientName",\n                ],\n                ascending=[True, True, False, True],\n                kind="mergesort",\n            )\n            .reset_index(drop=True)\n        )\n\n    def total_requirements(self, product_plan: pd.DataFrame) -> pd.DataFrame:\n        rows = self.requirements_by_product(product_plan)\n        return (\n            rows.groupby(\n                ["IngredientID", "IngredientName", "IngredientUnit"],\n                as_index=False,\n            )\n            .agg(\n                NormalIngredientRequirement=(\n                    "NormalIngredientRequirement",\n                    "sum",\n                ),\n                BulkIngredientRequirement=(\n                    "BulkIngredientRequirement",\n                    "sum",\n                ),\n                PlannedIngredientRequirement=(\n                    "PlannedIngredientRequirement",\n                    "sum",\n                ),\n            )\n            .sort_values(\n                [\n                    "IngredientUnit",\n                    "PlannedIngredientRequirement",\n                    "IngredientName",\n                ],\n                ascending=[True, False, True],\n                kind="mergesort",\n            )\n            .reset_index(drop=True)\n        )\n'
BACKEND_SOURCE = '\nfrom __future__ import annotations\n\nfrom pathlib import Path\nfrom typing import Any, Iterable\n\nimport pandas as pd\n\nfrom engine.forecast_engine import ForecastEngine\nfrom engine.ingredient_engine import IngredientEngine\nfrom engine.planning_engine import PlanningEngine\n\n\nclass EdenDemoBackend:\n    """\n    Stable application-facing facade.\n\n    Codex/UI code should call this class instead of reaching into model files.\n    """\n\n    def __init__(self, workspace_root: str | Path | None = None):\n        if workspace_root is None:\n            workspace_root = Path(__file__).resolve().parents[1]\n        self.root = Path(workspace_root).resolve()\n\n        self.forecaster = ForecastEngine(self.root)\n        self.planner = PlanningEngine(self.forecaster.catalogue())\n        self.ingredients = IngredientEngine(\n            self.root / "data" / "illustrative_ingredient_mapping.csv"\n        )\n\n    def catalogue(self) -> pd.DataFrame:\n        catalogue = self.forecaster.catalogue()\n        catalogue["HasIllustrativeIngredientMapping"] = (\n            catalogue["CanonicalProductID"]\n            .astype(str)\n            .isin(self.ingredients.mapped_ids)\n        )\n        return catalogue\n\n    def mapped_ingredient_products(self) -> pd.DataFrame:\n        return self.ingredients.mapped_catalogue()\n\n    def forecast_date(\n        self,\n        date: str,\n        *,\n        bulk_orders: pd.DataFrame | list[dict[str, Any]] | None = None,\n        closed_dates: Iterable[str] | None = None,\n        extra_operating_dates: Iterable[str] | None = None,\n        allow_extrapolation: bool = False,\n    ) -> dict[str, pd.DataFrame]:\n        forecast = self.forecaster.forecast_date(\n            date,\n            closed_dates=closed_dates,\n            extra_operating_dates=extra_operating_dates,\n            allow_extrapolation=allow_extrapolation,\n        )\n        plan = self.planner.apply_bulk_orders(forecast, bulk_orders)\n        restaurant = self.planner.restaurant_daily(plan)\n        ingredient_daily = self.ingredients.daily_totals(plan)\n        ingredient_total = self.ingredients.total_requirements(plan)\n        return {\n            "product_plan": plan,\n            "restaurant_total": restaurant,\n            "ingredient_daily": ingredient_daily,\n            "ingredient_total": ingredient_total,\n        }\n\n    def forecast_week(\n        self,\n        monday: str,\n        *,\n        bulk_orders: pd.DataFrame | list[dict[str, Any]] | None = None,\n        closed_dates: Iterable[str] | None = None,\n        extra_operating_dates: Iterable[str] | None = None,\n        allow_extrapolation: bool = False,\n    ) -> dict[str, pd.DataFrame]:\n        raw = self.forecaster.forecast_week(\n            monday,\n            closed_dates=closed_dates,\n            extra_operating_dates=extra_operating_dates,\n            allow_extrapolation=allow_extrapolation,\n        )\n        plan = self.planner.apply_bulk_orders(\n            raw["daily_product"],\n            bulk_orders,\n        )\n        return {\n            "daily_product_plan": plan,\n            "product_week_totals": self.planner.product_week_totals(plan),\n            "restaurant_daily": self.planner.restaurant_daily(plan),\n            "restaurant_total": self.planner.restaurant_total(plan),\n            "ingredient_daily": self.ingredients.daily_totals(plan),\n            "ingredient_week_total": self.ingredients.total_requirements(plan),\n        }\n\n    def forecast_remaining_week(\n        self,\n        monday: str,\n        actual_updates: pd.DataFrame,\n        *,\n        bulk_orders: pd.DataFrame | list[dict[str, Any]] | None = None,\n        closed_dates: Iterable[str] | None = None,\n        extra_operating_dates: Iterable[str] | None = None,\n        allow_extrapolation: bool = False,\n    ) -> dict[str, pd.DataFrame]:\n        raw = self.forecaster.forecast_remaining_week(\n            monday,\n            actual_updates,\n            closed_dates=closed_dates,\n            extra_operating_dates=extra_operating_dates,\n            allow_extrapolation=allow_extrapolation,\n        )\n        remaining = raw["remaining_forecast_product"].copy()\n        if len(remaining):\n            remaining_plan = self.planner.apply_bulk_orders(\n                remaining,\n                bulk_orders,\n            )\n            ingredient_remaining = self.ingredients.daily_totals(\n                remaining_plan\n            )\n        else:\n            remaining_plan = remaining\n            ingredient_remaining = pd.DataFrame()\n\n        return {\n            **raw,\n            "remaining_product_plan": remaining_plan,\n            "remaining_ingredient_daily": ingredient_remaining,\n        }\n'
CLI_SOURCE = '\nfrom __future__ import annotations\n\nimport argparse\n\nfrom engine.backend import EdenDemoBackend\n\n\ndef main() -> None:\n    parser = argparse.ArgumentParser(\n        description="Eden demand-forecasting demonstration backend CLI"\n    )\n    subparsers = parser.add_subparsers(dest="mode", required=True)\n\n    daily = subparsers.add_parser("daily")\n    daily.add_argument("date")\n    daily.add_argument("--allow-extrapolation", action="store_true")\n\n    weekly = subparsers.add_parser("weekly")\n    weekly.add_argument("monday")\n    weekly.add_argument("--allow-extrapolation", action="store_true")\n\n    args = parser.parse_args()\n    backend = EdenDemoBackend()\n\n    if args.mode == "daily":\n        result = backend.forecast_date(\n            args.date,\n            allow_extrapolation=args.allow_extrapolation,\n        )\n        print(result["restaurant_total"].to_string(index=False))\n        print()\n        print(\n            result["product_plan"]\n            .sort_values("PlannedQuantity", ascending=False)\n            .head(15)\n            .to_string(index=False)\n        )\n    else:\n        result = backend.forecast_week(\n            args.monday,\n            allow_extrapolation=args.allow_extrapolation,\n        )\n        print(result["restaurant_total"].to_string(index=False))\n        print()\n        print(\n            result["product_week_totals"]\n            .head(15)\n            .to_string(index=False)\n        )\n\n\nif __name__ == "__main__":\n    main()\n'
TEST_SMOKE_SOURCE = '\nfrom __future__ import annotations\n\nimport numpy as np\nimport pandas as pd\n\nfrom engine.backend import EdenDemoBackend\n\n\nEXPECTED_DAILY = 727.756809\nEXPECTED_WEEK = 3341.504320\n\n\ndef test_frozen_demo_reproduction():\n    backend = EdenDemoBackend()\n\n    daily = backend.forecast_date("2026-03-02")\n    assert len(daily["product_plan"]) == 88\n    daily_total = float(\n        daily["restaurant_total"]["PredictedNormalDemand"].iloc[0]\n    )\n    assert np.isclose(daily_total, EXPECTED_DAILY, atol=1e-6, rtol=0)\n\n    week = backend.forecast_week("2026-03-02")\n    assert week["daily_product_plan"]["Date"].nunique() == 5\n    assert (\n        week["daily_product_plan"]\n        .groupby("Date")["CanonicalProductID"]\n        .nunique()\n        .eq(88)\n        .all()\n    )\n    week_total = float(\n        week["restaurant_total"]["PredictedNormalDemand"].iloc[0]\n    )\n    assert np.isclose(week_total, EXPECTED_WEEK, atol=1e-6, rtol=0)\n\n\ndef test_ingredient_scope_and_bulk_formula():\n    backend = EdenDemoBackend()\n    assert len(backend.catalogue()) == 88\n    assert len(backend.mapped_ingredient_products()) == 14\n\n    mapped_id = (\n        backend.mapped_ingredient_products()["CanonicalProductID"]\n        .astype(str)\n        .iloc[0]\n    )\n    bulk = pd.DataFrame(\n        [\n            {\n                "Date": "2026-03-02",\n                "CanonicalProductID": mapped_id,\n                "ConfirmedBulkDemand": 10.0,\n            }\n        ]\n    )\n    baseline = backend.forecast_date("2026-03-02")\n    with_bulk = backend.forecast_date(\n        "2026-03-02",\n        bulk_orders=bulk,\n    )\n    baseline_total = float(\n        baseline["restaurant_total"]["PlannedQuantity"].iloc[0]\n    )\n    bulk_total = float(\n        with_bulk["restaurant_total"]["PlannedQuantity"].iloc[0]\n    )\n    assert np.isclose(bulk_total - baseline_total, 10.0, atol=1e-9)\n\n    ingredient_rows = backend.ingredients.mapping.loc[\n        backend.ingredients.mapping["CanonicalProductID"].astype(str)\n        == mapped_id\n    ]\n    before = baseline["ingredient_total"].set_index("IngredientID")\n    after = with_bulk["ingredient_total"].set_index("IngredientID")\n\n    for row in ingredient_rows.itertuples(index=False):\n        ingredient_id = row.IngredientID\n        expected_delta = 10.0 * float(row.QuantityPerProductUnit)\n        actual_delta = float(\n            after.loc[ingredient_id, "PlannedIngredientRequirement"]\n            - before.loc[ingredient_id, "PlannedIngredientRequirement"]\n        )\n        assert np.isclose(\n            actual_delta,\n            expected_delta,\n            atol=1e-9,\n            rtol=0,\n        )\n\n\ndef test_no_unmapped_product_is_reported_as_zero_recipe():\n    backend = EdenDemoBackend()\n    catalogue = backend.catalogue()\n    assert catalogue["HasIllustrativeIngredientMapping"].sum() == 14\n    assert (~catalogue["HasIllustrativeIngredientMapping"]).sum() == 74\n'


# =============================================================================
# HELPERS
# =============================================================================

def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


def write_text(path: Path, text: str) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(text, encoding="utf-8")


def write_json(path: Path, payload: dict) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(
        json.dumps(payload, indent=2, ensure_ascii=False, default=str) + "\n",
        encoding="utf-8",
    )


def write_csv(path: Path, frame: pd.DataFrame) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    frame.to_csv(path, index=False)


def normalize_family(values: pd.Series) -> pd.Series:
    return values.astype("string").fillna("__MISSING_FAMILY__").astype(str)


def build_history_state(
    training_frame: pd.DataFrame,
    active_ids: set[str],
) -> dict[str, dict[str, list]]:
    active_history = training_frame.loc[
        training_frame[PRODUCT_ID_COLUMN].astype(str).isin(active_ids)
    ].copy()
    ordered = active_history.sort_values(
        [PRODUCT_ID_COLUMN, DATE_COLUMN, SEQUENCE_COLUMN],
        kind="mergesort",
    )
    state = {}
    for product_id, product_frame in ordered.groupby(
        PRODUCT_ID_COLUMN,
        sort=False,
    ):
        state[str(product_id)] = {
            "demand": product_frame[TARGET_COLUMN].astype(float).tolist(),
            "sequence": product_frame[SEQUENCE_COLUMN].astype(float).tolist(),
        }
    if set(state) != active_ids:
        raise AssertionError(
            "Packaged history state does not contain every active product."
        )
    return state


def build_hierarchy_statistics(training_frame: pd.DataFrame) -> dict:
    train = training_frame.copy()
    train[PRODUCT_ID_COLUMN] = train[PRODUCT_ID_COLUMN].astype(str)
    train["_FamilyKey"] = normalize_family(train[FAMILY_COLUMN])
    train[DAY_OF_WEEK_COLUMN] = pd.to_numeric(
        train[DAY_OF_WEEK_COLUMN],
        errors="raise",
    ).astype(int)

    return {
        "global_mean": float(train[TARGET_COLUMN].mean()),
        "product_weekday": train.groupby(
            [PRODUCT_ID_COLUMN, DAY_OF_WEEK_COLUMN],
            dropna=False,
        )[TARGET_COLUMN].mean(),
        "product_mean": train.groupby(
            PRODUCT_ID_COLUMN,
            dropna=False,
        )[TARGET_COLUMN].mean(),
        "family_weekday": train.groupby(
            ["_FamilyKey", DAY_OF_WEEK_COLUMN],
            dropna=False,
        )[TARGET_COLUMN].mean(),
        "family_mean": train.groupby(
            "_FamilyKey",
            dropna=False,
        )[TARGET_COLUMN].mean(),
        "global_weekday": train.groupby(
            DAY_OF_WEEK_COLUMN,
            dropna=False,
        )[TARGET_COLUMN].mean(),
    }


def build_manifest(root: Path) -> pd.DataFrame:
    records = []
    for path in sorted(root.rglob("*")):
        if path.is_file() and ".git" not in path.parts:
            records.append(
                {
                    "RelativePath": str(path.relative_to(root)),
                    "Bytes": int(path.stat().st_size),
                    "SHA256": sha256_file(path),
                }
            )
    return pd.DataFrame(records)


def compare_ingredient_totals(
    generated: pd.DataFrame,
    reference: pd.DataFrame,
    label: str,
) -> float:
    keys = ["IngredientID", "IngredientUnit"]
    metric = "PlannedIngredientRequirement"

    for frame in [generated, reference]:
        if metric not in frame.columns:
            raise AssertionError(
                f"{label}: missing {metric}."
            )

    left = (
        generated.groupby(keys, as_index=False)[metric]
        .sum()
        .rename(columns={metric: "Generated"})
    )
    right = (
        reference.groupby(keys, as_index=False)[metric]
        .sum()
        .rename(columns={metric: "Reference"})
    )
    compare = left.merge(
        right,
        on=keys,
        how="outer",
        validate="one_to_one",
    ).fillna(0.0)
    max_diff = float(
        np.max(
            np.abs(
                compare["Generated"].to_numpy(dtype=float)
                - compare["Reference"].to_numpy(dtype=float)
            )
        )
    )
    if max_diff > 1e-8:
        raise AssertionError(
            f"{label} ingredient totals do not reproduce ND11. "
            f"Maximum difference: {max_diff}"
        )
    return max_diff


# =============================================================================
# PREFLIGHT
# =============================================================================

required_paths = [
    ALL_ROUTES_PATH,
    ACTIVE_CATALOGUE_PATH,
    ND09A_CONTRACT_PATH,
    ND09A_SCOPE_LOCK_PATH,
    ND11_MAPPING_PATH,
    ND11_INGREDIENT_MASTER_PATH,
    ND11_BACKEND_CONTRACT_PATH,
    ND11_DISCLAIMER_PATH,
    *CHECKPOINT_PATHS.values(),
    *[ND09_MODEL_DIR / filename for filename in MODEL_ARTIFACTS],
    *ND10A_EXAMPLE_FILES.values(),
    *ND11_EXAMPLE_FILES.values(),
]
missing = [path for path in required_paths if not path.is_file()]
if missing:
    raise FileNotFoundError(
        "ND12 required source files are missing:\n"
        + "\n".join(f"- {path}" for path in missing)
    )

actual_checkpoint_hashes = {
    name: sha256_file(path)
    for name, path in CHECKPOINT_PATHS.items()
}
for name, expected in EXPECTED_CHECKPOINT_HASHES.items():
    actual = actual_checkpoint_hashes[name]
    if actual != expected:
        raise AssertionError(
            f"{name} checkpoint mismatch.\n"
            f"Expected: {expected}\n"
            f"Actual:   {actual}"
        )

protected_hashes_before = {
    str(path): sha256_file(path)
    for path in required_paths
}

if DEMO_WORKSPACE_ROOT.exists():
    if not ALLOW_OVERWRITE:
        raise FileExistsError(
            "The demonstration workspace already exists. "
            "Nothing was changed:\n"
            f"{DEMO_WORKSPACE_ROOT}"
        )
    shutil.rmtree(DEMO_WORKSPACE_ROOT)

zip_path = DEMO_WORKSPACE_ROOT.parent / (
    DEMO_WORKSPACE_ROOT.name + ".zip"
)
if zip_path.exists():
    if not ALLOW_OVERWRITE:
        raise FileExistsError(
            "The demonstration workspace ZIP already exists. "
            "Nothing was changed:\n"
            f"{zip_path}"
        )
    zip_path.unlink()

STAGING_ROOT = DEMO_WORKSPACE_ROOT.parent / (
    f".{DEMO_WORKSPACE_ROOT.name}_staging_{uuid.uuid4().hex}"
)
STAGING_ROOT.mkdir(parents=True, exist_ok=False)


# =============================================================================
# BUILD CLEAN WORKSPACE
# =============================================================================

try:
    directories = {
        "engine": STAGING_ROOT / "engine",
        "model": STAGING_ROOT / "model",
        "data": STAGING_ROOT / "data",
        "contracts": STAGING_ROOT / "contracts",
        "examples": STAGING_ROOT / "examples",
        "tests": STAGING_ROOT / "tests",
        "docs": STAGING_ROOT / "docs",
        "exports": STAGING_ROOT / "exports",
        "control": STAGING_ROOT / "control",
        "app": STAGING_ROOT / "app",
    }
    for directory in directories.values():
        directory.mkdir(parents=True, exist_ok=True)

    # -------------------------------------------------------------------------
    # Read authoritative active catalogue and minimal development state source.
    # -------------------------------------------------------------------------
    active_catalogue = pd.read_csv(
        ACTIVE_CATALOGUE_PATH,
        low_memory=False,
    )
    active_catalogue[PRODUCT_ID_COLUMN] = (
        active_catalogue[PRODUCT_ID_COLUMN].astype(str)
    )
    active_catalogue["IncludeInForecast"] = True

    if len(active_catalogue) != EXPECTED_ACTIVE_PRODUCTS:
        raise AssertionError(
            f"Expected {EXPECTED_ACTIVE_PRODUCTS} active products; "
            f"found {len(active_catalogue)}."
        )
    if active_catalogue[PRODUCT_ID_COLUMN].duplicated().any():
        raise AssertionError("Active catalogue contains duplicate product IDs.")

    active_ids = set(active_catalogue[PRODUCT_ID_COLUMN])

    all_routes = pd.read_csv(
        ALL_ROUTES_PATH,
        low_memory=False,
        parse_dates=[DATE_COLUMN],
    )
    all_routes[PRODUCT_ID_COLUMN] = all_routes[PRODUCT_ID_COLUMN].astype(str)
    all_routes[TARGET_COLUMN] = pd.to_numeric(
        all_routes[TARGET_COLUMN],
        errors="raise",
    )
    all_routes[SEQUENCE_COLUMN] = pd.to_numeric(
        all_routes[SEQUENCE_COLUMN],
        errors="raise",
    )

    cutoff_date = pd.Timestamp(all_routes[DATE_COLUMN].max()).normalize()
    cutoff_sequence = int(
        all_routes.loc[
            all_routes[DATE_COLUMN] == cutoff_date,
            SEQUENCE_COLUMN,
        ].max()
    )
    if cutoff_date != pd.Timestamp("2026-02-27"):
        raise AssertionError(
            f"Expected frozen cutoff 2026-02-27; found {cutoff_date.date()}."
        )

    history_state = build_history_state(all_routes, active_ids)
    hierarchy_statistics = build_hierarchy_statistics(all_routes)

    # -------------------------------------------------------------------------
    # Copy frozen model artifacts exactly.
    # -------------------------------------------------------------------------
    for filename in MODEL_ARTIFACTS:
        shutil.copy2(
            ND09_MODEL_DIR / filename,
            directories["model"] / filename,
        )

    joblib.dump(
        history_state,
        directories["model"] / "active_history_state.joblib",
    )
    joblib.dump(
        hierarchy_statistics,
        directories["model"] / "hierarchy_statistics.joblib",
    )

    # -------------------------------------------------------------------------
    # Copy clean data/contracts only.
    # -------------------------------------------------------------------------
    write_csv(
        directories["data"] / "active_product_catalogue.csv",
        active_catalogue,
    )

    mapping = pd.read_csv(ND11_MAPPING_PATH, low_memory=False)
    mapping[PRODUCT_ID_COLUMN] = mapping[PRODUCT_ID_COLUMN].astype(str)
    mapped_ids = set(mapping[PRODUCT_ID_COLUMN])
    if len(mapped_ids) != EXPECTED_MAPPED_PRODUCTS:
        raise AssertionError(
            f"Expected {EXPECTED_MAPPED_PRODUCTS} ingredient-mapped products; "
            f"found {len(mapped_ids)}."
        )
    if not mapped_ids.issubset(active_ids):
        raise AssertionError(
            "At least one ingredient-mapped product is not active."
        )
    if set(mapping["IngredientMappingStatus"].astype(str)) != {
        "ILLUSTRATIVE_ASSUMED_RECIPE"
    }:
        raise AssertionError(
            "Ingredient mapping is not uniformly labelled illustrative."
        )

    write_csv(
        directories["data"] / "illustrative_ingredient_mapping.csv",
        mapping,
    )
    shutil.copy2(
        ND11_INGREDIENT_MASTER_PATH,
        directories["data"] / "ingredient_master.csv",
    )

    shutil.copy2(
        ND09A_CONTRACT_PATH,
        directories["contracts"] / "ND09A_corrected_inference_contract.json",
    )
    shutil.copy2(
        ND09A_SCOPE_LOCK_PATH,
        directories["contracts"] / "ND09A_active_catalogue_scope_lock.json",
    )
    shutil.copy2(
        ND11_BACKEND_CONTRACT_PATH,
        directories["contracts"] / "ND11_demo_backend_contract.json",
    )
    shutil.copy2(
        ND11_DISCLAIMER_PATH,
        directories["docs"] / "INGREDIENT_MAPPING_DISCLAIMER.md",
    )

    # -------------------------------------------------------------------------
    # Runtime contract: one stable application-facing source of truth.
    # -------------------------------------------------------------------------
    runtime_contract = {
        "ContractVersion": "EDEN_DEMO_RUNTIME_V1",
        "Purpose": (
            "Frozen academic demonstration backend for Eden normal-demand "
            "forecasting, planning and illustrative ingredient conversion."
        ),
        "DataCutoffDate": cutoff_date.strftime("%Y-%m-%d"),
        "DataCutoffOperatingSequence": cutoff_sequence,
        "ActiveForecastCatalogueProducts": EXPECTED_ACTIVE_PRODUCTS,
        "IngredientMappedProducts": EXPECTED_MAPPED_PRODUCTS,
        "CataloguePolicy": "LATEST_KNOWN_ACTIVE_PANEL",
        "Methods": {
            "NextOperatingDay": "NESTED_MEDIAN_ENSEMBLE",
            "MultiStepOrWeekStart": "ROLLING_MEAN_5",
            "DailyUpdatedRemainingWeek": "ROLLING_MEAN_5",
        },
        "MaxValidatedRecursiveOperatingDays": 5,
        "MaxAllowedRecursiveOperatingDays": 60,
        "PlanningFormula": (
            "PlannedQuantity = PredictedNormalDemand + ConfirmedBulkDemand"
        ),
        "IngredientFormula": (
            "PlannedIngredientRequirement = PlannedQuantity * "
            "QuantityPerProductUnit"
        ),
        "IngredientMappingStatus": "ILLUSTRATIVE_ASSUMED_RECIPE",
        "IngredientMappingIsActualEdenRecipeData": False,
        "RawTransactionDataIncluded": False,
        "MarchActualTargetsIncluded": False,
        "ModelTrainingCodeIncluded": False,
        "ModelRefitAllowed": False,
        "AuthoritativeSourceCheckpoints": actual_checkpoint_hashes,
        "ReferenceDemonstration": {
            "DailyDate": "2026-03-02",
            "ExpectedRestaurantNormalDemand": EXPECTED_DAILY_TOTAL,
            "WeekStart": "2026-03-02",
            "WeekEnd": "2026-03-06",
            "ExpectedWeekRestaurantNormalDemand": EXPECTED_WEEK_TOTAL,
        },
    }
    write_json(
        directories["contracts"] / "runtime_contract.json",
        runtime_contract,
    )

    ui_contract = {
        "UIContractVersion": "EDEN_UI_V1",
        "BackendFacade": "engine.backend.EdenDemoBackend",
        "RequiredFlows": {
            "DailyForecast": {
                "Inputs": ["date"],
                "Outputs": [
                    "88-product normal-demand forecast",
                    "restaurant total",
                    "confirmed bulk entry",
                    "planned quantities",
                    "optional ingredient requirements for mapped products",
                ],
            },
            "WeekForecast": {
                "Inputs": ["Monday week start"],
                "Outputs": [
                    "daily product plans",
                    "product-week totals",
                    "restaurant daily totals",
                    "restaurant week total",
                    "optional ingredient requirements",
                ],
            },
            "ConfirmedBulk": {
                "InputColumns": [
                    "Date",
                    "CanonicalProductID",
                    "ConfirmedBulkDemand",
                ],
                "Rule": (
                    "Bulk demand is external confirmed demand and is added "
                    "after normal-demand forecasting."
                ),
            },
            "RemainingWeekUpdate": {
                "Input": (
                    "Complete 88-product NormalDemand panel for each completed "
                    "operating day supplied."
                ),
                "BackendMethod": "forecast_remaining_week",
                "Purpose": (
                    "Replace predicted history with known completed-day demand "
                    "and reforecast only the remaining week."
                ),
            },
            "Ingredients": {
                "MappedProducts": EXPECTED_MAPPED_PRODUCTS,
                "UnmappedProductBehaviour": "UNMAPPED_NOT_ZERO_REQUIREMENT",
                "RequiredDisclaimer": (
                    "Prototype ingredient estimates based on illustrative "
                    "assumed recipes, not Eden Restaurant operational recipes."
                ),
            },
            "Exports": [
                "product planning CSV",
                "restaurant totals CSV",
                "ingredient requirements CSV",
            ],
        },
        "HorizonRule": (
            "The recursive system was evaluated through 5 operating days. "
            "Do not silently enable longer-horizon extrapolation in the UI."
        ),
    }
    write_json(
        directories["contracts"] / "ui_contract.json",
        ui_contract,
    )

    # -------------------------------------------------------------------------
    # Runtime source code.
    # -------------------------------------------------------------------------
    write_text(
        directories["engine"] / "__init__.py",
        'from .backend import EdenDemoBackend\n',
    )
    write_text(
        directories["engine"] / "forecast_engine.py",
        textwrap.dedent(FORECAST_ENGINE_SOURCE).lstrip(),
    )
    write_text(
        directories["engine"] / "planning_engine.py",
        textwrap.dedent(PLANNING_ENGINE_SOURCE).lstrip(),
    )
    write_text(
        directories["engine"] / "ingredient_engine.py",
        textwrap.dedent(INGREDIENT_ENGINE_SOURCE).lstrip(),
    )
    write_text(
        directories["engine"] / "backend.py",
        textwrap.dedent(BACKEND_SOURCE).lstrip(),
    )
    write_text(
        STAGING_ROOT / "demo_cli.py",
        textwrap.dedent(CLI_SOURCE).lstrip(),
    )
    write_text(
        directories["tests"] / "test_smoke.py",
        textwrap.dedent(TEST_SMOKE_SOURCE).lstrip(),
    )
    write_text(directories["tests"] / "__init__.py", "")

    # Compile all packaged Python files before runtime testing.
    for source_path in [
        directories["engine"] / "forecast_engine.py",
        directories["engine"] / "planning_engine.py",
        directories["engine"] / "ingredient_engine.py",
        directories["engine"] / "backend.py",
        STAGING_ROOT / "demo_cli.py",
        directories["tests"] / "test_smoke.py",
    ]:
        compile(
            source_path.read_text(encoding="utf-8"),
            str(source_path),
            "exec",
        )

    # -------------------------------------------------------------------------
    # Example outputs for UI development.
    # -------------------------------------------------------------------------
    for target_name, source_path in ND10A_EXAMPLE_FILES.items():
        shutil.copy2(
            source_path,
            directories["examples"] / target_name,
        )
    for target_name, source_path in ND11_EXAMPLE_FILES.items():
        shutil.copy2(
            source_path,
            directories["examples"] / target_name,
        )

    bulk_template = pd.DataFrame(
        [
            {
                "Date": "2026-03-02",
                PRODUCT_ID_COLUMN: "PLU_42533",
                "ConfirmedBulkDemand": 0.0,
            }
        ]
    )
    write_csv(
        directories["examples"] / "confirmed_bulk_orders_template.csv",
        bulk_template,
    )

    actual_update_template = active_catalogue[
        [PRODUCT_ID_COLUMN, PRODUCT_NAME_COLUMN]
    ].copy()
    actual_update_template.insert(0, DATE_COLUMN, "2026-03-02")
    actual_update_template[TARGET_COLUMN] = 0.0
    write_csv(
        directories["examples"] / "remaining_week_actual_update_template.csv",
        actual_update_template,
    )

    # -------------------------------------------------------------------------
    # Documentation for human user and Codex.
    # -------------------------------------------------------------------------
    readme = f"""# Eden Demand Forecasting Demonstration

This is the clean standalone application workspace for the MEng Eden Restaurant
demand-forecasting prototype.

## Frozen backend

- Data cutoff: {cutoff_date.date()}
- Active forecasting catalogue: {EXPECTED_ACTIVE_PRODUCTS} products
- Next-operating-day method: `NESTED_MEDIAN_ENSEMBLE`
- Multi-step / Monday-origin week method: `ROLLING_MEAN_5`
- Daily-updated remaining-week method: `ROLLING_MEAN_5`
- Ingredient-mapped prototype subset: {EXPECTED_MAPPED_PRODUCTS} products
- Ingredient recipes: `ILLUSTRATIVE_ASSUMED_RECIPE`

The application backend must not refit or reselect any model.

## Quick backend check

```bash
python demo_cli.py daily 2026-03-02
python demo_cli.py weekly 2026-03-02
```

Expected normal-demand totals:

- 2026-03-02: {EXPECTED_DAILY_TOTAL:.6f}
- 2026-03-02 to 2026-03-06: {EXPECTED_WEEK_TOTAL:.6f}

## UI development

Read `AGENTS.md`, `docs/CODEX_UI_BRIEF.md`, and
`contracts/ui_contract.json` before changing the app.

UI code should call `engine.backend.EdenDemoBackend`. Do not duplicate or
rewrite forecasting mathematics inside the UI.

## Data boundary

No raw Eden transaction/POS file and no March actual target vault are included.
Only frozen model artifacts, derived runtime state, the active product catalogue,
illustrative ingredient mapping, contracts and example outputs are present.
"""
    write_text(STAGING_ROOT / "README.md", readme)

    agents = f"""# AGENTS.md — Eden Demonstration UI Workspace

## Mission

Build a polished local demonstration UI around the existing frozen backend.
The backend is complete. This workspace is for application/UI implementation,
not model development.

## Non-negotiable model rules

1. Do not fit, refit, tune, calibrate, retrain or reselect any forecasting model.
2. Do not modify files under `model/`.
3. Do not change the 88-product active catalogue policy.
4. Do not replace the selected forecasting methods:
   - next operating day: `NESTED_MEDIAN_ENSEMBLE`
   - multi-step / Monday-origin week: `ROLLING_MEAN_5`
   - daily-updated remaining week: `ROLLING_MEAN_5`
5. Do not introduce March actual targets for scoring or optimisation.
6. Do not call `100 - WAPE` model accuracy.
7. The UI must use `engine.backend.EdenDemoBackend` as its backend facade.

## Required application behaviour

- Forecast an individual operating date.
- Forecast a Monday-Friday week.
- Display product-level forecasts for all {EXPECTED_ACTIVE_PRODUCTS} active products.
- Display restaurant daily and weekly totals.
- Allow confirmed bulk orders to be added after the normal-demand forecast.
- Display final planned quantities = normal forecast + confirmed bulk.
- Provide an optional ingredient view for the {EXPECTED_MAPPED_PRODUCTS} mapped products.
- Preserve unmapped products as `unmapped`; never interpret them as zero ingredient requirement.
- Allow a remaining-week reforecast after a complete actual product-demand panel is supplied for completed day(s).
- Allow CSV export of product plans, restaurant totals and ingredient estimates.
- Make horizon limitations visible. Five operating days is the evaluated recursive horizon; longer forecasts must never be silently presented as equally validated.

## Ingredient disclaimer

Ingredient mappings are realistic illustrative assumptions for the academic prototype.
They are not Eden Restaurant's actual recipes or procurement specifications.
This disclaimer must be visible anywhere ingredient requirements are shown.

## UI design freedom

You may choose the UI framework and visual design. Keep the forecasting and
planning engine code unchanged unless a genuine integration bug is found.
If a backend bug is suspected, document it first rather than silently changing
the model logic.

## Tests

Before considering the UI complete:

```bash
pytest -q
```

The frozen reference forecast must remain:

- Daily 2026-03-02 restaurant normal demand: {EXPECTED_DAILY_TOTAL:.6f}
- Week 2026-03-02 to 2026-03-06 restaurant normal demand: {EXPECTED_WEEK_TOTAL:.6f}
"""
    write_text(STAGING_ROOT / "AGENTS.md", agents)

    codex_brief = f"""# Codex UI Brief

## Goal

Create the final presentation/demo interface for an academic AI demand-forecasting
prototype for Eden Restaurant. The modelling work is frozen and complete.

The interface should make the operational story easy to demonstrate:

1. Select a single future operating date or a Monday-Friday week.
2. Generate normal-demand forecasts for the {EXPECTED_ACTIVE_PRODUCTS}-product active catalogue.
3. Show the restaurant total and ranked product quantities.
4. Add known confirmed bulk orders.
5. Recalculate final planned product quantities.
6. Optionally open **Ingredient Requirements** for the {EXPECTED_MAPPED_PRODUCTS} prototype-mapped products.
7. Export the results.
8. For a week already in progress, allow a complete actual product-demand CSV
   for completed day(s) to update the remaining-week forecast.

## Backend API

Use:

```python
from engine.backend import EdenDemoBackend

backend = EdenDemoBackend()

daily = backend.forecast_date("2026-03-02")
weekly = backend.forecast_week("2026-03-02")
catalogue = backend.catalogue()
mapped = backend.mapped_ingredient_products()
```

Bulk orders are passed as rows with:

- `Date`
- `CanonicalProductID`
- `ConfirmedBulkDemand`

The backend returns pandas DataFrames ready for display/export.

## Suggested screens

A compact single-page dashboard is acceptable. Suggested areas:

- forecast controls;
- restaurant-level KPI cards;
- product planning table;
- confirmed bulk editor;
- product/restaurant charts;
- ingredient requirements panel;
- exports;
- methodology/disclaimer information.

## Presentation language

Use **forecast**, **predicted demand**, and **planned quantity**. Do not label
`100 - WAPE` as accuracy.

Ingredient outputs must visibly state:

> Prototype ingredient estimates based on illustrative assumed recipes, not Eden Restaurant's operational recipes.

## Runtime boundary

The recursive system was evaluated through five operating days. A longer date
can technically be forecast only through explicit extrapolation. The UI should
either restrict normal presentation use to the validated horizon or clearly
require an explicit acknowledgement for extrapolation.

## Frozen reference check

The packaged backend must reproduce:

- {EXPECTED_DAILY_TOTAL:.6f} units on 2 March 2026;
- {EXPECTED_WEEK_TOTAL:.6f} units for 2-6 March 2026.

Do not change backend/model code merely to obtain a different number.
"""
    write_text(
        directories["docs"] / "CODEX_UI_BRIEF.md",
        codex_brief,
    )

    data_boundary = """# Data Boundary

This clean application workspace intentionally excludes the original Eden
transaction/POS dataset and the reserved March actual target vault.

The files under `model/` are frozen deployment artifacts and derived runtime
state required to reproduce the already selected forecasting system. They are
not a substitute for the original transaction dataset and must not be used to
retrain the model.

The visible product catalogue contains only the 88 products in the latest known
active forecasting panel at the 27 February 2026 cutoff.
"""
    write_text(
        directories["docs"] / "DATA_BOUNDARY.md",
        data_boundary,
    )

    architecture = """# Backend Architecture

```text
UI
 |
 v
engine.backend.EdenDemoBackend
 |----------------------|
 v                      v
ForecastEngine       PlanningEngine
                        |
                        v
                  IngredientEngine
```

ForecastEngine loads only frozen model artifacts and packaged runtime state.
PlanningEngine adds confirmed external bulk demand.
IngredientEngine performs deterministic illustrative recipe conversion.

The ingredient layer never changes the forecast.
"""
    write_text(
        directories["docs"] / "BACKEND_ARCHITECTURE.md",
        architecture,
    )

    output_schema = """# Output Semantics

## PredictedNormalDemand
Normal restaurant demand forecast generated by the frozen forecasting system.

## ConfirmedBulkDemand
Known external/confirmed demand entered by the user. It is not predicted.

## PlannedQuantity
`PredictedNormalDemand + ConfirmedBulkDemand`

## PlannedIngredientRequirement
`PlannedQuantity * QuantityPerProductUnit`

Ingredient quantities are available only for the explicitly mapped 14-product
prototype subset.
"""
    write_text(
        directories["docs"] / "OUTPUT_SEMANTICS.md",
        output_schema,
    )

    # -------------------------------------------------------------------------
    # Backend environment requirements.
    # -------------------------------------------------------------------------
    versions = {}
    package_modules = {
        "numpy": "numpy",
        "pandas": "pandas",
        "joblib": "joblib",
        "scikit-learn": "sklearn",
        "xgboost": "xgboost",
        "catboost": "catboost",
    }
    for package_name, module_name in package_modules.items():
        module = importlib.import_module(module_name)
        version = getattr(module, "__version__", None)
        if version is None:
            import importlib.metadata
            version = importlib.metadata.version(package_name)
        versions[package_name] = str(version)

    requirements = "\n".join(
        f"{package}=={version}"
        for package, version in versions.items()
    ) + "\npytest\n"
    write_text(
        STAGING_ROOT / "requirements-backend.txt",
        requirements,
    )
    write_json(
        directories["control"] / "environment_versions.json",
        {
            "Python": platform.python_version(),
            "Packages": versions,
        },
    )

    gitignore = """__pycache__/
*.pyc
.pytest_cache/
.DS_Store
exports/*
!exports/.gitkeep
.venv/
venv/
"""
    write_text(STAGING_ROOT / ".gitignore", gitignore)
    write_text(directories["exports"] / ".gitkeep", "")
    write_text(
        directories["app"] / "README.md",
        (
            "# UI implementation directory\n\n"
            "Codex should place the application/UI implementation here while "
            "keeping the frozen backend under `engine/` unchanged.\n"
        ),
    )

    # -------------------------------------------------------------------------
    # Runtime reproduction tests BEFORE publishing the workspace.
    # -------------------------------------------------------------------------
    sys.path.insert(0, str(STAGING_ROOT))
    try:
        from engine.backend import EdenDemoBackend

        backend = EdenDemoBackend(STAGING_ROOT)

        daily_result = backend.forecast_date("2026-03-02")
        daily_product_plan = daily_result["product_plan"]
        daily_total = float(
            daily_result["restaurant_total"]["PredictedNormalDemand"].iloc[0]
        )
        if len(daily_product_plan) != EXPECTED_ACTIVE_PRODUCTS:
            raise AssertionError(
                f"Packaged daily forecast has {len(daily_product_plan)} "
                f"products instead of {EXPECTED_ACTIVE_PRODUCTS}."
            )
        if not np.isclose(
            daily_total,
            EXPECTED_DAILY_TOTAL,
            atol=1e-6,
            rtol=0.0,
        ):
            raise AssertionError(
                "Packaged daily engine does not reproduce ND09A/ND10A.\n"
                f"Expected: {EXPECTED_DAILY_TOTAL:.9f}\n"
                f"Actual:   {daily_total:.9f}"
            )

        week_result = backend.forecast_week("2026-03-02")
        week_total = float(
            week_result["restaurant_total"]["PredictedNormalDemand"].iloc[0]
        )
        if week_result["daily_product_plan"][DATE_COLUMN].nunique() != 5:
            raise AssertionError(
                "Packaged week forecast does not contain five operating dates."
            )
        if not np.isclose(
            week_total,
            EXPECTED_WEEK_TOTAL,
            atol=1e-6,
            rtol=0.0,
        ):
            raise AssertionError(
                "Packaged weekly engine does not reproduce ND09A/ND10A.\n"
                f"Expected: {EXPECTED_WEEK_TOTAL:.9f}\n"
                f"Actual:   {week_total:.9f}"
            )

        if len(backend.mapped_ingredient_products()) != EXPECTED_MAPPED_PRODUCTS:
            raise AssertionError(
                "Packaged ingredient engine does not contain 14 mapped products."
            )

        # Compare generated ingredient totals with the authoritative ND11 output.
        reference_daily_ingredients = pd.read_csv(
            ND11_EXAMPLE_FILES["daily_ingredient_requirements.csv"],
            low_memory=False,
        )
        reference_week_ingredients = pd.read_csv(
            ND11_EXAMPLE_FILES["week_ingredient_requirements.csv"],
            low_memory=False,
        )
        daily_ingredient_max_diff = compare_ingredient_totals(
            daily_result["ingredient_total"],
            reference_daily_ingredients,
            "Daily",
        )
        week_ingredient_max_diff = compare_ingredient_totals(
            week_result["ingredient_week_total"],
            reference_week_ingredients,
            "Week",
        )

        # Confirm bulk-order propagation into product and ingredient planning.
        mapped_id = (
            backend.mapped_ingredient_products()[PRODUCT_ID_COLUMN]
            .astype(str)
            .iloc[0]
        )
        bulk_test = pd.DataFrame(
            [
                {
                    DATE_COLUMN: "2026-03-02",
                    PRODUCT_ID_COLUMN: mapped_id,
                    "ConfirmedBulkDemand": 10.0,
                }
            ]
        )
        bulk_result = backend.forecast_date(
            "2026-03-02",
            bulk_orders=bulk_test,
        )
        bulk_total = float(
            bulk_result["restaurant_total"]["PlannedQuantity"].iloc[0]
        )
        if not np.isclose(
            bulk_total - daily_total,
            10.0,
            atol=1e-9,
            rtol=0.0,
        ):
            raise AssertionError(
                "Confirmed bulk demand did not propagate correctly."
            )

        smoke_summary = {
            "DailyReferenceExpected": EXPECTED_DAILY_TOTAL,
            "DailyReferenceGenerated": daily_total,
            "WeekReferenceExpected": EXPECTED_WEEK_TOTAL,
            "WeekReferenceGenerated": week_total,
            "DailyProducts": int(len(daily_product_plan)),
            "MappedIngredientProducts": int(
                len(backend.mapped_ingredient_products())
            ),
            "DailyIngredientMaximumDifferenceVsND11": daily_ingredient_max_diff,
            "WeekIngredientMaximumDifferenceVsND11": week_ingredient_max_diff,
            "BulkPropagationTestUnits": 10.0,
            "BulkPropagationPassed": True,
        }
        write_json(
            directories["control"] / "runtime_smoke_test.json",
            smoke_summary,
        )
    finally:
        if str(STAGING_ROOT) in sys.path:
            sys.path.remove(str(STAGING_ROOT))
        for module_name in list(sys.modules):
            if module_name == "engine" or module_name.startswith("engine."):
                del sys.modules[module_name]

    # -------------------------------------------------------------------------
    # Provenance / control.
    # -------------------------------------------------------------------------
    provenance = {
        "PackageStatus": STATUS,
        "CreatedLocalTime": NOW_LOCAL.isoformat(),
        "SourceModelRoot": str(MODEL_ROOT),
        "DemoWorkspace": str(DEMO_WORKSPACE_ROOT),
        "SourceCheckpointHashes": actual_checkpoint_hashes,
        "FrozenDataCutoff": cutoff_date,
        "ActiveProducts": EXPECTED_ACTIVE_PRODUCTS,
        "IngredientMappedProducts": EXPECTED_MAPPED_PRODUCTS,
        "RawEdenTransactionDataCopied": False,
        "MarchActualTargetVaultCopied": False,
        "TrainingOrTuningOutputsCopied": False,
        "ForecastingModelsRefitted": False,
        "ForecastingMethodsReselected": False,
    }
    write_json(
        directories["control"] / "SOURCE_PROVENANCE.json",
        provenance,
    )

    # Build manifest after all user-facing files except manifest/checkpoint itself.
    manifest = build_manifest(STAGING_ROOT)
    write_csv(
        directories["control"] / "artifact_manifest.csv",
        manifest,
    )
    manifest_hash = sha256_file(
        directories["control"] / "artifact_manifest.csv"
    )

    package_checkpoint = {
        "Status": STATUS,
        "CreatedLocalTime": NOW_LOCAL.isoformat(),
        "DemoWorkspace": str(DEMO_WORKSPACE_ROOT),
        "ActiveProducts": EXPECTED_ACTIVE_PRODUCTS,
        "IngredientMappedProducts": EXPECTED_MAPPED_PRODUCTS,
        "DataCutoffDate": cutoff_date,
        "DailyReferenceTotal": daily_total,
        "WeekReferenceTotal": week_total,
        "RuntimeSmokeTestPassed": True,
        "DailyIngredientReproductionPassed": True,
        "WeekIngredientReproductionPassed": True,
        "RawTransactionDataIncluded": False,
        "MarchActualTargetsIncluded": False,
        "ModelsRefitted": False,
        "ManifestSHA256": manifest_hash,
        "SourceCheckpointHashes": actual_checkpoint_hashes,
        "NextStep": "CODEX_UI_IMPLEMENTATION",
    }
    write_json(
        directories["control"] / "DEMO_PACKAGE_CHECKPOINT.json",
        package_checkpoint,
    )
    checkpoint_hash = sha256_file(
        directories["control"] / "DEMO_PACKAGE_CHECKPOINT.json"
    )
    write_text(
        directories["control"] / "DEMO_PACKAGE_CHECKPOINT.sha256",
        checkpoint_hash + "\n",
    )

    # Confirm source files did not change.
    protected_hashes_after = {
        str(path): sha256_file(path)
        for path in required_paths
    }
    changed_sources = [
        path
        for path in protected_hashes_before
        if protected_hashes_before[path] != protected_hashes_after[path]
    ]
    if changed_sources:
        raise AssertionError(
            "A protected modelling source changed during ND12:\n"
            + "\n".join(f"- {path}" for path in changed_sources)
        )

    # Atomic publish.
    os.replace(STAGING_ROOT, DEMO_WORKSPACE_ROOT)

    # Portable ZIP backup next to the workspace.
    shutil.make_archive(
        str(DEMO_WORKSPACE_ROOT),
        "zip",
        root_dir=DEMO_WORKSPACE_ROOT,
        base_dir=".",
    )

    final_manifest = pd.read_csv(
        DEMO_WORKSPACE_ROOT / "control" / "artifact_manifest.csv"
    )

    print("=" * 118)
    print("EDEN NORMAL-DEMAND MODEL V2 — ND12 CLEAN DEMONSTRATION WORKSPACE COMPLETE")
    print("=" * 118)
    print(f"Status: {STATUS}")
    print(f"Local time: {NOW_LOCAL.isoformat()}")
    print(f"Workspace: {DEMO_WORKSPACE_ROOT}")
    print(f"ZIP backup: {zip_path}")
    print()
    print("SOURCE VERIFICATION")
    for name, value in actual_checkpoint_hashes.items():
        print(f"{name} checkpoint SHA-256: {value}")
    print("Previous modelling inputs modified: False")
    print("Raw Eden transaction/POS data copied: False")
    print("March target vault copied/opened: False")
    print()
    print("PACKAGED RUNTIME")
    print(f"Active forecast products: {EXPECTED_ACTIVE_PRODUCTS}")
    print(f"Ingredient-mapped products: {EXPECTED_MAPPED_PRODUCTS}")
    print(f"Frozen data cutoff: {cutoff_date.date()}")
    print("Forecasting models refitted: False")
    print("Forecasting methods reselected: False")
    print()
    print("RUNTIME REPRODUCTION")
    print(
        f"Daily 2026-03-02 expected/generated: "
        f"{EXPECTED_DAILY_TOTAL:.6f} / {daily_total:.6f}"
    )
    print(
        f"Week 2026-03-02 to 2026-03-06 expected/generated: "
        f"{EXPECTED_WEEK_TOTAL:.6f} / {week_total:.6f}"
    )
    print(
        f"Daily ingredient max difference vs ND11: "
        f"{daily_ingredient_max_diff:.12g}"
    )
    print(
        f"Week ingredient max difference vs ND11: "
        f"{week_ingredient_max_diff:.12g}"
    )
    print("Confirmed-bulk propagation smoke test: PASSED")
    print()
    print("WORKSPACE CONTENT")
    print(f"Files in manifest: {len(final_manifest):,}")
    print(f"Backend facade: {DEMO_WORKSPACE_ROOT / 'engine' / 'backend.py'}")
    print(f"Codex brief: {DEMO_WORKSPACE_ROOT / 'docs' / 'CODEX_UI_BRIEF.md'}")
    print(f"AGENTS.md: {DEMO_WORKSPACE_ROOT / 'AGENTS.md'}")
    print(f"UI contract: {DEMO_WORKSPACE_ROOT / 'contracts' / 'ui_contract.json'}")
    print(f"Tests: {DEMO_WORKSPACE_ROOT / 'tests'}")
    print(
        f"Package checkpoint: "
        f"{DEMO_WORKSPACE_ROOT / 'control' / 'DEMO_PACKAGE_CHECKPOINT.json'}"
    )
    print(f"Package checkpoint SHA-256: {checkpoint_hash}")
    print()
    print("NEXT STEP")
    print(
        "Open this standalone folder in Codex and ask Codex to build the UI "
        "according to AGENTS.md and docs/CODEX_UI_BRIEF.md."
    )
    print("=" * 118)

except Exception:
    if STAGING_ROOT.exists():
        shutil.rmtree(STAGING_ROOT, ignore_errors=True)
    raise

/Users/ryansmac/Desktop/Meng Project/.Eden_Demand_Forecasting_Demo_staging_574e34ad5138400c9194a839cf5233e3/engine/planning_engine.py:123: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  plan[BULK_COLUMN] = plan[BULK_COLUMN].fillna(0.0).astype(float)
/Users/ryansmac/Desktop/Meng Project/.Eden_Demand_Forecasting_Demo_staging_574e34ad5138400c9194a839cf5233e3/engine/planning_engine.py:123: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  plan[BULK_COLUMN] = plan[BULK_COLUMN].fillna(0.0).astype(float)


EDEN NORMAL-DEMAND MODEL V2 — ND12 CLEAN DEMONSTRATION WORKSPACE COMPLETE
Status: ND12_CLEAN_DEMONSTRATION_WORKSPACE_CREATED_READY_FOR_CODEX_UI
Local time: 2026-08-10T13:30:15.194998+01:00
Workspace: /Users/ryansmac/Desktop/Meng Project/Eden_Demand_Forecasting_Demo
ZIP backup: /Users/ryansmac/Desktop/Meng Project/Eden_Demand_Forecasting_Demo.zip

SOURCE VERIFICATION
ND09A checkpoint SHA-256: 54767758d9052cb1571043d2f39cc167bc7491b300afbd3605a5416b7aa95c10
ND10A checkpoint SHA-256: 6bcc97c5fe76667c2bdcdde7d0e81b4bd1534c3a601f39fd7acd795e6c87dbff
ND10RA checkpoint SHA-256: fe24555b1562cce7b4300f62051d427492f4e9b3259e662e21f400bac6972c75
ND11 checkpoint SHA-256: 58d497e8eb0e82f18f648537d9d754e438731d4b2ee4a2bc256ec18adba50879
Previous modelling inputs modified: False
Raw Eden transaction/POS data copied: False
March target vault copied/opened: False

PACKAGED RUNTIME
Active forecast products: 88
Ingredient-mapped products: 14
Frozen data cutoff: 2026-02-27
Forecasting models refitted: Fal